# Databricks Inspire AI

**Infuse your Data with Databricks Genie Code AI capabilities to deliver business value.**

Inspire AI reads your Unity Catalog metadata (schemas, table and column names) and uses a 7-phase multi-model AI pipeline to discover, score, and generate production-ready use cases — complete with Genie Code instruction notebooks, documents, and presentations.

---

## What You Get

- **Genie Code Notebooks** — One Databricks notebook per use case with Genie Code instructions, business metadata cards, and markdown documentation. These instructions are designed to be executed directly by Databricks Genie Code.
- **PDF Catalog** — Professional PDF document cataloging all discovered use cases with executive summary, domain grouping, and scoring.
- **PowerPoint Presentation** — Executive-ready slides summarizing domains, priorities, and top use cases.
- **Excel Use Cases Catalog** — Spreadsheet with all use cases, scores, metadata, and prioritization.

**23 Languages**: English, French, German, Spanish, Hindi, Chinese (Mandarin), Japanese, Arabic, Portuguese, Russian, Swedish, Danish, Norwegian, Finnish, Italian, Polish, Romanian, Ukrainian, Dutch, Korean, Indonesian, Malay, Tamil.

---

## Quick Start

1. **Configure**: Fill in **Business Name** and **UC Metadata** (required). Optionally set **Inspire Database** for tracking.
2. **Run**: Click **Run All** to start the engine.
3. **Explore**: Find all generated artifacts in your **Generation Path**.

---

## Configuration Guide

### Operation
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 00 | **Operation** | Controls the top-level workflow: | `Discover Use Cases` |
| | | `Discover Use Cases` — Runs the full discovery pipeline (phases 1-7) **except** Genie Code Instruction generation. Each use case gets a skeleton notebook containing the full metadata and a marker cell with `generate_genie_code_instruction = False`. The `__inspire_usecases` table persists every field needed to later regenerate the Genie Code Instruction (directly-involved schema, FK relationships, full-catalog listing, technique guidance). The `__inspire_session` table persists per-session prompt context (enriched business context, strategic goals, priorities, strategic initiative, value chain, revenue model, generation instructions section). | |
| | | `Generate Use Cases` — **Skips discovery entirely** and only regenerates Genie Code Instructions for use cases selected via one of two routes. <br/>**Route 1 (preferred, DB flag):** Flag rows in Inspire's tracking table via SQL (the Inspire UI also exposes this as a toggle): `UPDATE <inspire_database>.__inspire_usecases SET generate_genie_code_instruction = 'Yes' WHERE id = '<use_case_id>' AND session_id = <session_id>;` then run this notebook. After generation, flagged rows are reset back to `'No'`. <br/>**Route 2 (fallback, notebook flag):** If no DB rows are flagged, Inspire scans notebooks under the **Generation Path** and picks up any whose marker cell sets `generate_genie_code_instruction = True`. The marker is cross-referenced with `__inspire_usecases` to rehydrate the cached prompt fields; regenerating the notebook replaces the marker cell in-place. <br/>**Note:** PDF Catalog and Presentation selections are ignored in this mode. | |

### Required Inputs
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 01 | **Business Name** | Your organization or project name (used for titles, context, and AI prompts). | *(Empty)* |
| 02 | **UC Metadata** | Comma-separated Unity Catalog paths: Catalogs, Schemas, or individual Tables (e.g., `main.finance, sales_catalog, main.hr.employees`). You can mix all three levels. Alternatively, provide an absolute path to a JSON file to run in **Docs-Only mode** (skip discovery, generate artifacts from existing data). | *(Empty)* |

### Optional Tracking
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 03 | **Inspire Database** | A `catalog.schema` path where Inspire stores its tracking tables and result tables (e.g., `my_catalog.inspire_results`). The schema will be created if it does not exist. If left empty, defaults to `main._inspire` as a placeholder. | *(Empty — optional)* |

### Table Election
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 05 | **Table Election** | Controls which tables are used for use case generation: | `Let Inspire Decides` |
| | | `Let Inspire Decides` — AI automatically classifies tables as transactional vs. dimensional and selects the optimal set based on volume thresholds. | |
| | | `All Tables` — Use every business table (no filtering). | |
| | | `Transactional Only` — Only use tables classified as transactional. | |

### Use Cases Quality
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 06 | **Use Cases Quality** | Controls BOTH the generation-time quality gate AND post-generation filtering: | `Balanced` |
| | | `Coverage Mode (All)` — Maximum discovery. Generation gate is permissive and NO post-scoring filter is applied; all scored use cases are retained. Use when you want the widest possible coverage. | |
| | | `Balanced` — Default. Generation gate is stricter and post-scoring filter keeps Medium, High, Very High, and Ultra High quality use cases. | |
| | | `Strict Quality` — Maximum rigor. Tightest generation gate and post-scoring filter keeps only High, Very High, and Ultra High quality use cases. | |
| | | Legacy labels (`Good Quality`, `High Quality`, `Very High Quality`) are still accepted and mapped automatically to the new names. | |

### Strategic Configuration
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 07 | **Business Domains** | Comma-separated business domains to focus on (e.g., `Risk, Finance, Operations`). If provided, use cases will be aligned **only** to these domains. If left empty, the AI will auto-detect and cluster domains from your data. | *(Empty — auto-detected)* |
| 08 | **Business Priorities** | Multi-select strategic priorities. Use cases aligned to selected priorities will score higher: `Increase Revenue`, `Reduce Cost`, `Optimize Operations`, `Mitigate Risk`, `Empower Talent`, `Enhance Experience`, `Drive Innovation`, `Achieve ESG`, `Protect Revenue`, `Execute Strategy`. | `Increase Revenue` |
| 09 | **Generation Instructions** | Free-text instructions for use case generation (e.g., `join table X with Y on column Z, focus on fraud detection, use XGBoost, ignore tables starting with __log`). When provided, these are interpreted by AI and applied as non-negotiable instructions across all generation phases. | *(Empty)* |

### Generation Options
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 10 | **Generation Options** | Multi-select artifacts: | `PDF Catalog` |
| | | `PDF Catalog` — Generate professional PDF documentation per language. | |
| | | `Presentation` — Generate executive-ready PowerPoint slides per language. | |
| | | *(Note: Genie Code Instruction generation is now controlled by the top-level **Operation** widget — see the `Operation` section above.)* | |
| 12 | **Generation Path** | Workspace folder where all outputs are saved (notebooks, PDFs, PPTX, Excel). | `./../demos/inspire_gen/` |
| 13 | **Documents Languages** | Target language(s) for PDF and Presentation artifacts (multi-select). Required when PDF or Presentation is selected. | `English` |

### Advanced Options
| # | Widget | Description | Default |
|:--|:---|:---|:---|
| 14 | **Inspire Session ID** | Optional. Provide a fixed session ID to resume or correlate runs. If left empty, a unique ID is auto-generated. The session ID is used as the merge key in the tracking table. | *(Auto-generated)* |

---

## How It Works (v0.7.3 production release)

Inspire runs a 7-phase AI pipeline using a multi-model cascade (Claude Opus 4.6 / Sonnet 4.6 / Opus 4.5 / Sonnet 4.5 / GPT-OSS 120B / 20B with automatic fallback):

| Phase | What Happens |
|:------|:-------------|
| **1. Initialization** | Extracts business context, interprets the 7-category generation-instruction extraction (see below), infers industry, filters out technical/system tables aggressively. |
| **2. Use Case Generation** | Two passes cover structured analytics, AI/ML, and statistical techniques. All 19 quality gates (D1–D19) are instructed at generation time so the LLM tries to satisfy them as it writes each UC. |
| **3. Domain Clustering** | Clusters use cases into business domains, detects subdomains, merges overlapping domains, deduplicates subdomains across domains, and splits oversized mega-domains. |
| **4. Scoring, Dedup, BoB Ranking & Stratified Trim** | (a) Combined Value + Quality scoring — one LLM pass scores every UC; any HARD VETO caps quality at 2.0. (b) Balanced Quality Filter drops Low / Very Low / Ultra Low. (c) LLM Semantic Dedup removes near-duplicates. (d) **Target-Zone Vetting** — if count > 75 trigger **Best-of-Best ranking** and stratified-by-idea-theme trim. v0.7.3 adds: (i) hard-veto pre-filter that drops UCs with `bob_hard_veto_gate` set BEFORE the theme-floor rule applies, (ii) effective cap jitter so the portfolio size is non-round (default range [91, 110] = `90 + randint(1, 20)`). |
| **5. Genie Code Instructions** | Generates goal-oriented Genie Code instructions per use case. Each instruction describes WHAT to achieve and WHY, letting Genie Code determine the optimal implementation. |
| **6. Summary & Artifacts** | Executive summary, per-use-case Genie Code notebooks (organized by domain/subdomain), PDF catalog, PPTX presentation, Excel catalog with BoB scorecard columns. |
| **7. Translation** | Translates all artifacts into selected languages. |

### The 19 Quality Gates — Ruthless but Fair

Every use case must survive 19 gates. **Every gate is LLM-evaluated — zero regex, zero pattern matching, pure judgment.** Full spec: the **The 19 Quality Gates** section in [`readme.md`](readme.md#the-19-quality-gates).

In v0.7.2 the gates operate in **two modes**:

- **Binary (generation + Combined Scoring)**: each gate passes or fails. HARD VETO caps quality_score at 2.0 so the Balanced filter can drop the UC.
- **0–10 fine-grained (Best-of-Best ranking, when count > 75)**: each gate scored 0–10 per UC. Produces a ranked shortlist that feeds the stratified trim.

**Tier 1 — Technical Grounding (D1–D11)**

| # | Gate | Question |
|---|------|----------|
| D1 | Causal Signal | Do the data columns directly cause or indicate the outcome being predicted? |
| D2 | Cause-Effect Validity | Is the cause-effect relationship logical, proven, industry-recognized? |
| D3 | Data Granularity | Is the data at the right level of detail for the analysis? |
| D4 | Critical Dimensions | Are all required columns actually present in the schema? |
| D5 | Logical Possibility | Can the operation be performed on the given types? |
| D6 | Metric Validity | Can the claimed metric actually be calculated from available fields? |
| D7 | Design–Schema Match | Does the technical design cite columns that exist? |
| D8 | Semantic Uniqueness | After verb / noun / entity / metric normalization — is this a duplicate? |
| D9 | Analytical Depth (Strip Test) | Remove ML-technique mentions — is there real substance left? |
| D10 | Activation Quality (Deliverable Test) | Does the "with …" suffix name a business deliverable, not just a technique? |
| D11 | Domain & Technique Balance | Does the portfolio cluster too heavily on one domain+technique? |

**Tier 2 — Business Value (D12–D14)** — *"would you actually fund this?"*

| # | Gate | Persona | Question |
|---|------|---------|----------|
| D12 | Business Relevance & Opportunity | Stakeholder | Is it relevant day-to-day? Does it deliver tangible value? Can you name the team or role that consumes it? Is it a current pain or a future opportunity? |
| D13 | Sponsor Test | Sponsor / CFO | Would you fund it with a real budget line? Could you pitch it to the CEO without caveats? |
| D14 | Engineering Test | Production Engineer | Can it go to production on existing Databricks capabilities? Can you support and maintain it long-term? Can you fix issues without bespoke infrastructure? |

**Tier 3 — Principal-Business-Analyst (D15–D19)** — the questions a 20-year PBA asks before sign-off

| # | Gate | Question |
|---|------|----------|
| D15 | Decision Cadence Match | Does data refresh speed match decision speed? |
| D16 | The Monday Test | Would a stakeholder's Monday look different? Must name a concrete action. |
| D17 | Explainability & Audit Defensibility | Can you explain the output in 5 minutes without proprietary magic? |
| D18 | 18-Month Longevity | In 18 months, is this still running or abandoned? |
| D19 | Attributable Impact | Can the team prove 6 months later that this moved a measurable dollar amount? |

**Target portfolio after all 19 gates**: 50–100 fundable, operable, explainable, durable, measurable use cases per database. Sweet spot = **75**.

### Best-of-Best Ranking + Stratified Trim (NEW in v0.7.2)

Whenever the portfolio from Phase 2 has at least `bob_min_threshold` UCs (default 20), Inspire scores every UC with the Best-of-Best LLM pass so `bob_score` is populated. When the count additionally exceeds `aggressive_prune_threshold` (default 75), the same flow continues into the Stratified Trim. (v0.8.7: BoB scoring decoupled from trim gate — earlier releases skipped scoring for portfolios in the 20–75 zone, leaving `bob_score = NULL`.) The flow is:

**Step 1 — Best-of-Best scoring**
- Prompt: `BEST_OF_BEST_SCORING_PROMPT`
- Batched (25 UCs per batch), parallel
- For each UC the LLM returns: 19 per-gate scores 0–10 + a 2–4 word `idea_theme` label + short justification

**Step 2 — Tier-weighted composite**

```
tier1 = mean(D1..D11)  # technical grounding
tier2 = mean(D12..D14) # business value
tier3 = mean(D15..D19) # PBA sign-off

bob_score = 0.30*tier1 + 0.35*tier2 + 0.35*tier3

# Hard-veto floor
if any gate D1..D19 < 3:
    bob_score = min(bob_score, 2.0)
```

Weight rationale: technical grounding is a pass/fail check (survivors already sound); business fundability and PBA actionability are the differentiators, equally weighted.

**Step 3 — Stratified trim by idea theme**

```
group UCs by idea_theme
cap_per_theme = target_max * bob_theme_max_share   # default 15% → 15 of 100

FLOOR rule:   every theme keeps ≥ 1 UC            # coverage guarantee
CAP rule:     no theme exceeds cap_per_theme       # no single theme dominates
BUDGET rule:  remaining slots allocated proportional to theme size

within each theme:
    sort by bob_score DESC
    take top-K
```

**Why stratified beats flat top-N**: a flat "keep top 100 by bob_score" would silently wipe out narrow-but-valuable ideas (e.g. 3 strong pricing-optimization UCs beaten by 20 mediocre anomaly-detection UCs). Stratified preserves the discovery breadth — the whole point of Inspire — while ruthlessly culling redundancy within each idea.

**Step 4 — Persisted scorecard**

5 new columns land in `__inspire_usecases` and the Excel/CSV export:

| Column | Type | Values |
|---|---|---|
| `idea_theme` | STRING | e.g. "Anomaly Detection", "Pricing Optimization" |
| `bob_score` | DOUBLE | 0–10 composite (2.0 = hard-veto floor hit) |
| `bob_tier1_score` | DOUBLE | mean of D1–D11, 0–10 |
| `bob_tier2_score` | DOUBLE | mean of D12–D14, 0–10 |
| `bob_tier3_score` | DOUBLE | mean of D15–D19, 0–10 |

Migration is idempotent — safe on existing tables.

**Validation evidence** (tpcds_sf1 Samples database, session `8041728042000030`):

| Metric | Value |
|---|---|
| Phase 2 raw UCs | 378 |
| After dedup + Combined Scoring | 308 |
| After Balanced filter | 244 |
| **After BoB + Stratified Trim** | **100** |
| Distinct idea themes | 75 |
| Largest theme | Anomaly Detection — 11 UCs |
| Avg bob_score | 6.7 |
| bob_score range | 2.0 → 8.23 |
| Tier averages | t1 7.23 · t2 6.92 · t3 6.23 |
| Phase L final-gate | 16/19 passed |
| Run duration | 68 min |

### What changed in v0.7.3

1. ✅ **Hard-veto pre-filter (fix)** — v0.7.2 let hard-veto UCs (`bob_score = 2.0`) survive the trim when alone in their theme. v0.7.3 drops UCs with `bob_hard_veto_gate` set BEFORE grouping by theme. Validated on tpcds_sf1 (session 8042073003073003): 13 hard-veto UCs dropped, 0 survivors in final portfolio.
2. ✅ **Target-max jitter (new)** — v0.7.2 always produced exactly 100 UCs. v0.7.3 computes `effective_cap = target_max_use_cases + randint(1, target_max_jitter)` → default range [91, 110]. Portfolio sizes are now non-round (tpcds v0.7.3 = 105).
3. ✅ **Multi-language filename fix** — Excel + Markdown were overwriting each other across languages. v0.7.3 uses `_{lang_abbr}` suffix when multiple languages are selected (e.g., `readme_en.md`, `readme_de.md`).
4. ✅ **Parallel Route 1/2 Genie regeneration** — v0.7.2 processed UCs sequentially (100 × 3 min = 5 hours). v0.7.3 uses a ThreadPoolExecutor with `genie_regen_parallelism = 10` default → ~30 min for 100 UCs.

### Remaining (non-blocking) issues

1. **Idea themes still too granular** — the BoB prompt asks for 15–25 themes but produces 60–75. Future work: second-pass theme canonicalizer.
2. **Phase L `L12_q3_drop_under_50pct`** flags when >50% of scored UCs are trimmed. Behaviourally correct (we intended the trim) but the gate may need its threshold tuned.

### `Generation Instructions` — 7 Categories (v0.7.0+)

The free-text **09 Generation Instructions** widget is interpreted by an LLM into seven categories. Write naturally — no special syntax. If a category is not mentioned, it is left empty and the pipeline runs as if not provided.

| # | Category | Example Phrase | Effect |
|---|---|---|---|
| 1 | Table Joins | "join orders with customers on customer_id" | Genie Code treats tables as pre-joined |
| 2 | Goal Setting | "focus on fraud detection" | Goal Alignment Filter removes non-aligned UCs |
| 3 | Forced Techniques | "use XGBoost for all classification" | Generator prefers the named technique |
| 4 | Table Ignores | "ignore tables starting with __log" | Filter treats matching tables as TECHNICAL |
| 5 | **Construct Exclusions** (v0.7.0) | "we don't have a loyalty program" | UCs referencing excluded constructs are rejected |
| 6 | **Change-Management Posture** (v0.7.0) | "we're conservative about production changes" | Conservative → rollback mandate; experimental → pilot-scope mandate |
| 7 | **Data Characteristics** (v0.7.0) | "claims table has no labeled fraud outcomes" | Technique selection respects these — no supervised ML on un-labeled tables |

### Runtime config keys (v0.7.2)

| Key | Default | Purpose |
|---|---|---|
| `aggressive_prune_threshold` | 75 | Trigger Stratified Trim when portfolio count exceeds this. (v0.8.7: BoB scoring is no longer gated on this — see `bob_min_threshold`.) |
| `bob_min_threshold` | **20** | v0.8.7: Run Best-of-Best LLM scoring whenever portfolio has at least this many UCs, regardless of trim. Ensures `bob_score` is populated in the tracking table even for small portfolios that don't need trimming. |
| `target_max_use_cases` | **90** | Base for the effective cap (v0.7.3 semantics; was the fixed ceiling pre-v0.7.3) |
| `target_max_jitter` | **20** | Jitter amount: `effective_cap = target_max_use_cases + randint(1, jitter)`; default (90, 20) → [91, 110] |
| `target_min_use_cases` | 50 | Portfolio floor; if undershot, recover from rejection pool |
| `bob_scoring_batch_size` | 25 | UCs per BoB LLM call (smaller than Combined Scoring because 19-score JSON is heavier) |
| `bob_theme_max_share` | 0.15 | Max fraction of target_max per idea theme (default → ~15 UCs per theme) |
| `genie_regen_parallelism` | **10** | Concurrent Route 1/2 Genie regenerations per session (v0.7.3) — ~5h → ~30min for 100 UCs |

### What's new since v0.7.0

- **v0.7.3** — Hard-veto pre-filter (resolves the v0.7.2 hard-veto-through-theme-floor slip-through); target-max jitter for non-round portfolio sizes (default [91, 110]); per-language filename suffix fix (Excel + Markdown now get `_en` / `_de` suffixes in multi-language runs); parallel Route 1/2 Genie regeneration (100 UCs regenerated in ~30 min instead of ~5h).
- **v0.7.2** — Best-of-Best ranking + stratified-by-idea-theme trim; new `BEST_OF_BEST_SCORING_PROMPT`; 5 persisted BoB columns; tier-weighted composite + hard-veto floor.
- **v0.7.1** — 19 quality gates (D1–D19) fully documented in readme; D12–D14 business/sponsor/engineering gates and D15–D19 principal-analyst gates enforced at Combined Scoring; unified MD style across all docs; NameError fix in aggressive vetting.
- **v0.7.0** — Context-extraction upgrade: 7 categories extracted from the `09 Generation Instructions` widget (construct exclusions, change-management posture, data characteristics); `has_genie_code` column added to tracking table.

---

## Docs-Only Mode (JSON Path)

If you provide an absolute file path (starting with `/`) in **UC Metadata** instead of catalog/schema names, Inspire enters **Docs-Only mode**: it loads use cases from the JSON file and generates notebooks, PDFs, and presentations without running discovery or scoring.

---

## Privacy & Trust

<div style="background-color:#f0f7ed; color:#1b5e20; padding:15px; border-radius:8px; border: 1px solid #c3e6cb;">
  <b>Metadata Only</b>: Inspire AI reads <b>only</b> your metadata (catalog, schema, table, and column names) to generate use cases. It does <b>NOT</b> access, read, or sample your actual data at any point during the pipeline.
</div>

---

## Tips

- **Business Priorities vs. Generation Instructions**: Business Priorities are pre-defined categories (multi-select) that influence scoring weights. Generation Instructions are free-text and are parsed by AI into structured directives (table joins, goals, forced algorithms, table ignores) that are applied as non-negotiable rules across all generation phases.
- **Table Election**: For large catalogs (hundreds of tables), `Let Inspire Decides` typically yields the best results by focusing on transactional tables that drive the most valuable analytics.
- **Use Cases Quality**: The default `Balanced` applies a stricter generation-time gate and filters out lower-quality use cases after scoring. Use `Coverage Mode (All)` when you explicitly want every generated candidate (broader coverage, noisier portfolio), or `Strict Quality` when you want only the strongest use cases.
- **Inspire Database**: This is where Inspire persists tracking tables (`__inspire_session`, `__inspire_step`, `__inspire_usecases`) with all generated use cases, scores, and session metadata. Use it to track runs over time or resume work.
- **Session ID**: Provide the same Session ID across runs to update existing use cases in the tracking table rather than creating new entries.
- **Technical Table Filtering**: Inspire always aggressively filters out system tables, audit logs, and technical metadata to focus on business-relevant use cases.
- **Genie Code Instructions**: Each generated notebook contains a detailed instruction that Genie Code can execute autonomously. The instruction describes the business goal, analytical approach, and compliance rules — Genie Code decides the implementation details (SQL, joins, algorithms).



In [0]:
DATABRICKS_INSPIRE_BANNER = r"""
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ____        _        _          _      _                             ┃
┃   |  _ \  __ _| |_ __ _| |__  _ __(_) ___| | _____                      ┃
┃   | | | |/ _` | __/ _` | '_ \| '__| |/ __| |/ / __|                     ┃
┃   | |_| | (_| | || (_| | |_) | |  | | (__|   <\__ \                     ┃
┃   |____/ \__,_|\__\__,_|_.__/|_|  |_|\___|_|\_\___/                     ┃
┃       ___                      _                  _    ___              ┃
┃      |_ _| _ __   ___  _ __   (_) _ __  ___      / \  |_ _|             ┃
┃       | | | '_ \ / __|| '_ \  | || '__|/ _ \    / _ \  | |              ┃
┃       | | | | | |\__ \| |_) | | || |  |  __/   / ___ \ | |              ┃
┃      |___||_| |_||___/| .__/  |_||_|   \___|  /_/   \_\___|             ┃
┃                       |_|                                               ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
"""

import logging

# ==============================================================================
# TECHNICAL CONTEXT - Prompt to Model Type Cascade Configuration
# ==============================================================================
# Each prompt is mapped to a model_type + model_size. At runtime, the system
# resolves (type, size) to an ordered list of enabled models (sorted by "order").
# The lowest-order enabled model is tried first; on failure, the next one is used.
# To change which model bucket a prompt uses, modify "model_type"/"model_size" below.
# To add/remove/reorder models, modify the "models" section.
# ==============================================================================
DEFAULT_MODEL_SIZE_BY_TYPE = {
    "worker": "large",
    "thinker": "large",
}

TECHNICAL_CONTEXT = {
    "runtime": {
        "llm_timeout_seconds": 300,
        "max_retry_attempts": 1,
        "scan_parallelism": 8,
        "metadata_parallelism": 8,
        "global_min_parallelism": 4,
        "global_max_parallelism": 20,
        "default_max_parallelism": 16,
        "cluster_memory_gb": 32,
        "cluster_worker_count": 2,
        "quality_scoring_batch_size": 30,
        "quality_scoring_max_workers": 10,
        "value_scoring_retry_batch_size": 40,
        "value_scoring_max_retry_rounds": 2,
        "batch_processing_timeout": 900,
        "batch_processing_max_attempts": 3,
        "use_combined_scoring": True,
        "table_election_threshold": 5,
        "table_election_transactional_tables_min": 3,
        "min_use_cases_for_selection": 10,
        "aggressive_prune_threshold": 75,  # Trigger Stratified Trim when portfolio > this; LLM aims to settle around 75 (50-100 zone).
        "bob_min_threshold": 20,  # v0.8.7: Run Best-of-Best LLM scoring (without trim) whenever portfolio >= this. Decoupled from trim gate so bob_score is populated for portfolios in the 20-75 zone too.
        "aggressive_vetting_score_floor": 8.5,  # Floor applied during trim-down.
        "target_min_use_cases": 50,  # Recover UP when portfolio < this by pulling from rejection pool.
        "target_max_use_cases": 90,   # Base for effective cap; actual cap = target_max_use_cases + randint(1, target_max_jitter).
        "target_max_jitter": 20,      # Jitter amount; default (90, 20) → effective cap in [91, 110] to avoid round-number artifacts.
        "pass1_temperature": 0.25,
        "pass2_temperature": 0.1,  # B12: lowered from 0.5 (creative gap-fill) to 0.1 (table-grounded gap fill)
        "inspire_score_quality_weight": 0.65,
        "inspire_score_priority_weight": 0.35,
        "model_fallback_chain": [
            {"type": "thinker", "size": "large"},
            {"type": "worker", "size": "large"},
            {"type": "thinker", "size": "small"},
            {"type": "worker", "size": "small"},
            {"type": "thinker", "size": "tiny"},
            {"type": "worker", "size": "tiny"},
        ],
    },
    "prompts_models": [
        # === PHASE 1: INITIALIZATION & CONTEXT EXTRACTION ===
        # Temperature: 0.3-0.4 for accurate extraction of business context
        {"prompt_name": "BUSINESS_CONTEXT_WORKER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Context", "step_name": "Business Context Worker", "progress_increment": 2.0},      # Step 1: Extract business context, goals, priorities
        {"prompt_name": "GENERAL_INSTRUCTION_INTERPRETER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.2, "stage_name": "Initialization", "step_name": "Generation Instructions Interpreter", "progress_increment": 1.0},
        {"prompt_name": "FILTER_BUSINESS_TABLES_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.2, "stage_name": "Table Filtering", "step_name": "Business Table Filter", "progress_increment": 2.0},       # Step 3: Filter business vs technical tables (precision needed)
        
        # === PHASE 2: USE CASE GENERATION (PARALLEL) ===
        # Temperature: 0.25 for Pass 1 (near-deterministic exhaustive discovery), 0.5 for Pass 2 (creative gap-filling)
        {"prompt_name": "BASE_USE_CASE_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.25, "stage_name": "Use Cases Generation", "step_name": "Use Cases", "progress_increment": 4.0},
        
        # === PHASE 3: DOMAIN CLUSTERING ===
        # Temperature: 0.4-0.5 for balanced clustering decisions
        {"prompt_name": "DOMAIN_FINDER_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.5, "stage_name": "Business Domains Clustering", "step_name": "Domain Finder", "progress_increment": 2.0},                # Step 5: Cluster use cases into business domains
        {"prompt_name": "SUBDOMAIN_DETECTOR_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.4, "stage_name": "Business Domain Clustering", "step_name": "Subdomain Detector", "progress_increment": 2.0},           # Step 6: Detect subdomains within each domain
        {"prompt_name": "DOMAINS_MERGER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.4, "stage_name": "Business Domains Clustering", "step_name": "Domain Merger", "progress_increment": 1.0},               # Step 7: Merge similar domains (optional)
        {"prompt_name": "SUBDOMAINS_MERGER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Domain Clustering", "step_name": "Subdomain Merger", "progress_increment": 1.0},            # Step 7.5: Merge duplicate subdomains across domains
        {"prompt_name": "DOMAIN_SPLITTER_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.3, "stage_name": "Business Domain Clustering", "step_name": "Domain Splitter", "progress_increment": 1.0},                # Step 7.6: Split oversized mega-domains
        
        # === PHASE 4: SCORING & DEDUPLICATION (PARALLEL SCORING) ===
        # Temperature: 0.2 for consistent, accurate scoring
        # VALUE SCORE: Focuses on business value, ROI, strategic alignment
        # QUALITY SCORE: Focuses on data sufficiency, cause-effect, high level design validity
        {"prompt_name": "USE_CASE_VALUE_SCORE_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.2, "stage_name": "Use Cases Scoring", "step_name": "Value Scoring", "progress_increment": 2.0},        # Step 8a: Score use cases (ROI, Strategic Alignment) - VALUE focus
        {"prompt_name": "USE_CASE_QUALITY_SCORE_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.1, "stage_name": "Use Cases Scoring", "step_name": "Quality Scoring", "progress_increment": 2.0},      # Step 8b: Score use cases (Data Sufficiency, Cause-Effect) - QUALITY focus
        {"prompt_name": "COMBINED_VALUE_QUALITY_SCORE_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.15, "stage_name": "Use Cases Scoring", "step_name": "Combined Value Quality Scoring", "progress_increment": 3.0}, # Step 8c: Combined value+quality scoring in one pass
        {"prompt_name": "BEST_OF_BEST_SCORING_PROMPT", "model_type": "thinker", "model_size": "large", "temperature": 0.1, "stage_name": "Use Case Prioritization", "step_name": "Best of Best Ranking", "progress_increment": 1.0}, # Step 8d: Per-gate 0-10 scoring + idea theme for aggressive vetting
        {"prompt_name": "REVIEW_USE_CASES_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.1, "stage_name": "Review", "step_name": "Use Cases Review", "progress_increment": 2.0},            # Step 9: Deduplication (conservative - true duplicates only, low temp for consistency)
        {"prompt_name": "GOAL_ALIGNMENT_FILTER_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.0, "stage_name": "Goal Filtering", "step_name": "Goal Alignment Check", "progress_increment": 1.0},
        
        
        # === PHASE 5b: GENIE CODE INSTRUCTIONS GENERATION ===
        # Temperature: 0.3 for balanced, detailed implementation plans
        {"prompt_name": "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.3, "stage_name": "Genie Code Instructions", "step_name": "Genie Instruction Generator", "progress_increment": 4.0},
        
        # === PHASE 6: SUMMARY & ARTIFACTS ===
        # Temperature: 0.5 for engaging summaries
        {"prompt_name": "SUMMARY_GEN_PROMPT", "model_type": "worker", "model_size": "large", "temperature": 0.5, "stage_name": "Summary", "step_name": "Executive Summary", "progress_increment": 1.0},                  # Step 12: Generate executive summary
        
        # === PHASE 7: TRANSLATION (MULTI-LANGUAGE) ===
        # Temperature: 0.2-0.3 for accurate translation
        {"prompt_name": "KEYWORDS_TRANSLATE_PROMPT", "model_type": "worker", "model_size": "tiny", "temperature": 0.2, "stage_name": "Translation", "step_name": "Keywords Translation", "progress_increment": 1.0},           # Step 14: Translate keywords (accuracy)
        {"prompt_name": "USE_CASE_TRANSLATE_PROMPT", "model_type": "worker", "model_size": "tiny", "temperature": 0.3, "stage_name": "Translation", "step_name": "Use Cases Translation", "progress_increment": 1.0},           # Step 15: Translate use cases
    ],
    "models": [
        {
            "name": "claude-opus-4-6",
            "order": 1,
            "type": "thinker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-opus-4-6",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 128000
        },
        {
            "name": "claude-sonnet-4-6",
            "order": 2,
            "type": "worker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-sonnet-4-6",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "claude-opus-4-5",
            "order": 3,
            "type": "thinker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-opus-4-5",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "claude-sonnet-4-5",
            "order": 4,
            "type": "worker",
            "size": "large",
            "enabled": True,
            "llm_endpoint_name": "databricks-claude-sonnet-4-5",
            "llm_input_context_tokens_count": 200000,
            "llm_output_context_tokens_count": 64000
        },
        {
            "name": "gpt-oss-120b",
            "order": 5,
            "type": "worker",
            "size": "small",
            "enabled": True,
            "llm_endpoint_name": "databricks-gpt-oss-120b",
            "llm_input_context_tokens_count": 131072,
            "llm_output_context_tokens_count": 25000
        },
        {
            "name": "gpt-oss-20b",
            "order": 6,
            "type": "worker",
            "size": "tiny",
            "enabled": True,
            "llm_endpoint_name": "databricks-gpt-oss-20b",
            "llm_input_context_tokens_count": 131072,
            "llm_output_context_tokens_count": 25000
        }
    ]
}

MULTI_ENTITY_PROGRESS_PROMPTS = {
    "USE_CASE_VALUE_SCORE_PROMPT",
    "USE_CASE_QUALITY_SCORE_PROMPT",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT",
    "BEST_OF_BEST_SCORING_PROMPT",
    "REVIEW_USE_CASES_PROMPT",
    "KEYWORDS_TRANSLATE_PROMPT",
    "USE_CASE_TRANSLATE_PROMPT",
}

PROMPT_STORY_TITLES = {
    "BUSINESS_CONTEXT_WORKER_PROMPT": ("Business Understanding", "Understanding Business Priorities"),
    "GENERAL_INSTRUCTION_INTERPRETER_PROMPT": ("Business Understanding", "Interpreting Generation Instructions"),
    "FILTER_BUSINESS_TABLES_PROMPT": ("Data Preparation", "Selecting Business Tables"),
    "BASE_USE_CASE_GEN_PROMPT": ("Use Case Design", "Generating Use Cases"),
    "DOMAIN_FINDER_PROMPT": ("Domain Mapping", "Mapping Business Domains"),
    "SUBDOMAIN_DETECTOR_PROMPT": ("Domain Mapping", "Refining Business Subdomains"),
    "DOMAINS_MERGER_PROMPT": ("Domain Mapping", "Consolidating Overlapping Domains"),
    "SUBDOMAINS_MERGER_PROMPT": ("Domain Mapping", "Deduplicating Subdomains Across Domains"),
    "DOMAIN_SPLITTER_PROMPT": ("Domain Mapping", "Rebalancing Oversized Domains"),
    "USE_CASE_VALUE_SCORE_PROMPT": ("Use Case Prioritization", "Scoring Business Value"),
    "USE_CASE_QUALITY_SCORE_PROMPT": ("Use Case Prioritization", "Scoring Use Case Quality"),
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": ("Use Case Prioritization", "Ranking Use Cases"),
    "BEST_OF_BEST_SCORING_PROMPT": ("Use Case Prioritization", "Best-of-Best Ranking"),
    "REVIEW_USE_CASES_PROMPT": ("Use Case Prioritization", "Reviewing and Removing Duplicates"),
    "GOAL_ALIGNMENT_FILTER_PROMPT": ("Goal Filtering", "Filtering Use Cases by Goal Alignment"),
    "SUMMARY_GEN_PROMPT": ("Executive Readout", "Building Executive Summary"),
    "KEYWORDS_TRANSLATE_PROMPT": ("Localization", "Translating Business Keywords"),
    "USE_CASE_TRANSLATE_PROMPT": ("Localization", "Translating Use Cases"),
}

PROMPT_STORY_STARTED_MESSAGES = {
    "BUSINESS_CONTEXT_WORKER_PROMPT": "Inspire has started understanding business priorities and strategic goals",
    "GENERAL_INSTRUCTION_INTERPRETER_PROMPT": "Inspire has started interpreting generation instructions",
    "FILTER_BUSINESS_TABLES_PROMPT": "Inspire has started inspecting business tables for relevance",
    "BASE_USE_CASE_GEN_PROMPT": "Inspire has started generating business use cases",
    "DOMAIN_FINDER_PROMPT": "Inspire has started organizing use cases into business domains",
    "SUBDOMAIN_DETECTOR_PROMPT": "Inspire has started refining domains into subdomains",
    "DOMAINS_MERGER_PROMPT": "Inspire has started consolidating overlapping domains",
    "SUBDOMAINS_MERGER_PROMPT": "Inspire has started deduplicating subdomains across domains",
    "DOMAIN_SPLITTER_PROMPT": "Inspire has started rebalancing oversized domains",
    "USE_CASE_VALUE_SCORE_PROMPT": "Inspire has started evaluating business value for each use case",
    "USE_CASE_QUALITY_SCORE_PROMPT": "Inspire has started evaluating quality and feasibility for each use case",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "Inspire has started ranking use cases by combined value and quality",
    "BEST_OF_BEST_SCORING_PROMPT": "Inspire has started the Best-of-Best ranking with 19-gate per-UC scoring",
    "REVIEW_USE_CASES_PROMPT": "Inspire has started reviewing use cases to remove overlap and improve quality",
    "GOAL_ALIGNMENT_FILTER_PROMPT": "Inspire has started filtering use cases that do not align with user-specified goal",
    "SUMMARY_GEN_PROMPT": "Inspire has started creating the executive summary",
    "KEYWORDS_TRANSLATE_PROMPT": "Inspire has started translating business terminology",
    "USE_CASE_TRANSLATE_PROMPT": "Inspire has started translating use case outcomes",
}

def _validate_runtime_config():
    _rt = TECHNICAL_CONTEXT["runtime"]
    _bounds = {
        "llm_timeout_seconds": (30, 1800),
        "max_retry_attempts": (0, 5),
        "scan_parallelism": (1, 20),
        "metadata_parallelism": (1, 20),
        "global_min_parallelism": (1, 20),
        "global_max_parallelism": (2, 50),
        "default_max_parallelism": (1, 50),
        "cluster_memory_gb": (4, 512),
        "cluster_worker_count": (1, 64),
        "quality_scoring_batch_size": (5, 200),
        "quality_scoring_max_workers": (1, 20),
        "value_scoring_retry_batch_size": (5, 100),
        "value_scoring_max_retry_rounds": (0, 5),
        "batch_processing_timeout": (60, 3600),
        "batch_processing_max_attempts": (1, 10),
        "table_election_threshold": (1, 500),
        "table_election_transactional_tables_min": (1, 100),
    }
    for key, (lo, hi) in _bounds.items():
        val = _rt.get(key)
        if val is None:
            raise ValueError(f"TECHNICAL_CONTEXT.runtime missing required key: '{key}'")
        if not isinstance(val, (int, float)):
            raise TypeError(f"TECHNICAL_CONTEXT.runtime['{key}'] must be numeric, got {type(val).__name__}")
        if val < lo or val > hi:
            raise ValueError(f"TECHNICAL_CONTEXT.runtime['{key}']={val} out of bounds [{lo}, {hi}]")

    _chain = _rt.get("model_fallback_chain")
    if not isinstance(_chain, list) or not _chain:
        raise ValueError("TECHNICAL_CONTEXT.runtime.model_fallback_chain must be a non-empty list")
    _valid_types = {"worker", "thinker"}
    _valid_sizes = {"tiny", "small", "large"}
    for _idx, _entry in enumerate(_chain):
        if not isinstance(_entry, dict):
            raise TypeError(f"model_fallback_chain[{_idx}] must be a dict, got {type(_entry).__name__}")
        if _entry.get("type") not in _valid_types:
            raise ValueError(f"model_fallback_chain[{_idx}].type must be one of {sorted(_valid_types)}")
        if _entry.get("size") not in _valid_sizes:
            raise ValueError(f"model_fallback_chain[{_idx}].size must be one of {sorted(_valid_sizes)}")

_validate_runtime_config()

def _validate_model_config():
    valid_types = {"worker", "thinker"}
    valid_sizes = {"tiny", "small", "large"}
    models = TECHNICAL_CONTEXT.get("models", [])
    prompt_models = TECHNICAL_CONTEXT.get("prompts_models", [])

    if not isinstance(models, list) or not models:
        raise ValueError("TECHNICAL_CONTEXT.models must be a non-empty list")

    for idx, model in enumerate(models):
        model_type = model.get("type")
        model_size = model.get("size")
        enabled = model.get("enabled")
        if model_type not in valid_types:
            raise ValueError(f"TECHNICAL_CONTEXT.models[{idx}].type must be one of {sorted(valid_types)}")
        if model_size not in valid_sizes:
            raise ValueError(f"TECHNICAL_CONTEXT.models[{idx}].size must be one of {sorted(valid_sizes)}")
        if not isinstance(enabled, bool):
            raise TypeError(f"TECHNICAL_CONTEXT.models[{idx}].enabled must be boolean")

    enabled_pairs = {(m.get("type"), m.get("size")) for m in models if m.get("enabled") is True}
    fallback_chain = TECHNICAL_CONTEXT["runtime"].get("model_fallback_chain", [])
    for idx, pm in enumerate(prompt_models):
        model_type = pm.get("model_type", "worker")
        model_size = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(model_type, "small"))
        if model_type not in valid_types:
            raise ValueError(f"TECHNICAL_CONTEXT.prompts_models[{idx}].model_type must be one of {sorted(valid_types)}")
        if model_size not in valid_sizes:
            raise ValueError(f"TECHNICAL_CONTEXT.prompts_models[{idx}].model_size must be one of {sorted(valid_sizes)}")
        found_in_chain = False
        started = False
        for chain_entry in fallback_chain:
            ct, cs = chain_entry.get("type"), chain_entry.get("size")
            if ct == model_type and cs == model_size:
                started = True
            if started and (ct, cs) in enabled_pairs:
                found_in_chain = True
                break
        if not found_in_chain:
            raise ValueError(
                f"No enabled model reachable for prompts_models[{idx}] '{pm.get('prompt_name')}' "
                f"(designated bucket: {model_type}/{model_size}) via fallback chain"
            )

_validate_model_config()


def get_models_by_type(model_type: str) -> list:
    """
    Get all models of a given type (thinker/worker) sorted by order (ascending).
    The lowest-order model is tried first; on failure, the next one is used.
    """
    return sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if m.get("type") == model_type and bool(m.get("enabled", True))],
        key=lambda m: m.get("order", 999)
    )

def get_models_by_type_and_size(model_type: str, model_size: str) -> list:
    """
    Get enabled models matching (type, size), sorted by order (ascending).
    """
    return sorted(
        [
            m for m in TECHNICAL_CONTEXT["models"]
            if m.get("type") == model_type and m.get("size") == model_size and bool(m.get("enabled", True))
        ],
        key=lambda m: m.get("order", 999)
    )

def get_model_cascade_for_prompt(prompt_name: str) -> list:
    """
    Get the ordered list of model configs for a prompt's (model_type, model_size).
    Used for cascade fallback: try model[0], on failure try model[1], etc.

    Falls back to worker models with default worker size if prompt_name is not in prompts_models
    (e.g., dynamic prompt names like 'Assign_User_Domains_English').
    """
    for pm in TECHNICAL_CONTEXT["prompts_models"]:
        if pm["prompt_name"] == prompt_name:
            model_type = pm.get("model_type", "worker")
            model_size = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(model_type, "small"))
            return get_models_by_type_and_size(model_type, model_size)
    return get_models_by_type_and_size("worker", DEFAULT_MODEL_SIZE_BY_TYPE.get("worker", "small"))

def _get_prompt_bucket(prompt_name: str) -> tuple:
    for pm in TECHNICAL_CONTEXT["prompts_models"]:
        if pm["prompt_name"] == prompt_name:
            mt = pm.get("model_type", "worker")
            ms = pm.get("model_size", DEFAULT_MODEL_SIZE_BY_TYPE.get(mt, "small"))
            return (mt, ms)
    default_type = "worker"
    return (default_type, DEFAULT_MODEL_SIZE_BY_TYPE.get(default_type, "small"))

def _get_prompt_temperature(prompt_name: str) -> float:
    for pm in TECHNICAL_CONTEXT["prompts_models"]:
        if pm["prompt_name"] == prompt_name:
            return pm.get("temperature", 0.3)
    return 0.3


def get_full_fallback_cascade(prompt_name: str) -> list:
    designated_type, designated_size = _get_prompt_bucket(prompt_name)
    chain = TECHNICAL_CONTEXT["runtime"].get("model_fallback_chain", [])
    started = False
    cascade = []
    for entry in chain:
        ct, cs = entry.get("type"), entry.get("size")
        if ct == designated_type and cs == designated_size:
            started = True
        if started:
            bucket_models = get_models_by_type_and_size(ct, cs)
            cascade.extend(bucket_models)
    if not cascade:
        cascade = get_models_by_type_and_size(designated_type, designated_size)
    if not cascade:
        all_enabled = sorted(
            [m for m in TECHNICAL_CONTEXT["models"] if bool(m.get("enabled", True))],
            key=lambda m: m.get("order", 999)
        )
        cascade = all_enabled
    return cascade


def get_model_endpoint_for_prompt(prompt_name: str) -> str:
    """
    Get the primary LLM endpoint name for a given prompt using model_type cascade.
    Returns the endpoint of the lowest-order model matching the prompt's model_type.
    """
    cascade = get_model_cascade_for_prompt(prompt_name)
    if cascade:
        return cascade[0]["llm_endpoint_name"]
    all_models = sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if bool(m.get("enabled", True))],
        key=lambda m: m.get("order", 999)
    )
    return all_models[0]["llm_endpoint_name"] if all_models else "databricks-gpt-oss-20b"

def get_model_config_for_prompt(prompt_name: str) -> dict:
    """
    Get the primary model configuration for a given prompt using model_type cascade.
    Returns the config of the lowest-order model matching the prompt's model_type.
    """
    cascade = get_model_cascade_for_prompt(prompt_name)
    if cascade:
        return cascade[0]
    all_models = sorted(
        [m for m in TECHNICAL_CONTEXT["models"] if bool(m.get("enabled", True))],
        key=lambda m: m.get("order", 999)
    )
    return all_models[0] if all_models else {}

PROMPT_TRACKING_CONFIG = {}
for _pm in TECHNICAL_CONTEXT.get("prompts_models", []):
    _prompt_name = _pm.get("prompt_name")
    if not _prompt_name:
        continue
    PROMPT_TRACKING_CONFIG[_prompt_name] = {
        "stage_name": _pm.get("stage_name", "Prompt Execution"),
        "step_name": _pm.get("step_name", _prompt_name),
        "progress_increment": float(_pm.get("progress_increment", 0.5)),
    }

def log_print(message: str, level: str = "INFO", flush: bool = True):
    """Print a message with timestamp in logger format for immediate console output.
    For ERROR/CRITICAL levels, also writes to stderr for visibility.
    """
    import time as _time
    import sys as _sys
    timestamp = _time.strftime('%H:%M:%S')
    formatted_msg = f"{timestamp} - {level} - {message}"
    print(formatted_msg, flush=flush)
    if flush:
        _sys.stdout.flush()
    
    level_upper = level.upper()
    if level_upper in ("ERROR", "CRITICAL"):
        print(formatted_msg, file=_sys.stderr, flush=True)


# =============================================================================
# PHASE A / N / H / I SCAFFOLDS (foundation substrate)
# Industry-agnostic, import-free, module-level singletons that later phases
# extend. Kept minimal so they do not affect pipeline behavior today.
# =============================================================================

def _utc_iso8601():
    """Phase N1 / Gap 16: UTC ISO-8601 with millisecond precision and Z suffix."""
    import datetime as _dt
    return _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


class _NextActionsCollector:
    """Phase H1 scaffold — singleton for operator-actionable findings.
    Full taxonomy + triage + persistence live in Phase H (later).
    Today it just keeps an in-memory list callers can grow from any phase."""
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            inst = super().__new__(cls)
            inst._actions = []
            import threading as _threading
            inst._lock = _threading.Lock()
            cls._instance = inst
        return cls._instance

    def add(self, rule_id, severity="INFO", phase="", step="", evidence="",
            suggested_user_action="", fix_hint="", **provenance):
        allowed = ("BLOCKING", "SAFE_IGNORE", "INFO", "HIGH")
        sev = severity if severity in allowed else "INFO"
        entry = {
            "ts": _utc_iso8601(),
            "rule_id": str(rule_id or "UNKNOWN"),
            "severity": sev,
            "phase": str(phase or ""),
            "step": str(step or ""),
            "evidence": str(evidence or "")[:500],
            "suggested_user_action": str(suggested_user_action or "")[:500],
            "fix_hint": str(fix_hint or "")[:500],
            "provenance": {k: str(v)[:200] for k, v in provenance.items()},
        }
        with self._lock:
            self._actions.append(entry)
        return entry

    def summary(self):
        with self._lock:
            counts = {}
            for a in self._actions:
                counts[a["severity"]] = counts.get(a["severity"], 0) + 1
            return {"total": len(self._actions), "by_severity": counts}

    def snapshot(self):
        """Return a shallow copy of current entries for read-only inspection."""
        with self._lock:
            return list(self._actions)

    def assert_no_blocking(self):
        with self._lock:
            blocking = [a for a in self._actions if a["severity"] == "BLOCKING"]
        if blocking:
            raise RuntimeError(
                f"NEXT_ACTIONS has {len(blocking)} BLOCKING entries: "
                + ", ".join(a["rule_id"] for a in blocking[:10])
            )

    def export_jsonl(self, path):
        with self._lock:
            snapshot = list(self._actions)
        with open(path, "w", encoding="utf-8") as f:
            import json as _json
            for entry in snapshot:
                f.write(_json.dumps(entry, ensure_ascii=False) + "\n")


NEXT_ACTIONS = _NextActionsCollector()


class _EventsEmitter:
    """Phase A11 / Phase N scaffold — append-only JSONL event stream.
    Emits to /tmp/{run_id}/events.jsonl when run_id is set via bind(). Full
    Delta dual-write (Phase N1a) lives in Phase N implementation later."""
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            inst = super().__new__(cls)
            inst._path = None
            inst._run_id = None
            inst._session_id = None
            inst._business_name = None
            inst._seq = 0
            import threading as _threading
            inst._lock = _threading.Lock()
            cls._instance = inst
        return cls._instance

    def bind(self, run_id=None, session_id=None, business_name=None, base_tmp="/tmp"):
        import os as _os
        with self._lock:
            self._run_id = run_id
            self._session_id = session_id
            self._business_name = business_name
            if run_id:
                run_dir = _os.path.join(base_tmp, str(run_id))
                try:
                    _os.makedirs(run_dir, exist_ok=True)
                    self._path = _os.path.join(run_dir, "events.jsonl")
                except Exception:
                    self._path = None

    def emit(self, phase, step, status, duration_ms=None, rss_mb=None,
             tokens_in=None, tokens_out=None, llm_response_id=None,
             prompt_template=None, evidence=None, exception_type=None,
             exception_msg=None, next_actions_emitted=None,
             integrity_gate_result=None):
        with self._lock:
            self._seq += 1
            seq = self._seq
            path = self._path
            run_id = self._run_id
            sid = self._session_id
            bn = self._business_name
        event = {
            "ts": _utc_iso8601(),
            "run_id": str(run_id or ""),
            "session_id": sid,
            "business_name": bn,
            "phase": str(phase or ""),
            "step": str(step or ""),
            "status": str(status or ""),
            "duration_ms": duration_ms,
            "rss_mb": rss_mb,
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "llm_response_id": llm_response_id,
            "prompt_template": prompt_template,
            "evidence": evidence,
            "exception_type": exception_type,
            "exception_msg": exception_msg,
            "next_actions_emitted": list(next_actions_emitted or []),
            "integrity_gate_result": integrity_gate_result,
            "event_seq": seq,
        }
        if path:
            try:
                import json as _json
                with open(path, "a", encoding="utf-8") as f:
                    f.write(_json.dumps(event, ensure_ascii=False) + "\n")
            except Exception:
                pass
        return event


EVENTS = _EventsEmitter()


# --- Phase H2: closed taxonomy of NEXT_ACTIONS rule_ids ---
NEXT_ACTIONS_TAXONOMY = {
    "BLOCKING": {
        "METADATA_ONLY_CONTRACT_VIOLATED", "INSPIRE_DATA_ACCESS_PROMPT_LEAKAGE",
        "UC_GENERATION_FAILED", "GENIE_INSTRUCTION_EMPTY_FOR_FLAGGED_UC",
        "UC_REFERENCES_NONEXISTENT_TABLE", "UC_NOT_GROUNDED", "UC_NOT_RELEVANT",
        "UC_DEDUP_RATE_HIGH", "UC_RECALL_GAP",
        "UC_TRANSLATION_DROPPED_IDENTIFIER", "UC_TRANSLATION_DROPPED_NUMBER",
        "UC_TRANSLATION_LENGTH_DRIFT", "UC_TRANSLATION_SEMANTIC_DRIFT",
        "UC_STALE_CATALOG_DRIFT", "SCHEMA_MIGRATION_FAILED",
        "WIDGET_COMPLIANCE_VIOLATED", "PLACEHOLDER_LEAKAGE", "NULL_OUTPUT_SENTINEL",
        "REGRESSED_WARNING", "SUSPICIOUS_SAFE_IGNORE", "VOLUME_ESCALATED",
        "SHADOW_DIFF_DIVERGENCE", "BASELINE_QUALITY_REGRESSION",
        "PREFLIGHT_INVARIANT_FAILURE", "VALIDATION_PARSE_DEFAULT_REJECT",
        "TABLE_FILTER_PARSE_FAILED", "MEMORY_BUDGET_EXCEEDED",
        "VOLUME_LOG_UPLOAD_FAILED", "STEP_TABLE_DUAL_WRITE_DRIFT",
        "BUSINESS_NAME_LOCK_HELD",
        "BUSINESS_NAME_LOCK_FORCE_ACQUIRED",
        "SENSITIVE_CREDENTIAL_OVERRIDE_REFUSED", "LOG_CONTENT_LEAKAGE",
        "PLAN_ANCHOR_DRIFT",
    },
    "SAFE_IGNORE": {
        "LOW_QUALITY_UC", "MISSING_FK_FOR_UC_TABLES", "UC_NAMING_QUIBBLE",
        "BUSINESS_PRIORITY_PARTIAL_MATCH", "PHASE_5B_TIMEOUT_RECOVERED",
        "LANGUAGE_TRANSLATION_FALLBACK", "JOBLAUNCHER_PATH_MIGRATION_NOTICE",
        "DOMAIN_AUTO_DETECTED", "UC_NEAR_DUPLICATE_MERGED",
        "UC_EXACT_DUPLICATE_DROPPED", "UC_RECALL_GAP_ACCEPTED_BY_WIDGET",
        "UC_RECALL_GAP_ACCEPTED_SENSITIVITY", "COLUMN_ID_FUZZY_MATCH_REJECTED",
        "GENIE_CODE_DATA_ACCESS_BY_DESIGN", "R3_DETERMINISTIC_FALLBACK_PASS",
        "R3_DETERMINISTIC_FALLBACK_DROP", "BUSINESS_NAME_PATH_COLLISION_AVOIDED",
        "CATALOG_DRIFT_CHECK_NO_DRIFT",
    },
    "HIGH": {
        "R3_LLM_UNAVAILABLE_FALLBACK_USED", "NO_OPERATIONAL_PATTERNS_DETECTED",
        "DEDUP_LLM_JUDGE_UNAVAILABLE", "MICRO_FIXTURE_NOT_SEEDED",
        "UC_SCHEMA_INCOMPLETE", "UC_CATCHALL_TECHNIQUE", "UC_COLUMN_UNGROUNDED",
    },
    "INFO": {
        "LEXICON_NOT_AVAILABLE",
    },
}


# ============================================================
# Phase F: UseCaseBundle + Visitor + Validator Registry
# ============================================================
# Structured representation of a single use-case for validator/visitor pipelines.
# Lives alongside the dict-based UC representation used across the codebase. Adapters
# (from_dict / to_dict) keep it interop-safe with existing callers.

from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional, Tuple


@dataclass
class UseCaseBundle:
    no: str = ""
    name: str = ""
    business_domain: str = ""
    subdomain: str = ""
    type: str = ""
    analytics_technique: str = ""
    priority: str = ""
    primary_table: str = ""
    tables_involved: str = ""
    statement: str = ""
    solution: str = ""
    business_value: str = ""
    quality: str = ""
    result_table: str = ""
    genie_instruction: str = ""
    code_structure_mandate: str = ""
    provenance: Dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "UseCaseBundle":
        g = lambda k, default="": d.get(k, default) if d else default
        return cls(
            no=str(g("No") or ""),
            name=str(g("Name") or ""),
            business_domain=str(g("Business Domain") or ""),
            subdomain=str(g("Subdomain") or ""),
            type=str(g("type") or ""),
            analytics_technique=str(g("Analytics Technique") or ""),
            priority=str(g("Priority") or ""),
            primary_table=str(g("Primary Table") or ""),
            tables_involved=str(g("Tables Involved") or ""),
            statement=str(g("Statement") or ""),
            solution=str(g("Solution") or ""),
            business_value=str(g("Business Value") or ""),
            quality=str(g("Quality") or ""),
            result_table=str(g("result_table") or ""),
            genie_instruction=str(g("genie_instruction") or ""),
            code_structure_mandate=str(g("code_structure_mandate") or ""),
            provenance={k: v for k, v in (d.items() if d else [])
                        if k not in {"No", "Name", "Business Domain", "Subdomain", "type",
                                     "Analytics Technique", "Priority", "Primary Table",
                                     "Tables Involved", "Statement", "Solution",
                                     "Business Value", "Quality", "result_table",
                                     "genie_instruction", "code_structure_mandate"}},
        )

    def to_dict(self) -> Dict[str, Any]:
        out = {
            "No": self.no, "Name": self.name, "Business Domain": self.business_domain,
            "Subdomain": self.subdomain, "type": self.type,
            "Analytics Technique": self.analytics_technique, "Priority": self.priority,
            "Primary Table": self.primary_table, "Tables Involved": self.tables_involved,
            "Statement": self.statement, "Solution": self.solution,
            "Business Value": self.business_value, "Quality": self.quality,
            "result_table": self.result_table, "genie_instruction": self.genie_instruction,
            "code_structure_mandate": self.code_structure_mandate,
        }
        out.update(self.provenance or {})
        return out


class UseCaseBundleVisitor:
    """Visitor-pattern base. Override visit_<field> or visit_bundle to inspect/mutate."""
    def visit_bundle(self, bundle: UseCaseBundle) -> UseCaseBundle:
        return bundle

    def visit_batch(self, bundles: List[UseCaseBundle]) -> List[UseCaseBundle]:
        return [self.visit_bundle(b) for b in bundles]


# Registry: {name: callable(bundle) -> (passed: bool, reason: str)}
# Callers import this and register their own domain-agnostic validators.
VALIDATOR_REGISTRY: Dict[str, Callable[[UseCaseBundle], Tuple[bool, str]]] = {}


def register_validator(name: str):
    def _deco(fn: Callable[[UseCaseBundle], Tuple[bool, str]]):
        VALIDATOR_REGISTRY[name] = fn
        return fn
    return _deco


@register_validator("has_primary_table")
def _v_has_primary_table(b: UseCaseBundle) -> Tuple[bool, str]:
    if not (b.primary_table or "").strip():
        return False, "missing primary table"
    return True, ""


@register_validator("statement_min_length")
def _v_statement_min_length(b: UseCaseBundle) -> Tuple[bool, str]:
    if len((b.statement or "").strip()) < 20:
        return False, "statement too short"
    return True, ""


@register_validator("solution_min_length")
def _v_solution_min_length(b: UseCaseBundle) -> Tuple[bool, str]:
    if len((b.solution or "").strip()) < 20:
        return False, "solution too short"
    return True, ""


@register_validator("technique_present")
def _v_technique_present(b: UseCaseBundle) -> Tuple[bool, str]:
    if not (b.analytics_technique or "").strip():
        return False, "missing analytics technique"
    return True, ""


def run_validators(bundle: UseCaseBundle,
                   validators: Optional[List[str]] = None) -> Dict[str, Tuple[bool, str]]:
    """Run registered validators against a bundle. Returns {name: (passed, reason)}."""
    targets = validators or list(VALIDATOR_REGISTRY.keys())
    out = {}
    for n in targets:
        fn = VALIDATOR_REGISTRY.get(n)
        if not fn:
            continue
        try:
            out[n] = fn(bundle)
        except Exception as _err:
            out[n] = (False, f"validator_exception: {str(_err)[:80]}")
    return out







class _MemoryGuardScaffold:
    """Phase A3 / Phase I scaffold. Later phase extends with RSS checks,
    spill-to-disk, and budget enforcement."""
    def __init__(self, budget_mb=24 * 1024):
        self.budget_mb = int(budget_mb)

    def check(self, step_name=""):
        try:
            import psutil as _psutil, os as _os
            rss_mb = _psutil.Process(_os.getpid()).memory_info().rss / (1024 * 1024)
            return {"step": step_name, "rss_mb": rss_mb, "over_budget": rss_mb > self.budget_mb}
        except Exception:
            return {"step": step_name, "rss_mb": None, "over_budget": False}


_MEMORY_GUARD = _MemoryGuardScaffold()

# =============================================================================
# END PHASE A/N/H/I SCAFFOLDS
# =============================================================================


# =============================================================================
# PHASE C — METADATA-ONLY CONTRACT GUARD
# C1 (stub): inspire_sql wrapper with allowlist. Call sites not yet migrated —
# use as a standalone utility + run C3 banned-phrase scan at load.
# =============================================================================

_INSPIRE_SQL_ALLOWLIST = [
    r"^\s*SELECT\b.*\bFROM\s+[`\w.]+\.information_schema\.",
    r"^\s*(SELECT|MERGE|UPDATE|INSERT|DELETE|CREATE|ALTER|DESCRIBE|DROP)\b",
    r"^\s*CREATE\s+(TABLE|DATABASE|SCHEMA|VOLUME)\s+IF\s+NOT\s+EXISTS\b",
    r"^\s*CREATE\s+OR\s+REPLACE\s+TEMP\s+VIEW\b",
    r"^\s*SHOW\s+(TABLES|SCHEMAS|DATABASES|COLUMNS)\b",
]
_INSPIRE_SQL_BANNED = [
    r"\bDESCRIBE\s+TABLE\s+EXTENDED\b",
    r"\bDESCRIBE\s+EXTENDED\b",
]


def inspire_sql(query, args=None, spark_session=None):
    """Phase C1 wrapper around spark.sql with allowlist + banned-phrase check.
    Drop-in replacement for spark.sql(); raises if the query violates the
    metadata-only contract. This is a utility — existing call sites have NOT
    been migrated yet (39 total). Use for new call sites."""
    import re as _re
    q = str(query or "")
    for pat in _INSPIRE_SQL_BANNED:
        if _re.search(pat, q, _re.IGNORECASE):
            try:
                NEXT_ACTIONS.add("METADATA_ONLY_CONTRACT_VIOLATED", severity="BLOCKING",
                                 phase="C1", evidence=f"Banned pattern matched: {pat}")
            except Exception:
                pass
            raise PermissionError(f"Phase C1: query violates metadata-only contract (banned): {q[:80]}")
    allowed = any(_re.search(pat, q, _re.IGNORECASE) for pat in _INSPIRE_SQL_ALLOWLIST)
    if not allowed:
        try:
            NEXT_ACTIONS.add("METADATA_ONLY_CONTRACT_VIOLATED", severity="BLOCKING",
                             phase="C1", evidence=f"Not on allowlist: {q[:120]}")
        except Exception:
            pass
        raise PermissionError(f"Phase C1: query not on allowlist: {q[:80]}")
    _spark = spark_session
    if _spark is None:
        _spark = globals().get("spark")
    if _spark is None:
        raise RuntimeError("Phase C1: no spark session available")
    if args is None:
        return _spark.sql(q)
    return _spark.sql(q, args)


_C3_BANNED_PROMPT_PHRASES = [
    r"\bsample\s+rows\b",
    r"\bselect\s+\*\s+from\s+\{[^}]*tables_involved",
    r"\bdescribe\s+detail\b",
    r"\bdescribe\s+table\s+extended\b",
    r"\bshow\s+me\s+data\b",
    r"\bquery\s+the\s+table\b",
    r"\bcount\s+the\s+rows\b",
    r"\bread\s+from\s+the\s+table\b",
    r"\bfetch\s+sample\b",
    r"\bget\s+the\s+data\b",
    r"\binspect\s+the\s+rows\b",
]


def audit_prompts_for_data_access_leak(prompt_templates_dict, logger=None):
    """Phase C3: scan every prompt body for banned data-access pressure phrases,
    ignoring fenced code blocks (which belong to generated Genie code per C5 carve-out).
    Emits NEXT_ACTIONS BLOCKING on match; returns list of (prompt_name, phrase, context)."""
    import re as _re
    if not isinstance(prompt_templates_dict, dict):
        return []
    findings = []
    fence_re = _re.compile(r"```.*?```", _re.DOTALL)
    for name, body in prompt_templates_dict.items():
        if not isinstance(body, str):
            continue
        # Strip fenced code blocks (C5 carve-out)
        stripped = fence_re.sub("", body)
        for pat in _C3_BANNED_PROMPT_PHRASES:
            m = _re.search(pat, stripped, _re.IGNORECASE)
            if m:
                context = stripped[max(0, m.start()-40):m.end()+40]
                findings.append({"prompt": name, "pattern": pat, "context": context})
                try:
                    NEXT_ACTIONS.add(
                        "INSPIRE_DATA_ACCESS_PROMPT_LEAKAGE",
                        severity="BLOCKING", phase="C3",
                        evidence=f"{name}: {pat} ~ {context[:100]}",
                        suggested_user_action="Rewrite prompt to work from supplied schema only (no data SELECT pressure).",
                    )
                except Exception:
                    pass
                if logger is not None:
                    logger.warning(f"C3 prompt leak: {name} ~ {pat}")
    return findings

# =============================================================================
# END PHASE C
# =============================================================================


class DuplicateMessageFilter(logging.Filter):
    def __init__(self, window_seconds: float = 1.5):
        super().__init__()
        import threading as _threading
        self.window_seconds = float(window_seconds)
        self._last_seen = {}
        self._lock = _threading.Lock()

    def filter(self, record):
        import time
        msg = record.getMessage()
        key = f"{record.name}|{record.levelno}|{msg}"
        now = time.time()
        with self._lock:
            last = self._last_seen.get(key)
            self._last_seen[key] = now
        if last is None:
            return True
        return (now - last) > self.window_seconds

def get_clean_error_message(exception: Exception, max_lines: int = 3, max_chars: int = 600) -> str:
    """
    Extract clean error message from exception without stack traces or embedded SQL.
    Strips JVM stack traces, '== SQL ==' blocks, and Spark internal frames.
    """
    error_str = str(exception)

    if '== SQL ==' in error_str:
        error_str = error_str.split('== SQL ==')[0].strip()

    if 'JVM stacktrace:' in error_str:
        error_str = error_str.split('JVM stacktrace:')[0].strip()

    lines = []
    for line in error_str.split('\n'):
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith('at org.apache') or stripped.startswith('at com.databricks') or stripped.startswith('at scala.'):
            continue
        if stripped.startswith("'") and ('+- ' in stripped or '   ' in stripped):
            continue
        lines.append(stripped)

    result = ' '.join(lines[:max_lines])
    if len(result) > max_chars:
        result = result[:max_chars]
    return result


COMPACT_OUTPUT_PROMPTS = frozenset({
    "USE_CASE_QUALITY_SCORE_PROMPT",
    "USE_CASE_VALUE_SCORE_PROMPT",
    "REVIEW_USE_CASES_PROMPT",
    "FILTER_BUSINESS_TABLES_PROMPT",
    "BUSINESS_CONTEXT_WORKER_PROMPT",
    "SUBDOMAIN_DETECTOR_PROMPT",
    "DOMAIN_FINDER_PROMPT",
})

MEDIUM_OUTPUT_PROMPTS = frozenset({
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT",
})

# ==============================================================================
# TRULY ADAPTIVE PARALLELISM CALCULATOR
# ==============================================================================
# Parallelism is calculated dynamically based on ACTUAL DATA:
# - Number of items to process (tables, use cases, domains)
# - Estimated prompt/payload size
# - LLM vs non-LLM operations
# - Risk of rate limiting or timeouts
#
# BOUNDS: MIN=2, MAX=10 (enforced globally)
# METADATA QUERIES: FIXED at 5 (no LLM, but too many connections can hang)
# ==============================================================================

METADATA_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["metadata_parallelism"]

def calculate_adaptive_parallelism(
    step_name: str,
    max_parallelism: int,
    num_items: int = 0,
    total_columns: int = 0,
    avg_prompt_chars: int = 0,
    num_domains: int = 0,
    is_llm_operation: bool = True,
    logger=None,
    estimated_mb_per_worker: float = 5.0
) -> tuple:
    """
    Calculate truly adaptive parallelism based on actual data characteristics
    and available memory on the cluster.
    
    Args:
        step_name: Name of the step (for logging)
        max_parallelism: User-configured maximum parallelism
        num_items: Number of items to process (tables, use cases, domains, etc.)
        total_columns: Total columns involved (affects prompt size)
        avg_prompt_chars: Average prompt size in characters
        num_domains: Number of business domains
        is_llm_operation: Whether this step makes LLM calls
        logger: Optional logger
        estimated_mb_per_worker: Estimated MB each worker thread needs (callers
            that hold large data like schema markdown should pass higher values,
            e.g. 20 for quality scoring, 10 for SQL gen)
    
    Returns:
        Tuple of (parallelism: int, reason: str)
    """
    GLOBAL_MIN_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["global_min_parallelism"]
    GLOBAL_MAX_PARALLELISM = TECHNICAL_CONTEXT["runtime"]["global_max_parallelism"]
    CLUSTER_MEMORY_GB = TECHNICAL_CONTEXT["runtime"].get("cluster_memory_gb", 32)
    _SPARK_AND_OS_OVERHEAD_MB = 17500
    available_app_mb = max(1024, (CLUSTER_MEMORY_GB * 1024) - _SPARK_AND_OS_OVERHEAD_MB)
    memory_ceiling = max(GLOBAL_MIN_PARALLELISM, int(available_app_mb / max(1.0, estimated_mb_per_worker)))
    
    effective_max = min(max(1, max_parallelism), GLOBAL_MAX_PARALLELISM)
    effective_min = min(GLOBAL_MIN_PARALLELISM, effective_max)
    
    base = effective_max
    
    # =========================================================================
    # METADATA OPERATIONS (Fixed at 5 - no LLM but DB connections can saturate)
    # =========================================================================
    if step_name in ["schema_discovery", "table_discovery", "column_fetch"]:
        result = METADATA_PARALLELISM
        reason = f"FIXED={METADATA_PARALLELISM} for metadata queries (DB connection limit)"
        if logger:
            logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} | {reason}")
        return (result, reason)
    
    # =========================================================================
    # FILE I/O OPERATIONS (Can use higher parallelism, but cap based on items)
    # =========================================================================
    if step_name in ["notebook_generation", "artifact_writing"]:
        # Scale with number of items, but cap reasonably
        if num_items <= 5:
            result = min(base, num_items + 2)
            reason = f"few items ({num_items}), using {result} workers"
        elif num_items <= 15:
            result = min(base, 6)
            reason = f"moderate items ({num_items}), capped at 6"
        else:
            result = min(base, 8)
            reason = f"many items ({num_items}), capped at 8 for I/O stability"
        
        result = max(effective_min, min(effective_max, result))
        if logger:
            logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} | {reason}")
        return (result, reason)
    
    # =========================================================================
    # LLM OPERATIONS - Adaptive based on workload severity
    # =========================================================================
    # Each factor contributes an independent severity score (0.0 = no concern, 1.0 = max concern).
    # We combine the top-2 scores (weighted 70/30) into a single reduction applied ONCE to base.
    # This avoids cascading multiplication that previously drove any input to 0.
    
    factors = []
    severity_scores = []
    
    if num_items > 0:
        if num_items <= 10:
            severity_scores.append(0.0)
            factors.append(f"{num_items} items (small)")
        elif num_items <= 30:
            severity_scores.append(0.15)
            factors.append(f"{num_items} items (medium)")
        elif num_items <= 100:
            severity_scores.append(0.3)
            factors.append(f"{num_items} items (large)")
        else:
            severity_scores.append(0.45)
            factors.append(f"{num_items} items (very large)")
    
    if avg_prompt_chars > 0:
        if avg_prompt_chars > 100000:
            severity_scores.append(0.5)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (huge)")
        elif avg_prompt_chars > 50000:
            severity_scores.append(0.35)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (large)")
        elif avg_prompt_chars > 20000:
            severity_scores.append(0.15)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (medium)")
        else:
            severity_scores.append(0.0)
            factors.append(f"~{avg_prompt_chars//1000}K chars/prompt (small)")
    
    if num_domains > 0:
        if num_domains > 15:
            severity_scores.append(0.45)
            factors.append(f"{num_domains} domains (many)")
        elif num_domains > 8:
            severity_scores.append(0.3)
            factors.append(f"{num_domains} domains (moderate)")
        else:
            severity_scores.append(0.1)
            factors.append(f"{num_domains} domains (few)")
    
    if total_columns > 0:
        if total_columns > 2000:
            severity_scores.append(0.45)
            factors.append(f"{total_columns} cols (massive schema)")
        elif total_columns > 1000:
            severity_scores.append(0.3)
            factors.append(f"{total_columns} cols (large schema)")
        elif total_columns > 500:
            severity_scores.append(0.15)
            factors.append(f"{total_columns} cols (medium schema)")
        else:
            severity_scores.append(0.0)
            factors.append(f"{total_columns} cols (small schema)")
    
    step_severities = {
        "scoring": (0.25, "scoring is rate-limit sensitive"),
        "deduplication": (0.15, "dedup: global one-shot (thinker), fallback per-domain (worker)"),
        "use_case_generation": (0.3, "LLM-intensive, 2-pass for transactional tables"),
        "domain_clustering": (0.3, "domain detection is heavy"),
        "subdomain_detection": (0.15, "subdomain per domain"),
        "translation": (0.15, "translation LLM calls"),
    }
    
    if step_name in step_severities:
        step_sev, adj_reason = step_severities[step_name]
        severity_scores.append(step_sev)
        factors.append(adj_reason)
    
    if severity_scores:
        sorted_scores = sorted(severity_scores, reverse=True)
        if len(sorted_scores) >= 2:
            combined_severity = sorted_scores[0] * 0.7 + sorted_scores[1] * 0.3
        else:
            combined_severity = sorted_scores[0]
        reduction_factor = max(0.25, 1.0 - combined_severity)
        calculated = max(1, int(base * reduction_factor))
    else:
        calculated = base
    
    work_items = num_items if num_items > 0 else (num_domains if num_domains > 0 else calculated)
    calculated = min(calculated, max(1, work_items))
    
    result = max(effective_min, min(effective_max, calculated))
    
    if not factors and is_llm_operation:
        result = max(effective_min, min(4, base))
        factors.append("LLM operation, conservative default")
    
    pre_mem_result = result
    result = min(result, memory_ceiling)
    if result < pre_mem_result:
        factors.append(f"memory-capped={result} ({CLUSTER_MEMORY_GB}GB, ~{estimated_mb_per_worker:.0f}MB/worker, ceiling={memory_ceiling})")
    
    reason = " + ".join(factors) if factors else "default calculation"
    reason = f"calculated={result} based on: {reason}"
    
    if logger:
        logger.info(f"🔧 [{step_name.upper()}] Parallelism = {result} (from max={max_parallelism}, mem_ceil={memory_ceiling}) | {reason}")
    
    return (result, reason)


def log_adaptive_parallelism_decision(step_name: str, parallelism: int, max_parallelism: int, reason: str):
    """
    Log the adaptive parallelism decision with full context.
    """
    log_print(f"🔧 [{step_name.upper()}] Workers: {parallelism} (max={max_parallelism})")
    log_print(f"   └─ Reason: {reason}")


def create_widgets():
    """
    Creates widgets if they don't exist. Retains existing widget values.
    
    Widget Order:
    0- Operation (Discover Use Cases / Generate Use Cases)
    1- Business Name
    2- UC Metadata
    3- Inspire Database
    4- Table Election
    5- Use Cases Quality
    6- Business Domains
    7- Business Priorities
    8- Strategic Goals
    9- Generation Options (PDF Catalog / Presentation)
    10- Generation Path
    11- Documents Languages
    12- Inspire Session ID
    """
    
    log_print("Creating widgets (retaining existing values)...")
    
    widget_errors = []
    
    try:
        operation_options = [
            "Discover Use Cases",
            "Generate Use Cases",
        ]
        dbutils.widgets.dropdown("15_operation", "Discover Use Cases", operation_options, "00. Operation")
    except Exception as e:
        widget_errors.append(f"Operation: {e}")
    
    # --- 0. Business Name (REQUIRED) ---
    try:
        dbutils.widgets.text("00_business_name", "", "01. Business Name")
    except Exception as e:
        widget_errors.append(f"Business Name: {e}")
    
    # --- 1. UC Metadata (catalogs/schemas/tables OR JSON file path) ---
    try:
        dbutils.widgets.text("01_uc_metadata", "", "02. UC Metadata")
    except Exception as e:
        widget_errors.append(f"UC Metadata: {e}")
    
    # --- 2. Inspire Database (catalog.database format - where all results are stored) ---
    try:
        dbutils.widgets.text("02_inspire_database", "", "03. Inspire Database")
    except Exception as e:
        widget_errors.append(f"Inspire Database: {e}")
    

    # --- 4. Table Election (controls which tables are used for use case generation) ---
    try:
        table_election_options = [
            "Let Inspire Decides",
            "All Tables",
            "Transactional Only"
        ]
        dbutils.widgets.dropdown("04_table_election", "Let Inspire Decides", table_election_options, "05. Table Election")
    except Exception as e:
        widget_errors.append(f"Table Election: {e}")
    
    # --- 5. Use Cases Quality (controls both generation gate AND post-generation quality filtering) ---
    try:
        use_cases_quality_options = [
            "Coverage Mode (All)",
            "Balanced",
            "Strict Quality"
        ]
        dbutils.widgets.dropdown("05_use_cases_quality", "Balanced", use_cases_quality_options, "06. Use Cases Quality")
    except Exception as e:
        widget_errors.append(f"Use Cases Quality: {e}")
    
    # --- 6. Business Domains (comma-separated list of domains) ---
    try:
        dbutils.widgets.text("06_business_domains", "", "07. Business Domains")
    except Exception as e:
        widget_errors.append(f"Business Domains: {e}")
    
    # --- 7. Business Priorities (multi-select) ---
    try:
        business_priorities_options = [
            "Increase Revenue",
            "Reduce Cost",
            "Optimize Operations",
            "Mitigate Risk",
            "Empower Talent",
            "Enhance Experience",
            "Drive Innovation",
            "Achieve ESG",
            "Protect Revenue",
            "Execute Strategy"
        ]
        dbutils.widgets.multiselect("07_business_priorities", "Increase Revenue", business_priorities_options, "08. Business Priorities")
    except Exception as e:
        widget_errors.append(f"Business Priorities: {e}")
    
    # --- 8. Generation Instructions ---
    try:
        dbutils.widgets.text("08_generation_instructions", "", "09. Generation Instructions")
    except Exception as e:
        widget_errors.append(f"Generation Instructions: {e}")
    
    # --- 9. Generation Options (multiselect with generation choices) ---
    try:
        generation_options = [
            "PDF Catalog",
            "Presentation",
            "Genie Code Instructions",
        ]
        dbutils.widgets.multiselect(
            "09_generation_options", 
            "PDF Catalog,Genie Code Instructions",
            generation_options, 
            "10. Generation Options"
        )
    except Exception as e:
        widget_errors.append(f"Generation Options: {e}")
    
    # --- 10. Generation Path ---
    try:
        dbutils.widgets.text("11_generation_path", "./../demos/", "12. Generation Path")
    except Exception as e:
        widget_errors.append(f"Generation Path: {e}")
    
    # --- 12. Documents Languages (multiselect) ---
    try:
        lang_choices = [
            "English", "French", "German", "Spanish", "Hindi",
            "Chinese (Mandarin)", "Japanese", "Arabic", "Portuguese", "Russian",
            "Swedish", "Danish", "Norwegian", "Finnish",
            "Italian", "Polish", "Romanian", "Ukrainian", "Dutch", "Korean",
            "Indonesian", "Malay", "Tamil"
        ]
        dbutils.widgets.multiselect("12_documents_languages", "English", lang_choices, "13. Documents Languages")
    except Exception as e:
        widget_errors.append(f"Documents Languages: {e}")
    
    # --- 13. Inspire Session ID (optional - auto-generated if empty) ---
    try:
        dbutils.widgets.text("14_session_id", "", "14. Inspire Session ID")
    except Exception as e:
        widget_errors.append(f"Inspire Session ID: {e}")

    try:
        dbutils.widgets.text(
            "13_generate_genie_code_for", "5",
            "15. Generate Genie Code Instruction For"
        )
    except Exception as e:
        widget_errors.append(f"13_generate_genie_code_for: {e}")

    if widget_errors:
        log_print(f"⚠️ Some widgets had errors during creation:", level="WARNING")
        for err in widget_errors:
            log_print(f"   - {err}", level="WARNING")
        log_print("   Try running: dbutils.widgets.removeAll() and then run this cell again")
    else:
        log_print("✅ Widgets created successfully.")
    
    log_print("")
    log_print(">>> Fill in the widget values at the TOP of this notebook, then run main().")

# ---
# Run this cell to create widgets.
# Fill in the widget values at the TOP of the notebook.
# Then, proceed to run the 'main()' cell below.
# ---

create_widgets()

_NOTEBOOK_WIDGET_NAMES = [
    "15_operation",
    "00_business_name",
    "01_uc_metadata",
    "02_inspire_database",
    "04_table_election",
    "05_use_cases_quality",
    "06_business_domains",
    "07_business_priorities",
    "08_generation_instructions",
    "09_generation_options",
    "11_generation_path",
    "12_documents_languages",
    "13_generate_genie_code_for",
    "14_session_id",
]

# COMMAND ----------

# DBTITLE 1,Imports & Commons
# ==============================================================================
# 0. IMPORTS & CONFIGURATION
# ==============================================================================
import os
import pandas as pd
import logging
import re
import subprocess
import sys
import json
import csv
import io
import uuid
import base64
import random
import tempfile
import shutil
import datetime
import html
import warnings
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException
from collections import defaultdict
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
import threading
import gc

# --- Databricks SDK Imports for Notebook Creation ---
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import workspace

# --- Global Configuration ---
AI_MODEL_NAME = "databricks-gpt-oss-20b"

# Token-to-Character Ratios (for context limit calculations)
# English: 1 token ≈ 4 characters
# Non-English: 1 token ≈ 2 characters
TOKEN_TO_CHAR_RATIO_ENGLISH = 4
TOKEN_TO_CHAR_RATIO_NON_ENGLISH = 2

def get_model_token_limit(prompt_name: str = None) -> int:
    """
    Get the input token limit for a model based on TECHNICAL_CONTEXT.
    Uses the primary model (lowest-order) from the prompt's model_type cascade.
    """
    if prompt_name:
        model_config = get_model_config_for_prompt(prompt_name)
        if model_config:
            return model_config.get("llm_input_context_tokens_count", 200000)
    
    first_worker = get_models_by_type("worker")
    return first_worker[0].get("llm_input_context_tokens_count", 200000) if first_worker else 200000

def get_model_output_token_limit(prompt_name: str = None) -> int:
    """
    Get the OUTPUT token limit for a model based on TECHNICAL_CONTEXT.
    Uses the primary model (lowest-order) from the prompt's model_type cascade.
    """
    _safe_default = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
    if prompt_name:
        model_config = get_model_config_for_prompt(prompt_name)
        if model_config:
            return model_config.get("llm_output_context_tokens_count", _safe_default)
    
    first_worker = get_models_by_type("worker")
    return first_worker[0].get("llm_output_context_tokens_count", _safe_default) if first_worker else _safe_default

def get_max_context_chars(language: str = "English", prompt_name: str = None) -> int:
    """
    Calculate the maximum character limit based on language and model's token limit.
    Uses TECHNICAL_CONTEXT to determine model-specific token limits.
    
    Applies the same conservative limit (tokens * 3.2) used by the actual LLM call
    pre-flight check, ensuring the pre-check here matches the runtime check.
    
    Args:
        language: Target language (default: "English")
        prompt_name: Name of the prompt to lookup model for (optional)
    
    Returns:
        Maximum character limit for the given language and model
    """
    max_tokens = get_model_token_limit(prompt_name)
    
    if language.lower() == "english":
        char_limit = max_tokens * TOKEN_TO_CHAR_RATIO_ENGLISH
    else:
        char_limit = max_tokens * TOKEN_TO_CHAR_RATIO_NON_ENGLISH
    
    conservative_limit = int(max_tokens * 3.2)
    return min(char_limit, conservative_limit)

def get_context_limit_chars_for_model(model_config: dict, language: str = "English") -> int:
    """
    Calculate the effective character limit for a specific model config.
    Mirrors the pre-flight logic in _call_ai_query for consistency.
    """
    input_tokens = model_config.get("llm_input_context_tokens_count", 131000)
    char_ratio = TOKEN_TO_CHAR_RATIO_NON_ENGLISH if language.lower() != "english" else TOKEN_TO_CHAR_RATIO_ENGLISH
    max_context_chars = input_tokens * char_ratio
    conservative_limit = int(input_tokens * 3.2)
    return min(max_context_chars, conservative_limit)

def get_safe_context_limit(language: str = "English", buffer_percent: float = 0.9, prompt_name: str = None) -> int:
    """
    Calculate a safe context limit with buffer to proactively avoid LLM errors.
    Uses TECHNICAL_CONTEXT to determine model-specific token limits.
    
    Formula: model_token_limit * char_ratio * buffer_percent
    
    Args:
        language: Target language (default: "English")
        buffer_percent: Safety buffer (default: 0.9 = 10% buffer)
        prompt_name: Name of the prompt to lookup model for (optional)
    
    Returns:
        Safe character limit with buffer applied
    """
    max_chars = get_max_context_chars(language, prompt_name)
    safe_limit = int(max_chars * buffer_percent)
    return safe_limit

# Legacy constants for backward compatibility (uses default model's English ratio)
MAX_CONTEXT_TOKENS = get_model_token_limit()
MAX_CONTEXT_CHARS = min(MAX_CONTEXT_TOKENS * TOKEN_TO_CHAR_RATIO_ENGLISH, int(MAX_CONTEXT_TOKENS * 3.2))

# ==============================================================================
# IDENTIFIER NORMALIZATION UTILITIES
# ==============================================================================
# These functions handle SQL identifier parsing and quoting consistently.
# All user input should be normalized (backticks stripped) on input,
# and backticks should be added when constructing SQL queries.
# ==============================================================================

def normalize_identifier(identifier: str) -> str:
    """
    Strip backticks from an identifier.
    
    Args:
        identifier: A SQL identifier that may or may not have backticks
        
    Returns:
        The identifier without backticks
        
    Examples:
        normalize_identifier("`my-schema`") -> "my-schema"
        normalize_identifier("my_table") -> "my_table"
    """
    if identifier is None:
        return ""
    return identifier.strip().strip('`')

def quote_identifier(identifier: str) -> str:
    """
    Add backticks to an identifier for safe SQL usage.
    First normalizes (strips existing backticks) then adds fresh backticks.
    
    Args:
        identifier: A SQL identifier (normalized or not)
        
    Returns:
        The identifier wrapped in backticks
        
    Examples:
        quote_identifier("my-schema") -> "`my-schema`"
        quote_identifier("`my-schema`") -> "`my-schema`" (not double-quoted)
    """
    if identifier is None:
        return "``"
    normalized = normalize_identifier(identifier)
    return f"`{normalized}`"

def compute_result_table_name(use_case_id: str, use_case_name: str, tables_involved: str, inspire_database: str = "") -> str:
    """
    Compute the Unity Catalog result table name for a use case.
    
    Format: <inspire_database>._inspire_<id_underscored>_<snake_case_name>
    
    Rules:
        - Database comes from the user-provided inspire_database (catalog.schema format)
        - Prefix is always '_inspire_'
        - Use case ID has hyphens/spaces converted to underscores, lowercased
        - Name is lowercased and non-alphanumeric chars replaced with underscores
        - Max 255 chars total (Unity Catalog limit); table part capped at 200
    """
    import re as _re_rt
    uc_id_underscored = str(use_case_id).lower().replace('-', '_').replace(' ', '_')
    uc_snake_name = _re_rt.sub(r'[^a-z0-9]+', '_', use_case_name.lower()).strip('_')
    table_name = f"_inspire_{uc_id_underscored}_{uc_snake_name}"
    if len(table_name) > 200:
        table_name = table_name[:200]
    if inspire_database:
        return f"{inspire_database}.{table_name}"
    return f"main._inspire.{table_name}"

def parse_three_level_name(name: str) -> tuple:
    """
    Parse a three-level name (catalog.schema.table or catalog.schema.column) into parts.
    Handles names with or without backticks at any level.
    
    Args:
        name: A three-level name like "catalog.schema.table" or "`cat`.`sch`.`tbl`"
        
    Returns:
        Tuple of (part1, part2, part3) with backticks stripped from each part,
        or (None, None, None) if parsing fails
        
    Examples:
        parse_three_level_name("cat.schema.table") -> ("cat", "schema", "table")
        parse_three_level_name("`cat`.`my-schema`.`table`") -> ("cat", "my-schema", "table")
        parse_three_level_name("invalid") -> (None, None, None)
    """
    if not name:
        return (None, None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.')
    
    if len(parts) == 3:
        return (parts[0].strip(), parts[1].strip(), parts[2].strip())
    return (None, None, None)

def parse_two_level_name(name: str) -> tuple:
    """
    Parse a two-level name (catalog.schema) into parts.
    Handles names with or without backticks.
    
    Args:
        name: A two-level name like "catalog.schema" or "`cat`.`my-schema`"
        
    Returns:
        Tuple of (part1, part2) with backticks stripped from each part,
        or (None, None) if parsing fails
        
    Examples:
        parse_two_level_name("cat.schema") -> ("cat", "schema")
        parse_two_level_name("`cat`.`my-schema`") -> ("cat", "my-schema")
    """
    if not name:
        return (None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.', 1)
    
    if len(parts) == 2:
        return (parts[0].strip(), parts[1].strip())
    return (None, None)

def parse_four_level_name(name: str) -> tuple:
    """
    Parse a four-level name (catalog.schema.table.column) into parts.
    Handles names with or without backticks at any level.
    
    Args:
        name: A four-level name like "catalog.schema.table.column"
        
    Returns:
        Tuple of (catalog, schema, table, column) with backticks stripped,
        or (None, None, None, None) if parsing fails
    """
    if not name:
        return (None, None, None, None)
    
    clean_name = name.replace('`', '')
    parts = clean_name.split('.')
    
    if len(parts) == 4:
        return (parts[0].strip(), parts[1].strip(), parts[2].strip(), parts[3].strip())
    return (None, None, None, None)

def build_fqn(catalog: str, schema: str, table: str = None) -> str:
    """
    Build a fully qualified name with proper backtick quoting.
    
    Args:
        catalog: Catalog name (will be normalized and quoted)
        schema: Schema name (will be normalized and quoted)
        table: Optional table name (will be normalized and quoted if provided)
        
    Returns:
        Properly quoted FQN like `catalog`.`schema` or `catalog`.`schema`.`table`
        
    Examples:
        build_fqn("cat", "my-schema") -> "`cat`.`my-schema`"
        build_fqn("cat", "schema", "table") -> "`cat`.`schema`.`table`"
    """
    cat_quoted = quote_identifier(catalog)
    schema_quoted = quote_identifier(schema)
    
    if table:
        table_quoted = quote_identifier(table)
        return f"{cat_quoted}.{schema_quoted}.{table_quoted}"
    return f"{cat_quoted}.{schema_quoted}"

# --- LLM Model Resolution ---
# Model resolution is now handled dynamically via model_type + model_size cascade.
# Each prompt bucket is resolved at runtime to an ordered list of enabled models.
# On failure, the next enabled model in the same bucket is tried automatically.
# See: get_model_cascade_for_prompt(), _call_ai_query()

# DBTITLE 1,Prompts
# --- 1. Main Prompt Templates Dictionary ---
PROMPT_TEMPLATES = {}


def smoke_render_all_prompts(templates=None, dummy_values=None, verbose=False) -> dict:
    """Phase A1 helper: best-effort render every PROMPT_TEMPLATES entry with
    dummy values for every `{placeholder}` detected. Returns a per-template
    report {name: {"ok": bool, "error": str, "placeholders": [...]}.

    Use this in calibration mode or test suites to catch stray placeholders
    after prompt edits. No network calls, no LLM invocation.
    """
    import re as _sr
    t = templates if templates is not None else PROMPT_TEMPLATES
    dv = dict(dummy_values or {})
    report = {}
    for name, body in sorted(t.items()):
        if not isinstance(body, str):
            report[name] = {"ok": False, "error": f"non-string body type={type(body).__name__}", "placeholders": []}
            continue
        # Extract simple {name} placeholders (no nested braces)
        placeholders = sorted(set(_sr.findall(r"\{([A-Za-z_][A-Za-z0-9_]*)\}", body)))
        fill = {k: dv.get(k, f"<{k}>") for k in placeholders}
        try:
            # Use format_map so missing keys fall through to __missing__ (default dict)
            class _Defaulting(dict):
                def __missing__(self, key):
                    return f"<missing:{key}>"
            rendered = body.format_map(_Defaulting(fill))
            report[name] = {"ok": True, "error": None, "placeholders": placeholders, "rendered_len": len(rendered)}
        except (KeyError, IndexError, ValueError) as _re:
            report[name] = {"ok": False, "error": f"{type(_re).__name__}: {_re}", "placeholders": placeholders}
        except Exception as _re:
            report[name] = {"ok": False, "error": f"{type(_re).__name__}: {_re}", "placeholders": placeholders}
    if verbose:
        for n, r in report.items():
            status = "OK " if r["ok"] else "FAIL"
            print(f"[A1] {status} {n} (placeholders={len(r['placeholders'])})")
            if not r["ok"]:
                print(f"      -> {r['error']}")
    return report

QUALITY_GATE_RULES = """
**D1 - CAUSAL SIGNAL STRENGTH** (Critical):
Does the data contain fields that DIRECTLY CAUSE or INDICATE the outcome being predicted/analyzed?
If you cannot trace a logical cause-effect chain from available columns to the outcome, the use case is INVALID.
No causal signals = No use case.

**D2 - CAUSE-EFFECT VALIDITY** (Critical):
Does the high level design use variables with a LOGICAL, PROVEN, INDUSTRY-RECOGNIZED cause-effect relationship with the outcome?
Correlation is NOT causation. If the relationship is speculative, the use case is INVALID.

**D3 - DATA GRANULARITY**:
Does the data exist at the RIGHT LEVEL OF DETAIL for the analysis?
If the analysis requires per-customer data but only per-store aggregates exist, the use case is INVALID.

**D4 - CRITICAL DIMENSIONS PRESENT**:
Are ALL critical dimensions (columns) required for the analysis present in the schema?
If even ONE critical column is missing, the use case is INVALID. Do NOT assume derived data can compensate.

**D5 - LOGICAL POSSIBILITY**:
Is the high level design logically possible with the given data types and structure?
If operations cannot be performed on available data, the use case is INVALID.

**D6 - METRIC VALIDITY**:
Can the claimed business metrics ACTUALLY be calculated from available fields?
If the key metric requires fabrication or heavy approximation, the use case is INVALID.

**D7 - HIGH LEVEL DESIGN MATCH**:
Does the high level design match what the data can actually support?
If the design assumes columns, tables, or relationships that do not exist in the schema, the use case is INVALID.

**D8 - SEMANTIC UNIQUENESS** (Critical — ZERO TOLERANCE):
Does this use case address a FUNDAMENTALLY DIFFERENT business question than all others? Apply ALL 5 layers of duplicate detection — a match on ANY layer means DUPLICATE:

**LAYER 1 — VERB NORMALIZATION + SUFFIX STRIP**:
Normalize verb (Forecast/Predict/Anticipate/Envision = PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover/Expose = DETECT; Classify/Segment/Categorize/Tier/Rank = CLASSIFY; Simulate/Model/Project = SIMULATE; Optimize/Maximize/Minimize = OPTIMIZE; Assess/Evaluate/Measure/Gauge/Quantify = ASSESS). Strip "with [...]" activation suffix. If remaining core phrase matches another use case → DUPLICATE.

**LAYER 2 — NOUN SYNONYM DETECTION**:
Treat these nouns as IDENTICAL: Revenue=Income=Sales=Earnings; Cost=Expense=Spend=Expenditure; Churn=Attrition=Turnover=Defection; Leakage=Loss=Erosion=Slippage; Demand=Consumption=Usage=Volume; Risk=Exposure=Vulnerability=Threat; Delay=Latency=Bottleneck=Slowdown; Fraud=Anomaly=Irregularity; Customer=Client=Account=Subscriber; Employee=Staff=Worker=Personnel; Supplier=Vendor=Provider; Inventory=Stock=Supply.
If after replacing synonyms the core phrases match → DUPLICATE.

**LAYER 3 — ENTITY-METRIC PAIR OVERLAP**:
Extract the BUSINESS ENTITY (what is being analyzed: customers, orders, suppliers, products, etc.) and the BUSINESS METRIC (what is being measured: revenue, churn, cost, risk, etc.). If two use cases operate on the SAME entity + SAME metric → DUPLICATE regardless of wording.
Example: "Forecast Customer Churn Rate" and "Predict Client Attrition Probability" → same entity (customer), same metric (churn) → DUPLICATE.

**LAYER 4 — TECHNIQUE SWAP (NARROW — SAME QUESTION ONLY)**:
Two use cases are duplicates ONLY if they ask the SAME business question and merely swap the technique. Example: "Forecast Customer Churn" and "Predict Customer Churn" = DUPLICATE (same question: will the customer leave?). BUT "Forecast Revenue" and "Detect Revenue Anomalies" are NOT duplicates — the first asks "what will revenue be?" and the second asks "is revenue behaving abnormally?". Different QUESTIONS on the same entity/metric are DISTINCT use cases — keep BOTH. Do NOT collapse technique diversity.

**LAYER 5 — TABLE OVERLAP + SIMILAR INTENT**:
If two use cases use the SAME primary tables AND address a SIMILAR business question (even with different wording) → DUPLICATE. Different tables = potentially different use case. Same tables + same intent = definitely DUPLICATE.

**D9 - ANALYTICAL DEPTH** (Critical — RUTHLESS STANDARD):
Apply the "STRIP TEST": mentally remove all advanced analytical techniques (forecasting, classification, anomaly detection, simulation, etc.) from the High Level Design. What remains? If what remains is a trivial operation, the advanced techniques are DECORATION, not substance.

TRIVIAL (D9 ≤ 1.5 — AUTOMATIC REJECTION):
- Date arithmetic: simple date comparisons → trivial calendar logic
- Threshold checks: simple value filters → trivial filter
- Basic aggregation: counting or summing by one dimension → any junior analyst can do this
- Single-column statistical measure: basic deviation from mean → basic statistics
- Ratio computation: dividing one column by another → arithmetic, not intelligence
- Sorting/ranking: ordering by a metric → trivial

WEAK (D9 = 2.0-2.5 — NOT ACCEPTABLE FOR EXTREME QUALITY):
- Two-column correlation without cross-table context
- Simple moving averages or period-over-period comparisons
- Grouping by one dimension with a threshold overlay

GENUINE ANALYTICAL DEPTH (D9 ≥ 3.5 — MINIMUM FOR ACCEPTANCE):
- Multi-dimensional pattern detection across 3+ variables
- Cross-table correlation requiring JOINs on 2+ tables with derived features
- Time-series forecasting with external factor integration
- Multi-step classification with contextual enrichment
- Simulation or what-if analysis with scenario comparison
- Network/graph analysis of entity relationships
- Trained ML models (XGBoost, Random Forest, LightGBM, K-Means, etc.) applied to business problems with feature engineering
- Survival analysis or time-to-event modeling (e.g., customer lifetime, equipment failure)
- Optimization models (resource allocation, scheduling, pricing) with constraint satisfaction
- Recommendation systems using collaborative filtering or content-based approaches
- Ensemble methods combining multiple models or analytical approaches for robust scoring

**D10 - ACTIVATION QUALITY** (Critical — ZERO TOLERANCE FOR TECHNICAL METHOD SUFFIXES):
Apply the "DELIVERABLE TEST": the "with [...]" suffix must describe a business DELIVERABLE — the actionable output the user receives (recommendation, action plan, priority queue, strategy, intervention, escalation brief). It must NEVER describe a technical method, data source, statistical technique, or analytical approach.

BANNED CATEGORIES (D10 = 1.0 — AUTOMATIC REJECTION):
- Technical method suffixes: "with Z-Score Deviation", "with Correlation Analysis", "with Rolling STDDEV", "with Regression Slope"
- Data source suffixes: "with Payment History Cross-Reference", "with Invoice-to-Payment Date Gap Analysis", "with RFM Scoring and Cohort Aging"
- Technique suffixes: "with Seasonal Decomposition", "with Anomaly Profiling", "with Monte Carlo Simulation", "with Trend Analysis"
- Buzzword suffixes: "with Intelligence", "with Insights", "with Analytics", "with Context"
- Circular suffixes: where the suffix merely restates the verb/name (e.g., "Segment Customers with Customer Segmentation")
- Domain jargon in action part: Any industry-specific technical acronym (e.g., RASK, PNR, AWB, MEL, CDL, ULD, GDS, MTOW for airlines; SKU, BOM, WIP, OEE for manufacturing; LTV, CAC, MRR, ARR for SaaS; NPA, NIM, CAR for banking) or statistical term (Z-score, variance, coefficient, kurtosis) appearing ANYWHERE in the name

ACCEPTABLE (D10 ≥ 3.5 — MINIMUM FOR EXTREME QUALITY):
- Describes a specific business deliverable: "with Pricing Recalibration Recommendations", "with Pre-Invoice Correction Actions"
- Describes an actionable intervention: "with Prioritized Outreach Queue", "with Audit Escalation Brief", "with Collection Escalation Actions"
- Describes a decision-support artifact: "with Budget Reallocation Strategy", "with Capacity Planning Recommendations", "with Retention Intervention Playbook"
- The activation suffix answers "What does the business user DO with this?" not "How was this analyzed?"

**D11 - DOMAIN & TECHNIQUE BALANCE**:
Is the portfolio diverse across business domains and analytics techniques?
- No single domain should exceed 25% of all use cases
- No single analytics technique should exceed 30% of all use cases
- If generating into an over-represented area, produce use cases in under-represented areas instead.

**D12 - BUSINESS RELEVANCE & OPPORTUNITY** (Critical — put yourself in the business stakeholder's seat):
Answer ALL of these for THIS use case. If you answer "no" to ANY, the use case FAILS this gate:
1. Is this use case RELEVANT to how the business actually operates day-to-day? (Not just analytically possible — actually relevant.)
2. Does it deliver TANGIBLE VALUE to the business — revenue protection, cost reduction, risk mitigation, or opportunity capture?
3. Does SOMEONE INSIDE THE BUSINESS have the exact issue this solves? Can you name the team/role/function that would consume this output and change their decisions?
4. Does this represent either a CURRENT PAIN the business wants solved, OR a FUTURE OPPORTUNITY the business could expand into?

If you cannot ground this use case in a specific business function or named stakeholder, it is an analytical exercise, not a fundable use case — FAIL.

**D13 - SPONSOR TEST** (Critical — put yourself in the Sponsor's/CFO's seat):
Answer these two questions from the SPONSOR's perspective:
1. ARE YOU WILLING TO FUND THIS USE CASE with a real budget line? (yes / no — if no, FAIL)
2. CAN YOU PITCH THIS TO THE CEO and get approval without caveats or hand-waving? (yes / no — if no, FAIL)

The sponsor is the person who signs the check. If you cannot personally stand behind funding this and defending it to the CEO, do NOT generate it.

**D14 - ENGINEERING TEST** (Critical — put yourself in the Production Engineer's seat):
Answer these three questions from the ENGINEERING / PRODUCTION team's perspective:
1. CAN YOU PUT THIS TO PRODUCTION with existing Databricks capabilities (ai_query, Delta, MLflow, Spark) and the available columns — no fabricated external services? (yes / no — if no, FAIL)
2. CAN YOU SUPPORT AND MAINTAIN THIS long-term once deployed (monitor drift, refresh, retrain) with the existing team skills? (yes / no — if no, FAIL)
3. CAN YOU FIX ISSUES with this use case when they arise (bad predictions, schema drift, edge cases) without external dependencies or bespoke infrastructure? (yes / no — if no, FAIL)

Use cases that cannot be operated by the production team are shelfware. FAIL them here.

**D15 - DECISION CADENCE MATCH** (Critical):
Does the data refresh cadence match the business decision cadence?
- Weekly operational decisions REQUIRE at-least-weekly data freshness.
- Monthly strategic decisions can use monthly refresh.
- If decision cadence is FASTER than what the data can deliver, the UC is unactionable in practice -> FAIL.

**D16 - THE MONDAY TEST** (Critical):
If a stakeholder reads this UC's output on Monday morning, does their Monday look DIFFERENT from last Monday? Does it change a specific action they would take?
- If the output is "interesting to know" but changes nothing in the user's actual workflow -> decorative analytics -> FAIL.
- Must name at least one concrete action the output triggers.

**D17 - EXPLAINABILITY & AUDIT DEFENSIBILITY**:
If a regulator, auditor, CFO, or unhappy customer asked "how did you reach this conclusion?" -> can the output + methodology be explained in 5 minutes without invoking proprietary magic?
- For high-stakes decisions (compliance, pricing, risk, customer-facing), black-box models without local explainability -> FAIL.
- For operational/internal decisions, statistical-grounded techniques are acceptable.

**D18 - 18-MONTH LONGEVITY**:
In 18 months, will this UC still be running, or will it be abandoned?
- If the UC encodes a SHORT-TERM TACTICAL pattern (e.g., "this quarter's specific promotion mix") -> FAIL, it's a one-off not a use case.
- If it provides PERSISTENT operational advantage (always-on anomaly detection, recurring forecast, ongoing optimization) -> PASS.

**D19 - ATTRIBUTABLE IMPACT**:
Can the business team PROVE, 6 months after deployment, that this UC changed a measurable dollar amount in revenue / cost / risk?
- The output must enable a counterfactual or A/B-testable outcome.
- If impact is diffuse ("awareness", "insight", "visibility") without a traceable decision chain -> FAIL.

**COMBINED GATE RESOLUTION**:
- If D12 fails on ANY of the 4 sub-questions -> quality_score CAPPED at 2.0 (Low).
- If D13 fails on EITHER sub-question -> quality_score CAPPED at 2.0 (Low).
- If D14 fails on ANY of the 3 sub-questions -> quality_score CAPPED at 2.0 (Low).
- If D15-D19 FAIL -> quality_score CAPPED at 2.0 (Low).
- These are business-value vetoes — they override any technical quality score.
"""

QUALITY_GATE_GOOD = f"""
### DATA QUALITY GATE — MAXIMUM DISCOVERY MODE

**🚨 MAXIMIZE USE CASE DISCOVERY — GENERATE ALL POSSIBLE USE CASES 🚨**:
Your PRIMARY goal is to discover and generate the MAXIMUM number of valid, non-duplicate use cases from the provided schema. Quality will be assessed and filtered in a SEPARATE downstream scoring step — your job is to CAST A WIDE NET and generate EVERY possible use case that can be supported by the data.

**DO NOT self-censor or skip use cases because they might not be "high enough quality".**
**DO NOT limit yourself to only "exceptional" use cases.**
**Generate ALL use cases that are implementable from the schema — from simple analytical insights to complex ML pipelines.**

The ONLY reasons to skip a use case are:
1. It is a DUPLICATE of another use case you already generated (D8 check below)
2. It references columns/tables that DO NOT EXIST in the schema (hallucination)
3. It is purely trivial (e.g., just counting rows or selecting a column with no business insight)

**SELF-CHECK PROCEDURE**: Before writing each CSV row, verify D8 (no duplicates) and schema validity. If the use case references real columns and produces a genuine business insight, INCLUDE IT.

**D8 (5-LAYER DUPLICATE DETECTION)** — the ONLY hard gate during generation:
- Layer 1: Normalize verb + strip suffix → compare core phrase
- Layer 2: Replace noun synonyms (Revenue=Income=Sales, Churn=Attrition, etc.) → compare
- Layer 3: Extract entity + metric → same entity + same metric = DUPLICATE
- Layer 4: Technique swap on SAME QUESTION only = DUPLICATE (different questions on same entity are NOT duplicates)
- Layer 5: Same tables + similar intent = DUPLICATE
Match on ANY layer → DO NOT GENERATE (it's a duplicate).

**QUALITY BEST PRACTICES** (aim for these but DO NOT skip use cases that fall short):
- **D9 (ANALYTICAL DEPTH)**: Prefer multi-dimensional analysis, ML models, cross-table feature engineering. Simple aggregations are acceptable — they will be scored later.
- **D10 (DELIVERABLE TEST)**: Prefer "with [...]" suffixes that describe business DELIVERABLES (recommendation, action plan, queue, strategy, intervention). Technical method suffixes are discouraged but not grounds for skipping.
- **D11 (BALANCE)**: Spread use cases across domains and techniques. If one domain has many use cases already, look for opportunities in other domains too — but DO NOT stop generating valid use cases just because a domain has many.
{QUALITY_GATE_RULES}
**FINAL CHECK** (for each use case):
1. "Can an engineer implement this using ONLY the columns in the schema?" If NO → DO NOT GENERATE IT.
2. "Is this substantially different from every other use case I have generated so far (D8 check)?" If NO → DO NOT GENERATE IT.
3. "Does this produce a genuine business insight or actionable output?" If YES → GENERATE IT.
4. "Have I chosen an appropriate analytical approach for this problem?" If unsure, default to the technique that best fits the data.
"""

QUALITY_GATE_HIGH = f"""
### 🚨 DATA QUALITY GATE — BALANCED MODE 🚨

**GOAL: Generate a comprehensive set of HIGH-QUALITY use cases.** Balance breadth of discovery with analytical rigor. Generate all use cases that meet a reasonable quality bar — do not self-censor excessively, but do apply quality checks before including each use case.

**SELF-CHECK PROCEDURE**: Before writing each CSV row, verify all checks below. If a use case fails D8 (duplicate) or D9 (too trivial) → SKIP it. For D10 and D11, prefer compliance but do not skip otherwise strong use cases.

**🚨 HARD GATES (skip use case if any fails) 🚨**:

**D8 (5-LAYER DUPLICATE DETECTION)**:
- Layer 1: Normalize verb + strip suffix → compare core phrase
- Layer 2: Replace noun synonyms (Revenue=Income=Sales, Churn=Attrition, etc.) → compare
- Layer 3: Extract entity + metric → same entity + same metric = DUPLICATE
- Layer 4: Technique swap on SAME QUESTION only = DUPLICATE (different questions on same entity are NOT duplicates)
- Layer 5: Same tables + similar intent = DUPLICATE
Match on ANY layer → DO NOT GENERATE (it's a duplicate).

**D9 (ANALYTICAL DEPTH — STRIP TEST)**: Remove all AI function calls mentally. If what remains is ONLY a WHERE clause, single threshold check, basic GROUP BY count, date arithmetic, or single-column ratio → DO NOT GENERATE. The use case must require at least multi-column analysis, cross-table JOINs with derived insights, or genuine statistical/predictive reasoning.

**QUALITY GUIDELINES (apply but do not over-filter)**:
- **D10 (DELIVERABLE TEST)**: The "with [...]" suffix should describe a business DELIVERABLE (recommendation, action plan, queue, strategy, intervention), NOT a technical method. Rewrite weak activations rather than skipping the use case.
- **D11 (BALANCE)**: Spread use cases across domains and techniques. If one domain exceeds ~25% or one technique exceeds ~30%, prioritize under-represented areas. But do not skip valid use cases solely for balance — generate them and let scoring handle prioritization.
{QUALITY_GATE_RULES}
**FINAL CHECK** (for each use case):
1. "Can an engineer implement this using ONLY the columns in the schema, with logical cause-effect reasoning, producing a real calculable metric?" If NO → DO NOT GENERATE IT.
2. "Is this substantially different from every other use case I have generated so far (D8 check)?" If NO → DO NOT GENERATE IT.
3. "Does this require genuine analytical work beyond basic aggregation (D9 check)?" If NO → DO NOT GENERATE IT.
4. "Does the 'with [activation]' part describe a business deliverable?" If not, try to rewrite. If impossible, still include if the core analysis is strong.
5. "Have I chosen an appropriate analytical approach for this problem?" If unsure, default to the technique that best fits the data.
"""

QUALITY_GATE_VERY_HIGH = f"""
### 🚨🚨🚨 MANDATORY DATA QUALITY GATE (Apply BEFORE generating EACH use case) 🚨🚨🚨

**🚨 QUALITY IS ABSOLUTE — QUANTITY IS IRRELEVANT 🚨**: Generate ONLY use cases that pass ALL 11 quality dimensions AND all 5 duplicate detection layers.
A use case that fails ANY dimension or ANY duplicate layer MUST NOT be generated. It is VASTLY better to produce 10 exceptional use cases than 100 mediocre ones. Your output will be scored by a ruthless quality assessor that will reject anything below "Very High" quality. Every mediocre use case you generate WASTES resources and DAMAGES the portfolio.

**SELF-CHECK PROCEDURE**: Before writing each CSV row, mentally verify all 11 dimensions. If ANY fails, SKIP that use case entirely.

**🚨 CRITICAL CHECKS D8-D11 (EXTREME — ZERO TOLERANCE) 🚨**:
- **D8 (5-LAYER DUPLICATE DETECTION)**: Apply ALL 5 layers before generating EACH use case:
  Layer 1: Normalize verb + strip suffix → compare core phrase
  Layer 2: Replace noun synonyms (Revenue=Income=Sales, Churn=Attrition, etc.) → compare
  Layer 3: Extract entity + metric → same entity + same metric = DUPLICATE
  Layer 4: Technique swap on SAME QUESTION only = DUPLICATE (different questions on same entity are NOT duplicates)
  Layer 5: Same tables + similar intent = DUPLICATE
  Match on ANY layer → DO NOT GENERATE.
- **D9 (STRIP TEST)**: Remove all AI function calls mentally. If what remains is just a WHERE clause, threshold, GROUP BY, date check, ratio, or single-column Z-score → DO NOT GENERATE. It must REQUIRE multi-dimensional analysis, trained ML models, cross-table feature engineering, or genuine statistical/predictive reasoning.
- **D10 (DELIVERABLE TEST)**: Your "with [...]" suffix must describe a business DELIVERABLE (recommendation, action plan, queue, strategy, intervention), NOT a technical method, data source, or analytical technique. If the suffix describes HOW the analysis works instead of WHAT the user gets → REWRITE or DO NOT GENERATE.
- **D11**: Count use cases per domain and per technique. If one domain >20% or one technique >25%, STOP and move to under-represented areas.
{QUALITY_GATE_RULES}
**FINAL GATE**: For each use case ask ALL of these:
1. "Can an engineer implement this using ONLY the columns in the schema, with PROVEN cause-effect logic, producing a REAL calculable metric — whether via SQL, Python ML, statistical modeling, or any combination?" If NO → DO NOT GENERATE IT.
2. "Is this substantially different from every other use case I have generated so far?" If NO → DO NOT GENERATE IT.
3. "Would a junior analyst replicate this easily with basic filtering, grouping, or simple aggregation alone — without needing multi-step analysis, trained models, or cross-table feature engineering?" If YES (too simple) → DO NOT GENERATE IT.
4. "Does the 'with [activation]' part describe a business DELIVERABLE (recommendation, action plan, queue, strategy) rather than a technical method?" If it describes HOW the analysis works → REWRITE the activation or DO NOT GENERATE IT.
5. "Does the action part (before 'with') contain ANY technical jargon, domain acronyms, or statistical terms?" If YES → REWRITE using plain business language or DO NOT GENERATE IT.
6. "Have I chosen the BEST analytical approach for this problem — SQL statistical analysis, trained ML model, AI function, or hybrid — rather than defaulting to the simplest approach?" If you defaulted to SQL when an ML model would be more accurate → RECONSIDER the approach.
"""

QUALITY_GATE_GENERATION_BLOCKS = {
    "Coverage Mode (All)": QUALITY_GATE_GOOD,
    "Balanced": QUALITY_GATE_HIGH,
    "Strict Quality": QUALITY_GATE_VERY_HIGH,
    "Good Quality": QUALITY_GATE_GOOD,
    "High Quality": QUALITY_GATE_HIGH,
    "Very High Quality": QUALITY_GATE_VERY_HIGH,
}

USE_CASES_QUALITY_ALIASES = {
    "Good Quality": "Coverage Mode (All)",
    "High Quality": "Balanced",
    "Very High Quality": "Strict Quality",
}

USE_CASES_QUALITY_VALID = ("Coverage Mode (All)", "Balanced", "Strict Quality")

QUALITY_GATE_SCORING_BLOCK = f"""
## QUALITY SCORING DIMENSIONS

For EACH use case, evaluate the following 11 dimensions. These are the SAME quality gates applied during generation — your job is to VERIFY they were followed correctly.

**🚨 D8-D11 SCORING PROCEDURE (MANDATORY — follow step by step) 🚨**

**D8 — SEMANTIC UNIQUENESS SCORING PROCEDURE (5-LAYER DETECTION — apply ALL layers):**

**LAYER 1 — VERB NORMALIZATION + SUFFIX STRIP**:
1. Normalize the leading verb using the EXPANDED synonym map: Forecast/Predict/Anticipate/Envision → PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover/Expose → DETECT; Classify/Segment/Categorize/Tier/Rank → CLASSIFY; Simulate/Model/Project → SIMULATE; Optimize/Maximize/Minimize → OPTIMIZE; Assess/Evaluate/Measure/Gauge/Quantify → ASSESS
2. Strip "with [...]" activation suffix entirely
3. Compare the remaining core phrase against ALL other use cases in this batch
4. Match on LAYER 1 → D8 = 1.0 for the later use case

**LAYER 2 — NOUN SYNONYM DETECTION**:
After LAYER 1, also replace noun synonyms: Revenue=Income=Sales=Earnings; Cost=Expense=Spend; Churn=Attrition=Turnover=Defection; Leakage=Loss=Erosion=Slippage; Demand=Consumption=Usage=Volume; Risk=Exposure=Vulnerability; Delay=Latency=Bottleneck; Customer=Client=Account; Employee=Staff=Worker; Supplier=Vendor=Provider; Inventory=Stock=Supply.
Match after noun replacement → D8 = 1.0

**LAYER 3 — ENTITY-METRIC PAIR**:
Extract the business ENTITY (what?) and METRIC (measuring what?). If two use cases share the same entity + same metric → D8 = 1.0.
Example: "Forecast Customer Churn Rate" and "Predict Client Attrition Probability" → entity=Customer, metric=Churn → D8 = 1.0 for the later one.

**LAYER 4 — TECHNIQUE SWAP (NARROW — SAME QUESTION ONLY)**:
Two use cases are duplicates ONLY if they ask the SAME business question and merely swap the technique. "Forecast Customer Churn" and "Predict Customer Churn" → D8 = 1.0 for the later one. BUT "Forecast Revenue" and "Detect Revenue Anomalies" are NOT duplicates (different questions) → score each independently.

**LAYER 5 — TABLE + INTENT OVERLAP**:
Same primary tables AND similar business question → D8 = 1.0.

**SCORING**: D8 = 5.0 means ZERO overlap on ALL 5 layers. D8 = 3.0 means minor similarity detected but different business question. D8 ≤ 2.0 = DUPLICATE on any layer → triggers VETO.

**D9 — ANALYTICAL DEPTH SCORING PROCEDURE (apply the "STRIP TEST" AND the "TECHNIQUE MATCH TEST"):**
Mentally REMOVE all AI function calls from the High Level Design. What analytical substance remains? Score based on what's LEFT — considering SQL operations, ML model training, statistical modeling, and hybrid approaches:

D9 = 1.0 (TRIVIAL — VETO): Date arithmetic, threshold checks, basic aggregation, simple filtering, sorting, ratio computation. Could a junior analyst replicate this with basic tools alone?
D9 = 1.5 (TRIVIAL — VETO): Single-column Z-score, one-dimensional anomaly detection, basic period-over-period comparison.
D9 = 2.0 (WEAK — VETO): Two-column correlation without cross-table context, simple moving average, one-dimensional grouping + threshold.
D9 = 2.0 (TECHNIQUE MISMATCH — VETO): When the Analytics Technique is any of [Anomaly Detection, Clustering, Predictive Modeling, Classification, Forecasting, Regression Analysis, Segmentation, Survival Analysis, Recommendation, Network Analysis, Optimization, Simulation] but the High Level Design describes ONLY SQL operations (z-scores, window functions, CASE WHEN, GROUP BY, NTILE, simple thresholds) without any ML model training, statistical modeling, or technique-appropriate algorithm. A technique mismatch means the use case CLAIMS to use ML-grade analytics but actually proposes SQL-grade implementation — this is analytically dishonest regardless of SQL complexity.
D9 = 3.0 (MODERATE): Multiple statistical operations on 2-3 variables, but no genuine multi-dimensional reasoning or ML modeling. NOT acceptable for Extreme Quality. Acceptable ONLY for techniques where SQL is genuinely appropriate (Pareto Analysis, basic Cohort Analysis, Funnel Analysis).
D9 = 3.5 (ACCEPTABLE MINIMUM): Multi-dimensional analysis across 3+ variables with derived features; OR basic ML model training (single algorithm) on well-prepared features.
D9 = 4.0-4.5 (STRONG): Cross-table feature engineering with trained ML model (XGBoost, Random Forest, LightGBM, Isolation Forest, K-Means, DBSCAN); OR multi-step predictive pipeline with model evaluation; OR clustering/segmentation with profiling and actionable insights; OR survival analysis with time-to-event modeling; OR simulation with Monte Carlo or scenario modeling.
D9 = 5.0 (EXCEPTIONAL): Novel multi-technique pipeline: feature engineering → multiple model comparison → ensemble scoring → optimization/simulation → ai_query reasoning. OR: graph/network analysis combined with ML. OR: sophisticated time-series with multiple external factors and scenario modeling.

**DEFAULT IS 3.0**. The use case must EARN its way above 3.0 with concrete evidence of analytical complexity. ML model training, when appropriate for the problem, is a STRONG indicator of depth. SQL-only implementations of ML-designated techniques (Anomaly Detection, Clustering, Classification, Predictive Modeling, Forecasting, etc.) are CAPPED at D9 = 2.0 regardless of SQL complexity.

**D10 — ACTIVATION QUALITY SCORING PROCEDURE (apply the "DELIVERABLE TEST"):**
The "with [...]" suffix must describe a business DELIVERABLE — the actionable output the user receives. Score based on whether the activation describes WHAT THE USER GETS (high score) vs HOW THE ANALYSIS WORKS (low score).

D10 = 1.0 (TECHNICAL METHOD — VETO): Activation describes an analytical technique or statistical method — "with Z-Score Deviation", "with Correlation Analysis", "with Rolling STDDEV", "with Regression Slope", "with Seasonal Decomposition", "with RFM Scoring", "with Cross-Reference"
D10 = 1.0 (GENERIC FILLER — VETO): "with Intelligence", "with Insights", "with Analytics", "with Context", "with Pattern Recognition", "with Anomaly Profiling", "with Trend Analysis"
D10 = 1.0 (DATA SOURCE — VETO): Activation describes where data comes from, not what user gets — "with Payment History and Contract Term Signals", "with Invoice Date Gap Analysis", "with Load Factor Variance"
D10 = 2.0 (CIRCULAR): The suffix restates the analysis verb or name — "Segment Customers with Customer Segmentation", "Detect Anomalies with Anomaly Detection"
D10 = 3.0 (VAGUE DELIVERABLE): Mentions an output but too generic — "with Recommendations", "with Action Plan", "with Strategy"
D10 = 4.0-5.0 (SPECIFIC DELIVERABLE): Names a concrete, role-appropriate business deliverable — "with Pricing Recalibration Recommendations", "with Prioritized Outreach Queue", "with Pre-Invoice Correction Actions", "with Audit Escalation Brief", "with Collection Escalation Actions", "with Retention Intervention Playbook", "with Capacity Planning Recommendations"

Also check the ACTION PART (before "with"):
D10 = 1.0 (JARGON — VETO): Action part contains domain-specific technical acronyms (e.g., RASK, PNR, AWB, MEL, CDL, ULD, GDS, MTOW for airlines; SKU, BOM, WIP, OEE for manufacturing; LTV, CAC, MRR, ARR for SaaS; NPA, NIM, CAR for banking; RFM, CLTV for marketing) or statistical terms (Z-score, variance, coefficient, kurtosis, regression, STDDEV, deviation, correlation, decomposition, cross-reference).
A C-suite executive must be able to read the FULL name in 3 seconds and understand both the problem AND the deliverable.

**DEFAULT IS 3.0**. The activation must EARN its way above 3.0 by describing a specific, actionable business deliverable.

**D11 — DOMAIN BALANCE SCORING:**
Count how many use cases in this batch share the same Business Domain and the same Analytics Technique. If this batch is heavily concentrated in one area, reduce scores for excess use cases.
{QUALITY_GATE_RULES}
"""

HONESTY_CHECK_CSV = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - INTEGRATED IN CSV) 🎯

You MUST include honesty self-assessment as TWO ADDITIONAL COLUMNS at the END of your CSV:
- **"honesty_score"**: Your honest score 0-100 for the ENTIRE output quality
- **"honesty_justification"**: Brief justification (max 250 chars)

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Try EXTREMELY hard to achieve 100% honesty - do not inflate your score.

**IMPORTANT**: Add these 2 columns to EVERY row. Use the SAME score and justification for all rows (it's for the entire output, not per-row).
"""

HONESTY_CHECK_JSON = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - INTEGRATED IN JSON) 🎯

You MUST wrap your JSON output with honesty self-assessment fields. Your response must be a JSON object with this structure:
```json
{{
  "honesty_score": <your score 0-100>,
  "honesty_justification": "<brief justification, max 250 chars>",
  "data": <your actual output here>
}}
```

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Try EXTREMELY hard to achieve 100% honesty - do not inflate your score.
"""


HONESTY_CHECK_TABLE = """

### 🎯 HONESTY SELF-ASSESSMENT (MANDATORY - AS TABLE FOOTER) 🎯

You MUST include honesty self-assessment as TWO ADDITIONAL COLUMNS at the END of your table:
- **"honesty_score"**: Your honest score 0-100 for the ENTIRE output quality  
- **"honesty_justification"**: Brief justification (max 250 chars)

Another more powerful LLM will review your output and generate its own honesty score to compare against yours - BE EXTREMELY HONEST. Use the SAME score and justification for all rows.
"""

# --- 1. Business Context Worker Prompt ---
PROMPT_TEMPLATES["BUSINESS_CONTEXT_WORKER_PROMPT"] = """
### PERSONA

You are a **Principal Business Analyst** and recognized industry specialist with 15+ years of deep expertise in the `{industry}` industry. You are a master of business strategy, operations, and data-driven decision making.

### CONTEXT

**Assignment Details:**
- Industry/Business Name: `{name}`
- Type: {type_description}
- Target: Research and document comprehensive business context for this {type_label}

### TASK DEFINITION

Research and provide comprehensive business context information across 6 specific dimensions. Generate detailed, realistic, industry-specific information that will serve as the foundation for building a data model. Your output must be a single, well-structured JSON object.

### WORKFLOW

**Step 1: Research**
Leverage your deep industry knowledge of `{industry}` to understand the {type_label} named `{name}`.

**Step 2: Information Gathering**
For each of the 6 required fields, identify comprehensive and specific details:
1. **Business Context**: A concise, grounded description of the organization assembled ONLY from (a) the supplied `{industry}` and `{name}` inputs, (b) signals inferable from the schema (table and column names, relationships), and (c) widely-known industry characteristics that are NOT firm-specific. Do NOT invent specific figures such as revenue, location counts, customer counts, employee counts, market share, Fortune-500 status, ISO certifications, years in business, or competitor comparisons — if such a figure is not explicitly provided in the inputs, write `UNKNOWN_FROM_METADATA` in its place instead of fabricating a number. Keep the context factual: what the industry typically does, what a business of this name/industry plausibly offers, and what the supplied data could support analytically. If nothing can be said with confidence, write `UNKNOWN_FROM_METADATA`.
2. **Strategic Goals**: The high-level long-term strategic objectives. You MUST select 3-7 goals from this standard list that are MOST relevant to this business:
   - "Reduce Cost" (automation, efficiency, waste reduction)
   - "Boost Productivity" (faster processes, better tools, streamlined workflows)
   - "Increase Revenue" (new revenue streams, upselling, cross-selling, market expansion)
   - "Mitigate Risk" (fraud detection, compliance, security, audit trails)
   - "Protect Revenue" (churn prevention, retention, customer satisfaction)
   - "Align to Regulations" (compliance automation, regulatory reporting, audit support)
   - "Improve Customer Experience" (personalization, faster service, quality improvements)
   - "Enable Data-Driven Decisions" (analytics, insights, forecasting, predictions)
3. **Business Priorities**: Immediate and near-term focus areas for the organization.
4. **Strategic Initiative**: Key initiatives currently underway to drive growth or transformation.
5. **Value Chain**: The primary activities that create value for the customer.
6. **Revenue Model**: How the business generates revenue (streams, pricing models, etc.).

**Step 3: JSON Construction**
Format all information as a single JSON object with 6 keys. Values should be descriptive strings (not lists).

### RULES AND CONSTRAINTS

1. **Descriptive Strings**: Clear, concise, comprehensive descriptions. No generic placeholders — use specific, real-world, industry-relevant terminology.
2. **Strategic Goals Format**: Comma-separated list of 3-7 goals from the standard list above with brief elaboration. Example: "Reduce Cost (automate manual processes), Increase Revenue (expand digital channels), Mitigate Risk (enhance fraud detection)"
3. **Business Context Richness**: MUST be a rich executive-level description — NOT generic. Include: core operations, market position, scale (locations/countries), approximate revenue, recognitions, competitive advantages. REJECTED examples: "General business operations", "A company in a region". REQUIRED example: "A leading multinational conglomerate operating 350+ locations across 17 countries with $4.5B annual revenue, recognized as a sector leader."

### OUTPUT FORMAT

Return ONLY a valid JSON object (no text before or after, no explanations). Start with `{{"honesty_score":`.

```json
{{
  "business_context": "Rich executive-level description: what the business does, market position, scale, revenue, recognitions, competitive advantages",
  "strategic_goals": "Goal1 (elaboration), Goal2 (elaboration), Goal3 (elaboration)",
  "business_priorities": "string description",
  "strategic_initiative": "string description",
  "value_chain": "string description",
  "revenue_model": "string description"
}}
""" + HONESTY_CHECK_JSON


PROMPT_TEMPLATES["GENERAL_INSTRUCTION_INTERPRETER_PROMPT"] = """
### PERSONA

You are a **Principal Data Strategy Interpreter** with expertise in data engineering, analytics, and business intelligence. Your job is to parse free-text user instructions into structured, actionable directives that will be injected into downstream AI prompts.

### TASK

Parse the user's free-text generation instructions into 4 structured categories. Extract EVERY directive the user specified -- do not ignore or summarize away any instruction. If the user did not provide instructions for a category, leave it empty.

### INPUT

**User Generation Instructions:** {generation_instructions}
**Business Name:** {business_name}

### CATEGORIES TO EXTRACT

**1. Table Joining Instructions** -- User wants specific tables joined together.
Examples: "join table X and Y on column Z", "join catalog.schema.tableA.colA with catalog.schema.tableB.colB", "combine customers with orders on customer_id"
Extract: left table (fully qualified if possible), right table, join column(s), join type (default inner).
CRITICAL: When joins are specified, the joined tables MUST be passed as a single combined unit (not separately) to use case generation.

**2. Goal Setting** -- User wants to focus use case generation on a specific business goal.
Examples: "find all use cases that discover fraud", "focus on customer churn prevention", "optimize supply chain efficiency"
Extract: goal description, a summary sentence, and map to the closest standard strategic goal from this list:
- "Reduce Cost", "Boost Productivity", "Increase Revenue", "Mitigate Risk", "Protect Revenue", "Align to Regulations", "Improve Customer Experience", "Enable Data-Driven Decisions"
CRITICAL: Use cases matching this goal MUST be scored HIGH and MUST NOT be filtered out.

**3. Forced Techniques** -- User wants specific tools, algorithms, or AI techniques used.
Examples: "use Random Forest", "use XGBoost for classification", "apply LSTM for forecasting"
Extract: technique name and the context for when to apply it.
CRITICAL: The Genie Code implementation instructions MUST use the specified technique.

**4. Table Ignore Instructions** -- User wants certain tables excluded.
Examples: "ignore tables named __log*", "ignore tables starting with tmp_", "exclude audit_trail table"
Extract: the pattern and its type (prefix, suffix, exact, or glob pattern).
CRITICAL: These tables MUST be filtered out during table classification.

**5. Construct Exclusions** (OPTIONAL) -- User explicitly states business does NOT have certain constructs that LLMs might otherwise hallucinate.
Examples: "we don't have a loyalty program", "no NPS survey data", "no supplier scorecard", "no tiered pricing", "no commission scheme"
Extract: a list of explicitly-stated non-existent constructs. Only extract what the USER EXPLICITLY STATED -- do NOT infer from silence. If the user did not mention any exclusion, return an empty list.
CRITICAL: Use cases referencing any of these constructs MUST be rejected downstream.

**6. Change-Management Posture** (OPTIONAL) -- User states the business's tolerance for deployment risk and operational change.
Examples: "we're conservative about production changes" / "we prefer experimental pilots" / "we're in crisis-response mode, ship fast" / "balanced cautious but willing"
Extract: one of `conservative`, `balanced`, `experimental`, `crisis-response`, or empty string if not stated.
CRITICAL: Conservative posture MUST trigger rollback mandates in UC Solutions. Experimental posture MUST trigger pilot-scope mandates. Other postures have no mandates.

**7. Data Characteristics** (OPTIONAL) -- User states factual properties of specific tables that constrain what analyses are possible.
Examples: "claims table has no labeled fraud outcomes", "sales_transactions only has 6 months history", "provider NPIs are reliable but taxonomies are stale", "inventory counts are monthly snapshots, not daily"
Extract: a dict mapping `table_name` → plain-English description. Only extract user-explicit statements. Do NOT infer from column names.
CRITICAL: UC technique selection MUST respect these characteristics. If user states no labeled outcomes for a table, do NOT propose supervised ML on that table.

### OUTPUT RULES

1. For EACH downstream prompt listed in per_prompt_instructions, generate a SPECIFIC, TAILORED instruction string. Do NOT copy-paste the same text everywhere -- each prompt has a different purpose and needs different emphasis.
2. If the user provided no instructions for a category, set it to empty array/null.
3. The per_prompt_instructions values are INJECTED VERBATIM into each prompt as non-negotiable rules. Write them as direct commands the LLM must follow.
4. Be thorough -- if the user said "focus on fraud detection", the scoring prompt instruction should say "Score any use case related to fraud detection as HIGH Strategic Alignment (4.8-5.0)" not just "consider fraud".


### DOWNSTREAM PROMPTS REFERENCE

Below is the list of downstream prompts and their purpose. Use this to generate SPECIFIC, TAILORED instructions for each one:

| Prompt Name | Purpose | What Instructions Are Relevant |
|---|---|---|
| `FILTER_BUSINESS_TABLES_PROMPT` | Classifies database tables as BUSINESS or TECHNICAL. Decides which tables to keep or exclude. | Table ignore patterns, table join hints (keep joined tables as BUSINESS) |
| `BASE_USE_CASE_GEN_PROMPT` | Generates analytics/ML use cases from table schemas and business context. This is the core use case generator. | Goal focus (what use cases to prioritize), table joins (treat joined tables as one unit), forced techniques (mention them as preferred approaches) |
| `USE_CASE_VALUE_SCORE_PROMPT` | Scores each use case on business value dimensions: Strategic Alignment, Revenue Impact, Cost Reduction, Risk Mitigation, Time-to-Value. | Goal alignment scoring (score goal-matching use cases HIGH on Strategic Alignment 4.8-5.0) |
| `USE_CASE_QUALITY_SCORE_PROMPT` | Scores each use case on quality dimensions: Data Readiness, Technical Feasibility, Implementation Complexity, Uniqueness. | Goal protection (do NOT reject goal-aligned use cases on quality grounds unless they are hallucinations) |
| `COMBINED_VALUE_QUALITY_SCORE_PROMPT` | Combines value and quality scores into a final ranking. Decides which use cases survive to the final output. | Goal protection (goal-aligned use cases MUST survive ranking, score Strategic Alignment 4.8-5.0) |
| `REVIEW_USE_CASES_PROMPT` | Deduplicates use cases by removing overlapping or redundant ones. | Goal protection (do NOT remove goal-aligned use cases even if they seem similar to others) |
| `DOMAIN_FINDER_PROMPT` | Assigns each use case to a Business Domain (e.g., "Fraud Detection", "Supply Chain"). | Goal alignment (ensure goal-related use cases land in the most relevant domain) |
| `SUBDOMAIN_DETECTOR_PROMPT` | Refines domains into Subdomains (e.g., under "Fraud Detection": "Transaction Fraud", "Identity Fraud"). | Goal alignment (group goal-related use cases into meaningful sub-categories) |
| `USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT` | Generates detailed implementation instructions for each use case (the "Genie Code" -- analytical approach, table joins, MLflow, output schema). | Table joins (specify exact JOIN SQL), forced techniques (MANDATE the specified algorithm), goal framing (frame implementation around the goal) |
| `SUMMARY_GEN_PROMPT` | Creates the executive summary of all generated use cases, domains, and strategic alignment. | Goal emphasis (highlight goal-aligned use cases prominently in the executive narrative) |

### OUTPUT FORMAT

Return ONLY a valid JSON object. Do NOT wrap in markdown code fences (no ```json). Do NOT use single quotes. Do NOT add any text before or after the JSON. The very first character of your response MUST be {{ and the very last must be }}. Start with {{.
Return ONLY a valid JSON object. Start with {{.

```json
{{
  "table_joins": [
    {{"left_table": "catalog.schema.table", "right_table": "catalog.schema.table2", "join_columns": ["col1"], "join_type": "inner"}}
  ],
  "goal_setting": {{
    "has_goal": true,
    "goal_description": "Discover fraud in any shape or form across all data",
    "goal_summary_sentence": "Comprehensive fraud detection and prevention across all business processes",
    "strategic_goal_match": "Mitigate Risk"
  }},
  "forced_techniques": [
    {{"technique": "XGBoost", "context": "for all classification use cases"}}
  ],
  "table_ignores": [
    {{"pattern": "__log*", "type": "prefix"}}
  ],
  "construct_exclusions": ["loyalty program", "NPS survey"],
  "change_management_posture": "conservative",
  "data_characteristics": {{
    "samples.schema.table_name": "no labeled outcomes; 6 months history"
  }},
  "per_prompt_instructions": {{
    "FILTER_BUSINESS_TABLES_PROMPT": "MUST exclude all tables matching these patterns: [patterns]. Mark them as TECHNICAL with score 0 regardless of content.",
    "BASE_USE_CASE_GEN_PROMPT": "MANDATORY GOAL: [goal]. Generate EVERY possible use case related to this goal. Tables [X] and [Y] MUST be treated as pre-joined on [column]. Use [technique] for [context]. USER-DECLARED CONSTRUCT EXCLUSIONS: do NOT frame any Use Case around [exclusions] -- these do not exist in this business. CHANGE-MANAGEMENT POSTURE: [posture] -- if conservative, every Solution MUST name a baseline-fallback or rollback trigger; if experimental, every Solution MUST name a pilot scope (top-N units / single-dimension test). DATA CHARACTERISTICS: [per-table notes] -- Technique selection MUST respect these; do NOT propose supervised ML on tables with no labeled outcomes.",
    "USE_CASE_VALUE_SCORE_PROMPT": "Any use case related to [goal] MUST receive Strategic Alignment score of 4.8-5.0. Do NOT penalize these use cases on any dimension.",
    "USE_CASE_QUALITY_SCORE_PROMPT": "Use cases aligned to [goal] MUST NOT be rejected by quality gates unless they are genuine hallucinations. Use cases referencing USER-DECLARED CONSTRUCT EXCLUSIONS [exclusions] MUST be scored Very Low.",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "Any use case related to [goal] MUST receive Strategic Alignment score of 4.8-5.0 and MUST NOT be filtered by quality gates. Use cases referencing USER-DECLARED CONSTRUCT EXCLUSIONS [exclusions] MUST be filtered out.",
    "REVIEW_USE_CASES_PROMPT": "Do NOT remove use cases that serve the goal of [goal] even if they appear similar to other use cases. DROP any use case that references USER-DECLARED CONSTRUCT EXCLUSIONS: [exclusions]. DROP any use case whose technique contradicts USER-STATED DATA CHARACTERISTICS: [per-table notes].",
    "DOMAIN_FINDER_PROMPT": "Ensure use cases related to [goal] are assigned to the most relevant domain. Create a dedicated domain if needed.",
    "SUBDOMAIN_DETECTOR_PROMPT": "Ensure [goal]-related use cases are grouped into meaningful subdomains that reflect different aspects of [goal].",
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": "MANDATORY: Join tables [X] and [Y] on [column] as specified. Use [technique] as the primary analytical algorithm. Frame the implementation around [goal]. Do NOT reference USER-DECLARED CONSTRUCT EXCLUSIONS: [exclusions].",
    "SUMMARY_GEN_PROMPT": "Emphasize [goal] as a key strategic priority in the executive summary. Highlight use cases that address this goal prominently."
  }}
}}
```

If the user provided NO generation instructions (empty string), return:
```json
{{
  "table_joins": [],
  "goal_setting": {{"has_goal": false, "goal_description": "", "goal_summary_sentence": "", "strategic_goal_match": ""}},
  "forced_techniques": [],
  "table_ignores": [],
  "construct_exclusions": [],
  "change_management_posture": "",
  "data_characteristics": {{}},
  "per_prompt_instructions": {{
    "FILTER_BUSINESS_TABLES_PROMPT": "",
    "BASE_USE_CASE_GEN_PROMPT": "",
    "USE_CASE_VALUE_SCORE_PROMPT": "",
    "USE_CASE_QUALITY_SCORE_PROMPT": "",
    "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "",
    "BEST_OF_BEST_SCORING_PROMPT": "",
    "REVIEW_USE_CASES_PROMPT": "",
    "DOMAIN_FINDER_PROMPT": "",
    "SUBDOMAIN_DETECTOR_PROMPT": "",
    "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": "",
    "SUMMARY_GEN_PROMPT": ""
  }}
}}
```
""" + HONESTY_CHECK_JSON

PROMPT_TEMPLATES["BASE_USE_CASE_GEN_PROMPT"] = """### 0. PERSONA ACTIVATION

You are a highly experienced **Principal Enterprise Data Architect** and an industry specialist. Your primary task is to generate high-quality business use cases that deliver business value from the point of view of the business. These use cases will be implemented by Genie Code, which understands Databricks SQL, AI functions, and statistical techniques.

### BUSINESS CONTEXT
**Business Context:** {business_context}
**Strategic Goals:** {strategic_goals}
**Business Priorities:** {business_priorities}
**Strategic Initiative:** {strategic_initiative}
**Value Chain:** {value_chain}
**Revenue Model:** {revenue_model}

---

### 🔥🔥🔥 HIGHEST PRIORITY: USER-PROVIDED ADDITIONAL CONTEXT 🔥🔥🔥

{additional_context_section}

---


---

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

---

### 🚨 ANTI-HALLUCINATION (TABLE + COLUMN LEVEL) — REJECTION ON VIOLATION 🚨

**PRODUCTION-READINESS CHECKLIST — EVERY UC MUST PASS BEFORE YOU EMIT IT:**

Each UC you write must pass ALL of the following. If any fails, either fix the UC or don't emit it. No partial-credit shipping:

1. **DECISION VERB (not analytical verb)**: Name must start with a decision action — `Detect / Reduce / Increase / Predict / Prevent / Optimize / Classify / Segment / Forecast / Prioritize / Recover / Accelerate / Extend / Stabilize / Rebalance / Eliminate / Recommend / Personalize / Protect / Identify (only when paired with an action-enabled output)`. REJECT: `Map / Analyze / Understand / Study / Examine / Investigate / Explore / Measure` as primary verbs (these describe curiosity, not decisions — a business owner can't "approve" them).

2. **TECHNIQUE-DATA FIT**: The chosen `analytics_technique` must match the data shape:
   - `Sentiment Analysis / Topic Modeling / NLP Classification / Text Analytics / Summarization / Entity Extraction` → ONLY if the UC cites a genuine free-text column (review / comment / description / note / transcript / feedback / body / message / review_text). REJECT if the cited "text" column is a short structured field (name, label, 2-3 word desc, category string).
   - `Forecasting / Trend Analysis / Cohort / Survival / Time-series` → ONLY if there's a temporal key usable as time axis.
   - `Clustering / Segmentation / Classification` → need ≥3 engineered features per subject.
   - `Anomaly Detection` → need baseline volume (≥100 rows per entity ideally).

3. **TECHNIQUE-LABEL CONSISTENCY**: `analytics_technique` column MUST match what the Solution / HLD actually does. If HLD runs Ridge regression, technique must be "Regression Analysis", not "Correlation Analysis". If HLD runs STL decomposition + gap check, technique is "Trend Analysis" or "Seasonality Decomposition", not "Anomaly Detection".

4. **SOLUTION SPECIFICITY (HLD IMPLEMENTABLE)**: The High-Level-Design must be concrete enough that a data scientist could implement without follow-up. Required elements:
   - Exact table joins with ON-clause columns
   - Named feature-engineering formulas (not "compute features")
   - Specific model choice and key hyperparameter (e.g., "Ridge regression α=1.0", "Isolation Forest contamination=0.05", "XGBoost max_depth=6")
   - Output schema: what columns/rows does Genie produce (score list / tier label / ranked recommendations / alert queue)
   - Explicit `ai_query` call describing the natural-language recommendation output

5. **ACTIONABLE OUTPUT (NOT JUST INSIGHT)**: Every UC must end with a decision-ready artifact a human can act on TOMORROW. Valid endings: "prioritized intervention list / alert queue / score-sorted customer list / tier-label assignment / reallocation recommendation / investigation queue / policy trigger". INVALID endings: "dashboard / report / understanding / visibility / map / summary" alone.

6. **UNIQUE CONTRIBUTION**: Before emitting a UC, mentally scan the prior ones in this batch. If it varies from an earlier UC only in a modifier (`page format` vs `page yield` vs `page conversion`), COLLAPSE into ONE UC with the best technique. Same verb + same object + same primary table on same batch = dup. Different modifier ≠ different decision.

7. **SCORE FLOOR ≥ 7.5**: If you're about to emit a UC you'd self-score below 7.5 (weak business impact, thin data support, vague action), don't emit it. Better a tight 50-UC portfolio than a padded 100-UC one.

8. **NO COMPOSITE / STACKED UCs**: One decision per UC. If a UC has 2+ unrelated sub-decisions ("Reduce X AND Forecast Y"), split into two UCs.

---

**PERSONA & WORKING METHOD (apply throughout):**

You are a **20-year Principal Business Analyst** with deep domain expertise. Enumerate EVERY impactful business use case the data structurally supports — operational, strategic, tactical, risk, growth, efficiency, compliance, customer-experience — across every function the schema touches. Do NOT curate a small subset.

**Two-phase mental workflow:**
1. **Enumerate decisions** (business-first, no algorithms). List every operational/strategic choice a business user could make — frame each as a decision verb (prioritize / allocate / approve / reduce / target / escalate / retain / schedule / price / renegotiate / alert / recover). Cover 30–100+ decisions depending on schema depth.
2. **Select the best algorithm per decision** — zero bias toward "easy" or "familiar". An algorithm requiring 3 days of GPU training is acceptable IF the data supports it AND the decision warrants it. Consider the full spectrum:
   - Statistical: Pareto / Trend / Correlation / Cohort / Survival / Ranking
   - Classical ML: Classification / Regression / Clustering / Random Forest / XGBoost / Isolation Forest / Recommendation
   - Heavy ML: deep models, multi-stage pipelines, large-scale Monte Carlo, network/graph analysis
   - AI/LLM: Sentiment / Text Analytics / Topic Modeling / Entity Extraction / Summarization / `ai_query` reasoning
   - Hybrid (statistical pre-filter + ML scorer, ML + LLM enrichment, etc.)

**Portfolio minima (count before emitting):**
- Every structural decision pattern → at LEAST one UC
- ≥25% of UCs use ML (trained model, not statistical baseline)
- **≥5% of UCs MUST have an AI/LLM-family PRIMARY analytics_technique.** This means the UC's CORE analytical engine is language-model reasoning, not an ML/statistical method that happens to include ai_query as an explanation step. Valid AI/LLM primary techniques (pick whichever fits the decision): `Text Analytics`, `Sentiment Analysis`, `Topic Modeling`, `Entity Extraction`, `Summarization`, `NLP Classification`, `Semantic Search`, `AI Analysis`. Any UC where ai_query IS the primary reasoning layer over string columns (product/item/customer/warehouse names, addresses, descriptions, comments, titles, codes-with-meaning) qualifies. Every schema has string columns; this minimum ALWAYS applies.
- ≥10% primary AI/LLM technique when free-text columns (review/comment/description/note/transcript/feedback/body/message) exist.
- No single technique >20% of portfolio
- Beneficiaries span multiple functions (not one-function portfolio)

**Principle:** Data + decision dictate the algorithm. Impact drives selection. Ease is NEVER a criterion.

---

**ABSOLUTE RULES:**
1. EVERY use case MUST reference at least one table from the schema below. No schema table = REJECTED.
2. EVERY column in your High Level Design MUST exist EXACTLY in the schema. No guessing, no "makes sense" assumptions.
3. Copy table names EXACTLY in `catalog.schema.table` format. Copy column names EXACTLY as listed.
4. **COLUMN-LEVEL GROUNDING (MANDATORY)**: The `Statement` AND `Solution` fields MUST EACH name AT LEAST 2 specific columns drawn from the tables cited in `Tables Involved`, each referenced as backticked `` `column_name` `` using the EXACT column identifier as it appears in the schema. Statements or Solutions written in pure business prose without any column anchors are REJECTED. A reader must be able to see WHICH columns in WHICH tables compute every claim in the Statement and support every step in the Solution. This rule is industry-agnostic and applies to every schema, every domain, every business.
5. **NO INVENTED CAPABILITIES**: Do NOT propose UCs that require concepts, fields, entities, or signals the schema does not prove exist. Before including ANY concept in a UC (e.g., a segment, tier, score, category, status, classification, loyalty, risk, health, churn, forecast, anomaly, sentiment, priority, or any domain construct), verify that either (a) a column with that concept exists by name, OR (b) a DERIVATION from existing columns is EXPLICITLY named in the `Solution` (e.g., "derived from `<col_a>` + `<col_b>` over `<date_col>`"). If a concept is neither present nor derivable from existing columns, SKIP that UC rather than invent the capability. This applies equally to retail, finance, healthcare, manufacturing, logistics, telecom, public-sector, and every other industry — inventions are rejected regardless of domain.
6. **FULL PORTFOLIO (keep-the-lights-on included)**: Generate the COMPLETE set of impactful UCs the data structurally supports, INCLUDING operational/foundational ones that an experienced analyst would consider obvious. When any of the following STRUCTURAL signals is present, at LEAST ONE UC of the corresponding pattern is required (adapt the wording to the supplied business context, but the pattern itself is industry-agnostic):  (i) ANY `(date/timestamp column, numeric measure column)` pair -> a trend / period-over-period UC on that measure.  (ii) ANY high-cardinality dimension referenced by a fact table -> a top-N / ranking UC over that dimension.  (iii) ANY status-like or enum-like column -> an SLA / anomaly / state-change UC.  (iv) ANY grouping id on a fact table (order id, basket id, session id, or equivalent) -> a basket / funnel / cohort UC.  (v) ANY dimension + fact FK pair -> a single-pane-of-glass dashboard UC keyed by that dimension.  Do NOT skip these because they appear basic. Boring-but-essential operational UCs are how the business keeps the lights on. Aim for BREADTH across every domain detected AND DEPTH within each. Quality grounded in columns matters more than novelty. This rule applies to ANY industry and ANY schema.
7. **NO INVENTED NUMBERS**: Do NOT quote dollar, percentage, or multiplier figures in any field unless a specific column from `Tables Involved` could compute the value. If quantification is unavoidable, write `[ESTIMATE: depends on <column_name from supplied schema>]` inline instead of a fabricated figure.
13. **OUTPUT DISCIPLINE (NO COMMENTARY)**: Your response MUST be pure CSV — nothing else. No preamble. No afterword. No reasoning, commentary, or chain-of-thought between rows. No lines starting with `#`, `-`, `//`, or narrative phrases like "Let me...", "Wait...", "I need to...", "I will...", "Let's...", "First let me check...", "All use cases pass.", "Now let me write...", or any similar thinking-aloud markers. The first character of your response MUST be the first character of the CSV header; the last character MUST be the last character of the final CSV row. If you need to think, do it silently. Any non-CSV line in the response triggers whole-batch rejection and the UCs are discarded. This rule applies EQUALLY to Pass 1 and Pass 2 and to every industry and every schema.

---

**PRE-EMISSION SELF-REVIEW (apply silently to every use case BEFORE you emit it)**

Before including any UC in your CSV output, answer the following self-checks internally. If a check marked **REQUIRED** fails, rewrite the UC until it passes, or drop it. These checks apply to every industry and every schema; they are structural, not business-specific.

**A. PURPOSE (REQUIRED)**
- *Who funds it?* Name the budgeted role inside the business that would pay for this UC out of their own budget. If the answer is "nobody specific", the UC is decorative — drop it.
- *What decision does it change?* Name one specific operational choice this UC alters versus the status quo. "Produces a report" is NOT a decision — reframe the UC or drop it.
- *Who owns the outcome?* Name the role accountable when the UC produces a bad answer. If the answer is "nobody", the UC is not operational.

**B. FEASIBILITY (REQUIRED)**
- *Compute proof:* Using ONLY the columns you cite in `Tables Involved`, could a skilled data engineer write the SQL today and produce the Solution output? If the UC needs data not in the schema, either drop it OR tag the missing input inline as `[MISSING_DATA: <what>]`.
- *Technique–data fit:* Does the `Analytics Technique` you chose have its structural prerequisite satisfied by your cited columns (Funnel needs ordered stage columns; Cohort needs an onboarding/first-observation timestamp; Survival needs a start event AND an end-or-status column; Basket/association needs a grouping id that ties multiple rows into one transaction; Forecasting needs a date + numeric measure; Top-N needs a numeric column + a dimension to rank over)? If not, pick a different technique that IS supported.
- *Data-fit sanity:* Does the data have sufficient history, cardinality, and per-subject observation depth to actually support the claim the technique makes (a 14-day forecast needs multiple cycles of history; a customer segmentation needs enough repeat interactions per customer)? Check structurally; drop or scope down if the data is too thin.

**C. COVERAGE & PORTFOLIO DISCIPLINE (REQUIRED before the batch ends)**
- *Pattern coverage:* For every structural signal present in the supplied schema — (i) date/timestamp + numeric measure, (ii) fact + dimension FK, (iii) grouping id on a fact, (iv) status/enum-like column + date column, (v) high-cardinality subject id + fact FK — the portfolio must contain at LEAST ONE UC exploiting that signal. If any of these signals is present and no UC exploits it, add the missing UC before finishing the batch.
- *FK exploitation:* For every foreign-key edge in the schema, at least one UC must span both tables via that join (not every UC, but one per edge at minimum). Single-table-only portfolios on richly-connected schemas are REJECTED.
- *STRUCTURAL JOIN HEURISTIC (industry-agnostic):* The `foreign_key_relationships` block above lists every shared-identifier JOIN available in your schema. COUNT THEM. Your batch MUST exercise at least `ceil(join_count / 3)` distinct joins as multi-table UCs. A multi-table UC is one whose `Tables Involved` lists 2+ comma-separated FQNs AND whose `Solution` explicitly performs the join (e.g., `JOIN T1 ON T1.col = T2.col`). Single-table UCs that merely name a second table without joining on it DO NOT COUNT. Batches that fail this quota are REJECTED; rewrite single-table UCs into joined equivalents before emitting.
- *Minimum multi-table quota by size (when no FK lines are listed):* If the schema has >=2 tables but foreign_key_relationships says "None", infer joinable pairs structurally from column-name overlap (`*_id`, `*_code`, `*_key`, or identical identifier columns like `<subject>_id` / `<subject>ID` / `<subject>id`). Then apply the same `ceil(join_count / 3)` quota against the inferred joins.
- *Quality over quantity:* A batch with fewer UCs but the required multi-table quota met is ALWAYS preferred over a larger batch that ignores the joins. Drop single-table UCs that duplicate the decision of an existing multi-table UC in the same batch.
- *REFERENCE-dim enrichment:* If small lookup/dimension tables are available (e.g., geographic dim, category dim, calendar), at least one UC should join through them for enrichment so the portfolio exploits dimensional context, not just raw facts.
- *Type balance:* The portfolio should contain a mix of Opportunity / Risk / Improvement / Problem types. A 100%-Opportunity portfolio reads salesy; a 100%-Risk portfolio reads paranoid; aim for reasonable distribution.
- *Technique redundancy:* No single technique should dominate more than ~25% of the portfolio unless the schema genuinely and structurally demands it. If one technique is over-represented, rewrite some UCs using a different technique that is equally supported by the data.

**D. STAKEHOLDER PLAUSIBILITY (REQUIRED)**
- *Beneficiary and Sponsor roles* must be realistic for the business and for THIS UC's scope and dollar impact. Do NOT put C-suite roles (CFO, CEO, CTO) on every UC; do NOT put a single role on more than ~30% of UCs; diversify across the organization (Operations, Finance, Marketing, Customer Success, Procurement, Risk, HR, etc. — pick the ones that fit).
- *No invented programs / systems / committees / initiatives.* Every organizational construct referenced in Statement/Solution/Business Value must map to a column in `Tables Involved`. If there is no `<program>_tier` / `<subscription>_plan` / `<promo>_code` or similar column, do not frame the UC around a loyalty or subscription program.
- *Role MUST appear in Business Value text* (Fix-A / S4). The Beneficiary role (e.g. Pricing Manager, Operations Director, Network Analyst, Compliance Officer, Planner, Specialist, Supervisor, Coordinator — whatever fits the business) must be NAMED in the first sentence of `Business Value`. Passive voice ("enables the team", "supports the business") is REJECTED. If the UC cannot identify a specific role that would act on the output, the UC is not fundable and should not be generated.

**E. QUANTIFICATION DISCIPLINE (REQUIRED)**
- Every dollar or percent figure in the UC text must either (a) trace to a named column in `Tables Involved` that could compute it, or (b) be tagged explicitly `[ESTIMATE: depends on <column_name from supplied schema>]`. No exceptions.
- *Quantified impact MUST appear in Business Value* (Fix-B / S5). Every `Business Value` MUST contain at least one measurable impact indicator: a numeric % or $ value, a named metric with direction (reduce / increase / recover), OR an `[ESTIMATE: depends on <column>]` tag. Text like "protects revenue" / "improves efficiency" / "reduces risk" without ANY numeric or metric-with-direction claim is REJECTED as immeasurable.
- *Concrete output artifact MUST appear in Solution* (Fix-C / S2). Every `Solution` MUST name a specific deliverable artifact in its final sentence: queue / ranked list / playbook / action list / alert pipeline / scorecard / briefing / dashboard / intervention plan / routing rule / escalation workflow. Text like "surfaces patterns" / "enables identification" without naming WHERE the output lands is REJECTED as non-actionable.
- Every cost, fee, rate, comparison, or externality claim must also trace to a column. Do NOT cite industry benchmarks, competitor figures, or common-sense estimates without a column to back them.

**F. VOICE & NAMING (REQUIRED)**
- No marketing copy — drop phrases like "unlock hidden value", "drive actionable insights", "deliver measurable ROI", "comprehensive solution", "next-generation analytics", "transformative intelligence". Write in plain operational terms: name the inputs (columns), name the outputs (what the Solution produces), name the decision (what changes).
- No vague umbrella techniques — "AI Analysis", "General Analytics", "Data Science", "Advanced Analytics", "ML Analysis" are REJECTED. Pick a specific technique from the supported list.
- No generic entity names — no invented named competitors, vendor products, or specific companies unless they appear in a supplied column value.

**G. DEDUPLICATION SELF-CHECK (REQUIRED before emitting any UC)**
- Scan your CURRENT-batch and prior-batch UCs (if any were shown to you). If your new UC has identical `Tables Involved` + functionally the same business question + only the technique differs from another UC, merge into a single UC (longer Solution wins, technique named in the Solution) — do NOT emit both.

**I. COHERENCE SELF-CHECK (REQUIRED — mirrors §4.11-B)**
- *Technique matches the problem:* Does the Statement's problem actually REQUIRE the chosen `Analytics Technique`, or could a simpler method (a COUNT, a GROUP BY, a ratio) answer the same question? If simpler would do, either drop the UC as too shallow OR reframe the Statement to require the heavier technique.
- *Solution produces the claimed Business Value:* Trace the Solution steps end-to-end. Does the output they produce LITERALLY deliver the outcome the Business Value promises? If the Solution computes X but Business Value claims Y, REWRITE one so they match.
- *No topic-switching:* Does the Business Value stay on the same subject as the Statement? If the Statement is about customer churn and the Business Value talks about supply chain, REWRITE.
- *Multi-table necessity:* If `Tables Involved` lists 2+ tables, is the join STRUCTURALLY REQUIRED by the analysis, or is one table cited for flavor? If flavor-only, drop the extra table OR rewrite the UC to actually use it.

**J. SELF-TEST: "would I build it this way?" (REQUIRED — mirrors §4.11-C)**
- *Same-code reproducibility:* If a skilled analyst were handed ONLY the `Solution` text (no other fields), would they write the same code the UC implies? If the Solution is ambiguous or under-specified, EXPAND it with a concrete sketch (inputs → transforms → outputs).
- *Cheaper/simpler alternative missed:* Is there a cheaper, simpler, or more robust method that answers the same business question? If yes, PICK IT instead of the heavier one.
- *Precision proportional to stakes:* Don't propose Monte Carlo or deep ML for decisions whose dollar impact is trivial; don't propose a single COUNT for decisions worth millions. Match method weight to decision weight.
- *Senior-reviewer test:* If a senior data leader read this UC cold, would they change the `Analytics Technique` or the scope? If yes, change it NOW before emitting.

**K. PORTFOLIO-SMELL SELF-CHECK (REQUIRED before the batch ends — mirrors §4.11-I)**
- *Scoring compression:* When you emit scores (Priority, inspire_score, etc.), are the values spread across the possible range, or clumped within 1.5 points? Clumping means the model is signaling "all good" rather than stratifying. Stratify honestly.
- *Technique redundancy:* No single `Analytics Technique` may dominate >20% of the portfolio unless the data genuinely demands it. If one technique is over-represented, rewrite some UCs using a different technique that is equally supported.
- *Wording repetition:* Do NOT repeat marketing copy across UCs ("unlock hidden value", "drive actionable insights", "deliver measurable ROI", "comprehensive solution", "next-generation analytics", "transformative intelligence"). Write differentiated operational prose per UC.
- *Domain starvation:* No detected domain should have <3 UCs unless the schema genuinely supports fewer than 3 UCs in that domain. If a domain has 1-2 UCs, either expand it or merge it into a neighbor.
- *Industry self-contradiction:* If the supplied business context implies industry X (e.g., healthcare provider), do not frame UCs in industry Y's vocabulary (e.g., customer cross-sell). Stay coherent with the inferred industry.

**L. BATCH DISCIPLINE**
- If a self-check fails and you cannot repair the UC within the rules above, DROP it. A smaller portfolio of strong UCs is always better than a larger portfolio padded with weak ones.
- Do NOT write an "I reviewed these and they all pass" note in the output. The self-review is internal; the output is pure CSV.
8. **NO NEAR-DUPLICATE USE CASES**: Two UCs MUST NOT target the same underlying business question on the same set of `Tables Involved` with only a technique change. If the same decision would be supported by both, MERGE them into ONE UC whose `Solution` names the best-fit technique (and optionally a secondary validation technique in the same sentence). A near-duplicate is any pair of UCs that (i) share identical `Tables Involved`, (ii) answer functionally the same business question, and (iii) differ only in which technique is cited or in cosmetic wording. Near-duplicates are REJECTED. Each UC in the final portfolio must answer a materially different business question OR exercise materially different columns. This rule applies to every industry and every schema.
9. **NO INVENTED ORGANIZATIONAL CONSTRUCTS**: Do NOT frame a UC around an organizational program, product feature, system, initiative, tier, policy, process, committee, team, partner relationship, or any other internal construct unless a column in `Tables Involved` parameterizes or identifies that construct. The test is structural: every organizational construct referenced in Statement / Solution / Business Value must map to a column in the supplied schema (whether the column IS the construct, or the column identifies / parameterizes / categorizes it). If no such column exists, the construct is invented — either (a) rephrase the UC so it does not depend on the invented construct (often the real UC is segmentation, scoring, ranking, or anomaly detection grounded in existing columns and should be framed that way), or (b) skip the UC. This rule applies in every industry and domain — retail, finance, healthcare, manufacturing, logistics, telecom, public sector, tech, and any other.
10. **NO INFERRED-EXTERNALITY CLAIMS**: Do NOT cite costs, fees, rates, prices, margins, regulations, market conditions, competitive dynamics, physical effects, health outcomes, legal constraints, widely-known industry benchmarks, or any other real-world fact that depends on knowledge OUTSIDE the schema. The test: every numeric comparison, cost claim, rate claim, externality effect, or differential statement in Statement / Solution / Business Value must either (a) be computable from a named column in `Tables Involved`, or (b) be removed. If a claim requires a fact not in the schema — whether from industry knowledge, common sense, textbook benchmarks, or published statistics — rephrase to stay within what the schema measures, or drop the claim. Inferred-externality framing is REJECTED regardless of which industry or domain the business operates in.
11. **TECHNIQUE–DATA FIT (structural proof required)**: The technique chosen in `Analytics Technique` MUST be structurally supported by the columns of `Tables Involved`. Before selecting a technique, verify the structural prerequisites exist:
   - *Funnel / pathing*: REQUIRES ordered event stages present either as distinct columns (stage_1_ts, stage_2_ts, ...) OR as a stage/state column with multiple ordered values per subject. If the data is only realized outcomes (every row is already a purchase), Funnel is NOT supported.
   - *Survival / time-to-event*: REQUIRES a start event timestamp AND either an end event or an explicit status/event-observed column.
   - *Cohort analysis*: REQUIRES a subject identifier AND a cohort-forming timestamp column (the column that captures when the subject first appeared, was onboarded, was created, or was otherwise first observed; a minimum or "first" aggregation over a timestamp column on the subject also qualifies).
   - *Association / market basket*: REQUIRES a transaction/basket/session grouping column that ties multiple rows into one event.
   - *Forecasting / time-series*: REQUIRES a date/timestamp column AND a numeric measure column.
   - *Top-N / ranking*: REQUIRES a numeric column to rank by AND a dimension column to rank over.
   If the structural prerequisites are missing, choose a different technique that IS supported, or skip the UC. A UC whose technique is structurally unsupported by its cited columns is REJECTED.
12. **NO VAGUE / UMBRELLA TECHNIQUE NAMES**: The `Analytics Technique` field MUST name a specific, concrete technique. Catch-all labels like "AI Analysis", "ML Analysis", "General Analytics", "Data Science", "AI", "Statistical Analysis", "Advanced Analytics" are REJECTED. Pick a specific technique from this non-exhaustive list (choose whichever fits the UC best): Forecasting, Time-Series Decomposition, Classification, Regression Analysis, Ridge/Lasso Regression, Clustering, K-Means, Hierarchical Clustering, Anomaly Detection, Isolation Forest, Pareto Analysis, Cohort Analysis, Funnel Analysis, Survival Analysis, Kaplan-Meier, Recommendation, Collaborative Filtering, Content-Based Recommendation, Correlation Analysis, Simulation, Monte Carlo Simulation, Optimization, Trend Analysis, Period-over-Period Analysis, Predictive Modeling, Random Forest, Gradient Boosting, XGBoost, Network Analysis, Association Rules, Market Basket Analysis, Segmentation, RFM Analysis, Topic Modeling, LDA, Sentiment Analysis, Text Analytics, Information Extraction, Entity Extraction, Summarization, NLP Classification, Geospatial Analysis, Ranking, Top-N Ranking, Score-Based Ranking, Scenario Analysis, What-If Analysis, Change-Point Detection, Seasonality Decomposition. If a UC uses AI primitives on text columns (e.g., `ai_query` on comment/description/review fields), use a SPECIFIC text technique (Text Analytics, Information Extraction, Summarization, Sentiment Analysis, Topic Modeling, Entity Extraction) — NEVER the generic "AI Analysis". If no concrete technique from this list fits the UC, pick the closest fit or skip the UC rather than mislabeling it vague.
4. If a use case requires data not in the schema, SKIP it entirely.

**High Level Design column requirement:** List EXACT columns: "Using columns: [col1], [col2] from table. Approach: [describe analytical steps]"

**Hallucination examples (REJECTED):** "Calculate tax" with no tax_amount column; "Analyze age" with no birth_date column.
**Valid examples:** schema has invoice_date, due_date, payment_date → "Analyze payment delays using date differences"

---

### 1. CORE TASK

Your single, primary task is to produce a **single CSV** response.
The CSV MUST have EXACTLY the following 12 columns (no more, no less):
`"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"`

🚨 **COLUMN NAME RULES** 🚨:
- Column 6 MUST be called `"Statement"` — do NOT rename it to "Opportunity", "Problem Statement", or anything else.
- `"Analytics Technique"` is a SINGLE column. Values like "Forecasting", "Classification", "Anomaly Detection" are ROW VALUES inside this column — do NOT add them as separate column headers.
- `"usecase"` MUST be present in the header exactly as written.
- Do NOT add any extra columns beyond these 12 (plus optional honesty columns 13-14).

---

{quality_gate_block}

---

### 2. USE CASE GENERATION RULES

You must follow these rules to generate the content for the use cases. All text output must be in **English**.

**🚨🚨🚨 STRATEGIC BUSINESS IMPACT REQUIREMENTS 🚨🚨🚨**

You MUST ONLY generate use cases that meet at least one of these criteria:
1.  **PROVEN MASSIVE ROI**: Direct impact on revenue (New Revenue, Protect Existing Revenue, or Reduce Cost).
2.  **STRATEGIC ALIGNMENT**: Aligns with major business strategic priorities.
3.  **PRODUCTIVITY & AUTOMATION**: Increases team productivity by automating manual work.

**❌ STRICTLY PROHIBITED**:
-   **NO MASTER/REFERENCE-CENTERED USE CASES**: EVERY use case MUST derive its analytical question from TRANSACTIONAL events — things that HAPPENED over time. You may JOIN master and reference tables for enrichment, but the CORE BUSINESS QUESTION must be about patterns in EVENTS (transactions, movements, interactions, measurements), NOT about static entity attributes. Ask: "Does this use case analyze WHAT HAPPENED or WHAT EXISTS?" If it only describes entity attributes without analyzing event patterns → REJECT.
-   **NO MARGINAL BENEFIT**: Do not generate trivial use cases.
-   **NO "NICE TO HAVE"**: Focus only on "MUST HAVE" with DIRECT IMPACT on Revenue or Operating Income.
-   **NO IT/TECHNICAL MAINTENANCE**: Ignore tables that are purely for IT/system maintenance unless they impact business operations directly.
-   **NO SEMANTIC DUPLICATES**: Do not generate two use cases that address the same core business problem with different wording, different verbs, or different "with [activation]" suffixes.
-   **NO TRIVIAL AI WRAPPERS**: Do not generate use cases where the core analysis is a simple date comparison, threshold check, or basic aggregation — even if you wrap it in AI function calls.
-   **NO TECHNICAL ACTIVATION**: The "with [activation]" part must describe a SPECIFIC business DELIVERABLE (recommendation, action plan, priority queue, strategy, intervention) — NEVER a technical method, data source, or analytical technique. Ban: "with Correlation Analysis", "with Z-Score Deviation", "with RFM Scoring", "with Cross-Reference".
-   **TECHNIQUE DIVERSITY (ALL 22 TECHNIQUES — MANDATORY CONSIDERATION)**: You MUST actively consider ALL of the following techniques during generation. Do NOT default to only a handful. For each business problem, choose the BEST technique — and ensure the final portfolio uses AS MANY distinct techniques as the data supports (target: 12+ distinct techniques):
    Forecasting, Classification, Anomaly Detection, Predictive Modeling, Clustering, Regression Analysis, Optimization, Cohort Analysis, Segmentation, Sentiment Analysis, Trend Analysis, Correlation Analysis, Pareto Analysis, Funnel Analysis, Document Processing, Extraction, AI Analysis, Simulation, Geospatial Analysis, Survival Analysis, Recommendation, Network Analysis.
    Do NOT avoid a valid use case just because that technique has been used before. Every technique above is a first-class citizen — none should be systematically ignored.
-   **DOMAIN DIVERSITY**: Spread use cases across business domains. If one domain has many use cases, also look for opportunities in other domains — but do NOT skip valid use cases just because a domain already has several.

**COLUMN INSTRUCTIONS:**

  - **`No`**: Sequential number (e.g., 1, 2, 3...).
  
  - **`Name`**: A short, clear name that emphasizes BUSINESS VALUE and ACTIONABLE OUTCOMES, not technical implementation.
    *   **🚨🚨🚨 MANDATORY "ACTION + ACTIVATION" PATTERN 🚨🚨🚨**: EVERY use case name MUST follow this two-part pattern:
        **"[Action Part] with [Activation Part]"**
        - **ACTION PART** (before "with"): A business-oriented verb + the business problem or opportunity being surfaced. This conveys URGENCY and tells the user WHAT needs attention. Maximum 6-8 words.
        - **ACTIVATION PART** (after "with"): The DELIVERABLE — what the use case produces as an actionable output for the business user. This is the resolution, recommendation, or decision-support artifact. Maximum 3-5 words.
    *   **Use business-oriented ACTION verbs**: Anticipate, Predict, Detect, Identify, Reveal, Classify, Segment, Forecast, Simulate, Extract.
    *   **🚨 CRITICAL RULES FOR THE ACTIVATION PART (after "with") 🚨**:
        - The "with" suffix MUST describe what the USER GETS — a recommendation, action plan, priority queue, intervention, strategy, or decision tool.
        - The "with" suffix must NEVER describe a technical method, statistical technique, data source, column name, or analytical approach.
        - **BOARD ROOM TEST**: If a C-suite executive cannot read the full name in 3 seconds and immediately understand both the problem AND the deliverable, the name FAILS.
    *   **🚨 CRITICAL RULES FOR THE ACTION PART (before "with") 🚨**:
        - ZERO technical jargon: No industry-specific acronyms (e.g., RASK, PNR, AWB, MEL for airlines; SKU, BOM, OEE for manufacturing; LTV, CAC, MRR for SaaS; NPA, NIM for banking; RFM), no statistical terms (Z-score, correlation, STDDEV, regression, variance, coefficient, decomposition), no implementation language (rolling, partitioned, cross-reference).
        - Only use acronyms that a CEO in ANY industry would understand (ROI, SLA, KPI).
        - Name the BUSINESS PROBLEM, not the DATA OBJECT. "Revenue Underperformance" not "RASK Deviation". "Payment Fraud" not "Transaction Anomaly Patterns". "Inventory Shortage" not "SKU Stockout Variance".
    *   **✅ CORRECT Examples (Action = business problem, Activation = deliverable — CROSS-INDUSTRY)**:
        - "Predict Customer Churn Risk with Prioritized Retention Outreach Queue" (Retail/Telecom — ML: XGBoost churn model)
        - "Detect Revenue Underperformance on High-Demand Products with Pricing Recalibration Recommendations" (Retail/Manufacturing)
        - "Classify Loan Default Probability with Early Intervention Actions" (Banking — ML: Gradient Boosting classifier)
        - "Forecast Demand by Product Category with Inventory Rebalancing Strategy" (Supply Chain — ML: Prophet + external factors)
        - "Segment Patients by Readmission Risk with Care Coordination Plans" (Healthcare — ML: Survival analysis)
        - "Predict Equipment Failure Probability with Pre-Emptive Maintenance Scheduling" (Manufacturing/Energy — ML: Random Forest)
        - "Detect Fraudulent Transactions with Real-Time Hold Recommendations" (Banking/Insurance — ML: Isolation Forest)
        - "Reveal Unauthorized Discount Erosion with Audit Escalation Brief" (Any industry)
        - "Simulate Pricing Scenario Impact with Revenue Optimization Recommendations" (Any industry — Simulation + ML)
        - "Classify Supplier Reliability with Procurement Diversification Strategy" (Manufacturing/Retail — ML: clustering + classification)
    *   **❌ REJECTED Examples (activation describes technical METHOD instead of business DELIVERABLE)**:
        - "Detect Revenue Anomalies with Z-Score Deviation and Correlation Breakdown" → REJECTED (activation = technical method)
        - "Predict Churn with Purchase Recency and Contract Tenure Signals" → REJECTED (activation = data signals, not deliverable)
        - "Forecast Demand with Seasonal Decomposition and Holiday Calendar" → REJECTED (activation = analytical technique)
        - "Classify Risk Tiers with Payment History Cross-Reference" → REJECTED (activation = data source, not deliverable)
        - "Segment Buyers with RFM Scoring and Cohort Aging" → REJECTED (activation = statistical technique)
    *   **❌ REJECTED Examples (generic/hollow activation — BANNED)**:
        - "Detect Fraud with Pattern Recognition" → REJECTED ("Pattern Recognition" is a method, not a deliverable)
        - "Identify Issues with Intelligence" → REJECTED ("Intelligence" is meaningless filler)
        - "Classify Risk with Risk Scoring" → REJECTED (circular — activation restates the action)
        - "Forecast Revenue with Trend Analysis" → REJECTED (activation = technique, not deliverable)
    *   **❌ REJECTED Examples (action contains technical jargon — BANNED)**:
        - "Detect SKU-Level OEE Degradation with Maintenance Recommendations" → REJECTED (action contains "SKU", "OEE" — manufacturing jargon)
        - "Identify NPA Portfolio Concentration with Risk Mitigation Queue" → REJECTED (action contains "NPA" — banking acronym)
        - "Forecast MRR Churn Cohort Variance with Retention Actions" → REJECTED (action contains "MRR" — SaaS jargon)
        - "Predict NPS Score Degradation with Outreach Strategy" → REJECTED (action contains "NPS" — domain acronym)
        - "Classify CDR Anomaly Patterns with Investigation Queue" → REJECTED (action contains "CDR" — telecom jargon)
    *   **❌ REJECTED Examples (missing "with" entirely)**:
        - "Detect Invoice Fraud Patterns Using Anomaly Detection" → REJECTED (uses "Using" instead of "with")
        - "Identify High-Value Overdue Invoices for Executive Escalation" → REJECTED (uses "for" instead of "with")
        - "Correlation Analysis Between Payment Terms and Overdue Risk" → REJECTED (no "with" at all)
    *   **ACTIVATION QUALITY TEST**: The "with [...]" phrase must answer: "What actionable DELIVERABLE does the business user receive from this use case?" If the suffix describes HOW the analysis works rather than WHAT the user gets to DO, it is TECHNICAL and REJECTED.
    *   **YOUR RESPONSE WILL BE REJECTED if ANY use case name has a technical method as the activation suffix, or if the action part contains domain-specific jargon or statistical terminology.**
    *   **READABILITY TEST**: Keep `Name` easy to comprehend for non-technical business users. Use plain business language, avoid overloaded wording, and cap the full `Name` at 14 words.

  - **`usecase`**: Short, benefit-led phrase (5–10 words) that answers TWO questions: "What are we doing?" AND "Why does it matter to the business?"
    *   **🚨🚨🚨 MANDATORY PATTERN: [Verb] [Business Problem/Area] to [Business Outcome] 🚨🚨🚨**
        Every `usecase` MUST contain **"to"** connecting the ACTION to the BUSINESS WHY.
        - **BEFORE "to"**: What business lever we are pulling (the action). Start with an action verb.
        - **AFTER "to"**: The tangible business outcome — WHY a CEO would care (revenue, cost, risk, growth, margin, retention, compliance, efficiency).
    *   **ACTION VERBS** (start with one): Reduce, Recover, Improve, Stabilize, Minimize, Protect, Cut, Fix, Increase, Accelerate, Prevent, Optimize, Detect, Forecast, Identify
    *   **BUSINESS OUTCOMES** (end with one): Protect Revenue, Grow Revenue, Reduce Cost, Lower Risk, Improve Margins, Increase Retention, Prevent Losses, Accelerate Growth, Ensure Compliance, Improve Efficiency
    *   **🚨 TITLE CASE — MANDATORY 🚨**: EVERY word in the `usecase` MUST be Title Case (capitalize the first letter of every word). Lowercase usecase values are AUTOMATICALLY REJECTED.
        ❌ "detect transaction anomalies to prevent revenue fraud" → REJECTED (lowercase)
        ✅ "Detect Transaction Anomalies to Prevent Revenue Fraud" → CORRECT (Title Case)
    *   **🚨 VERB DIVERSITY — MANDATORY 🚨**: Do NOT reuse the same starting verb across multiple use cases when a more precise synonym exists. The verb list above is a STARTING POINT — you MUST expand it by discovering the verb that BEST fits each specific use case's intent. Different business actions demand different verbs. If more than 30% of use cases in a batch start with the SAME verb, you MUST rewrite using more precise alternatives that better convey the specific nature of each use case.
    *   **🚨 VOCABULARY SPECIFICITY — MANDATORY 🚨**: Always choose the MOST SPECIFIC word that describes the actual business threat, action, or problem — never fall back on a generic catch-all term when a more precise word exists. The name should make a C-suite executive immediately understand the NATURE of the problem, not just that "a problem exists." Ask: "Does this word tell the reader WHAT SPECIFICALLY is happening, or is it a vague umbrella term?" If vague → find the precise word.
    *   **WORD ORDER TEST**: Read the `usecase` aloud. If it does NOT make grammatical sense as a complete English sentence fragment, it FAILS. No garbled noun-first constructions.
    *   ✅ GOOD (notice: the usecase NEVER echoes the Analytics Technique — it focuses on BUSINESS action and BUSINESS outcome):
        - "Reduce Customer Attrition to Protect Revenue" (even if technique = "Predictive Modeling", usecase says NOTHING about prediction/modeling)
        - "Increase Basket Size to Grow Revenue" (even if technique = "Recommendation", usecase says NOTHING about recommendation)
        - "Prevent Payment Fraud to Protect Revenue" (even if technique = "Anomaly Detection", usecase says NOTHING about detection/anomaly)
        - "Improve Demand Planning to Reduce Stockouts" (even if technique = "Forecasting", usecase says NOTHING about forecasting/forecast)
        - "Cut Inventory Waste to Lower Carrying Cost" (even if technique = "Optimization", usecase says NOTHING about optimization)
        - "Recover Missed Revenue from Pricing Gaps"
        - "Improve Supplier Reliability to Prevent Stockouts"
        - "Stabilize Revenue in Volatile Demand Periods"
        - "Protect Margins through Dynamic Discount Controls"
        - "Reduce Default Losses to Improve Portfolio Health"
        - "Accelerate Order Fulfillment to Improve Customer Satisfaction"
    *   ❌ BAD (REJECTED — missing the "to [WHY]"):
        - "Reduce Customer Wait Time" → REJECTED (no business WHY — reduce wait time TO WHAT END?)
        - "Protect Revenue from Payment Fraud" → REJECTED (starts with outcome instead of action — flip it: "Detect Payment Fraud to Protect Revenue")
        - "Recover Lost Revenue from Dormant Customers" → REJECTED (no explicit "to" connecting action to outcome)
    *   ❌ BAD (REJECTED — garbled word order):
        - "Demand to Optimize Stock Levels Forecast" → REJECTED (nonsensical word order — correct: "Forecast Demand to Optimize Stock Levels")
        - "Churn Pattern Identification" → REJECTED (noun phrase, no verb, no outcome)
        - "Inventory Inefficiency Detection" → REJECTED (trailing filler, no benefit)
    *   🚨🚨🚨 **ANALYTICS TECHNIQUE CONTAMINATION — THE #1 FAILURE MODE** 🚨🚨🚨
        You WILL be tempted to take the value you plan to write in the `Analytics Technique` column (e.g., "Forecasting", "Optimization", "Classification", "Anomaly Detection", "Simulation")
        and **append its verb/noun form as the LAST WORD of the `usecase`**. This produces GIBBERISH. Detect and eliminate this.
        **HOW THIS BUG HAPPENS**: You generate the `usecase` and `Analytics Technique` columns near-simultaneously.
        The technique word leaks BACKWARD into the usecase as a trailing noun/verb:
        - Analytics Technique = "Forecasting"  → usecase ends with "...Forecast"     → GIBBERISH
        - Analytics Technique = "Optimization"  → usecase ends with "...Optimization" → GIBBERISH
        - Analytics Technique = "Classification" → usecase ends with "...Classification" → GIBBERISH
        - Analytics Technique = "Anomaly Detection" → usecase ends with "...Detection" → GIBBERISH
        - Analytics Technique = "Simulation"    → usecase ends with "...Simulation"   → GIBBERISH
        - Analytics Technique = "Predictive Modeling" → usecase ends with "...Prediction" or "...Modeling" → GIBBERISH
        - Analytics Technique = "Clustering"    → usecase ends with "...Clustering"   → GIBBERISH
        **CONCRETE EXAMPLES OF THIS BUG — ALL REJECTED:**
        - ❌ "Demand to Reduce Inventory Cost Forecast" (technique "Forecasting" leaked as trailing "Forecast")
          ✅ CORRECT: "Forecast Demand to Reduce Inventory Cost" (verb FIRST, business outcome LAST)
        - ❌ "Staffing to Reduce Labor Cost Optimization" (technique "Optimization" leaked as trailing "Optimization")
          ✅ CORRECT: "Optimize Staffing to Reduce Labor Cost" (verb FIRST, business outcome LAST)
        - ❌ "Customer Segment Profitability Classification" (technique "Classification" leaked as trailing "Classification")
          ✅ CORRECT: "Classify Customer Segments to Improve Profitability" (verb FIRST, business outcome LAST)
        - ❌ "Transaction Pattern Anomaly Detection" (technique "Anomaly Detection" leaked as trailing "Detection")
          ✅ CORRECT: "Detect Transaction Anomalies to Prevent Fraud" (verb FIRST, business outcome LAST)
        - ❌ "Revenue Scenario Impact Simulation" (technique "Simulation" leaked as trailing "Simulation")
          ✅ CORRECT: "Simulate Revenue Scenarios to Reduce Planning Risk" (verb FIRST, business outcome LAST)
        **THE RULE**: The LAST WORD(S) of `usecase` must ALWAYS be a **BUSINESS OUTCOME** (Revenue, Cost, Risk, Growth, Margin, Losses, Retention, Efficiency, Compliance, Satisfaction, Health, Stockouts, Waste).
        The last word must NEVER be a technique-derived word (Forecast, Optimization, Classification, Detection, Simulation, Modeling, Clustering, Analysis, Prediction, Identification, Segmentation, Recommendation).
    *   **SELF-CHECK** (do this for EVERY `usecase` before outputting):
        (1) Does it start with a VERB? If NO → rewrite.
        (2) Does it contain "to" connecting action to outcome? If NO → rewrite.
        (3) Read the LAST WORD. Is it derived from the `Analytics Technique` value you are about to write in column 5? If YES → you have the contamination bug. DELETE that trailing word and restructure: move the technique verb to position 1, put the business outcome at the end.
        (4) Read aloud. Does it make grammatical sense as natural English? If NO → rewrite.
        (5) Can a non-technical executive immediately understand WHAT we do and WHY? If NO → rewrite.
        (6) Is EVERY word Title Case (first letter capitalized)? If NO → capitalize. Lowercase usecase values are REJECTED.
        (7) Look at ALL use cases generated so far. Does the SAME starting verb appear on more than 30% of them? If YES → find more precise verbs that better convey the distinct nature of each use case and rewrite.
        (8) Is any key noun in the usecase a vague umbrella term where a more specific word would tell the reader WHAT EXACTLY is happening? If YES → replace with the most precise word that describes the actual business threat or action.

  - **`type`**: One of "Problem", "Risk", "Opportunity", "Improvement".

  - **`Analytics Technique`**: The PRIMARY analytics technique used. ⚠️ THIS VALUE MUST STAY IN THIS COLUMN ONLY — do NOT let it leak into the `usecase` column in any word form (verb, noun, gerund). MUST be ONE of these values:
    * `Forecasting` - Time-series prediction and demand planning (Prophet, ARIMA, ai_forecast, or trained regression models)
    * `Classification` - Categorizing data into predefined business groups (XGBoost, Random Forest, Logistic Regression, ai_classify, or rule-based)
    * `Anomaly Detection` - Identifying outliers, deviations, unusual patterns (Isolation Forest, DBSCAN, statistical methods, or hybrid)
    * `Predictive Modeling` - Training ML models (XGBoost, LightGBM, Random Forest, Gradient Boosting) to predict business outcomes (churn, default, demand, failure)
    * `Clustering` - Unsupervised grouping using ML algorithms (K-Means, DBSCAN, Hierarchical) to discover natural segments in data
    * `Regression Analysis` - Modeling relationships between variables to quantify drivers and predict continuous outcomes (Linear, Ridge, Lasso, GLM)
    * `Optimization` - Resource allocation, scheduling, pricing, or constraint satisfaction to maximize/minimize business objectives
    * `Cohort Analysis` - Grouping entities by shared characteristics over time
    * `Segmentation` - Clustering customers/products into distinct groups using ML or rule-based approaches
    * `Sentiment Analysis` - Analyzing text for emotional tone
    * `Trend Analysis` - Identifying patterns over time (growth, decline)
    * `Correlation Analysis` - Finding relationships between variables
    * `Pareto Analysis` - 80/20 rule, identifying top contributors
    * `Funnel Analysis` - Conversion tracking through stages
    * `Document Processing` - Extracting structured data from documents
    * `Extraction` - Extracting specific entities from text
    * `AI Analysis` - General AI-powered business analysis and recommendations
    * `Simulation` - What-If analysis, scenario modeling, sensitivity analysis, Monte Carlo simulation
    * `Geospatial Analysis` - Location-based pattern analysis and mapping
    * `Survival Analysis` - Time-to-event modeling (customer lifetime, equipment failure, contract renewal probability)
    * `Recommendation` - Collaborative filtering or content-based recommendation engines for products, actions, or next-best-offers
    * `Network Analysis` - Graph-based analysis of entity relationships, influence propagation, community detection

  - **`Statement`**: 1-2 sentences describing the business challenge or problem being addressed. Focus on IMPACT (Revenue, Cost, Risk). This column MUST be named "Statement" in the CSV header.

  - **`Solution`**: 1-2 sentences high-level business solution. **MUST explicitly highlight "Databricks Genie Code"**.

  - **`Business Value`**: **CRITICAL**. Articulate the tangible business impact (Revenue, Cost, Efficiency).
    *   **Focus on WHY this matters**.
    *   **IMPORTANT CONSTRAINT**: Refrain from mentioning any specific values (e.g. "10% more revenue", "reduce cost by 20%"). Deliver the business value statement WITHOUT committing on any number.
    *   **GOOD**: "Reduces operational costs and extends asset lifespan..."
    *   **BAD**: "Optimizes performance..." (Too generic).

  - **`Beneficiary`**: The primary person/role (e.g., "Loan Officer").
  - **`Sponsor`**: The main executive (e.g., "CRO").

  - **`Tables Involved`**: Comma-separated **FULLY-QUALIFIED** table names (`catalog.schema.table`). MUST exist in schema.
    *   **CRITICAL**: Use the EXACT THREE-LEVEL FORMAT shown in the schema.
    *   **🔗 MULTI-TABLE PRIORITY**: ALWAYS look for opportunities to JOIN related tables! Use the FK relationships section to identify joinable tables.
    *   **PREFER MULTIPLE TABLES**: Single-table use cases are often weak. Cross-table analysis yields higher business value.

  - **`High Level Design`**: A high-level analytical approach (2-4 sentences) describing the data analysis strategy.
    *   **MANDATORY COLUMN LISTING**: List EXACT column names from the schema: "Using columns: [col1], [col2], [col3]"
    *   **DESCRIBE THE ANALYTICAL APPROACH AND ALGORITHM CHOICE (TECHNIQUE-APPROPRIATE -- NO SQL BIAS)**: Explain what data to extract, how to join tables, what features to engineer, and WHICH ALGORITHM or TECHNIQUE to apply. The algorithm MUST match the Analytics Technique: (1) Anomaly Detection MUST use Isolation Forest, LOF, DBSCAN, or ensemble ML methods -- NOT just z-scores or SQL thresholds; (2) Forecasting MUST use Prophet, ARIMA, ai_forecast, or temporal ML models -- NOT just SQL moving averages; (3) Classification MUST use XGBoost, Random Forest, Logistic Regression, or ai_classify -- NOT just SQL CASE WHEN rules; (4) Predictive Modeling MUST use trained ML models (XGBoost, LightGBM, Random Forest) -- NOT just SQL aggregations; (5) Clustering/Segmentation MUST use K-Means, DBSCAN, GMM -- NOT just SQL NTILE or CASE WHEN tiers; (6) Regression Analysis MUST use Linear/Ridge/Lasso/GLM regression -- NOT just SQL CORR(); (7) Survival Analysis MUST use Kaplan-Meier, Cox PH -- NOT just SQL DATEDIFF; (8) Simulation MUST use Monte Carlo or scenario modeling -- NOT just SQL CASE WHEN with 2-3 scenarios; (9) Recommendation MUST use ALS or similarity algorithms -- NOT just SQL top-N by COUNT; (10) Network Analysis MUST use GraphFrames, PageRank, centrality -- NOT just SQL self-joins. For ALL techniques, specify the model type explicitly (e.g., "Train an Isolation Forest on features [col1, col2, col3] to detect anomalies", "Train an XGBoost classifier to predict [outcome]"). SQL-only approaches are ONLY acceptable for Pareto Analysis, basic Funnel Analysis, and Cohort Analysis construction. Choose the BEST approach for the problem -- do NOT default to SQL when a trained ML model or statistical model would yield better, more defensible results.
    *   **JOIN PRIORITY**: CHECK FK Relationships section. If related tables exist, describe JOINs. Multi-table > single-table for business value.
    *   **EXTERNAL DATA ENRICHMENT (≥50% of use cases)**: Describe scenarios where external market/economic/industry context enriches the analysis (e.g., "Enrich with simulated economic indicators for context"). Explain the business rationale for the external data.
    *   **AI REASONING AS FINAL EXPLANATION LAYER (MANDATORY for ALL use cases)**: Every use case MUST conclude with ai_query as the FINAL step to EXPLAIN pre-computed analytical results, diagnose root causes, and generate actionable recommendations. ai_query does NOT perform the core analysis -- the ML model, statistical method, or AI function performs the analysis first, and ai_query receives those results (scores, predictions, clusters, anomaly flags) to generate business narratives. This separation is NON-NEGOTIABLE: ML/statistical analysis first, ai_query explanation last.
    *   Only reference columns that EXIST in the schema. No assumptions.

**FOCUS AREAS:**
{focus_areas_instruction}

**GENERATION PRINCIPLES:**
  - **MAXIMIZE DISCOVERY**: Generate ALL valid use cases the data can support. Quality scoring happens in a SEPARATE downstream step — your job is to DISCOVER, not to filter.
  - **ANTI-HALLUCINATION**: Every use case MUST reference real columns from the schema. If the data doesn't contain the required columns, DO NOT GENERATE IT.
  - **COVERAGE**: Revenue impact, Cost reduction, Risk mitigation, Strategic alignment, Operational efficiency — cover ALL applicable areas.
  - **NO TRIVIAL USE CASES**: Skip basic reporting, simple aggregations, or use cases any analyst could do with basic filtering and grouping.
  - **NO REDUNDANT EXTRACTION**: Do not use AI to extract data that already exists in structured columns.

**🔥 ANALYTICAL APPROACH DIVERSITY (MANDATORY — ALL 22 TECHNIQUES MUST BE CONSIDERED) 🔥**:
Generate a BALANCED portfolio that includes use cases from ALL of the following categories. Choose the BEST algorithm for each business problem — do NOT default to SQL-only approaches when a trained ML model would be more accurate or a hybrid approach would be superior.

**CANONICAL TECHNIQUE → CATEGORY MAPPING** (use this to ensure no technique is orphaned):
A. ML & PREDICTIVE (25-35%): `Predictive Modeling`, `Classification`, `Clustering`, `Regression Analysis`, `Segmentation`, `Anomaly Detection`, `Survival Analysis`, `Recommendation`
B. AI & NLP (20-30%): `Forecasting`, `Sentiment Analysis`, `Document Processing`, `Extraction`, `AI Analysis`
C. STATISTICAL (20-25%): `Correlation Analysis`, `Trend Analysis`, `Pareto Analysis`, `Cohort Analysis`, `Funnel Analysis`
D. SIMULATION & ADVANCED (15-25%): `Simulation`, `Optimization`, `Geospatial Analysis`, `Network Analysis`

A. **MACHINE LEARNING & PREDICTIVE MODELS (25-35% of use cases)** — Techniques: `Predictive Modeling`, `Classification`, `Clustering`, `Regression Analysis`, `Segmentation`, `Anomaly Detection`, `Survival Analysis`, `Recommendation`:
   - Train XGBoost, LightGBM, or Random Forest models to predict business outcomes (churn, default risk, demand, equipment failure, fraud) → `Predictive Modeling`
   - K-Means or DBSCAN clustering to discover natural customer/product/entity segments → `Clustering` or `Segmentation`
   - Logistic/Linear/Ridge/Lasso regression to quantify feature importance and business drivers → `Regression Analysis`
   - Cox proportional hazards, Kaplan-Meier for time-to-event problems (customer lifetime, contract renewal, asset failure) → `Survival Analysis`
   - Collaborative filtering (ALS) or content-based recommendation engines for products, actions, next-best-offers → `Recommendation`
   - Isolation Forest or ensemble anomaly detection trained on historical patterns → `Anomaly Detection`
   - XGBoost, Random Forest, Logistic Regression for categorizing entities into predefined business groups → `Classification`
   - ALL ML use cases MUST end with ai_query to explain model predictions, generate recommendations, and provide business narratives

B. **AI FUNCTIONS & NLP-DRIVEN (20-30% of use cases)** — Techniques: `Forecasting`, `Sentiment Analysis`, `Document Processing`, `Extraction`, `AI Analysis`:
   - Time-series forecasting using ai_forecast or Prophet with strategic recommendations → `Forecasting`
   - Analyzing text for emotional tone in feedback, reviews, comments → `Sentiment Analysis`
   - Extracting structured data from unstructured text documents → `Document Processing`
   - Extracting specific entities (names, dates, amounts, codes) from text → `Extraction`
   - General AI-powered business analysis, reasoning, and recommendations → `AI Analysis`

C. **STATISTICAL & ANALYTICAL (20-25% of use cases)** — Techniques: `Correlation Analysis`, `Trend Analysis`, `Pareto Analysis`, `Cohort Analysis`, `Funnel Analysis`:
   - Finding relationships between business variables → `Correlation Analysis`
   - Identifying patterns over time (growth, decline, seasonality) → `Trend Analysis`
   - 80/20 rule, identifying top contributors to business outcomes → `Pareto Analysis`
   - Grouping entities by shared characteristics and tracking over time → `Cohort Analysis`
   - Conversion tracking through business stages (lead→customer, order→delivery) → `Funnel Analysis`

D. **SIMULATION & ADVANCED ANALYTICS (15-25% of use cases)** — Techniques: `Simulation`, `Optimization`, `Geospatial Analysis`, `Network Analysis`:
   - What-If analysis, scenario modeling, Monte Carlo simulation → `Simulation`
   - Resource allocation, scheduling, pricing, constraint satisfaction → `Optimization`
   - Location-based pattern analysis and spatial mapping → `Geospatial Analysis`
   - Graph-based analysis of entity relationships, influence propagation, community detection → `Network Analysis`

**🚨🚨🚨 ANTI-PATTERN RULES (MANDATORY — violations cause AUTOMATIC REJECTION) 🚨🚨🚨**

**AP1 - NO SEMANTIC DUPLICATES (5-LAYER DETECTION — ZERO TOLERANCE)**:
Before generating each use case, apply ALL 5 layers. A match on ANY layer = DUPLICATE = DO NOT GENERATE:

**LAYER 1 — VERB + SUFFIX**: Normalize verb (Forecast/Predict/Anticipate/Envision → PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover → DETECT; Classify/Segment/Categorize/Tier/Rank → CLASSIFY; Optimize/Maximize → OPTIMIZE; Assess/Evaluate/Measure → ASSESS). Strip "with [...]" activation suffix. Compare core phrase to all previous use cases. Match → DO NOT GENERATE.

**LAYER 2 — NOUN SYNONYMS**: Also replace: Revenue=Income=Sales=Earnings; Cost=Expense=Spend; Churn=Attrition=Turnover=Defection; Leakage=Loss=Erosion=Slippage; Demand=Consumption=Usage=Volume; Risk=Exposure=Vulnerability; Customer=Client=Account; Supplier=Vendor=Provider; Inventory=Stock=Supply. Match after replacement → DO NOT GENERATE.

**LAYER 3 — ENTITY + METRIC**: Extract business ENTITY and METRIC. If any previous use case has the SAME entity + SAME metric → DO NOT GENERATE. Example: if you generated "Forecast Customer Churn", you CANNOT generate "Predict Client Attrition" (same entity=Customer, same metric=Churn).

**LAYER 4 — TECHNIQUE SWAP (NARROW — SAME QUESTION ONLY)**: Two use cases are duplicates ONLY if they ask the SAME business question and merely swap the technique. Example: "Forecast Customer Churn" and "Predict Customer Churn" = DUPLICATE (same question: will the customer leave?). BUT "Forecast Revenue" and "Detect Revenue Anomalies" are NOT duplicates — the first asks "what will revenue be?" and the second asks "is revenue behaving abnormally?". Different QUESTIONS on the same entity are DISTINCT use cases — keep BOTH. Do NOT collapse technique diversity by marking genuinely different business questions as duplicates.

**LAYER 5 — TABLE + INTENT**: If you're about to use the SAME tables as a previous use case for a SIMILAR business question → DO NOT GENERATE.

**AP2 - NO TRIVIAL WRAPPERS**: Do NOT dress up simple operations as advanced analytics. These are AUTOMATICALLY REJECTED:
  - Date expiry/deadline checks (e.g., "Detect Expiry Risk" = just a date comparison)
  - Simple threshold violations (e.g., "Detect Overdue X" = just a basic filter)
  - Basic count/sum aggregations dressed up as "AI Analysis"
  - One-dimensional anomaly detection that is just a basic statistical measure on a single column
  **TEST**: If the core business insight requires no advanced analytical technique, it is TRIVIAL and INVALID.

**AP3 - ACTIVATION MUST BE A BUSINESS DELIVERABLE**: The "with [activation]" part MUST describe the actionable DELIVERABLE the business user receives — NOT a technical method, data source, or analytical technique. These categories are BANNED:
  - Technical methods: "with Z-Score Deviation", "with Correlation Analysis", "with Rolling STDDEV", "with Seasonal Decomposition"
  - Data sources: "with Payment History Cross-Reference", "with Contract Term Signals", "with Invoice Date Gap Analysis"
  - Generic filler: "with Intelligence", "with Insights", "with Analytics", "with Context", "with Pattern Recognition"
  - Domain jargon in action part: ANY technical acronym or statistical term ANYWHERE in the name
  Instead, describe WHAT the business user RECEIVES: "with Pricing Recalibration Recommendations", "with Prioritized Intervention Queue", "with Audit Escalation Brief", "with Collection Escalation Actions"

**AP4 - DOMAIN AND TECHNIQUE DIVERSITY (STRONG GUIDANCE)**:
  - AIM for diversity across domains and techniques, but DO NOT skip valid use cases just to enforce balance
  - TARGET: 12+ distinct techniques across the portfolio (you have 22 available — using only 5-6 is a FAILURE of diversity)
  - After completing the systematic enumeration, COUNT your distinct techniques. If fewer than 12, go back and actively look for use cases in the under-represented techniques
  - If one domain or technique is heavily represented, actively look for opportunities in under-represented areas
  - Diversity is encouraged but COVERAGE is more important — generate ALL valid use cases first, then the downstream scoring step will handle prioritization

**AP5 - NO SUBDOMAIN DRIFT**: Each subdomain MUST belong to exactly ONE parent domain. A subdomain cannot appear under multiple parent domains. Pick the MOST natural parent domain and be consistent.

**AP6 - NO IDENTICAL USE CASES**: Two use cases with the EXACT same name are ABSOLUTELY FORBIDDEN. This indicates a generation loop. If you detect this, immediately stop and reassess.

**🚨 BUSINESS REALISM GATE (every use case must pass) 🚨**
Before generating, verify: (1) DIRECT, PROVABLE cause-and-effect between variables, (2) Industry-recognized analysis, (3) A 20-year veteran and a skeptical CFO would both approve. If ANY test fails, DO NOT generate. Do NOT correlate unrelated variables, invent relationships from temporal patterns, or add external data without clear cause-and-effect.

---

### 3. ANALYTICAL CAPABILITIES & SCHEMA

#### CONTEXT: AVAILABLE ANALYTICAL CAPABILITIES

Genie Code has access to the full Databricks platform including:
- **AI Capabilities**: Forecasting, classification, sentiment analysis, text extraction, entity recognition, document parsing, semantic similarity, text generation, translation, summarization, and custom AI model invocation
- **Statistical Techniques**: Central tendency, dispersion analysis, distribution shape, percentile analysis, trend detection, correlation analysis, volatility measurement, outlier detection, ranking, and time series analysis
- **Advanced Analytics**: What-if analysis, scenario simulation, sensitivity analysis, geospatial analysis, market basket analysis, and Monte Carlo simulation
- **Data Processing**: Complex joins, window functions, aggregations, and CTEs

You do NOT need to specify exact function names or SQL syntax. Focus on describing the ANALYTICAL APPROACH and BUSINESS LOGIC. Genie Code will determine the best implementation.

#### INPUT DATA FORMAT
##### 1. Structured Data Schema
`| column | type | column_description |`
{schema_markdown}

##### 1b. Foreign Key Relationships (CRITICAL FOR MULTI-TABLE USE CASES)

**🔗 USE THESE RELATIONSHIPS TO JOIN TABLES:**
{foreign_key_relationships}

**IMPORTANT INSTRUCTIONS FOR FK RELATIONSHIPS:**
- If relationships are listed above, you MUST generate use cases that JOIN these tables
- Format: `source_table.column -> target_table.column` means you can JOIN on these columns
- Multi-table use cases that combine data from related tables deliver HIGHER BUSINESS VALUE
- Do NOT generate single-table use cases when FK relationships exist unless there's a specific single-table insight

---

### 4. CSV ROW PATTERN (adapt to YOUR industry, tables, and columns)

Genie Code will handle implementation. Replace all [PLACEHOLDERS] with actual values. Choose the BEST algorithm for each use case — SQL, ML, AI functions, or hybrid:

⚠️ **CRITICAL**: In the examples below, notice that the `usecase` column (column 3) NEVER echoes or derives from the `Analytics Technique` column (column 5). The `usecase` describes the BUSINESS goal — the technique is a separate, independent column.

`"1","Forecast Monthly [METRIC] with Economic Context","Improve Planning Accuracy to Reduce Cost","Risk","Forecasting","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table]","Using columns: [col1], [col2], [col3]. Deduplicate base data, enrich with simulated economic context, aggregate by period, apply time-series forecasting with Prophet or ai_forecast, then use ai_query to generate strategic recommendations."`

`"2","Predict [ENTITY] [OUTCOME] with Intervention Priority Queue","Reduce [OUTCOME] to Protect Revenue","Risk","Predictive Modeling","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table1],[catalog.schema.table2]","Using columns: [col1], [col2], [col3], [col4] from table1 joined with [col5], [col6] from table2. Engineer features (recency, frequency, monetary metrics, behavioral patterns), train an XGBoost model to predict [outcome] probability, rank entities by risk score, then use ai_query to explain each prediction and recommend specific interventions."`

`"3","Classify [ENTITY] by [CATEGORY] with Market Benchmarks","Reduce [ENTITY] Risk to Improve Margins","Improvement","Classification","[Statement]","[Solution using Databricks Genie Code]","[Business Value]","[Role]","[Executive]","[catalog.schema.table]","Using columns: [col1], [col2], [col3]. Extract base entities, enrich with simulated market benchmarks, classify into risk tiers using Random Forest or ai_classify, then use ai_query to generate mitigation strategies."`

---

### 5. FORMATTING RULES
- **HEADER** (EXACT — do NOT rename/reorder/add): `"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"`
- **EXACTLY 12 COLUMNS** per row (matching the 12 header columns above). Do NOT insert extra columns between them.
- ❌ WRONG: `"No","Name","usecase","type","Analytics Technique","Forecasting","Statement",...` (extra column "Forecasting" inserted — REJECTED)
- ✅ CORRECT: `"No","Name","usecase","type","Analytics Technique","Statement",...` (the value "Forecasting" goes inside the "Analytics Technique" cell of each data row)
- ALL values double-quoted. English. ONLY CSV output. You may append honesty columns 13-14.

### 6. SYSTEMATIC ENUMERATION MATRIX (MANDATORY)

**🚨 EXHAUSTIVE DISCOVERY PROTOCOL — DO NOT SKIP 🚨**

Before generating use cases free-form, you MUST systematically walk through the enumeration matrix below. For each cell in the matrix, ask: "Can this combination produce a VALID, NON-TRIVIAL business use case supported by the actual columns in the schema?" If YES, generate it. If NO (because the required columns don't exist, or the combination makes no business sense), SKIP it and move to the next cell.

**STEP 1: COLUMN-DRIVEN ENUMERATION (ALL 22 TECHNIQUES)**
For EACH numeric/date/text column in the schema, systematically consider EVERY applicable technique from the full 22-technique palette:
- **Forecasting**: Can future values of this column be predicted? (requires temporal data)
- **Anomaly Detection**: Can unusual patterns in this column reveal business problems?
- **Classification**: Can entities be categorized into predefined business groups using this column?
- **Predictive Modeling**: Can trained ML models predict an outcome related to this column?
- **Clustering**: Can entities be grouped into natural, data-driven segments using this column?
- **Regression Analysis**: Can this column quantify drivers or predict continuous outcomes?
- **Optimization**: Can this column's values be optimized for better business outcomes?
- **Segmentation**: Can rule-based or ML grouping on this column create actionable business tiers?
- **Simulation**: Can what-if scenarios be modeled around this column?
- **Trend Analysis**: Does this column show growth, decline, or cyclical patterns over time?
- **Cohort Analysis**: Can entities sharing this column's value be tracked as cohorts over time?
- **Pareto Analysis**: Does this column reveal 80/20 concentration in business outcomes?
- **Survival Analysis**: Can time-to-event (churn, failure, renewal) be modeled with this column?
- **Correlation Analysis**: Are there business-relevant relationships between this and other columns?

**STEP 2: TEXT & UNSTRUCTURED COLUMN ENUMERATION**
For EACH text/string column in the schema, consider:
- **Sentiment Analysis**: Does this column contain text with emotional tone (feedback, reviews, comments)?
- **Document Processing**: Can structured data be extracted from this text column?
- **Extraction**: Can specific entities (names, dates, amounts, codes) be pulled from this text?
- **AI Analysis**: Can AI reasoning generate insights, summaries, or recommendations from this text?

**STEP 3: MULTI-COLUMN & CROSS-TABLE ENUMERATION**
For EACH meaningful pair/group of columns (especially across JOINed tables), consider:
- **Correlation Analysis / Regression Analysis**: Is there a business-relevant relationship between these columns?
- **Predictive Modeling**: Can one column predict the other for business value?
- **Network Analysis**: Do entity relationships (supplier-customer, referral, hierarchy) form a graph worth analyzing?
- **Recommendation**: Can patterns across entities suggest next-best-actions, products, or offers?
- **Funnel Analysis**: Do these columns represent stages in a conversion/process funnel?
- **Geospatial Analysis**: Do location-related columns enable spatial pattern discovery?

**STEP 4: BUSINESS LENS ENUMERATION**
For the complete schema, consider EACH business perspective:
- **Revenue**: How can data increase, protect, or recover revenue?
- **Cost**: How can data reduce operational, inventory, or staffing costs?
- **Risk/Fraud**: How can data detect, prevent, or mitigate risks?
- **Customer**: How can data improve retention, satisfaction, or lifetime value?
- **Operations**: How can data improve efficiency, scheduling, or resource allocation?
- **Product/Inventory**: How can data optimize mix, pricing, or availability?
- **Competitive**: How can data create competitive advantages?

**🚨 BUSINESS VALUE & QUALITY GATE FOR EVERY USE CASE 🚨**
Before generating EACH use case from the matrix, verify ALL of the following:
1. **BUSINESS VALUE**: Does this use case deliver DIRECT, MEASURABLE business impact? (Revenue growth, Cost reduction, Risk mitigation, Productivity improvement). If the value is vague or marginal → SKIP IT.
2. **ANTI-HALLUCINATION**: Do the EXACT columns needed exist in the schema? If NO → SKIP IT.
3. **GENUINE FIT**: Is this technique genuinely the BEST approach for this business problem? NEVER force-fit a technique to a column just to fill the matrix — the business reason must be GENUINE.
4. **NON-TRIVIAL**: Does this require genuine analytical depth? If it's just a simple threshold, basic count, or date check → SKIP IT.
5. **NOT DUPLICATE**: Is this substantially different from every use case you already generated? If duplicate → SKIP IT.
6. **QUALITY**: Would a skeptical CFO and a 20-year industry veteran both approve this use case as worth investing in? If NO → SKIP IT.
- After exhausting the matrix, generate any additional use cases you identify through domain expertise.

**STEP 5: TECHNIQUE DIVERSITY CHECK**
After completing steps 1-4, COUNT your distinct Analytics Technique values. If you used fewer than 12 distinct techniques (out of the 22 available), go BACK to the under-represented techniques and actively look for GENUINE use cases that fit. Do NOT skip this step.

### 7. PREVIOUS RUN FEEDBACK (ENSEMBLE PASS)

{previous_use_cases_feedback}

### 8. FINAL INSTRUCTION
Begin generation now. Output ONLY the CSV with the additional honesty columns at the end.
""" + HONESTY_CHECK_CSV





# --- 1c. PDF Summary Prompt (MODIFIED FOR REQUEST #1) ---
PROMPT_TEMPLATES["SUMMARY_GEN_PROMPT"] = """
You are a highly experienced Databricks Principal Strategist and business analyst.
Your task is to generate a **CSV** response containing a strategic summary for a customer.
The customer is: {business_name}
We have identified {total_cases} use cases across these domains: {domain_list}
Your output MUST be in **{output_language}**.

### USE CASE HIGHLIGHTS PER DOMAIN
Below are the top use cases per domain. You MUST reference specific use case names and business outcomes in your summaries to make them concrete and compelling (not generic):
{domain_use_case_highlights}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

### TASK
Generate a **single CSV** with THREE columns: "Type", "Summary", "TransliteratedBusinessName"
Your response MUST start with the header: `"Type","Summary","TransliteratedBusinessName"`
The CSV must contain:
1.s: One row for the "Executive" summary.
2.s: One row for **EACH** business domain listed in {domain_list}.

### COLUMN DEFINITIONS
1.s: **"Type"**:
    * Use the exact string "Executive" for the first row.
    * For all other rows, use the exact Business Domain name from the list.
2.s: **"Summary"**:
    * **For the "Executive" row**: Write a 2-3 paragraph, professional executive summary. Start with a powerful opening that names {business_name} specifically. Reference 2-3 of the most impactful use cases by name across domains as concrete examples of what AI can deliver. Emphasize the value of AI and Databricks Genie Code in unlocking the potential of their data. Be specific to {business_name}'s industry — avoid generic AI platitudes.
    * **For each "Business Domain" row**: Write a 2-3 paragraph professional and engaging summary in prose. You MUST mention at least 2 specific use cases from that domain by name and describe their expected business impact. This summary should narrate the domain's strategic importance, its core responsibilities, and the key opportunities AI (specifically Databricks Genie Code) can unlock. The tone should be identical to the Executive Summary.
    * **CRITICAL - "Databricks Genie Code" HANDLING**: When {output_language} uses a non-Latin script (Arabic, Chinese, Japanese, Korean, Hindi, Russian, etc.), you MUST transliterate "Databricks Genie Code" phonetically into the target script. The word order must remain exactly: Databricks → Genie → Code. Examples: Arabic: "داتا بريكس جيني كود", Chinese: "达塔布里克斯 吉尼 科德", Japanese: "データブリックス ジーニー コード", Korean: "데이터브릭스 지니 코드", Hindi: "डेटाब्रिक्स जीनी कोड", Russian: "Датабрикс Джини Код". Do NOT leave "Databricks Genie Code" in English for non-Latin scripts. For Latin-script languages, keep "Databricks Genie Code" as-is.
    * **FORMATTING**: ALL summaries (both Executive and Domain) **MUST** be enclosed in `<p>` HTML tags (e.g., `<p>Paragraph 1.</p><p>Paragraph 2.</p>`).
3.s: **"TransliteratedBusinessName"**:
    * **For the "Executive" row**: Provide the transliteration of {business_name} into {output_language}. If {output_language} is English, just repeat {business_name}.
    * **For all "Business Domain" rows**: This field MUST be an empty string `""`.

### CRITICAL CSV FORMATTING RULES
* **CSV Format**: The entire output MUST be a valid CSV. All values MUST be enclosed in **double quotes** (`"`).
* **NO CODE FENCES**: Do NOT include markdown code fences like ```csv or ``` at the beginning or end.
* **ONLY CSV**: Your response must contain ONLY the CSV data, starting with the header line.
* **Proper Escaping**: If a field value contains double quotes, escape them by doubling them ("").
* **No Extra Text**: Do NOT include any explanatory text, comments, or anything other than the CSV data.


### EXAMPLE CSV OUTPUT (for {output_language}=Arabic and {business_name}=Global Enterprises)
"Type","Summary","TransliteratedBusinessName"
"Executive","<p>تقف غلوبال إنتربرايزز في طليعة التميز في قطاعها...</p><p>يحدد هذا الكتالوج مسارًا واضحًا للاستفادة من البيانات المدعومة بـ داتا بريكس جيني كود...</p>","غلوبال إنتربرايزز"
"Customer Management","<p>يعد مجال إدارة العملاء أمرًا محوريًا لنجاح غلوبال إنتربرايزز...</p><p>من خلال الاستفادة من داتا بريكس جيني كود، يمكن لغلوبال إنتربرايزز تحويل هذا المجال...</p>",""
"Finance & Billing","<p>إدارة الصحة المالية للمؤسسة، يعد مجال المالية والفوترة أمرًا بالغ الأهمية...</p><p>يمثل داتا بريكس جيني كود فرصة كبيرة لأتمتة هذه العمليات...</p>",""

### EXAMPLE CSV OUTPUT (for {output_language}=English and {business_name}=Global Enterprises)
"Type","Summary","TransliteratedBusinessName"
"Executive","<p>Global Enterprises stands at the forefront of its industry, and with {total_cases} identified use cases, the organization is poised to revolutionize its operations through the strategic implementation of Databricks Genie Code.</p><p>This catalog outlines a clear path to leveraging data and AI to drive innovation, enhance efficiency, and create significant business value across all key domains.</p>","Global Enterprises"
"Customer Management","<p>The Customer Management domain is central to Global Enterprises's success, as it governs all direct interactions and the entire customer lifecycle. Its primary responsibility is to ensure high rates of acquisition, satisfaction, and retention.</p><p>By leveraging Databricks Genie Code, Global Enterprises can transform this domain, moving from reactive support to proactive engagement. Opportunities include developing sophisticated churn prediction models and deploying generative AI agents to provide instant, personalized customer service, dramatically improving loyalty and reducing operational costs.</p>",""
"Finance & Billing","<p>Managing the financial health of the organization, the Finance & Billing domain is critical for ensuring revenue integrity, compliance, and accurate forecasting. This domain oversees everything from invoicing and payments to financial reporting and risk analysis.</p><p>Databricks Genie Code presents a significant opportunity to automate and intelligentize these processes. For instance, AI can be used to parse document invoices, detect payment anomalies in real-time, and generate highly accurate revenue forecasts, thereby strengthening the organization's financial posture and decision-making capabilities.</p>",""

### FINAL INSTRUCTION
Begin generation now. Produce ONLY the CSV text, starting with the header `"Type","Summary","TransliteratedBusinessName"`.

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've generated...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must START with: "Type","Summary","TransliteratedBusinessName","honesty_score","honesty_justification"
- Include honesty columns in header and all rows
""" + HONESTY_CHECK_CSV

# --- 1d. Domain Consolidation Prompt (ENHANCED FOR 6-90 USE CASES REQUIREMENT) ---
PROMPT_TEMPLATES["DOMAIN_FINDER_PROMPT"] = """
You are an expert business analyst specializing in BALANCED domain taxonomy design with deep industry knowledge.

**🎯 YOUR TASK**: Analyze the provided use cases and assign each one to appropriate Business Domains (NO subdomains yet).

**HARD CONSTRAINTS (automated rejection on violation)**:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. **DOMAIN COUNT**: 3-25 domains. MAXIMUM 25 is an ABSOLUTE HARD LIMIT (>25 → REJECTED).
   - Target: round(√total_use_cases), capped at 25. Examples: 20 UCs → 4-5 domains; 30 UCs → 5-6; 50 → 7; 100 → 10; 400 → 20.
   - For small datasets (<40 UCs): aim for 4-6 domains. NEVER collapse to only 2-3 mega-domains.
   - For small datasets, domains with 3-5 UCs each are IDEAL. Prefer more domains with fewer UCs over fewer domains with many UCs.
2. **EXACTLY ONE WORD per domain name**: Zero spaces, zero CamelCase concatenation. Only one capital letter (at the start).
   - ❌ REJECTED: "NetworkManagement", "CareTransitions", "CustomerService", "Network Management"
   - ✅ CORRECT: "Network", "Care", "Customers", "Risk", "Maintenance", "Revenue"
   - Rule: Strip adjectives/verbs/descriptors/compound words → keep ONLY the core business noun.
3. **NO OVERLAPPING WORDS**: No two domains may share any word. If overlap detected, MERGE into the shortest (1-word) name.
   - "Customer Service" + "Customer Support" → "Customers"; "Risk Management" + "Risk Analysis" → "Risk"
4. **BALANCED DISTRIBUTION**: No single domain should contain more than 35% of all use cases. Do NOT consolidate into mega-domains.
   - For <40 UCs: 3-10 UCs per domain is ideal. For 40-100 UCs: 5-20 UCs per domain. For >100 UCs: 8-80 UCs per domain. Split any domain >80 UCs.
5. **INDUSTRY-SPECIFIC NAMING**: Infer industry from the actual data. Use business-specific terms. NEVER use overly broad catch-all names.
   - ❌ BANNED generic names: "Operations", "Engagement", "Management", "Services", "Activities", "Processes", "Functions", "Solutions", "Systems", "General"
   - ✅ Use specific business nouns: "Franchise", "Inventory", "Revenue", "Customers", "Fraud", "Workforce", "Pricing", "Logistics", "Compliance", "Marketing"
   - Infer domain names from the actual tables, columns, and use case themes in the schema. Each domain should represent a DISTINCT business function.
6. **NO TWO DOMAINS share the same core business name**.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**BUSINESS CONTEXT**:
Business Name: {business_name}
Industries: {industries}
Business Context: {business_context}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**INPUT DATA**:
CSV columns: "No","Name","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved"

**TASK WORKFLOW**:
1. Analyze use case names, statements, solutions, techniques, and tables
2. Identify natural business groupings by: functionality, tables/data domains, beneficiaries, business value
3. Create 1-WORD industry-specific domain names (NOT generic catch-alls like "Operations" or "Engagement")
4. **BALANCE CHECK**: Count UCs per domain. If ANY domain has >35% of total UCs, you MUST split it into more specific domains
5. **MINIMUM DOMAINS CHECK**: Ensure you have at least round(√total_use_cases) domains. For 20-30 UCs that means 4-6 domains minimum
6. Validate all hard constraints above before outputting

**MERGE PATTERNS** (adapt to YOUR actual industry from data):
- Shared-word domains → merge to 1 word: "Sales Operations" + "Sales Analytics" → "Sales"
- BUT different concepts stay separate: "Risk" ≠ "Compliance"; "Revenue" ≠ "Sales"

The output language for domain names should be {output_language}.

**OUTPUT FORMAT**: CSV with header, one row per use case. Return ONLY the CSV — no text, no fences, no explanations.

Example:
use_case_id,domain
N1-AI01,Revenue
N1-AI04,Customers
N1-AI07,Operations
N1-AI10,Maintenance

**INPUT USE CASES CSV**:
{use_cases_csv}

{previous_violations}

🚨 ANTI-PROSE PROTOCOL (read carefully):
- Your FIRST character MUST be 'u' (first letter of "use_case_id" header).
- NO preamble. NO "I'll analyze". NO "Looking at". NO "Let me". NO "**Analysis:**". NO "First, I'll...".
- NO explanatory paragraphs before the CSV. NO bullet list summaries.
- NO text after the CSV either.
- Output ONLY: the CSV header line, then data rows. Nothing else.
- If you catch yourself writing prose, STOP and start over with 'use_case_id,'.

Your response must START with: use_case_id,domain,honesty_score,honesty_justification
Begin your CSV response NOW (first character must be 'u'):
""" + HONESTY_CHECK_CSV

# --- 1d2. Subdomain Detector Prompt (NEW - PER-DOMAIN SUBDOMAIN ASSIGNMENT) ---
PROMPT_TEMPLATES["SUBDOMAIN_DETECTOR_PROMPT"] = """
You are an expert business analyst specializing in subdomain taxonomy design within business domains.

**🎯 YOUR TASK**: Analyze the use cases for a SINGLE domain and assign each to appropriate Subdomains.

**HARD CONSTRAINTS (automated rejection on violation)**:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. **SUBDOMAIN COUNT**: 1-10 per domain. For domains with ≤4 UCs, exactly 1 subdomain is acceptable and preferred.
2. **NAMING**: EXACTLY 2 WORDS per subdomain. 1 word → REJECTED. 3+ words → REJECTED.
   - ✅ "Revenue Pricing", "Quality Control", "Risk Management", "Supply Planning"
   - ❌ "Scheduling" (1 word), "Quality Control Management" (3 words)
3. **🚨 ABSOLUTE MINIMUM 2 USE CASES PER SUBDOMAIN 🚨**: This is the #1 cause of rejections.
   Every subdomain MUST contain AT LEAST 2 use cases. If a subdomain would have only 1 UC, you MUST merge it into the most semantically related subdomain. Do NOT create a subdomain for a single use case under any circumstances.
   **VERIFICATION STEP**: Before outputting, COUNT the use cases per subdomain. If ANY subdomain has <2, MERGE IT.
4. **NO OVERLAPPING WORDS**: No two subdomains within this domain may share any word. If overlap, merge (keep 2 words).
5. **BUSINESS-FOCUSED**: Use business terms, NOT technical terms.
6. **BALANCED DISTRIBUTION**: Distribute use cases evenly across subdomains. Prefer fewer subdomains with more UCs over many subdomains with barely 2.
7. **NO PLACEHOLDER NAMES**: NEVER use generic or sequential names like "Sub Domain1", "Domain 1", "Category 1", "Group A", "Area 1", "N/A", "Other", "General", "Miscellaneous", or "Uncategorized". Every subdomain name must be a MEANINGFUL 2-word business term that describes the actual functional area. If you cannot find a meaningful name, merge those use cases into an existing subdomain.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**PRACTICAL SIZING GUIDANCE** (CRITICAL — follow strictly):
- For 2-4 use cases: create EXACTLY 1 subdomain (all UCs go into one meaningful subdomain)
- For 5-6 use cases: create EXACTLY 2 subdomains (split roughly evenly)
- For 7-12 use cases: create 2-3 subdomains
- For 13-20 use cases: create 3-5 subdomains
- For 21+ use cases: create 4-8 subdomains
FEWER subdomains is ALWAYS better. A subdomain with 4-5 UCs is ideal. A subdomain with only 2 UCs is acceptable only if semantically necessary. NEVER create subdomains where each has exactly 2 UCs if you could have 1-2 broader subdomains with 3-5 UCs each.

**BUSINESS CONTEXT**:
Domain Name: {domain_name}
Business Name: {business_name}
Industries: {industries}
Business Context: {business_context}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**INPUT DATA**:
CSV of use cases belonging to domain "{domain_name}":
"No","Name","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved"

**TASK WORKFLOW**:
1. Analyze use case names, statements, solutions, techniques, and tables
2. Identify natural business groupings by: processes, functional areas, activities, stakeholders
3. Create 2-WORD business-focused subdomain names
4. **CRITICAL**: Count use cases per subdomain. Merge any subdomain that has fewer than 2 UCs into the nearest related subdomain
5. Validate all hard constraints before outputting

The output language for subdomain names should be {output_language}.

**OUTPUT FORMAT**: CSV with header, one row per use case. Return ONLY the CSV — no text, no fences, no explanations.

Example for "Revenue" domain:
use_case_id,subdomain
N1-AI01,Pricing Strategy
N1-AI04,Yield Management
N1-AI07,Revenue Forecasting

**INPUT USE CASES CSV FOR DOMAIN "{domain_name}"**:
{use_cases_csv}

{previous_violations}

Your response must START with: use_case_id,subdomain,honesty_score,honesty_justification
If you cannot meet requirements, still output CSV (merge subdomains as needed).
🚨 ANTI-PROSE PROTOCOL (read carefully):
- Your FIRST character MUST be 'u' (first letter of "use_case_id" header).
- NO preamble. NO "I'll analyze". NO "Looking at". NO "Let me". NO "**Analysis:**". NO bullet lists.
- Output ONLY: CSV header line, then data rows. Nothing else.
- If you catch yourself writing prose, STOP and start over with 'use_case_id,'.

Begin your CSV response now:
""" + HONESTY_CHECK_CSV
# --- 1f. Translation Prompts (MODIFIED FOR REQUEST #1, #2) ---
PROMPT_TEMPLATES["KEYWORDS_TRANSLATE_PROMPT"] = """You are an expert translator. Your task is to translate the *values* (and only the values) of the following JSON object into {target_language}.
Do NOT translate the keys.

**CRITICAL JSON SYNTAX VALIDATION**:
- Every key-value pair MUST have a colon (:) separator
- Format: "key": "translated value"
- **MOST COMMON ERROR**: Missing colon - `"key" "value"` is WRONG, must be `"key": "value"`
- **FOR ARABIC/CHINESE/JAPANESE/HINDI**: Verify the colon `:` exists between key and value
- Count your colons - you need ONE colon per key-value pair
- Validate your JSON structure before responding

SPECIAL INSTRUCTIONS:
🚨 CRITICAL: You MUST translate ALL text values in the JSON object. Do NOT leave ANY value in English (except for the specific exceptions listed below).

**MANDATORY TRANSLATION REQUIREMENT:**
- EVERY single value in the JSON MUST be translated to {target_language}
- This includes short words like "Type", "Quality", "Subdomain", "Statement", "Solution", "Beneficiary", "Sponsor", etc.
- DO NOT assume any value should stay in English just because it's short or seems technical
- If a value is in English in the input, it MUST be in {target_language} in the output (except for the specific exceptions below)

**ONLY THESE SPECIFIC EXCEPTIONS - DO NOT TRANSLATE:**
1. "N/A" - Keep as "N/A" in all languages

**SPECIAL RULE FOR TITLES CONTAINING "Powered by Databricks Genie Code":**
- "Powered by" MUST be TRANSLATED into the target language (e.g., Arabic: "مدعومة بـ", Spanish: "Impulsados por", French: "Propulsés par", German: "Unterstützt von", Chinese: "由...驱动", Japanese: "を活用した", Korean: "기반의", Hindi: "द्वारा संचालित", Russian: "на базе", Portuguese: "Impulsionados por", Italian: "Alimentati da")
- "Databricks Genie Code" MUST be TRANSLITERATED phonetically for non-Latin scripts (Arabic: "داتا بريكس جيني كود", Chinese: "达塔布里克斯 吉尼 科德", Japanese: "データブリックス ジーニー コード", Korean: "데이터브릭스 지니 코드", Hindi: "डेटाब्रिक्स जीनी कोड", Russian: "Датабрикс Джини Код"). For Latin-script languages, keep "Databricks Genie Code" as-is.
- The word order of "Databricks Genie Code" MUST remain exactly: Databricks, then Genie, then Code (transliterated in same order)

**EVERYTHING ELSE MUST BE TRANSLATED WITHOUT EXCEPTION**

**FOR ALL LANGUAGES - UNIVERSAL REQUIREMENTS:**
- Provide EXACT, DIRECT translations of the English text
- Do NOT use special phrasings or adaptations
- Translate literally and accurately
- Use standard terminology for the target language
- Do NOT leave any English words in the translation (except the 3 exceptions above)
- Ensure complete and proper character encoding for all scripts (Arabic, Chinese, Japanese, Hindi, Cyrillic, etc.)

**MANDATORY TRANSLATIONS FOR ALL LANGUAGES (these MUST be translated, NOT left in English):**

**Common UI Terms (translate to your target language):**
  * "Type" → MUST translate
  * "Subdomain" → MUST translate
  * "Analytics Technique" → MUST translate
  * "Primary Table" → MUST translate
  * "Quality" → MUST translate
  * "Statement" → MUST translate
  * "Solution" → MUST translate
  * "Business Value" → MUST translate
  * "Beneficiary" → MUST translate
  * "Sponsor" → MUST translate
  * "Business Domain" → MUST translate
  * "Tables Involved" → MUST translate

**Value Terms (translate to your target language):**
  * "Risk" → MUST translate
  * "Problem" → MUST translate
  * "Opportunity" → MUST translate
  * "Improvement" → MUST translate
  * "Very High" → MUST translate
  * "High" → MUST translate
  * "Medium" → MUST translate
  * "Low" → MUST translate
  * "Very Low" → MUST translate

**REFERENCE TRANSLATIONS BY LANGUAGE:**

**Arabic:**
  * "Type" = "النوع", "Subdomain" = "المجال الفرعي", "Analytics Technique" = "تقنية التحليل", "Primary Table" = "الجدول الرئيسي", "Quality" = "الجودة"
  * "Statement" = "البيان", "Solution" = "الحل", "Business Value" = "القيمة التجارية"
  * "Beneficiary" = "المستفيد", "Sponsor" = "الراعي", "Business Domain" = "مجال الأعمال"
  * "Risk" = "مخاطرة", "Problem" = "مشكلة", "Opportunity" = "فرصة", "Improvement" = "تحسين"
  * "High" = "عالية", "Very High" = "عالية جداً", "Medium" = "متوسطة", "Low" = "منخفضة", "Very Low" = "منخفضة جداً"

**Spanish:**
  * "Type" = "Tipo", "Subdomain" = "Subdominio", "Analytics Technique" = "Técnica de Análisis", "Primary Table" = "Tabla Principal", "Quality" = "Calidad"
  * "Statement" = "Declaración", "Solution" = "Solución", "Business Value" = "Valor Comercial"
  * "Beneficiary" = "Beneficiario", "Sponsor" = "Patrocinador", "Business Domain" = "Dominio Empresarial"
  * "Risk" = "Riesgo", "Problem" = "Problema", "Opportunity" = "Oportunidad", "Improvement" = "Mejora"
  * "High" = "Alto", "Very High" = "Muy Alto", "Medium" = "Medio", "Low" = "Bajo", "Very Low" = "Muy Bajo"

**French:**
  * "Type" = "Type", "Subdomain" = "Sous-domaine", "Analytics Technique" = "Technique d'Analyse", "Primary Table" = "Table Principale", "Quality" = "Qualité"
  * "Statement" = "Déclaration", "Solution" = "Solution", "Business Value" = "Valeur Commerciale"
  * "Beneficiary" = "Bénéficiaire", "Sponsor" = "Sponsor", "Business Domain" = "Domaine d'Activité"
  * "Risk" = "Risque", "Problem" = "Problème", "Opportunity" = "Opportunité", "Improvement" = "Amélioration"
  * "High" = "Élevé", "Very High" = "Très Élevé", "Medium" = "Moyen", "Low" = "Faible", "Very Low" = "Très Faible"

**German:**
  * "Type" = "Typ", "Subdomain" = "Unterbereich", "Analytics Technique" = "Analysetechnik", "Primary Table" = "Haupttabelle", "Quality" = "Qualität"
  * "Statement" = "Aussage", "Solution" = "Lösung", "Business Value" = "Geschäftswert"
  * "Beneficiary" = "Begünstigter", "Sponsor" = "Sponsor", "Business Domain" = "Geschäftsbereich"
  * "Risk" = "Risiko", "Problem" = "Problem", "Opportunity" = "Chance", "Improvement" = "Verbesserung"
  * "High" = "Hoch", "Very High" = "Sehr Hoch", "Medium" = "Mittel", "Low" = "Niedrig", "Very Low" = "Sehr Niedrig"

**Chinese (Simplified):**
  * "Type" = "类型", "Subdomain" = "子域", "Analytics Technique" = "分析技术", "Primary Table" = "主表", "Quality" = "质量"
  * "Statement" = "声明", "Solution" = "解决方案", "Business Value" = "业务价值"
  * "Beneficiary" = "受益人", "Sponsor" = "赞助者", "Business Domain" = "业务领域"
  * "Risk" = "风险", "Problem" = "问题", "Opportunity" = "机会", "Improvement" = "改进"
  * "High" = "高", "Very High" = "非常高", "Medium" = "中等", "Low" = "低", "Very Low" = "非常低"

**Japanese:**
  * "Type" = "タイプ", "Subdomain" = "サブドメイン", "Analytics Technique" = "分析技術", "Primary Table" = "主要テーブル", "Quality" = "品質"
  * "Statement" = "ステートメント", "Solution" = "ソリューション", "Business Value" = "ビジネス価値"
  * "Beneficiary" = "受益者", "Sponsor" = "スポンサー", "Business Domain" = "ビジネスドメイン"
  * "Risk" = "リスク", "Problem" = "問題", "Opportunity" = "機会", "Improvement" = "改善"
  * "High" = "高", "Very High" = "非常に高い", "Medium" = "中程度", "Low" = "低", "Very Low" = "非常に低い"

**Portuguese:**
  * "Type" = "Tipo", "Subdomain" = "Subdomínio", "Analytics Technique" = "Técnica de Análise", "Primary Table" = "Tabela Principal", "Quality" = "Qualidade"
  * "Statement" = "Declaração", "Solution" = "Solução", "Business Value" = "Valor Comercial"
  * "Beneficiary" = "Beneficiário", "Sponsor" = "Patrocinador", "Business Domain" = "Domínio de Negócio"
  * "Risk" = "Risco", "Problem" = "Problema", "Opportunity" = "Oportunidade", "Improvement" = "Melhoria"
  * "High" = "Alto", "Very High" = "Muito Alto", "Medium" = "Médio", "Low" = "Baixo", "Very Low" = "Muito Baixo"

**Russian:**
  * "Type" = "Тип", "Subdomain" = "Поддомен", "Analytics Technique" = "Аналитическая техника", "Primary Table" = "Основная таблица", "Quality" = "Качество"
  * "Statement" = "Заявление", "Solution" = "Решение", "Business Value" = "Бизнес-ценность"
  * "Beneficiary" = "Бенефициар", "Sponsor" = "Спонсор", "Business Domain" = "Бизнес-домен"
  * "Risk" = "Риск", "Problem" = "Проблема", "Opportunity" = "Возможность", "Improvement" = "Улучшение"
  * "High" = "Высокий", "Very High" = "Очень Высокий", "Medium" = "Средний", "Low" = "Низкий", "Very Low" = "Очень Низкий"

**Hindi:**
  * "Type" = "प्रकार", "Subdomain" = "उपडोमेन", "Analytics Technique" = "विश्लेषण तकनीक", "Primary Table" = "प्राथमिक तालिका", "Quality" = "गुणवत्ता"
  * "Statement" = "कथन", "Solution" = "समाधान", "Business Value" = "व्यावसायिक मूल्य"
  * "Beneficiary" = "लाभार्थी", "Sponsor" = "प्रायोजक", "Business Domain" = "व्यावसायिक डोमेन"
  * "Risk" = "जोखिम", "Problem" = "समस्या", "Opportunity" = "अवसर", "Improvement" = "सुधार"
  * "High" = "उच्च", "Very High" = "बहुत उच्च", "Medium" = "मध्यम", "Low" = "निम्न", "Very Low" = "बहुत निम्न"

**VALIDATION CHECKLIST - Before submitting your translation:**
✓ Every value that was in English is now in {target_language} (except the 3 exceptions)
✓ "Type" is translated (NOT left as "Type")
✓ "Subdomain" is translated (NOT left as "Subdomain")
✓ "Analytics Technique" is translated (NOT left as "Analytics Technique")
✓ "Primary Table" is translated (NOT left as "Primary Table")
✓ "Quality" is translated (NOT left as "Quality")
✓ "Statement" is translated (NOT left as "Statement")
✓ "Solution" is translated (NOT left as "Solution")
✓ "Beneficiary" is translated (NOT left as "Beneficiary")
✓ "Sponsor" is translated (NOT left as "Sponsor")
✓ "Business Value" is translated (NOT left as "Business Value")
✓ "Business Domain" is translated (NOT left as "Business Domain")
✓ "Tables Involved" is translated (NOT left as "Tables Involved")
✓ ALL keys with "Type", "Quality" values are translated (including "type", "quality", "aspect_quality")
✓ All other English words are translated
✓ ONLY "N/A" remains in English. Title phrases have "Powered by" translated and "Databricks Genie Code" transliterated

**CRITICAL - COUNT YOUR TRANSLATIONS:**
The input JSON has approximately 70+ key-value pairs. Your output JSON must have THE SAME number of key-value pairs with 99%+ of values in {target_language} (only "N/A" stays in English; title phrases have "Powered by" translated and "Databricks Genie Code" transliterated for non-Latin scripts).

If you see ANY English words in your output other than the 3 exceptions, you FAILED. Go back and translate them.

Return ONLY a single, valid JSON object with the exact same structure, but with the values translated.

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must be wrapped in honesty JSON: {{"honesty_score": XX, "honesty_justification": "...", "data": <your_translated_json>}}
- Start with: {{"honesty_score":

Input JSON:
{json_payload}

Respond with the honesty-wrapped JSON object.
""" + HONESTY_CHECK_JSON

PROMPT_TEMPLATES["USE_CASE_TRANSLATE_PROMPT"] = """You are an expert translator. Your task is to translate a list of use cases into {target_language}.

**🚨 CRITICAL OUTPUT REQUIREMENT - READ THIS FIRST 🚨**
Your response MUST be PURE CSV DATA ONLY. 
- NO explanations before or after the CSV
- NO conversational text
- NO SQL code fragments
- NO commentary
- NO markdown fences (```)
- ONLY the CSV header and data rows
- Start immediately with the header row: "No","Name",...

**FIELDS TO TRANSLATE** (translate these 11 fields):
"Name", "Business Domain", "Subdomain", "type", "Statement", "Solution", "Business Value", "Beneficiary", "Sponsor", "Business Priority Alignment", "Strategic Goals Alignment"

Do NOT translate: "No", "Tables Involved", "Analytics Technique", "Quality"

**SPECIAL RULES FOR STRATEGIC GOAL ALIGNMENT FIELD**:
- This field contains strategic goal values like: "Reduce Cost", "Increase Revenue", "Boost Productivity", "Mitigate Risk", "Protect Revenue", "Align to Regulations", "Improve Customer Experience", "Enable Data-Driven Decisions", "General Improvement"
- Translate these goal values to {target_language} using proper business terminology
- If multiple goals are comma-separated (e.g., "Reduce Cost, Increase Revenue"), translate each goal separately and keep them comma-separated
- Examples for MULTIPLE languages:
  * Arabic: "Reduce Cost" → "تقليل التكلفة", "Increase Revenue" → "زيادة الإيرادات"
  * French: "Reduce Cost" → "Réduire les coûts", "Boost Productivity" → "Améliorer la productivité"
  * Spanish: "Reduce Cost" → "Reducir costos", "Mitigate Risk" → "Mitigar riesgos"
  * Chinese: "Reduce Cost" → "降低成本", "Increase Revenue" → "增加收入"
  * German: "Reduce Cost" → "Kosten senken", "Protect Revenue" → "Umsatz schützen"

**NOTE**: The SQL field is NOT included in the input to reduce payload size. You only need to translate the fields listed above.

**SPECIAL RULES FOR SPONSOR FIELD**:
- If the Sponsor field contains a name in the format "Name (Title)" (e.g., "John Smith (Chief Technology Officer)"), you MUST transliterate/translate BOTH the name AND the title for ALL languages
- Person names should be transliterated phonetically into the target language writing system
- **CRITICAL**: This applies to ALL languages, not just non-Latin scripts. Transliterate names appropriately for each target language.
- Examples for MULTIPLE languages:
  * French: "John Smith (Chief Technology Officer)" → "Jean Smith (Directeur de la Technologie)"
  * Spanish: "John Smith (Chief Technology Officer)" → "Juan Smith (Director de Tecnología)"
  * Arabic: "John Smith (Chief Technology Officer)" → "جون سميث (المدير التنفيذي للتكنولوجيا)"
  * Chinese: "John Smith (Chief Technology Officer)" → "约翰·史密斯 (首席技术官)"
  * Japanese: "John Smith (Chief Technology Officer)" → "ジョン・スミス (最高技術責任者)"
  * German: "John Smith (Chief Technology Officer)" → "Johann Schmidt (Technischer Leiter)"
  * Russian: "John Smith (Chief Technology Officer)" → "Джон Смит (Технический директор)"
  * Hindi: "John Smith (Chief Technology Officer)" → "जॉन स्मिथ (मुख्य प्रौद्योगिकी अधिकारी)"
- If the Sponsor field contains ONLY a title (no name), translate it normally
- Always adapt the name to sound natural in the target language and culture

**OUTPUT FORMAT: CSV (NOT JSON)**
Your response MUST be in CSV format with the following structure:
- First line: Header row with 15 column names (SQL field is not included)
- Subsequent lines: One row per use case with 15 fields

**CSV HEADER (MUST BE EXACTLY THIS)**:
"No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Strategic Goals Alignment","Tables Involved","Quality"

**CRITICAL CSV FORMATTING RULES**:
1. ALL fields MUST be enclosed in double quotes (")
2. Use comma (,) as the field separator
3. For fields containing commas or quotes, keep them inside the double quotes
4. Each row must have exactly 15 fields (SQL is NOT included)
5. Keep untranslated fields (No, Tables Involved, Analytics Technique, Quality) EXACTLY as they appear in the input
6. The Analytics Technique field should be copied EXACTLY as-is in English (do not translate)
7. The Quality field should be copied EXACTLY as-is in English (do not translate "Very High", "High", "Medium", "Low", "Very Low")

**IMPORTANT**: Analytics Technique and Quality values remain in English in the CSV. The UI will handle translation separately.

**TRANSLATION REQUIREMENTS**:
1. Translate ALL 11 designated fields for EVERY use case — including ALL long text fields
2. "Business Domain" - MUST be translated to {target_language}
3. "Subdomain" - MUST be translated to {target_language}
4. "Name" - MUST be translated to {target_language}
5. "Business Priority Alignment" - MUST be translated to {target_language} (translate the priority values like "Reduce Cost" → appropriate translation)
6. "Strategic Goals Alignment" - MUST be translated to {target_language} (this is FREE-TEXT describing strategic goals; translate the entire text naturally)
7. "Statement" - MUST be FULLY translated to {target_language} (this is a long text field — do NOT leave it in English)
8. "Solution" - MUST be FULLY translated to {target_language} (this is a long text field — do NOT leave it in English)
9. "Business Value" - MUST be FULLY translated to {target_language} (this is a long text field — do NOT leave it in English)
10. "Beneficiary" - MUST be translated to {target_language} (translate role titles like "Revenue Management Analyst")

🚨 **CRITICAL FOR NON-LATIN SCRIPTS (Chinese, Arabic, Japanese, Korean, Hindi, Russian)**:
- You MUST translate ALL 11 fields including long text (Statement, Solution, Business Value)
- If ANY field remains in English/Latin script (except No, Tables Involved, Analytics Technique, Quality), your translation is WRONG
- The Statement, Solution, and Business Value fields are LONG paragraphs — you MUST translate them FULLY, not skip them

**SPECIAL INSTRUCTIONS**:
- "Powered by Databricks Genie Code" → "Powered by" MUST be TRANSLATED into {target_language}. "Databricks Genie Code" MUST be TRANSLITERATED phonetically for non-Latin scripts (Arabic: "مدعومة بـ داتا بريكس جيني كود", Chinese: "由 达塔布里克斯 吉尼 科德 驱动", Japanese: "データブリックス ジーニー コード を活用した", Korean: "데이터브릭스 지니 코드 기반의", Hindi: "डेटाब्रिक्स जीनी कोड द्वारा संचालित", Russian: "на базе Датабрикс Джини Код"). For Latin-script languages, translate "Powered by" and keep "Databricks Genie Code" as-is. Word order of "Databricks Genie Code" must be preserved exactly.
- Keep numeric IDs (like "AI-001") unchanged
- Keep Analytics Technique and Quality values in English (not translated in CSV)
- For Arabic: Ensure complete and proper Arabic text encoding. Double-check all Arabic characters are properly formed.

**🛑 ABSOLUTELY FORBIDDEN - DO NOT INCLUDE 🛑**:
- ❌ NO text before the CSV header
- ❌ NO explanatory text like "Here is the translation..." or "I've translated..."
- ❌ NO SQL code fragments or queries outside the CSV structure
- ❌ NO examples or demonstrations
- ❌ NO line breaks or empty lines before the header
- ❌ NO commentary about the translation process
- ❌ NO additional rows beyond the exact number provided in the input
- ✅ ONLY: Pure CSV starting with the header, nothing else

**VALIDATION CHECKLIST**:
✓ Response starts with the exact 15-column header (no text before it)
✓ All fields are enclosed in double quotes
✓ Each row has exactly 15 comma-separated fields (SQL is NOT included)
✓ No markdown fences (```) around the CSV
✓ Analytics Technique field is copied exactly as received (do not translate it)
✓ Quality field is copied exactly as received (do not translate it)
✓ Business Priority Alignment field is translated to target language
✓ Strategic Goals Alignment field is translated to target language
✓ Statement, Solution, Business Value fields are FULLY translated (not left in English)
✓ All 11 translatable fields are translated for every row

**CRITICAL ROW COUNT REQUIREMENT**:
- You will receive EXACTLY a specific number of use cases in the input
- Your CSV output MUST contain EXACTLY the same number of data rows (plus the header row)
- DO NOT add any extra rows beyond what was provided in the input
- DO NOT omit any rows from the input
- Each row in your output MUST correspond to exactly one row from the input, matched by the "No" field

**INPUT USE CASES** (as JSON for readability):
{json_payload}

🚨 FINAL INSTRUCTION 🚨
Your ENTIRE response must be ONLY the CSV data.
- Start your response with: "No","Name","Business Domain","Subdomain","type","Statement",...
- Do NOT write anything before this header
- Do NOT write anything after the last data row
- Do NOT include markdown code fences (```)
- Do NOT include any explanatory text
- NO thoughts or reasoning
- NO "Here is the translation..." or similar statements
- The number of data rows MUST exactly match the number of use cases above

🚨 ABSOLUTE RULE - OUTPUT FORMAT 🚨:

❌ DO NOT INCLUDE:
- Any conversational or explanatory text ("Here is...", "I've...", "The...")
- Any thoughts or analysis descriptions

✅ OUTPUT REQUIREMENTS:
- Your response must START with: "No","Name","Business Domain",...,"honesty_score","honesty_justification"
- Include honesty columns in header and all rows

Begin your CSV response now (header row first, include honesty columns):
""" + HONESTY_CHECK_CSV


# --- 1f. Use Case Review Prompt (RENAMED, ENHANCED) ---
PROMPT_TEMPLATES["REVIEW_USE_CASES_PROMPT"] = """You are a **Deduplication Specialist** for an analytics use case portfolio. Your job is to remove semantic duplicates — use cases that address the SAME business problem with the SAME core analysis on the SAME entity/metric, OR that analyze the SAME underlying data with the SAME analytical intent even if the wording differs. You must PRESERVE genuinely distinct use cases.

**🚨 DECISION BIAS: When in doubt, FLAG for human review (keep the use case AND tag it), do NOT silently keep it. Silent-keeps are how duplicate portfolios leak through. 🚨**
**TARGET: Remove genuine duplicates AND flag suspected duplicates. Typical removal rate is 10-30%. Suspected-but-ambiguous cases must be KEPT with honesty_justification tagged `_FLAG_DUPLICATE_SUSPECT` so humans can review.**

**INPUT**: You receive ID, Name, Business Domain, Analytics Technique, Tables (primary table names, without catalog/schema), and a 1-line Statement. Judge duplicates on BUSINESS INTENT AND DATA GROUNDING (tables + technique + statement together).

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**REMOVAL CRITERIA** — a use case is removed if ANY of the following is true:

**LAYER A — IDENTICAL NAME:** Two or more use cases share the EXACT same Name (case-insensitive, whitespace-normalized). Keep the earliest ID; remove the rest.

**LAYER B — SEMANTIC DUPLICATE (verb + noun synonym + entity/metric all match):**
  1. Normalize leading verb: Forecast/Predict/Anticipate/Envision → PREDICT; Detect/Identify/Reveal/Find/Discover/Uncover/Expose → DETECT; Classify/Segment/Categorize/Tier/Rank → CLASSIFY; Simulate/Model/Project → SIMULATE; Optimize/Maximize/Minimize → OPTIMIZE; Assess/Evaluate/Measure/Gauge/Quantify → ASSESS. Strip "with [...]" activation suffix.
  2. Apply noun synonym map: Revenue=Income=Sales; Cost=Expense=Spend; Churn=Attrition=Turnover; Demand=Consumption=Volume; Customer=Client=Account; Supplier=Vendor; Inventory=Stock=Supply; Fraud=Anomaly; Leakage=Loss=Erosion.
  3. Extract ENTITY and METRIC. If (normalized verb + entity + metric) all match → DUPLICATE. Example: "Forecast Customer Churn Rate" and "Predict Client Attrition Probability" → entity=Customer, metric=Churn → DUPLICATE.

**LAYER C — TECHNIQUE SWAP ON SAME QUESTION:** Same entity + same metric + same analytical intent (from Statement), but the Analytics Technique column differs. This is a duplicate — the underlying business question is identical; only the technique label varies. Example: two use cases on "predicting customer churn" where one lists Technique=Classification and the other lists Technique=Predictive Modeling → DUPLICATE.

**LAYER D — TABLE + INTENT OVERLAP:** Primary Tables heavily overlap (≥60% common base-table names) AND the Statements describe the same analytical question (same entity + same metric + same direction of analysis) even if names and techniques differ. This is the most common silent-duplicate pattern. EXAMPLE: UC-X "Identify low-utilization assets via clustering" on table `asset_utilization` and UC-Y "Detect underused equipment with anomaly analysis" on `asset_utilization` → same tables, same question → DUPLICATE.

**EXPLICITLY NOT DUPLICATES — KEEP BOTH (even under Layer D):**
- Different analytical DIMENSIONS on the same entity ("Revenue by Product" vs "Revenue by Customer").
- Different METRICS on the same entity ("Customer Churn" vs "Customer Lifetime Value").
- Different analytical QUESTIONS on the same data ("Forecast demand" vs "Detect demand anomalies" — prediction ≠ anomaly detection).
- Different BUSINESS DOMAINS even with superficial textual similarity (treat each domain's portfolio independently unless Layer A matches exactly).

**AMBIGUOUS CASES — FLAG, DO NOT HIDE:**
If you are unsure whether two use cases are duplicates (partial match, close but not identical business intent), KEEP both but set honesty_justification to `_FLAG_DUPLICATE_SUSPECT: <brief reason referencing the other ID>`. This surfaces the ambiguity to humans for review. Ambiguous-kept-silent is FORBIDDEN.

**DUPLICATE RESOLUTION — which ID to KEEP when removing:**
a. Keep the use case whose Statement is more specific and actionable.
b. Prefer the one whose Tables cover a broader analytical join (more tables listed).
c. Tie-break: keep the earliest ID.

**SOFT QUALITY FLAGS (KEEP the use case but tag honesty_justification):**
- `_TRIVIAL_WRAPPER`: Name is just "[Technique] + [Column Name]" with no business framing (e.g., "Cluster transaction_amount").
- `_TECH_ONLY`: Name describes a technical method rather than a business outcome (e.g., "Apply Random Forest to Sales").
These flags help downstream scoring penalize low-quality framing without losing coverage.

**BUSINESS VALUE GATES (apply DURING review — drop if in doubt when count > 75):**
When the portfolio has more than 75 use cases, apply these additional checks and DROP if in doubt on ANY:
- **BUSINESS RELEVANCE**: Is it relevant to how the business operates? Does someone in the business have this exact problem? Does it represent current pain OR future opportunity?
- **SPONSOR TEST**: Would a Sponsor/CFO fund this? Can they pitch it to the CEO without caveats?
- **ENGINEERING TEST**: Can the production team deploy it, support it long-term, AND fix issues without external dependencies?

If ANY of these is "unclear" or "no", DROP the use case (tag `honesty_justification=_FAIL_BUSINESS_VALUE_GATE: <which gate>`). The goal is to land in the 50-100 UC range — ideally settling around 75 (the sweet spot). Be aggressive about dropping borderline UCs when above 75; preserve core coverage when near 50.

**RULES**: Keep earliest ID on duplicates. Reviewing ALL {total_count} use cases in one pass.

Here is the markdown table of ALL use cases to analyze:
{use_case_markdown}

**OUTPUT**: CSV listing IDs to KEEP. Every KEPT id that hits a soft flag or `_FLAG_DUPLICATE_SUSPECT` condition MUST carry that tag in `honesty_justification`. No markdown fences, no prose.

Your response must START with: use_case_id,honesty_score,honesty_justification
""" + HONESTY_CHECK_CSV

# --- 1g. Use Case VALUE Scoring Prompt (COMPREHENSIVE WITH BUSINESS CONTEXT) ---
PROMPT_TEMPLATES["USE_CASE_VALUE_SCORE_PROMPT"] = """# Persona

You are the **Chief Investment Officer & Strategic Value Architect**. You are known for being ruthless, evidence-based, and ROI-obsessed. You do not care about "cool tech" or "easy wins" unless they drive massive financial impact. Your job is to allocate finite capital only to use cases that drive the specific strategic goals of this business.

# Context & Inputs

**Business Context:** {business_context}
**Strategic Goals:** {strategic_goals}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**Business Priorities:** {business_priorities}
**Strategic Initiative:** {strategic_initiative}
**Value Chain:** {value_chain}
**Revenue Model:** {revenue_model}

**Use Cases to Score:**
{use_case_markdown}

# Instructions

### Task Overview

Score the provided use cases based strictly on their potential to drive the specific **Business Priorities**, **Strategic Goals**, and **Revenue Model** provided in the **Context & Inputs** section above.

🚨🚨🚨 **CRITICAL: BUSINESS PRIORITIES DRIVE RANKING** 🚨🚨🚨

Use cases that directly achieve any of the Business Priorities MUST receive significantly higher scores. Strategic Goals define the intended outcomes and must be used for alignment.

**Common Strategic Goals to Consider:**
- **Reduce Cost**: Automation, efficiency improvements, waste reduction
- **Boost Productivity**: Faster processes, better tools, streamlined workflows  
- **Increase Revenue**: New revenue streams, upselling, cross-selling, market expansion
- **Mitigate Risk**: Fraud detection, compliance, security, audit trails
- **Protect Revenue**: Churn prevention, retention, customer satisfaction
- **Align to Regulations**: Compliance automation, regulatory reporting, audit support
- **Improve Customer Experience**: Personalization, faster service, quality improvements
- **Enable Data-Driven Decisions**: Analytics, insights, forecasting, predictions

**For EVERY use case, you MUST:**
1. Identify which Strategic Goal(s) it aligns to (if any)
2. Score higher if it DIRECTLY achieves a stated Business Priority
3. In the justification, EXPLICITLY mention which Business Priority and Strategic Goal(s) the use case supports

**🚨 SCORING RULES: AGGRESSIVE BUSINESS VALUE 🚨**

1.  **NO CURVE / NO DISTRIBUTION:** Do not force a normal distribution. If all use cases are weak, score them all low. If all are critical, score them all high. Score based on **Absolute Merit**.
2.  **ZERO-BASED SCORING:** Start every score at 1.0. The use case must *earn* points by showing explicit alignment to the data provided in the Context. Do not assume value exists unless clearly demonstrated.
3.  **IGNORE "NICE TO HAVES":** If a use case improves a process that does not directly impact revenue, margin, or strategic competitive advantage, it is a **Low Value** case, regardless of how easy it is to implement.
4.  **STRATEGIC GOAL BONUS:** Use cases that DIRECTLY achieve a stated Strategic Goal should receive a +0.5 to +1.0 bonus to their Strategic Alignment score.

**🚨🚨🚨 CRITICAL: BUSINESS RELEVANCY & REALISM PENALTY 🚨🚨🚨**

5.  **IRRELEVANT CORRELATIONS = LOW SCORE:** Use cases that correlate variables with NO logical, provable cause-and-effect relationship MUST receive low scores (ROI ≤ 2.0, Alignment ≤ 2.0).
6.  **NONSENSICAL EXTERNAL DATA = LOW SCORE:** Use cases that include external data enrichment without a clear, industry-recognized business connection MUST be penalized heavily.
7.  **RELEVANCY TEST:** For EVERY use case, ask: "Can I explain in ONE sentence why these variables/factors are logically connected?" If NO, score LOW.
8.  **BOARDROOM TEST:** Would a senior executive approve budget for this analysis without questioning the logic? If the correlation seems invented or far-fetched, score LOW and note in the justification: "Irrelevant correlation - no logical business connection."

### The 75/25 Priority Formula (Critical)

You must use a weighted formula for the final Priority Score to heavily favor Business Value over Feasibility.

1.  **Calculate Value Score (1.0 - 5.0):** Weighted average of ROI (60%), Alignment (25%), TTV (7.5%), Reusability (7.5%).
2.  **Calculate Feasibility Score (1.0 - 5.0):** Simple average of the 8 feasibility factors.
3.  **CALCULATE PRIORITY SCORE (2.0 - 10.0):**
    
    $$ Priority = (Value * 1.5) + (Feasibility * 0.5) $$

*Note: This formula ensures that Business Value accounts for 7.5 points of the total score, while Feasibility only accounts for 2.5 points.*

---

### Scoring Factors (Detailed Assessment Criteria)

**I. VALUE FACTORS (The "Why")**

**1. Return on Investment (ROI)** 🚨 **WEIGHT: 60% of Value Score** 🚨
    * **Contextual ROI Check:** Compare the use case against the **Revenue Model** listed in the Context. Does this use case directly impact the way this specific company makes money?
    * **4.8 - 5.0 (Exponential):** Directly impacts top-line revenue or prevents massive bottom-line leakage (>10x return). *Examples: Dynamic Pricing, Demand Forecasting, Churn Prevention for high-value customers.*
    * **4.0 - 4.7 (High):** Significant measurable impact on P&L (5-10x return). *Examples: Supply Chain Optimization, Fraud Detection, Predictive Maintenance.*
    * **3.0 - 3.9 (Moderate):** Incremental efficiency gains (2-5x return). *Examples: Automated Invoice Processing, Intelligent Document Classification.*
    * **1.0 - 2.9 (Low/Soft):** "Soft" benefits (efficiency, happiness) that do not clearly translate to dollars in the **Revenue Model**. *Examples: Internal Wiki Search, Employee Sentiment Dashboard.*
    * **CRITICAL**: Evaluate ROI based on the ACTUAL industry and business model from the provided context - do NOT assume any specific industry.

**2. Strategic Alignment** 🚨 **WEIGHT: 25% of Value Score** 🚨
    * **Strict Alignment Check:** Look at the **Business Priorities** and **Strategic Goals** listed in the Context. Is this use case mentioned?
    * **4.8 - 5.0 (Direct Hit):** The use case is EXPLICITLY named in or required by the **Business Priorities** or **Strategic Goals**. *Pattern: If priority mentions "[X]", use case directly addresses "[X]".*
    * **3.5 - 4.7 (Strong Link):** Supports a stated **Business Priority** directly. *Pattern: Priority is about retention/growth/efficiency, use case directly enables that outcome.*
    * **1.0 - 3.4 (Weak/None):** Generic improvement (e.g., "Better Reporting") that does not touch the specific **Business Priorities** listed in the Context.
    * **CRITICAL**: Evaluate alignment based on the ACTUAL Business Priorities and Strategic Goals provided in the context - do NOT assume any default goals.

**3. Time to Value (TTV)** (Weight: 7.5%)
    * **Definition:** How fast until the business *sees* the money?
    * **4.8 - 5.0 (Immediate):** < 4 weeks. Quick wins, dashboarding existing data.
    * **3.0 - 4.7 (Quarterly):** 1-3 months. Standard agile cycle.
    * **1.0 - 2.9 (Long Term):** > 6 months. Long infrastructure build-outs before any value is realized.

**4. Reusability** (Weight: 7.5%)
    * **Definition:** Does this create a permanent asset (Feature Store, Data Product)?
    * **4.8 - 5.0 (Platform Asset):** Creates a "Customer 360" or "Product Master" table that 10+ other use cases *will* leverage.
    * **3.0 - 4.7 (Modular):** Code is clean and reusable, but data is specific to this use case.
    * **1.0 - 2.9 (One-Off):** Ad-hoc analysis or script solving exactly one isolated problem.

**II. FEASIBILITY FACTORS (The "How" - Average of all 8)**

**5. Data Availability**
    * **Check:** Does the specific data required exist in this industry/business context?
    * **4.8 - 5.0 (Perfect):** Data is standard, transactional, and historically logged (e.g., Sales Records, ERP logs).
    * **3.0 - 4.7 (Likely):** Data likely exists but might be scattered or require some engineering to consolidate.
    * **1.0 - 2.9 (Missing):** Requires new sensors, external purchases, or starting logs from scratch.

**6. Data Accessibility**
    * **Check:** Are there Legal, Privacy, or Tech barriers?
    * **4.8 - 5.0 (Open):** Internal, non-PII, open access data.
    * **3.0 - 4.7 (Restricted):** PII present but manageable via standard RBAC/Masking.
    * **1.0 - 2.9 (Blocked):** Highly sensitive (Medical/Banking) or owned by a 3rd party refusing to share.

**7. Architecture Fitness**
    * **Check:** Does it fit the Databricks Lakehouse platform capabilities?
    * **4.8 - 5.0 (Native):** Solvable using Databricks-native capabilities: SQL, Python, PySpark, scikit-learn, XGBoost, LightGBM, MLlib, MLflow, AI functions, or any combination. Fits Medallion Architecture and Databricks runtime.
    * **3.0 - 4.7 (Adaptable):** Requires external API calls, custom library installs beyond the standard Databricks runtime, or non-standard configurations.
    * **1.0 - 2.9 (Incompatible):** Requires mainframe, on-prem appliances, real-time streaming infrastructure, or non-cloud tech.

**8. Team Skills**
    * **Check:** Does a typical data team in this industry have these skills?
    * **4.8 - 5.0 (Standard):** SQL, Python, scikit-learn, XGBoost, standard ML/statistical modeling, Databricks AI functions — skills widely available in modern data teams.
    * **3.0 - 4.7 (Specialized):** Advanced NLP, Computer Vision, custom deep learning, complex optimization solvers.
    * **1.0 - 2.9 (Niche):** Requires PhD-level research math, proprietary algorithms, or archaic languages (COBOL).

**9. Domain Knowledge**
    * **Check:** Is the business logic clear?
    * **4.8 - 5.0 (Documented):** Logic is clear, rules are written, SMEs are available.
    * **3.0 - 4.7 (Tribal):** "Head knowledge" exists but isn't written down.
    * **1.0 - 2.9 (Unknown):** Logic is a "Black Box" or lost.

**10. People Allocation**
    * **Check:** Staffing difficulty.
    * **4.8 - 5.0 (Minimal):** 1-2 Engineers.
    * **3.0 - 4.7 (Squad):** Full agile squad (4-6 people).
    * **1.0 - 2.9 (Heavy):** Requires massive cross-functional teams or external consultants.

**11. Budget Allocation**
    * **Check:** Likelihood of funding.
    * **4.8 - 5.0 (Secured):** Critical path for the **Strategic Initiative** listed in the Context.
    * **3.0 - 4.7 (Discretionary):** Funded via normal OPEX.
    * **1.0 - 2.9 (CapEx Required):** Needs board approval for new money.

**12. Time to Production**
    * **Check:** Engineering effort magnitude.
    * **4.8 - 5.0 (Sprint):** < 2 weeks dev time.
    * **3.0 - 4.7 (Quarterly):** 1-3 months dev time.
    * **1.0 - 2.9 (Major Project):** > 6 months dev time.

-----

### Scoring Methodology - Execution Steps

**STEP 1: CALCULATE RAW VALUE (High Precision)**
* Score ROI (0-5) based on the **Revenue Model** in the Context.
* Score Alignment (0-5) based on the **Strategic Goals** in the Context.
* Score TTV and Reusability.
* Calculate:
  $$ Value = (ROI * 0.60) + (Alignment * 0.25) + (TTV * 0.075) + (Reusability * 0.075) $$

**STEP 2: CALCULATE RAW FEASIBILITY**
* Average the 8 feasibility factors (Factors 5 through 12).
* Calculate:
  $$ Feasibility = (Sum of 8 Factors) / 8 $$

**STEP 3: APPLY THE "VALUE-FIRST" PRIORITY FORMULA**
* Calculate:
  $$ Priority Score = (Value * 1.5) + (Feasibility * 0.5) $$
* *Validation Logic:*
    * If Value is 5.0 and Feasibility is 1.0 -> Priority = 8.0 (High Priority)
    * If Value is 1.0 and Feasibility is 5.0 -> Priority = 4.0 (Low Priority)
    * **This mathematically forces High Value cases to always outrank High Feasibility cases.**

**STEP 4: GENERATE JUSTIFICATION**
* Write a sharp, executive summary (max 200 chars).
* **Must** reference specific **Strategic Goals** or **Revenue Model** elements found in the Context.
* **Must** justify the score based on BUSINESS IMPACT, not technical ease.

🚨 **JUSTIFICATION QUALITY RULES - CRITICAL** 🚨:
1. **USE CASE SPECIFIC**: The justification MUST be specific to THIS use case. It should mention the core capability or outcome (e.g., "network congestion prediction", "churn prevention", "demand forecasting").
2. **NO GENERIC BUZZWORDS**: Do NOT use generic phrases that could apply to any use case. The following are PROHIBITED unless directly relevant to the use case domain:
   - "digital transformation" (too vague)
   - "workflow automation" (unless the use case IS about workflow automation)
   - "revenue recognition" (unless the use case IS about revenue recognition)
   - "operational efficiency" (too generic)
   - "data-driven insights" (too vague)
3. **CONNECT TO USE CASE DOMAIN**: If the use case is about "Network Congestion Prediction", the justification MUST mention network, capacity, congestion, or infrastructure concepts - NOT unrelated benefits like "CSAT" or "invoice processing".
4. **EXAMPLES OF BAD vs GOOD JUSTIFICATIONS**:
   - ❌ BAD for "Predict Network Congestion": "Accelerates revenue recognition and supports digital transformation through workflow automation."
   - ✅ GOOD for "Predict Network Congestion": "Proactively prevents service degradation by predicting network hotspots, directly reducing churn and protecting recurring revenue from enterprise clients."
   - ❌ BAD for "Churn Prediction": "Improves operational efficiency and enables data-driven decision making."
   - ✅ GOOD for "Churn Prediction": "Identifies at-risk customers 30 days before cancellation, enabling targeted retention campaigns that protect $2M annual recurring revenue."

-----

### Output Format

Return **ONLY** a valid CSV.

**Columns:**
"No","Strategic Alignment","Return on Investment","Reusability","Time to Value","Data Availability","Data Accessibility","Architecture Fitness","Team Skills","Domain Knowledge","People Allocation","Budget Allocation","Time to Production","Value","Feasibility","Priority","Business Priority Alignment","Strategic Goals Alignment","Justification","AI_Confidence","AI_Feedback"

**CRITICAL - Business Priority Alignment Column:**
For each use case, identify which business priority(ies) it aligns to. Use the following format:
- If aligned to ONE priority: "Reduce Cost" or "Increase Revenue" etc.
- If aligned to MULTIPLE priorities: "Reduce Cost, Mitigate Risk" (comma-separated)
- If NO clear alignment: "General Improvement"

Standard business priorities: Increase Revenue | Reduce Cost | Optimize Operations | Mitigate Risk | Empower Talent | Enhance Experience | Drive Innovation | Achieve ESG | Protect Revenue | Execute Strategy

**CRITICAL - Strategic Goals Alignment Column:**
For each use case, identify which strategic goal(s) it aligns to.
- Look at the **Strategic Goals** provided in the Context section above (these are either user-provided OR generated from business context)
- Match each use case to one or more strategic goals from the context
- If aligned to ONE goal: output that goal exactly as written
- If aligned to MULTIPLE goals: list them comma-separated
- If the use case does NOT align to ANY of the strategic goals in context: output "General Improvement"
- Be specific: use the EXACT wording of the strategic goals from the context

**CRITICAL - AI_Confidence and AI_Feedback Columns (LAST 2 COLUMNS - MANDATORY):**
For EACH use case, you MUST provide:
- **AI_Confidence**: A decimal score from 0.0 to 1.0 representing your honesty score - how truthfully and completely you achieved this scoring task. Consider: data quality, domain expertise applied, clarity of the use case statement.
- **AI_Feedback**: A comprehensive explanation that MUST include: 1) All reasons justifying your AI_Confidence score, 2) If score < 1.0, what specific improvements are needed to reach 1.0, 3) A MANDATORY "MISSING DATA" section listing all data/context that if provided would have improved your scoring accuracy. Be 100% honest - your output will be reviewed by another more powerful AI to judge your score.

**Example Output:**
```csv
"No","Strategic Alignment","Return on Investment","Reusability","Time to Value","Data Availability","Data Accessibility","Architecture Fitness","Team Skills","Domain Knowledge","People Allocation","Budget Allocation","Time to Production","Value","Feasibility","Priority","Business Priority Alignment","Strategic Goals Alignment","Justification","AI_Confidence","AI_Feedback"
"AI-U001",4.9,4.8,4.5,4.2,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.76,4.00,9.14,"Increase Revenue, Reduce Cost","Improve carbon footprint tracking, Optimize workforce efficiency","Directly drives Revenue Growth by optimizing pricing engine. Achieves Increase Revenue priority and aligns to strategic goals.",0.85,"Score Justification: High confidence due to clear business value proposition and well-defined table relationships. Improvements Needed: Historical implementation success rates would raise score to 0.95. MISSING DATA: Industry benchmark ROI metrics, competitor pricing data, historical pricing model performance statistics."
"AI-U002",1.2,1.5,2.0,3.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,1.58,5.00,4.87,"General Improvement","General Improvement","Does not align with any stated Business Priorities or Strategic Goals. Purely administrative 'nice-to-have'.",0.72,"Moderate confidence. Use case statement is vague - would benefit from specific metrics and expected outcomes."

```

**OUTPUT RULES:**

* Return ONLY the CSV data (with header row).
* The `Priority Score` in the CSV MUST reflect the `(Value * 1.5) + (Feasibility * 0.5)` formula.
* Do not normalize. Real scores only.
* The `Business Priority Alignment` column MUST specify which business priority(ies) the use case achieves.
* The `Strategic Goals Alignment` column MUST specify which strategic goal(s) from the context the use case achieves, or "General Improvement" if none match.
* The `AI_Confidence` column MUST be a decimal between 0.0 and 1.0 (your honesty score).
* The `AI_Feedback` column MUST include: reasons for score, improvements needed if < 1.0, and MISSING DATA section.

**🚨🚨🚨 CRITICAL: SCORE EVERY SINGLE USE CASE 🚨🚨🚨**
* You MUST output a CSV row for EVERY use case in the input table
* Count the use cases: If there are N use cases in the input, you MUST output EXACTLY N data rows (plus header)
* DO NOT skip any use case. DO NOT truncate the output.
* If you're running low on output space, use shorter justifications but NEVER omit rows
* Missing use case scores = CRITICAL FAILURE

Begin your CSV response now:
"""

# --- 1g-1. USE CASE QUALITY SCORE PROMPT (DATA SUFFICIENCY & CAUSE-EFFECT SCORING) ---
PROMPT_TEMPLATES["USE_CASE_QUALITY_SCORE_PROMPT"] = """# USE CASE DATA QUALITY SCORER

## YOUR ROLE: DATA QUALITY ASSESSOR

You are a **Chief Data Scientist** and **Domain Expert** responsible for assessing the DATA QUALITY aspect of each use case. Your task is to evaluate whether the data ACTUALLY SUPPORTS the claimed analysis and assign a quality score.

**YOUR MISSION**: Score each use case based on how well the available data can support the high level design and achieve the intended goal.

---

## CONTEXT

**Business Context:** {business_context}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**Industry:** {industry}

---

## AVAILABLE SCHEMA (THE ONLY DATA THAT EXISTS)

{schema_details}

---

## USE CASES TO SCORE

{use_cases_to_validate}

---

""" + QUALITY_GATE_SCORING_BLOCK + """

### SCORING GUIDE — ANTI-LENIENCY CALIBRATION (apply to EACH dimension)

**🚨 CRITICAL: You are known to over-score. Consciously resist the urge to give 4.0+. The DEFAULT score is 3.0. A use case must EARN every point above 3.0 with CONCRETE EVIDENCE. 🚨**

- **5.0** (EXCEPTIONAL — rare, <5% of use cases): Flawless execution with zero gaps, novel approach, multi-dimensional depth, would impress a skeptical CTO. Reserve this for truly outstanding use cases.
- **4.5** (EXCELLENT — uncommon, <10%): Nearly flawless, demonstrates sophisticated analytical thinking, strong cross-table integration.
- **4.0** (STRONG — selective, ~15%): Clearly above average with evidence of depth. NOT just "looks good" — must demonstrate concrete analytical value beyond standard patterns.
- **3.5** (ABOVE AVERAGE — common): Solid but with clear room for improvement. This is the MINIMUM for "High" quality label.
- **3.0** (AVERAGE — the DEFAULT starting point): Acceptable but unremarkable. Standard patterns without differentiation. If you cannot articulate WHY a dimension deserves above 3.0, it IS 3.0.
- **2.5** (BELOW AVERAGE): Notable weaknesses or gaps. Triggers soft veto on critical dimensions.
- **2.0** (WEAK): Significant failures. Triggers hard veto on critical dimensions.
- **1.5** (POOR): Major failures across the dimension.
- **1.0** (FAILED): Dimension completely not satisfied.

**DISTRIBUTION EXPECTATION**: In a well-calibrated scoring run, ~20% of use cases should score below 3.0 overall, ~40% should score 3.0-3.5, ~25% should score 3.5-4.0, and only ~15% should score 4.0+. If your scores skew higher, you are being lenient.

### CALIBRATION ANCHORS — what each score LOOKS LIKE in practice:

**D8 (Uniqueness) = 1.0**: "Forecast Customer Churn with Retention Campaign Prioritization" when "Predict Client Attrition with Proactive Outreach Queue" already exists. Same entity (customer), same metric (churn/attrition), just synonyms.
**D8 = 5.0**: A use case analyzing supplier delivery reliability when no other use case touches supplier operations. Completely novel business question.

**D9 (Depth) = 1.0**: "Detect Overdue Invoice Risk" — this is just WHERE due_date < CURRENT_DATE. No advanced analysis required.
**D9 = 2.0 (TECHNIQUE MISMATCH)**: Analytics Technique says "Anomaly Detection" but High Level Design describes only z-scores and SQL thresholds without Isolation Forest, LOF, or any ML-based anomaly detection. OR: Technique says "Clustering" but design uses SQL NTILE/CASE WHEN instead of K-Means/DBSCAN. SQL-only implementation of an ML-designated technique is ALWAYS D9 = 2.0 regardless of SQL complexity.
**D9 = 3.0**: Multi-column aggregation with a statistical comparison — requires some effort but no genuine predictive modeling or ML. Acceptable ONLY when the Analytics Technique is inherently SQL-appropriate (Pareto Analysis, Cohort Analysis, Funnel Analysis).
**D9 = 4.0**: Cross-table feature engineering → trained ML model (XGBoost/Random Forest/Isolation Forest/K-Means) → prediction/detection/clustering with evaluation metrics → ai_query explanation. The technique matches the Analytics Technique designation.
**D9 = 5.0**: Multi-step pipeline: cross-table feature engineering → multiple model comparison (XGBoost vs LightGBM vs Random Forest) → ensemble scoring → survival analysis or optimization → counterfactual scenario analysis → ai_query with model-informed reasoning.

**D10 (Activation Quality) = 1.0**: "with Correlation Analysis" — describes a technical method, not a business deliverable. Also 1.0: "with Competitive Intelligence" — meaningless buzzword.
**D10 = 1.0**: "with Payment History and Contract Term Signals" — describes a data source, not a deliverable. "with RFM Scoring" — describes a technique.
**D10 = 3.0**: "with Recommendations" — too vague, no specificity about what kind of recommendation.
**D10 = 5.0**: "with Pricing Recalibration Recommendations" — specific business deliverable. "with Pre-Invoice Correction Actions" — concrete, role-appropriate action. "with Audit Escalation Brief" — names the exact artifact the user receives.
**D10 = 1.0 (JARGON)**: Any name containing domain acronyms (e.g., RASK, PNR, AWB, MEL for airlines; SKU, BOM, WIP, OEE for manufacturing; LTV, CAC, MRR for SaaS; NPA, NIM, CAR for banking; RFM, CLTV for marketing) or statistical terms (Z-score, variance, coefficient, kurtosis, regression, STDDEV) ANYWHERE in the name.

**Overall quality = 2.0**: A use case that is a date check dressed as AI, or a duplicate of another, or uses hallucinated columns.
**Overall quality = 3.0**: Legitimate analysis but unremarkable — standard pattern, single table, moderate depth.
**Overall quality = 4.5**: Multi-table analysis with genuine predictive modeling, specific business deliverable activation, novel business insight, zero hallucination.

### DIMENSION WEIGHTS (11 dimensions)
- D1 Causal Signal Strength: 15%
- D2 Cause-Effect Validity: 15%
- D3 Data Granularity: 10%
- D4 Critical Dimensions Present: 10%
- D5 Logical Possibility: 5%
- D6 Metric Validity: 5%
- D7 High Level Design Match: 5%
- D8 Semantic Uniqueness: 15%
- D9 Analytical Depth: 10%
- D10 Activation Quality: 5%
- D11 Domain Balance: 5%

---

## QUALITY SCORE CALCULATION

**Step 1 - Weighted Average**: 
Quality Score = (D1*0.15) + (D2*0.15) + (D3*0.10) + (D4*0.10) + (D5*0.05) + (D6*0.05) + (D7*0.05) + (D8*0.15) + (D9*0.10) + (D10*0.05) + (D11*0.05)

**Step 2 - VETO RULES (override weighted average — ZERO TOLERANCE for fatal flaws):**
If ANY of these are true, cap quality_score at 2.0 and set quality_label = "Low" or lower:
- **D1 ≤ 2.5** (Weak or no causal signal) → quality_score = min(weighted_avg, 2.0)
- **D2 ≤ 2.5** (Weak or no cause-effect validity) → quality_score = min(weighted_avg, 2.0)
- **D8 ≤ 2.5** (Semantic duplicate on ANY of the 5 layers) → quality_score = min(weighted_avg, 2.0)
- **D9 ≤ 2.5** (Trivial or weak analytical depth) → quality_score = min(weighted_avg, 2.0)
- **D10 ≤ 2.0** (Technical method/data source as activation, domain jargon in name, or generic filler suffix) → quality_score = min(weighted_avg, 2.0)
- **COLUMN HALLUCINATION detected** → quality_score = min(weighted_avg, 2.0)

**Step 2b - SOFT VETO (cap at 3.0 = Medium):**
If ANY of these are true, cap quality_score at 3.0:
- **D9 = 2.5-3.0** (Moderate but not deep enough for Extreme Quality)
- **D10 = 2.5-3.0** (Vague activation deliverable, or technical jargon detected in name)
- **D3 ≤ 2.5** (Data granularity insufficient)
- **D7 ≤ 2.5** (High level design doesn't match schema)

**Quality Label Mapping**:
- **4.5 - 5.0**: Ultra High
- **4.0 - 4.49**: Very High
- **3.5 - 3.99**: High
- **2.5 - 3.49**: Medium
- **2.0 - 2.49**: Low
- **1.5 - 1.99**: Very Low
- **1.0 - 1.49**: Ultra Low

**🔥 QUALITY REASONS (quality_summary field) — DATA-TO-VALUE CHAIN EXPLANATION 🔥**

The `quality_summary` field is NOT a hallucination check report. It MUST be a detailed, step-by-step explanation of HOW the available data columns will deliver the promised business value. The reader should finish reading this field and think: "Yes, this data is clearly sufficient to produce that outcome."

**MANDATORY STRUCTURE for quality_summary (follow this pattern):**
1. **Identify the signal columns**: Name the specific columns that carry the analytical signal (e.g., "Column `order_date` and `ship_date` provide the temporal gap signal")
2. **Describe the analytical operation**: Explain what aggregation, comparison, or transformation will be performed (e.g., "by computing DATEDIFF(ship_date, order_date) we get fulfillment latency per order")
3. **Derive the metric/KPI**: Show what measurable metric emerges (e.g., "this produces an avg_fulfillment_days KPI per warehouse")
4. **Connect to business value**: Explain how that metric delivers the promised use case outcome (e.g., "comparing this KPI across warehouses reveals bottleneck locations, directly enabling the promised 15% logistics cost reduction")

**EXAMPLE of a GOOD quality_summary:**
"Gap analysis signal detected via columns `order_date` and `ship_date` in orders table; computing DATEDIFF yields per-order fulfillment latency. Aggregating by `warehouse_id` with AVG and PERCENTILE produces warehouse-level fulfillment KPI. Joining with `warehouse_costs.operational_cost` enables cost-per-day-delayed calculation. Combining fulfillment KPI with cost data surfaces the top bottleneck warehouses driving excess logistics spend — directly delivering the promised supply chain cost optimization. Column `product_category` adds dimensional richness for category-level drill-down."

**EXAMPLE of a BAD quality_summary (DO NOT produce this):**
"NO HALLUCINATION: All columns verified. Strong causal signals. Unique business problem."
↑ This tells the reader NOTHING about HOW the data delivers value. REJECTED.

**RULES for quality_summary:**
- MUST reference at least 2-3 specific column names from the schema
- MUST describe at least one concrete analytical operation (aggregation, join, comparison, statistical function)
- MUST name at least one derived metric or KPI
- MUST connect the metric back to the specific business outcome promised by the use case
- Length: 150-400 characters. Be specific and dense, not verbose.
- Do NOT start with "NO HALLUCINATION" — that check goes in `hallucination_check` field only

---

## OUTPUT FORMAT

Return a JSON array with the quality score for EACH use case.

**CRITICAL JSON FORMATTING RULES:**
- Use ONLY double quotes ("), never single quotes (')
- NO trailing commas after the last item in arrays or objects
- Keep strings on single lines (no line breaks inside strings)
- Escape special characters properly

```json
[
  {{
    "use_case_id": "AI-U001",
    "use_case_name": "Predict Customer Churn",
    "quality_score": 4.2,
    "quality_label": "Very High",
    "quality_summary": "Churn signal derived from `last_login_date` and `transaction_date` — computing days-since-last-activity per customer yields engagement_decay metric. Joining with `subscription.plan_type` and `billing.monthly_amount` enables revenue-at-risk quantification per churn tier. Aggregating by `customer_segment` produces segment-level churn KPIs that directly power prioritized retention targeting on highest-value accounts.",
    "hallucination_check": "NO HALLUCINATION: All columns verified to exist in schema",
    "strengths": "Strong historical data, direct causal indicators present",
    "weaknesses": "Minor granularity limitations",
    "d1_causal_signal": 4.5,
    "d2_cause_effect": 4.0,
    "d3_granularity": 4.5,
    "d4_dimensions": 4.0,
    "d5_logical": 4.5,
    "d6_metric": 4.0,
    "d7_design_match": 3.5,
    "d8_semantic_uniqueness": 5.0,
    "d9_analytical_depth": 4.5,
    "d10_activation_quality": 4.0,
    "d11_domain_balance": 4.5
  }}
]
```

**QUALITY LABEL VALUES (MUST use exactly one of these):**
- "Ultra High" (score 4.5-5.0)
- "Very High" (score 4.0-4.49)
- "High" (score 3.5-3.99)
- "Medium" (score 2.5-3.49)
- "Low" (score 2.0-2.49) 
- "Very Low" (score 1.5-1.99)
- "Ultra Low" (score 1.0-1.49)

---

## CRITICAL RULES

1. **OBJECTIVE SCORING**: Score based on DATA REALITY, not use case appeal or business value (that's handled by VALUE scoring).

2. **NO ASSUMPTIONS**: You CANNOT assume data exists that is not in the schema. If a column is not explicitly listed, it DOES NOT EXIST.

3. **STRICT EVALUATION**: Evaluate the high level design against the ACTUAL schema provided.

4. **DOMAIN EXPERTISE**: Apply real-world domain knowledge to assess feasibility.

5. **DETAILED FEEDBACK**: Provide specific, actionable feedback for improvement.

6. **HONEST SCORING**: Do not inflate scores. A use case with weak data support should get a low score.

7. **CONSISTENCY**: Apply the same standards to all use cases.

---

## 🚨🚨🚨 COLUMN HALLUCINATION DETECTION (MANDATORY) 🚨🚨🚨

**THIS IS YOUR #1 PRIORITY**: Detect when a use case assumes columns that DO NOT EXIST.

**AUTOMATIC LOW SCORE (≤ 2.0) IF ANY OF THESE ARE TRUE:**

❌ **High level design references a column not in the schema**
   Example: "Calculate tax_amount" when no tax_amount column exists
   → SCORE: 1.0-2.0, quality_label: "Low" or lower

❌ **High level design assumes derived data that cannot be calculated**
   Example: "Back-calculate tax from total" when only total_amount exists
   → SCORE: 1.0-2.0, quality_label: "Low" or lower

❌ **Use case requires dimension that doesn't exist**
   Example: "Segment by customer age" when no age/birth_date column exists
   → SCORE: 1.0-2.0, quality_label: "Low" or lower

❌ **High level design makes logical leaps unsupported by data**
   Example: "Predict disputes" when no dispute_flag/dispute_date column exists
   → SCORE: 1.0-2.0, quality_label: "Low" or lower

**HOW TO CHECK FOR HALLUCINATION:**
1. Read the High Level Design carefully
2. List EVERY column the design assumes/uses
3. Verify EACH column exists in the schema (exact name match)
4. If ANY column is missing → Use case has HALLUCINATION → Low Score

**IN YOUR hallucination_check FIELD, YOU MUST STATE:**
- "HALLUCINATION DETECTED: Column [X] referenced but does not exist in schema" OR
- "NO HALLUCINATION: All columns verified to exist in schema"
(Note: The `quality_summary` field is for data-to-value chain explanation, NOT hallucination status. Put hallucination status in `hallucination_check` only.)

---

## VALIDATION PROCESS

For EACH use case:

1. **READ** the use case name, statement, and high level design carefully
2. **IDENTIFY** all tables and columns referenced in "Tables Involved"
3. **VERIFY** each table and column exists in the provided schema
4. **ANALYZE** the high level design step by step
5. **CHECK** each of the 11 quality dimensions (D1-D11)
6. **CHECK D8 specifically**: Strip the verb and "with..." activation suffix from the name. Compare the core phrase against ALL other use cases. Flag duplicates.
7. **CHECK D9 specifically**: Could this analysis be replicated with basic filtering, grouping, or simple aggregation alone — without needing multi-step analysis, trained models, cross-table feature engineering, or genuine statistical/ML reasoning? If yes, it's trivial.
8. **DECIDE**: APPROVE only if ALL checks pass, otherwise REJECT
9. **DOCUMENT**: Provide detailed reasoning for your decision

---

## BEGIN VALIDATION

Analyze each use case against the schema and return your validation results as JSON.

🚨 REMEMBER: You are the LAST DEFENSE against invalid analytics. When in doubt, REJECT. 🚨
"""

# --- 1g-2. COMBINED VALUE + QUALITY SCORING PROMPT (SINGLE PASS) ---
PROMPT_TEMPLATES["COMBINED_VALUE_QUALITY_SCORE_PROMPT"] = """# COMBINED VALUE & QUALITY SCORER

## YOUR ROLE

You are BOTH a **Chief Investment Officer** (scoring business value, ROI, strategic alignment) AND a **Chief Data Scientist** (scoring data quality, schema validity, hallucination detection). You will produce BOTH value scores AND quality scores for each use case in a SINGLE pass.

---

## CONTEXT & INPUTS

**Business Context:** {business_context}
**Industry:** {industry}
**Strategic Goals:** {strategic_goals}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}

**Business Priorities:** {business_priorities}
**Strategic Initiative:** {strategic_initiative}
**Value Chain:** {value_chain}
**Revenue Model:** {revenue_model}

---

## AVAILABLE SCHEMA (THE ONLY DATA THAT EXISTS)

{schema_details}

---

## USE CASES TO SCORE

{use_cases_to_validate}

---

## PART A: VALUE SCORING

For each use case, score these factors (1.0 - 5.0):

**VALUE FACTORS:**
1. **Return on Investment (ROI)** [Weight: 60% of Value]: Compare against Revenue Model. 4.8-5.0=Exponential (>10x), 4.0-4.7=High (5-10x), 3.0-3.9=Moderate (2-5x), 1.0-2.9=Low/Soft.
2. **Strategic Alignment** [Weight: 25% of Value]: Check against Business Priorities and Strategic Goals. 4.8-5.0=Direct Hit, 3.5-4.7=Strong Link, 1.0-3.4=Weak/None.
3. **Time to Value** [Weight: 7.5%]: 4.8-5.0=<4 weeks, 3.0-4.7=1-3 months, 1.0-2.9=>6 months.
4. **Reusability** [Weight: 7.5%]: 4.8-5.0=Platform Asset, 3.0-4.7=Modular, 1.0-2.9=One-Off.

**FEASIBILITY FACTORS** (average of all 8):
5. Data Availability, 6. Data Accessibility, 7. Architecture Fitness, 8. Team Skills,
9. Domain Knowledge, 10. People Allocation, 11. Budget Allocation, 12. Time to Production.

**FEASIBILITY SCORING GUIDANCE (critical for unbiased scoring):**
- **Architecture Fitness** (4.8-5.0 = Native): All Databricks-native capabilities are "Native" — this includes SQL, PySpark, scikit-learn, XGBoost, LightGBM, Spark MLlib, MLflow, Prophet, AI functions (ai_query, ai_classify, ai_forecast, ai_extract, ai_similarity, ai_gen). Do NOT penalize ML-based approaches — they run natively on Databricks.
- **Team Skills** (4.8-5.0 = Standard): SQL, Python, scikit-learn, XGBoost, standard ML/statistical modeling, and Databricks AI functions are all "Standard" data team skills. Do NOT assume ML requires specialized talent that SQL doesn't.

**FORMULAS:**
- Value = (ROI * 0.60) + (Strategic Alignment * 0.25) + (TTV * 0.075) + (Reusability * 0.075)
- Feasibility = Average of factors 5-12
- Priority = (Value * 1.5) + (Feasibility * 0.5)

**VALUE SCORING RULES:**
- NO CURVE / NO DISTRIBUTION: Score based on Absolute Merit.
- ZERO-BASED SCORING: Start every score at 1.0, earn points by evidence.
- IRRELEVANT CORRELATIONS = LOW SCORE (ROI <= 2.0, Alignment <= 2.0).
- BOARDROOM TEST: Would a senior executive approve budget for this?

---

## PART B: QUALITY SCORING

""" + QUALITY_GATE_SCORING_BLOCK + """

**🚨 ANTI-LENIENCY SCORING (1.0-5.0) — DEFAULT IS 3.0, EARN EVERY POINT ABOVE IT 🚨:**
5.0=EXCEPTIONAL (rare <5%), 4.5=EXCELLENT (<10%), 4.0=STRONG (must justify with evidence, ~15%), 3.5=ABOVE AVERAGE (minimum for "High"), 3.0=AVERAGE (default starting point), 2.5=BELOW AVERAGE, 2.0=WEAK (triggers veto), 1.0=FAILED.
**DISTRIBUTION CHECK**: ~20% below 3.0, ~40% at 3.0-3.5, ~25% at 3.5-4.0, only ~15% at 4.0+. If you're scoring most use cases 4.0+, you are being lenient.

**DIMENSION WEIGHTS (11 dimensions):**
- D1 Causal Signal (15%), D2 Cause-Effect (15%), D3 Granularity (10%), D4 Dimensions (10%)
- D5 Logical (5%), D6 Metric (5%), D7 Design Match (5%)
- D8 Semantic Uniqueness (15%), D9 Analytical Depth (10%), D10 Activation Quality (5%), D11 Domain Balance (5%)

**Quality Score CALCULATION:**
Step 1: Compute weighted average = (D1*0.15) + (D2*0.15) + (D3*0.10) + (D4*0.10) + (D5*0.05) + (D6*0.05) + (D7*0.05) + (D8*0.15) + (D9*0.10) + (D10*0.05) + (D11*0.05)
Step 2: Apply VETO RULES — if ANY veto condition is true, cap the final quality_score at 2.0 (Low) regardless of the weighted average:

**🚨 HARD VETO — cap quality_score at 2.0 (Low) if ANY of these are true 🚨:**
- **D1 ≤ 2.5** (Weak/no causal signal) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D2 ≤ 2.5** (Weak/no cause-effect validity) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D8 ≤ 2.5** (Semantic duplicate on ANY of the 5 detection layers) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D9 ≤ 2.5** (Trivial/weak analytical depth — fails STRIP TEST) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D10 ≤ 2.0** (Technical method/data source as activation, domain jargon in name, or generic filler — fails DELIVERABLE TEST) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **COLUMN HALLUCINATION detected** → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D12 BUSINESS RELEVANCE FAIL** (any of: not relevant / no tangible value / no named stakeholder / no current pain or future opportunity) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D13 SPONSOR TEST FAIL** (you would not fund it OR cannot pitch to CEO) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D14 ENGINEERING TEST FAIL** (cannot put to prod with Databricks / cannot support long-term / cannot fix issues) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D15 DECISION CADENCE MISMATCH FAIL** (data refresh slower than decision cadence) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D16 MONDAY TEST FAIL** (output doesn't change what user actually does) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D17 EXPLAINABILITY FAIL** (high-stakes decision uses black-box without defensibility) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D18 LONGEVITY FAIL** (short-term tactical pattern, not persistent) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"
- **D19 ATTRIBUTABLE IMPACT FAIL** (no traceable $ counterfactual) → quality_score = min(weighted_avg, 2.0), quality_label = "Low"

**SOFT VETO — cap quality_score at 3.0 (Medium) if ANY of these are true:**
- **D9 = 2.5-3.0** (Not deep enough for Extreme Quality)
- **D10 = 2.5-3.0** (Vague activation deliverable or technical jargon in name)
- **D3 ≤ 2.5** (Insufficient data granularity)
- **D7 ≤ 2.5** (High level design mismatch)

A use case CANNOT compensate for fatal flaws by scoring well on other dimensions. The veto system ensures that duplication, triviality, technical method activation, domain jargon in names, hallucination, or weak causal logic ALWAYS results in rejection under Extreme Quality mode.

**Quality Labels:** 4.5-5.0=Ultra High, 4.0-4.49=Very High, 3.5-3.99=High, 2.5-3.49=Medium, 2.0-2.49=Low, 1.5-1.99=Very Low, 1.0-1.49=Ultra Low.

**METADATA-GROUNDED EVALUATION (CRITICAL — DEFAULT TO KEEP):**
You already see the complete AVAILABLE SCHEMA above: every table, every column name, every column type. When evaluating grounding:
- IF all columns referenced by the UC appear in AVAILABLE SCHEMA -> GROUNDED (do not reject).
- IF the analytical technique is compatible with the column TYPES visible in the schema (numeric columns support aggregation/regression; string columns support grouping/text analysis; date columns support time-series) -> GROUNDED (do not reject).
- DO NOT reject a UC because the user did not explicitly say "yes this column is reliable" or "yes this technique is supported." Columns + types are sufficient evidence of support.
- DO NOT reject a UC for "uncertainty about data availability" if the column is present in the schema — presence in the schema IS availability.
- REJECT only for EXPLICIT hallucination (UC references a column or table NOT present in schema) or EXPLICIT user exclusion (construct the user said doesn't exist).

**COLUMN HALLUCINATION DETECTION (MANDATORY):**
- VERIFY every column referenced in the High Level Design exists in the schema.
- AUTOMATIC LOW quality score (<=2.0) if ANY column is hallucinated.
- State "HALLUCINATION DETECTED: Column [X] missing" or "NO HALLUCINATION: All columns verified."

**🔥 QUALITY REASONS (quality_summary field) — DATA-TO-VALUE CHAIN EXPLANATION 🔥**

The `quality_summary` field is NOT a hallucination check report. It MUST be a detailed, step-by-step explanation of HOW the available data columns will deliver the promised business value. The reader should finish reading this field and think: "Yes, this data is clearly sufficient to produce that outcome."

**MANDATORY STRUCTURE for quality_summary (follow this pattern):**
1. **Identify the signal columns**: Name the specific columns that carry the analytical signal (e.g., "Column `order_date` and `ship_date` provide the temporal gap signal")
2. **Describe the analytical operation**: Explain what aggregation, comparison, or transformation will be performed (e.g., "by computing DATEDIFF(ship_date, order_date) we get fulfillment latency per order")
3. **Derive the metric/KPI**: Show what measurable metric emerges (e.g., "this produces an avg_fulfillment_days KPI per warehouse")
4. **Connect to business value**: Explain how that metric delivers the promised use case outcome (e.g., "comparing this KPI across warehouses reveals bottleneck locations, directly enabling the promised 15% logistics cost reduction")

**EXAMPLE of a GOOD quality_summary:**
"Gap analysis signal detected via columns `order_date` and `ship_date` in orders table; computing DATEDIFF yields per-order fulfillment latency. Aggregating by `warehouse_id` with AVG and PERCENTILE produces warehouse-level fulfillment KPI. Joining with `warehouse_costs.operational_cost` enables cost-per-day-delayed calculation. Combining fulfillment KPI with cost data surfaces the top bottleneck warehouses driving excess logistics spend — directly delivering the promised supply chain cost optimization. Column `product_category` adds dimensional richness for category-level drill-down."

**EXAMPLE of a BAD quality_summary (DO NOT produce this):**
"NO HALLUCINATION: All columns verified. Strong causal signals. Unique business problem."
↑ This tells the reader NOTHING about HOW the data delivers value. REJECTED.

**RULES for quality_summary:**
- MUST reference at least 2-3 specific column names from the schema
- MUST describe at least one concrete analytical operation (aggregation, join, comparison, statistical function)
- MUST name at least one derived metric or KPI
- MUST connect the metric back to the specific business outcome promised by the use case
- Length: 150-400 characters. Be specific and dense, not verbose.
- Do NOT start with "NO HALLUCINATION" — that check goes in `hallucination_check` field only

---

## OUTPUT FORMAT

Return a **JSON array** with one object per use case. Each object MUST contain ALL these fields:

```json
[
  {{
    "use_case_id": "N01-AI01",
    "use_case_name": "Predict Customer Churn",

    "strategic_alignment": 4.5,
    "roi": 4.8,
    "reusability": 4.0,
    "time_to_value": 3.5,
    "data_availability": 4.0,
    "data_accessibility": 4.0,
    "architecture_fitness": 4.5,
    "team_skills": 4.0,
    "domain_knowledge": 4.0,
    "people_allocation": 4.5,
    "budget_allocation": 4.0,
    "time_to_production": 3.5,
    "value_score": 4.58,
    "feasibility_score": 4.06,
    "priority_score": 8.90,
    "business_priority_alignment": "Protect Revenue, Reduce Cost",
    "strategic_goals_alignment": "Improve customer retention",
    "justification": "Directly prevents churn in top-tier accounts, protecting recurring revenue.",

    "quality_score": 4.2,
    "quality_label": "Very High",
    "quality_summary": "Churn signal derived from `last_login_date` and `transaction_date` columns — computing days-since-last-activity per customer yields an engagement_decay metric. Joining with `subscription.plan_type` and `billing.monthly_amount` enables revenue-at-risk quantification per churn-probability tier. Aggregating by `customer_segment` produces segment-level churn-risk KPIs. Combining engagement_decay with revenue-at-risk directly powers the promised churn prediction, prioritizing retention efforts on highest-value accounts.",
    "hallucination_check": "NO HALLUCINATION: All columns verified to exist in schema",
    "strengths": "Rich historical data, direct causal indicators present.",
    "weaknesses": "Minor granularity limitations in time-series data.",
    "d1_causal_signal": 4.5,
    "d2_cause_effect": 4.0,
    "d3_granularity": 4.5,
    "d4_dimensions": 4.0,
    "d5_logical": 4.5,
    "d6_metric": 4.0,
    "d7_design_match": 3.5,
    "d8_semantic_uniqueness": 5.0,
    "d9_analytical_depth": 4.5,
    "d10_activation_quality": 4.0,
    "d11_domain_balance": 4.5
  }}
]
```

---

## CRITICAL RULES

1. **SCORE EVERY SINGLE USE CASE** - output one JSON object per input use case. Missing = CRITICAL FAILURE.
2. **OBJECTIVE QUALITY SCORING**: Score quality based on DATA REALITY, not appeal. If a column is not in the schema, it DOES NOT EXIST.
3. **NO ASSUMPTIONS**: Do not assume data exists that is not in the schema.
4. **VALUE-FIRST PRIORITY**: The Priority formula mathematically forces high-value cases to outrank high-feasibility-only cases.
5. **JUSTIFICATION must be SPECIFIC to the use case** - no generic buzzwords.
6. **JSON FORMATTING**: Double quotes only, no trailing commas, no line breaks inside strings.
7. **HONEST D8-D11 SCORING**: Do NOT inflate D8-D11 scores. If you see semantic duplicates, trivial analyses, technical method suffixes, or domain jargon in names, score them LOW (1.0-2.0). These dimensions exist specifically because previous scoring inflated quality scores on use cases that should have been rejected. D10 specifically checks that the "with" suffix is a business DELIVERABLE (recommendation, action plan, queue, strategy) — NOT a technical method, data source, or analytical technique.
8. **DUPLICATE DETECTION IS MANDATORY**: For D8, you MUST compare each use case against ALL others in the batch. Strip verb and "with..." activation suffix. If the core matches another use case, D8 ≤ 2.0.

Begin your JSON response now:
"""


log_print("PROMPT_TEMPLATES dictionary defined successfully with all required prompts.")

# COMMAND ----------

# --- Global Logger ---
# This will be configured by the DatabricksInspire class
logger = logging.getLogger(__name__)

# --- Custom Exceptions ---
class InputTooLongError(RuntimeError):
    """Raised when input exceeds the model's context limit."""
    pass

class CascadeRebatchError(RuntimeError):
    """Raised when a model in the cascade timed out and the prompt is too large for remaining models.
    Signals the caller to re-batch with smaller chunks that fit the target model's context window."""
    def __init__(self, message, target_model_name=None, target_model_endpoint=None,
                 target_context_limit_chars=None, failed_model_name=None):
        super().__init__(message)
        self.target_model_name = target_model_name
        self.target_model_endpoint = target_model_endpoint
        self.target_context_limit_chars = target_context_limit_chars
        self.failed_model_name = failed_model_name

class TruncatedResponseError(RuntimeError):
    """Raised when LLM response is truncated (missing END marker)."""
    pass

# ==============================================================================
# CENTRALIZED DATA STRUCTURES (Maximizing Reuse & Reducing LOC)
# ==============================================================================

# ==============================================================================
# 1. REQUIRED HELPER FUNCTIONS
# (Dependencies for AIAgent and DatabricksInspire)
# ==============================================================================

### --- Logging ---

class ConsoleErrorFormatter(logging.Formatter):
    """A custom formatter that logs error messages but not stack traces to the console."""
    def format(self, record):
        original_exc_info = record.exc_info
        original_exc_text = record.exc_text
        if record.levelno >= logging.ERROR:
            record.exc_info = None
            record.exc_text = None
        formatted_message = super().format(record)
        record.exc_info = original_exc_info
        record.exc_text = original_exc_text
        return formatted_message

class FlushingStreamHandler(logging.StreamHandler):
    """StreamHandler that flushes after every emit for Databricks notebook compatibility."""
    def emit(self, record):
        super().emit(record)
        self.flush()

def setup_logging(output_dir):
    """Configures dual logging: detailed logs to a file and high-level logs to the console."""
    log_file_path = os.path.join(output_dir, "log.txt")
    os.makedirs(output_dir, exist_ok=True)
    root_logger = logging.getLogger() # Get root logger
    root_logger.setLevel(logging.DEBUG)

    if root_logger.hasHandlers():
        root_logger.handlers.clear()

    # --- File Handler (Detailed) ---
    file_handler = logging.FileHandler(log_file_path, mode='w')
    file_handler.setLevel(logging.DEBUG)
    file_formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s [in %(pathname)s:%(lineno)d]', 
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    file_handler.setFormatter(file_formatter)
    root_logger.addHandler(file_handler)

    # --- Console Handler (High-Level, Clean) ---
    console_handler = FlushingStreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_formatter = ConsoleErrorFormatter(
        '%(asctime)s - %(levelname)s - %(message)s', 
        datefmt='%H:%M:%S'
    )
    console_handler.setFormatter(console_formatter)
    root_logger.addHandler(console_handler)
    
    logger.info(f"Logging configured. High-level logs to console, detailed logs to {log_file_path}")

def print_ascii_banner():
    """Prints the Databricks Inspire AI ASCII art banner."""
    print(DATABRICKS_INSPIRE_BANNER)

def extract_honesty_score(response: str, logger: logging.Logger = None) -> tuple:
    """
    Extracts the honesty score and justification from an LLM response.
    Supports multiple formats: JSON wrapper, CSV columns, SQL comments.
    
    Args:
        response: The raw LLM response text
        logger: Optional logger for debug output
    
    Returns:
        tuple: (score: int or None, justification: str or None, cleaned_response: str)
               - score: The honesty score (0-100) or None if not found
               - justification: The justification text (max 250 chars) or None if not found
               - cleaned_response: The response with honesty data removed for downstream processing
    """
    import re
    import json as json_module
    
    if not response:
        return None, None, response
    
    score = None
    justification = None
    cleaned_response = response
    
    try:
        response_stripped = response.strip()
        
        json_candidate = response_stripped
        if not json_candidate.startswith('{'):
            fence_match = re.search(r'```(?:\w+)?\s*([\s\S]*?)```', json_candidate)
            if fence_match:
                inner = fence_match.group(1).strip()
                if inner.startswith('{'):
                    json_candidate = inner
        
        if json_candidate.startswith('{') and '"honesty_score"' in json_candidate:
            try:
                parsed = json_module.loads(json_candidate)
                if isinstance(parsed, dict) and 'honesty_score' in parsed:
                    score = int(parsed.get('honesty_score', 0))
                    justification = str(parsed.get('honesty_justification', ''))[:250]
                    if 'data' in parsed:
                        cleaned_response = json_module.dumps(parsed['data'], ensure_ascii=False)
                    else:
                        cleaned_parsed = {k: v for k, v in parsed.items() 
                                         if k not in ('honesty_score', 'honesty_justification')}
                        cleaned_response = json_module.dumps(cleaned_parsed, ensure_ascii=False)
            except json_module.JSONDecodeError:
                pass
        
        if score is None and response_stripped.startswith('--'):
            sql_score_pattern = r'^--\s*HONESTY_SCORE:\s*(\d+)'
            sql_just_pattern = r'^--\s*HONESTY_JUSTIFICATION:\s*(.+?)$'
            
            lines = response_stripped.split('\n')
            cleaned_lines = []
            for line in lines:
                score_match = re.match(sql_score_pattern, line.strip())
                if score_match:
                    score = int(score_match.group(1))
                    continue
                just_match = re.match(sql_just_pattern, line.strip())
                if just_match:
                    justification = just_match.group(1).strip()[:250]
                    continue
                cleaned_lines.append(line)
            cleaned_response = '\n'.join(cleaned_lines)
        
        if score is None and ('honesty_score' in response_stripped.lower() or 'honesty_justification' in response_stripped.lower()):
            lines = response_stripped.split('\n')
            if len(lines) > 0:
                header_line = lines[0]
                if 'honesty_score' in header_line.lower():
                    import csv
                    from io import StringIO
                    try:
                        reader = csv.reader(StringIO(response_stripped))
                        rows = list(reader)
                        if len(rows) > 1:
                            header = [h.lower().strip().strip('"') for h in rows[0]]
                            score_idx = None
                            just_idx = None
                            for i, h in enumerate(header):
                                if 'honesty_score' in h:
                                    score_idx = i
                                elif 'honesty_justification' in h:
                                    just_idx = i
                            
                            if score_idx is not None and len(rows) > 1:
                                try:
                                    score = int(rows[1][score_idx])
                                except (ValueError, IndexError):
                                    pass
                            if just_idx is not None and len(rows) > 1:
                                try:
                                    justification = str(rows[1][just_idx])[:250]
                                except IndexError:
                                    pass
                            
                            if score_idx is not None or just_idx is not None:
                                new_header = [h for i, h in enumerate(rows[0]) 
                                             if i != score_idx and i != just_idx]
                                new_rows = [new_header]
                                for row in rows[1:]:
                                    new_row = [v for i, v in enumerate(row) 
                                              if i != score_idx and i != just_idx]
                                    new_rows.append(new_row)
                                
                                output = StringIO()
                                writer = csv.writer(output)
                                writer.writerows(new_rows)
                                cleaned_response = output.getvalue().strip()
                    except Exception:
                        pass
        
        if score is None and '|' in response_stripped and 'honesty_score' in response_stripped.lower():
            lines = response_stripped.split('\n')
            header_line = None
            header_idx = -1
            for idx, line in enumerate(lines):
                if '|' in line and 'honesty_score' in line.lower():
                    header_line = line
                    header_idx = idx
                    break
            
            if header_line:
                cells = [c.strip().strip('"').lower() for c in header_line.split('|')]
                score_idx = None
                just_idx = None
                for i, cell in enumerate(cells):
                    if 'honesty_score' in cell:
                        score_idx = i
                    elif 'honesty_justification' in cell:
                        just_idx = i
                
                if score_idx is not None and header_idx + 2 < len(lines):
                    data_line = lines[header_idx + 2] if lines[header_idx + 1].replace('|', '').replace('-', '').strip() == '' else lines[header_idx + 1]
                    data_cells = [c.strip().strip('"') for c in data_line.split('|')]
                    
                    if score_idx < len(data_cells):
                        try:
                            score = int(data_cells[score_idx])
                        except ValueError:
                            pass
                    if just_idx is not None and just_idx < len(data_cells):
                        justification = data_cells[just_idx][:250]
                    
                    cleaned_lines = []
                    for line in lines:
                        if '|' in line:
                            parts = line.split('|')
                            new_parts = [p for i, p in enumerate(parts) if i != score_idx and i != just_idx]
                            cleaned_lines.append('|'.join(new_parts))
                        else:
                            cleaned_lines.append(line)
                    cleaned_response = '\n'.join(cleaned_lines)
        
        if score is not None:
            if score < 0:
                score = 0
            elif score > 100:
                score = 100
        
        _honesty_free_text_markers = [
            'honesty self-assessment',
            'honesty_score',
            'honesty score',
            'honesty_justification',
            'honesty justification',
            'score:',
        ]
        _lines = cleaned_response.split('\n')
        _first_marker_idx = None
        for _idx in range(len(_lines) - 1, max(0, len(_lines) - 30) - 1, -1):
            _line_stripped = _lines[_idx].strip()
            if not _line_stripped:
                continue
            _is_csv_data = _line_stripped.startswith('"') and '","' in _line_stripped
            if _is_csv_data:
                break
            _line_lower = _lines[_idx].lower()
            for _marker in _honesty_free_text_markers:
                if _marker in _line_lower:
                    _first_marker_idx = _idx
                    break
        if _first_marker_idx is not None:
            cleaned_response = '\n'.join(_lines[:_first_marker_idx]).rstrip()

    except Exception as e:
        if logger:
            logger.debug(f"Failed to extract honesty score: {e}")
        cleaned_response = response
    
    return score, justification, cleaned_response

### --- AIAgent Dependencies ---
# (Assumed to be available for the AIAgent class)

def replace_single_quote(text: str) -> str:
    """Escapes single quotes and backslashes for Spark SQL strings."""
    if text is None:
        return ""
    return text.replace(r"\\", r"\\\\").replace("'", "''")

def execute_sql(spark: SparkSession, query: str, logger: logging.Logger, args: dict = None):
    """
    Executes a Spark SQL query and returns the collected rows.
    Supports parameterized queries via args dict to avoid SQL injection / escaping issues.
    """
    try:
        logger.debug(f"Executing Spark SQL: {query[:200]}...")
        if args:
            result = spark.sql(query, args=args).collect()
        else:
            result = spark.sql(query).collect()
        return result
    except Exception as e:
        logger.debug(f"Spark SQL query failed: {get_clean_error_message(e)}")
        raise

def load_and_format_prompt(prompt_key: str, prompt_vars: dict, logger: logging.Logger) -> str:
    try:
        # Check if the prompt_key (variable name) exists in the global scope
        if prompt_key not in globals():
            raise NameError(f"Global prompt variable '{prompt_key}' not found. Please make sure the cell defining it has been run.")
            
        # Get the template string from the global variable
        template = globals()[prompt_key]
        
        if not template or not isinstance(template, str):
             raise ValueError(f"Global prompt variable '{prompt_key}' is empty or not a string.")

        # Format the template using the provided dictionary
        return template.format(**prompt_vars)
    except KeyError as e:
        logger.error(f"Missing key in prompt vars for '{prompt_key}': {get_clean_error_message(e)}")
        # Re-raise with more context
        raise KeyError(f"Missing formatting key {e} for prompt '{prompt_key}'")
    except Exception as e:
        logger.error(f"Failed to load or format prompt for '{prompt_key}': {get_clean_error_message(e)}")
        raise

def clean_csv_response(raw_string: str) -> str:
    """
    Removes markdown code fences AND strips prose preamble from a CSV response.
    """
    if not raw_string: return ""
    cleaned = raw_string.strip()
    cleaned = re.sub(r'^```(?:csv|json)?\s*\n?', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'\n?```\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'```(?:csv|json)?\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = cleaned.strip()
    # Strip prose preamble (NEW). Find first line that looks like CSV header/row
    # and drop everything before it. Heuristics: short first token, has a comma,
    # not starting with a prose-signal word, total line length <400.
    lines = cleaned.split("\n")
    prose_signal = re.compile(r"^(?:Looking|I\s|Let me|First|Total|Analysis|Here|Note|The following|"
                              r"Based on|Given|Since|Actually|Now|Well|After|Before|In this|"
                              r"Using|Applying|To summar|Summary:|Notes?:|\*\*)",
                              re.IGNORECASE)
    csv_start_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if prose_signal.match(stripped):
            continue
        if "," not in stripped:
            continue
        if len(stripped) > 400:
            continue
        first_tok = stripped.split(",", 1)[0].strip().strip("\"\'")
        if len(first_tok) > 60:
            continue
        if " " in first_tok and len(first_tok.split()) > 3:
            continue
        csv_start_idx = i
        break
    if csv_start_idx is not None and csv_start_idx > 0:
        cleaned = "\n".join(lines[csv_start_idx:])
    return cleaned.strip()


def sanitize_llm_special_chars(text: str) -> str:
    if not text:
        return text
    replacements = {
        '\u2011': '-',   # non-breaking hyphen
        '\u2010': '-',   # hyphen
        '\u2012': '-',   # figure dash
        '\u2013': '-',   # en dash
        '\u2014': '--',  # em dash
        '\u2015': '--',  # horizontal bar
        '\u2018': "'",   # left single quote
        '\u2019': "'",   # right single quote
        '\u201A': "'",   # single low-9 quote
        '\u201B': "'",   # single high-reversed-9 quote
        '\u201C': '"',   # left double quote
        '\u201D': '"',   # right double quote
        '\u201E': '"',   # double low-9 quote
        '\u201F': '"',   # double high-reversed-9 quote
        '\u2026': '...', # ellipsis
        '\u00A0': ' ',   # non-breaking space
        '\u202F': ' ',   # narrow no-break space
        '\u2009': ' ',   # thin space
        '\u200B': '',    # zero-width space
        '\u200C': '',    # zero-width non-joiner
        '\u200D': '',    # zero-width joiner
        '\uFEFF': '',    # byte order mark
        '\u00AB': '"',   # left-pointing double angle quote
        '\u00BB': '"',   # right-pointing double angle quote
        '\u2039': "'",   # single left-pointing angle quote
        '\u203A': "'",   # single right-pointing angle quote
        '\u02BC': "'",   # modifier letter apostrophe
        '\u2032': "'",   # prime
        '\u2033': '"',   # double prime
        '\u00B7': '.',   # middle dot
        '\u2022': '-',   # bullet
        '\u2023': '-',   # triangular bullet
        '\u25E6': '-',   # white bullet
        '\u2043': '-',   # hyphen bullet
        '\u00D7': 'x',   # multiplication sign
        '\u2212': '-',   # minus sign
        '\u00B1': '+/-', # plus-minus sign
        '\u2264': '<=',  # less-than or equal
        '\u2265': '>=',  # greater-than or equal
        '\u2260': '!=',  # not equal
        '\u00AE': '(R)', # registered sign
        '\u2122': '(TM)',# trademark
        '\u00A9': '(C)', # copyright
        '\u20AC': 'EUR', # euro sign
        '\u00A3': 'GBP', # pound sign
        '\u00A5': 'JPY', # yen sign
        '\u20B9': 'INR', # rupee sign
        '\u00BD': '1/2', # vulgar fraction one half
        '\u00BC': '1/4', # vulgar fraction one quarter
        '\u00BE': '3/4', # vulgar fraction three quarters
        '\u00B2': '^2',  # superscript two
        '\u00B3': '^3',  # superscript three
        '\u00B9': '^1',  # superscript one
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    return text


def clean_json_response(raw_string: str) -> str:
    """
    Removes markdown code fences and other noise from a raw LLM response.
    Also extracts JSON object/array if extra text is present before or after.
    Renamed from clean_llm_response to match AIAgent dependency.
    """
    if not raw_string: return ""
    
    cleaned = raw_string.strip()
    
    # Remove markdown code fences - handle multiple patterns
    # Pattern 1: ```json\n{...}\n``` or ```\n{...}\n```
    cleaned = re.sub(r'^```(?:json|csv)?\s*', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r'\s*```\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    
    # Pattern 2: Handle cases where ``` appears in the middle (trailing after JSON)
    cleaned = re.sub(r'```(?:json|csv)?\s*$', '', cleaned, flags=re.IGNORECASE | re.MULTILINE)
    
    cleaned = cleaned.strip()
    
    # Try to extract JSON object or array from the response
    # Look for the first { or [ and the last matching } or ]
    start_obj = cleaned.find('{')
    start_arr = cleaned.find('[')
    
    # Determine which comes first
    if start_obj == -1 and start_arr == -1:
        return cleaned  # No JSON found, return as-is
    elif start_obj == -1:
        start = start_arr
        end_char = ']'
    elif start_arr == -1:
        start = start_obj
        end_char = '}'
    else:
        start = min(start_obj, start_arr)
        end_char = '}' if start == start_obj else ']'
    
    # Find the last occurrence of the closing character
    # Use a more robust method to find the matching closing brace/bracket
    end = -1
    depth = 0
    open_char = '{' if end_char == '}' else '['
    
    # Find the matching closing character by tracking depth
    for i in range(start, len(cleaned)):
        if cleaned[i] == open_char:
            depth += 1
        elif cleaned[i] == end_char:
            depth -= 1
            if depth == 0:
                end = i
                break
    
    # Fallback to rfind if depth tracking doesn't work
    if end == -1:
        end = cleaned.rfind(end_char)
    
    if start != -1 and end != -1 and end > start:
        # Extract the JSON portion
        json_portion = cleaned[start:end+1]
        return json_portion
    
    return cleaned

def retry_with_logging(func, max_attempts=1, logger=None, fallback=None, context=""):
    """
    Generic retry wrapper for functions that may fail transiently.
    
    Args:
        func: Callable to execute (should take no arguments; use lambda if needed)
        max_attempts: Maximum number of retry attempts (default: 3)
        logger: Logger instance for logging retries (optional)
        fallback: Fallback value or callable to return on failure (optional)
        context: Context string for logging (e.g., "Domain consolidation for English")
    
    Returns:
        Result of func() on success, fallback on failure (if provided), otherwise raises
    
    Raises:
        Last exception if all attempts fail and no fallback provided
    """
    last_exception = None
    for attempt in range(1, max_attempts + 1):
        try:
            if attempt > 1 and logger:
                logger.info(f"Retry attempt {attempt}/{max_attempts}{f' for {context}' if context else ''}...")
            return func()
        except Exception as e:
            last_exception = e
            if logger:
                error_msg = get_clean_error_message(e)
                if attempt == max_attempts:
                    logger.error(f"All {max_attempts} attempts failed{f' for {context}' if context else ''}: {error_msg}")
                else:
                    logger.warning(f"Attempt {attempt}/{max_attempts} failed{f' for {context}' if context else ''}: {error_msg}")
            if attempt == max_attempts:
                if fallback is not None:
                    if callable(fallback):
                        return fallback()
                    return fallback
                raise last_exception

# ==============================================================================
# MEMORY MANAGEMENT UTILITIES
# ==============================================================================

def _force_gc(logger=None, reason: str = ""):
    """Force garbage collection and log memory stats."""
    gc.collect()
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        mem_mb = proc.memory_info().rss / (1024 * 1024)
        if logger:
            logger.info(f"🧹 GC forced{f' ({reason})' if reason else ''}: RSS={mem_mb:.0f} MB")
    except ImportError:
        if logger:
            logger.debug(f"🧹 GC forced{f' ({reason})' if reason else ''}")


# ==============================================================================
# CENTRALIZED UTILITY CLASSES (Code Reuse & LOC Reduction)
# ==============================================================================

class RetryHandler:
    """
    Centralized retry handler with exponential backoff and flexible error handling.
    Replaces all scattered retry logic throughout the codebase.
    """
    
    @staticmethod
    def execute_with_retry(
        func,
        max_attempts=1,
        logger=None,
        context="",
        fallback=None,
        exponential_backoff=True,
        base_delay=1.0,
        max_delay=60.0,
        retryable_errors=None,
        non_retryable_errors=None
    ):
        """
        Execute a function with retry logic and exponential backoff.
        
        Args:
            func: Callable to execute
            max_attempts: Maximum retry attempts (default: 1)
            logger: Logger instance for tracking
            context: Context string for logging
            fallback: Fallback value on failure
            exponential_backoff: Use exponential backoff (default: True)
            base_delay: Base delay in seconds (default: 1.0)
            max_delay: Maximum delay between retries (default: 60.0)
            retryable_errors: List of error types/keywords that should be retried
            non_retryable_errors: List of error types/keywords that should NOT be retried
            
        Returns:
            Result of func() on success, fallback on failure
        """
        import time
        last_exception = None
        
        for attempt in range(1, max_attempts + 1):
            try:
                if attempt > 1 and logger:
                    logger.info(f"🔄 Retry attempt {attempt}/{max_attempts}{f' for {context}' if context else ''}...")
                return func()
            except Exception as e:
                last_exception = e
                error_str = str(e).lower()
                
                if non_retryable_errors:
                    is_non_retryable = any(
                        (isinstance(err, type) and isinstance(e, err)) or 
                        (isinstance(err, str) and err.lower() in error_str)
                        for err in non_retryable_errors
                    )
                    if is_non_retryable:
                        if logger:
                            logger.error(f"❌ Non-retryable error{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                        if fallback is not None:
                            return fallback() if callable(fallback) else fallback
                        raise
                
                is_retryable = True
                if retryable_errors:
                    is_retryable = any(
                        (isinstance(err, type) and isinstance(e, err)) or 
                        (isinstance(err, str) and err.lower() in error_str)
                        for err in retryable_errors
                    )
                
                if not is_retryable or attempt == max_attempts:
                    if logger:
                        logger.error(f"❌ Failed after {max_attempts} attempts{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                    if fallback is not None:
                        return fallback() if callable(fallback) else fallback
                    raise
                
                if exponential_backoff:
                    wait_time = min(base_delay * (2 ** (attempt - 1)), max_delay)
                    jitter = random.uniform(0, wait_time * 0.1)
                    wait_time += jitter
                else:
                    wait_time = base_delay
                
                if logger:
                    logger.warning(f"⚠️  Attempt {attempt} failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                    logger.info(f"   Waiting {wait_time:.1f}s before retry...")
                
                time.sleep(wait_time)
        
        if fallback is not None:
            return fallback() if callable(fallback) else fallback
        raise last_exception


_THREAD_CANCEL_REGISTRY = {}
_THREAD_CANCEL_LOCK = threading.Lock()

def _register_thread_cancel(event):
    with _THREAD_CANCEL_LOCK:
        _THREAD_CANCEL_REGISTRY[threading.current_thread().ident] = event

def _unregister_thread_cancel():
    with _THREAD_CANCEL_LOCK:
        _THREAD_CANCEL_REGISTRY.pop(threading.current_thread().ident, None)

def _is_thread_cancelled():
    with _THREAD_CANCEL_LOCK:
        ev = _THREAD_CANCEL_REGISTRY.get(threading.current_thread().ident)
    return ev is not None and ev.is_set()


def safe_as_completed(futures, timeout=None):
    """Drop-in replacement for concurrent.futures.as_completed() that keeps
    the main thread active via a polling loop. On Databricks, the notebook
    output buffer only flushes when the main thread executes Python code.

    Uses manual future.done() polling + time.sleep() instead of
    concurrent.futures.wait(). The latter uses C-level Condition.wait()
    which can block far beyond its timeout when the GIL is contended by
    worker threads executing C-extensions (e.g. PySpark JNI calls via
    Py4J). time.sleep() properly releases the GIL and always returns on
    schedule, guaranteeing the main thread stays responsive.
    """
    import time as _time_mod
    import sys as _sys_mod

    if isinstance(futures, dict):
        future_set = set(futures.keys())
    else:
        future_set = set(futures)

    total = len(future_set)
    remaining = set(future_set)
    start_time = _time_mod.time()
    heartbeat_interval = 10
    sleep_interval = 2
    completed_count = 0
    last_heartbeat = start_time

    while remaining:
        elapsed = _time_mod.time() - start_time
        if timeout is not None and elapsed >= timeout:
            raise concurrent.futures.TimeoutError(
                f"safe_as_completed timed out after {timeout:.0f}s with {len(remaining)} futures remaining"
            )

        newly_done = {f for f in remaining if f.done()}
        remaining -= newly_done

        if newly_done:
            for future in newly_done:
                completed_count += 1
                yield future
            _sys_mod.stdout.flush()
            _sys_mod.stderr.flush()
        else:
            now = _time_mod.time()
            if now - last_heartbeat >= heartbeat_interval:
                timeout_str = f"{timeout:.0f}" if timeout is not None else "∞"
                timestamp = _time_mod.strftime('%H:%M:%S', _time_mod.localtime())
                _sys_mod.stdout.write(f"{timestamp} - INFO - ⏳ [{completed_count}/{total}] {len(remaining)} task(s) in progress ({elapsed:.0f}s / {timeout_str}s)\n")
                last_heartbeat = now
            _sys_mod.stdout.flush()
            _sys_mod.stderr.flush()
            remaining_timeout = (timeout - elapsed) if timeout is not None else sleep_interval
            _time_mod.sleep(min(sleep_interval, max(0.1, remaining_timeout)))


def _safe_future_result(future, timeout=None, poll_interval=2):
    """Get a future's result with guaranteed timeout enforcement.

    Replaces direct future.result(timeout=X) calls which use C-level
    Condition.wait() that can block beyond the timeout when the GIL is
    contended (e.g. PySpark JNI calls). This polls future.done() +
    time.sleep() instead, both of which behave correctly under GIL pressure.
    """
    import time as _time_mod
    import sys as _sys_mod

    if future.done():
        return future.result(timeout=0)

    start = _time_mod.time()
    while not future.done():
        elapsed = _time_mod.time() - start
        if timeout is not None and elapsed >= timeout:
            raise concurrent.futures.TimeoutError(
                f"_safe_future_result timed out after {timeout:.0f}s"
            )
        remaining = (timeout - elapsed) if timeout is not None else poll_interval
        _time_mod.sleep(min(poll_interval, max(0.1, remaining)))
        _sys_mod.stdout.flush()
        _sys_mod.stderr.flush()

    return future.result(timeout=0)


class _SafeExecutorContext:
    """
    Wraps a ThreadPoolExecutor with safe shutdown on timeout.
    Call mark_timed_out() when a timeout occurs so __exit__ uses
    shutdown(wait=False, cancel_futures=True) instead of blocking forever.
    """
    __slots__ = ('_executor', '_timed_out', '_logger', '_name')

    def __init__(self, max_workers, thread_name_prefix="Worker", logger=None, name=""):
        self._executor = ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix=thread_name_prefix)
        self._timed_out = False
        self._logger = logger
        self._name = name or thread_name_prefix

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._timed_out:
            if self._logger:
                self._logger.warning(f"🛑 [{self._name}] Forcing executor shutdown (wait=False) after timeout")
        self._executor.shutdown(wait=False, cancel_futures=self._timed_out)
        return False

    def submit(self, fn, *args, **kwargs):
        return self._executor.submit(fn, *args, **kwargs)

    def mark_timed_out(self):
        self._timed_out = True


class ParallelExecutor:
    """
    Centralized parallel execution manager using ThreadPoolExecutor.
    Replaces all scattered ThreadPoolExecutor usage throughout the codebase.
    """
    
    @staticmethod
    def execute_parallel(
        tasks,
        max_workers,
        task_name="Task",
        logger=None,
        timeout_per_task=None,
        total_timeout=None,
        thread_name_prefix="Worker",
        return_exceptions=False
    ):
        """
        Execute multiple tasks in parallel using ThreadPoolExecutor.
        
        Uses a polling loop instead of as_completed() to keep the main thread
        active. This is critical on Databricks where the notebook output buffer
        only flushes when the main thread executes Python code. Without polling,
        worker thread output (logs, heartbeats) accumulates invisibly while the
        main thread is blocked in as_completed()'s C-level Condition.wait().
        
        Args:
            tasks: List of callables or (callable, args) tuples
            max_workers: Maximum number of parallel workers
            task_name: Name for logging purposes
            logger: Logger instance
            timeout_per_task: Timeout per individual task in seconds
            total_timeout: Total timeout for all tasks in seconds
            thread_name_prefix: Prefix for thread names
            return_exceptions: If True, return exceptions instead of raising them
            
        Returns:
            List of results (or exceptions if return_exceptions=True)
        """
        import time as _time_mod
        import sys as _sys_mod
        import traceback as _tb_mod
        
        results = []
        exceptions = []
        poll_interval = 15
        
        cancel_events = {}

        def _safe_wrapper(idx, fn, fn_args, _cancel_evt=None):
            if _cancel_evt is not None:
                _register_thread_cancel(_cancel_evt)
            try:
                return fn(*fn_args) if fn_args else fn()
            except Exception as exc:
                _sys_mod.stderr.write(f"[ParallelExecutor] {task_name} #{idx} FAILED: {exc}\n{_tb_mod.format_exc()}")
                _sys_mod.stderr.flush()
                raise
            finally:
                if _cancel_evt is not None:
                    _unregister_thread_cancel()
        
        timed_out = False
        executor = ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix=thread_name_prefix)
        try:
            future_to_task = {}
            for i, task in enumerate(tasks):
                evt = threading.Event()
                cancel_events[i] = evt
                if isinstance(task, tuple):
                    func, args = task
                    future = executor.submit(_safe_wrapper, i, func, args, evt)
                else:
                    future = executor.submit(_safe_wrapper, i, task, None, evt)
                future_to_task[future] = i
            
            remaining = set(future_to_task.keys())
            completed_indices = set()
            start_time = _time_mod.time()
            last_heartbeat = start_time
            heartbeat_interval = poll_interval
            sleep_interval = 2
            
            while remaining:
                elapsed = _time_mod.time() - start_time
                
                if total_timeout and elapsed >= total_timeout:
                    timed_out_count = len(remaining)
                    msg = f"⏱️  Total timeout ({total_timeout}s) exceeded for {task_name}. {timed_out_count} task(s) did not complete."
                    if logger:
                        logger.error(msg)
                    else:
                        log_print(msg, level="ERROR")
                    timed_out = True
                    for evt in cancel_events.values():
                        evt.set()
                    if return_exceptions:
                        for future in remaining:
                            task_idx = future_to_task[future]
                            results.append((task_idx, TimeoutError(f"{task_name} #{task_idx} exceeded total timeout of {total_timeout}s")))
                            future.cancel()
                    else:
                        raise concurrent.futures.TimeoutError(msg)
                    break
                
                done = {f for f in remaining if f.done()}
                remaining -= done
                
                for future in done:
                    task_idx = future_to_task[future]
                    completed_indices.add(task_idx)
                    try:
                        result = future.result(timeout=timeout_per_task)
                        results.append((task_idx, result))
                    except concurrent.futures.TimeoutError:
                        error_msg = f"{task_name} #{task_idx} timed out"
                        if logger:
                            logger.warning(f"⏱️  {error_msg}")
                        if return_exceptions:
                            results.append((task_idx, TimeoutError(error_msg)))
                        else:
                            exceptions.append((task_idx, TimeoutError(error_msg)))
                    except Exception as e:
                        if logger:
                            logger.warning(f"❌ {task_name} #{task_idx} failed: {get_clean_error_message(e)}")
                        if return_exceptions:
                            results.append((task_idx, e))
                        else:
                            exceptions.append((task_idx, e))
                
                if remaining:
                    now = _time_mod.time()
                    if not done or now - last_heartbeat >= heartbeat_interval:
                        elapsed = _time_mod.time() - start_time
                        timeout_str = f"{total_timeout:.0f}" if total_timeout else "∞"
                        log_print(f"⏳ [{task_name}] {len(completed_indices)}/{len(future_to_task)} done, {len(remaining)} in progress ({elapsed:.0f}s / {timeout_str}s)")
                        last_heartbeat = now
                    _sys_mod.stdout.flush()
                    _sys_mod.stderr.flush()
                    if not done:
                        remaining_timeout = (total_timeout - elapsed) if total_timeout else sleep_interval
                        _time_mod.sleep(min(sleep_interval, max(0.1, remaining_timeout)))
        finally:
            if timed_out:
                if logger:
                    logger.warning(f"🛑 [{task_name}] Forcing executor shutdown (wait=False) after timeout to prevent hang")
                else:
                    log_print(f"🛑 [{task_name}] Forcing executor shutdown (wait=False) after timeout to prevent hang", level="WARNING")
            executor.shutdown(wait=False, cancel_futures=timed_out)
        
        results.sort(key=lambda x: x[0])
        
        if not return_exceptions and exceptions:
            if logger:
                logger.error(f"❌ {len(exceptions)} {task_name}(s) failed")
            raise exceptions[0][1]
        
        return [r[1] for r in results]


class CSVParser:
    """
    Centralized CSV parsing utility with consistent error handling.
    Replaces all scattered csv.DictReader usage throughout the codebase.
    """
    
    @staticmethod
    def parse_csv_string(
        csv_data,
        logger=None,
        context="",
        quoting=csv.QUOTE_ALL,
        delimiter=',',
        skipinitialspace=True,
        expected_fields=None
    ):
        """
        Parse CSV string into list of dictionaries.
        
        Args:
            csv_data: CSV string data
            logger: Logger instance
            context: Context string for logging
            quoting: CSV quoting mode (default: QUOTE_ALL)
            delimiter: Field delimiter (default: ',')
            skipinitialspace: Skip initial spaces (default: True)
            expected_fields: Optional list of expected field names for validation
            
        Returns:
            List of dictionaries (one per row)
        """
        if not csv_data or not csv_data.strip():
            if logger:
                logger.warning(f"⚠️  Empty CSV data{f' for {context}' if context else ''}")
            return []
        
        try:
            reader = csv.DictReader(
                io.StringIO(csv_data),
                quoting=quoting,
                delimiter=delimiter,
                skipinitialspace=skipinitialspace
            )
            rows = list(reader)
            
            if expected_fields and rows:
                actual_fields_lower = {k.lower().strip(): k for k in rows[0].keys() if k is not None}
                missing_fields = {f for f in expected_fields if f.lower().strip() not in actual_fields_lower}
                if missing_fields and logger:
                    logger.warning(f"⚠️  Missing expected CSV fields{f' for {context}' if context else ''}: {missing_fields}")
            
            if logger:
                logger.debug(f"✅ Parsed {len(rows)} CSV rows{f' for {context}' if context else ''}")
            
            return rows
        except Exception as e:
            if logger:
                logger.error(f"❌ CSV parsing failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return []
    
    @staticmethod
    def parse_csv_list(
        csv_data,
        logger=None,
        context="",
        quoting=csv.QUOTE_ALL,
        delimiter=',',
        quotechar='"',
        skipinitialspace=True
    ):
        """
        Parse CSV string into list of lists (for non-dictionary CSV).
        
        Args:
            csv_data: CSV string data
            logger: Logger instance
            context: Context string for logging
            quoting: CSV quoting mode (default: QUOTE_ALL)
            delimiter: Field delimiter (default: ',')
            quotechar: Quote character (default: '"')
            skipinitialspace: Skip initial spaces (default: True)
            
        Returns:
            List of lists (one per row)
        """
        if not csv_data or not csv_data.strip():
            if logger:
                logger.warning(f"⚠️  Empty CSV data{f' for {context}' if context else ''}")
            return []
        
        try:
            reader = csv.reader(
                io.StringIO(csv_data),
                delimiter=delimiter,
                quotechar=quotechar,
                quoting=quoting,
                skipinitialspace=skipinitialspace
            )
            rows = list(reader)
            
            if logger:
                logger.debug(f"✅ Parsed {len(rows)} CSV rows{f' for {context}' if context else ''}")
            
            return rows
        except Exception as e:
            if logger:
                logger.error(f"❌ CSV parsing failed{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return []


class JSONParser:
    """
    Centralized JSON parsing utility with consistent error handling.
    Replaces scattered json.loads/dumps usage throughout the codebase.
    """
    
    @staticmethod
    def safe_loads(json_string, logger=None, context="", fallback=None):
        """
        Safely parse JSON string with error handling.
        
        Args:
            json_string: JSON string to parse
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback value on parsing failure
            
        Returns:
            Parsed JSON object or fallback value
        """
        if not json_string:
            return fallback
        
        try:
            return json.loads(json_string)
        except json.JSONDecodeError as e:
            if logger:
                logger.warning(f"⚠️  JSON parsing failed{f' for {context}' if context else ''}: {str(e)[:100]}")
            return fallback
        except Exception as e:
            if logger:
                logger.error(f"❌ Unexpected error parsing JSON{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return fallback
    
    @staticmethod
    def safe_dumps(obj, logger=None, context="", fallback="{}", indent=None, separators=None):
        """
        Safely serialize object to JSON string with error handling.
        
        Args:
            obj: Object to serialize
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback string on serialization failure
            indent: Indentation level (default: None)
            separators: Custom separators (default: None)
            
        Returns:
            JSON string or fallback value
        """
        try:
            if separators:
                return json.dumps(obj, indent=indent, separators=separators)
            return json.dumps(obj, indent=indent)
        except TypeError as e:
            if logger:
                logger.warning(f"⚠️  JSON serialization failed{f' for {context}' if context else ''}: {str(e)[:100]}")
            return fallback
        except Exception as e:
            if logger:
                logger.error(f"❌ Unexpected error serializing JSON{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
            return fallback


class TimeoutHandler:
    """
    Centralized timeout handling utility.
    Provides consistent timeout behavior across the codebase.
    """
    
    @staticmethod
    def execute_with_timeout(func, timeout_seconds, logger=None, context="", fallback=None):
        """
        Execute a function with a timeout.
        
        Args:
            func: Callable to execute
            timeout_seconds: Timeout in seconds
            logger: Logger instance
            context: Context string for logging
            fallback: Fallback value on timeout
            
        Returns:
            Result of func() or fallback on timeout
        """
        from concurrent.futures import TimeoutError as FuturesTimeoutError
        
        executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="Timeout")
        timed_out = False
        try:
            future = executor.submit(func)
            try:
                result = _safe_future_result(future, timeout=timeout_seconds)
                return result
            except (FuturesTimeoutError, concurrent.futures.TimeoutError):
                timed_out = True
                future.cancel()
                if logger:
                    logger.warning(f"⏱️  Timeout ({timeout_seconds}s) exceeded{f' for {context}' if context else ''}")
                if fallback is not None:
                    return fallback() if callable(fallback) else fallback
                raise TimeoutError(f"Operation timed out after {timeout_seconds}s{f' for {context}' if context else ''}")
            except Exception as e:
                if logger:
                    logger.error(f"❌ Error during execution{f' for {context}' if context else ''}: {get_clean_error_message(e)}")
                raise
        finally:
            executor.shutdown(wait=False, cancel_futures=timed_out)



# DBTITLE 1,Table Size Analyzer & Dynamic Batch Optimizer

class TableSizeInfo:
    """Metadata about a table's size and structure."""
    def __init__(self, catalog, schema, table, num_columns=0, estimated_row_count=0, size_category="unknown"):
        self.catalog = catalog
        self.schema = schema
        self.table = table
        self.num_columns = num_columns
        self.estimated_row_count = estimated_row_count
        self.size_category = size_category  # "small", "medium", "wide", "very_wide"
        self.memory_weight = self._calculate_memory_weight()
    
    def _calculate_memory_weight(self):
        """Calculate memory weight for batching decisions."""
        if self.num_columns > 1000:
            return self.num_columns * 10  # Very heavy
        elif self.num_columns > 250:
            return self.num_columns * 5   # Heavy
        elif self.num_columns > 100:
            return self.num_columns * 2   # Medium
        else:
            return self.num_columns       # Light
    
    def __repr__(self):
        return f"{self.catalog}.{self.schema}.{self.table} ({self.num_columns} cols, {self.size_category})"


class TableSizeAnalyzer:
    """
    Two-pass analyzer: First pass collects table sizes, second pass uses this info for intelligent batching.
    """
    def __init__(self, spark, logger):
        self.spark = spark
        self.logger = logger
        self.size_cache = {}  # {(catalog, schema, table): TableSizeInfo}
    
    def analyze_table_sizes_batch(self, table_tuples, use_info_schema_map, max_parallelism=20):
        """
        Analyze sizes for a batch of tables efficiently using parallel queries.
        
        Args:
            table_tuples: List of (catalog, schema, table) tuples
            use_info_schema_map: Dict mapping catalog -> bool for information_schema support
            max_parallelism: Maximum number of parallel queries (default: 20)
            
        Returns:
            List of TableSizeInfo objects
        """
        results = []
        
        # Group by catalog.schema for efficient querying
        schema_groups = defaultdict(list)
        for cat, schema, table in table_tuples:
            schema_groups[(cat, schema)].append(table)
        
        # Process schemas in parallel for speed
        def analyze_schema_group(schema_key_and_tables):
            (catalog, schema), tables = schema_key_and_tables
            use_info_schema = use_info_schema_map.get(catalog, False)
            schema_results = []
            
            self.logger.debug(f"   Analyzing {len(tables)} tables in {catalog}.{schema}...")
            
            try:
                if use_info_schema:
                    # Batch query using information_schema (much faster)
                    table_list = "','".join(tables)
                    query = f"""
                        SELECT table_name, COUNT(*) as num_columns
                        FROM `{catalog}`.`information_schema`.`columns`
                        WHERE table_schema = '{schema}'
                        AND table_name IN ('{table_list}')
                        GROUP BY table_name
                    """
                    df = self.spark.sql(query)
                    column_counts = {row.table_name: row.num_columns for row in df.collect()}
                    
                    for table in tables:
                        num_cols = column_counts.get(table, 0)
                        if num_cols == 0:
                            # info_schema-zero-fallback: some catalogs (samples, delta shares) expose
                            # tables without populating information_schema.columns. Fall back to DESCRIBE
                            # TABLE — otherwise the entire fact/dim stack silently vanishes.
                            try:
                                num_cols = self._get_column_count_fallback(catalog, schema, table)
                            except Exception as _fb_err:
                                self.logger.debug(f"DESCRIBE fallback failed for {catalog}.{schema}.{table}: {_fb_err}")
                                num_cols = 0
                            if num_cols == 0:
                                self.logger.warning(f"No columns available for {catalog}.{schema}.{table} (info_schema empty AND DESCRIBE failed); skipping")
                                continue
                            else:
                                self.logger.info(f"info_schema returned 0 for {catalog}.{schema}.{table}; DESCRIBE fallback found {num_cols} columns")
                        
                        size_info = TableSizeInfo(
                            catalog, schema, table,
                            num_columns=num_cols,
                            size_category=self._categorize_table(num_cols)
                        )
                        schema_results.append(size_info)
                        self.size_cache[(catalog, schema, table)] = size_info
                else:
                    # Fallback: DESCRIBE each table individually (slower)
                    for table in tables:
                        num_cols = self._get_column_count_fallback(catalog, schema, table)
                        if num_cols == 0:
                            continue
                        
                        size_info = TableSizeInfo(
                            catalog, schema, table,
                            num_columns=num_cols,
                            size_category=self._categorize_table(num_cols)
                        )
                        schema_results.append(size_info)
                        self.size_cache[(catalog, schema, table)] = size_info
                        
            except Exception as e:
                self.logger.warning(f"Error analyzing tables in {catalog}.{schema}: {get_clean_error_message(e)}")
                # Fallback: assume medium size
                for table in tables:
                    size_info = TableSizeInfo(catalog, schema, table, num_columns=50, size_category="medium")
                    schema_results.append(size_info)
                    self.size_cache[(catalog, schema, table)] = size_info
            
            return schema_results
        
        # Execute schema analysis - parallel or sequential based on max_parallelism
        if max_parallelism == 1:
            # Sequential execution when nested in another thread pool
            for item in schema_groups.items():
                try:
                    schema_results = analyze_schema_group(item)
                    results.extend(schema_results)
                except Exception as e:
                    schema_key = item[0]
                    self.logger.error(f"Failed to analyze schema {schema_key}: {get_clean_error_message(e)}")
        else:
            # Parallel execution for top-level calls
            with _SafeExecutorContext(max_workers=max_parallelism, thread_name_prefix="SchemaAnalyzer", logger=self.logger, name="SchemaAnalyzer") as ctx:
                futures = {ctx.submit(analyze_schema_group, item): item[0] 
                          for item in schema_groups.items()}
                
                total_timeout = len(futures) * 300
                try:
                    for future in safe_as_completed(futures, timeout=total_timeout):
                        schema_key = futures[future]
                        try:
                            schema_results = future.result(timeout=300)
                            results.extend(schema_results)
                        except concurrent.futures.TimeoutError:
                            self.logger.error(f"Schema analysis timed out for {schema_key} after 5 minutes")
                        except Exception as e:
                            self.logger.error(f"Failed to analyze schema {schema_key}: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Overall schema analysis timeout ({total_timeout}s). Proceeding with partial results.")
        
        return results
    
    def _get_column_count_fallback(self, catalog, schema, table):
        """Fallback method to get column count using DESCRIBE."""
        try:
            fq_table = f"`{catalog}`.`{schema}`.`{table}`"
            df = self.spark.sql(f"DESCRIBE TABLE {fq_table}")
            # Filter out partition info and metadata rows
            count = df.filter(~col("col_name").startswith("#")).count()
            return count
        except Exception as e:
            self.logger.debug(f"Error getting column count for {catalog}.{schema}.{table}: {get_clean_error_message(e, max_chars=2000)}")
            return 0
    
    def _categorize_table(self, num_columns):
        """Categorize table based on column count."""
        if num_columns > 1000:
            return "very_wide"
        elif num_columns > 250:
            return "wide"
        elif num_columns > 100:
            return "medium"
        else:
            return "small"
    
    def get_cached_size(self, catalog, schema, table):
        """Get cached size info for a table."""
        return self.size_cache.get((catalog, schema, table))


class DynamicBatchOptimizer:
    """
    Intelligently groups tables into batches based on their size and memory requirements.
    
    Strategy:
    - Very wide tables (>1000 cols): 1-2 per batch
    - Wide tables (250-1000 cols): 5-10 per batch
    - Medium tables (100-250 cols): 20-50 per batch
    - Small tables (<100 cols): 100-500 per batch
    """
    
    # Memory weights for batching (in arbitrary units)
    MAX_BATCH_WEIGHT = 10000  # Adjust based on cluster memory
    MIN_BATCH_SIZE = 1
    MAX_BATCH_SIZE = 500
    
    def __init__(self, logger, max_batch_weight=None):
        self.logger = logger
        self.max_batch_weight = max_batch_weight or self.MAX_BATCH_WEIGHT
    
    def create_optimized_batches(self, table_size_infos):
        """
        Create optimized batches from table size information.
        
        Args:
            table_size_infos: List of TableSizeInfo objects
            
        Returns:
            List of lists, where each inner list is a batch of (catalog, schema, table) tuples
        """
        if not table_size_infos:
            return []
        
        # Sort tables by size (largest first for better packing)
        sorted_tables = sorted(table_size_infos, key=lambda t: t.memory_weight, reverse=True)
        
        batches = []
        current_batch = []
        current_weight = 0
        
        for table_info in sorted_tables:
            table_tuple = (table_info.catalog, table_info.schema, table_info.table)
            table_weight = table_info.memory_weight
            
            # Check if adding this table would exceed batch weight
            if current_batch and (current_weight + table_weight > self.max_batch_weight or 
                                 len(current_batch) >= self.MAX_BATCH_SIZE):
                # Start new batch
                batches.append(current_batch)
                self.logger.debug(f"Created batch with {len(current_batch)} tables, weight={current_weight}")
                current_batch = []
                current_weight = 0
            
            # Add table to current batch
            current_batch.append(table_tuple)
            current_weight += table_weight
        
        # Add final batch
        if current_batch:
            batches.append(current_batch)
            self.logger.debug(f"Created final batch with {len(current_batch)} tables, weight={current_weight}")
        
        # Log batch statistics
        self._log_batch_stats(batches, table_size_infos)
        
        return batches
    
    def _log_batch_stats(self, batches, table_size_infos):
        """Log statistics about the created batches."""
        total_tables = len(table_size_infos)
        num_batches = len(batches)
        
        size_categories = defaultdict(int)
        for table_info in table_size_infos:
            size_categories[table_info.size_category] += 1
        
        avg_batch_size = total_tables / num_batches if num_batches > 0 else 0
        
        self.logger.info(f"📊 Dynamic Batch Optimization Complete:")
        self.logger.info(f"   • Total tables: {total_tables}")
        self.logger.info(f"   • Created batches: {num_batches}")
        self.logger.info(f"   • Average batch size: {avg_batch_size:.1f} tables")
        self.logger.info(f"   • Table size distribution:")
        self.logger.info(f"      - Small (<100 cols): {size_categories['small']}")
        self.logger.info(f"      - Medium (100-250 cols): {size_categories['medium']}")
        self.logger.info(f"      - Wide (250-1000 cols): {size_categories['wide']}")
        self.logger.info(f"      - Very Wide (>1000 cols): {size_categories['very_wide']}")


class ColumnSampler:
    """
    Samples columns from very wide tables to reduce memory footprint.
    
    For tables with >250 columns, intelligently selects representative columns:
    - All primary keys, foreign keys
    - Columns with business-meaningful names
    - Sample of remaining columns
    """
    
    WIDE_TABLE_THRESHOLD = 250
    TARGET_SAMPLE_SIZE = 200
    
    def __init__(self, logger):
        self.logger = logger
    
    def should_sample(self, num_columns):
        """Determine if column sampling is needed."""
        return num_columns > self.WIDE_TABLE_THRESHOLD
    
    def sample_columns(self, column_details, table_info):
        """
        Sample columns from a wide table.
        
        Args:
            column_details: List of (catalog, schema, table, col_name, data_type, comment) tuples
            table_info: TableSizeInfo object
            
        Returns:
            Sampled list of column details + metadata about sampling
        """
        if not self.should_sample(len(column_details)):
            return column_details, False  # No sampling needed
        
        self.logger.info(f"🎯 Sampling columns for wide table {table_info}: {len(column_details)} -> ~{self.TARGET_SAMPLE_SIZE} cols")
        
        # Categorize columns
        key_columns = []
        business_columns = []
        other_columns = []
        
        # Business keywords to identify important columns
        business_keywords = [
            'id', 'key', 'name', 'date', 'time', 'amount', 'total', 'count', 'quantity',
            'price', 'cost', 'revenue', 'customer', 'order', 'product', 'status',
            'type', 'category', 'description', 'address', 'email', 'phone'
        ]
        
        for col_detail in column_details:
            col_name = col_detail[3].lower()
            
            # Identify key columns (id, primary key patterns)
            if 'id' in col_name or 'key' in col_name or col_name.endswith('_pk') or col_name.endswith('_fk'):
                key_columns.append(col_detail)
            # Identify business-relevant columns
            elif any(keyword in col_name for keyword in business_keywords):
                business_columns.append(col_detail)
            else:
                other_columns.append(col_detail)
        
        # Build sampled list
        sampled = []
        
        # Always include all key columns
        sampled.extend(key_columns)
        
        # Include as many business columns as possible
        remaining_slots = self.TARGET_SAMPLE_SIZE - len(sampled)
        if remaining_slots > 0:
            sampled.extend(business_columns[:remaining_slots])
        
        # Fill remaining with evenly spaced sample from other columns
        remaining_slots = self.TARGET_SAMPLE_SIZE - len(sampled)
        if remaining_slots > 0 and other_columns:
            step = max(1, len(other_columns) // remaining_slots)
            sampled.extend(other_columns[::step][:remaining_slots])
        
        self.logger.info(f"   ✓ Sampled: {len(key_columns)} key cols + {len(business_columns[:remaining_slots])} business cols + "
                        f"{len(sampled) - len(key_columns) - len(business_columns[:remaining_slots])} other cols = {len(sampled)} total")
        
        return sampled, True  # Return sampled columns + flag indicating sampling occurred


# COMMAND ----------

# DBTITLE 1,DataLoader

class DataLoader:
    # === MODIFIED: Added tables parameter + memory optimization features ===
    def __init__(self, catalogs: str, schemas: str, tables: str, logger: logging.Logger, 
                 enable_two_pass=True, enable_column_sampling=True, streaming_batch_size=1000,
                 max_parallelism=10, schema_timeout_seconds=900):
        self.spark = SparkSession.builder.getOrCreate()
        self.max_parallelism = max_parallelism  # For parallel schema discovery and column loading
        self.schema_timeout_seconds = schema_timeout_seconds  # Timeout per schema query (15 minutes)
        self.logger = logger
        self.foreign_key_graph = defaultdict(list)
        
        # === NEW: Memory optimization features ===
        self.enable_two_pass = enable_two_pass  # Enable intelligent batching based on table sizes
        self.enable_column_sampling = enable_column_sampling  # Sample columns from very wide tables
        self.streaming_batch_size = streaming_batch_size  # Number of tables to process in each streaming chunk
        
        # === NEW: Initialize optimization components ===
        self.size_analyzer = TableSizeAnalyzer(self.spark, self.logger) if enable_two_pass else None
        self.batch_optimizer = DynamicBatchOptimizer(self.logger) if enable_two_pass else None
        self.column_sampler = ColumnSampler(self.logger) if enable_column_sampling else None

        # === MODIFIED: Process schemas, catalogs, and individual tables ===
        # Use utility functions to normalize identifiers (strip backticks from user input)
        self.schemas_to_process = [s.strip() for s in schemas.split(',') if s.strip()]
        self.catalogs_to_process = [normalize_identifier(c) for c in catalogs.split(',') if c.strip()]
        self.tables_to_process = [t.strip() for t in tables.split(',') if t.strip()]
        
        # Get all unique catalogs mentioned to check capabilities
        self.catalog_capabilities = {}
        unique_catalogs = set(self.catalogs_to_process)
        for s in self.schemas_to_process:
            cat, _ = parse_two_level_name(s)
            if cat:
                unique_catalogs.add(cat)
        for t in self.tables_to_process:
            cat, _, _ = parse_three_level_name(t)
            if cat:
                unique_catalogs.add(cat)
        
        self.logger.info(f"Initializing capabilities for {len(unique_catalogs)} catalogs in parallel: {unique_catalogs}")
        if len(unique_catalogs) > 1:
            with _SafeExecutorContext(max_workers=min(len(unique_catalogs), METADATA_PARALLELISM), thread_name_prefix="cat_cap", logger=self.logger, name="CatalogCapabilities") as ctx:
                cap_futures = {ctx.submit(self._check_catalog_capability, cat): cat for cat in unique_catalogs}
                try:
                    for future in safe_as_completed(cap_futures, timeout=120):
                        cat = cap_futures[future]
                        try:
                            self.catalog_capabilities[cat] = future.result(timeout=60)
                        except Exception as exc:
                            self.logger.warning(f"Catalog capability check failed for '{cat}': {get_clean_error_message(exc)}")
                            self.catalog_capabilities[cat] = False
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Catalog capability checks timed out globally (120s)")
                    for future in cap_futures:
                        if not future.done():
                            cat = cap_futures[future]
                            self.catalog_capabilities[cat] = False
        else:
            for catalog in unique_catalogs:
                self.catalog_capabilities[catalog] = self._check_catalog_capability(catalog)
        self.logger.info(f"Capabilities found: {self.catalog_capabilities}")

        # === MODIFIED: Build the database queue and individual tables list ===
        self.database_queue = []
        db_set = set()
        
        # Track individual tables separately
        self.individual_tables = []  # List of (catalog, schema, table) tuples
        
        # Track explicitly provided schemas (to know which schemas should expand ALL tables)
        self.explicit_schemas_set = set()  # Set of (catalog, schema) tuples

        # 1. From explicit schemas (use parse_two_level_name for consistent normalization)
        for s in self.schemas_to_process:
            cat, db = parse_two_level_name(s)
            if cat and db:
                db_set.add((cat, db))
                self.explicit_schemas_set.add((cat, db))  # Mark as explicitly provided
            else:
                self.logger.warning(f"Skipping malformed schema name: {s}")
        
        # 2. From individual tables (use parse_three_level_name for consistent normalization)
        for t in self.tables_to_process:
            cat, db, table = parse_three_level_name(t)
            if cat and db and table:
                self.individual_tables.append((cat, db, table))
                # Also add the schema to db_set so we can process it
                db_set.add((cat, db))
            else:
                self.logger.warning(f"Skipping malformed table name: {t} (expected format: catalog.schema.table)")
        
        # 3. From catalogs (schemas from catalogs should also expand ALL tables)
        eligible_catalogs = [cat for cat in self.catalogs_to_process if self.catalog_capabilities.get(cat, False)]
        skipped_catalogs = [cat for cat in self.catalogs_to_process if not self.catalog_capabilities.get(cat, False)]
        for cat in skipped_catalogs:
            self.logger.warning(f"Skipping schema discovery for catalog `{cat}`: Lacks information_schema support and fallback failed.")
        
        if len(eligible_catalogs) > 1:
            _schema_lock = threading.Lock()
            def _fetch_catalog_schemas(cat):
                self.logger.info(f"Fetching schemas for catalog: {cat}")
                return cat, self._fetch_schemas_for_catalog(cat)
            
            with _SafeExecutorContext(max_workers=min(len(eligible_catalogs), METADATA_PARALLELISM), thread_name_prefix="cat_sch", logger=self.logger, name="SchemaDiscovery") as ctx:
                sch_futures = {ctx.submit(_fetch_catalog_schemas, cat): cat for cat in eligible_catalogs}
                try:
                    for future in safe_as_completed(sch_futures, timeout=300):
                        try:
                            cat, schemas_in_cat = future.result(timeout=120)
                            with _schema_lock:
                                for db in schemas_in_cat:
                                    db_set.add((cat, db))
                                    self.explicit_schemas_set.add((cat, db))
                        except Exception as exc:
                            self.logger.error(f"Schema discovery failed for catalog: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Schema discovery timed out globally (300s)")
        else:
            for cat in eligible_catalogs:
                self.logger.info(f"Fetching schemas for catalog: {cat}")
                schemas_in_cat = self._fetch_schemas_for_catalog(cat)
                for db in schemas_in_cat:
                    db_set.add((cat, db))
                    self.explicit_schemas_set.add((cat, db))
        
        self.database_queue = sorted(list(db_set)) # Sort for deterministic order
        self.logger.info(f"Found {len(self.database_queue)} unique databases to process.")
        
        # Log explicit schemas (will expand ALL tables)
        if self.explicit_schemas_set:
            explicit_names = [f"{cat}.{db}" for cat, db in sorted(self.explicit_schemas_set)]
            self.logger.info(f"📋 Explicit schemas (will expand ALL tables): {', '.join(explicit_names)}")
        
        # Log individual tables (specific tables only)
        if self.individual_tables:
            db_stats = {}
            for cat, db, tbl in self.individual_tables:
                key = f"{cat}.{db}"
                db_stats[key] = db_stats.get(key, 0) + 1
            self.logger.info(f"Found {len(self.individual_tables)} individual tables to process across {len(db_stats)} databases:")
            for db_key, count in sorted(db_stats.items()):
                self.logger.info(f"  database {db_key}: {count} tables loaded")
        
        # === NEW: State variables for table-level batching ===
        self.all_table_tuples = []  # List of (catalog, schema, table) tuples
        self.optimized_batches = []  # List of optimized batches (two-pass mode)
        self.current_batch_idx = 0   # Current batch index (two-pass mode)
        self.current_table_idx = 0   # Current position in all_table_tuples (single-pass mode)
        self._tables_initialized = False
        self._size_analysis_complete = False

    def _check_catalog_capability(self, catalog_name: str) -> bool:
        try:
            self.spark.sql(f"SELECT 1 FROM `{catalog_name}`.`information_schema`.`schemata` LIMIT 1").collect()
            return True
        except Exception:
            self.logger.info(f"Catalog `{catalog_name}` does not support information_schema. Will attempt fallback 'SHOW' commands.")
            return False # Rely on fallbacks

    def _fetch_schemas_for_catalog(self, catalog_name: str):
        use_info_schema = self.catalog_capabilities.get(catalog_name, False)

        try:
            if not use_info_schema:
                raise Exception("Catalog does not support information_schema, using fallback.")
            
            query = f"""
                SELECT `schema_name` FROM `{catalog_name}`.`information_schema`.`schemata` 
                WHERE `schema_name` != 'information_schema' 
                ORDER BY `schema_name`
            """
            df = self.spark.sql(query)
            return [row.schema_name for row in df.collect()]
        except Exception as e_info:
            try:
                query = f"SHOW SCHEMAS IN `{catalog_name}`"
                df = self.spark.sql(query)
                col_name = "databaseName" if "databaseName" in df.columns else "namespace"
                all_schemas = [row[col_name] for row in df.orderBy(col_name).collect()]
                return [s for s in all_schemas if s != 'information_schema']
            except Exception as e_show_schemas:
                self.logger.error(f"Error listing schemas for catalog `{catalog_name}`. All fallbacks failed: {get_clean_error_message(e_show_schemas)}")
                return []

    # === MODIFIED: Added streaming/pagination support for memory efficiency ===
    def _fetch_tables_for_schema(self, catalog_name: str, schema_name: str, limit=None, offset=0):
        """
        Fetch tables for a schema with optional pagination.
        
        Args:
            catalog_name: Catalog name
            schema_name: Schema name
            limit: Optional limit for pagination (for very large schemas)
            offset: Offset for pagination
            
        Returns:
            List of fully qualified table names
        """
        cat_normalized = normalize_identifier(catalog_name)
        schema_normalized = normalize_identifier(schema_name)
        fq_schema = build_fqn(cat_normalized, schema_normalized)
        tables = []
        use_info_schema = self.catalog_capabilities.get(cat_normalized, False)

        try:
            if not use_info_schema:
                raise Exception("Catalog does not support information_schema, using fallback.")
            
            # Use pagination for large schemas
            limit_clause = f"LIMIT {limit} OFFSET {offset}" if limit else ""
            cat_quoted = quote_identifier(cat_normalized)
            query = f"""
                SELECT `table_name` FROM {cat_quoted}.`information_schema`.`tables`
                WHERE `table_schema` = '{schema_normalized}'
                ORDER BY `table_name`
                {limit_clause}
            """
            df = self.spark.sql(query)
            
            # Use iterative collection for large result sets
            if limit and limit > 1000:
                # Stream in chunks to avoid memory issues
                chunk_size = 1000
                collected = []
                temp_df = df
                while True:
                    chunk = temp_df.limit(chunk_size).collect()
                    if not chunk:
                        break
                    collected.extend(chunk)
                    if len(chunk) < chunk_size:
                        break
                tables = [f"{fq_schema}.`{row.table_name}`" for row in collected]
            else:
                tables = [f"{fq_schema}.`{row.table_name}`" for row in df.collect()]
                
        except Exception:
            try:
                query = f"SHOW TABLES IN {fq_schema}"
                df = self.spark.sql(query)
                if 'isTemporary' in df.columns:
                    df = df.filter(df.isTemporary == False)
                    
                # Apply pagination if specified
                if limit:
                    df = df.orderBy("tableName").limit(limit).offset(offset)
                else:
                    df = df.orderBy("tableName")
                    
                tables = [f"{fq_schema}.`{row.tableName}`" for row in df.collect()]
            except Exception as e:
                self.logger.warning(f"Error listing tables for {fq_schema}: {get_clean_error_message(e)}")
                tables = []
        
        return tables
    
    def _fetch_tables_for_schema_streaming(self, catalog_name: str, schema_name: str, chunk_size=1000):
        """
        Generator that yields tables in chunks for memory-efficient streaming.
        
        Args:
            catalog_name: Catalog name  
            schema_name: Schema name
            chunk_size: Number of tables to yield per chunk
            
        Yields:
            Lists of fully qualified table names
        """
        offset = 0
        while True:
            tables = self._fetch_tables_for_schema(catalog_name, schema_name, limit=chunk_size, offset=offset)
            if not tables:
                break
            yield tables
            if len(tables) < chunk_size:
                break  # Last chunk
            offset += chunk_size

    def _get_table_details(self, catalog: str, schema: str, table: str, apply_sampling=True):
        """
        Get table column details with optional column sampling for very wide tables.
        
        Args:
            catalog: Catalog name
            schema: Schema name
            table: Table name
            apply_sampling: Whether to apply column sampling for wide tables
            
        Returns:
            List of (catalog, schema, table, column_name, data_type, comment) tuples
        """
        details = []
        cat_normalized = normalize_identifier(catalog)
        schema_normalized = normalize_identifier(schema)
        table_normalized = normalize_identifier(table)
        fq_table_name = build_fqn(cat_normalized, schema_normalized, table_normalized)
        use_info_schema = self.catalog_capabilities.get(cat_normalized, False)
        
        try:
            if not use_info_schema:
                raise Exception("Catalog does not support information_schema, using fallback.")

            cat_quoted = quote_identifier(cat_normalized)
            query = f"""
                SELECT `table_catalog`, `table_schema`, `table_name`, `column_name`, `data_type`, `comment`
                FROM {cat_quoted}.`information_schema`.`columns`
                WHERE `table_schema` = '{schema_normalized}' AND `table_name` = '{table_normalized}'
                ORDER BY `ordinal_position`
            """
            df = self.spark.sql(query)
            for row in df.toLocalIterator():
                details.append((
                    row.table_catalog, 
                    row.table_schema, 
                    row.table_name, 
                    row.column_name, 
                    row.data_type, 
                    row.comment
                ))
            try:
                fk_rows = self._get_foreign_keys(catalog, schema, table)
                if fk_rows:
                    self.foreign_key_graph[(catalog, schema, table)] = fk_rows
            except Exception:
                pass
            if not details:
                # Don't raise exception, just try fallback
                pass
            else:
                # Apply column sampling if enabled and needed
                if apply_sampling and self.column_sampler and len(details) > ColumnSampler.WIDE_TABLE_THRESHOLD:
                    table_info = self.size_analyzer.get_cached_size(catalog, schema, table) if self.size_analyzer else None
                    if not table_info:
                        table_info = TableSizeInfo(catalog, schema, table, num_columns=len(details))
                    details, was_sampled = self.column_sampler.sample_columns(details, table_info)
                return details
                
        except Exception:
            pass # Fallthrough to DESCRIBE
            
        try:
            query = f"DESCRIBE TABLE {fq_table_name}"
            df = self.spark.sql(query)
            for row in df.toLocalIterator():
                if row.col_name and not row.col_name.startswith('#'):
                    details.append((
                        catalog,
                        schema,
                        table,
                        row.col_name,
                        row.data_type,
                        row.comment
                    ))
            
            try:
                fk_rows = self._get_foreign_keys(catalog, schema, table)
                if fk_rows:
                    self.foreign_key_graph[(catalog, schema, table)] = fk_rows
            except Exception:
                pass
            # Apply column sampling if enabled and needed
            if apply_sampling and self.column_sampler and len(details) > ColumnSampler.WIDE_TABLE_THRESHOLD:
                table_info = self.size_analyzer.get_cached_size(catalog, schema, table) if self.size_analyzer else None
                if not table_info:
                    table_info = TableSizeInfo(catalog, schema, table, num_columns=len(details))
                details, was_sampled = self.column_sampler.sample_columns(details, table_info)
                
            return details
        except Exception as e:
            # Suppress permission denied errors - this tool works at METADATA level only
            error_msg = str(e).lower()
            if "permission" in error_msg or "unauthorized" in error_msg or "access" in error_msg:
                # Silently skip tables without SELECT permission - this is expected behavior
                pass  # No logging for expected access permission issues
            else:
                # Log other errors at debug level only
                pass  # Suppress low-level metadata errors
            return []

    def _get_foreign_keys(self, catalog: str, schema: str, table: str):
        key = (catalog, schema, table)
        if key in self.foreign_key_graph:
            return self.foreign_key_graph[key]
        
        cat_normalized = normalize_identifier(catalog)
        schema_normalized = normalize_identifier(schema)
        table_normalized = normalize_identifier(table)
        cat_quoted = quote_identifier(cat_normalized)
        
        rels = []
        
        try:
            comprehensive_query = f"""
                SELECT 
                    kcu.table_catalog AS table_catalog,
                    kcu.table_schema AS table_schema,
                    kcu.table_name AS table_name,
                    kcu.column_name AS column_name,
                    rc.unique_constraint_catalog AS referenced_table_catalog,
                    rc.unique_constraint_schema AS referenced_table_schema,
                    kcu_ref.table_name AS referenced_table_name,
                    kcu_ref.column_name AS referenced_column_name
                FROM {cat_quoted}.`information_schema`.`table_constraints` tc
                JOIN {cat_quoted}.`information_schema`.`referential_constraints` rc
                    ON tc.constraint_catalog = rc.constraint_catalog
                    AND tc.constraint_schema = rc.constraint_schema
                    AND tc.constraint_name = rc.constraint_name
                JOIN {cat_quoted}.`information_schema`.`key_column_usage` kcu
                    ON tc.constraint_catalog = kcu.constraint_catalog
                    AND tc.constraint_schema = kcu.constraint_schema
                    AND tc.constraint_name = kcu.constraint_name
                JOIN {cat_quoted}.`information_schema`.`key_column_usage` kcu_ref
                    ON rc.unique_constraint_catalog = kcu_ref.constraint_catalog
                    AND rc.unique_constraint_schema = kcu_ref.constraint_schema
                    AND rc.unique_constraint_name = kcu_ref.constraint_name
                WHERE tc.table_catalog = '{cat_normalized}'
                    AND tc.table_schema = '{schema_normalized}'
                    AND tc.table_name = '{table_normalized}'
                    AND tc.constraint_type = 'FOREIGN KEY'
                ORDER BY tc.constraint_name
            """
            df = self.spark.sql(comprehensive_query)
            rels = [
                (
                    row.table_catalog,
                    row.table_schema,
                    row.table_name,
                    row.column_name,
                    row.referenced_table_catalog,
                    row.referenced_table_schema,
                    row.referenced_table_name,
                    row.referenced_column_name
                )
                for row in df.collect()
            ]
            if rels:
                self.logger.debug(f"Found {len(rels)} FK relationships for {cat_normalized}.{schema_normalized}.{table_normalized} via comprehensive query")
                self.foreign_key_graph[key] = rels
                return rels
        except Exception as e:
            self.logger.debug(f"Comprehensive FK query failed for {cat_normalized}.{schema_normalized}.{table_normalized}: {str(e)[:150]}. Falling back to simple query.")
        
        try:
            simple_query = f"""
                SELECT table_catalog,
                       table_schema,
                       table_name,
                       column_name,
                       referenced_table_catalog,
                       referenced_table_schema,
                       referenced_table_name,
                       referenced_column_name
                FROM {cat_quoted}.`information_schema`.`key_column_usage`
                WHERE table_schema = '{schema_normalized}'
                  AND table_name = '{table_normalized}'
                  AND referenced_table_name IS NOT NULL
            """
            df = self.spark.sql(simple_query)
            rels = [
                (
                    row.table_catalog,
                    row.table_schema,
                    row.table_name,
                    row.column_name,
                    row.referenced_table_catalog,
                    row.referenced_table_schema,
                    row.referenced_table_name,
                    row.referenced_column_name
                )
                for row in df.collect()
            ]
            if rels:
                self.logger.debug(f"Found {len(rels)} FK relationships for {cat_normalized}.{schema_normalized}.{table_normalized} via simple query")
        except Exception as e:
            error_str = str(e).lower()
            if 'table or view not found' in error_str or 'key_column_usage' in error_str:
                self.logger.debug(f"key_column_usage table not available in catalog {cat_normalized} (older Databricks version)")
            elif 'cannot resolve' in error_str or 'referenced_table' in error_str:
                self.logger.debug(f"referenced_table columns not available in key_column_usage (older Databricks version)")
            else:
                self.logger.debug(f"Simple FK query also failed for {cat_normalized}.{schema_normalized}.{table_normalized}: {str(e)[:150]}")
        
        self.foreign_key_graph[key] = rels
        return rels

    def get_foreign_key_relations(self, table_tuples: set):
        relations = []
        for tbl in table_tuples:
            relations.extend(self.foreign_key_graph.get(tbl, []))
        return relations
    
    def infer_foreign_keys_by_naming(self, all_column_details: list) -> dict:
        """
        Infer PK-FK relationships based on column naming conventions when explicit FK constraints don't exist.
        
        This uses common naming patterns to detect potential relationships:
        - table_id columns (e.g., customer_id in orders table likely references customers.id)
        - _fk suffix columns (e.g., customer_fk)
        - _ref suffix columns
        - Matching column names across tables (e.g., product_id in both products and order_items)
        
        Args:
            all_column_details: List of (catalog, schema, table, column, type, comment) tuples
            
        Returns:
            dict: Mapping of table tuples to list of inferred relationships
        """
        import re
        
        inferred_relations = defaultdict(list)
        
        tables_columns = defaultdict(list)
        for (catalog, schema, table, column, dtype, comment) in all_column_details:
            tables_columns[(catalog, schema, table)].append((column.lower(), column, dtype))
        
        table_names_map = {}
        for (catalog, schema, table) in tables_columns.keys():
            table_lower = table.lower()
            table_singular = table_lower.rstrip('s')
            table_names_map[table_lower] = (catalog, schema, table)
            table_names_map[table_singular] = (catalog, schema, table)
            if table_lower.endswith('ies'):
                table_names_map[table_lower[:-3] + 'y'] = (catalog, schema, table)
        
        id_patterns = [
            (r'^(.+)_id$', '_id'),
            (r'^(.+)_fk$', '_fk'),
            (r'^(.+)_ref$', '_ref'),
            (r'^(.+)_key$', '_key'),
            (r'^(.+)_sk$', '_sk'),    # surrogate key (Kimball dimensional modeling)
            (r'^(.+)_nk$', '_nk'),    # natural key (Kimball dimensional modeling)
            (r'^fk_(.+)$', 'fk_'),
            (r'^id_(.+)$', 'id_'),
            (r'^sk_(.+)$', 'sk_'),
            # camelCase / concatenated styles (no underscore): customerID, franchiseID, supplierID.
            # Match only when stem is >=3 chars to avoid "id"->"" or single-letter prefixes.
            (r'^([a-z][a-z0-9]{2,}?)id$', 'id'),
            (r'^([a-z][a-z0-9]{2,}?)key$', 'key'),
            (r'^([a-z][a-z0-9]{2,}?)code$', 'code'),
            (r'^([a-z][a-z0-9]{2,}?)sk$', 'sk'),
        ]
        
        for (catalog, schema, table), columns in tables_columns.items():
            for (col_lower, col_original, dtype) in columns:
                if 'int' not in dtype.lower() and 'long' not in dtype.lower() and 'string' not in dtype.lower():
                    continue
                    
                for pattern, suffix_type in id_patterns:
                    match = re.match(pattern, col_lower)
                    if match:
                        potential_table_name = match.group(1)
                        
                        # Try direct lookup first
                        ref_table_key = table_names_map.get(potential_table_name)
                        # If no direct match, try stripping short leading-prefix segments.
                        # Dimensional-modeling convention (Kimball): fact tables often prefix
                        # columns with a short table abbreviation, e.g. web_sales.ws_customer_sk
                        # points to `customer` via stem "ws_customer". Strip first 1 or 2
                        # underscore segments if each is <=4 chars (short prefix).
                        if not ref_table_key and "_" in potential_table_name:
                            parts = potential_table_name.split("_")
                            for _i in range(1, min(3, len(parts))):
                                if all(len(p) <= 4 for p in parts[:_i]):
                                    candidate = "_".join(parts[_i:])
                                    if not candidate:
                                        continue
                                    ref_table_key = table_names_map.get(candidate)
                                    if ref_table_key and ref_table_key != (catalog, schema, table):
                                        break
                                    cand_singular = candidate.rstrip("s")
                                    ref_table_key = table_names_map.get(cand_singular)
                                    if ref_table_key and ref_table_key != (catalog, schema, table):
                                        break
                        if ref_table_key and ref_table_key != (catalog, schema, table):
                            ref_catalog, ref_schema, ref_table = ref_table_key
                            
                            ref_cols = tables_columns.get(ref_table_key, [])
                            ref_pk_col = None
                            for (ref_col_lower, ref_col_original, _) in ref_cols:
                                if ref_col_lower in ('id', f'{potential_table_name}_id', 'pk', f'{ref_table.lower()}_id'):
                                    ref_pk_col = ref_col_original
                                    break
                            
                            if ref_pk_col is None:
                                for (ref_col_lower, ref_col_original, _) in ref_cols:
                                    if ref_col_lower.endswith('_id') or ref_col_lower == 'id':
                                        ref_pk_col = ref_col_original
                                        break
                            
                            if ref_pk_col:
                                rel_tuple = (
                                    catalog,
                                    schema,
                                    table,
                                    col_original,
                                    ref_catalog,
                                    ref_schema,
                                    ref_table,
                                    ref_pk_col
                                )
                                
                                existing = inferred_relations.get((catalog, schema, table), [])
                                if rel_tuple not in existing:
                                    inferred_relations[(catalog, schema, table)].append(rel_tuple)
                                    self.logger.debug(f"Inferred FK: {catalog}.{schema}.{table}.{col_original} -> {ref_catalog}.{ref_schema}.{ref_table}.{ref_pk_col}")
                        break
        
        return inferred_relations
    
    def merge_inferred_foreign_keys(self, all_column_details: list):
        """
        Merge inferred FK relationships into the foreign_key_graph.
        Only adds inferred relationships for tables that don't already have explicit FKs.
        
        Args:
            all_column_details: List of all column details to analyze
        """
        inferred = self.infer_foreign_keys_by_naming(all_column_details)
        
        added_count = 0
        for table_key, relations in inferred.items():
            existing = self.foreign_key_graph.get(table_key, [])
            if not existing:
                self.foreign_key_graph[table_key] = relations
                added_count += len(relations)
            else:
                for rel in relations:
                    if rel not in existing:
                        self.foreign_key_graph[table_key].append(rel)
                        added_count += 1
        
        if added_count > 0:
            self.logger.info(f"✅ Inferred {added_count} additional FK relationships from column naming patterns")
        
        return added_count

    # === REMOVED: _fill_from_direct_tables, _fill_from_schemas, _fill_from_catalogs ===

    # === MODIFIED: Support both single-pass and two-pass optimized modes ===
    def getNextTables(self, batch_size=None):
        """
        Returns a batch of tables with their column details.
        Maintains state across all catalogs and databases, continuing seamlessly
        across database boundaries.
        
        Supports two modes:
        1. Single-pass mode (default): Uses batch_size parameter for simple batching
        2. Two-pass mode (enable_two_pass=True): Uses pre-optimized batches, ignores batch_size
        
        Args:
            batch_size (int): Maximum number of tables to return in this batch (single-pass mode only)
            
        Returns:
            list of tuples: Each tuple is (catalog, schema, table, column_name, data_type, comment)
            None: if all tables have been processed
        """
        
        # Initialize table list on first call
        if not self._tables_initialized:
            self.logger.info("Initializing all tables from all databases...")
            self._initialize_all_tables()
            self._tables_initialized = True
        
        # === TWO-PASS OPTIMIZED MODE ===
        if self.enable_two_pass and self._size_analysis_complete:
            # Check if we've exhausted all batches
            if self.current_batch_idx >= len(self.optimized_batches):
                return None  # Signal no more batches
            
            # Get the next optimized batch
            batch_table_tuples = self.optimized_batches[self.current_batch_idx]
            batch_num = self.current_batch_idx + 1
            
            self.logger.info(f"📦 Fetching optimized batch {batch_num}/{len(self.optimized_batches)}: "
                           f"{len(batch_table_tuples)} tables")
            
            # Fetch column details for this optimized batch in parallel
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            column_parallelism, reason = calculate_adaptive_parallelism(
                "column_fetch", self.max_parallelism, 
                num_items=len(batch_table_tuples),
                is_llm_operation=False, logger=self.logger
            )
            
            all_column_details = []
            
            def fetch_details_with_sampling(args):
                return self._get_table_details(*args, apply_sampling=self.enable_column_sampling)
            
            with _SafeExecutorContext(max_workers=column_parallelism, thread_name_prefix="tbl_detail", logger=self.logger, name="TableDetailsBatch") as ctx:
                detail_futures = [ctx.submit(fetch_details_with_sampling, args) for args in batch_table_tuples]
                try:
                    for future in safe_as_completed(detail_futures, timeout=600):
                        try:
                            column_list = future.result(timeout=120)
                            if column_list:
                                all_column_details.extend(column_list)
                        except Exception as exc:
                            self.logger.warning(f"Table detail fetch failed: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Table details batch timed out globally (600s)")
            
            self.logger.info(f"   ✓ Batch {batch_num} loaded: {len(all_column_details)} columns from {len(batch_table_tuples)} tables")
            
            # Update position for next call
            self.current_batch_idx += 1
            
            # Return the column details for this optimized batch
            return all_column_details
        
        # === SINGLE-PASS MODE (backward compatible) ===
        else:
            if batch_size is None:
                self.logger.warning("batch_size not provided in single-pass mode, using default of 25")
                batch_size = 25
                
            # Check if we've exhausted all tables
            if self.current_table_idx >= len(self.all_table_tuples):
                return None  # Signal no more tables
                
            # Get the next batch of tables
            end_idx = min(self.current_table_idx + batch_size, len(self.all_table_tuples))
            batch_table_tuples = self.all_table_tuples[self.current_table_idx:end_idx]
            
            self.logger.info(f"Fetching batch of {len(batch_table_tuples)} tables (indices {self.current_table_idx} to {end_idx-1} of {len(self.all_table_tuples)} total)")
            
            # Fetch column details for this batch of tables in parallel
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            column_parallelism, reason = calculate_adaptive_parallelism(
                "column_fetch", self.max_parallelism,
                num_items=len(batch_table_tuples),
                is_llm_operation=False, logger=self.logger
            )
            
            all_column_details = []
            
            def fetch_details_with_sampling(args):
                return self._get_table_details(*args, apply_sampling=self.enable_column_sampling)
            
            with _SafeExecutorContext(max_workers=column_parallelism, thread_name_prefix="tbl_detail_sp", logger=self.logger, name="TableDetailsSinglePass") as ctx:
                detail_futures = [ctx.submit(fetch_details_with_sampling, args) for args in batch_table_tuples]
                try:
                    for future in safe_as_completed(detail_futures, timeout=600):
                        try:
                            column_list = future.result(timeout=120)
                            if column_list:
                                all_column_details.extend(column_list)
                        except Exception as exc:
                            self.logger.warning(f"Table detail fetch failed: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error("Table details single-pass batch timed out globally (600s)")
            
            # Batch complete - high-level logging only
            
            # Update position for next call
            self.current_table_idx = end_idx
            
            # Return the column details for this batch
            return all_column_details
    
    def reset(self):
        """Reset the data loader to start from the beginning."""
        self.current_table_idx = 0
        self.current_batch_idx = 0
        self.logger.info("DataLoader reset to beginning")
    
    def _initialize_all_tables(self):
        """
        Fetches all table names from all databases in the queue and stores them
        as (catalog, schema, table) tuples for later batch processing.
        
        Supports both streaming (memory-efficient) and two-pass (size-optimized) modes.
        """
        self.logger.info(f"Discovering all tables from {len(self.database_queue)} databases...")
        
        # Check if we have specific individual tables to filter by
        individual_tables_set = set(self.individual_tables) if self.individual_tables else None
        
        # Get set of explicitly provided schemas (these should expand ALL tables, not filter)
        explicit_schemas_set = getattr(self, 'explicit_schemas_set', set())
        
        # Log discovery strategy
        if explicit_schemas_set and individual_tables_set:
            explicit_schema_names = [f"{cat}.{db}" for cat, db in explicit_schemas_set]
            self.logger.info(f"📋 Discovery strategy: {len(explicit_schemas_set)} explicit schemas will expand ALL tables: {', '.join(explicit_schema_names)}")
            self.logger.info(f"📋 {len(individual_tables_set)} individual tables will be included from other schemas")
        
        # === TWO-PASS MODE: First analyze table sizes, then create optimized batches ===
        if self.enable_two_pass:
            self.logger.info("🔄 TWO-PASS MODE ENABLED: Analyzing table sizes for intelligent batching...")
            self.logger.info(f"⚡ Using parallel schema discovery with {self.max_parallelism} workers for speed")
            
            # PASS 1: Discover all tables and analyze their sizes IN PARALLEL
            all_table_size_infos = []
            
            # Function to process a single schema in parallel
            def discover_and_analyze_schema(schema_tuple):
                catalog_name, schema_name = schema_tuple
                cat_normalized = normalize_identifier(catalog_name)
                schema_normalized = normalize_identifier(schema_name)
                db_fqn = build_fqn(cat_normalized, schema_normalized)
                self.logger.info(f"🔍 Starting discovery for {db_fqn}...")
                
                schema_table_tuples = []
                schema_key = (cat_normalized, schema_normalized)
                is_explicit_schema = schema_key in explicit_schemas_set
                
                # Check if schema has a huge number of tables (>10K) - use streaming
                try:
                    self.logger.debug(f"   Counting tables in {db_fqn}...")
                    cat_quoted = quote_identifier(cat_normalized)
                    count_query = f"""
                        SELECT COUNT(*) as cnt FROM {cat_quoted}.`information_schema`.`tables`
                        WHERE `table_schema` = '{schema_normalized}'
                    """
                    table_count = self.spark.sql(count_query).collect()[0].cnt
                    self.logger.debug(f"   {db_fqn} has {table_count} tables")
                    use_streaming = table_count > 10000
                except Exception as e:
                    # Extract clean error message without stack trace for console
                    error_msg = get_clean_error_message(e)
                    self.logger.warning(f"   Could not count tables in {db_fqn}: {error_msg}. Using non-streaming mode.")
                    self.logger.debug(f"   Full error details: {get_clean_error_message(e, max_chars=2000)}")  # Full trace to log file only
                    use_streaming = False  # Fallback to non-streaming
                
                if use_streaming:
                    self.logger.info(f"   Large schema detected in {db_fqn} ({table_count:,} tables). Using streaming mode...")
                    # Process in chunks
                    for table_chunk in self._fetch_tables_for_schema_streaming(cat_normalized, schema_normalized, chunk_size=self.streaming_batch_size):
                        for fq_table_name in table_chunk:
                            cat, db, tbl = parse_three_level_name(fq_table_name)
                            if not (cat and db and tbl):
                                self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                                continue
                            table_tuple = (cat, db, tbl)
                            
                            # Only filter by individual_tables if this schema was NOT explicitly provided
                            # Explicit schemas should expand ALL tables
                            if not is_explicit_schema and individual_tables_set and table_tuple not in individual_tables_set:
                                continue
                            
                            schema_table_tuples.append(table_tuple)
                else:
                    # Non-streaming mode for smaller schemas
                    table_names_fq = self._fetch_tables_for_schema(cat_normalized, schema_normalized)
                    
                    if not table_names_fq:
                        self.logger.debug(f"No tables found in {db_fqn}. Skipping.")
                        return []  # Return empty list for this schema
                    
                    for fq_table_name in table_names_fq:
                        cat, db, tbl = parse_three_level_name(fq_table_name)
                        if not (cat and db and tbl):
                            self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                            continue
                        table_tuple = (cat, db, tbl)
                        
                        # Only filter by individual_tables if this schema was NOT explicitly provided
                        # Explicit schemas should expand ALL tables
                        if not is_explicit_schema and individual_tables_set and table_tuple not in individual_tables_set:
                            continue
                        
                        schema_table_tuples.append(table_tuple)
                
                # Analyze sizes for this schema's tables
                schema_size_infos = []
                if schema_table_tuples:
                    self.logger.debug(f"   Analyzing column counts for {len(schema_table_tuples)} tables in {db_fqn}...")
                    
                    # Process in chunks to avoid memory issues
                    # IMPORTANT: Use max_parallelism=1 to avoid nested parallelism deadlock
                    # (outer loop already runs schemas in parallel with self.max_parallelism workers)
                    chunk_size = 100
                    for i in range(0, len(schema_table_tuples), chunk_size):
                        chunk = schema_table_tuples[i:i+chunk_size]
                        size_infos = self.size_analyzer.analyze_table_sizes_batch(chunk, self.catalog_capabilities, max_parallelism=1)
                        schema_size_infos.extend(size_infos)
                    
                    self.logger.info(f"   ✓ {db_fqn}: Found {len(schema_table_tuples)} tables")
                
                return schema_size_infos
            
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            discovery_parallelism, reason = calculate_adaptive_parallelism(
                "schema_discovery", self.max_parallelism,
                num_items=len(self.database_queue),
                is_llm_operation=False, logger=self.logger
            )
            log_adaptive_parallelism_decision("schema_discovery", discovery_parallelism, self.max_parallelism, reason)
            
            with _SafeExecutorContext(max_workers=discovery_parallelism, thread_name_prefix="SchemaDiscovery", logger=self.logger, name="SchemaDiscovery") as ctx:
                futures = {ctx.submit(discover_and_analyze_schema, schema_tuple): schema_tuple 
                          for schema_tuple in self.database_queue}
                
                completed = 0
                total = len(self.database_queue)
                self.logger.info(f"      Submitted {total} schemas for parallel discovery...")
                
                timeout_per_schema = self.schema_timeout_seconds
                
                try:
                    for future in safe_as_completed(futures, timeout=timeout_per_schema * total):
                        schema_tuple = futures[future]
                        completed += 1
                        try:
                            schema_size_infos = future.result(timeout=timeout_per_schema)
                            all_table_size_infos.extend(schema_size_infos)
                            self.logger.info(f"      Progress: {completed}/{total} schemas analyzed, {len(all_table_size_infos)} tables discovered so far")
                        except concurrent.futures.TimeoutError:
                            timeout_minutes = timeout_per_schema // 60
                            self.logger.error(f"⏱️  Timeout analyzing schema {schema_tuple} (>{timeout_minutes} min). Skipping this schema.")
                        except Exception as e:
                            self.logger.error(f"Failed to analyze schema {schema_tuple}: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"⏱️  Overall schema discovery timeout ({timeout_per_schema * total}s). Proceeding with {completed}/{total} schemas.")
                
                if completed < total:
                    self.logger.warning(f"⚠️  Only {completed}/{total} schemas completed. {total - completed} schemas timed out or failed.")
            
            self.logger.info(f"📊 Pass 1 complete: Analyzed {len(all_table_size_infos)} tables")
            
            # Deduplicate tables (in case same table was included via both schema and individual table)
            seen_tables = set()
            unique_table_size_infos = []
            for info in all_table_size_infos:
                table_key = (info.catalog, info.schema, info.table)
                if table_key not in seen_tables:
                    seen_tables.add(table_key)
                    unique_table_size_infos.append(info)
            
            if len(unique_table_size_infos) < len(all_table_size_infos):
                duplicates_removed = len(all_table_size_infos) - len(unique_table_size_infos)
                self.logger.info(f"🔄 Deduplicated: Removed {duplicates_removed} duplicate tables, {len(unique_table_size_infos)} unique tables remaining")
                all_table_size_infos = unique_table_size_infos
            
            # PASS 2: Create optimized batches based on size analysis
            self.logger.info("🎯 Pass 2: Creating optimized batches based on table sizes...")
            self.optimized_batches = self.batch_optimizer.create_optimized_batches(all_table_size_infos)
            self._size_analysis_complete = True
            
            # Store all table tuples for reference
            self.all_table_tuples = [
                (info.catalog, info.schema, info.table) for info in all_table_size_infos
            ]
            
            self.logger.info(f"✅ Two-pass initialization complete: {len(self.all_table_tuples)} tables in {len(self.optimized_batches)} optimized batches")
        
        # === SINGLE-PASS MODE: Standard behavior (for backward compatibility) ===
        else:
            self.logger.info("📋 SINGLE-PASS MODE: Standard table discovery with parallel queries...")
            self.logger.info(f"⚡ Using {self.max_parallelism} parallel workers for schema queries")
            
            # Function to fetch tables for a single schema
            def fetch_schema_tables(schema_tuple):
                catalog_name, schema_name = schema_tuple
                cat_normalized = normalize_identifier(catalog_name)
                schema_normalized = normalize_identifier(schema_name)
                db_fqn = build_fqn(cat_normalized, schema_normalized)
                self.logger.info(f"🔍 Starting table fetch for {db_fqn}...")
                
                schema_key = (cat_normalized, schema_normalized)
                is_explicit_schema = schema_key in explicit_schemas_set
                
                # Fetch all table names for this database
                table_names_fq = self._fetch_tables_for_schema(cat_normalized, schema_normalized)
                
                schema_tuples = []
                if not table_names_fq:
                    self.logger.debug(f"No tables found in {db_fqn}. Skipping.")
                    return schema_tuples
                    
                # Parse into (cat, schema, table) tuples
                for fq_table_name in table_names_fq:
                    cat, db, tbl = parse_three_level_name(fq_table_name)
                    if not (cat and db and tbl):
                        self.logger.warning(f"Skipping malformed table name: {fq_table_name}")
                        continue
                    table_tuple = (cat, db, tbl)
                    
                    # Explicit schemas expand ALL tables; otherwise filter by individual_tables
                    if is_explicit_schema:
                        # Schema was explicitly provided - include ALL tables
                        schema_tuples.append(table_tuple)
                    elif individual_tables_set:
                        # Schema came from individual tables - only include those specific tables
                        if table_tuple in individual_tables_set:
                            schema_tuples.append(table_tuple)
                    else:
                        # No individual tables specified - include all
                        schema_tuples.append(table_tuple)
                
                self.logger.info(f"   ✓ {db_fqn}: Found {len(schema_tuples)} tables")
                return schema_tuples
            
            # ADAPTIVE PARALLELISM: Fixed for metadata (DB connection limits)
            table_discovery_parallelism, reason = calculate_adaptive_parallelism(
                "table_discovery", self.max_parallelism,
                num_items=len(self.database_queue),
                is_llm_operation=False, logger=self.logger
            )
            log_adaptive_parallelism_decision("table_discovery", table_discovery_parallelism, self.max_parallelism, reason)
            
            with _SafeExecutorContext(max_workers=table_discovery_parallelism, thread_name_prefix="TableDiscovery", logger=self.logger, name="TableDiscovery") as ctx:
                futures = {ctx.submit(fetch_schema_tables, schema_tuple): schema_tuple 
                          for schema_tuple in self.database_queue}
                
                completed = 0
                total = len(self.database_queue)
                self.logger.info(f"      Submitted {total} schemas for parallel discovery...")
                
                timeout_per_schema = self.schema_timeout_seconds
                
                try:
                    for future in safe_as_completed(futures, timeout=timeout_per_schema * total):
                        schema_tuple = futures[future]
                        completed += 1
                        try:
                            schema_tuples = future.result(timeout=timeout_per_schema)
                            self.all_table_tuples.extend(schema_tuples)
                            self.logger.info(f"      Progress: {completed}/{total} schemas processed, {len(self.all_table_tuples)} tables discovered so far")
                        except concurrent.futures.TimeoutError:
                            timeout_minutes = timeout_per_schema // 60
                            self.logger.error(f"⏱️  Timeout fetching tables for schema {schema_tuple} (>{timeout_minutes} min). Skipping this schema.")
                        except Exception as e:
                            self.logger.error(f"Failed to fetch tables for schema {schema_tuple}: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"⏱️  Overall table discovery timeout ({timeout_per_schema * total}s). Proceeding with {completed}/{total} schemas.")
                
                if completed < total:
                    self.logger.warning(f"⚠️  Only {completed}/{total} schemas completed. {total - completed} schemas timed out or failed.")
            
            # Deduplicate tables (in case same table was included via both schema and individual table)
            original_count = len(self.all_table_tuples)
            self.all_table_tuples = list(dict.fromkeys(self.all_table_tuples))  # Preserves order, removes duplicates
            if len(self.all_table_tuples) < original_count:
                duplicates_removed = original_count - len(self.all_table_tuples)
                self.logger.info(f"🔄 Deduplicated: Removed {duplicates_removed} duplicate tables, {len(self.all_table_tuples)} unique tables remaining")
            
            self.logger.info(f"Table discovery complete. Found {len(self.all_table_tuples)} total tables across all databases.")

# COMMAND ----------

# DBTITLE 1,Chart Type Reference Guide
# Chart Type Reference Guide:
# The generator maps SQL columns to chart fields in two ways:
# 1. By Prefix: Looks for columns starting with a specific prefix (e.g., `category_`, `value_`).
# 2. By Position: If no prefixes are found, it uses the column order from the SELECT statement.
#
# Supported formats (optional columns in brackets `[]`):
# --------------------------------------------------------------------------------
# Bar Chart:      category_col, value_col, [group_col]
# Pie Chart:      category_col, value_col
# Line Chart:     x_axis_col, y_axis_col, [group_col]
# Area Chart:     x_axis_col, y_axis_col, [group_col]
# Scatter Plot:   x_axis_col, y_axis_col, [group_col]
# Heatmap:        x_axis_col, y_axis_col, color_col
# Combo Chart:    x_axis_col, bar_value_col, line_value_col
#
# Counter:        value_col
# Histogram:      value_col
# Funnel Chart:   stage_col, value_col
#
# Box Plot:       category_col, min_col, q1_col, median_col, q3_col, max_col
# Sankey Chart:   source_col, destination_col, value_col
# Pivot Table:    row_col, column_col, cell_value_col
# Choropleth Map: location_col, value_col
# Symbol Map:     lat_col, lon_col, [size_col], [group_col]
#
# Table:          col_1, col_2, ... (all columns are displayed)
# --------------------------------------------------------------------------------

# (Chart Type Reference Guide... unchanged)

class WidgetFailedToCreate(Exception):
    """Custom exception raised when widget creation fails due to unmet requirements."""
    pass

TRAIN_INFER_TECHNIQUES = {
    "Classification", "Predictive Modeling", "Regression Analysis",
    "Anomaly Detection", "Clustering", "Segmentation",
    "Recommendation", "Survival Analysis", "Forecasting",
}

TECHNIQUE_IMPLEMENTATION_GUIDANCE = {
    "Anomaly Detection": (
        "ANALYTICAL APPROACH MANDATE -- ANOMALY DETECTION:\n"
        "You MUST use ML-based anomaly detection as the PRIMARY analytical engine. "
        "Recommended algorithms (choose best fit for data characteristics): "
        "Isolation Forest (best for high-dimensional tabular data, handles mixed feature types well), "
        "Local Outlier Factor / LOF (best for density-based local anomalies), "
        "DBSCAN (best for spatial/density-based clustering where anomalies are noise points), "
        "One-Class SVM (best for well-defined normal class with clear boundary), "
        "Ensemble methods combining multiple detectors for robust scoring. "
        "Use sklearn.ensemble.IsolationForest, sklearn.neighbors.LocalOutlierFactor, "
        "sklearn.cluster.DBSCAN, or spark.ml equivalents -- all run natively on Databricks Serverless. "
        "Statistical methods (z-scores, IQR, Grubbs test, MAD) are acceptable as SUPPLEMENTARY signals "
        "or fallback when sample size is too small for ML, but are NEVER sufficient as the sole detection method. "
        "The ML model produces anomaly scores, anomaly flags, and feature importance; "
        "ai_query then EXPLAINS the detected anomalies, diagnoses root causes, quantifies cost impact, "
        "and generates corrective action recommendations. "
        "MINIMUM BAR: Simple z-score thresholds or SQL CASE WHEN on a single metric is a FAILURE for Anomaly Detection."
    ),
    "Forecasting": (
        "ANALYTICAL APPROACH MANDATE -- FORECASTING:\n"
        "You MUST use time-series modeling as the PRIMARY analytical engine. "
        "Recommended approaches (choose best fit): "
        "Prophet (best for business time series with seasonality, holidays, and trend changes), "
        "ARIMA/SARIMA via statsmodels (best for stationary series with clear autocorrelation), "
        "ai_forecast() Databricks built-in function (simplest path for standard forecasting), "
        "Temporal regression models (XGBoost/LightGBM with lag features, rolling statistics, calendar features) "
        "for complex multi-variate forecasting where external regressors matter. "
        "Feature engineering MUST include: lag features, rolling means/std, day-of-week/month/quarter encodings, "
        "holiday indicators, trend decomposition components. "
        "The forecasting model produces point forecasts, confidence intervals, and trend/seasonality decomposition; "
        "ai_query then EXPLAINS the forecast, identifies key drivers of predicted trends, "
        "assesses forecast uncertainty, and generates strategic recommendations. "
        "MINIMUM BAR: Simple moving averages or SQL window function LAG/LEAD alone is a FAILURE for Forecasting."
    ),
    "Classification": (
        "ANALYTICAL APPROACH MANDATE -- CLASSIFICATION:\n"
        "You MUST use ML classification models as the PRIMARY analytical engine. "
        "Recommended algorithms (choose based on data size, interpretability needs, and feature types): "
        "XGBoost/LightGBM (best accuracy for tabular data, handles missing values, feature importance built-in), "
        "Random Forest (good baseline, robust to overfitting, interpretable feature importance), "
        "Logistic Regression (best when interpretability is paramount, works well with regularization), "
        "ai_classify() Databricks built-in (acceptable for text-based or semantic classification tasks). "
        "Pipeline MUST include: feature engineering, train/test split, model training, "
        "evaluation metrics (accuracy, precision, recall, F1, AUC-ROC, confusion matrix), "
        "and MLflow model registration (with infer_signature and input_example -- Unity Catalog requires model signatures). "
        "The ML model produces class predictions, prediction probabilities, and feature importance scores; "
        "ai_query then EXPLAINS each classification decision, provides confidence assessment, "
        "and generates actionable recommendations based on the predicted class. "
        "MINIMUM BAR: SQL CASE WHEN rules or manual threshold-based classification is a FAILURE for Classification."
    ),
    "Predictive Modeling": (
        "ANALYTICAL APPROACH MANDATE -- PREDICTIVE MODELING:\n"
        "You MUST train ML models as the PRIMARY analytical engine. "
        "Recommended algorithms: "
        "XGBoost (best general-purpose for tabular prediction), "
        "LightGBM (best for large datasets with many features), "
        "Random Forest (robust baseline with good interpretability), "
        "Gradient Boosting (sklearn GradientBoostingClassifier/Regressor for smaller datasets). "
        "Pipeline MUST include: exploratory feature analysis, feature engineering (derived features, interactions, encodings), "
        "train/validation/test split, hyperparameter selection, model training, "
        "comprehensive evaluation (regression: RMSE, MAE, R-squared, MAPE; classification: AUC, F1, precision, recall), "
        "feature importance extraction, and MLflow model registration (with infer_signature and input_example -- Unity Catalog requires model signatures). "
        "The ML model produces predictions and confidence scores; "
        "ai_query then EXPLAINS each prediction, interprets feature contributions, "
        "quantifies business impact, and generates prioritized action recommendations. "
        "MINIMUM BAR: SQL aggregations or rule-based scoring without trained ML model is a FAILURE for Predictive Modeling."
    ),
    "Clustering": (
        "ANALYTICAL APPROACH MANDATE -- CLUSTERING:\n"
        "You MUST use unsupervised ML algorithms as the PRIMARY analytical engine. "
        "Recommended algorithms: "
        "K-Means (best for spherical clusters with known k, use elbow method or silhouette score for k selection), "
        "DBSCAN (best for arbitrary-shaped clusters and noise detection, no need to pre-specify k), "
        "Gaussian Mixture Models / GMM (best for overlapping clusters with probabilistic assignment), "
        "Hierarchical/Agglomerative Clustering (best for exploring cluster hierarchy at multiple granularities). "
        "Pipeline MUST include: feature scaling/normalization, dimensionality assessment, "
        "optimal cluster count determination (silhouette score, elbow method, BIC for GMM), "
        "cluster assignment, cluster profiling (centroid analysis, feature distributions per cluster). "
        "The ML model produces cluster assignments, distance-to-centroid scores, and cluster profiles; "
        "ai_query then PROFILES each cluster in business terms, names the segments meaningfully, "
        "and generates targeted strategy recommendations per cluster. "
        "MINIMUM BAR: SQL GROUP BY or NTILE-based bucketing is a FAILURE for Clustering."
    ),
    "Regression Analysis": (
        "ANALYTICAL APPROACH MANDATE -- REGRESSION ANALYSIS:\n"
        "You MUST use statistical or ML regression models as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "Linear Regression with regularization (Ridge/Lasso/ElasticNet for feature selection and multicollinearity), "
        "Generalized Linear Models / GLM (for non-normal response distributions: Poisson, Gamma, Binomial), "
        "Polynomial regression or splines (for non-linear relationships), "
        "Tree-based regression (XGBoost/Random Forest regressor for complex non-linear patterns). "
        "Pipeline MUST include: feature correlation analysis, multicollinearity detection (VIF), "
        "residual diagnostics, R-squared/adjusted R-squared, coefficient interpretation, "
        "statistical significance testing (p-values, confidence intervals for coefficients). "
        "The regression model produces predicted values, residuals, coefficients, and significance metrics; "
        "ai_query then INTERPRETS the regression results in business terms, explains which factors drive outcomes, "
        "quantifies the marginal effect of each driver, and recommends interventions. "
        "MINIMUM BAR: SQL CORR() function or simple ratio computation alone is a FAILURE for Regression Analysis."
    ),
    "Segmentation": (
        "ANALYTICAL APPROACH MANDATE -- SEGMENTATION:\n"
        "You MUST use ML-driven segmentation as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "K-Means on RFM (Recency, Frequency, Monetary) or domain-specific multi-dimensional features, "
        "DBSCAN for density-based segmentation that handles outliers naturally, "
        "Gaussian Mixture Models for soft/probabilistic segment assignment, "
        "Hierarchical clustering for discovering natural segment hierarchy. "
        "Feature engineering MUST include: behavioral features (recency, frequency, monetary value, engagement metrics), "
        "demographic/firmographic features, lifecycle stage indicators, normalized/scaled features. "
        "Pipeline MUST include: feature engineering, scaling, optimal segment count determination, "
        "segment assignment, segment profiling with statistical summaries. "
        "The ML model produces segment labels, segment membership scores, and segment profiles; "
        "ai_query then NAMES each segment in business-friendly terms, profiles segment characteristics, "
        "and generates segment-specific strategy recommendations. "
        "MINIMUM BAR: SQL CASE WHEN tier assignments or NTILE-based quantile bucketing is a FAILURE for Segmentation."
    ),
    "Cohort Analysis": (
        "ANALYTICAL APPROACH MANDATE -- COHORT ANALYSIS:\n"
        "You should use a combination of cohort construction, retention/survival analysis, and optionally ML scoring. "
        "Recommended approaches: "
        "Cohort retention matrices (cohort definition by time period, tracking metric over subsequent periods), "
        "Kaplan-Meier survival curves per cohort for time-to-event analysis, "
        "ML-based cohort scoring (train a model to predict cohort-level outcomes like LTV, retention probability), "
        "Statistical comparison tests between cohorts (t-test, chi-square, Mann-Whitney U test). "
        "Pipeline MUST include: cohort definition logic, retention/conversion tracking over time periods, "
        "cohort comparison metrics, statistical significance of inter-cohort differences. "
        "ai_query then EXPLAINS cohort performance differences, identifies what distinguishes high-performing cohorts, "
        "and recommends strategies to improve underperforming cohorts. "
        "SQL-based cohort construction is acceptable as the foundation, but MUST be enriched with "
        "statistical testing or ML-based scoring beyond simple aggregation."
    ),
    "Trend Analysis": (
        "ANALYTICAL APPROACH MANDATE -- TREND ANALYSIS:\n"
        "You MUST use statistical trend detection methods beyond simple SQL window functions. "
        "Recommended approaches: "
        "Seasonal decomposition (STL decomposition to separate trend, seasonality, residuals), "
        "Mann-Kendall test for statistically significant trend detection, "
        "Change-point detection (PELT, Binary Segmentation, or Bayesian methods) to identify regime changes, "
        "Holt-Winters exponential smoothing for trend + seasonality modeling, "
        "Linear regression on time variable with statistical significance testing. "
        "Pipeline MUST include: time-series construction, decomposition into components, "
        "trend significance testing, change-point identification, volatility measurement. "
        "ai_query then INTERPRETS the trend patterns, explains inflection points and regime changes, "
        "assesses whether trends are statistically significant or noise, and recommends strategic responses. "
        "MINIMUM BAR: SQL LAG/LEAD or simple period-over-period percentage change alone is a FAILURE for Trend Analysis."
    ),
    "Correlation Analysis": (
        "ANALYTICAL APPROACH MANDATE -- CORRELATION ANALYSIS:\n"
        "You MUST use multi-dimensional correlation and relationship analysis beyond simple pairwise correlation. "
        "Recommended approaches: "
        "Correlation matrix with Pearson, Spearman (rank), and Kendall (ordinal) coefficients, "
        "Partial correlation (controlling for confounding variables), "
        "Mutual Information for detecting non-linear dependencies, "
        "Principal Component Analysis / PCA for dimensionality reduction and latent factor discovery, "
        "Variance Inflation Factor / VIF for multicollinearity detection. "
        "Pipeline MUST include: multi-variable correlation computation, statistical significance testing, "
        "identification of spurious correlations, partial correlation to isolate direct relationships, "
        "visualization-ready correlation matrices. "
        "ai_query then INTERPRETS correlations in business context, warns about correlation-vs-causation, "
        "identifies the strongest actionable relationships, and recommends causal investigation priorities. "
        "MINIMUM BAR: SQL CORR() on two columns alone is a FAILURE for Correlation Analysis."
    ),
    "Pareto Analysis": (
        "ANALYTICAL APPROACH MANDATE -- PARETO ANALYSIS:\n"
        "You should use concentration analysis with statistical rigor. "
        "Recommended approaches: "
        "Cumulative distribution analysis with Pareto frontier identification, "
        "Gini coefficient computation for inequality measurement, "
        "Lorenz curve construction for visual concentration assessment, "
        "Entropy-based concentration metrics (Herfindahl-Hirschman Index / HHI), "
        "Segmented Pareto analysis across multiple dimensions simultaneously. "
        "SQL-based cumulative percentage computation is acceptable as the foundation, "
        "but MUST include concentration metrics (Gini, HHI) and multi-dimensional cross-cutting analysis. "
        "ai_query then EXPLAINS the concentration pattern, identifies the vital few contributors, "
        "quantifies the resource allocation imbalance, and recommends rebalancing strategies. "
        "MINIMUM BAR: Simple PERCENT_RANK or cumulative sum alone without concentration metrics is insufficient."
    ),
    "Funnel Analysis": (
        "ANALYTICAL APPROACH MANDATE -- FUNNEL ANALYSIS:\n"
        "You should use conversion modeling with statistical rigor and optionally ML-based bottleneck prediction. "
        "Recommended approaches: "
        "Stage-to-stage conversion rate computation with confidence intervals, "
        "Markov chain modeling for transition probabilities between funnel stages, "
        "Bottleneck identification using drop-off rate anomaly detection, "
        "ML-based conversion prediction (predict which entities will convert at each stage), "
        "A/B test framework for comparing funnel performance across segments. "
        "SQL-based stage counting is acceptable as the foundation, "
        "but MUST include statistical significance testing and conversion prediction or Markov modeling. "
        "ai_query then EXPLAINS conversion bottlenecks, identifies root causes of drop-offs, "
        "compares funnel performance across segments, and recommends specific optimization actions. "
        "MINIMUM BAR: Simple COUNT(*) at each stage without statistical testing or modeling is insufficient."
    ),
    "Sentiment Analysis": (
        "ANALYTICAL APPROACH MANDATE -- SENTIMENT ANALYSIS:\n"
        "You MUST use NLP-based sentiment analysis as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "ai_classify() with sentiment-specific prompts for scalable text classification, "
        "ai_query for nuanced sentiment extraction with aspect-level granularity, "
        "TextBlob or VADER via Python UDFs for rule-based sentiment scoring as a baseline, "
        "Custom prompt engineering for domain-specific sentiment categories beyond positive/negative/neutral. "
        "Pipeline MUST include: text preprocessing (cleaning, normalization), "
        "sentiment scoring (positive/negative/neutral + confidence), "
        "aspect-level sentiment extraction (WHAT specifically is positive/negative about each entity), "
        "temporal sentiment trend analysis, entity-level sentiment aggregation. "
        "ai_query then SYNTHESIZES sentiment patterns across entities and time, "
        "identifies sentiment drivers and emerging concerns, and recommends response strategies. "
        "MINIMUM BAR: Simple keyword matching or SQL LIKE patterns is a FAILURE for Sentiment Analysis."
    ),
    "Document Processing": (
        "ANALYTICAL APPROACH MANDATE -- DOCUMENT PROCESSING:\n"
        "You MUST use NLP extraction pipelines as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "ai_query or ai_extract() for structured data extraction from unstructured text, "
        "Regex-based pattern extraction for well-defined formats (dates, IDs, amounts, codes), "
        "Multi-pass extraction pipeline: first extract entities, then classify document type, then validate cross-field consistency, then score confidence. "
        "Pipeline MUST include: text normalization, entity extraction, field validation/cross-referencing, "
        "confidence scoring for each extraction, structured output table creation, "
        "error handling for malformed or ambiguous documents. "
        "ai_query performs the core extraction AND validates results, generates quality assessments, "
        "and flags uncertain extractions for human review with specific reason codes. "
        "MINIMUM BAR: A single ai_query call without validation, confidence scoring, or error handling is insufficient."
    ),
    "Extraction": (
        "ANALYTICAL APPROACH MANDATE -- EXTRACTION:\n"
        "You MUST use NLP entity recognition and pattern extraction as the PRIMARY engine. "
        "Recommended approaches: "
        "ai_extract() for named entity recognition from text fields, "
        "ai_query with structured JSON output for complex multi-field extraction, "
        "Regex pipelines for well-defined patterns combined with AI for ambiguous cases, "
        "Multi-stage extraction: coarse extraction -> disambiguation -> normalization -> deduplication. "
        "Pipeline MUST include: entity extraction, entity normalization (standardize formats, resolve aliases), "
        "deduplication (merge duplicate entities), confidence scoring per extracted entity, "
        "cross-reference validation against known entity lists where available. "
        "ai_query validates extractions, resolves ambiguities, normalizes entities, "
        "and generates structured output with per-field confidence assessments. "
        "MINIMUM BAR: A single extraction call without normalization, deduplication, or confidence scoring is insufficient."
    ),
    "AI Analysis": (
        "ANALYTICAL APPROACH MANDATE -- AI ANALYSIS:\n"
        "You have full freedom to choose the optimal analytical approach based on data characteristics. "
        "This technique category is a catch-all -- you MUST select the BEST method for the specific problem: "
        "ML models (XGBoost, Random Forest, Isolation Forest) when predicting outcomes or detecting patterns, "
        "Clustering (K-Means, DBSCAN) when discovering natural groups in the data, "
        "Statistical methods (regression, correlation, hypothesis testing) when quantifying relationships, "
        "AI functions (ai_query, ai_classify, ai_extract) when semantic understanding is the primary requirement. "
        "The approach MUST be a HYBRID: statistical pre-processing and feature engineering to prepare the data, "
        "followed by the most appropriate ML or AI technique for the core analysis, "
        "with ai_query providing final business interpretation and recommendation generation. "
        "MINIMUM BAR: Pure SQL aggregation wrapped in ai_query is insufficient for AI Analysis -- "
        "there must be genuine analytical depth (multi-dimensional feature engineering, statistical testing, "
        "or ML model training) in the pre-ai_query pipeline."
    ),
    "Simulation": (
        "ANALYTICAL APPROACH MANDATE -- SIMULATION:\n"
        "You MUST use computational simulation techniques as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "Monte Carlo simulation (generate 1000+ random scenarios from estimated input distributions), "
        "Sensitivity analysis (systematically vary each input parameter +/- 10-50% to measure output elasticity), "
        "Scenario modeling (define optimistic/baseline/pessimistic parameter sets with business justification, compute outcomes for each), "
        "What-If analysis with parameter sweeps across business-relevant ranges, "
        "Bootstrap resampling for confidence interval estimation on business metrics. "
        "Pipeline MUST include: input distribution estimation from historical data, "
        "scenario generation (1000+ iterations for Monte Carlo, 3-5 named scenarios for scenario modeling), "
        "outcome computation for each scenario, probability distribution of outcomes, "
        "sensitivity ranking of input parameters by impact on output variance. "
        "ai_query then INTERPRETS simulation results, explains which factors drive outcome uncertainty, "
        "quantifies risk probabilities (e.g., P(loss > $1M) = 12%), and recommends risk mitigation strategies. "
        "MINIMUM BAR: SQL CASE WHEN with 2-3 hardcoded scenarios is a FAILURE for Simulation."
    ),
    "Geospatial Analysis": (
        "ANALYTICAL APPROACH MANDATE -- GEOSPATIAL ANALYSIS:\n"
        "You MUST use spatial analysis techniques as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "H3 hexagonal indexing for spatial aggregation and region-to-region comparison, "
        "Geohash encoding for efficient spatial grouping and proximity queries, "
        "DBSCAN spatial clustering for identifying geographic hotspots and coldspots, "
        "Haversine/Vincenty distance computation for accurate distance-based analysis, "
        "Spatial autocorrelation (Moran's I) for detecting geographic clustering patterns, "
        "Kernel density estimation for continuous spatial density surfaces. "
        "Pipeline MUST include: coordinate normalization and validation, spatial indexing, "
        "proximity analysis, density estimation, spatial clustering, "
        "distance matrix computation where relevant, spatial statistics. "
        "ai_query then INTERPRETS spatial patterns in business terms, explains geographic concentrations, "
        "identifies underserved areas or coverage gaps, and recommends location-based strategies. "
        "MINIMUM BAR: Simple latitude/longitude column inclusion without spatial computation is a FAILURE."
    ),
    "Survival Analysis": (
        "ANALYTICAL APPROACH MANDATE -- SURVIVAL ANALYSIS:\n"
        "You MUST use time-to-event statistical or ML models as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "Kaplan-Meier estimator for non-parametric survival curves per group, "
        "Cox Proportional Hazards model for identifying risk factors affecting survival time, "
        "Accelerated Failure Time / AFT models for parametric survival modeling, "
        "ML-based survival: Random Survival Forests, XGBoost with survival objectives (via scikit-survival). "
        "Use lifelines Python library (works on Databricks Serverless) or sklearn-compatible survival packages. "
        "Pipeline MUST include: survival time computation, censoring handling (right-censored observations), "
        "Kaplan-Meier curves with confidence bands, hazard ratio estimation, "
        "covariate significance testing (log-rank test for group comparisons). "
        "ai_query then INTERPRETS survival curves and hazard ratios in business terms, "
        "explains which factors accelerate or delay the event, and recommends intervention timing. "
        "MINIMUM BAR: SQL DATEDIFF to compute time-to-event without survival modeling is a FAILURE."
    ),
    "Recommendation": (
        "ANALYTICAL APPROACH MANDATE -- RECOMMENDATION:\n"
        "You MUST use recommendation algorithms as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "ALS (Alternating Least Squares) collaborative filtering via spark.ml for implicit/explicit feedback, "
        "Cosine similarity or Pearson correlation for item-based or user-based collaborative filtering, "
        "Content-based filtering using TF-IDF or feature vectors with similarity metrics, "
        "Hybrid approaches combining collaborative and content-based signals with weighted scoring, "
        "Matrix factorization (SVD, NMF) for latent factor discovery. "
        "Pipeline MUST include: interaction matrix construction, train/test split, model training, "
        "recommendation generation with relevance scores, diversity/novelty metrics, cold-start handling strategy. "
        "ai_query then EXPLAINS why each item is recommended, provides business context for the recommendation, "
        "and generates personalized recommendation narratives with expected impact. "
        "MINIMUM BAR: SQL-based top-N by simple COUNT/SUM without learned similarity is a FAILURE."
    ),
    "Network Analysis": (
        "ANALYTICAL APPROACH MANDATE -- NETWORK ANALYSIS:\n"
        "You MUST use graph-based analysis as the PRIMARY analytical engine. "
        "Recommended approaches: "
        "GraphFrames on Spark for scalable graph processing (available on Databricks), "
        "PageRank for influence/importance scoring of nodes, "
        "Community detection (Label Propagation, Connected Components, Louvain) for group discovery, "
        "Centrality metrics (betweenness, closeness, degree, eigenvector) for node importance ranking, "
        "Shortest path algorithms for relationship distance computation, "
        "networkx for smaller graphs that fit in driver memory. "
        "Pipeline MUST include: graph construction from relational data (nodes and edges), "
        "centrality computation, community detection, influence propagation analysis, "
        "graph-level statistics (density, diameter, clustering coefficient). "
        "ai_query then INTERPRETS network structure in business terms, identifies key influencers and brokers, "
        "explains community composition, and recommends network-based strategies. "
        "MINIMUM BAR: SQL self-joins without graph metrics is a FAILURE for Network Analysis."
    ),
    "Optimization": (
        "ANALYTICAL APPROACH MANDATE -- OPTIMIZATION:\n"
        "You MUST use mathematical optimization or heuristic algorithms as the PRIMARY engine. "
        "Recommended approaches: "
        "Linear Programming / LP via scipy.optimize.linprog for linear objective with linear constraints, "
        "Mixed Integer Programming / MIP via PuLP or scipy for discrete decisions, "
        "Constraint satisfaction via scipy.optimize.minimize for non-linear optimization, "
        "Genetic algorithms or simulated annealing for complex combinatorial problems, "
        "Multi-objective optimization with Pareto frontier identification. "
        "Pipeline MUST include: objective function definition, constraint specification, "
        "feasibility checking, optimal solution computation, sensitivity analysis on constraints, "
        "comparison of optimal vs current state to quantify improvement potential. "
        "ai_query then INTERPRETS the optimal solution in business terms, explains tradeoffs, "
        "quantifies improvement over current state, and recommends implementation steps. "
        "MINIMUM BAR: SQL-based ranking or sorting without mathematical optimization is a FAILURE."
    ),
}

PROMPT_TEMPLATES["BEST_OF_BEST_SCORING_PROMPT"] = """# BEST-OF-BEST RANKING — per-UC 19-gate scoring + idea theme

You are a Principal Business Analyst ranking a shortlist of already-accepted use cases so the portfolio can be trimmed to a fundable ~{target_max} size while preserving idea coverage.

**The 19 quality gates (D1-D19)** — you already know these; each UC here has already passed the HARD VETO floor. Your job now is to give each gate a fine-grained **0-10 score** per UC:

- D1 Causal Signal — how directly do the data columns CAUSE or indicate the outcome?
- D2 Cause-Effect Validity — how industry-recognized is the cause-effect relationship?
- D3 Data Granularity — how well does the data level match the analysis level?
- D4 Critical Dimensions Present — are ALL required columns actually in the schema?
- D5 Logical Possibility — is the operation possible given data types?
- D6 Metric Validity — is the claimed metric truly calculable from the fields?
- D7 Design-Schema Match — does the technical design cite columns that exist?
- D8 Semantic Uniqueness — after verb/noun/entity/metric normalization, how unique is this?
- D9 Analytical Depth (Strip Test) — remove ML mentions, is real substance left?
- D10 Activation Quality (Deliverable Test) — "with [...]" names a business deliverable, not a technique?
- D11 Domain and Technique Balance — how distinctive vs the portfolio's cluster centers?
- D12 Business Relevance & Opportunity — (business stakeholder) relevant to day-to-day ops? tangible value? named consumer team? current pain or future opportunity?
- D13 Sponsor Test — (CFO) fundable with a real budget line? pitchable to the CEO without caveats?
- D14 Engineering Test — (production engineer) deployable on existing Databricks capabilities? supportable long-term? fixable without bespoke infra?
- D15 Decision Cadence Match — does data refresh speed match decision speed?
- D16 The Monday Test — would a stakeholder's Monday look DIFFERENT? names a concrete action?
- D17 Explainability & Audit Defensibility — can the output be defended in 5 minutes without proprietary magic?
- D18 18-Month Longevity — still running in 18 months or abandoned?
- D19 Attributable Impact — can a measurable dollar amount be proven 6 months post-deployment?

**Scoring rubric** — use the full 0-10 range. Do NOT default to 7-8 for everything.

- 0-2: Catastrophic fail. This gate is NOT met. (Rare at this stage — most HARD VETOes already dropped upstream.)
- 3-4: Weak. Technically passes but the answer to the gate question is "barely".
- 5-6: Acceptable. Clear "yes" but not exceptional.
- 7-8: Strong. Gate clearly satisfied with evidence.
- 9-10: Exceptional. Gate is a clear strength of this UC; best-in-class.

**Spread scores**. If two UCs tie at 8 across every gate you're not doing your job — force yourself to differentiate. This is a ranking exercise; the whole point is separation.

## Idea theme

Assign each UC to a **2-4 word idea theme**. The theme names the ANALYTICAL IDEA the UC embodies, NOT the business domain. Examples:

- "Anomaly detection"
- "Churn prediction"
- "Demand forecasting"
- "Pricing optimization"
- "Customer segmentation"
- "Root-cause attribution"
- "Risk scoring"
- "Growth/expansion planning"
- "Fraud detection"
- "Inventory optimization"

Use CONSISTENT theme labels across the batch so UCs on the same idea get the same label. Aim for ~15-25 distinct themes across the full input.

## Output

Return a JSON array, one object per UC, in the same order as the input. Zero prose outside the JSON.

```json
[
  {{
    "No": "N01-U03",
    "bob_scores": {{"D1": 8, "D2": 7, "D3": 9, "D4": 9, "D5": 10, "D6": 8, "D7": 9, "D8": 7, "D9": 6, "D10": 8, "D11": 6, "D12": 9, "D13": 7, "D14": 8, "D15": 7, "D16": 9, "D17": 6, "D18": 8, "D19": 7}},
    "idea_theme": "Anomaly detection",
    "bob_justification_short": "Strong causal signal from transaction-level fraud indicators, explainable rules-based baseline, clear Monday trigger (investigation queue)."
  }},
  ...
]
```

**Rules**
- Every UC gets all 19 D-scores. None omitted.
- `bob_justification_short` is one sentence, ≤160 chars.
- Idea themes should be SHORT, TITLE CASE (no emoji, no punctuation).
- Return ONLY the JSON array.

## Inputs

**Business Context**: {business_context}
**Industry**: {industry}
**Strategic Goals**: {strategic_goals}
**Business Priorities**: {business_priorities}

{generation_instructions_section}

**Target portfolio size after trim**: {target_max}

## Use Cases to Score

{use_cases_to_validate}
"""

PROMPT_TEMPLATES["USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"] = """You are generating a **goal-oriented instruction** for code implementation.

**RULES:**
- ZERO executable code in your output (no SELECT, no Python, no PySpark) -- except the ai_query prompt template in Section 4. Code = FAILURE.
- You MUST name the analytical approach category and algorithm families. Missing approach = FAILURE.
- ai_query is the FINAL EXPLANATION layer only -- it does NOT perform core analysis. Core analysis uses ML/statistical models BEFORE ai_query.
- Do NOT expand table columns or list schemas -- the implementer inspects schemas directly.
- Be CONCISE throughout. Every sentence must add value. No filler, no repetition.

**Your job:** The instruction must communicate: (1) business goal and why (Sections 1-2), (2) input tables by name only (Section 3), (3) analytical approach directive (Section 4A), (4) output column categories and naming conventions (Section 5), (5) ai_query contract (Section 4C). Everything else -- joins, features, hyperparameters, code structure -- is your decision as the implementer.

**BUSINESS CONTEXT:**
- Company: {business_name}
- Context: {enriched_business_context}
- Strategic Goals: {enriched_strategic_goals}
- Business Priorities: {enriched_business_priorities}
- Strategic Initiative: {enriched_strategic_initiative}
- Value Chain: {enriched_value_chain}
- Revenue Model: {enriched_revenue_model}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**USE CASE:**
- ID: {use_case_id} | Name: {use_case_name}
- Domain: {business_domain} > {subdomain}
- Type: {use_case_type} | Technique: {analytics_technique}
- Statement: {statement}
- Solution: {solution}
- Business Value: {business_value}
- High Level Design: {high_level_design}
- Tables: {tables_involved} | Primary: {primary_table}
- Result Table: {result_table}

**TABLES (directly involved -- you will inspect columns autonomously):**
{directly_involved_schema}

**FOREIGN KEY RELATIONSHIPS:**
{foreign_key_relationships}

**ALL AVAILABLE TABLES (you may use ANY for enrichment):**
{available_tables_listing}

---

Generate the instruction using EXACTLY this 7-section structure. Each section starts with `## N. Section Name`. Output ONLY the instruction text -- no preamble, no markdown fences wrapping the output.

**CRITICAL: Use ONLY plain ASCII characters in your output.** No smart quotes, no em dashes, no non-breaking hyphens, no special Unicode characters. Use straight quotes (' and "), double hyphens (--) instead of em dashes, and regular hyphens (-) instead of en dashes. The output must contain ONLY characters in the ASCII range (0x00-0x7E).

## 1. Persona

Generate a compact role-based persona (NO person names -- role titles only). This SAME persona condensed to 1 sentence must be reused in the ai_query prompt.

**QUALITY BAR**: EXACTLY 2-3 sentences, 80 words MAXIMUM, ZERO invented facts:
- Sentence 1: "You are a [Senior Title] at {business_name} with analytics responsibility for this use case."
- Sentence 2: State WHICH supplied columns and tables the persona reasons over (e.g., "Grounded in `<table>.<column>`, `<table>.<column>` from `Tables Involved`"). Do NOT invent revenue figures, fleet size, customer counts, transaction volumes, employee counts, location counts, Fortune rankings, or any other firm-scale number. If a scale fact is not supplied as an input or directly derivable from a column in `Tables Involved`, OMIT it.
- Sentence 3 (optional, only if supported by data): One sentence stating this use case's criticality in terms of what the supplied columns can measure -- e.g., "Improving this decision optimizes `<measure_col>` rolled up over `<date_col>`." Do NOT quantify with `$` or `%` values unless those values are computable from a named column in `Tables Involved`.
Do NOT elaborate on competitive dynamics, hedging strategies, or institutional knowledge details -- those belong in Section 2.

## 2. Task

**A. Mission (1-2 sentences):** State the mission with the specific system name and primary business outcome. Include: `MERGE INTO {result_table}` (the table will be created if it does not exist and data will be upserted, NOT overwritten)

**B. Business Value (1 paragraph, 5-8 sentences max, GROUNDED ONLY):** Describe the decisions this UC enables and the operational consequences of poor decisions. Frame risk/cost categories QUALITATIVELY (e.g., "stockout risk", "margin erosion", "late payment exposure") and tie each category to a SPECIFIC column in `Tables Involved` that would evidence the category (e.g., "stockout risk is measurable via low `quantity` events on `dateTime`"). Any numeric quantification MUST be tagged as `[ESTIMATE: depends on <column_name>]` rather than stated as a fabricated figure. Do NOT write dollar ranges, percentage ranges, multi-million or multi-billion impact claims, market-share statements, industry-benchmark claims, fleet size, customer count, transaction volume, employee count, location count, or any other firm-scale number that is not computable from a named supplied column. Connect to {business_name}'s goals: "{enriched_strategic_goals}" without inventing figures to prove alignment.

**C. Success Scenario (1 paragraph):** WHO uses it (job titles only -- no team-size guesses unless supplied), HOW (workflow built on specific filters over named columns from `Tables Involved`), WHAT downstream systems they hand off to (describe by CATEGORY -- "the operational scheduling system", "the finance posting system" -- NOT by a named vendor product unless supplied as input), WHAT decisions they change. Pattern: "The [role] opens this, filters by `<column>` / `<column>`, reviews the AI-produced fields, and updates the [downstream-system-category] accordingly." Do NOT invent team sizes, meeting names, or specific vendor product names.

**D. Failure Scenario (3-5 sentences):** Describe what degrades if the UC is not implemented, grounded in the columns available (e.g., "Without this decomposition of `<measure_col>` by `<date_col>` and `<dimension_col>`, the team sees only aggregate roll-ups and misses the pattern"). Do NOT name specific competitors or specific external companies; do NOT cite invented dollar or percentage figures. If quantitative impact is genuinely implied, tag it `[ESTIMATE: depends on <column_name>]`. End with: "This is a ZERO-FAILURE use case."

**QUALITY BAR**: Section 2 must be specific and quantified but concise -- no filler or repetition. B: 1 focused paragraph with 2-4 numbered cost categories and dollar figures. C: 1 paragraph with WHO/HOW/WHAT specifics. D: 3-5 sentences. Every sentence must contain specific dollar figures or operational details.

## 3. Input

List the tables you should use. For each table provide:
- Fully qualified name (e.g., `catalog.schema.table`)
- Role: fact table, dimension, reference, or enrichment
- 1-sentence description of what business data it contains and why it matters for this use case

Structure as:
**Primary Fact Table(s):** -- tables where each row is a business event
**Dimension Tables:** -- tables describing entities (who, what, where)
**Enrichment Opportunities:** -- additional tables from the available schema that could add value

Do NOT describe join paths, foreign keys, column names, column schemas, or SQL logic here. Do NOT expand or list individual columns from these tables. You have direct access to all table schemas and will inspect columns, discover join keys, and figure out the implementation yourself.

**Enrichment Table Usage (MANDATORY):** ALL tables listed in the Enrichment Opportunities section MUST be loaded and incorporated into the analysis pipeline. If a listed enrichment table cannot be used (e.g., no viable join key, incompatible granularity, empty table), you MUST document the specific reason in ai_sys_feedback with estimated accuracy impact. Silently ignoring listed enrichment tables is FORBIDDEN.

**Table Discovery (MANDATORY):** You must NOT be limited to only the tables listed here. Before implementation, you MUST scan all tables in the same schema(s) and catalog(s) as the listed tables (using SHOW TABLES IN, INFORMATION_SCHEMA, or DESCRIBE SCHEMA) to discover additional relevant tables. For each discovered table that might be relevant, you MUST inspect its schema (DESCRIBE TABLE or SHOW COLUMNS) to evaluate enrichment potential. Document all discovered tables in ai_sys_feedback and log as an MLflow artifact -- including tables inspected but not used, with the reason.


**Lineage and Relationship Discovery (MANDATORY):** Before implementation, you MUST query Unity Catalog metadata to discover which tables are most commonly joined with your input tables. Use `INFORMATION_SCHEMA.TABLE_CONSTRAINTS`, `INFORMATION_SCHEMA.KEY_COLUMN_USAGE`, and `system.access.table_lineage` (if available) to find upstream/downstream tables and common join patterns. For each discovered related table, evaluate whether it adds analytical value to this use case. If it does, incorporate it into your implementation. Document all lineage-discovered tables in ai_sys_feedback and MLflow artifacts. If lineage tables are unavailable or access is restricted, proceed with the listed tables but note the limitation in ai_sys_feedback.

**External Data Recommendation:** If external data would materially improve accuracy (20%+), mention what data source would help and the estimated accuracy improvement. You will decide how to simulate or integrate it.

## 4. Implementation Strategy

**EXACTLY 4 PARTS (A-D). No executable code, no column formulas, no hyperparameters.** You provide: (A) analytical approach directive, (B) mission goal, (C) ai_query contract, (D) MLflow contract. You decide joins, features, hyperparameters, and code structure.

**YOU MUST** name the analytical approach CATEGORY (ML, statistical, simulation, hybrid) and algorithm FAMILY (e.g., "Isolation Forest", "XGBoost", "K-Means", "Monte Carlo"). You must NOT write executable code or specify hyperparameter values.

**PART A -- Analytical Approach Directive (THIS IS THE MOST CRITICAL PART):**

The analytics technique for this use case is: `{analytics_technique}`

{technique_implementation_guidance}

Based on the above technique guidance, state in 3-5 sentences: (1) analytical approach category, (2) algorithm family names, (3) what the pipeline produces, (4) that ai_query ONLY explains pre-computed results -- not performs analysis, (5) what constitutes a FAILURE for this technique.

**PART B -- Mission Goal (3-5 sentences MAXIMUM):**
State what this use case must achieve in plain business language. What decision does the output enable? Who benefits? What is the expected outcome? Reference the analytical approach from Part A: "Using [technique] to [detect/predict/cluster/forecast], this system enables [decision] for [beneficiary]."

**PART C -- ai_query Contract (you must follow these rules EXACTLY):**
- Model: `'databricks-gpt-oss-120b'`
- Prompt construction: `format_string(...)` only -- never concatenation, f-strings, or CONCAT()
- Temperature: `modelParameters => named_struct('temperature', CAST(0.1 AS DOUBLE))`
- Banned: systemPrompt, system_prompt, temperature as direct param
- ai_query is the FINAL pipeline stage -- it receives pre-computed analytical results (ML scores, predictions, clusters, statistical measures) and generates all `ai_cat_*`, `ai_txt_*`, and `ai_sys_*` columns
- ai_query does NOT perform core analysis -- it EXPLAINS and RECOMMENDS based on results computed upstream by the ML/statistical pipeline
- The SAME format_string expression must populate both the ai_query call AND the `ai_sys_prompt` auditability column
- No deep learning (no TensorFlow/PyTorch/Keras)

**CRITICAL -- PROMPT FORMATTING FOR USER EDITABILITY:**
The per-record prompt MUST be defined as a triple-quoted multi-line Python string (using triple quotes: \"\"\"...\"\"\") so that formatting is visually apparent in the notebook cell. Users will read and edit this prompt -- it must look like a well-structured document, NOT a concatenation of single-line strings with escape characters. Each section must start on its own line with a `# Section` header. Blank lines between sections.

The prompt must follow this multi-line structure:
```
# Persona
[Same role from Section 1, condensed to 1-2 sentences for per-record context]

# Task
[1-2 sentences: EXPLAIN the pre-computed analytical results, diagnose root causes, quantify financial impact, recommend actions]

# Input
[ALL analytical fields AND ML/statistical scores injected via %s placeholders, CAST non-strings to STRING.
Group related fields logically with BLANK LINES between groups (not just line breaks -- actual empty lines for visual separation):
- Entity identifiers (ID, type, name, age, etc.)

- Operational metrics (utilization, efficiency, volumes, etc.)

- ML/statistical outputs (scores, predictions, clusters, optimal values, NPVs, etc.)

- Sensitivity and probability metrics

- Peer comparison and fleet/cohort metrics]

# Output
[JSON with every ai_cat_, ai_txt_, ai_sys_ field. List each field on its own line with allowed values:
- ai_cat_xxx (values: Value1, Value2, Value3, Value4)
- ai_cat_yyy (values: ...)
- ai_txt_executive_summary
- ai_txt_xxx
- ai_sys_importance (values: Critical, High, Medium, Low)
- ai_sys_urgency (values: Immediate, Near-Term, Medium-Term, Long-Term)
- ai_sys_confidence (0.00 to 1.00)
- ai_sys_feedback]

# Constraints
- Zero hallucination: reference only the Input data provided.
- Quantify every statement with specific numbers from Input fields.
- Output ONLY valid JSON with no markdown, no explanatory text, no code blocks.
```
The Input section MUST include ALL pre-computed ML/statistical outputs (anomaly scores, prediction probabilities, cluster assignments, forecast values, regression coefficients, scenario NPVs, sensitivity rankings, etc.) so ai_query can reference them in its explanation. List every specific ai_cat_, ai_txt_, ai_sys_ field name the prompt must request, with allowed values for categorical fields.

**PART D -- MLflow Experiment Registration (MANDATORY):**
You MUST register the pipeline as an MLflow experiment (NON-NEGOTIABLE for every use case). Experiment name: `{use_case_id}_<descriptive_name>`. Log: parameters (config choices, data filters, model type, row count), metrics (record count, completeness, ML evaluation metrics if applicable, segment/cluster size distribution if clustering, ai_sys_confidence mean from output table), artifacts (ai_query prompt template as .txt, feature engineering logic summary, discovered tables list with usage status, model output profiles as JSON), tags (output table, use_case_id, technique). Register model in MLflow Model Registry if ML is used (name: "{use_case_id}_model"). **MODEL SIGNATURE (CRITICAL -- NON-NEGOTIABLE for Unity Catalog):** When registering ANY model via `mlflow.<flavor>.log_model(..., registered_model_name=...)`, you MUST provide a model signature using `mlflow.models.infer_signature(X_train, model.predict(X_train))` passed as the `signature` parameter, AND provide `input_example=X_train[:5]` (or similar small sample). Unity Catalog REJECTS models without signature metadata -- omitting the signature causes a fatal `Execution error: Model passed for registration did not contain any signature metadata`. This is a HARD REQUIREMENT, not optional. You decide exact API calls and code structure.

## 5. Output

**Target Table:** `{result_table}`

**Table Write Strategy (MANDATORY -- MERGE, NOT OVERWRITE):** You MUST use MERGE INTO (upsert) to write results into the target table, NOT `CREATE OR REPLACE TABLE` or `.mode("overwrite").saveAsTable(...)`. The correct pattern in PySpark is:

```
# Step 1: Ensure table exists (first run only)
spark.sql("CREATE TABLE IF NOT EXISTS {result_table} AS SELECT * FROM temp_results WHERE 1=0")

# Step 2: Create temp view from results DataFrame
output_df.createOrReplaceTempView("_inspire_merge_source")

# Step 3: MERGE (upsert) -- update existing, insert new
spark.sql(\"\"\"
    MERGE INTO {result_table} AS target
    USING _inspire_merge_source AS source
    ON target.<primary_key> = source.<primary_key>
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
\"\"\")
```

Identify the natural primary key for the merge condition (e.g., transactionID, assetID, customerID, entityID). If NO natural primary key exists in the source data, generate a deterministic surrogate key using `md5(concat_ws('|', col1, col2, ...))` on the columns that together uniquely identify a record, and add this as a `record_key` column to the output. This prevents data loss from accidental overwrites, supports incremental updates, and is safe for re-runs. `CREATE OR REPLACE TABLE` and `.mode("overwrite")` are FORBIDDEN for the final output table -- they destroy historical data and are not idempotent in a multi-user environment. You decide the specific merge key and implementation details.

You decide HOW to populate this table AND which specific columns to include. This section defines the COLUMN CATEGORIES and NAMING CONVENTIONS only. Do NOT enumerate every individual column -- you will inspect the input table schemas and determine the appropriate columns autonomously.

**Verification (you must run after table creation):**
1. `SELECT * FROM {result_table} LIMIT 5`
2. `SELECT COUNT(*) as total_records, COUNT(DISTINCT <primary_entity_column>) as unique_entities FROM {result_table}`

**Required Column Categories (you determine specific columns within each -- describe the CATEGORY purpose and minimum expectations, NOT individual column names):**

1. **Business & Entity Columns (expect 5-15 columns)** -- Entity identifiers (IDs, codes, names), geographic dimensions, temporal dimensions, and categorical attributes from the source tables. Describe what KINDS of entity attributes matter for this use case (e.g., "customer identifiers and segmentation attributes", "asset IDs and location hierarchies"). You inspect input schemas and select the relevant columns.

2. **Analytical & Statistical Columns (expect 8-15 columns)** -- Pre-computed outputs from the ML/statistical pipeline that feed into ai_query. MUST include technique-appropriate model outputs (e.g., anomaly scores, predictions, cluster assignments, forecasts, scenario NPVs, sensitivity rankings -- whichever matches the technique from Section 4 Part A). Also include computed business metrics, statistical measures, peer comparisons, and derived features. You decide specifics.

3. **Data Quality Columns (REQUIRED naming convention, 2-4 columns):**
   - `data_sufficiency_flag` (STRING), `data_completeness_score` (DECIMAL), and outlier flags as appropriate for the domain

4. **AI-Generated Columns (produced by ai_query -- REQUIRED naming conventions, 8-12 columns):**
   - `ai_cat_<name>` (STRING) -- 1-3 categorical classification columns with domain-relevant categories
   - `ai_txt_executive_summary` (STRING), `ai_txt_business_outcome` (STRING), `ai_txt_recommended_actions` (STRING)
   - 2-4 additional `ai_txt_<name>` columns specific to this use case domain -- describe the KINDS of AI insight needed (e.g., "root cause analysis", "competitive context", "risk factors")
   - `ai_sys_importance` (STRING), `ai_sys_urgency` (STRING), `ai_sys_confidence` (DECIMAL), `ai_sys_feedback` (STRING)

5. **MLflow Tracking Columns:** `mlflow_experiment_name` (STRING), `mlflow_run_id` (STRING)

6. **Metadata Columns:** `analysis_timestamp` (TIMESTAMP), `use_case_id` (STRING) = "{use_case_id}", `use_case_name` (STRING) = "{use_case_name}"

7. **Auditability (MUST BE LAST):** `ai_sys_prompt` (STRING) -- Complete verbatim prompt sent to ai_query

The total output table should have approximately 25-40 columns across all categories. You determine the exact columns by inspecting the input table schemas. You define the CATEGORIES, NAMING CONVENTIONS, and MINIMUM EXPECTATIONS -- not individual column names.

**ai_query Prompt Structure (you must use for EVERY record -- USERS WILL EDIT THIS):**
The prompt MUST be a triple-quoted multi-line Python string (\"\"\"...\"\"\") with visual structure matching Section 4 Part C: `# Section` headers on their own lines, blank lines between sections, logical field grouping in Input, per-field listing with allowed values in Output. Users need to read and modify this prompt -- it must NOT be a concatenated single-line string.
The SAME format_string must populate both the ai_query call AND the `ai_sys_prompt` column.

**QUALITY BAR**: Describe column CATEGORIES with naming conventions. Do NOT list 30-50 individual columns. Focus on the ai_query contract, naming conventions, and ensuring the prompt template is comprehensive and well-formatted for user editing.

## 6. Constraints

Generate constraints that are SPECIFIC to this use case -- reference the actual table names and business thresholds:

- **Zero Hallucination:** Name the specific tables you must use as starting points. You will discover available columns by inspecting table schemas directly and MUST also scan nearby tables in the same schema(s) and catalog(s) for additional relevant data (using SHOW TABLES IN, INFORMATION_SCHEMA, or DESCRIBE SCHEMA). If required data doesn't exist in any discovered tables, flag in ai_sys_feedback with estimated accuracy impact (e.g., "Missing X reduces accuracy by ~Y%"). **CRITICAL: NEVER fabricate data using random number generators (np.random, random.uniform, etc.) to simulate metrics that cannot be derived from actual data.** If a metric (e.g., response lag, competitor reaction time, customer lifetime value) cannot be computed from available tables, set the column to NULL and document in ai_sys_feedback: "Column X set to NULL -- not derivable from available data. Would require [specific data source] to compute." Fabricating data with random numbers violates the Zero Hallucination constraint and constitutes implementation failure.
- **Enrichment Table Mandate:** ALL tables listed in Section 3 Enrichment Opportunities MUST be loaded and used. Do NOT silently skip enrichment tables. If a listed table cannot be joined or provides no usable signal, document the specific reason in ai_sys_feedback.
- **Statistical Integrity:** Define specific sample size thresholds for this use case (e.g., "Flag entities with <10 observations by setting `data_sufficiency_flag = 'Insufficient'` and capping `ai_sys_confidence` at 0.50"). Specify precision requirements: standard deviations to 2 decimal places, correlation coefficients to 3 decimal places, percentages to 1 decimal place, confidence intervals at 95% level. When time-window features require a minimum observation period (e.g., 90 days for volatility, 12 months for seasonality), validate data availability per entity and flag in ai_sys_feedback if insufficient: "Entity X has only N days of observations; volatility estimate may be unstable." When time-series decomposition requires more cycles than available, flag the metric as "Insufficient Data" rather than computing a misleading value.
- **ai_query Compliance:** Model `'databricks-gpt-oss-120b'` only, `format_string` only, `modelParameters => named_struct('temperature', CAST(0.1 AS DOUBLE))`. Banned: systemPrompt, temperature as direct param, concatenation.
- **Auditability:** `ai_sys_prompt` must contain the COMPLETE verbatim prompt -- not a summary. Same format_string expression for ai_query and ai_sys_prompt column.
- **Notebook Identity Preservation (CRITICAL -- NON-NEGOTIABLE):** Do NOT rename, modify, or change the notebook name/title under any circumstances. The notebook already has a name assigned by the user or the system. Your job is ONLY to generate code cells within the existing notebook. Renaming the notebook is FORBIDDEN and constitutes an implementation failure. Leave the notebook name exactly as it is.
- **Databricks Serverless Compatibility (CRITICAL -- NON-NEGOTIABLE):** ALL code MUST run on Databricks Serverless compute. The following are FORBIDDEN and will cause runtime errors: (1) `spark.conf.get("spark.databricks.*")` -- this includes ALL configs under the `spark.databricks` namespace such as `spark.databricks.clusterUsageTags.*`, `spark.databricks.notebook.userName`, `spark.databricks.notebook.path`, `spark.databricks.job.*`, etc. NONE of these exist on Serverless and ALL raise `[CONFIG_NOT_AVAILABLE]` gRPC errors. For getting the current user, use `spark.sql('SELECT current_user() AS user').collect()[0]['user']` (most reliable, works everywhere) or `from databricks.sdk import WorkspaceClient; w = WorkspaceClient(); user = w.current_user.me().user_name` as an alternative. For getting the notebook path, use `dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()` instead. For MLflow experiment paths, use `_user = spark.sql('SELECT current_user() AS user').collect()[0]['user']; mlflow.set_experiment(f"/Users/{{_user}}/{{EXPERIMENT_NAME}}")` or simply `mlflow.set_experiment(f"/{{EXPERIMENT_NAME}}")`. (2) `spark.sparkContext`, `sc`, `SparkContext`, or any direct SparkContext access; (3) `.cache()`, `.persist()`, `.unpersist()`, `.checkpoint()` on DataFrames; (4) `spark.catalog.setCurrentDatabase()` -- use fully qualified table names instead; (5) Any reference to cluster IDs, driver IPs, or executor counts via Spark config. For REST API authentication (REST API operations, etc.), use `from databricks.sdk import WorkspaceClient; w = WorkspaceClient(); host = w.config.host; token = w.config.token`. Alternatively, `dbutils.notebook.entry_point.getDbutils().notebook().getContext()` works on Serverless for `apiUrl()`, `apiToken()`, and `jobId().isDefined()`.

## 7. Remarks

Generate remarks covering these topics with domain-specific detail:

- **Quantification Mandate:** Every ai_txt_ column must cite specific numbers from the input data. Give examples of good vs bad (e.g., "load_volatility of 12.3 percentage points, which is 2.1 standard deviations above cohort mean" vs "high volatility"). Never use vague qualifiers without numbers.
- **Analytical Approach Mandate:** You MUST use the technique from Section 4 Part A (`{analytics_technique}`). Non-negotiables: (1) approach matches the directive -- for ML-designated techniques, SQL-only (thresholds, z-scores, CASE WHEN) is a FAILURE; (2) output table follows Section 5 column categories and naming; (3) ai_query is the final EXPLANATION layer only. You have full autonomy over algorithm choice within the family, hyperparameters, features, joins, and code structure.
- **End-to-End Execution:** You must run the complete pipeline end-to-end: data extraction, analysis, ai_query enrichment, table creation (via MERGE INTO), verification queries, and job cleanup. If any step fails, implement error handling (flag affected records with data quality columns, continue processing remaining records). The pipeline MUST complete without unhandled exceptions -- wrap risky operations (API calls, ai_query) in try/except blocks with meaningful error messages and graceful degradation.
- **Self-Critique (MUST be a SEPARATE markdown cell):** Create a dedicated markdown cell (cell_type="markdown") -- NOT a print() statement inside code -- rating the implementation 1-10. Include a Missing Data Table: | Data Source | Impact on Accuracy | What It Would Enable | Priority |. Be SPECIFIC about each missing source and its estimated impact. Note: when constructing the markdown cell content in Python f-strings, use a single `%` for literal percent signs (f-strings do not require `%%` escaping -- that is only for old-style `%` formatting).
- No `ai_sys_missing_data` column -- missing data commentary belongs in ai_sys_feedback and the self-critique cell.
- **Table Discovery:** You must scan the same schema(s) and catalog(s) as the listed input tables (SHOW TABLES IN, INFORMATION_SCHEMA, DESCRIBE SCHEMA) to find additional relevant tables before implementation. For each potentially relevant discovered table, inspect its schema (DESCRIBE TABLE) to evaluate join keys and enrichment potential. Document ALL discovered tables in ai_sys_feedback and MLflow artifacts -- both used and not used, with reasoning. Do NOT limit analysis to only the explicitly listed tables. ALL enrichment tables listed in Section 3 MUST be incorporated or explicitly justified as unusable in ai_sys_feedback.
- **ai_query Prompt Formatting (NON-NEGOTIABLE):** The prompt_tmpl MUST be a triple-quoted multi-line string (\"\"\"...\"\"\") with `# Section` headers, blank lines between sections, logical field grouping in Input, and per-field listing in Output. Users will edit this -- NOT a concatenated single-line string.
- **Verified Execution (CRITICAL -- PASS/FAIL GATE):** The code you generate MUST run successfully end-to-end in a Databricks Job without ANY errors. The Job Launcher will submit the notebook as a job run -- if that job run fails, the ENTIRE implementation is a FAILURE regardless of how good the code looks. Before considering the implementation complete, you MUST: (1) ensure every code path is exercised and produces valid output, (2) verify the output table is actually created and populated with the expected number of records by running the verification queries from Section 5, (3) confirm all SQL statements use valid syntax and reference only columns that exist in the data, (4) test that all API calls (MLflow, Lakeview, ai_query) succeed or are wrapped in try/except with graceful fallback. If the code cannot run without errors on Databricks Serverless compute, it is NOT done. "It should work" is NOT acceptable -- it MUST work.
- **Job Cleanup After Execution (MANDATORY):** After the job run completes (success or failure), the job created for this notebook run MUST be cleaned up. Add a cleanup step at the END of each main function (`main()`, `main_train()`, or `main_infer()` -- whichever applies to this use case) after all verification queries that deletes the job used for the test run. Use the Databricks SDK: `from databricks.sdk import WorkspaceClient; w = WorkspaceClient(); ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext(); job_id = ctx.jobId().get() if ctx.jobId().isDefined() else None; if job_id: w.jobs.delete(job_id=int(job_id))`. Wrap in try/except so cleanup failure does not mask pipeline success. This prevents job accumulation in the workspace from repeated runs.
- **Do NOT Rename the Notebook (NON-NEGOTIABLE):** You MUST NOT rename, retitle, or modify the notebook name in any way. The notebook has an existing name -- leave it unchanged. Only generate code cells and markdown cells inside the notebook. Any attempt to change the notebook name is an IMPLEMENTATION FAILURE.
- **MERGE Into Final Table (NON-NEGOTIABLE -- NO BLIND INSERT/OVERWRITE):** You MUST use MERGE INTO (upsert) to write data into the final output table, NOT `CREATE OR REPLACE TABLE`, `.mode("overwrite")`, or `.write.saveAsTable(...)` with overwrite mode. The correct pattern is: (1) `CREATE TABLE IF NOT EXISTS {result_table} (...)` with full schema definition; (2) Create a temporary view from the results DataFrame; (3) `MERGE INTO {result_table} AS target USING <temp_view> AS source ON target.<primary_key> = source.<primary_key> WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *`. The primary key should be the natural entity identifier (e.g., transactionID, assetID, customerID). This ensures: idempotent re-runs (running twice does not duplicate data), preservation of historical records from prior runs, safe concurrent writes in multi-user environments. Using `CREATE OR REPLACE TABLE` or `.mode("overwrite")` on the final output table is IMPLEMENTATION FAILURE.
- **100% Complete and Working Code Delivery (NON-NEGOTIABLE):** You MUST deliver code that is 100% complete, fully functional, and runs without modification. Partial implementations, placeholder code, TODO comments, stub functions, commented-out sections, or "left as exercise" patterns are ALL implementation failures. Every function must be fully implemented. Every SQL query must be syntactically valid and reference real columns. Every API call must use correct endpoints and parameters. Every import must be used. Every variable must be defined before use. The notebook must execute from top to bottom in a fresh Databricks Serverless job context and produce the final output table, MLflow experiment without human intervention. If you are uncertain whether a code path works, add defensive error handling with logging -- do NOT leave it broken or incomplete.
{code_structure_mandate}

**OUTPUT FORMAT**: Generate ONLY the 7-section instruction (Sections 1-7). Section 8 (Job Launcher Cell) is auto-appended by the system -- do NOT generate it. Each section starts with `## N. Section Name`. No preamble, no code blocks wrapping the output, no explanation outside the sections.
"""

# --- 1i. Business vs Technical Table Filtering Prompt (NEW - CRITICAL FOR QUALITY) ---
PROMPT_TEMPLATES["GOAL_ALIGNMENT_FILTER_PROMPT"] = """You are a **Goal Alignment Classifier**. Your ONLY job is to determine whether each use case aligns with the user's stated goal.

### USER GOAL
{goal_description}

### USE CASES TO CLASSIFY
{use_cases_table}

### RULES
1. For EACH use case, output ONLY the use case ID and YES or NO.
2. YES = the use case DIRECTLY serves the user's stated goal.
3. NO = the use case does NOT serve the user's stated goal, even if it is a valid business use case.
4. **EXPANSION — the following ARE ALWAYS operational/growth decisions (mark YES when detected)**, regardless of how the goal is worded:
   - **Fraud, abuse, anomaly detection, misuse detection, leakage, chargeback, arbitrage** → these protect revenue (operational goal) or prevent loss (growth goal). ALWAYS YES.
   - **Compliance / tax / regulatory / policy-exposure checks** → these reduce operational risk and financial exposure. ALWAYS YES when the goal contains "operational", "growth", "risk", "revenue", "margin", or "cost".
   - **Return-loss / refund-abuse / pricing-integrity / discount-abuse** → operational margin protection. ALWAYS YES.
   - **Data-quality / listing-integrity / catalog-standardization** → reduces operational cost of errors. YES when goal contains "operational".
5. Be STRICT OTHERWISE. If the use case is tangentially related but not directly about the goal, mark it NO. But do NOT reject a use case merely because it mentions "fraud" or "compliance" — those ARE operational decisions.
6. Do NOT add explanations, comments, or any other text.

### OUTPUT FORMAT
Return a CSV with exactly 2 columns: ID,Aligned
No header row. One row per use case. Example:
AI-P1_1-U1,YES
AI-P1_1-U2,NO
AI-P1_1-U3,YES

Start your response with the first ID immediately. No preamble.
"""

PROMPT_TEMPLATES["FILTER_BUSINESS_TABLES_PROMPT"] = """You are a **Senior Data Architect** classifying database tables as BUSINESS or TECHNICAL.

**BUSINESS CONTEXT**:
- **Business**: {business_name} | **Industry**: {industry}
- **Description**: {business_context}
- **Exclusion Strategy**: {exclusion_strategy}

{additional_context_section}

### NON-NEGOTIABLE GENERATION INSTRUCTIONS

{generation_instructions_section}


**CONTEXT-AWARE RULE**: Terms technical for most businesses may be BUSINESS for companies where they are core product (e.g., `clusters` is BUSINESS for Databricks but TECHNICAL for a retailer. `cluster_logs` is always TECHNICAL).

**UNIVERSAL TECHNICAL PATTERNS** (always TECHNICAL regardless of context):
`*_logs`, `*_audit_trail`, `*_changelog`, `*_snapshot`, `*_backup`, `information_schema.*`, `sys.*`, `*_metrics`, `*_health`, `*_monitoring`, `etl_*`, `pipeline_*` (orchestration), `*_error`, `*_exception`, `*_debug`, `*_config` (system), `*_test`, `*_staging`, `*_temp`, ML/platform infra metadata.
**Exception**: If business sells these as products (e.g., observability company), they're BUSINESS.


**🧱 MEDALLION + RAG/AI-PREP PATTERNS — CURATED BUSINESS TABLES, NOT TECHNICAL**

Tables named with medallion-architecture suffixes describe DATA QUALITY TIERS, not ETL artifacts. The content is still business data:

- `*_bronze_*` / `*_bronze` — raw landing; **TECHNICAL** (lake raw-tier, still transformed downstream)
- `*_silver_*` / `*_silver` — cleaned, joined; **BUSINESS** — ready-to-analyze business entities
- `*_gold_*` / `*_gold` — curated, aggregated, BI-ready; **BUSINESS** — the HIGHEST-VALUE tier, almost always master or transactional business data
- `*_chunked` / `*_embeddings` / `*_embeds` / `*_vectors` / `*_rag` — RAG-prepared variants of business data for LLM consumption. **If the base entity is BUSINESS, the chunked/embedding variant is ALSO BUSINESS.** These are first-class inputs for AI-powered analytics use cases (semantic search, topic modeling, sentiment analysis, LLM summarization) — NOT ETL artifacts.

**Example**: `media_gold_reviews_chunked` = gold-layer (curated) + reviews (customer-feedback, obviously business) + chunked (RAG-ready). This is BUSINESS with Data Category MASTER. **Never classify as TECHNICAL solely because of 'chunked', 'gold', 'silver', 'embedded', 'vectorized' suffixes** — reason about the BASE ENTITY.

**STRATEGY: {exclusion_strategy}**
{strategy_rules}

---

**DATA CATEGORY CLASSIFICATION (SEMANTIC — not score-based)**:

Use the business context above to understand what this company does, then classify EACH table by applying the 3-QUESTION FRAMEWORK below. Do NOT rely on table names alone — reason about what data the table holds in the context of THIS specific industry and business.

**🚨 3-QUESTION FRAMEWORK (apply to EVERY table) 🚨**

**Q1: "Does a NEW row get inserted every time a business EVENT occurs?"**
  YES → likely **TRANSACTIONAL**. The table records discrete business events — things that HAPPENED at a specific moment. Each row represents one occurrence (one sale, one payment, one inspection, one shipment, one measurement). The table grows continuously over time. Rows are rarely updated after insertion.

**Q2: "Does this table describe WHAT something IS — its identity, attributes, or configuration — rather than WHAT HAPPENED?"**
  YES → likely **MASTER**. The table represents business ENTITIES — things that EXIST and have a lifecycle (active/inactive). Think: people, organizations, physical assets, products, locations, contracts, plans. The table is updated occasionally when entity attributes change, but new rows are NOT inserted for each business event.

**Q3: "Is this a finite, controlled lookup set that classifies or categorizes data in other tables?"**
  YES → likely **REFERENCE**. The table is a small, stable set of codes, types, categories, or rules. It changes rarely (quarterly/annually). Other tables reference it via foreign keys to categorize their data.

**TRANSACTIONAL** — "VERBS" of the business (events that HAPPENED):
- Each row = one discrete business event with a PRIMARY BUSINESS TIMESTAMP — a timestamp that records WHEN THE EVENT OCCURRED (not when the row was created/modified).
- Insert-heavy, rarely updated after creation. High volume, grows continuously over time.
- Typically references MASTER entities via FK (e.g., a transaction row points to a customer row).

**MASTER** — "NOUNS" of the business (entities that EXIST):
- Core business entity with lifecycle. Has a natural business identifier (ID, code, number).
- Timestamps on master tables are METADATA ABOUT the entity (created_at, last_updated, effective_date, hire_date, registration_date) — these do NOT make it transactional. These timestamps describe WHEN THE ENTITY WAS CREATED/CHANGED, not when business events happened.
- Updated occasionally when attributes change. Does NOT grow with each business transaction.

**REFERENCE** — "ADJECTIVES" of the business (lookup/configuration codes):
- Finite controlled set (typically <1000 rows). Classifies, categorizes, or configures other data.
- Changes rarely (quarterly, annually, or never). Other tables point to it for classification.

**🚨 CRITICAL ANTI-MISCLASSIFICATION RULES (apply to ALL industries) 🚨**

The #1 error is classifying MASTER tables as TRANSACTIONAL. Apply these rules to catch it:

1. **THE ENTITY TEST**: If the table represents a WHO (person, organization, partner), a WHAT (asset, product, service, resource), or a WHERE (location, facility, infrastructure) → it is MASTER or REFERENCE, NOT transactional. These tables describe things that EXIST, regardless of whether business events happen.

2. **THE EVENT TEST**: A table is ONLY transactional if each row captures a DISCRETE BUSINESS EVENT — something that happened at a specific point in time that would NOT exist if the event hadn't occurred. Ask: "If this business event hadn't happened, would this row not exist?" If yes → TRANSACTIONAL. If the row exists because the entity exists (regardless of events) → MASTER.

3. **THE TIMESTAMP TEST**: EVERY table has timestamps. The question is: does the timestamp represent the REASON the row exists (an event occurring), or is it just metadata about the row? If the table's primary purpose is to record "something happened at time X" → TRANSACTIONAL. If the timestamp is just "this entity was created/updated at time X" → MASTER/REFERENCE.

4. **THE GROWTH TEST**: Does this table grow proportionally with business activity? If transaction volume doubles, does this table's row count roughly double? If yes → TRANSACTIONAL. If the table stays roughly the same size regardless of business volume → MASTER/REFERENCE.

5. **THE FK DIRECTION TEST**: Does this table POINT TO other tables (has FK columns referencing master/reference entities), or do other tables POINT TO it (other tables have FK columns referencing this table's ID)? Tables that are POINTED TO by transactions are typically MASTER. Tables that POINT TO master entities are typically TRANSACTIONAL.

6. **PLANS/SCHEDULES/CONFIGURATIONS are MASTER, not TRANSACTIONAL**: A planned schedule, a pricing rule, a capacity allocation, a route definition, or a configuration table describes HOW things SHOULD work — not events that happened. Even if schedules change over time, each row represents a PLAN, not an event.

7. **CATALOGS/DIRECTORIES/REGISTRIES are MASTER or REFERENCE**: A product catalog, a partner directory, an asset registry, a service listing — these describe what CAN exist, not what DID happen.

**WHEN IN DOUBT → MASTER**: It is ALWAYS safer to classify as MASTER than as TRANSACTIONAL. MASTER tables are still available as JOIN context for use case generation. Misclassifying as TRANSACTIONAL wastes generation resources on static data and produces low-quality use cases centered on entity attributes rather than business events.

**TECHNICAL tables**: ONLY for IT infrastructure with ZERO business value: IT system logs, DB monitoring, server health, ETL orchestration, CI/CD, schema migrations, system configs.
**Edge cases default to BUSINESS**: Business user activity logs → TRANSACTIONAL. Business audit tables → TRANSACTIONAL. Business config/pricing → REFERENCE/MASTER. Mixed (>10% business columns) → BUSINESS.

**BUSINESS SCORE** (criticality, NOT category): 90-100=high-frequency transactional; 80-89=core master; 60-79=operational; 40-59=supporting; 20-39=static reference; 1-19=marginal. TECHNICAL=0.

---

**TABLES TO CLASSIFY**:
{tables_markdown}

**OUTPUT**: CSV with columns: "Table Name","Classification","Data Category","Business Score","Reason"
- Classification: "BUSINESS" or "TECHNICAL". Data Category: "MASTER", "TRANSACTIONAL", "REFERENCE", or "TECHNICAL".
- ALL fields double-quoted. Classify EVERY table. When in doubt, prefer BUSINESS.

Your response must START with: "Table Name","Classification","Data Category","Business Score","Reason","honesty_score","honesty_justification"
""" + HONESTY_CHECK_CSV

# COMMAND ----------

# DBTITLE 1,AIAgent
# ==============================================================================
# 2.4. AI AGENT MANAGER (Centralised AI Query Authority)
# ==============================================================================

class AIAgentManager:
    def __init__(self, max_concurrent_ai_calls: int, fallback_chain: list, logger=None):
        import threading
        self._max_concurrent = max(1, int(max_concurrent_ai_calls))
        self._semaphore = threading.BoundedSemaphore(self._max_concurrent)
        self._fallback_chain = list(fallback_chain)
        self._logger = logger
        self._lock = threading.Lock()
        self._model_stats = {}

    def _ensure_model_stats(self, model_name: str):
        if model_name not in self._model_stats:
            self._model_stats[model_name] = {"calls": 0, "successes": 0, "timeouts": 0, "errors": 0}

    def record_call(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["calls"] += 1

    def record_success(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["successes"] += 1

    def record_timeout(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["timeouts"] += 1

    def record_error(self, model_name: str):
        with self._lock:
            self._ensure_model_stats(model_name)
            self._model_stats[model_name]["errors"] += 1

    def get_stats_summary(self) -> dict:
        with self._lock:
            return {k: dict(v) for k, v in self._model_stats.items()}

    def build_full_cascade(self, prompt_name: str) -> list:
        return get_full_fallback_cascade(prompt_name)

    def acquire(self):
        self._semaphore.acquire()

    def release(self):
        self._semaphore.release()

    @property
    def max_concurrent(self) -> int:
        return self._max_concurrent


# ==============================================================================
# 2.5. AI AGENT CLASS (MODIFIED)
# ==============================================================================

class AIAgent:
    _total_ai_calls = 0
    _total_input_chars = 0
    _total_output_chars = 0
    _step_stats = defaultdict(lambda: {"calls": 0, "input_chars": 0, "output_chars": 0})

    # === MODIFIED: Added prompt_templates parameter ===
    def __init__(self, spark, logger, worker_llm_config, judge_llm_config, prompt_templates: dict,
                 default_timeout_seconds: int = None, max_retry_attempts: int = None, status_emitter=None):
        if default_timeout_seconds is None:
            default_timeout_seconds = TECHNICAL_CONTEXT["runtime"]["llm_timeout_seconds"]
        if max_retry_attempts is None:
            max_retry_attempts = TECHNICAL_CONTEXT["runtime"]["max_retry_attempts"]
        self.spark = spark
        self.logger = logger
        self.worker_llm = worker_llm_config
        self.judge_llm = judge_llm_config
        self.prompt_templates = prompt_templates  # <-- NEW: Store the dictionary
        self._language_local = threading.local()
        self._language_local.language = "English"
        self.default_timeout_seconds = default_timeout_seconds
        self.max_retry_attempts = max(0, max_retry_attempts)
        self.status_emitter = status_emitter
        self._status_step_ids = {}
        self._warning_seq = 0
        self._status_lock = threading.Lock()
        
        self._llm_cache_dir = None
        self._llm_cache_lock = threading.Lock()

        _runtime = TECHNICAL_CONTEXT.get("runtime", {})
        _max_ai_calls = _runtime.get("global_max_parallelism", 20)
        _fallback_chain = _runtime.get("model_fallback_chain", [])
        self._manager = AIAgentManager(
            max_concurrent_ai_calls=_max_ai_calls,
            fallback_chain=_fallback_chain,
            logger=self.logger,
        )
        self.logger.info(
            f"AIAgentManager initialized: max_concurrent_ai_calls={self._manager.max_concurrent}, "
            f"fallback_chain_length={len(_fallback_chain)}"
        )
        
        if not self.prompt_templates:
            self.logger.warning("AIAgent initialized with an empty prompt_templates dictionary.")

    def _utc_event_at(self):
        from datetime import datetime, timezone
        return datetime.now(timezone.utc).isoformat()

    def _derive_entity_id(self, prompt_name: str, step_name: str, result_json) -> str:
        if isinstance(result_json, dict):
            for key in ("entity_id", "domain_name", "use_case_id", "batch_id"):
                value = result_json.get(key)
                if value:
                    return f"{prompt_name}:{value}"
        return f"{prompt_name}:{step_name}"

    def _enrich_status_payload(self, prompt_name: str, step_name: str, result_json):
        payload = result_json if isinstance(result_json, dict) else {}
        if "event_at" not in payload:
            payload["event_at"] = self._utc_event_at()
        if "entity_id" not in payload:
            payload["entity_id"] = self._derive_entity_id(prompt_name, step_name, payload)
        payload.setdefault("step_name", step_name)
        payload.setdefault("prompt_name", prompt_name)
        return payload

    def _emit_runtime_warning(self, prompt_name: str, step_name: str, message: str, payload: dict = None):
        import time
        with self._status_lock:
            self._warning_seq += 1
            seq = self._warning_seq
        warning_step = f"{step_name}__warning_{int(time.time() * 1000)}_{seq}"
        self._emit_status(
            prompt_name=prompt_name,
            step_name=warning_step,
            status="ended_warning",
            message=message,
            result_json=payload if payload is not None else {}
        )

    def _emit_status(self, prompt_name: str, step_name: str, status: str, message: str = "", result_json=None):
        if not self.status_emitter:
            return
        try:
            payload = self._enrich_status_payload(prompt_name, step_name, result_json if result_json is not None else {})
            key = f"{prompt_name}|{step_name}"
            if status == "started":
                step_id = self.status_emitter(
                    prompt_name=prompt_name,
                    step_name=step_name,
                    status=status,
                    message=message,
                    result_json=payload,
                    step_id=None
                )
                if step_id is not None:
                    with self._status_lock:
                        self._status_step_ids[key] = step_id
                return

            with self._status_lock:
                existing_step_id = self._status_step_ids.get(key)
            self.status_emitter(
                prompt_name=prompt_name,
                step_name=step_name,
                status=status,
                message=message,
                result_json=payload,
                step_id=existing_step_id
            )
            if status.startswith("ended_"):
                with self._status_lock:
                    self._status_step_ids.pop(key, None)
        except Exception:
            pass

    def _count_csv_rows_hint(self, csv_text: str) -> int:
        if not csv_text:
            return 0
        lines = [ln for ln in str(csv_text).splitlines() if ln.strip()]
        if not lines:
            return 0
        return max(0, len(lines) - 1)

    def _extract_csv_rows(self, raw_response: str, max_rows: int = 200) -> tuple:
        import csv
        from io import StringIO
        cleaned = clean_csv_response(raw_response) if raw_response else ""
        if not cleaned or "," not in cleaned:
            return [], 0, False
        try:
            reader = csv.DictReader(StringIO(cleaned))
            rows = []
            total = 0
            for row in reader:
                total += 1
                if len(rows) < max_rows:
                    rows.append({str(k): str(v) for k, v in row.items()})
            return rows, total, total > len(rows)
        except Exception:
            return [], 0, False

    def _extract_json_object(self, raw_response: str):
        import json
        candidates = []
        if raw_response:
            candidates.append(raw_response)
            try:
                candidates.append(clean_json_response(raw_response))
            except Exception:
                pass
        for candidate in candidates:
            if not candidate:
                continue
            try:
                return json.loads(candidate)
            except Exception:
                continue
        return None

    def _build_started_status(self, prompt_name: str, step_name: str, prompt_vars: dict) -> tuple:
        pv = prompt_vars if isinstance(prompt_vars, dict) else {}
        summary = {"step_name": step_name, "prompt_name": prompt_name}
        if "schema_markdown" in pv:
            summary["schema_chars"] = len(str(pv.get("schema_markdown", "")))
        if "foreign_key_relationships" in pv:
            summary["fk_chars"] = len(str(pv.get("foreign_key_relationships", "")))
        if "use_cases_csv" in pv:
            summary["use_cases_input_count"] = self._count_csv_rows_hint(pv.get("use_cases_csv", ""))
        if "batch_csv" in pv:
            summary["batch_rows"] = self._count_csv_rows_hint(pv.get("batch_csv", ""))
        if "domain_name" in pv:
            summary["domain_name"] = str(pv.get("domain_name", ""))
        if "target_language" in pv:
            summary["target_language"] = str(pv.get("target_language", ""))
        if "strategic_goals" in pv:
            summary["strategic_goals_chars"] = len(str(pv.get("strategic_goals", "")))
        message = f"{prompt_name} started"
        details = []
        if "schema_chars" in summary:
            details.append(f"schema_chars={summary['schema_chars']}")
        if "use_cases_input_count" in summary:
            details.append(f"use_cases={summary['use_cases_input_count']}")
        if "domain_name" in summary and summary["domain_name"]:
            details.append(f"domain={summary['domain_name']}")
        if details:
            message = f"{prompt_name} started ({', '.join(details)})"
        return message, summary

    def _build_success_status(self, prompt_name: str, step_name: str, raw_response: str, response_schema) -> tuple:
        payload = {
            "step_name": step_name,
            "prompt_name": prompt_name,
            "response_type": "json" if response_schema else "raw",
            "response_chars": len(raw_response or "")
        }
        rows, total_rows, truncated = self._extract_csv_rows(raw_response)
        parsed_json = self._extract_json_object(raw_response)

        if prompt_name == "BASE_USE_CASE_GEN_PROMPT":
            use_cases = []
            for r in rows:
                use_cases.append({
                    "id": r.get("No", ""),
                    "name": r.get("Name", ""),
                    "usecase": r.get("usecase", ""),
                    "business_domain": r.get("Business Domain", ""),
                    "subdomain": r.get("Subdomain", ""),
                    "type": r.get("type", ""),
                    "priority": r.get("Priority", "")
                })
            payload["use_cases"] = use_cases
            payload["use_cases_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} use cases", payload

        if prompt_name in ("DOMAIN_FINDER_PROMPT", "SUBDOMAIN_DETECTOR_PROMPT", "DOMAINS_MERGER_PROMPT", "SUBDOMAINS_MERGER_PROMPT", "DOMAIN_SPLITTER_PROMPT"):
            if isinstance(parsed_json, dict):
                domains = parsed_json.get("domains", parsed_json.get("domain_clusters", []))
                payload["domains"] = domains if isinstance(domains, list) else []
                payload["domains_count"] = len(payload["domains"])
            elif rows:
                payload["domains"] = rows
                payload["domains_count"] = total_rows
                payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {payload.get('domains_count', 0)} domains", payload

        if prompt_name in ("USE_CASE_VALUE_SCORE_PROMPT", "USE_CASE_QUALITY_SCORE_PROMPT", "COMBINED_VALUE_QUALITY_SCORE_PROMPT", "REVIEW_USE_CASES_PROMPT"):
            priorities = []
            scored_use_cases = []
            for r in rows:
                item = {
                    "id": r.get("No", ""),
                    "name": r.get("Name", ""),
                    "usecase": r.get("usecase", ""),
                    "priority": r.get("Priority", r.get("priority", "")),
                    "value": r.get("Value", r.get("value", "")),
                    "feasibility": r.get("Feasibility", r.get("feasibility", "")),
                    "quality": r.get("Quality", r.get("quality", ""))
                }
                scored_use_cases.append(item)
                if item["priority"]:
                    priorities.append({"id": item["id"], "priority": item["priority"]})
            payload["scored_use_cases"] = scored_use_cases
            payload["priorities"] = priorities
            payload["scored_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} scored use cases", payload

        if rows:
            payload["rows"] = rows
            payload["rows_count"] = total_rows
            payload["is_truncated"] = truncated
            return f"{prompt_name} completed with {total_rows} rows", payload
        if parsed_json is not None:
            payload["json"] = parsed_json
            return f"{prompt_name} completed", payload
        return f"{prompt_name} completed", payload
    
    @property
    def current_language(self) -> str:
        return getattr(self._language_local, 'language', 'English')

    @current_language.setter
    def current_language(self, value: str):
        self._language_local.language = value

    def set_language(self, language: str):
        self._language_local.language = language

    def enable_llm_cache(self, session_id):
        import tempfile, os
        self._llm_cache_dir = os.path.join(tempfile.gettempdir(), f"inspire_{session_id}", "llm_cache")
        os.makedirs(self._llm_cache_dir, exist_ok=True)
        self.logger.info(f"LLM disk cache enabled: {self._llm_cache_dir}")

    def cleanup_llm_cache(self):
        import shutil
        if self._llm_cache_dir:
            try:
                shutil.rmtree(self._llm_cache_dir, ignore_errors=True)
                self.logger.info(f"LLM disk cache cleaned up: {self._llm_cache_dir}")
            except Exception:
                pass
            self._llm_cache_dir = None

    def _llm_cache_key(self, prompt: str, model: str) -> str:
        import hashlib
        h = hashlib.sha256((prompt + "||" + model).encode('utf-8', errors='replace')).hexdigest()[:32]
        return h

    def _llm_cache_get(self, prompt: str, model: str):
        if not self._llm_cache_dir:
            return None
        import os
        key = self._llm_cache_key(prompt, model)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        if os.path.exists(path):
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    cached = f.read()
                if cached:
                    self.logger.info(f"   LLM cache HIT: key={key[:12]}... ({len(cached):,} chars)")
                    return cached
            except Exception:
                pass
        return None

    def _llm_cache_put(self, prompt: str, model: str, response: str):
        if not self._llm_cache_dir or not response:
            return
        import os
        key = self._llm_cache_key(prompt, model)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        try:
            with self._llm_cache_lock:
                with open(path, 'w', encoding='utf-8') as f:
                    f.write(response)
        except Exception:
            pass

    def _llm_cache_invalidate(self, prompt: str, model: str):
        if not self._llm_cache_dir:
            return
        import os
        key = self._llm_cache_key(prompt, model)
        path = os.path.join(self._llm_cache_dir, f"{key}.txt")
        try:
            if os.path.exists(path):
                with self._llm_cache_lock:
                    os.remove(path)
                self.logger.info(f"   LLM cache INVALIDATED: key={key[:12]}... (failed validation)")
        except Exception:
            pass

    def _llm_cache_invalidate_all_models(self, prompt: str):
        if not self._llm_cache_dir:
            return
        for model_config in TECHNICAL_CONTEXT.get("models", []):
            self._llm_cache_invalidate(prompt, model_config["llm_endpoint_name"])

    _UNESCAPED_BRACE_RE = re.compile(r'(?<!\{)\{([^{}]+)\}(?!\})')

    def _sanitize_template_braces(self, template: str, prompt_key: str) -> str:
        """
        Pre-processes a prompt template to auto-escape {content} patterns where
        the content is NOT a valid Python .format() variable name.

        Prevents KeyError from unescaped JSON/text braces (e.g. {"key": "val"})
        that were accidentally left in the template string. Logs a warning for
        each pattern it auto-escapes so the developer can fix the source.
        """
        auto_fixed = []

        def _replacer(match):
            content = match.group(1)
            field_part = content.split('!')[0].split(':')[0].strip()
            base_name = field_part.split('.')[0].split('[')[0]
            if base_name and base_name.isidentifier():
                return match.group(0)
            auto_fixed.append(content[:100])
            return '{{' + content + '}}'

        sanitized = self._UNESCAPED_BRACE_RE.sub(_replacer, template)

        if auto_fixed:
            for fix_desc in auto_fixed:
                self.logger.warning(
                    f"[{prompt_key}] ⚠️ TEMPLATE BUG: Auto-escaped unescaped braces "
                    f"containing non-variable content: '{fix_desc}'"
                )

        return sanitized

    def _load_and_format_prompt(self, prompt_key: str, prompt_vars: dict) -> str:
        """
        Loads a prompt template from the internal dictionary and formats it.

        Includes a safety layer that auto-escapes any {content} pattern in the
        template where content is not a valid .format() variable name (e.g.
        unescaped JSON like {"key": "val"}). This prevents cryptic KeyError
        failures that silently break entire generation pipelines.
        """
        if prompt_key not in self.prompt_templates:
            self.logger.error(f"Prompt key '{prompt_key}' not found in prompt dictionary.")
            raise KeyError(f"Prompt key '{prompt_key}' not found in AIAgent's prompt_templates.")

        prompt_template = self._sanitize_template_braces(
            self.prompt_templates[prompt_key], prompt_key
        )

        try:
            return prompt_template.format(**prompt_vars)
        except KeyError as e:
            self.logger.error(f"Failed to format prompt '{prompt_key}'. Missing variable: {get_clean_error_message(e)}")
            raise e
        except Exception as e:
            self.logger.error(f"An unexpected error occurred during prompt formatting for '{prompt_key}': {get_clean_error_message(e)}")
            raise

    def _call_ai_query(self, prompt: str, prompt_name: str, response_schema=None, model_override=None, timeout_seconds=None, max_retries=None, display_name=None, start_from_model=None, skip_cache=False, status_step_name=None, temperature_override=None) -> str:
        # Phase N2 / N4: capture tokens_in / tokens_out / llm_response_id / duration_ms
        import time as _jl_t2
        _t0 = _jl_t2.perf_counter()
        _tin = max(1, len(str(prompt or "")) // 4)
        self._manager.acquire()
        try:
            _resp = self._call_ai_query_impl(
                prompt, prompt_name, response_schema, model_override,
                timeout_seconds, max_retries, display_name, start_from_model,
                skip_cache, status_step_name, temperature_override
            )
            _tout = max(1, len(str(_resp or "")) // 4)
            try:
                EVENTS.emit("llm_call", f"_call_ai_query::{prompt_name}", "ok",
                            duration_ms=int((_jl_t2.perf_counter() - _t0) * 1000),
                            tokens_in=_tin, tokens_out=_tout,
                            prompt_template=prompt_name)
            except Exception:
                pass
            return _resp
        except Exception as _e:
            try:
                EVENTS.emit("llm_call", f"_call_ai_query::{prompt_name}", "error",
                            duration_ms=int((_jl_t2.perf_counter() - _t0) * 1000),
                            tokens_in=_tin, exception_type=type(_e).__name__,
                            exception_msg=str(_e)[:500], prompt_template=prompt_name)
            except Exception:
                pass
            raise
        finally:
            self._manager.release()

    def _call_ai_query_impl(self, prompt: str, prompt_name: str, response_schema=None, model_override=None, timeout_seconds=None, max_retries=None, display_name=None, start_from_model=None, skip_cache=False, status_step_name=None, temperature_override=None) -> str:
        """
        Calls Databricks AI query function with the given prompt.
        
        Uses cross-type model fallback chain: resolves the prompt's designated model bucket
        and builds a cascade through all remaining buckets in the fallback chain.
        Tries models in chain order; on failure, cascades to the next enabled model.
        Each model gets up to max_retries attempts before cascading.
        Concurrency is gated by AIAgentManager's BoundedSemaphore (acquired in _call_ai_query).
        
        Args:
            prompt: The prompt text
            prompt_name: Name of the prompt template (for logging and model_type resolution)
            response_schema: Optional JSON schema for structured responses
            model_override: Optional endpoint name override (skips cascade, uses only this model)
            timeout_seconds: Timeout in seconds for LLM call
            max_retries: Maximum number of retry attempts per model for throttling/timeouts
            display_name: Optional display name for heartbeat logs (defaults to prompt_name)
            start_from_model: Optional model name/endpoint to start cascade from (skips earlier models)
        
        Returns:
            Raw response string from the LLM
        """
        import time
        
        heartbeat_name = display_name if display_name else prompt_name
        status_step = status_step_name if status_step_name else heartbeat_name
        
        attempts_per_model = (max_retries if max_retries is not None else self.max_retry_attempts) + 1
        timeout_val = timeout_seconds if timeout_seconds is not None else self.default_timeout_seconds
        
        if not timeout_val or timeout_val <= 0:
            self.logger.warning(f"Timeout value '{timeout_val}' is invalid. Enforcing safety fallback of 300s.")
            timeout_val = 300

        # --- Build model cascade ---
        if model_override:
            override_config = next(
                (m for m in TECHNICAL_CONTEXT["models"] if m["llm_endpoint_name"] == model_override),
                None
            )
            if not override_config:
                min_output_limit = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
                override_config = {
                    "name": model_override,
                    "llm_endpoint_name": model_override,
                    "llm_input_context_tokens_count": 131000,
                    "llm_output_context_tokens_count": min_output_limit,
                    "order": 999,
                    "type": "worker"
                }
            model_cascade = [override_config]
        elif start_from_model:
            full_cascade = self._manager.build_full_cascade(prompt_name)
            found_start = False
            model_cascade = []
            for m in full_cascade:
                if m["llm_endpoint_name"] == start_from_model or m.get("name") == start_from_model:
                    found_start = True
                if found_start:
                    model_cascade.append(m)
            if not model_cascade:
                self.logger.warning(f"start_from_model '{start_from_model}' not found in cascade, using full cascade")
                model_cascade = full_cascade
        else:
            model_cascade = self._manager.build_full_cascade(prompt_name)
        if not model_cascade:
            raise ValueError(f"No enabled models available for prompt bucket of '{prompt_name}'")

        cascade_names = [m.get("name", m["llm_endpoint_name"]) for m in model_cascade]
        self.logger.info(f"   [{prompt_name}] Model cascade ({len(model_cascade)}): {' -> '.join(cascade_names)}")

        if self._llm_cache_dir and not skip_cache:
            for _mc in model_cascade:
                cached = self._llm_cache_get(prompt, _mc["llm_endpoint_name"])
                if cached is not None:
                    return cached
        elif skip_cache and self._llm_cache_dir:
            self.logger.debug(f"   [{prompt_name}] Cache SKIPPED (retry wave — forcing fresh LLM call)")
            for _mc in model_cascade:
                self._llm_cache_invalidate(prompt, _mc["llm_endpoint_name"])

        last_error = None
        prompt_len = len(prompt)
        any_model_attempted = False

        # --- Outer loop: iterate through models in cascade order ---
        for model_idx, model_config in enumerate(model_cascade):
            if _is_thread_cancelled():
                raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")

            is_last_model = (model_idx == len(model_cascade) - 1)
            model = model_config["llm_endpoint_name"]
            model_name = model_config.get("name", model)
            input_token_limit_val = model_config.get("llm_input_context_tokens_count", 131000)
            _min_output_default = min(m.get("llm_output_context_tokens_count", 25000) for m in TECHNICAL_CONTEXT["models"])
            output_token_limit = model_config.get("llm_output_context_tokens_count", _min_output_default)

            if model_idx > 0:
                self.logger.info(
                    f"🔀 [{prompt_name}] Cascading to model '{model_name}' "
                    f"(order: {model_config.get('order', '?')}, type: {model_config.get('type', '?')}, "
                    f"input: {input_token_limit_val:,} tokens, output: {output_token_limit:,} tokens)"
                )

            # PRE-FLIGHT: Check if prompt fits this model's context window
            char_ratio = TOKEN_TO_CHAR_RATIO_NON_ENGLISH if self.current_language.lower() != "english" else TOKEN_TO_CHAR_RATIO_ENGLISH
            max_context_chars = input_token_limit_val * char_ratio
            conservative_char_limit = int(input_token_limit_val * 3.2)
            effective_limit = min(max_context_chars, conservative_char_limit)

            if prompt_len > effective_limit:
                estimated_tokens = int(prompt_len / 3.2)

                if any_model_attempted:
                    remaining_models = model_cascade[model_idx:]
                    best_remaining = max(remaining_models, key=lambda m: m.get("llm_input_context_tokens_count", 0))
                    best_limit = get_context_limit_chars_for_model(best_remaining, self.current_language)
                    failed_name = cascade_names[model_idx - 1] if model_idx > 0 else "unknown"
                    self.logger.warning(
                        f"🔄 [{prompt_name}] Prompt ({prompt_len:,} chars) exceeds remaining models' limits after "
                        f"'{failed_name}' failed. Best remaining: '{best_remaining.get('name')}' ({best_limit:,} chars). "
                        f"Raising CascadeRebatchError so caller can split into smaller batches."
                    )
                    raise CascadeRebatchError(
                        f"Prompt ({prompt_len:,} chars) exceeds '{best_remaining.get('name')}' limit ({best_limit:,} chars) "
                        f"after '{failed_name}' failed. Caller should rebatch to fit {best_limit:,} chars.",
                        target_model_name=best_remaining.get("name"),
                        target_model_endpoint=best_remaining["llm_endpoint_name"],
                        target_context_limit_chars=best_limit,
                        failed_model_name=failed_name
                    )

                if not is_last_model:
                    self.logger.warning(
                        f"⚠️  [{prompt_name}] Prompt ({prompt_len:,} chars, ~{estimated_tokens:,} tokens) exceeds "
                        f"'{model_name}' limit ({effective_limit:,} chars / {input_token_limit_val:,} tokens). "
                        f"Skipping to next model in cascade."
                    )
                    continue
                self.logger.error(
                    f"Prompt length ({prompt_len:,} chars, ~{estimated_tokens:,} est. tokens) exceeds limit "
                    f"({effective_limit:,} chars / {input_token_limit_val:,} tokens) "
                    f"for language '{self.current_language}' and prompt: {prompt_name}. All cascade models exhausted."
                )
                raise InputTooLongError(
                    f"Input length: {prompt_len:,} characters (~{estimated_tokens:,} tokens) exceeds context limit of {effective_limit:,} characters "
                    f"({input_token_limit_val:,} tokens) "
                    f"for language '{self.current_language}'. "
                    f"Prompt: {prompt_name}, Model: {model_name}. All {len(model_cascade)} cascade models exhausted. Please batch your input."
                )

            any_model_attempted = True
            self._manager.record_call(model_name)

            # --- Inner loop: retry attempts on this specific model ---
            cascade_to_next = False

            for attempt in range(1, attempts_per_model + 1):
                if _is_thread_cancelled():
                    raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")
                try:
                    if attempt > 1:
                        self.logger.info(f"🔄 [{prompt_name}] Retry attempt {attempt}/{attempts_per_model} on model '{model_name}'...")
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} retry attempt {attempt}/{attempts_per_model} on {model_name}",
                            payload={"model": model_name, "attempt": attempt, "attempts_per_model": attempts_per_model}
                        )

                    response_format_str = ""
                    if prompt_name in COMPACT_OUTPUT_PROMPTS:
                        max_output_tokens = min(int(output_token_limit * 0.25), output_token_limit)
                    elif prompt_name in MEDIUM_OUTPUT_PROMPTS:
                        max_output_tokens = min(20480, output_token_limit)
                    else:
                        max_output_tokens = min(int(output_token_limit * 0.9), output_token_limit)
                    max_output_tokens = min(max_output_tokens, output_token_limit)

                    self.logger.info(f"   [{prompt_name}] Setting max_tokens={max_output_tokens:,} (model: {model_name}, limit: {output_token_limit:,})")

                    effective_temperature = temperature_override if temperature_override is not None else _get_prompt_temperature(prompt_name)
                    self.logger.debug(f"   [{prompt_name}] Using temperature={effective_temperature} (override={temperature_override is not None})")
                    ai_query_args = {"prompt_text": replace_single_quote(prompt)}
                    if response_schema:
                        response_format_str = json.dumps({"type": "json_schema", "json_schema": response_schema}, separators=(',', ':'))
                        ai_query_args["resp_fmt"] = response_format_str
                        ai_query_sql = f"SELECT ai_query('{model}', :prompt_text, responseFormat => :resp_fmt, modelParameters => named_struct('max_tokens', {max_output_tokens}, 'temperature', CAST({effective_temperature} AS DOUBLE))) AS ai_response"
                    else:
                        ai_query_sql = f"SELECT ai_query('{model}', :prompt_text, modelParameters => named_struct('max_tokens', {max_output_tokens}, 'temperature', CAST({effective_temperature} AS DOUBLE))) AS ai_response"

                    _done_event = threading.Event()
                    _tctx = {'response': None, 'error': None, 'done': False}
                    _query_tag = f"llm_{prompt_name}_{uuid.uuid4().hex[:8]}"

                    def execute_query(_ctx=_tctx, _sql=ai_query_sql, _args=ai_query_args, _evt=_done_event, _tag=_query_tag, _spark=self.spark):
                        _tag_added = False
                        try:
                            try:
                                _spark.addTag(_tag)
                                _tag_added = True
                            except Exception:
                                pass
                            _ctx['error'] = None
                            result = execute_sql(_spark, _sql, self.logger, args=_args)
                            _ctx['response'] = result
                            _ctx['done'] = True
                        except Exception as e:
                            _ctx['error'] = e
                            _ctx['done'] = True
                        finally:
                            if _tag_added:
                                try:
                                    _spark.removeTag(_tag)
                                except Exception:
                                    pass
                            _evt.set()

                    query_thread = threading.Thread(target=execute_query, name=f"LLM_Query_{prompt_name}")
                    query_thread.daemon = True
                    start_time = time.time()
                    query_thread.start()

                    poll_interval = 5
                    last_heartbeat_minute = -1
                    while True:
                        _done_event.wait(timeout=poll_interval)
                        elapsed = time.time() - start_time

                        if _done_event.is_set() or _tctx['done'] or not query_thread.is_alive():
                            break

                        if _is_thread_cancelled():
                            self.logger.warning(f"🛑 [{prompt_name}] Parent task cancelled - aborting LLM poll (model: {model_name})")
                            try:
                                self.spark.interruptTag(_query_tag)
                            except Exception:
                                pass
                            raise Exception(f"[{prompt_name}] Cascade aborted: parent task was cancelled/timed out")

                        if elapsed >= timeout_val:
                            self.logger.error(f"⏱️  [{prompt_name}] LLM call timed out after {elapsed:.1f}s (model: {model_name}, attempt {attempt}/{attempts_per_model})")
                            self._emit_runtime_warning(
                                prompt_name=prompt_name,
                                step_name=status_step,
                                message=f"{prompt_name} timeout on {model_name} after {int(elapsed)}s",
                                payload={
                                    "model": model_name,
                                    "attempt": attempt,
                                    "attempts_per_model": attempts_per_model,
                                    "elapsed_seconds": int(elapsed),
                                    "event_type": "timeout"
                                }
                            )
                            try:
                                self.spark.interruptTag(_query_tag)
                                self.logger.info(f"🛑 [{prompt_name}] Sent interruptTag for timed-out query (tag: {_query_tag})")
                            except AttributeError:
                                self.logger.debug(f"[{prompt_name}] spark.interruptTag not available (non-Spark-Connect runtime)")
                            except Exception as _cancel_err:
                                self.logger.debug(f"[{prompt_name}] interruptTag failed: {str(_cancel_err)[:100]}")
                            break

                        current_minute = int(elapsed) // 60
                        if current_minute > 0 and current_minute != last_heartbeat_minute:
                            last_heartbeat_minute = current_minute
                            log_print(f"[{heartbeat_name}] Still waiting... {elapsed:.0f}s elapsed (timeout: {timeout_val}s, model: {model_name})")

                    if not _done_event.is_set() and query_thread.is_alive():
                        raise Exception(f"LLM call timed out after {timeout_val} seconds (model: {model_name})")

                    if _tctx['error'] is not None:
                        raise _tctx['error']

                    raw_response = _tctx['response'][0].ai_response if _tctx['response'] and _tctx['response'][0] else ""
                    _tctx = None
                    ai_query_sql = None
                    ai_query_args = None

                    honesty_score, honesty_justification, cleaned_response = extract_honesty_score(raw_response, self.logger)
                    del raw_response

                    if honesty_score is not None:
                        if honesty_justification:
                            self.logger.info(f"🔮✨ HONESTY CHECK [{prompt_name}] Score: {honesty_score}% | {honesty_justification} ✨🔮")
                        else:
                            self.logger.info(f"🔮✨ HONESTY CHECK [{prompt_name}] Score: {honesty_score}% ✨🔮")

                    input_len = len(prompt)
                    output_len = len(cleaned_response)

                    AIAgent._total_ai_calls += 1
                    AIAgent._total_input_chars += input_len
                    AIAgent._total_output_chars += output_len

                    AIAgent._step_stats[prompt_name]["calls"] += 1
                    AIAgent._step_stats[prompt_name]["input_chars"] += input_len
                    AIAgent._step_stats[prompt_name]["output_chars"] += output_len

                    self._llm_cache_put(prompt, model, cleaned_response)
                    self._manager.record_success(model_name)
                    return cleaned_response

                except InputTooLongError:
                    raise
                except CascadeRebatchError:
                    raise
                except TruncatedResponseError:
                    raise
                except Exception as e:
                    last_error = e
                    error_msg = str(e).lower()

                    is_sql_construction_error = (
                        'parse_syntax_error' in error_msg and
                        'select ai_query' in error_msg
                    )
                    is_max_tokens_error = 'max_new_tokens' in error_msg and 'cannot be greater' in error_msg
                    if is_sql_construction_error or is_max_tokens_error:
                        self.logger.error(
                            f"❌ [{prompt_name}] Non-retryable AI query error: {get_clean_error_message(e)}"
                        )
                        raise

                    is_input_too_long = any(keyword in error_msg for keyword in [
                        'input is too long', 'too long for requested model', 'input length',
                        'exceeds context limit', 'context window', 'token limit exceeded',
                        'maximum context length'
                    ])
                    is_databricks_input_error = (
                        ('400' in error_msg or 'bad_request' in error_msg or 'bad request' in error_msg) and
                        ('input' in error_msg or 'length' in error_msg or 'model' in error_msg)
                    )

                    if is_input_too_long or is_databricks_input_error:
                        self.logger.error(
                            f"❌ [{prompt_name}] Input too long on model '{model_name}': {get_clean_error_message(e)}"
                        )
                        if not is_last_model:
                            next_cfg = model_cascade[model_idx + 1]
                            next_char_limit = int(next_cfg.get("llm_input_context_tokens_count", 131000) * 3.2)
                            if prompt_len <= next_char_limit:
                                self.logger.info(
                                    f"🔀 [{prompt_name}] Prompt fits next model '{next_cfg.get('name')}' "
                                    f"({prompt_len:,} chars <= {next_char_limit:,} limit). Cascading."
                                )
                                cascade_to_next = True
                                break
                        raise InputTooLongError(
                            f"Input length: {len(prompt)} characters exceeds model's context limit. "
                            f"Prompt: {prompt_name}, Model: {model_name}. All cascade models exhausted."
                        ) from e

                    is_throttling = any(keyword in error_msg for keyword in [
                        'throttl', 'rate limit', 'too many requests', 'quota', '429',
                        'resource exhausted', 'capacity', 'overload'
                    ])
                    is_timeout = any(keyword in error_msg for keyword in [
                        'timeout', 'timed out', 'deadline', 'time limit'
                    ])
                    is_server_error = any(keyword in error_msg for keyword in [
                        '500', '502', '503', '504', 'internal server', 'service unavailable',
                        'bad gateway', 'gateway timeout'
                    ])
                    is_retryable = is_throttling or is_timeout or is_server_error
                    if is_timeout:
                        self._manager.record_timeout(model_name)
                    else:
                        self._manager.record_error(model_name)

                    error_type = "Throttling" if is_throttling else ("Timeout" if is_timeout else ("Server error" if is_server_error else "Error"))

                    clean_err_msg = get_clean_error_message(e)

                    if is_retryable and attempt < attempts_per_model:
                        wait_time = min(2 ** attempt * 5, 120)
                        self.logger.warning(
                            f"⚠️  [{prompt_name}] {error_type} on model '{model_name}' "
                            f"(attempt {attempt}/{attempts_per_model}): {clean_err_msg}"
                        )
                        self.logger.info(f"   Waiting {wait_time}s before retry on same model '{model_name}'...")
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} transient {error_type.lower()} on {model_name}, retrying in {wait_time}s",
                            payload={
                                "model": model_name,
                                "attempt": attempt,
                                "attempts_per_model": attempts_per_model,
                                "wait_seconds": wait_time,
                                "event_type": "retry_scheduled",
                                "error_type": error_type
                            }
                        )
                        time.sleep(wait_time)
                        continue

                    if not is_last_model:
                        self.logger.warning(
                            f"⚠️  [{prompt_name}] {error_type} on model '{model_name}' "
                            f"(attempt {attempt}/{attempts_per_model}, retries exhausted). "
                            f"Cascading to next model. Error: {clean_err_msg}"
                        )
                        next_model_name = model_cascade[model_idx + 1].get("name", model_cascade[model_idx + 1]["llm_endpoint_name"])
                        self._emit_runtime_warning(
                            prompt_name=prompt_name,
                            step_name=status_step,
                            message=f"{prompt_name} cascading from {model_name} to {next_model_name}",
                            payload={
                                "from_model": model_name,
                                "to_model": next_model_name,
                                "attempt": attempt,
                                "event_type": "cascade"
                            }
                        )
                        cascade_to_next = True
                        break

                    if is_retryable:
                        self.logger.error(
                            f"❌ [{prompt_name}] All {len(model_cascade)} cascade models and retries exhausted "
                            f"for retryable error: {clean_err_msg}"
                        )
                    self.logger.error(f"❌ [{prompt_name}] AI Query failed on '{model_name}' (last in cascade): {get_clean_error_message(e)}")
                    raise

            if cascade_to_next:
                continue

        if last_error:
            raise last_error
        raise Exception(f"All {len(model_cascade)} cascade models exhausted for prompt: {prompt_name}")

    def run_worker(self, step_name, worker_prompt_path, prompt_vars, response_schema, model_override=None, timeout_override=None, max_retries_override=None, start_from_model=None, skip_cache=False, temperature_override=None):
        try:
            start_message, start_payload = self._build_started_status(worker_prompt_path, step_name, prompt_vars)
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="started",
                message=start_message,
                result_json=start_payload
            )
            worker_prompt = self._load_and_format_prompt(worker_prompt_path, prompt_vars)
            
            if not worker_prompt:
                raise ValueError(f"Failed to load prompt: {worker_prompt_path}")
            
            raw_response = self._call_ai_query(
                worker_prompt,
                worker_prompt_path,
                response_schema,
                model_override,
                timeout_seconds=timeout_override,
                max_retries=max_retries_override,
                display_name=step_name,
                start_from_model=start_from_model,
                skip_cache=skip_cache,
                status_step_name=step_name,
                temperature_override=temperature_override
            )
            
            if response_schema:
                parsed = clean_json_response(raw_response)
                end_message, end_payload = self._build_success_status(worker_prompt_path, step_name, raw_response, response_schema)
                self._emit_status(
                    prompt_name=worker_prompt_path,
                    step_name=step_name,
                    status="ended_success",
                    message=end_message,
                    result_json=end_payload
                )
                return parsed
            else:
                if not raw_response:
                    self.logger.warning(f"AI Worker for {step_name} (Raw) returned an empty response.")
                    self._emit_status(
                        prompt_name=worker_prompt_path,
                        step_name=step_name,
                        status="ended_warning",
                        message=f"{worker_prompt_path} returned empty response",
                        result_json={"step_name": step_name}
                    )
                    return ""
                end_message, end_payload = self._build_success_status(worker_prompt_path, step_name, raw_response, response_schema)
                self._emit_status(
                    prompt_name=worker_prompt_path,
                    step_name=step_name,
                    status="ended_success",
                    message=end_message,
                    result_json=end_payload
                )
                return raw_response
        except CascadeRebatchError:
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="ended_warning",
                message=f"{worker_prompt_path} requires rebatching",
                result_json={"step_name": step_name}
            )
            raise
        except Exception as e:
            self._emit_status(
                prompt_name=worker_prompt_path,
                step_name=step_name,
                status="ended_error",
                message=f"{worker_prompt_path} failed",
                result_json={"step_name": step_name, "error": get_clean_error_message(e, max_chars=300)}
            )
            self.logger.error(f"AI Worker process failed for {step_name}: {get_clean_error_message(e)}")
            raise

    # --- START OF MODIFICATIONS ---

    def _deep_parse_json_values(self, data, task):
        """
        (Helper) Parses known stringified keys ('attributes', 'domains') within a data object.
        Operates on the dictionary, not the JSON string.
        """
        if not isinstance(data, dict):
            return data # Not a dict, can't fix

        keys_to_check = []
        if task == 'attributes':
            keys_to_check = ['attributes']
        elif task == 'domains':
            keys_to_check = ['domains']
        
        for key in keys_to_check:
            value = data.get(key)
            if isinstance(value, str):
                try:
                    data[key] = json.loads(value) # Replace string with parsed object
                except (json.JSONDecodeError, TypeError):
                    self.logger.warning(f"Failed to deep-parse stringified key '{key}' in task '{task}'.")
                    data[key] = [] # Set to empty list on failure
        return data

    # === MODIFIED: Uses internal _load_and_format_prompt ===
    def run_worker_judge(self, step_name, worker_prompt_path, judge_prompt_path, base_prompt_vars, worker_response_schema, config, randomization_params={}, task_info_lambda=None, validation_lambda=None):
        
        task_type, log_context = task_info_lambda(base_prompt_vars) if task_info_lambda else ("unknown task", "")
        # Worker/judge process starting - high-level logging only
        
        # --- Added Retry Loop ---
        for attempt in range(config["MAX_RETRIES"]):
            try:
                # --- Worker Generation ---
                worker_outputs = []
                for i in range(2):
                    randomized_vars = base_prompt_vars.copy()
                    for key, val in randomization_params.items():
                        base_min = int(config["PROMPT_VARIABLES"][f"min_{key}"])
                        base_max = int(config["PROMPT_VARIABLES"][f"max_{key}"])
                        new_min = base_min + random.randint(0, val)
                        randomized_vars[f"min_{key}"] = new_min
                        randomized_vars[f"max_{key}"] = max(new_min + 1, base_max + random.randint(0, val))
                    
                    # === MODIFIED ===
                    worker_prompt = self._load_and_format_prompt(worker_prompt_path, randomized_vars)
                    
                    # --- Worker call ---
                    worker_step_name = f"{step_name}_worker_{i+1}"
                    raw_response = self._call_ai_query(worker_prompt, worker_step_name, worker_response_schema) 
                    worker_outputs.append(clean_json_response(raw_response))

                # --- Helper Functions (Modified to use _deep_parse) ---
                def summarize_output(json_string, task):
                    try:
                        data = json.loads(json_string)
                        data = self._deep_parse_json_values(data, task) # Fix stringified values
                        if task == 'domains': return f"domains: {', '.join([d.get('domain', 'N/A') for d in data.get('domains', [])])}"
                        if task == 'attributes': return f"{len(data.get('attributes', []))} attributes"
                        if 'dashboard' in task: return f"{sum(len(p.get('queries', [])) for p in data)} dashboard queries"
                        return "summary not available"
                    except (json.JSONDecodeError, TypeError): return "invalid JSON"

                def is_response_empty(json_string, task):
                    try:
                        data = json.loads(json_string)
                        if not data: return True
                        data = self._deep_parse_json_values(data, task) # Fix stringified values
                        if task == 'domains': return not data.get('domains')
                        if task == 'attributes': return not data.get('attributes')
                        if 'dashboard' in task: return not isinstance(data, list) or not any(p.get('queries') for p in data)
                        return True
                    except (json.JSONDecodeError, TypeError):
                        return True

                def get_best_worker_output(outputs, task):
                    best_output, max_count = "", -1
                    for out_str in outputs:
                        if is_response_empty(out_str, task): continue
                        try:
                            data, count = json.loads(out_str), 0
                            data = self._deep_parse_json_values(data, task) # Fix stringified values
                            if task == 'domains': count = len(data.get('domains', []))
                            elif task == 'attributes': count = len(data.get('attributes', []))
                            elif 'dashboard' in task: count = sum(len(p.get('queries', [])) for p in data)
                            if count > max_count:
                                max_count, best_output = count, out_str
                        except (json.JSONDecodeError, TypeError): continue
                    return best_output if best_output else max(outputs, key=len, default="")
                
                # --- End Helper Functions ---

                for i, output in enumerate(worker_outputs): self.logger.debug(f"LLM Worker {i+1} suggested {log_context}: {summarize_output(output, task_type)}")
                
                best_worker_fallback = get_best_worker_output(worker_outputs, task_type)
                
                # --- START: NEW JUDGE-SKIP LOGIC ---
                skip_judge = False
                final_cleaned_json = ""
                
                if task_type == 'domains' and judge_prompt_path:
                    try:
                        data1 = json.loads(worker_outputs[0])
                        data1 = self._deep_parse_json_values(data1, task_type)
                        domains1 = set(d.get('domain') for d in data1.get('domains', []))
                        
                        data2 = json.loads(worker_outputs[1])
                        data2 = self._deep_parse_json_values(data2, task_type)
                        domains2 = set(d.get('domain') for d in data2.get('domains', []))
                        
                        if domains1 and domains1 == domains2:
                            self.logger.debug("Worker domain outputs match. Skipping judge and using worker 1 output.")
                            final_cleaned_json = worker_outputs[0]
                            skip_judge = True
                    except Exception as e:
                        self.logger.warning(f"Could not compare worker outputs for judge skip: {get_clean_error_message(e)}")
                # --- END: NEW JUDGE-SKIP LOGIC ---

                if not judge_prompt_path or skip_judge:
                    if not judge_prompt_path:
                        self.logger.debug("No judge prompt provided. Skipping judge and using best worker output.")
                    
                    if not final_cleaned_json:
                        final_cleaned_json = best_worker_fallback
                else:
                    judge_prompt_vars = {**base_prompt_vars, **{f'llm{i+1}_output': out for i, out in enumerate(worker_outputs)}}
                    
                    # === MODIFIED ===
                    judge_prompt = self._load_and_format_prompt(judge_prompt_path, judge_prompt_vars)
                    
                    if len(judge_prompt) > 120000:
                        self.logger.debug(f"Judge prompt too long ({len(judge_prompt)} chars). Using best worker response.")
                        final_cleaned_json = best_worker_fallback
                    else:
                        judge_step_name = f"{step_name}_judge"
                        final_raw_response = self._call_ai_query(judge_prompt, judge_step_name, worker_response_schema) 
                        final_cleaned_json = clean_json_response(final_raw_response)
                        if is_response_empty(final_cleaned_json, task_type):
                            self.logger.warning(f"Judge returned a malformed or empty result for {task_type}. Rejecting and using best worker output.")
                            final_cleaned_json = best_worker_fallback
                
                try:
                    final_data = json.loads(final_cleaned_json)
                    final_data = self._deep_parse_json_values(final_data, task_type)
                    final_fixed_json_string = json.dumps(final_data)
                except (json.JSONDecodeError, TypeError):
                    self.logger.error(f"Failed to parse or fix final JSON output for {task_type}. Using raw cleaned JSON for validation.")
                    final_fixed_json_string = final_cleaned_json

                if validation_lambda:
                    validation_lambda(final_fixed_json_string, task_type)
                
                self.logger.debug(f"Judge adjudicated the final output {log_context}, resulting in: {summarize_output(final_fixed_json_string, task_type)}")
                return final_fixed_json_string

            except Exception as e:
                self.logger.warning(f"Attempt {attempt + 1}/{config['MAX_RETRIES']} for {step_name} {log_context} failed: {get_clean_error_message(e)}")
                if attempt == config["MAX_RETRIES"] - 1:
                    self.logger.error(f"FAILED to generate valid output for {step_name} {log_context} after all retries.")
                    raise e
        
        raise Exception(f"AI Worker/Judge for {step_name} failed after all retries.")

    @staticmethod
    def get_summary_report():
        """
        Generates a summary report of AI usage, grouped by prompt type.
        More generic and aggregated across all instances.
        """
        report = []
        report.append("\n" + "="*70)
        report.append("--- 📊 AI Usage Summary ---")
        report.append("="*70)
        report.append(f"Total AI Calls:     {AIAgent._total_ai_calls}")
        
        # Calculate estimated tokens (chars / 4)
        input_tokens = AIAgent._total_input_chars / 4
        output_tokens = AIAgent._total_output_chars / 4
        
        report.append(f"Total Input Tokens:  ~{input_tokens:,.2f}  ({AIAgent._total_input_chars:,} chars)")
        report.append(f"Total Output Tokens: ~{output_tokens:,.2f}  ({AIAgent._total_output_chars:,} chars)")
        report.append("\n--- Prompt Type Details ---")
        
        if not AIAgent._step_stats:
            report.append("No AI calls were tracked.")
        else:
            # Group by prompt type (extract the general prompt name from step names)
            prompt_aggregates = {}
            for step_name, stats in AIAgent._step_stats.items():
                # Extract prompt type from step name (e.g., "Generate_UseCases_Batch_5" -> "Generate_UseCases")
                # Common patterns: "PromptName_Language_Batch_X", "PromptName_Batch_X", etc
                parts = step_name.split('_')
                
                # Identify the core prompt name
                prompt_type = step_name
                if 'Batch' in step_name:
                    # Extract everything before "_Batch"
                    batch_idx = step_name.find('_Batch')
                    if batch_idx > 0:
                        prompt_type = step_name[:batch_idx]
                elif any(lang in step_name for lang in ['English', 'Arabic', 'Chinese', 'French', 'Spanish']):
                    # Remove language suffix
                    for lang in ['English', 'Arabic', 'Chinese', 'French', 'Spanish', 'German', 'Portuguese', 'Italian', 'Japanese', 'Korean']:
                        if f"_{lang}" in step_name:
                            prompt_type = step_name.replace(f"_{lang}", "")
                            break
                
                # Aggregate stats by prompt type
                if prompt_type not in prompt_aggregates:
                    prompt_aggregates[prompt_type] = {'calls': 0, 'input_chars': 0, 'output_chars': 0}
                
                prompt_aggregates[prompt_type]['calls'] += stats['calls']
                prompt_aggregates[prompt_type]['input_chars'] += stats['input_chars']
                prompt_aggregates[prompt_type]['output_chars'] += stats['output_chars']
            
            # Display aggregated results
            prompt_col_width = 40
            header = f"{'Prompt Type':<{prompt_col_width}} | {'Calls':<7} | {'Input Tokens':<18} | {'Output Tokens':<18}"
            report.append(header)
            report.append("-" * len(header))
            
            # Sort by number of calls (descending)
            for prompt_type, stats in sorted(prompt_aggregates.items(), key=lambda x: x[1]['calls'], reverse=True):
                input_tok = stats['input_chars'] / 4
                output_tok = stats['output_chars'] / 4
                report.append(f"{prompt_type:<{prompt_col_width}} | {stats['calls']:<7} | ~{input_tok:<16,.0f} | ~{output_tok:<16,.0f}")
        
        report.append("="*70)

        final_report_str = "\n".join(report)
        log_print(final_report_str)
        return final_report_str

    def get_manager_stats_report(self) -> str:
        if not hasattr(self, '_manager'):
            return ""
        mgr_stats = self._manager.get_stats_summary()
        if not mgr_stats:
            return ""
        lines = []
        lines.append("\n" + "="*70)
        lines.append("--- AIAgentManager Model Stats ---")
        lines.append(f"Max concurrent AI calls: {self._manager.max_concurrent}")
        mgr_header = f"{'Model':<30} | {'Calls':<7} | {'Success':<9} | {'Timeouts':<9} | {'Errors':<7}"
        lines.append(mgr_header)
        lines.append("-" * len(mgr_header))
        for mname, mstats in sorted(mgr_stats.items(), key=lambda x: x[1].get("calls", 0), reverse=True):
            lines.append(
                f"{mname:<30} | {mstats.get('calls',0):<7} | "
                f"{mstats.get('successes',0):<9} | {mstats.get('timeouts',0):<9} | "
                f"{mstats.get('errors',0):<7}"
            )
        lines.append("="*70)
        report_str = "\n".join(lines)
        log_print(report_str)
        return report_str

# COMMAND ----------

# DBTITLE 1,Translations
# ==============================================================================
# 1.5. TRANSLATION SERVICE (MODIFIED)
# ==============================================================================

class TranslationService:
    """
    Handles all language translation tasks by calling an AI agent in parallel.
    Relies on an AIAgent that has been initialized with the PROMPT_TEMPLATES dictionary.
    """
    
    # === MODIFIED: Added pdf_disclaimer_title and updated titles ===
    ENGLISH_TRANSLATIONS = {
        "main_title": "Use Case Generator Powered by Databricks Genie Code",
        "intro": "This notebook contains AI-generated use cases based on your schemas. Below is a summary of the generated scenarios by business domain.",
        "domain": "Business Domain",
        "total": "Total Use Cases",
        "summaries": "Use Case Summaries",
        "sum_id": "ID",
        "sum_name": "Use Case",
        "sum_value": "Business Value",
        "sum_outcome": "Expected Outcome",
        "warning_header": "WARNING",
        "warning_body": "Do not run this notebook. It is intended for demonstration and cataloging purposes only.",
        "disclaimer": "This content is AI-generated and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from the use of this content.",
        "detailed_scenarios": "Use Cases Details",
        "aspect": "Aspect",
        "description": "Description",
        "aspect_domain": "Business Domain",
        "type": "Type",
        "analytics_technique": "Analytics Technique",
        "primary_table": "Primary Table",
        "quality": "Quality",
        "priority": "Priority",
        # Value translations for Type field
        "value_type_problem": "Problem",
        "value_type_risk": "Risk",
        "value_type_opportunity": "Opportunity",
        "value_type_improvement": "Improvement",
        # Value translations for Priority field
        "value_priority_ultra_high": "Ultra High",
        "value_priority_very_high": "Very High",
        "value_priority_high": "High",
        "value_priority_medium": "Medium",
        "value_priority_low": "Low",
        "value_priority_very_low": "Very Low",
        "value_priority_ultra_low": "Ultra Low",
        "statement": "Statement",
        "solution": "Solution",
        "aspect_beneficiary": "Beneficiary",
        "beneficiary": "Beneficiary",
        "aspect_sponsor": "Sponsor",
        "sponsor": "Sponsor",
        "business_priority_alignment": "Business Priority Alignment",
        "strategic_goals_alignment": "Strategic Goals Alignment",
        "subdomain": "Subdomain",
        "usecase": "Use Case",
        "description_label": "Description",
        "aspect_value": "Business Value",
        "business_value": "Business Value",
        "aspect_tables": "Tables Involved",
        "quality_reasons": "Quality Reasons",
        "aspect_ai_function": "AI Function",
        "aspect_analytics_technique": "Analytics Technique",
        "aspect_primary_table": "Primary Table",
        "aspect_quality": "Quality",
        "aspect_priority": "Priority",
        # Scoring field labels (for Excel)
        "strategic_alignment": "Strategic Alignment",
        "return_on_investment": "Return on Investment",
        "reusability": "Reusability",
        "time_to_value": "Time to Value",
        "data_availability": "Data Availability",
        "data_accessibility": "Data Accessibility",
        "architecture_fitness": "Architecture Fitness",
        "team_skills": "Team Skills",
        "domain_knowledge": "Domain Knowledge",
        "people_allocation": "People Allocation",
        "budget_allocation": "Budget Allocation",
        "time_to_production": "Time to Production",
        "value_score": "Value",
        "feasibility_score": "Feasibility",
        "priority_score": "Priority",
        "inspire_score": "Inspire Score",
        "quality_score_header": "Quality Score",
        "pdf_title": "Strategic AI Use Cases Powered by Databricks Genie Code", # MODIFIED
        "pdf_for": "For",
        "pdf_exec_summary": "Executive Summary",
        "pdf_toc_title": "Use Case Domains",
        "pdf_detailed_view": "Detailed Use Case Catalog",
        "pdf_disclaimer_title": "Disclaimer", # NEW
        "pdf_fallback_summary_p1": "This document outlines {total_cases} high-value analytical use cases identified for {business_name}. These scenarios, powered by Databricks Genie Code, are designed to drive significant business outcomes by leveraging your existing data assets.",
        "pdf_fallback_summary_p2": "The following pages provide a detailed breakdown of these opportunities, categorized by business domain, to help prioritize your AI initiatives.",
        "pptx_main_title": "Strategic AI Use Cases Powered by Databricks Genie Code", # MODIFIED
        "pptx_for": "For",
        "pptx_disclaimer_title": "Disclaimer",
        "pptx_domain_suffix": "Use Cases",
        # NEW: Query result example translations
        "example_results": "Example Results",
        "error_no_results": "Could not generate results, Check Notebook: {notebook_name} and use case id {use_case_id}",
        "input_data_original": "Input Data (Original Values)",
        "ai_generated_output": "AI-Generated Results (Output)",
        "column": "Column",
        "value": "Value",
        "executive_summary_not_available": "Executive summary not available.",
        "domain_summary_not_available": "Domain summary not available.",
        "summary_not_available": "Summary not available.",
        # Strategic Goal/Priority Alignment value translations
        "value_general_improvement": "General Improvement",
        "value_reduce_cost": "Reduce Cost",
        "value_increase_revenue": "Increase Revenue",
        "value_boost_productivity": "Boost Productivity",
        "value_mitigate_risk": "Mitigate Risk",
        "value_protect_revenue": "Protect Revenue",
        "value_align_to_regulations": "Align to Regulations",
        "value_improve_customer_experience": "Improve Customer Experience",
        "value_enable_data_driven_decisions": "Enable Data-Driven Decisions",
        "value_optimize_operations": "Optimize Operations",
        "value_empower_talent": "Empower Talent",
        "value_enhance_experience": "Enhance Experience",
        "value_drive_innovation": "Drive Innovation",
        "value_achieve_esg": "Achieve ESG",
        "value_execute_strategy": "Execute Strategy",
        # Analytics Technique value translations
        "value_forecasting": "Forecasting",
        "value_classification": "Classification",
        "value_anomaly_detection": "Anomaly Detection",
        "value_cohort_analysis": "Cohort Analysis",
        "value_segmentation": "Segmentation",
        "value_sentiment_analysis": "Sentiment Analysis",
        "value_trend_analysis": "Trend Analysis",
        "value_prescriptive_analytics": "Prescriptive Analytics",
        "value_root_cause_analysis": "Root Cause Analysis",
        "value_optimization": "Optimization",
        "value_recommendation": "Recommendation",
        "value_time_series_analysis": "Time Series Analysis",
        "value_predictive_analytics": "Predictive Analytics",
        "value_descriptive_analytics": "Descriptive Analytics"
    }

    def __init__(self, ai_agent, logger=None):
        """
        Initializes the TranslationService with an AIAgent instance.
        """
        self.ai_agent = ai_agent
        self.logger = logger or logging.getLogger(self.__class__.__name__)
        self.translation_cache = {}
        self._translation_cache_lock = threading.Lock()

    # Complete fallback translations for ALL keys - ensures translations are 100% reliable
    TRANSLATION_FALLBACKS = {
        "Arabic": {
            "main_title": "مولد حالات الاستخدام مدعوم بـ داتا بريكس جيني كود",
            "intro": "يحتوي هذا الدفتر على حالات استخدام تم إنشاؤها بواسطة الذكاء الاصطناعي استناداً إلى مخططاتك. فيما يلي ملخص للسيناريوهات المُنشأة حسب مجال الأعمال.",
            "domain": "مجال الأعمال",
            "total": "إجمالي حالات الاستخدام",
            "summaries": "ملخصات حالات الاستخدام",
            "sum_id": "المعرف",
            "sum_name": "الاسم",
            "sum_value": "القيمة التجارية",
            "sum_outcome": "النتيجة المتوقعة",
            "warning_header": "تحذير",
            "warning_body": "لا تقم بتشغيل هذا الدفتر. وهو مخصص للعرض التوضيحي والفهرسة فقط.",
            "disclaimer": "هذا المحتوى تم إنشاؤه بواسطة الذكاء الاصطناعي ويجب مراجعته من قبل مهندس مؤهل قبل استخدامه في الإنتاج. داتابريكس غير مسؤولة عن أي مشاكل ناتجة عن استخدام هذا المحتوى.",
            "detailed_scenarios": "تفاصيل حالات الاستخدام",
            "aspect": "الجانب",
            "description": "الوصف",
            "aspect_domain": "مجال الأعمال",
            "type": "النوع",
            "analytics_technique": "تقنية التحليل",
            "primary_table": "الجدول الرئيسي",
            "quality": "الجودة",
            "priority": "الأولوية",
            "usecase": "حالة الاستخدام",
            "description_label": "الوصف",
            "value_type_problem": "مشكلة",
            "value_type_risk": "مخاطرة",
            "value_type_opportunity": "فرصة",
            "value_type_improvement": "تحسين",
            "value_priority_ultra_high": "عالية للغاية",
            "value_priority_very_high": "عالية جداً",
            "value_priority_high": "عالية",
            "value_priority_medium": "متوسطة",
            "value_priority_low": "منخفضة",
            "value_priority_very_low": "منخفضة جداً",
            "value_priority_ultra_low": "منخفضة للغاية",
            "statement": "البيان",
            "solution": "الحل",
            "aspect_beneficiary": "المستفيد",
            "beneficiary": "المستفيد",
            "aspect_sponsor": "الراعي",
            "sponsor": "الراعي",
            "business_priority_alignment": "توافق أولوية الأعمال",
            "strategic_goals_alignment": "التوافق مع الأهداف الاستراتيجية",
            "subdomain": "النطاق الفرعي",
            "aspect_value": "القيمة التجارية",
            "business_value": "القيمة التجارية",
            "aspect_tables": "الجداول المستخدمة",
            "quality_reasons": "أسباب الجودة",
            "aspect_ai_function": "وظيفة الذكاء الاصطناعي",
            "aspect_analytics_technique": "تقنية التحليل",
            "aspect_primary_table": "الجدول الرئيسي",
            "aspect_quality": "الجودة",
            "aspect_priority": "الأولوية",
            "strategic_alignment": "التوافق الاستراتيجي",
            "return_on_investment": "العائد على الاستثمار",
            "reusability": "قابلية إعادة الاستخدام",
            "time_to_value": "الوقت للقيمة",
            "data_availability": "توفر البيانات",
            "data_accessibility": "إمكانية الوصول للبيانات",
            "architecture_fitness": "ملاءمة البنية",
            "team_skills": "مهارات الفريق",
            "domain_knowledge": "المعرفة بالمجال",
            "people_allocation": "تخصيص الموارد البشرية",
            "budget_allocation": "تخصيص الميزانية",
            "time_to_production": "الوقت للإنتاج",
            "value_score": "درجة القيمة",
            "feasibility_score": "درجة الجدوى",
            "priority_score": "درجة الأولوية",
            "inspire_score": "درجة إنسباير",
            "quality_score_header": "درجة الجودة",
            "pdf_title": "حالات استخدام الذكاء الاصطناعي الاستراتيجية مدعومة بـ داتا بريكس جيني كود",
            "pdf_for": "لـ",
            "pdf_exec_summary": "الملخص التنفيذي",
            "pdf_toc_title": "مجالات حالات الاستخدام",
            "pdf_detailed_view": "كتالوج حالات الاستخدام التفصيلية",
            "pdf_disclaimer_title": "إخلاء المسؤولية",
            "pdf_fallback_summary_p1": "يوضح هذا المستند {total_cases} حالة استخدام تحليلية عالية القيمة تم تحديدها لـ {business_name}. هذه السيناريوهات، مدعومة بـ داتا بريكس جيني كود، مصممة لتحقيق نتائج أعمال مهمة من خلال الاستفادة من أصول البيانات الحالية.",
            "pdf_fallback_summary_p2": "توفر الصفحات التالية تفصيلاً مفصلاً لهذه الفرص، مصنفة حسب مجال الأعمال، للمساعدة في تحديد أولويات مبادرات الذكاء الاصطناعي الخاصة بك.",
            "pptx_main_title": "حالات استخدام الذكاء الاصطناعي الاستراتيجية مدعومة بـ داتا بريكس جيني كود",
            "pptx_for": "لـ",
            "pptx_disclaimer_title": "إخلاء المسؤولية",
            "pptx_domain_suffix": "حالات الاستخدام",
            "example_results": "نتائج المثال",
            "error_no_results": "تعذر إنشاء النتائج، تحقق من الدفتر: {notebook_name} ومعرف حالة الاستخدام {use_case_id}",
            "input_data_original": "بيانات الإدخال (القيم الأصلية)",
            "ai_generated_output": "مخرجات الذكاء الاصطناعي",
            "column": "العمود",
            "value": "القيمة",
            "executive_summary_not_available": "الملخص التنفيذي غير متوفر.",
            "domain_summary_not_available": "ملخص المجال غير متوفر.",
            "summary_not_available": "الملخص غير متوفر.",
            "value_general_improvement": "تحسين عام",
            "value_reduce_cost": "تقليل التكلفة",
            "value_increase_revenue": "زيادة الإيرادات",
            "value_boost_productivity": "تعزيز الإنتاجية",
            "value_mitigate_risk": "تخفيف المخاطر",
            "value_protect_revenue": "حماية الإيرادات",
            "value_align_to_regulations": "الامتثال للوائح",
            "value_improve_customer_experience": "تحسين تجربة العملاء",
            "value_enable_data_driven_decisions": "تمكين القرارات المبنية على البيانات",
            "value_optimize_operations": "تحسين العمليات",
            "value_empower_talent": "تمكين المواهب",
            "value_enhance_experience": "تعزيز التجربة",
            "value_drive_innovation": "دفع الابتكار",
            "value_achieve_esg": "تحقيق ESG",
            "value_execute_strategy": "تنفيذ الاستراتيجية",
            "value_forecasting": "التنبؤ",
            "value_classification": "التصنيف",
            "value_anomaly_detection": "كشف الشذوذ",
            "value_cohort_analysis": "تحليل الأتراب",
            "value_segmentation": "التجزئة",
            "value_sentiment_analysis": "تحليل المشاعر",
            "value_trend_analysis": "تحليل الاتجاهات",
            "value_prescriptive_analytics": "التحليلات الوصفية",
            "value_root_cause_analysis": "تحليل السبب الجذري",
            "value_optimization": "التحسين",
            "value_recommendation": "التوصية",
            "value_time_series_analysis": "تحليل السلاسل الزمنية",
            "value_predictive_analytics": "التحليلات التنبؤية",
            "value_descriptive_analytics": "التحليلات الوصفية",
        },
        "Spanish": {
            "main_title": "Generador de Casos de Uso Impulsado por Databricks Genie Code",
            "intro": "Este cuaderno contiene casos de uso generados por IA basados en sus esquemas. A continuación se muestra un resumen de los escenarios generados por dominio de negocio.",
            "domain": "Dominio de Negocio",
            "total": "Total de Casos de Uso",
            "summaries": "Resúmenes de Casos de Uso",
            "sum_id": "ID",
            "sum_name": "Nombre",
            "sum_value": "Valor Comercial",
            "sum_outcome": "Resultado Esperado",
            "warning_header": "ADVERTENCIA",
            "warning_body": "No ejecute este cuaderno. Está destinado solo para demostración y catalogación.",
            "disclaimer": "Este contenido es generado por IA y debe ser revisado por un ingeniero calificado antes de su uso en producción. Databricks no es responsable de ningún problema derivado del uso de este contenido.",
            "detailed_scenarios": "Detalles de Casos de Uso",
            "aspect": "Aspecto",
            "description": "Descripción",
            "aspect_domain": "Dominio de Negocio",
            "type": "Tipo",
            "analytics_technique": "Técnica de Análisis",
            "primary_table": "Tabla Principal",
            "quality": "Calidad",
            "priority": "Prioridad",
            "usecase": "Caso de Uso",
            "description_label": "Descripción",
            "value_type_problem": "Problema",
            "value_type_risk": "Riesgo",
            "value_type_opportunity": "Oportunidad",
            "value_type_improvement": "Mejora",
            "value_priority_ultra_high": "Extremadamente Alta",
            "value_priority_very_high": "Muy Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Media",
            "value_priority_low": "Baja",
            "value_priority_very_low": "Muy Baja",
            "value_priority_ultra_low": "Extremadamente Baja",
            "statement": "Declaración",
            "solution": "Solución",
            "aspect_beneficiary": "Beneficiario",
            "beneficiary": "Beneficiario",
            "aspect_sponsor": "Patrocinador",
            "sponsor": "Patrocinador",
            "business_priority_alignment": "Alineación de Prioridad Empresarial",
            "strategic_goals_alignment": "Alineación con Objetivos Estratégicos",
            "subdomain": "Subdominio",
            "aspect_value": "Valor Comercial",
            "business_value": "Valor Comercial",
            "aspect_tables": "Tablas Involucradas",
            "quality_reasons": "Razones de Calidad",
            "aspect_ai_function": "Función de IA",
            "aspect_analytics_technique": "Técnica de Análisis",
            "aspect_primary_table": "Tabla Principal",
            "aspect_quality": "Calidad",
            "aspect_priority": "Prioridad",
            "strategic_alignment": "Alineación Estratégica",
            "return_on_investment": "Retorno de Inversión",
            "reusability": "Reusabilidad",
            "time_to_value": "Tiempo al Valor",
            "data_availability": "Disponibilidad de Datos",
            "data_accessibility": "Accesibilidad de Datos",
            "architecture_fitness": "Aptitud de Arquitectura",
            "team_skills": "Habilidades del Equipo",
            "domain_knowledge": "Conocimiento del Dominio",
            "people_allocation": "Asignación de Personal",
            "budget_allocation": "Asignación de Presupuesto",
            "time_to_production": "Tiempo a Producción",
            "value_score": "Puntuación de Valor",
            "feasibility_score": "Puntuación de Viabilidad",
            "priority_score": "Puntuación de Prioridad",
            "inspire_score": "Puntuación Inspire",
            "quality_score_header": "Puntuación de Calidad",
            "pdf_title": "Casos de Uso de IA Estratégica Impulsados por Databricks Genie Code",
            "pdf_for": "Para",
            "pdf_exec_summary": "Resumen Ejecutivo",
            "pdf_toc_title": "Dominios de Casos de Uso",
            "pdf_detailed_view": "Catálogo Detallado de Casos de Uso",
            "pdf_disclaimer_title": "Descargo de Responsabilidad",
            "pdf_fallback_summary_p1": "Este documento describe {total_cases} casos de uso analíticos de alto valor identificados para {business_name}.",
            "pdf_fallback_summary_p2": "Las siguientes páginas proporcionan un desglose detallado de estas oportunidades, categorizadas por dominio de negocio.",
            "pptx_main_title": "Casos de Uso de IA Estratégica Impulsados por Databricks Genie Code",
            "pptx_for": "Para",
            "pptx_disclaimer_title": "Descargo de Responsabilidad",
            "pptx_domain_suffix": "Casos de Uso",
            "example_results": "Resultados de Ejemplo",
            "error_no_results": "No se pudieron generar resultados. Verificar Cuaderno: {notebook_name} y caso de uso {use_case_id}",
            "input_data_original": "Datos de Entrada (Valores Originales)",
            "ai_generated_output": "Resultados Generados por IA",
            "column": "Columna",
            "value": "Valor",
            "executive_summary_not_available": "Resumen ejecutivo no disponible.",
            "domain_summary_not_available": "Resumen del dominio no disponible.",
            "summary_not_available": "Resumen no disponible.",
            "value_general_improvement": "Mejora General",
            "value_reduce_cost": "Reducir Costos",
            "value_increase_revenue": "Aumentar Ingresos",
            "value_boost_productivity": "Impulsar Productividad",
            "value_mitigate_risk": "Mitigar Riesgos",
            "value_protect_revenue": "Proteger Ingresos",
            "value_align_to_regulations": "Cumplir Regulaciones",
            "value_improve_customer_experience": "Mejorar Experiencia del Cliente",
            "value_enable_data_driven_decisions": "Habilitar Decisiones Basadas en Datos",
            "value_optimize_operations": "Optimizar Operaciones",
            "value_empower_talent": "Empoderar Talento",
            "value_enhance_experience": "Mejorar Experiencia",
            "value_drive_innovation": "Impulsar Innovación",
            "value_achieve_esg": "Lograr ESG",
            "value_execute_strategy": "Ejecutar Estrategia",
            "value_forecasting": "Pronóstico",
            "value_classification": "Clasificación",
            "value_anomaly_detection": "Detección de Anomalías",
            "value_cohort_analysis": "Análisis de Cohortes",
            "value_segmentation": "Segmentación",
            "value_sentiment_analysis": "Análisis de Sentimiento",
            "value_trend_analysis": "Análisis de Tendencias",
            "value_prescriptive_analytics": "Analítica Prescriptiva",
            "value_root_cause_analysis": "Análisis de Causa Raíz",
            "value_optimization": "Optimización",
            "value_recommendation": "Recomendación",
            "value_time_series_analysis": "Análisis de Series Temporales",
            "value_predictive_analytics": "Analítica Predictiva",
            "value_descriptive_analytics": "Analítica Descriptiva",
        },
        "French": {
            "main_title": "Générateur de Cas d'Utilisation Propulsé par Databricks Genie Code",
            "intro": "Ce carnet contient des cas d'utilisation générés par l'IA basés sur vos schémas. Voici un résumé des scénarios générés par domaine d'activité.",
            "domain": "Domaine d'Activité",
            "total": "Total des Cas d'Utilisation",
            "summaries": "Résumés des Cas d'Utilisation",
            "sum_id": "ID",
            "sum_name": "Nom",
            "sum_value": "Valeur Commerciale",
            "sum_outcome": "Résultat Attendu",
            "warning_header": "AVERTISSEMENT",
            "warning_body": "N'exécutez pas ce carnet. Il est destiné uniquement à la démonstration et au catalogage.",
            "disclaimer": "Ce contenu est généré par l'IA et doit être examiné par un ingénieur qualifié avant toute mise en production. Databricks n'est pas responsable des problèmes découlant de l'utilisation de ce contenu.",
            "detailed_scenarios": "Détails des Cas d'Utilisation",
            "aspect": "Aspect",
            "description": "Description",
            "aspect_domain": "Domaine d'Activité",
            "type": "Type",
            "analytics_technique": "Technique d'Analyse",
            "primary_table": "Table Principale",
            "quality": "Qualité",
            "priority": "Priorité",
            "usecase": "Cas d'Utilisation",
            "description_label": "Description",
            "value_type_problem": "Problème",
            "value_type_risk": "Risque",
            "value_type_opportunity": "Opportunité",
            "value_type_improvement": "Amélioration",
            "value_priority_ultra_high": "Extrêmement Haute",
            "value_priority_very_high": "Très Haute",
            "value_priority_high": "Haute",
            "value_priority_medium": "Moyenne",
            "value_priority_low": "Basse",
            "value_priority_very_low": "Très Basse",
            "value_priority_ultra_low": "Extrêmement Basse",
            "statement": "Énoncé",
            "solution": "Solution",
            "aspect_beneficiary": "Bénéficiaire",
            "beneficiary": "Bénéficiaire",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Alignement de Priorité d'Entreprise",
            "strategic_goals_alignment": "Alignement aux Objectifs Stratégiques",
            "subdomain": "Sous-domaine",
            "aspect_value": "Valeur Commerciale",
            "business_value": "Valeur Commerciale",
            "aspect_tables": "Tables Impliquées",
            "quality_reasons": "Raisons de Qualité",
            "aspect_ai_function": "Fonction IA",
            "aspect_analytics_technique": "Technique d'Analyse",
            "aspect_primary_table": "Table Principale",
            "aspect_quality": "Qualité",
            "aspect_priority": "Priorité",
            "strategic_alignment": "Alignement Stratégique",
            "return_on_investment": "Retour sur Investissement",
            "reusability": "Réutilisabilité",
            "time_to_value": "Délai de Valorisation",
            "data_availability": "Disponibilité des Données",
            "data_accessibility": "Accessibilité des Données",
            "architecture_fitness": "Adéquation Architecturale",
            "team_skills": "Compétences de l'Équipe",
            "domain_knowledge": "Connaissance du Domaine",
            "people_allocation": "Allocation du Personnel",
            "budget_allocation": "Allocation Budgétaire",
            "time_to_production": "Délai de Production",
            "value_score": "Score de Valeur",
            "feasibility_score": "Score de Faisabilité",
            "priority_score": "Score de Priorité",
            "inspire_score": "Score Inspire",
            "quality_score_header": "Score de Qualité",
            "pdf_title": "Cas d'Utilisation IA Stratégiques Propulsés par Databricks Genie Code",
            "pdf_for": "Pour",
            "pdf_exec_summary": "Résumé Exécutif",
            "pdf_toc_title": "Domaines des Cas d'Utilisation",
            "pdf_detailed_view": "Catalogue Détaillé des Cas d'Utilisation",
            "pdf_disclaimer_title": "Avertissement",
            "pdf_fallback_summary_p1": "Ce document présente {total_cases} cas d'utilisation analytiques à forte valeur identifiés pour {business_name}.",
            "pdf_fallback_summary_p2": "Les pages suivantes fournissent une répartition détaillée de ces opportunités, classées par domaine d'activité.",
            "pptx_main_title": "Cas d'Utilisation IA Stratégiques Propulsés par Databricks Genie Code",
            "pptx_for": "Pour",
            "pptx_disclaimer_title": "Avertissement",
            "pptx_domain_suffix": "Cas d'Utilisation",
            "example_results": "Résultats d'Exemple",
            "error_no_results": "Impossible de générer les résultats. Vérifier le Carnet: {notebook_name} et cas d'utilisation {use_case_id}",
            "input_data_original": "Données d'Entrée (Valeurs Originales)",
            "ai_generated_output": "Résultats Générés par l'IA",
            "column": "Colonne",
            "value": "Valeur",
            "executive_summary_not_available": "Résumé exécutif non disponible.",
            "domain_summary_not_available": "Résumé du domaine non disponible.",
            "summary_not_available": "Résumé non disponible.",
            "value_general_improvement": "Amélioration Générale",
            "value_reduce_cost": "Réduire les Coûts",
            "value_increase_revenue": "Augmenter les Revenus",
            "value_boost_productivity": "Améliorer la Productivité",
            "value_mitigate_risk": "Atténuer les Risques",
            "value_protect_revenue": "Protéger les Revenus",
            "value_align_to_regulations": "Se Conformer aux Réglementations",
            "value_improve_customer_experience": "Améliorer l'Expérience Client",
            "value_enable_data_driven_decisions": "Permettre les Décisions Basées sur les Données",
            "value_optimize_operations": "Optimiser les Opérations",
            "value_empower_talent": "Autonomiser les Talents",
            "value_enhance_experience": "Améliorer l'Expérience",
            "value_drive_innovation": "Stimuler l'Innovation",
            "value_achieve_esg": "Atteindre ESG",
            "value_execute_strategy": "Exécuter la Stratégie",
            "value_forecasting": "Prévision",
            "value_classification": "Classification",
            "value_anomaly_detection": "Détection d'Anomalies",
            "value_cohort_analysis": "Analyse de Cohortes",
            "value_segmentation": "Segmentation",
            "value_sentiment_analysis": "Analyse de Sentiments",
            "value_trend_analysis": "Analyse des Tendances",
            "value_prescriptive_analytics": "Analytique Prescriptive",
            "value_root_cause_analysis": "Analyse des Causes Profondes",
            "value_optimization": "Optimisation",
            "value_recommendation": "Recommandation",
            "value_time_series_analysis": "Analyse de Séries Temporelles",
            "value_predictive_analytics": "Analytique Prédictive",
            "value_descriptive_analytics": "Analytique Descriptive",
        },
        "German": {
            "main_title": "Anwendungsfall-Generator Unterstützt von Databricks Genie Code",
            "intro": "Dieses Notebook enthält KI-generierte Anwendungsfälle basierend auf Ihren Schemas. Nachfolgend finden Sie eine Zusammenfassung der generierten Szenarien nach Geschäftsbereich.",
            "domain": "Geschäftsbereich",
            "total": "Gesamtzahl der Anwendungsfälle",
            "summaries": "Zusammenfassungen der Anwendungsfälle",
            "sum_id": "ID",
            "sum_name": "Name",
            "sum_value": "Geschäftswert",
            "sum_outcome": "Erwartetes Ergebnis",
            "warning_header": "WARNUNG",
            "warning_body": "Führen Sie dieses Notebook nicht aus. Es dient nur zur Demonstration und Katalogisierung.",
            "disclaimer": "Dieser Inhalt wurde von KI generiert und sollte von einem qualifizierten Ingenieur überprüft werden, bevor er in der Produktion eingesetzt wird. Databricks übernimmt keine Haftung für Probleme, die aus der Verwendung dieses Inhalts entstehen.",
            "detailed_scenarios": "Anwendungsfall-Details",
            "aspect": "Aspekt",
            "description": "Beschreibung",
            "aspect_domain": "Geschäftsbereich",
            "type": "Typ",
            "analytics_technique": "Analysetechnik",
            "primary_table": "Haupttabelle",
            "quality": "Qualität",
            "priority": "Priorität",
            "usecase": "Anwendungsfall",
            "description_label": "Beschreibung",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Chance",
            "value_type_improvement": "Verbesserung",
            "value_priority_ultra_high": "Extrem Hoch",
            "value_priority_very_high": "Sehr Hoch",
            "value_priority_high": "Hoch",
            "value_priority_medium": "Mittel",
            "value_priority_low": "Niedrig",
            "value_priority_very_low": "Sehr Niedrig",
            "value_priority_ultra_low": "Extrem Niedrig",
            "statement": "Aussage",
            "solution": "Lösung",
            "aspect_beneficiary": "Begünstigter",
            "beneficiary": "Begünstigter",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Geschäftspriorität Ausrichtung",
            "strategic_goals_alignment": "Strategische Zielausrichtung",
            "subdomain": "Subdomäne",
            "aspect_value": "Geschäftswert",
            "business_value": "Geschäftswert",
            "aspect_tables": "Beteiligte Tabellen",
            "quality_reasons": "Qualitätsgründe",
            "aspect_ai_function": "KI-Funktion",
            "aspect_analytics_technique": "Analysetechnik",
            "aspect_primary_table": "Haupttabelle",
            "aspect_quality": "Qualität",
            "aspect_priority": "Priorität",
            "strategic_alignment": "Strategische Ausrichtung",
            "return_on_investment": "Kapitalrendite",
            "reusability": "Wiederverwendbarkeit",
            "time_to_value": "Zeit bis zum Wert",
            "data_availability": "Datenverfügbarkeit",
            "data_accessibility": "Datenzugänglichkeit",
            "architecture_fitness": "Architektureignung",
            "team_skills": "Teamfähigkeiten",
            "domain_knowledge": "Fachwissen",
            "people_allocation": "Personalzuweisung",
            "budget_allocation": "Budgetzuweisung",
            "time_to_production": "Zeit bis zur Produktion",
            "value_score": "Wertpunktzahl",
            "feasibility_score": "Machbarkeitspunktzahl",
            "priority_score": "Prioritätspunktzahl",
            "inspire_score": "Inspire-Punktzahl",
            "quality_score_header": "Qualitätspunktzahl",
            "pdf_title": "Strategische KI-Anwendungsfälle Unterstützt von Databricks Genie Code",
            "pdf_for": "Für",
            "pdf_exec_summary": "Zusammenfassung",
            "pdf_toc_title": "Anwendungsfall-Bereiche",
            "pdf_detailed_view": "Detaillierter Anwendungsfallkatalog",
            "pdf_disclaimer_title": "Haftungsausschluss",
            "pdf_fallback_summary_p1": "Dieses Dokument beschreibt {total_cases} hochwertige analytische Anwendungsfälle, die für {business_name} identifiziert wurden.",
            "pdf_fallback_summary_p2": "Die folgenden Seiten bieten eine detaillierte Aufschlüsselung dieser Möglichkeiten, kategorisiert nach Geschäftsbereich.",
            "pptx_main_title": "Strategische KI-Anwendungsfälle Unterstützt von Databricks Genie Code",
            "pptx_for": "Für",
            "pptx_disclaimer_title": "Haftungsausschluss",
            "pptx_domain_suffix": "Anwendungsfälle",
            "example_results": "Beispielergebnisse",
            "error_no_results": "Ergebnisse konnten nicht generiert werden. Prüfen Sie Notebook: {notebook_name} und Anwendungsfall {use_case_id}",
            "input_data_original": "Eingabedaten (Originalwerte)",
            "ai_generated_output": "KI-generierte Ergebnisse",
            "column": "Spalte",
            "value": "Wert",
            "executive_summary_not_available": "Zusammenfassung nicht verfügbar.",
            "domain_summary_not_available": "Bereichszusammenfassung nicht verfügbar.",
            "summary_not_available": "Zusammenfassung nicht verfügbar.",
            "value_general_improvement": "Allgemeine Verbesserung",
            "value_reduce_cost": "Kosten Reduzieren",
            "value_increase_revenue": "Umsatz Steigern",
            "value_boost_productivity": "Produktivität Steigern",
            "value_mitigate_risk": "Risiken Mindern",
            "value_protect_revenue": "Umsatz Schützen",
            "value_align_to_regulations": "Vorschriften Einhalten",
            "value_improve_customer_experience": "Kundenerlebnis Verbessern",
            "value_enable_data_driven_decisions": "Datengestützte Entscheidungen Ermöglichen",
            "value_optimize_operations": "Betrieb Optimieren",
            "value_empower_talent": "Talente Fördern",
            "value_enhance_experience": "Erlebnis Verbessern",
            "value_drive_innovation": "Innovation Vorantreiben",
            "value_achieve_esg": "ESG Erreichen",
            "value_execute_strategy": "Strategie Umsetzen",
            "value_forecasting": "Prognose",
            "value_classification": "Klassifizierung",
            "value_anomaly_detection": "Anomalieerkennung",
            "value_cohort_analysis": "Kohortenanalyse",
            "value_segmentation": "Segmentierung",
            "value_sentiment_analysis": "Stimmungsanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Präskriptive Analytik",
            "value_root_cause_analysis": "Ursachenanalyse",
            "value_optimization": "Optimierung",
            "value_recommendation": "Empfehlung",
            "value_time_series_analysis": "Zeitreihenanalyse",
            "value_predictive_analytics": "Prädiktive Analytik",
            "value_descriptive_analytics": "Deskriptive Analytik",
        },
        "Portuguese": {
            "main_title": "Gerador de Casos de Uso Impulsionado por Databricks Genie Code",
            "intro": "Este notebook contém casos de uso gerados por IA baseados em seus esquemas. Abaixo está um resumo dos cenários gerados por domínio de negócio.",
            "domain": "Domínio de Negócio",
            "total": "Total de Casos de Uso",
            "summaries": "Resumos de Casos de Uso",
            "sum_id": "ID",
            "sum_name": "Nome",
            "sum_value": "Valor de Negócio",
            "sum_outcome": "Resultado Esperado",
            "warning_header": "AVISO",
            "warning_body": "Não execute este notebook. É destinado apenas para demonstração e catalogação.",
            "disclaimer": "Este conteúdo foi gerado por IA e deve ser revisado por um engenheiro qualificado antes de ser utilizado em produção. A Databricks não é responsável por quaisquer problemas decorrentes do uso deste conteúdo.",
            "detailed_scenarios": "Detalhes dos Casos de Uso",
            "aspect": "Aspecto",
            "description": "Descrição",
            "aspect_domain": "Domínio de Negócio",
            "type": "Tipo",
            "analytics_technique": "Técnica de Análise",
            "primary_table": "Tabela Principal",
            "quality": "Qualidade",
            "priority": "Prioridade",
            "usecase": "Caso de Uso",
            "description_label": "Descrição",
            "value_type_problem": "Problema",
            "value_type_risk": "Risco",
            "value_type_opportunity": "Oportunidade",
            "value_type_improvement": "Melhoria",
            "value_priority_ultra_high": "Extremamente Alta",
            "value_priority_very_high": "Muito Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Média",
            "value_priority_low": "Baixa",
            "value_priority_very_low": "Muito Baixa",
            "value_priority_ultra_low": "Extremamente Baixa",
            "statement": "Declaração",
            "solution": "Solução",
            "aspect_beneficiary": "Beneficiário",
            "beneficiary": "Beneficiário",
            "aspect_sponsor": "Patrocinador",
            "sponsor": "Patrocinador",
            "business_priority_alignment": "Alinhamento de Prioridade de Negócio",
            "strategic_goals_alignment": "Alinhamento com Objetivos Estratégicos",
            "subdomain": "Subdomínio",
            "aspect_value": "Valor de Negócio",
            "business_value": "Valor de Negócio",
            "aspect_tables": "Tabelas Envolvidas",
            "quality_reasons": "Razões de Qualidade",
            "aspect_ai_function": "Função de IA",
            "aspect_analytics_technique": "Técnica de Análise",
            "aspect_primary_table": "Tabela Principal",
            "aspect_quality": "Qualidade",
            "aspect_priority": "Prioridade",
            "strategic_alignment": "Alinhamento Estratégico",
            "return_on_investment": "Retorno sobre Investimento",
            "reusability": "Reusabilidade",
            "time_to_value": "Tempo para Valor",
            "data_availability": "Disponibilidade de Dados",
            "data_accessibility": "Acessibilidade de Dados",
            "architecture_fitness": "Adequação da Arquitetura",
            "team_skills": "Habilidades da Equipe",
            "domain_knowledge": "Conhecimento do Domínio",
            "people_allocation": "Alocação de Pessoas",
            "budget_allocation": "Alocação de Orçamento",
            "time_to_production": "Tempo para Produção",
            "value_score": "Pontuação de Valor",
            "feasibility_score": "Pontuação de Viabilidade",
            "priority_score": "Pontuação de Prioridade",
            "inspire_score": "Pontuação Inspire",
            "quality_score_header": "Pontuação de Qualidade",
            "pdf_title": "Casos de Uso de IA Estratégica Impulsionados por Databricks Genie Code",
            "pdf_for": "Para",
            "pdf_exec_summary": "Resumo Executivo",
            "pdf_toc_title": "Domínios de Casos de Uso",
            "pdf_detailed_view": "Catálogo Detalhado de Casos de Uso",
            "pdf_disclaimer_title": "Aviso Legal",
            "pdf_fallback_summary_p1": "Este documento descreve {total_cases} casos de uso analíticos de alto valor identificados para {business_name}.",
            "pdf_fallback_summary_p2": "As páginas seguintes fornecem uma análise detalhada dessas oportunidades, categorizadas por domínio de negócio.",
            "pptx_main_title": "Casos de Uso de IA Estratégica Impulsionados por Databricks Genie Code",
            "pptx_for": "Para",
            "pptx_disclaimer_title": "Aviso Legal",
            "pptx_domain_suffix": "Casos de Uso",
            "example_results": "Resultados de Exemplo",
            "error_no_results": "Não foi possível gerar resultados. Verificar Notebook: {notebook_name} e caso de uso {use_case_id}",
            "input_data_original": "Dados de Entrada (Valores Originais)",
            "ai_generated_output": "Resultados Gerados por IA",
            "column": "Coluna",
            "value": "Valor",
            "executive_summary_not_available": "Resumo executivo não disponível.",
            "domain_summary_not_available": "Resumo do domínio não disponível.",
            "summary_not_available": "Resumo não disponível.",
            "value_general_improvement": "Melhoria Geral",
            "value_reduce_cost": "Reduzir Custos",
            "value_increase_revenue": "Aumentar Receita",
            "value_boost_productivity": "Aumentar Produtividade",
            "value_mitigate_risk": "Mitigar Riscos",
            "value_protect_revenue": "Proteger Receita",
            "value_align_to_regulations": "Cumprir Regulamentos",
            "value_improve_customer_experience": "Melhorar Experiência do Cliente",
            "value_enable_data_driven_decisions": "Habilitar Decisões Baseadas em Dados",
            "value_optimize_operations": "Otimizar Operações",
            "value_empower_talent": "Capacitar Talentos",
            "value_enhance_experience": "Melhorar Experiência",
            "value_drive_innovation": "Impulsionar Inovação",
            "value_achieve_esg": "Alcançar ESG",
            "value_execute_strategy": "Executar Estratégia",
            "value_forecasting": "Previsão",
            "value_classification": "Classificação",
            "value_anomaly_detection": "Detecção de Anomalias",
            "value_cohort_analysis": "Análise de Coorte",
            "value_segmentation": "Segmentação",
            "value_sentiment_analysis": "Análise de Sentimento",
            "value_trend_analysis": "Análise de Tendências",
            "value_prescriptive_analytics": "Análise Prescritiva",
            "value_root_cause_analysis": "Análise de Causa Raiz",
            "value_optimization": "Otimização",
            "value_recommendation": "Recomendação",
            "value_time_series_analysis": "Análise de Séries Temporais",
            "value_predictive_analytics": "Análise Preditiva",
            "value_descriptive_analytics": "Análise Descritiva",
        },
        "Italian": {
            "main_title": "Generatore di Casi d'Uso Alimentato da Databricks Genie Code",
            "intro": "Questo notebook contiene casi d'uso generati dall'IA basati sui tuoi schemi. Di seguito è riportato un riepilogo degli scenari generati per dominio aziendale.",
            "domain": "Dominio Aziendale",
            "total": "Totale Casi d'Uso",
            "summaries": "Riepiloghi dei Casi d'Uso",
            "sum_id": "ID",
            "sum_name": "Nome",
            "sum_value": "Valore Aziendale",
            "sum_outcome": "Risultato Atteso",
            "warning_header": "AVVERTIMENTO",
            "warning_body": "Non eseguire questo notebook. È destinato solo a scopi dimostrativi e di catalogazione.",
            "disclaimer": "Questo contenuto è generato dall'IA e deve essere verificato da un ingegnere qualificato prima dell'uso in produzione. Databricks non è responsabile per eventuali problemi derivanti dall'uso di questo contenuto.",
            "detailed_scenarios": "Dettagli dei Casi d'Uso",
            "aspect": "Aspetto",
            "description": "Descrizione",
            "aspect_domain": "Dominio Aziendale",
            "type": "Tipo",
            "analytics_technique": "Tecnica di Analisi",
            "primary_table": "Tabella Principale",
            "quality": "Qualità",
            "priority": "Priorità",
            "usecase": "Caso d'Uso",
            "description_label": "Descrizione",
            "value_type_problem": "Problema",
            "value_type_risk": "Rischio",
            "value_type_opportunity": "Opportunità",
            "value_type_improvement": "Miglioramento",
            "value_priority_ultra_high": "Estremamente Alta",
            "value_priority_very_high": "Molto Alta",
            "value_priority_high": "Alta",
            "value_priority_medium": "Media",
            "value_priority_low": "Bassa",
            "value_priority_very_low": "Molto Bassa",
            "value_priority_ultra_low": "Estremamente Bassa",
            "statement": "Dichiarazione",
            "solution": "Soluzione",
            "aspect_beneficiary": "Beneficiario",
            "beneficiary": "Beneficiario",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Allineamento Priorità Aziendale",
            "strategic_goals_alignment": "Allineamento agli Obiettivi Strategici",
            "subdomain": "Sottodominio",
            "aspect_value": "Valore Aziendale",
            "business_value": "Valore Aziendale",
            "aspect_tables": "Tabelle Coinvolte",
            "quality_reasons": "Ragioni di Qualità",
            "aspect_ai_function": "Funzione IA",
            "aspect_analytics_technique": "Tecnica di Analisi",
            "aspect_primary_table": "Tabella Principale",
            "aspect_quality": "Qualità",
            "aspect_priority": "Priorità",
            "strategic_alignment": "Allineamento Strategico",
            "return_on_investment": "Ritorno sull'Investimento",
            "reusability": "Riutilizzabilità",
            "time_to_value": "Tempo al Valore",
            "data_availability": "Disponibilità dei Dati",
            "data_accessibility": "Accessibilità dei Dati",
            "architecture_fitness": "Idoneità dell'Architettura",
            "team_skills": "Competenze del Team",
            "domain_knowledge": "Conoscenza del Dominio",
            "people_allocation": "Allocazione del Personale",
            "budget_allocation": "Allocazione del Budget",
            "time_to_production": "Tempo alla Produzione",
            "value_score": "Punteggio di Valore",
            "feasibility_score": "Punteggio di Fattibilità",
            "priority_score": "Punteggio di Priorità",
            "inspire_score": "Punteggio Inspire",
            "quality_score_header": "Punteggio di Qualità",
            "pdf_title": "Casi d'Uso IA Strategici Alimentati da Databricks Genie Code",
            "pdf_for": "Per",
            "pdf_exec_summary": "Riepilogo Esecutivo",
            "pdf_toc_title": "Domini dei Casi d'Uso",
            "pdf_detailed_view": "Catalogo Dettagliato dei Casi d'Uso",
            "pdf_disclaimer_title": "Disclaimer",
            "pdf_fallback_summary_p1": "Questo documento descrive {total_cases} casi d'uso analitici ad alto valore identificati per {business_name}.",
            "pdf_fallback_summary_p2": "Le pagine seguenti forniscono un'analisi dettagliata di queste opportunità, categorizzate per dominio aziendale.",
            "pptx_main_title": "Casi d'Uso IA Strategici Alimentati da Databricks Genie Code",
            "pptx_for": "Per",
            "pptx_disclaimer_title": "Disclaimer",
            "pptx_domain_suffix": "Casi d'Uso",
            "example_results": "Risultati di Esempio",
            "error_no_results": "Impossibile generare i risultati. Verificare Notebook: {notebook_name} e caso d'uso {use_case_id}",
            "input_data_original": "Dati di Input (Valori Originali)",
            "ai_generated_output": "Risultati Generati dall'IA",
            "column": "Colonna",
            "value": "Valore",
            "executive_summary_not_available": "Riepilogo esecutivo non disponibile.",
            "domain_summary_not_available": "Riepilogo del dominio non disponibile.",
            "summary_not_available": "Riepilogo non disponibile.",
            "value_general_improvement": "Miglioramento Generale",
            "value_reduce_cost": "Ridurre i Costi",
            "value_increase_revenue": "Aumentare i Ricavi",
            "value_boost_productivity": "Aumentare la Produttività",
            "value_mitigate_risk": "Mitigare i Rischi",
            "value_protect_revenue": "Proteggere i Ricavi",
            "value_align_to_regulations": "Conformarsi alle Normative",
            "value_improve_customer_experience": "Migliorare l'Esperienza del Cliente",
            "value_enable_data_driven_decisions": "Abilitare Decisioni Basate sui Dati",
            "value_optimize_operations": "Ottimizzare le Operazioni",
            "value_empower_talent": "Valorizzare i Talenti",
            "value_enhance_experience": "Migliorare l'Esperienza",
            "value_drive_innovation": "Promuovere l'Innovazione",
            "value_achieve_esg": "Raggiungere ESG",
            "value_execute_strategy": "Eseguire la Strategia",
            "value_forecasting": "Previsione",
            "value_classification": "Classificazione",
            "value_anomaly_detection": "Rilevamento Anomalie",
            "value_cohort_analysis": "Analisi di Coorte",
            "value_segmentation": "Segmentazione",
            "value_sentiment_analysis": "Analisi del Sentimento",
            "value_trend_analysis": "Analisi delle Tendenze",
            "value_prescriptive_analytics": "Analisi Prescrittiva",
            "value_root_cause_analysis": "Analisi delle Cause Profonde",
            "value_optimization": "Ottimizzazione",
            "value_recommendation": "Raccomandazione",
            "value_time_series_analysis": "Analisi delle Serie Temporali",
            "value_predictive_analytics": "Analisi Predittiva",
            "value_descriptive_analytics": "Analisi Descrittiva",
        },
        "Chinese (Mandarin)": {
            "main_title": "由 达塔布里克斯 吉尼 科德 驱动的用例生成器",
            "intro": "本笔记本包含基于您的架构由AI生成的用例。以下是按业务领域生成的场景摘要。",
            "domain": "业务领域",
            "total": "用例总数",
            "summaries": "用例摘要",
            "sum_id": "ID",
            "sum_name": "名称",
            "sum_value": "商业价值",
            "sum_outcome": "预期结果",
            "warning_header": "警告",
            "warning_body": "请勿运行此笔记本。它仅用于演示和编目目的。",
            "disclaimer": "此内容由AI生成，应由合格工程师审核后方可用于生产环境。Databricks不对因使用此内容而产生的任何问题承担责任。",
            "detailed_scenarios": "用例详情",
            "aspect": "方面",
            "description": "描述",
            "aspect_domain": "业务领域",
            "type": "类型",
            "analytics_technique": "分析技术",
            "primary_table": "主表",
            "quality": "质量",
            "priority": "优先级",
            "usecase": "用例",
            "description_label": "描述",
            "value_type_problem": "问题",
            "value_type_risk": "风险",
            "value_type_opportunity": "机会",
            "value_type_improvement": "改进",
            "value_priority_ultra_high": "极高",
            "value_priority_very_high": "非常高",
            "value_priority_high": "高",
            "value_priority_medium": "中等",
            "value_priority_low": "低",
            "value_priority_very_low": "非常低",
            "value_priority_ultra_low": "极低",
            "statement": "陈述",
            "solution": "解决方案",
            "aspect_beneficiary": "受益者",
            "beneficiary": "受益者",
            "aspect_sponsor": "发起人",
            "sponsor": "发起人",
            "business_priority_alignment": "业务优先级对齐",
            "strategic_goals_alignment": "战略目标对齐",
            "subdomain": "子领域",
            "aspect_value": "商业价值",
            "business_value": "商业价值",
            "aspect_tables": "涉及的表",
            "quality_reasons": "质量原因",
            "aspect_ai_function": "AI功能",
            "aspect_analytics_technique": "分析技术",
            "aspect_primary_table": "主表",
            "aspect_quality": "质量",
            "aspect_priority": "优先级",
            "strategic_alignment": "战略对齐",
            "return_on_investment": "投资回报率",
            "reusability": "可重用性",
            "time_to_value": "价值实现时间",
            "data_availability": "数据可用性",
            "data_accessibility": "数据可访问性",
            "architecture_fitness": "架构适配性",
            "team_skills": "团队技能",
            "domain_knowledge": "领域知识",
            "people_allocation": "人员分配",
            "budget_allocation": "预算分配",
            "time_to_production": "投产时间",
            "value_score": "价值分数",
            "feasibility_score": "可行性分数",
            "priority_score": "优先级分数",
            "inspire_score": "Inspire 评分",
            "quality_score_header": "质量评分",
            "pdf_title": "战略AI用例 由 达塔布里克斯 吉尼 科德 驱动",
            "pdf_for": "为",
            "pdf_exec_summary": "执行摘要",
            "pdf_toc_title": "用例领域",
            "pdf_detailed_view": "详细用例目录",
            "pdf_disclaimer_title": "免责声明",
            "pdf_fallback_summary_p1": "本文档概述了为{business_name}识别的{total_cases}个高价值分析用例。",
            "pdf_fallback_summary_p2": "以下页面按业务领域分类提供这些机会的详细分析。",
            "pptx_main_title": "战略AI用例 由 达塔布里克斯 吉尼 科德 驱动",
            "pptx_for": "为",
            "pptx_disclaimer_title": "免责声明",
            "pptx_domain_suffix": "用例",
            "example_results": "示例结果",
            "error_no_results": "无法生成结果。请检查笔记本：{notebook_name}和用例{use_case_id}",
            "input_data_original": "输入数据（原始值）",
            "ai_generated_output": "AI生成结果",
            "column": "列",
            "value": "值",
            "executive_summary_not_available": "执行摘要不可用。",
            "domain_summary_not_available": "领域摘要不可用。",
            "summary_not_available": "摘要不可用。",
            "value_general_improvement": "一般改进",
            "value_reduce_cost": "降低成本",
            "value_increase_revenue": "增加收入",
            "value_boost_productivity": "提高生产力",
            "value_mitigate_risk": "降低风险",
            "value_protect_revenue": "保护收入",
            "value_align_to_regulations": "符合法规",
            "value_improve_customer_experience": "改善客户体验",
            "value_enable_data_driven_decisions": "实现数据驱动决策",
            "value_optimize_operations": "优化运营",
            "value_empower_talent": "赋能人才",
            "value_enhance_experience": "提升体验",
            "value_drive_innovation": "推动创新",
            "value_achieve_esg": "实现ESG",
            "value_execute_strategy": "执行战略",
            "value_forecasting": "预测",
            "value_classification": "分类",
            "value_anomaly_detection": "异常检测",
            "value_cohort_analysis": "队列分析",
            "value_segmentation": "细分",
            "value_sentiment_analysis": "情感分析",
            "value_trend_analysis": "趋势分析",
            "value_prescriptive_analytics": "规范性分析",
            "value_root_cause_analysis": "根因分析",
            "value_optimization": "优化",
            "value_recommendation": "推荐",
            "value_time_series_analysis": "时间序列分析",
            "value_predictive_analytics": "预测分析",
            "value_descriptive_analytics": "描述性分析",
        },
        "Japanese": {
            "main_title": "データブリックス ジーニー コード を活用したユースケースジェネレーター",
            "intro": "このノートブックには、スキーマに基づいてAIが生成したユースケースが含まれています。以下は、ビジネスドメイン別に生成されたシナリオの概要です。",
            "domain": "ビジネスドメイン",
            "total": "ユースケース総数",
            "summaries": "ユースケース概要",
            "sum_id": "ID",
            "sum_name": "名前",
            "sum_value": "ビジネス価値",
            "sum_outcome": "期待される成果",
            "warning_header": "警告",
            "warning_body": "このノートブックを実行しないでください。デモンストレーションとカタログ作成のみを目的としています。",
            "disclaimer": "このコンテンツはAIによって生成されたものであり、本番環境で使用する前に資格のあるエンジニアによるレビューが必要です。Databricksはこのコンテンツの使用に起因するいかなる問題についても責任を負いません。",
            "detailed_scenarios": "ユースケースの詳細",
            "aspect": "側面",
            "description": "説明",
            "aspect_domain": "ビジネスドメイン",
            "type": "タイプ",
            "analytics_technique": "分析技術",
            "primary_table": "主要テーブル",
            "quality": "品質",
            "priority": "優先度",
            "usecase": "ユースケース",
            "description_label": "説明",
            "value_type_problem": "問題",
            "value_type_risk": "リスク",
            "value_type_opportunity": "機会",
            "value_type_improvement": "改善",
            "value_priority_ultra_high": "極めて高い",
            "value_priority_very_high": "非常に高い",
            "value_priority_high": "高い",
            "value_priority_medium": "中程度",
            "value_priority_low": "低い",
            "value_priority_very_low": "非常に低い",
            "value_priority_ultra_low": "極めて低い",
            "statement": "ステートメント",
            "solution": "ソリューション",
            "aspect_beneficiary": "受益者",
            "beneficiary": "受益者",
            "aspect_sponsor": "スポンサー",
            "sponsor": "スポンサー",
            "business_priority_alignment": "ビジネス優先度整合性",
            "strategic_goals_alignment": "戦略目標との整合性",
            "subdomain": "サブドメイン",
            "aspect_value": "ビジネス価値",
            "business_value": "ビジネス価値",
            "aspect_tables": "関連テーブル",
            "quality_reasons": "品質理由",
            "aspect_ai_function": "AI機能",
            "aspect_analytics_technique": "分析技術",
            "aspect_primary_table": "主要テーブル",
            "aspect_quality": "品質",
            "aspect_priority": "優先度",
            "strategic_alignment": "戦略的整合性",
            "return_on_investment": "投資収益率",
            "reusability": "再利用性",
            "time_to_value": "価値実現までの時間",
            "data_availability": "データ可用性",
            "data_accessibility": "データアクセシビリティ",
            "architecture_fitness": "アーキテクチャ適合性",
            "team_skills": "チームスキル",
            "domain_knowledge": "ドメイン知識",
            "people_allocation": "人員配置",
            "budget_allocation": "予算配分",
            "time_to_production": "本番化までの時間",
            "value_score": "価値スコア",
            "feasibility_score": "実現可能性スコア",
            "priority_score": "優先度スコア",
            "inspire_score": "Inspireスコア",
            "quality_score_header": "品質スコア",
            "pdf_title": "データブリックス ジーニー コード を活用した戦略的AIユースケース",
            "pdf_for": "対象",
            "pdf_exec_summary": "エグゼクティブサマリー",
            "pdf_toc_title": "ユースケースドメイン",
            "pdf_detailed_view": "詳細ユースケースカタログ",
            "pdf_disclaimer_title": "免責事項",
            "pdf_fallback_summary_p1": "本ドキュメントは{business_name}向けに特定された{total_cases}件の高価値分析ユースケースを概説しています。",
            "pdf_fallback_summary_p2": "以下のページでは、ビジネスドメイン別に分類されたこれらの機会の詳細な分析を提供します。",
            "pptx_main_title": "データブリックス ジーニー コード を活用した戦略的AIユースケース",
            "pptx_for": "対象",
            "pptx_disclaimer_title": "免責事項",
            "pptx_domain_suffix": "ユースケース",
            "example_results": "サンプル結果",
            "error_no_results": "結果を生成できませんでした。ノートブック: {notebook_name} とユースケース {use_case_id} を確認してください",
            "input_data_original": "入力データ（元の値）",
            "ai_generated_output": "AI生成結果",
            "column": "列",
            "value": "値",
            "executive_summary_not_available": "エグゼクティブサマリーは利用できません。",
            "domain_summary_not_available": "ドメインサマリーは利用できません。",
            "summary_not_available": "サマリーは利用できません。",
            "value_general_improvement": "一般的な改善",
            "value_reduce_cost": "コスト削減",
            "value_increase_revenue": "収益増加",
            "value_boost_productivity": "生産性向上",
            "value_mitigate_risk": "リスク軽減",
            "value_protect_revenue": "収益保護",
            "value_align_to_regulations": "規制遵守",
            "value_improve_customer_experience": "顧客体験の改善",
            "value_enable_data_driven_decisions": "データ駆動型意思決定の実現",
            "value_optimize_operations": "業務最適化",
            "value_empower_talent": "人材育成",
            "value_enhance_experience": "体験向上",
            "value_drive_innovation": "イノベーション推進",
            "value_achieve_esg": "ESG達成",
            "value_execute_strategy": "戦略実行",
            "value_forecasting": "予測",
            "value_classification": "分類",
            "value_anomaly_detection": "異常検知",
            "value_cohort_analysis": "コホート分析",
            "value_segmentation": "セグメンテーション",
            "value_sentiment_analysis": "感情分析",
            "value_trend_analysis": "トレンド分析",
            "value_prescriptive_analytics": "処方的分析",
            "value_root_cause_analysis": "根本原因分析",
            "value_optimization": "最適化",
            "value_recommendation": "レコメンデーション",
            "value_time_series_analysis": "時系列分析",
            "value_predictive_analytics": "予測分析",
            "value_descriptive_analytics": "記述的分析",
        },
        "Korean": {
            "main_title": "데이터브릭스 지니 코드 기반의 유스케이스 생성기",
            "intro": "이 노트북에는 스키마를 기반으로 AI가 생성한 유스케이스가 포함되어 있습니다. 아래는 비즈니스 도메인별 생성된 시나리오 요약입니다.",
            "domain": "비즈니스 도메인",
            "total": "총 유스케이스",
            "summaries": "유스케이스 요약",
            "sum_id": "ID",
            "sum_name": "이름",
            "sum_value": "비즈니스 가치",
            "sum_outcome": "예상 결과",
            "warning_header": "경고",
            "warning_body": "이 노트북을 실행하지 마십시오. 데모 및 카탈로그 작성 목적으로만 사용됩니다.",
            "disclaimer": "이 콘텐츠는 AI가 생성한 것이며 프로덕션에 사용하기 전에 자격을 갖춘 엔지니어의 검토가 필요합니다. Databricks는 이 콘텐츠 사용으로 인해 발생하는 문제에 대해 책임을 지지 않습니다.",
            "detailed_scenarios": "유스케이스 세부정보",
            "aspect": "측면",
            "description": "설명",
            "aspect_domain": "비즈니스 도메인",
            "type": "유형",
            "analytics_technique": "분석 기법",
            "primary_table": "주요 테이블",
            "quality": "품질",
            "priority": "우선순위",
            "usecase": "유스케이스",
            "description_label": "설명",
            "value_type_problem": "문제",
            "value_type_risk": "위험",
            "value_type_opportunity": "기회",
            "value_type_improvement": "개선",
            "value_priority_ultra_high": "초고",
            "value_priority_very_high": "매우 높음",
            "value_priority_high": "높음",
            "value_priority_medium": "보통",
            "value_priority_low": "낮음",
            "value_priority_very_low": "매우 낮음",
            "value_priority_ultra_low": "초저",
            "statement": "설명",
            "solution": "솔루션",
            "aspect_beneficiary": "수혜자",
            "beneficiary": "수혜자",
            "aspect_sponsor": "후원자",
            "sponsor": "후원자",
            "business_priority_alignment": "비즈니스 우선순위 정렬",
            "strategic_goals_alignment": "전략 목표 정렬",
            "subdomain": "하위 도메인",
            "aspect_value": "비즈니스 가치",
            "business_value": "비즈니스 가치",
            "aspect_tables": "관련 테이블",
            "quality_reasons": "품질 사유",
            "aspect_ai_function": "AI 기능",
            "aspect_analytics_technique": "분석 기법",
            "aspect_primary_table": "주요 테이블",
            "aspect_quality": "품질",
            "aspect_priority": "우선순위",
            "strategic_alignment": "전략적 정렬",
            "return_on_investment": "투자 수익률",
            "reusability": "재사용성",
            "time_to_value": "가치 실현 시간",
            "data_availability": "데이터 가용성",
            "data_accessibility": "데이터 접근성",
            "architecture_fitness": "아키텍처 적합성",
            "team_skills": "팀 역량",
            "domain_knowledge": "도메인 지식",
            "people_allocation": "인력 배치",
            "budget_allocation": "예산 배분",
            "time_to_production": "프로덕션까지 시간",
            "value_score": "가치 점수",
            "feasibility_score": "실현 가능성 점수",
            "priority_score": "우선순위 점수",
            "inspire_score": "Inspire 점수",
            "quality_score_header": "품질 점수",
            "pdf_title": "데이터브릭스 지니 코드 기반의 전략적 AI 유스케이스",
            "pdf_for": "대상",
            "pdf_exec_summary": "경영진 요약",
            "pdf_toc_title": "유스케이스 도메인",
            "pdf_detailed_view": "상세 유스케이스 카탈로그",
            "pdf_disclaimer_title": "면책조항",
            "pdf_fallback_summary_p1": "이 문서는 {business_name}을 위해 식별된 {total_cases}개의 고가치 분석 유스케이스를 설명합니다.",
            "pdf_fallback_summary_p2": "다음 페이지에서는 비즈니스 도메인별로 분류된 이러한 기회의 상세 분석을 제공합니다.",
            "pptx_main_title": "데이터브릭스 지니 코드 기반의 전략적 AI 유스케이스",
            "pptx_for": "대상",
            "pptx_disclaimer_title": "면책조항",
            "pptx_domain_suffix": "유스케이스",
            "example_results": "예시 결과",
            "error_no_results": "결과를 생성할 수 없습니다. 노트북: {notebook_name} 및 유스케이스 {use_case_id}를 확인하세요",
            "input_data_original": "입력 데이터 (원본 값)",
            "ai_generated_output": "AI 생성 결과",
            "column": "열",
            "value": "값",
            "executive_summary_not_available": "경영진 요약을 사용할 수 없습니다.",
            "domain_summary_not_available": "도메인 요약을 사용할 수 없습니다.",
            "summary_not_available": "요약을 사용할 수 없습니다.",
            "value_general_improvement": "일반 개선",
            "value_reduce_cost": "비용 절감",
            "value_increase_revenue": "수익 증대",
            "value_boost_productivity": "생산성 향상",
            "value_mitigate_risk": "위험 완화",
            "value_protect_revenue": "수익 보호",
            "value_align_to_regulations": "규정 준수",
            "value_improve_customer_experience": "고객 경험 개선",
            "value_enable_data_driven_decisions": "데이터 기반 의사결정 지원",
            "value_optimize_operations": "운영 최적화",
            "value_empower_talent": "인재 역량 강화",
            "value_enhance_experience": "경험 향상",
            "value_drive_innovation": "혁신 추진",
            "value_achieve_esg": "ESG 달성",
            "value_execute_strategy": "전략 실행",
            "value_forecasting": "예측",
            "value_classification": "분류",
            "value_anomaly_detection": "이상 탐지",
            "value_cohort_analysis": "코호트 분석",
            "value_segmentation": "세분화",
            "value_sentiment_analysis": "감정 분석",
            "value_trend_analysis": "추세 분석",
            "value_prescriptive_analytics": "처방적 분석",
            "value_root_cause_analysis": "근본 원인 분석",
            "value_optimization": "최적화",
            "value_recommendation": "추천",
            "value_time_series_analysis": "시계열 분석",
            "value_predictive_analytics": "예측 분석",
            "value_descriptive_analytics": "기술적 분석",
        },
        "Hindi": {
            "main_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित उपयोग केस जनरेटर",
            "intro": "इस नोटबुक में आपके स्कीमा के आधार पर AI-जनित उपयोग केस शामिल हैं। नीचे व्यापार डोमेन द्वारा जनित परिदृश्यों का सारांश है।",
            "domain": "व्यापार डोमेन",
            "total": "कुल उपयोग केस",
            "summaries": "उपयोग केस सारांश",
            "sum_id": "ID",
            "sum_name": "नाम",
            "sum_value": "व्यापारिक मूल्य",
            "sum_outcome": "अपेक्षित परिणाम",
            "warning_header": "चेतावनी",
            "warning_body": "इस नोटबुक को न चलाएं। यह केवल प्रदर्शन और कैटलॉगिंग उद्देश्यों के लिए है।",
            "disclaimer": "यह सामग्री AI-जनित है और उत्पादन में उपयोग करने से पहले एक योग्य इंजीनियर द्वारा समीक्षा की जानी चाहिए। Databricks इस सामग्री के उपयोग से उत्पन्न किसी भी समस्या के लिए उत्तरदायी नहीं है।",
            "detailed_scenarios": "उपयोग केस विवरण",
            "aspect": "पहलू",
            "description": "विवरण",
            "aspect_domain": "व्यापार डोमेन",
            "type": "प्रकार",
            "analytics_technique": "विश्लेषण तकनीक",
            "primary_table": "प्राथमिक तालिका",
            "quality": "गुणवत्ता",
            "priority": "प्राथमिकता",
            "usecase": "उपयोग केस",
            "description_label": "विवरण",
            "value_type_problem": "समस्या",
            "value_type_risk": "जोखिम",
            "value_type_opportunity": "अवसर",
            "value_type_improvement": "सुधार",
            "value_priority_ultra_high": "अत्यधिक उच्च",
            "value_priority_very_high": "बहुत उच्च",
            "value_priority_high": "उच्च",
            "value_priority_medium": "मध्यम",
            "value_priority_low": "कम",
            "value_priority_very_low": "बहुत कम",
            "value_priority_ultra_low": "अत्यधिक कम",
            "statement": "कथन",
            "solution": "समाधान",
            "aspect_beneficiary": "लाभार्थी",
            "beneficiary": "लाभार्थी",
            "aspect_sponsor": "प्रायोजक",
            "sponsor": "प्रायोजक",
            "business_priority_alignment": "व्यापार प्राथमिकता संरेखण",
            "strategic_goals_alignment": "रणनीतिक लक्ष्य संरेखण",
            "subdomain": "उप-डोमेन",
            "aspect_value": "व्यापारिक मूल्य",
            "business_value": "व्यापारिक मूल्य",
            "aspect_tables": "शामिल तालिकाएं",
            "quality_reasons": "गुणवत्ता कारण",
            "aspect_ai_function": "AI फ़ंक्शन",
            "aspect_analytics_technique": "विश्लेषण तकनीक",
            "aspect_primary_table": "प्राथमिक तालिका",
            "aspect_quality": "गुणवत्ता",
            "aspect_priority": "प्राथमिकता",
            "strategic_alignment": "रणनीतिक संरेखण",
            "return_on_investment": "निवेश पर प्रतिफल",
            "reusability": "पुन: प्रयोज्यता",
            "time_to_value": "मूल्य तक समय",
            "data_availability": "डेटा उपलब्धता",
            "data_accessibility": "डेटा पहुंच",
            "architecture_fitness": "आर्किटेक्चर उपयुक्तता",
            "team_skills": "टीम कौशल",
            "domain_knowledge": "डोमेन ज्ञान",
            "people_allocation": "लोग आवंटन",
            "budget_allocation": "बजट आवंटन",
            "time_to_production": "उत्पादन तक समय",
            "value_score": "मूल्य स्कोर",
            "feasibility_score": "व्यवहार्यता स्कोर",
            "priority_score": "प्राथमिकता स्कोर",
            "inspire_score": "Inspire स्कोर",
            "quality_score_header": "गुणवत्ता स्कोर",
            "pdf_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित रणनीतिक AI उपयोग केस",
            "pdf_for": "के लिए",
            "pdf_exec_summary": "कार्यकारी सारांश",
            "pdf_toc_title": "उपयोग केस डोमेन",
            "pdf_detailed_view": "विस्तृत उपयोग केस कैटलॉग",
            "pdf_disclaimer_title": "अस्वीकरण",
            "pdf_fallback_summary_p1": "यह दस्तावेज़ {business_name} के लिए पहचाने गए {total_cases} उच्च-मूल्य विश्लेषणात्मक उपयोग केस का वर्णन करता है।",
            "pdf_fallback_summary_p2": "निम्नलिखित पृष्ठ व्यापार डोमेन द्वारा वर्गीकृत इन अवसरों का विस्तृत विश्लेषण प्रदान करते हैं।",
            "pptx_main_title": "डेटाब्रिक्स जीनी कोड द्वारा संचालित रणनीतिक AI उपयोग केस",
            "pptx_for": "के लिए",
            "pptx_disclaimer_title": "अस्वीकरण",
            "pptx_domain_suffix": "उपयोग केस",
            "example_results": "उदाहरण परिणाम",
            "error_no_results": "परिणाम उत्पन्न नहीं हो सके। नोटबुक: {notebook_name} और उपयोग केस {use_case_id} जांचें",
            "input_data_original": "इनपुट डेटा (मूल मान)",
            "ai_generated_output": "AI-जनित परिणाम",
            "column": "कॉलम",
            "value": "मूल्य",
            "executive_summary_not_available": "कार्यकारी सारांश उपलब्ध नहीं है।",
            "domain_summary_not_available": "डोमेन सारांश उपलब्ध नहीं है।",
            "summary_not_available": "सारांश उपलब्ध नहीं है।",
            "value_general_improvement": "सामान्य सुधार",
            "value_reduce_cost": "लागत कम करें",
            "value_increase_revenue": "राजस्व बढ़ाएं",
            "value_boost_productivity": "उत्पादकता बढ़ाएं",
            "value_mitigate_risk": "जोखिम कम करें",
            "value_protect_revenue": "राजस्व की रक्षा करें",
            "value_align_to_regulations": "नियमों का पालन करें",
            "value_improve_customer_experience": "ग्राहक अनुभव सुधारें",
            "value_enable_data_driven_decisions": "डेटा-आधारित निर्णय सक्षम करें",
            "value_optimize_operations": "संचालन अनुकूलित करें",
            "value_empower_talent": "प्रतिभा को सशक्त बनाएं",
            "value_enhance_experience": "अनुभव बढ़ाएं",
            "value_drive_innovation": "नवाचार को बढ़ावा दें",
            "value_achieve_esg": "ESG प्राप्त करें",
            "value_execute_strategy": "रणनीति निष्पादित करें",
            "value_forecasting": "पूर्वानुमान",
            "value_classification": "वर्गीकरण",
            "value_anomaly_detection": "विसंगति पहचान",
            "value_cohort_analysis": "समूह विश्लेषण",
            "value_segmentation": "विभाजन",
            "value_sentiment_analysis": "भावना विश्लेषण",
            "value_trend_analysis": "रुझान विश्लेषण",
            "value_prescriptive_analytics": "निर्देशात्मक विश्लेषण",
            "value_root_cause_analysis": "मूल कारण विश्लेषण",
            "value_optimization": "अनुकूलन",
            "value_recommendation": "अनुशंसा",
            "value_time_series_analysis": "समय श्रृंखला विश्लेषण",
            "value_predictive_analytics": "भविष्यवाणी विश्लेषण",
            "value_descriptive_analytics": "वर्णनात्मक विश्लेषण",
        },
        "Russian": {
            "main_title": "Генератор бизнес-кейсов на базе Датабрикс Джини Код",
            "intro": "Эта записная книжка содержит бизнес-кейсы, созданные ИИ на основе ваших схем. Ниже приведено резюме созданных сценариев по бизнес-доменам.",
            "domain": "Бизнес-домен",
            "total": "Всего бизнес-кейсов",
            "summaries": "Резюме бизнес-кейсов",
            "sum_id": "ID",
            "sum_name": "Название",
            "sum_value": "Бизнес-ценность",
            "sum_outcome": "Ожидаемый результат",
            "warning_header": "ПРЕДУПРЕЖДЕНИЕ",
            "warning_body": "Не запускайте эту записную книжку. Она предназначена только для демонстрации и каталогизации.",
            "disclaimer": "Этот контент создан ИИ и должен быть проверен квалифицированным инженером перед использованием в рабочей среде. Databricks не несёт ответственности за любые проблемы, возникающие в результате использования этого контента.",
            "detailed_scenarios": "Детали бизнес-кейсов",
            "aspect": "Аспект",
            "description": "Описание",
            "aspect_domain": "Бизнес-домен",
            "type": "Тип",
            "analytics_technique": "Аналитическая техника",
            "primary_table": "Основная таблица",
            "quality": "Качество",
            "priority": "Приоритет",
            "usecase": "Бизнес-кейс",
            "description_label": "Описание",
            "value_type_problem": "Проблема",
            "value_type_risk": "Риск",
            "value_type_opportunity": "Возможность",
            "value_type_improvement": "Улучшение",
            "value_priority_ultra_high": "Крайне высокий",
            "value_priority_very_high": "Очень высокий",
            "value_priority_high": "Высокий",
            "value_priority_medium": "Средний",
            "value_priority_low": "Низкий",
            "value_priority_very_low": "Очень низкий",
            "value_priority_ultra_low": "Крайне низкий",
            "statement": "Описание",
            "solution": "Решение",
            "aspect_beneficiary": "Выгодоприобретатель",
            "beneficiary": "Выгодоприобретатель",
            "aspect_sponsor": "Спонсор",
            "sponsor": "Спонсор",
            "business_priority_alignment": "Соответствие бизнес-приоритетам",
            "strategic_goals_alignment": "Соответствие стратегическим целям",
            "subdomain": "Поддомен",
            "aspect_value": "Бизнес-ценность",
            "business_value": "Бизнес-ценность",
            "aspect_tables": "Связанные таблицы",
            "quality_reasons": "Причины качества",
            "aspect_ai_function": "Функция ИИ",
            "aspect_analytics_technique": "Аналитическая техника",
            "aspect_primary_table": "Основная таблица",
            "aspect_quality": "Качество",
            "aspect_priority": "Приоритет",
            "strategic_alignment": "Стратегическое соответствие",
            "return_on_investment": "Рентабельность инвестиций",
            "reusability": "Возможность повторного использования",
            "time_to_value": "Время до получения ценности",
            "data_availability": "Доступность данных",
            "data_accessibility": "Доступ к данным",
            "architecture_fitness": "Соответствие архитектуре",
            "team_skills": "Навыки команды",
            "domain_knowledge": "Знание предметной области",
            "people_allocation": "Распределение персонала",
            "budget_allocation": "Распределение бюджета",
            "time_to_production": "Время до внедрения",
            "value_score": "Оценка ценности",
            "feasibility_score": "Оценка реализуемости",
            "priority_score": "Оценка приоритета",
            "inspire_score": "Оценка Inspire",
            "quality_score_header": "Оценка качества",
            "pdf_title": "Стратегические ИИ бизнес-кейсы на базе Датабрикс Джини Код",
            "pdf_for": "Для",
            "pdf_exec_summary": "Резюме для руководства",
            "pdf_toc_title": "Домены бизнес-кейсов",
            "pdf_detailed_view": "Подробный каталог бизнес-кейсов",
            "pdf_disclaimer_title": "Отказ от ответственности",
            "pdf_fallback_summary_p1": "Этот документ описывает {total_cases} высокоценных аналитических бизнес-кейсов, выявленных для {business_name}.",
            "pdf_fallback_summary_p2": "На следующих страницах представлен подробный анализ этих возможностей, классифицированных по бизнес-доменам.",
            "pptx_main_title": "Стратегические ИИ бизнес-кейсы на базе Датабрикс Джини Код",
            "pptx_for": "Для",
            "pptx_disclaimer_title": "Отказ от ответственности",
            "pptx_domain_suffix": "Бизнес-кейсы",
            "example_results": "Примеры результатов",
            "error_no_results": "Не удалось создать результаты. Проверьте записную книжку: {notebook_name} и бизнес-кейс {use_case_id}",
            "input_data_original": "Входные данные (исходные значения)",
            "ai_generated_output": "Результаты, созданные ИИ",
            "column": "Столбец",
            "value": "Значение",
            "executive_summary_not_available": "Резюме для руководства недоступно.",
            "domain_summary_not_available": "Резюме домена недоступно.",
            "summary_not_available": "Резюме недоступно.",
            "value_general_improvement": "Общее Улучшение",
            "value_reduce_cost": "Сократить Затраты",
            "value_increase_revenue": "Увеличить Доход",
            "value_boost_productivity": "Повысить Производительность",
            "value_mitigate_risk": "Снизить Риски",
            "value_protect_revenue": "Защитить Доход",
            "value_align_to_regulations": "Соответствовать Нормативам",
            "value_improve_customer_experience": "Улучшить Клиентский Опыт",
            "value_enable_data_driven_decisions": "Обеспечить Решения на Основе Данных",
            "value_optimize_operations": "Оптимизировать Операции",
            "value_empower_talent": "Развивать Таланты",
            "value_enhance_experience": "Улучшить Опыт",
            "value_drive_innovation": "Стимулировать Инновации",
            "value_achieve_esg": "Достичь ESG",
            "value_execute_strategy": "Реализовать Стратегию",
            "value_forecasting": "Прогнозирование",
            "value_classification": "Классификация",
            "value_anomaly_detection": "Обнаружение Аномалий",
            "value_cohort_analysis": "Когортный Анализ",
            "value_segmentation": "Сегментация",
            "value_sentiment_analysis": "Анализ Настроений",
            "value_trend_analysis": "Анализ Трендов",
            "value_prescriptive_analytics": "Предписывающая Аналитика",
            "value_root_cause_analysis": "Анализ Первопричин",
            "value_optimization": "Оптимизация",
            "value_recommendation": "Рекомендация",
            "value_time_series_analysis": "Анализ Временных Рядов",
            "value_predictive_analytics": "Прогнозная Аналитика",
            "value_descriptive_analytics": "Описательная Аналитика",
        },
        "Swedish": {
            "main_title": "Användningsfallsgenerator Driven av Databricks Genie Code",
            "intro": "Denna notebook innehåller AI-genererade användningsfall baserade på dina scheman. Nedan finns en sammanfattning av genererade scenarier per affärsdomän.",
            "domain": "Affärsdomän",
            "total": "Totalt antal användningsfall",
            "summaries": "Sammanfattningar av användningsfall",
            "sum_id": "ID",
            "sum_name": "Namn",
            "sum_value": "Affärsvärde",
            "sum_outcome": "Förväntat resultat",
            "warning_header": "VARNING",
            "warning_body": "Kör inte denna notebook. Den är avsedd enbart för demonstration och katalogisering.",
            "disclaimer": "Detta innehåll är AI-genererat och bör granskas av en kvalificerad ingenjör innan det används i produktion. Databricks ansvarar inte för eventuella problem som uppstår vid användning av detta innehåll.",
            "detailed_scenarios": "Detaljer om användningsfall",
            "aspect": "Aspekt",
            "description": "Beskrivning",
            "aspect_domain": "Affärsdomän",
            "type": "Typ",
            "analytics_technique": "Analysteknik",
            "primary_table": "Primärtabell",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Användningsfall",
            "description_label": "Beskrivning",
            "value_type_problem": "Problem",
            "value_type_risk": "Risk",
            "value_type_opportunity": "Möjlighet",
            "value_type_improvement": "Förbättring",
            "value_priority_ultra_high": "Extremt hög",
            "value_priority_very_high": "Mycket hög",
            "value_priority_high": "Hög",
            "value_priority_medium": "Medel",
            "value_priority_low": "Låg",
            "value_priority_very_low": "Mycket låg",
            "value_priority_ultra_low": "Extremt låg",
            "statement": "Uttalande",
            "solution": "Lösning",
            "aspect_beneficiary": "Förmånstagare",
            "beneficiary": "Förmånstagare",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Affärsprioritet anpassning",
            "strategic_goals_alignment": "Strategisk målanpassning",
            "subdomain": "Underdomän",
            "aspect_value": "Affärsvärde",
            "business_value": "Affärsvärde",
            "aspect_tables": "Involverade tabeller",
            "quality_reasons": "Kvalitetsskäl",
            "aspect_ai_function": "AI-funktion",
            "aspect_analytics_technique": "Analysteknik",
            "aspect_primary_table": "Primärtabell",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk anpassning",
            "return_on_investment": "Avkastning på investering",
            "reusability": "Återanvändbarhet",
            "time_to_value": "Tid till värde",
            "data_availability": "Datatillgänglighet",
            "data_accessibility": "Dataåtkomlighet",
            "architecture_fitness": "Arkitekturlämplighet",
            "team_skills": "Teamkompetens",
            "domain_knowledge": "Domänkunskap",
            "people_allocation": "Personalallokering",
            "budget_allocation": "Budgetallokering",
            "time_to_production": "Tid till produktion",
            "value_score": "Värdepoäng",
            "feasibility_score": "Genomförbarhetspoäng",
            "priority_score": "Prioritetspoäng",
            "inspire_score": "Inspire-poäng",
            "quality_score_header": "Kvalitetspoäng",
            "pdf_title": "Strategiska AI-användningsfall drivna av Databricks Genie Code",
            "pdf_for": "För",
            "pdf_exec_summary": "Sammanfattning",
            "pdf_toc_title": "Användningsfallsdomäner",
            "pdf_detailed_view": "Detaljerad användningsfallskatalog",
            "pdf_disclaimer_title": "Ansvarsfriskrivning",
            "pdf_fallback_summary_p1": "Detta dokument beskriver {total_cases} högvärdiga analytiska användningsfall identifierade för {business_name}.",
            "pdf_fallback_summary_p2": "Följande sidor ger en detaljerad uppdelning av dessa möjligheter, kategoriserade efter affärsdomän.",
            "pptx_main_title": "Strategiska AI-användningsfall drivna av Databricks Genie Code",
            "pptx_for": "För",
            "pptx_disclaimer_title": "Ansvarsfriskrivning",
            "pptx_domain_suffix": "Användningsfall",
            "example_results": "Exempelresultat",
            "error_no_results": "Kunde inte generera resultat. Kontrollera notebook: {notebook_name} och användningsfall {use_case_id}",
            "input_data_original": "Indata (originalvärden)",
            "ai_generated_output": "AI-genererade resultat",
            "column": "Kolumn",
            "value": "Värde",
            "executive_summary_not_available": "Sammanfattning ej tillgänglig.",
            "domain_summary_not_available": "Domänsammanfattning ej tillgänglig.",
            "summary_not_available": "Sammanfattning ej tillgänglig.",
            "value_general_improvement": "Allmän förbättring",
            "value_reduce_cost": "Minska kostnader",
            "value_increase_revenue": "Öka intäkter",
            "value_boost_productivity": "Öka produktivitet",
            "value_mitigate_risk": "Minska risker",
            "value_protect_revenue": "Skydda intäkter",
            "value_align_to_regulations": "Efterleva regelverk",
            "value_improve_customer_experience": "Förbättra kundupplevelse",
            "value_enable_data_driven_decisions": "Möjliggöra datadrivna beslut",
            "value_optimize_operations": "Optimera verksamheten",
            "value_empower_talent": "Stärka talanger",
            "value_enhance_experience": "Förbättra upplevelsen",
            "value_drive_innovation": "Driva innovation",
            "value_achieve_esg": "Uppnå ESG",
            "value_execute_strategy": "Genomföra strategi",
            "value_forecasting": "Prognos",
            "value_classification": "Klassificering",
            "value_anomaly_detection": "Anomalidetektering",
            "value_cohort_analysis": "Kohortanalys",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalys",
            "value_trend_analysis": "Trendanalys",
            "value_prescriptive_analytics": "Preskriptiv analys",
            "value_root_cause_analysis": "Rotorsaksanalys",
            "value_optimization": "Optimering",
            "value_recommendation": "Rekommendation",
            "value_time_series_analysis": "Tidsserieanalys",
            "value_predictive_analytics": "Prediktiv analys",
            "value_descriptive_analytics": "Deskriptiv analys",
        },
        "Danish": {
            "main_title": "Anvendelsescase-generator drevet af Databricks Genie Code",
            "intro": "Denne notebook indeholder AI-genererede anvendelsescases baseret på dine skemaer. Nedenfor er en oversigt over genererede scenarier pr. forretningsdomæne.",
            "domain": "Forretningsdomæne",
            "total": "Samlet antal anvendelsescases",
            "summaries": "Oversigter over anvendelsescases",
            "sum_id": "ID",
            "sum_name": "Navn",
            "sum_value": "Forretningsværdi",
            "sum_outcome": "Forventet resultat",
            "warning_header": "ADVARSEL",
            "warning_body": "Kør ikke denne notebook. Den er udelukkende beregnet til demonstration og katalogisering.",
            "disclaimer": "Dette indhold er AI-genereret og bør gennemgås af en kvalificeret ingeniør, før det bruges i produktion. Databricks er ikke ansvarlig for problemer, der opstår ved brug af dette indhold.",
            "detailed_scenarios": "Detaljer om anvendelsescases",
            "aspect": "Aspekt",
            "description": "Beskrivelse",
            "aspect_domain": "Forretningsdomæne",
            "type": "Type",
            "analytics_technique": "Analyseteknik",
            "primary_table": "Primærtabel",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Anvendelsescase",
            "description_label": "Beskrivelse",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Mulighed",
            "value_type_improvement": "Forbedring",
            "value_priority_ultra_high": "Ekstremt høj",
            "value_priority_very_high": "Meget høj",
            "value_priority_high": "Høj",
            "value_priority_medium": "Middel",
            "value_priority_low": "Lav",
            "value_priority_very_low": "Meget lav",
            "value_priority_ultra_low": "Ekstremt lav",
            "statement": "Erklæring",
            "solution": "Løsning",
            "aspect_beneficiary": "Modtager",
            "beneficiary": "Modtager",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Forretningsprioritets tilpasning",
            "strategic_goals_alignment": "Strategisk måltilpasning",
            "subdomain": "Underdomæne",
            "aspect_value": "Forretningsværdi",
            "business_value": "Forretningsværdi",
            "aspect_tables": "Involverede tabeller",
            "quality_reasons": "Kvalitetsårsager",
            "aspect_ai_function": "AI-funktion",
            "aspect_analytics_technique": "Analyseteknik",
            "aspect_primary_table": "Primærtabel",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk tilpasning",
            "return_on_investment": "Investeringsafkast",
            "reusability": "Genanvendelighed",
            "time_to_value": "Tid til værdi",
            "data_availability": "Datatilgængelighed",
            "data_accessibility": "Dataadgang",
            "architecture_fitness": "Arkitekturegnethed",
            "team_skills": "Teamkompetencer",
            "domain_knowledge": "Domæneviden",
            "people_allocation": "Personaleallokering",
            "budget_allocation": "Budgetallokering",
            "time_to_production": "Tid til produktion",
            "value_score": "Værdiscore",
            "feasibility_score": "Gennemførlighedsscore",
            "priority_score": "Prioritetsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kvalitetsscore",
            "pdf_title": "Strategiske AI-anvendelsescases drevet af Databricks Genie Code",
            "pdf_for": "For",
            "pdf_exec_summary": "Resumé",
            "pdf_toc_title": "Anvendelsescasedomæner",
            "pdf_detailed_view": "Detaljeret anvendelsescasekatalog",
            "pdf_disclaimer_title": "Ansvarsfraskrivelse",
            "pdf_fallback_summary_p1": "Dette dokument beskriver {total_cases} højværdi analytiske anvendelsescases identificeret for {business_name}.",
            "pdf_fallback_summary_p2": "De følgende sider giver en detaljeret opdeling af disse muligheder, kategoriseret efter forretningsdomæne.",
            "pptx_main_title": "Strategiske AI-anvendelsescases drevet af Databricks Genie Code",
            "pptx_for": "For",
            "pptx_disclaimer_title": "Ansvarsfraskrivelse",
            "pptx_domain_suffix": "Anvendelsescases",
            "example_results": "Eksempelresultater",
            "error_no_results": "Kunne ikke generere resultater. Tjek notebook: {notebook_name} og anvendelsescase {use_case_id}",
            "input_data_original": "Inputdata (originalværdier)",
            "ai_generated_output": "AI-genererede resultater",
            "column": "Kolonne",
            "value": "Værdi",
            "executive_summary_not_available": "Resumé ikke tilgængeligt.",
            "domain_summary_not_available": "Domæneoversigt ikke tilgængelig.",
            "summary_not_available": "Oversigt ikke tilgængelig.",
            "value_general_improvement": "Generel forbedring",
            "value_reduce_cost": "Reducer omkostninger",
            "value_increase_revenue": "Øg omsætning",
            "value_boost_productivity": "Øg produktivitet",
            "value_mitigate_risk": "Mindsk risici",
            "value_protect_revenue": "Beskyt omsætning",
            "value_align_to_regulations": "Overhold regler",
            "value_improve_customer_experience": "Forbedr kundeoplevelse",
            "value_enable_data_driven_decisions": "Muliggør datadrevne beslutninger",
            "value_optimize_operations": "Optimer drift",
            "value_empower_talent": "Styrk talenter",
            "value_enhance_experience": "Forbedr oplevelsen",
            "value_drive_innovation": "Driv innovation",
            "value_achieve_esg": "Opnå ESG",
            "value_execute_strategy": "Udfør strategi",
            "value_forecasting": "Prognose",
            "value_classification": "Klassificering",
            "value_anomaly_detection": "Anomalidetektion",
            "value_cohort_analysis": "Kohorteanalyse",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Præskriptiv analyse",
            "value_root_cause_analysis": "Rodårsagsanalyse",
            "value_optimization": "Optimering",
            "value_recommendation": "Anbefaling",
            "value_time_series_analysis": "Tidsserieanalyse",
            "value_predictive_analytics": "Prædiktiv analyse",
            "value_descriptive_analytics": "Deskriptiv analyse",
        },
        "Norwegian": {
            "main_title": "Brukscasegenerator drevet av Databricks Genie Code",
            "intro": "Denne notatboken inneholder AI-genererte brukscaser basert på skjemaene dine. Nedenfor er en oppsummering av genererte scenarier etter forretningsdomene.",
            "domain": "Forretningsdomene",
            "total": "Totalt antall brukscaser",
            "summaries": "Oppsummeringer av brukscaser",
            "sum_id": "ID",
            "sum_name": "Navn",
            "sum_value": "Forretningsverdi",
            "sum_outcome": "Forventet resultat",
            "warning_header": "ADVARSEL",
            "warning_body": "Ikke kjør denne notatboken. Den er kun ment for demonstrasjon og katalogisering.",
            "disclaimer": "Dette innholdet er AI-generert og bør gjennomgås av en kvalifisert ingeniør før det brukes i produksjon. Databricks er ikke ansvarlig for problemer som oppstår ved bruk av dette innholdet.",
            "detailed_scenarios": "Detaljer om brukscaser",
            "aspect": "Aspekt",
            "description": "Beskrivelse",
            "aspect_domain": "Forretningsdomene",
            "type": "Type",
            "analytics_technique": "Analyseteknikk",
            "primary_table": "Primærtabell",
            "quality": "Kvalitet",
            "priority": "Prioritet",
            "usecase": "Brukscase",
            "description_label": "Beskrivelse",
            "value_type_problem": "Problem",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Mulighet",
            "value_type_improvement": "Forbedring",
            "value_priority_ultra_high": "Ekstremt høy",
            "value_priority_very_high": "Veldig høy",
            "value_priority_high": "Høy",
            "value_priority_medium": "Middels",
            "value_priority_low": "Lav",
            "value_priority_very_low": "Veldig lav",
            "value_priority_ultra_low": "Ekstremt lav",
            "statement": "Utsagn",
            "solution": "Løsning",
            "aspect_beneficiary": "Mottaker",
            "beneficiary": "Mottaker",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Forretningsprioritetstilpasning",
            "strategic_goals_alignment": "Strategisk måltilpasning",
            "subdomain": "Underdomene",
            "aspect_value": "Forretningsverdi",
            "business_value": "Forretningsverdi",
            "aspect_tables": "Involverte tabeller",
            "quality_reasons": "Kvalitetsårsaker",
            "aspect_ai_function": "AI-funksjon",
            "aspect_analytics_technique": "Analyseteknikk",
            "aspect_primary_table": "Primærtabell",
            "aspect_quality": "Kvalitet",
            "aspect_priority": "Prioritet",
            "strategic_alignment": "Strategisk tilpasning",
            "return_on_investment": "Avkastning på investering",
            "reusability": "Gjenbrukbarhet",
            "time_to_value": "Tid til verdi",
            "data_availability": "Datatilgjengelighet",
            "data_accessibility": "Datatilgang",
            "architecture_fitness": "Arkitekturegnethet",
            "team_skills": "Teamkompetanse",
            "domain_knowledge": "Domenekunnskap",
            "people_allocation": "Personalallokering",
            "budget_allocation": "Budsjettallokering",
            "time_to_production": "Tid til produksjon",
            "value_score": "Verdiscore",
            "feasibility_score": "Gjennomførbarhetsscore",
            "priority_score": "Prioritetsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kvalitetsscore",
            "pdf_title": "Strategiske AI-brukscaser drevet av Databricks Genie Code",
            "pdf_for": "For",
            "pdf_exec_summary": "Sammendrag",
            "pdf_toc_title": "Brukscasedomener",
            "pdf_detailed_view": "Detaljert brukscasekatalog",
            "pdf_disclaimer_title": "Ansvarsfraskrivelse",
            "pdf_fallback_summary_p1": "Dette dokumentet beskriver {total_cases} høyverdige analytiske brukscaser identifisert for {business_name}.",
            "pdf_fallback_summary_p2": "De følgende sidene gir en detaljert oversikt over disse mulighetene, kategorisert etter forretningsdomene.",
            "pptx_main_title": "Strategiske AI-brukscaser drevet av Databricks Genie Code",
            "pptx_for": "For",
            "pptx_disclaimer_title": "Ansvarsfraskrivelse",
            "pptx_domain_suffix": "Brukscaser",
            "example_results": "Eksempelresultater",
            "error_no_results": "Kunne ikke generere resultater. Sjekk notatbok: {notebook_name} og brukscase {use_case_id}",
            "input_data_original": "Inndata (originalverdier)",
            "ai_generated_output": "AI-genererte resultater",
            "column": "Kolonne",
            "value": "Verdi",
            "executive_summary_not_available": "Sammendrag ikke tilgjengelig.",
            "domain_summary_not_available": "Domeneoversikt ikke tilgjengelig.",
            "summary_not_available": "Oversikt ikke tilgjengelig.",
            "value_general_improvement": "Generell forbedring",
            "value_reduce_cost": "Reduser kostnader",
            "value_increase_revenue": "Øk inntekter",
            "value_boost_productivity": "Øk produktivitet",
            "value_mitigate_risk": "Reduser risiko",
            "value_protect_revenue": "Beskytt inntekter",
            "value_align_to_regulations": "Overhold regler",
            "value_improve_customer_experience": "Forbedre kundeopplevelse",
            "value_enable_data_driven_decisions": "Muliggjøre datadrevne beslutninger",
            "value_optimize_operations": "Optimere drift",
            "value_empower_talent": "Styrke talenter",
            "value_enhance_experience": "Forbedre opplevelsen",
            "value_drive_innovation": "Drive innovasjon",
            "value_achieve_esg": "Oppnå ESG",
            "value_execute_strategy": "Gjennomføre strategi",
            "value_forecasting": "Prognose",
            "value_classification": "Klassifisering",
            "value_anomaly_detection": "Anomalideteksjon",
            "value_cohort_analysis": "Kohortanalyse",
            "value_segmentation": "Segmentering",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Preskriptiv analyse",
            "value_root_cause_analysis": "Rotårsaksanalyse",
            "value_optimization": "Optimalisering",
            "value_recommendation": "Anbefaling",
            "value_time_series_analysis": "Tidsserieanalyse",
            "value_predictive_analytics": "Prediktiv analyse",
            "value_descriptive_analytics": "Deskriptiv analyse",
        },
        "Finnish": {
            "main_title": "Käyttötapausgeneraattori Databricks Genie Coden avulla",
            "intro": "Tämä muistikirja sisältää tekoälyn luomia käyttötapauksia skeemojesi perusteella. Alla on yhteenveto luoduista skenaarioista liiketoiminta-alueittain.",
            "domain": "Liiketoiminta-alue",
            "total": "Käyttötapauksia yhteensä",
            "summaries": "Käyttötapausten yhteenvedot",
            "sum_id": "ID",
            "sum_name": "Nimi",
            "sum_value": "Liiketoiminta-arvo",
            "sum_outcome": "Odotettu tulos",
            "warning_header": "VAROITUS",
            "warning_body": "Älä suorita tätä muistikirjaa. Se on tarkoitettu vain esittely- ja luettelointitarkoituksiin.",
            "disclaimer": "Tämä sisältö on tekoälyn luomaa ja pätevän insinöörin tulee tarkistaa se ennen tuotantokäyttöä. Databricks ei ole vastuussa mistään tämän sisällön käytöstä aiheutuvista ongelmista.",
            "detailed_scenarios": "Käyttötapausten yksityiskohdat",
            "aspect": "Näkökulma",
            "description": "Kuvaus",
            "aspect_domain": "Liiketoiminta-alue",
            "type": "Tyyppi",
            "analytics_technique": "Analytiikkatekniikka",
            "primary_table": "Ensisijainen taulu",
            "quality": "Laatu",
            "priority": "Prioriteetti",
            "usecase": "Käyttötapaus",
            "description_label": "Kuvaus",
            "value_type_problem": "Ongelma",
            "value_type_risk": "Riski",
            "value_type_opportunity": "Mahdollisuus",
            "value_type_improvement": "Parannus",
            "value_priority_ultra_high": "Erittäin korkea",
            "value_priority_very_high": "Hyvin korkea",
            "value_priority_high": "Korkea",
            "value_priority_medium": "Keskitaso",
            "value_priority_low": "Matala",
            "value_priority_very_low": "Hyvin matala",
            "value_priority_ultra_low": "Erittäin matala",
            "statement": "Lausunto",
            "solution": "Ratkaisu",
            "aspect_beneficiary": "Edunsaaja",
            "beneficiary": "Edunsaaja",
            "aspect_sponsor": "Sponsori",
            "sponsor": "Sponsori",
            "business_priority_alignment": "Liiketoimintaprioriteetin kohdistaminen",
            "strategic_goals_alignment": "Strategisten tavoitteiden kohdistaminen",
            "subdomain": "Alidomain",
            "aspect_value": "Liiketoiminta-arvo",
            "business_value": "Liiketoiminta-arvo",
            "aspect_tables": "Liittyvät taulut",
            "quality_reasons": "Laadun syyt",
            "aspect_ai_function": "Tekoälytoiminto",
            "aspect_analytics_technique": "Analytiikkatekniikka",
            "aspect_primary_table": "Ensisijainen taulu",
            "aspect_quality": "Laatu",
            "aspect_priority": "Prioriteetti",
            "strategic_alignment": "Strateginen kohdistaminen",
            "return_on_investment": "Sijoitetun pääoman tuotto",
            "reusability": "Uudelleenkäytettävyys",
            "time_to_value": "Aika arvon saavuttamiseen",
            "data_availability": "Datan saatavuus",
            "data_accessibility": "Datan saavutettavuus",
            "architecture_fitness": "Arkkitehtuurin soveltuvuus",
            "team_skills": "Tiimin osaaminen",
            "domain_knowledge": "Toimialaosaaminen",
            "people_allocation": "Henkilöstöallokointi",
            "budget_allocation": "Budjettiallokointi",
            "time_to_production": "Aika tuotantoon",
            "value_score": "Arvopisteet",
            "feasibility_score": "Toteutettavuuspisteet",
            "priority_score": "Prioriteettipisteet",
            "inspire_score": "Inspire-pisteet",
            "quality_score_header": "Laatupisteet",
            "pdf_title": "Strategiset tekoälykäyttötapaukset Databricks Genie Coden avulla",
            "pdf_for": "Kohde",
            "pdf_exec_summary": "Yhteenveto",
            "pdf_toc_title": "Käyttötapausdomainit",
            "pdf_detailed_view": "Yksityiskohtainen käyttötapausluettelo",
            "pdf_disclaimer_title": "Vastuuvapauslauseke",
            "pdf_fallback_summary_p1": "Tämä dokumentti kuvaa {total_cases} korkea-arvoista analyyttistä käyttötapausta, jotka on tunnistettu {business_name} toimintaan.",
            "pdf_fallback_summary_p2": "Seuraavat sivut tarjoavat yksityiskohtaisen erittelyn näistä mahdollisuuksista liiketoiminta-alueittain.",
            "pptx_main_title": "Strategiset tekoälykäyttötapaukset Databricks Genie Coden avulla",
            "pptx_for": "Kohde",
            "pptx_disclaimer_title": "Vastuuvapauslauseke",
            "pptx_domain_suffix": "Käyttötapaukset",
            "example_results": "Esimerkkitulokset",
            "error_no_results": "Tuloksia ei voitu luoda. Tarkista muistikirja: {notebook_name} ja käyttötapaus {use_case_id}",
            "input_data_original": "Syötetiedot (alkuperäiset arvot)",
            "ai_generated_output": "Tekoälyn tuottamat tulokset",
            "column": "Sarake",
            "value": "Arvo",
            "executive_summary_not_available": "Yhteenveto ei saatavilla.",
            "domain_summary_not_available": "Alueen yhteenveto ei saatavilla.",
            "summary_not_available": "Yhteenveto ei saatavilla.",
            "value_general_improvement": "Yleinen parannus",
            "value_reduce_cost": "Vähennä kustannuksia",
            "value_increase_revenue": "Kasvata liikevaihtoa",
            "value_boost_productivity": "Lisää tuottavuutta",
            "value_mitigate_risk": "Vähennä riskejä",
            "value_protect_revenue": "Suojaa liikevaihtoa",
            "value_align_to_regulations": "Noudata säädöksiä",
            "value_improve_customer_experience": "Paranna asiakaskokemusta",
            "value_enable_data_driven_decisions": "Mahdollista datapohjaiset päätökset",
            "value_optimize_operations": "Optimoi toimintaa",
            "value_empower_talent": "Vahvista osaajia",
            "value_enhance_experience": "Paranna kokemusta",
            "value_drive_innovation": "Edistä innovaatiota",
            "value_achieve_esg": "Saavuta ESG",
            "value_execute_strategy": "Toteuta strategia",
            "value_forecasting": "Ennustaminen",
            "value_classification": "Luokittelu",
            "value_anomaly_detection": "Poikkeamien tunnistus",
            "value_cohort_analysis": "Kohortti-analyysi",
            "value_segmentation": "Segmentointi",
            "value_sentiment_analysis": "Sentimenttianalyysi",
            "value_trend_analysis": "Trendianalyysi",
            "value_prescriptive_analytics": "Preskriptiivinen analytiikka",
            "value_root_cause_analysis": "Juurisyyanalyysi",
            "value_optimization": "Optimointi",
            "value_recommendation": "Suositus",
            "value_time_series_analysis": "Aikasarja-analyysi",
            "value_predictive_analytics": "Prediktiivinen analytiikka",
            "value_descriptive_analytics": "Deskriptiivinen analytiikka",
        },
        "Polish": {
            "main_title": "Generator przypadków użycia oparty na Databricks Genie Code",
            "intro": "Ten notatnik zawiera przypadki użycia wygenerowane przez AI na podstawie Twoich schematów. Poniżej znajduje się podsumowanie wygenerowanych scenariuszy według domeny biznesowej.",
            "domain": "Domena biznesowa",
            "total": "Łączna liczba przypadków użycia",
            "summaries": "Podsumowania przypadków użycia",
            "sum_id": "ID",
            "sum_name": "Nazwa",
            "sum_value": "Wartość biznesowa",
            "sum_outcome": "Oczekiwany wynik",
            "warning_header": "OSTRZEŻENIE",
            "warning_body": "Nie uruchamiaj tego notatnika. Jest przeznaczony wyłącznie do celów demonstracyjnych i katalogowania.",
            "disclaimer": "Ta treść została wygenerowana przez AI i powinna zostać zweryfikowana przez wykwalifikowanego inżyniera przed wdrożeniem na produkcję. Databricks nie ponosi odpowiedzialności za jakiekolwiek problemy wynikające z korzystania z tej treści.",
            "detailed_scenarios": "Szczegóły przypadków użycia",
            "aspect": "Aspekt",
            "description": "Opis",
            "aspect_domain": "Domena biznesowa",
            "type": "Typ",
            "analytics_technique": "Technika analityczna",
            "primary_table": "Tabela główna",
            "quality": "Jakość",
            "priority": "Priorytet",
            "usecase": "Przypadek użycia",
            "description_label": "Opis",
            "value_type_problem": "Problem",
            "value_type_risk": "Ryzyko",
            "value_type_opportunity": "Szansa",
            "value_type_improvement": "Ulepszenie",
            "value_priority_ultra_high": "Ekstremalnie wysoki",
            "value_priority_very_high": "Bardzo wysoki",
            "value_priority_high": "Wysoki",
            "value_priority_medium": "Średni",
            "value_priority_low": "Niski",
            "value_priority_very_low": "Bardzo niski",
            "value_priority_ultra_low": "Ekstremalnie niski",
            "statement": "Stwierdzenie",
            "solution": "Rozwiązanie",
            "aspect_beneficiary": "Beneficjent",
            "beneficiary": "Beneficjent",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Zgodność z priorytetami biznesowymi",
            "strategic_goals_alignment": "Zgodność z celami strategicznymi",
            "subdomain": "Poddomena",
            "aspect_value": "Wartość biznesowa",
            "business_value": "Wartość biznesowa",
            "aspect_tables": "Powiązane tabele",
            "quality_reasons": "Przyczyny jakości",
            "aspect_ai_function": "Funkcja AI",
            "aspect_analytics_technique": "Technika analityczna",
            "aspect_primary_table": "Tabela główna",
            "aspect_quality": "Jakość",
            "aspect_priority": "Priorytet",
            "strategic_alignment": "Zgodność strategiczna",
            "return_on_investment": "Zwrot z inwestycji",
            "reusability": "Możliwość ponownego użycia",
            "time_to_value": "Czas do wartości",
            "data_availability": "Dostępność danych",
            "data_accessibility": "Dostęp do danych",
            "architecture_fitness": "Dopasowanie architektoniczne",
            "team_skills": "Umiejętności zespołu",
            "domain_knowledge": "Wiedza domenowa",
            "people_allocation": "Alokacja personelu",
            "budget_allocation": "Alokacja budżetu",
            "time_to_production": "Czas do produkcji",
            "value_score": "Wynik wartości",
            "feasibility_score": "Wynik wykonalności",
            "priority_score": "Wynik priorytetu",
            "inspire_score": "Wynik Inspire",
            "quality_score_header": "Wynik jakości",
            "pdf_title": "Strategiczne przypadki użycia AI oparte na Databricks Genie Code",
            "pdf_for": "Dla",
            "pdf_exec_summary": "Podsumowanie wykonawcze",
            "pdf_toc_title": "Domeny przypadków użycia",
            "pdf_detailed_view": "Szczegółowy katalog przypadków użycia",
            "pdf_disclaimer_title": "Zastrzeżenie",
            "pdf_fallback_summary_p1": "Ten dokument opisuje {total_cases} wartościowych analitycznych przypadków użycia zidentyfikowanych dla {business_name}.",
            "pdf_fallback_summary_p2": "Kolejne strony zawierają szczegółowy podział tych możliwości, skategoryzowanych według domeny biznesowej.",
            "pptx_main_title": "Strategiczne przypadki użycia AI oparte na Databricks Genie Code",
            "pptx_for": "Dla",
            "pptx_disclaimer_title": "Zastrzeżenie",
            "pptx_domain_suffix": "Przypadki użycia",
            "example_results": "Przykładowe wyniki",
            "error_no_results": "Nie udało się wygenerować wyników. Sprawdź notatnik: {notebook_name} i przypadek użycia {use_case_id}",
            "input_data_original": "Dane wejściowe (wartości oryginalne)",
            "ai_generated_output": "Wyniki wygenerowane przez AI",
            "column": "Kolumna",
            "value": "Wartość",
            "executive_summary_not_available": "Podsumowanie wykonawcze niedostępne.",
            "domain_summary_not_available": "Podsumowanie domeny niedostępne.",
            "summary_not_available": "Podsumowanie niedostępne.",
            "value_general_improvement": "Ogólne ulepszenie",
            "value_reduce_cost": "Redukcja kosztów",
            "value_increase_revenue": "Zwiększenie przychodów",
            "value_boost_productivity": "Zwiększenie produktywności",
            "value_mitigate_risk": "Ograniczenie ryzyka",
            "value_protect_revenue": "Ochrona przychodów",
            "value_align_to_regulations": "Zgodność z przepisami",
            "value_improve_customer_experience": "Poprawa doświadczenia klienta",
            "value_enable_data_driven_decisions": "Wsparcie decyzji opartych na danych",
            "value_optimize_operations": "Optymalizacja operacji",
            "value_empower_talent": "Wzmocnienie talentów",
            "value_enhance_experience": "Poprawa doświadczenia",
            "value_drive_innovation": "Napędzanie innowacji",
            "value_achieve_esg": "Osiągnięcie ESG",
            "value_execute_strategy": "Realizacja strategii",
            "value_forecasting": "Prognozowanie",
            "value_classification": "Klasyfikacja",
            "value_anomaly_detection": "Wykrywanie anomalii",
            "value_cohort_analysis": "Analiza kohortowa",
            "value_segmentation": "Segmentacja",
            "value_sentiment_analysis": "Analiza sentymentu",
            "value_trend_analysis": "Analiza trendów",
            "value_prescriptive_analytics": "Analityka preskrypcyjna",
            "value_root_cause_analysis": "Analiza przyczyn źródłowych",
            "value_optimization": "Optymalizacja",
            "value_recommendation": "Rekomendacja",
            "value_time_series_analysis": "Analiza szeregów czasowych",
            "value_predictive_analytics": "Analityka predykcyjna",
            "value_descriptive_analytics": "Analityka opisowa",
        },
        "Romanian": {
            "main_title": "Generator de cazuri de utilizare alimentat de Databricks Genie Code",
            "intro": "Acest notebook conține cazuri de utilizare generate de AI pe baza schemelor dvs. Mai jos este un rezumat al scenariilor generate pe domeniu de afaceri.",
            "domain": "Domeniu de afaceri",
            "total": "Total cazuri de utilizare",
            "summaries": "Rezumate ale cazurilor de utilizare",
            "sum_id": "ID",
            "sum_name": "Nume",
            "sum_value": "Valoare de afaceri",
            "sum_outcome": "Rezultat așteptat",
            "warning_header": "AVERTISMENT",
            "warning_body": "Nu rulați acest notebook. Este destinat exclusiv pentru demonstrație și catalogare.",
            "disclaimer": "Acest conținut este generat de AI și trebuie verificat de un inginer calificat înainte de utilizarea în producție. Databricks nu este responsabil pentru nicio problemă rezultată din utilizarea acestui conținut.",
            "detailed_scenarios": "Detalii cazuri de utilizare",
            "aspect": "Aspect",
            "description": "Descriere",
            "aspect_domain": "Domeniu de afaceri",
            "type": "Tip",
            "analytics_technique": "Tehnică analitică",
            "primary_table": "Tabel principal",
            "quality": "Calitate",
            "priority": "Prioritate",
            "usecase": "Caz de utilizare",
            "description_label": "Descriere",
            "value_type_problem": "Problemă",
            "value_type_risk": "Risc",
            "value_type_opportunity": "Oportunitate",
            "value_type_improvement": "Îmbunătățire",
            "value_priority_ultra_high": "Extrem de ridicat",
            "value_priority_very_high": "Foarte ridicat",
            "value_priority_high": "Ridicat",
            "value_priority_medium": "Mediu",
            "value_priority_low": "Scăzut",
            "value_priority_very_low": "Foarte scăzut",
            "value_priority_ultra_low": "Extrem de scăzut",
            "statement": "Declarație",
            "solution": "Soluție",
            "aspect_beneficiary": "Beneficiar",
            "beneficiary": "Beneficiar",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Aliniere la prioritățile de afaceri",
            "strategic_goals_alignment": "Aliniere la obiectivele strategice",
            "subdomain": "Subdomeniu",
            "aspect_value": "Valoare de afaceri",
            "business_value": "Valoare de afaceri",
            "aspect_tables": "Tabele implicate",
            "quality_reasons": "Motive de calitate",
            "aspect_ai_function": "Funcție AI",
            "aspect_analytics_technique": "Tehnică analitică",
            "aspect_primary_table": "Tabel principal",
            "aspect_quality": "Calitate",
            "aspect_priority": "Prioritate",
            "strategic_alignment": "Aliniere strategică",
            "return_on_investment": "Randamentul investiției",
            "reusability": "Reutilizabilitate",
            "time_to_value": "Timp până la valoare",
            "data_availability": "Disponibilitatea datelor",
            "data_accessibility": "Accesibilitatea datelor",
            "architecture_fitness": "Compatibilitate arhitecturală",
            "team_skills": "Competențe echipă",
            "domain_knowledge": "Cunoștințe de domeniu",
            "people_allocation": "Alocare personal",
            "budget_allocation": "Alocare buget",
            "time_to_production": "Timp până la producție",
            "value_score": "Scor valoare",
            "feasibility_score": "Scor fezabilitate",
            "priority_score": "Scor prioritate",
            "inspire_score": "Scor Inspire",
            "quality_score_header": "Scor calitate",
            "pdf_title": "Cazuri de utilizare AI strategice alimentate de Databricks Genie Code",
            "pdf_for": "Pentru",
            "pdf_exec_summary": "Rezumat executiv",
            "pdf_toc_title": "Domenii cazuri de utilizare",
            "pdf_detailed_view": "Catalog detaliat de cazuri de utilizare",
            "pdf_disclaimer_title": "Declinare a responsabilității",
            "pdf_fallback_summary_p1": "Acest document descrie {total_cases} cazuri de utilizare analitice de mare valoare identificate pentru {business_name}.",
            "pdf_fallback_summary_p2": "Paginile următoare oferă o defalcare detaliată a acestor oportunități, clasificate pe domenii de afaceri.",
            "pptx_main_title": "Cazuri de utilizare AI strategice alimentate de Databricks Genie Code",
            "pptx_for": "Pentru",
            "pptx_disclaimer_title": "Declinare a responsabilității",
            "pptx_domain_suffix": "Cazuri de utilizare",
            "example_results": "Rezultate exemplu",
            "error_no_results": "Nu s-au putut genera rezultate. Verificați notebook: {notebook_name} și cazul de utilizare {use_case_id}",
            "input_data_original": "Date de intrare (valori originale)",
            "ai_generated_output": "Rezultate generate de AI",
            "column": "Coloană",
            "value": "Valoare",
            "executive_summary_not_available": "Rezumat executiv indisponibil.",
            "domain_summary_not_available": "Rezumat domeniu indisponibil.",
            "summary_not_available": "Rezumat indisponibil.",
            "value_general_improvement": "Îmbunătățire generală",
            "value_reduce_cost": "Reducerea costurilor",
            "value_increase_revenue": "Creșterea veniturilor",
            "value_boost_productivity": "Creșterea productivității",
            "value_mitigate_risk": "Reducerea riscurilor",
            "value_protect_revenue": "Protejarea veniturilor",
            "value_align_to_regulations": "Conformitate cu reglementările",
            "value_improve_customer_experience": "Îmbunătățirea experienței clientului",
            "value_enable_data_driven_decisions": "Facilitarea deciziilor bazate pe date",
            "value_optimize_operations": "Optimizarea operațiunilor",
            "value_empower_talent": "Consolidarea talentelor",
            "value_enhance_experience": "Îmbunătățirea experienței",
            "value_drive_innovation": "Stimularea inovației",
            "value_achieve_esg": "Atingerea ESG",
            "value_execute_strategy": "Executarea strategiei",
            "value_forecasting": "Prognoză",
            "value_classification": "Clasificare",
            "value_anomaly_detection": "Detectarea anomaliilor",
            "value_cohort_analysis": "Analiză de cohortă",
            "value_segmentation": "Segmentare",
            "value_sentiment_analysis": "Analiza sentimentelor",
            "value_trend_analysis": "Analiza tendințelor",
            "value_prescriptive_analytics": "Analiză prescriptivă",
            "value_root_cause_analysis": "Analiza cauzelor profunde",
            "value_optimization": "Optimizare",
            "value_recommendation": "Recomandare",
            "value_time_series_analysis": "Analiza seriilor temporale",
            "value_predictive_analytics": "Analiză predictivă",
            "value_descriptive_analytics": "Analiză descriptivă",
        },
        "Ukrainian": {
            "main_title": "Генератор бізнес-кейсів на базі Датабрікс Джіні Код",
            "intro": "Цей ноутбук містить бізнес-кейси, створені ШІ на основі ваших схем. Нижче наведено резюме створених сценаріїв за бізнес-доменами.",
            "domain": "Бізнес-домен",
            "total": "Загальна кількість бізнес-кейсів",
            "summaries": "Резюме бізнес-кейсів",
            "sum_id": "ID",
            "sum_name": "Назва",
            "sum_value": "Бізнес-цінність",
            "sum_outcome": "Очікуваний результат",
            "warning_header": "ПОПЕРЕДЖЕННЯ",
            "warning_body": "Не запускайте цей ноутбук. Він призначений лише для демонстрації та каталогізації.",
            "disclaimer": "Цей контент створений ШІ і повинен бути перевірений кваліфікованим інженером перед використанням у виробничому середовищі. Databricks не несе відповідальності за будь-які проблеми, що виникають внаслідок використання цього контенту.",
            "detailed_scenarios": "Деталі бізнес-кейсів",
            "aspect": "Аспект",
            "description": "Опис",
            "aspect_domain": "Бізнес-домен",
            "type": "Тип",
            "analytics_technique": "Аналітична техніка",
            "primary_table": "Основна таблиця",
            "quality": "Якість",
            "priority": "Пріоритет",
            "usecase": "Бізнес-кейс",
            "description_label": "Опис",
            "value_type_problem": "Проблема",
            "value_type_risk": "Ризик",
            "value_type_opportunity": "Можливість",
            "value_type_improvement": "Покращення",
            "value_priority_ultra_high": "Надзвичайно високий",
            "value_priority_very_high": "Дуже високий",
            "value_priority_high": "Високий",
            "value_priority_medium": "Середній",
            "value_priority_low": "Низький",
            "value_priority_very_low": "Дуже низький",
            "value_priority_ultra_low": "Надзвичайно низький",
            "statement": "Опис ситуації",
            "solution": "Рішення",
            "aspect_beneficiary": "Вигодонабувач",
            "beneficiary": "Вигодонабувач",
            "aspect_sponsor": "Спонсор",
            "sponsor": "Спонсор",
            "business_priority_alignment": "Відповідність бізнес-пріоритетам",
            "strategic_goals_alignment": "Відповідність стратегічним цілям",
            "subdomain": "Піддомен",
            "aspect_value": "Бізнес-цінність",
            "business_value": "Бізнес-цінність",
            "aspect_tables": "Пов'язані таблиці",
            "quality_reasons": "Причини якості",
            "aspect_ai_function": "Функція ШІ",
            "aspect_analytics_technique": "Аналітична техніка",
            "aspect_primary_table": "Основна таблиця",
            "aspect_quality": "Якість",
            "aspect_priority": "Пріоритет",
            "strategic_alignment": "Стратегічна відповідність",
            "return_on_investment": "Рентабельність інвестицій",
            "reusability": "Можливість повторного використання",
            "time_to_value": "Час до отримання цінності",
            "data_availability": "Доступність даних",
            "data_accessibility": "Доступ до даних",
            "architecture_fitness": "Відповідність архітектурі",
            "team_skills": "Навички команди",
            "domain_knowledge": "Знання предметної області",
            "people_allocation": "Розподіл персоналу",
            "budget_allocation": "Розподіл бюджету",
            "time_to_production": "Час до впровадження",
            "value_score": "Оцінка цінності",
            "feasibility_score": "Оцінка реалізованості",
            "priority_score": "Оцінка пріоритету",
            "inspire_score": "Оцінка Inspire",
            "quality_score_header": "Оцінка якості",
            "pdf_title": "Стратегічні ШІ бізнес-кейси на базі Датабрікс Джіні Код",
            "pdf_for": "Для",
            "pdf_exec_summary": "Резюме для керівництва",
            "pdf_toc_title": "Домени бізнес-кейсів",
            "pdf_detailed_view": "Детальний каталог бізнес-кейсів",
            "pdf_disclaimer_title": "Відмова від відповідальності",
            "pdf_fallback_summary_p1": "Цей документ описує {total_cases} високоцінних аналітичних бізнес-кейсів, виявлених для {business_name}.",
            "pdf_fallback_summary_p2": "На наступних сторінках представлений детальний аналіз цих можливостей, класифікованих за бізнес-доменами.",
            "pptx_main_title": "Стратегічні ШІ бізнес-кейси на базі Датабрікс Джіні Код",
            "pptx_for": "Для",
            "pptx_disclaimer_title": "Відмова від відповідальності",
            "pptx_domain_suffix": "Бізнес-кейси",
            "example_results": "Приклади результатів",
            "error_no_results": "Не вдалося створити результати. Перевірте ноутбук: {notebook_name} та бізнес-кейс {use_case_id}",
            "input_data_original": "Вхідні дані (оригінальні значення)",
            "ai_generated_output": "Результати, створені ШІ",
            "column": "Стовпець",
            "value": "Значення",
            "executive_summary_not_available": "Резюме для керівництва недоступне.",
            "domain_summary_not_available": "Резюме домену недоступне.",
            "summary_not_available": "Резюме недоступне.",
            "value_general_improvement": "Загальне покращення",
            "value_reduce_cost": "Скорочення витрат",
            "value_increase_revenue": "Збільшення доходу",
            "value_boost_productivity": "Підвищення продуктивності",
            "value_mitigate_risk": "Зниження ризиків",
            "value_protect_revenue": "Захист доходу",
            "value_align_to_regulations": "Дотримання нормативів",
            "value_improve_customer_experience": "Покращення клієнтського досвіду",
            "value_enable_data_driven_decisions": "Забезпечення рішень на основі даних",
            "value_optimize_operations": "Оптимізація операцій",
            "value_empower_talent": "Розвиток талантів",
            "value_enhance_experience": "Покращення досвіду",
            "value_drive_innovation": "Стимулювання інновацій",
            "value_achieve_esg": "Досягнення ESG",
            "value_execute_strategy": "Реалізація стратегії",
            "value_forecasting": "Прогнозування",
            "value_classification": "Класифікація",
            "value_anomaly_detection": "Виявлення аномалій",
            "value_cohort_analysis": "Когортний аналіз",
            "value_segmentation": "Сегментація",
            "value_sentiment_analysis": "Аналіз настроїв",
            "value_trend_analysis": "Аналіз трендів",
            "value_prescriptive_analytics": "Приписна аналітика",
            "value_root_cause_analysis": "Аналіз першопричин",
            "value_optimization": "Оптимізація",
            "value_recommendation": "Рекомендація",
            "value_time_series_analysis": "Аналіз часових рядів",
            "value_predictive_analytics": "Прогнозна аналітика",
            "value_descriptive_analytics": "Описова аналітика",
        },
        "Dutch": {
            "main_title": "Gebruikscasegenerator aangedreven door Databricks Genie Code",
            "intro": "Dit notebook bevat door AI gegenereerde gebruikscases op basis van uw schema's. Hieronder vindt u een samenvatting van de gegenereerde scenario's per bedrijfsdomein.",
            "domain": "Bedrijfsdomein",
            "total": "Totaal aantal gebruikscases",
            "summaries": "Samenvattingen van gebruikscases",
            "sum_id": "ID",
            "sum_name": "Naam",
            "sum_value": "Bedrijfswaarde",
            "sum_outcome": "Verwacht resultaat",
            "warning_header": "WAARSCHUWING",
            "warning_body": "Voer dit notebook niet uit. Het is uitsluitend bedoeld voor demonstratie en catalogisering.",
            "disclaimer": "Deze inhoud is door AI gegenereerd en dient door een gekwalificeerde engineer te worden beoordeeld voordat deze in productie wordt gebruikt. Databricks is niet aansprakelijk voor problemen die voortvloeien uit het gebruik van deze inhoud.",
            "detailed_scenarios": "Details van gebruikscases",
            "aspect": "Aspect",
            "description": "Beschrijving",
            "aspect_domain": "Bedrijfsdomein",
            "type": "Type",
            "analytics_technique": "Analysetechniek",
            "primary_table": "Primaire tabel",
            "quality": "Kwaliteit",
            "priority": "Prioriteit",
            "usecase": "Gebruikscase",
            "description_label": "Beschrijving",
            "value_type_problem": "Probleem",
            "value_type_risk": "Risico",
            "value_type_opportunity": "Kans",
            "value_type_improvement": "Verbetering",
            "value_priority_ultra_high": "Extreem hoog",
            "value_priority_very_high": "Zeer hoog",
            "value_priority_high": "Hoog",
            "value_priority_medium": "Gemiddeld",
            "value_priority_low": "Laag",
            "value_priority_very_low": "Zeer laag",
            "value_priority_ultra_low": "Extreem laag",
            "statement": "Verklaring",
            "solution": "Oplossing",
            "aspect_beneficiary": "Begunstigde",
            "beneficiary": "Begunstigde",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Afstemming bedrijfsprioriteit",
            "strategic_goals_alignment": "Afstemming strategische doelen",
            "subdomain": "Subdomein",
            "aspect_value": "Bedrijfswaarde",
            "business_value": "Bedrijfswaarde",
            "aspect_tables": "Betrokken tabellen",
            "quality_reasons": "Kwaliteitsredenen",
            "aspect_ai_function": "AI-functie",
            "aspect_analytics_technique": "Analysetechniek",
            "aspect_primary_table": "Primaire tabel",
            "aspect_quality": "Kwaliteit",
            "aspect_priority": "Prioriteit",
            "strategic_alignment": "Strategische afstemming",
            "return_on_investment": "Rendement op investering",
            "reusability": "Herbruikbaarheid",
            "time_to_value": "Tijd tot waarde",
            "data_availability": "Databeschikbaarheid",
            "data_accessibility": "Datatoegang",
            "architecture_fitness": "Architectuurgeschiktheid",
            "team_skills": "Teamvaardigheden",
            "domain_knowledge": "Domeinkennis",
            "people_allocation": "Personeelstoewijzing",
            "budget_allocation": "Budgettoewijzing",
            "time_to_production": "Tijd tot productie",
            "value_score": "Waardescore",
            "feasibility_score": "Haalbaarheidsscore",
            "priority_score": "Prioriteitsscore",
            "inspire_score": "Inspire-score",
            "quality_score_header": "Kwaliteitsscore",
            "pdf_title": "Strategische AI-gebruikscases aangedreven door Databricks Genie Code",
            "pdf_for": "Voor",
            "pdf_exec_summary": "Samenvatting",
            "pdf_toc_title": "Gebruikscasedomeinen",
            "pdf_detailed_view": "Gedetailleerde gebruikscasecatalogus",
            "pdf_disclaimer_title": "Disclaimer",
            "pdf_fallback_summary_p1": "Dit document beschrijft {total_cases} hoogwaardige analytische gebruikscases geïdentificeerd voor {business_name}.",
            "pdf_fallback_summary_p2": "De volgende pagina's bieden een gedetailleerde uitsplitsing van deze kansen, gecategoriseerd per bedrijfsdomein.",
            "pptx_main_title": "Strategische AI-gebruikscases aangedreven door Databricks Genie Code",
            "pptx_for": "Voor",
            "pptx_disclaimer_title": "Disclaimer",
            "pptx_domain_suffix": "Gebruikscases",
            "example_results": "Voorbeeldresultaten",
            "error_no_results": "Kon geen resultaten genereren. Controleer notebook: {notebook_name} en gebruikscase {use_case_id}",
            "input_data_original": "Invoergegevens (originele waarden)",
            "ai_generated_output": "Door AI gegenereerde resultaten",
            "column": "Kolom",
            "value": "Waarde",
            "executive_summary_not_available": "Samenvatting niet beschikbaar.",
            "domain_summary_not_available": "Domeinsamenvatting niet beschikbaar.",
            "summary_not_available": "Samenvatting niet beschikbaar.",
            "value_general_improvement": "Algemene verbetering",
            "value_reduce_cost": "Kosten verlagen",
            "value_increase_revenue": "Omzet verhogen",
            "value_boost_productivity": "Productiviteit verhogen",
            "value_mitigate_risk": "Risico's beperken",
            "value_protect_revenue": "Omzet beschermen",
            "value_align_to_regulations": "Voldoen aan regelgeving",
            "value_improve_customer_experience": "Klantervaring verbeteren",
            "value_enable_data_driven_decisions": "Datagedreven besluitvorming mogelijk maken",
            "value_optimize_operations": "Operaties optimaliseren",
            "value_empower_talent": "Talent versterken",
            "value_enhance_experience": "Ervaring verbeteren",
            "value_drive_innovation": "Innovatie stimuleren",
            "value_achieve_esg": "ESG bereiken",
            "value_execute_strategy": "Strategie uitvoeren",
            "value_forecasting": "Voorspelling",
            "value_classification": "Classificatie",
            "value_anomaly_detection": "Anomaliedetectie",
            "value_cohort_analysis": "Cohortanalyse",
            "value_segmentation": "Segmentatie",
            "value_sentiment_analysis": "Sentimentanalyse",
            "value_trend_analysis": "Trendanalyse",
            "value_prescriptive_analytics": "Prescriptieve analyse",
            "value_root_cause_analysis": "Oorzaakanalyse",
            "value_optimization": "Optimalisatie",
            "value_recommendation": "Aanbeveling",
            "value_time_series_analysis": "Tijdreeksanalyse",
            "value_predictive_analytics": "Voorspellende analyse",
            "value_descriptive_analytics": "Beschrijvende analyse",
        },
        "Indonesian": {
            "main_title": "Generator Kasus Penggunaan Didukung oleh Databricks Genie Code",
            "intro": "Notebook ini berisi kasus penggunaan yang dihasilkan oleh AI berdasarkan skema Anda. Di bawah ini adalah ringkasan skenario yang dihasilkan berdasarkan domain bisnis.",
            "domain": "Domain Bisnis",
            "total": "Total Kasus Penggunaan",
            "summaries": "Ringkasan Kasus Penggunaan",
            "sum_id": "ID",
            "sum_name": "Nama",
            "sum_value": "Nilai Bisnis",
            "sum_outcome": "Hasil yang Diharapkan",
            "warning_header": "PERINGATAN",
            "warning_body": "Jangan jalankan notebook ini. Ini dimaksudkan hanya untuk demonstrasi dan katalogisasi.",
            "disclaimer": "Konten ini dihasilkan oleh AI dan harus ditinjau oleh insinyur yang berkualifikasi sebelum digunakan dalam produksi. Databricks tidak bertanggung jawab atas masalah apa pun yang timbul dari penggunaan konten ini.",
            "detailed_scenarios": "Detail Kasus Penggunaan",
            "aspect": "Aspek",
            "description": "Deskripsi",
            "aspect_domain": "Domain Bisnis",
            "type": "Tipe",
            "analytics_technique": "Teknik Analitik",
            "primary_table": "Tabel Utama",
            "quality": "Kualitas",
            "priority": "Prioritas",
            "usecase": "Kasus Penggunaan",
            "description_label": "Deskripsi",
            "value_type_problem": "Masalah",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Peluang",
            "value_type_improvement": "Peningkatan",
            "value_priority_ultra_high": "Sangat Tinggi Sekali",
            "value_priority_very_high": "Sangat Tinggi",
            "value_priority_high": "Tinggi",
            "value_priority_medium": "Sedang",
            "value_priority_low": "Rendah",
            "value_priority_very_low": "Sangat Rendah",
            "value_priority_ultra_low": "Sangat Rendah Sekali",
            "statement": "Pernyataan",
            "solution": "Solusi",
            "aspect_beneficiary": "Penerima Manfaat",
            "beneficiary": "Penerima Manfaat",
            "aspect_sponsor": "Sponsor",
            "sponsor": "Sponsor",
            "business_priority_alignment": "Keselarasan Prioritas Bisnis",
            "strategic_goals_alignment": "Keselarasan Tujuan Strategis",
            "subdomain": "Subdomain",
            "aspect_value": "Nilai Bisnis",
            "business_value": "Nilai Bisnis",
            "aspect_tables": "Tabel yang Terlibat",
            "quality_reasons": "Alasan Kualitas",
            "aspect_ai_function": "Fungsi AI",
            "aspect_analytics_technique": "Teknik Analitik",
            "aspect_primary_table": "Tabel Utama",
            "aspect_quality": "Kualitas",
            "aspect_priority": "Prioritas",
            "strategic_alignment": "Keselarasan Strategis",
            "return_on_investment": "Pengembalian Investasi",
            "reusability": "Dapat Digunakan Kembali",
            "time_to_value": "Waktu ke Nilai",
            "data_availability": "Ketersediaan Data",
            "data_accessibility": "Aksesibilitas Data",
            "architecture_fitness": "Kesesuaian Arsitektur",
            "team_skills": "Keterampilan Tim",
            "domain_knowledge": "Pengetahuan Domain",
            "people_allocation": "Alokasi Personel",
            "budget_allocation": "Alokasi Anggaran",
            "time_to_production": "Waktu ke Produksi",
            "value_score": "Skor Nilai",
            "feasibility_score": "Skor Kelayakan",
            "priority_score": "Skor Prioritas",
            "inspire_score": "Skor Inspire",
            "quality_score_header": "Skor Kualitas",
            "pdf_title": "Kasus Penggunaan AI Strategis Didukung oleh Databricks Genie Code",
            "pdf_for": "Untuk",
            "pdf_exec_summary": "Ringkasan Eksekutif",
            "pdf_toc_title": "Domain Kasus Penggunaan",
            "pdf_detailed_view": "Katalog Detail Kasus Penggunaan",
            "pdf_disclaimer_title": "Penafian",
            "pdf_fallback_summary_p1": "Dokumen ini menjelaskan {total_cases} kasus penggunaan analitik bernilai tinggi yang diidentifikasi untuk {business_name}.",
            "pdf_fallback_summary_p2": "Halaman-halaman berikut memberikan uraian terperinci tentang peluang-peluang ini, dikategorikan berdasarkan domain bisnis.",
            "pptx_main_title": "Kasus Penggunaan AI Strategis Didukung oleh Databricks Genie Code",
            "pptx_for": "Untuk",
            "pptx_disclaimer_title": "Penafian",
            "pptx_domain_suffix": "Kasus Penggunaan",
            "example_results": "Contoh Hasil",
            "error_no_results": "Tidak dapat menghasilkan hasil. Periksa notebook: {notebook_name} dan kasus penggunaan {use_case_id}",
            "input_data_original": "Data Input (Nilai Asli)",
            "ai_generated_output": "Hasil yang Dihasilkan AI",
            "column": "Kolom",
            "value": "Nilai",
            "executive_summary_not_available": "Ringkasan eksekutif tidak tersedia.",
            "domain_summary_not_available": "Ringkasan domain tidak tersedia.",
            "summary_not_available": "Ringkasan tidak tersedia.",
            "value_general_improvement": "Peningkatan Umum",
            "value_reduce_cost": "Mengurangi Biaya",
            "value_increase_revenue": "Meningkatkan Pendapatan",
            "value_boost_productivity": "Meningkatkan Produktivitas",
            "value_mitigate_risk": "Mengurangi Risiko",
            "value_protect_revenue": "Melindungi Pendapatan",
            "value_align_to_regulations": "Mematuhi Peraturan",
            "value_improve_customer_experience": "Meningkatkan Pengalaman Pelanggan",
            "value_enable_data_driven_decisions": "Memungkinkan Keputusan Berbasis Data",
            "value_optimize_operations": "Mengoptimalkan Operasi",
            "value_empower_talent": "Memberdayakan Talenta",
            "value_enhance_experience": "Meningkatkan Pengalaman",
            "value_drive_innovation": "Mendorong Inovasi",
            "value_achieve_esg": "Mencapai ESG",
            "value_execute_strategy": "Melaksanakan Strategi",
            "value_forecasting": "Peramalan",
            "value_classification": "Klasifikasi",
            "value_anomaly_detection": "Deteksi Anomali",
            "value_cohort_analysis": "Analisis Kohort",
            "value_segmentation": "Segmentasi",
            "value_sentiment_analysis": "Analisis Sentimen",
            "value_trend_analysis": "Analisis Tren",
            "value_prescriptive_analytics": "Analitik Preskriptif",
            "value_root_cause_analysis": "Analisis Akar Masalah",
            "value_optimization": "Optimasi",
            "value_recommendation": "Rekomendasi",
            "value_time_series_analysis": "Analisis Deret Waktu",
            "value_predictive_analytics": "Analitik Prediktif",
            "value_descriptive_analytics": "Analitik Deskriptif",
        },
        "Malay": {
            "main_title": "Penjana Kes Penggunaan Dikuasakan oleh Databricks Genie Code",
            "intro": "Buku nota ini mengandungi kes penggunaan yang dijana oleh AI berdasarkan skema anda. Di bawah ialah ringkasan senario yang dijana mengikut domain perniagaan.",
            "domain": "Domain Perniagaan",
            "total": "Jumlah Kes Penggunaan",
            "summaries": "Ringkasan Kes Penggunaan",
            "sum_id": "ID",
            "sum_name": "Nama",
            "sum_value": "Nilai Perniagaan",
            "sum_outcome": "Hasil Dijangka",
            "warning_header": "AMARAN",
            "warning_body": "Jangan jalankan buku nota ini. Ia bertujuan untuk demonstrasi dan pengkatalogan sahaja.",
            "disclaimer": "Kandungan ini dijana oleh AI dan perlu disemak oleh jurutera yang berkelayakan sebelum digunakan dalam pengeluaran. Databricks tidak bertanggungjawab atas sebarang masalah yang timbul daripada penggunaan kandungan ini.",
            "detailed_scenarios": "Butiran Kes Penggunaan",
            "aspect": "Aspek",
            "description": "Penerangan",
            "aspect_domain": "Domain Perniagaan",
            "type": "Jenis",
            "analytics_technique": "Teknik Analitik",
            "primary_table": "Jadual Utama",
            "quality": "Kualiti",
            "priority": "Keutamaan",
            "usecase": "Kes Penggunaan",
            "description_label": "Penerangan",
            "value_type_problem": "Masalah",
            "value_type_risk": "Risiko",
            "value_type_opportunity": "Peluang",
            "value_type_improvement": "Penambahbaikan",
            "value_priority_ultra_high": "Amat Tinggi Sekali",
            "value_priority_very_high": "Amat Tinggi",
            "value_priority_high": "Tinggi",
            "value_priority_medium": "Sederhana",
            "value_priority_low": "Rendah",
            "value_priority_very_low": "Amat Rendah",
            "value_priority_ultra_low": "Amat Rendah Sekali",
            "statement": "Pernyataan",
            "solution": "Penyelesaian",
            "aspect_beneficiary": "Penerima Manfaat",
            "beneficiary": "Penerima Manfaat",
            "aspect_sponsor": "Penaja",
            "sponsor": "Penaja",
            "business_priority_alignment": "Penjajaran Keutamaan Perniagaan",
            "strategic_goals_alignment": "Penjajaran Matlamat Strategik",
            "subdomain": "Subdomain",
            "aspect_value": "Nilai Perniagaan",
            "business_value": "Nilai Perniagaan",
            "aspect_tables": "Jadual Terlibat",
            "quality_reasons": "Sebab Kualiti",
            "aspect_ai_function": "Fungsi AI",
            "aspect_analytics_technique": "Teknik Analitik",
            "aspect_primary_table": "Jadual Utama",
            "aspect_quality": "Kualiti",
            "aspect_priority": "Keutamaan",
            "strategic_alignment": "Penjajaran Strategik",
            "return_on_investment": "Pulangan Pelaburan",
            "reusability": "Kebolehgunaan Semula",
            "time_to_value": "Masa ke Nilai",
            "data_availability": "Ketersediaan Data",
            "data_accessibility": "Kebolehcapaian Data",
            "architecture_fitness": "Kesesuaian Seni Bina",
            "team_skills": "Kemahiran Pasukan",
            "domain_knowledge": "Pengetahuan Domain",
            "people_allocation": "Peruntukan Kakitangan",
            "budget_allocation": "Peruntukan Bajet",
            "time_to_production": "Masa ke Pengeluaran",
            "value_score": "Skor Nilai",
            "feasibility_score": "Skor Kebolehlaksanaan",
            "priority_score": "Skor Keutamaan",
            "inspire_score": "Skor Inspire",
            "quality_score_header": "Skor Kualiti",
            "pdf_title": "Kes Penggunaan AI Strategik Dikuasakan oleh Databricks Genie Code",
            "pdf_for": "Untuk",
            "pdf_exec_summary": "Ringkasan Eksekutif",
            "pdf_toc_title": "Domain Kes Penggunaan",
            "pdf_detailed_view": "Katalog Terperinci Kes Penggunaan",
            "pdf_disclaimer_title": "Penafian",
            "pdf_fallback_summary_p1": "Dokumen ini menerangkan {total_cases} kes penggunaan analitik bernilai tinggi yang dikenal pasti untuk {business_name}.",
            "pdf_fallback_summary_p2": "Halaman-halaman berikut menyediakan perincian terperinci peluang-peluang ini, dikategorikan mengikut domain perniagaan.",
            "pptx_main_title": "Kes Penggunaan AI Strategik Dikuasakan oleh Databricks Genie Code",
            "pptx_for": "Untuk",
            "pptx_disclaimer_title": "Penafian",
            "pptx_domain_suffix": "Kes Penggunaan",
            "example_results": "Contoh Keputusan",
            "error_no_results": "Tidak dapat menjana keputusan. Semak buku nota: {notebook_name} dan kes penggunaan {use_case_id}",
            "input_data_original": "Data Input (Nilai Asal)",
            "ai_generated_output": "Keputusan Dijana oleh AI",
            "column": "Lajur",
            "value": "Nilai",
            "executive_summary_not_available": "Ringkasan eksekutif tidak tersedia.",
            "domain_summary_not_available": "Ringkasan domain tidak tersedia.",
            "summary_not_available": "Ringkasan tidak tersedia.",
            "value_general_improvement": "Penambahbaikan Umum",
            "value_reduce_cost": "Mengurangkan Kos",
            "value_increase_revenue": "Meningkatkan Hasil",
            "value_boost_productivity": "Meningkatkan Produktiviti",
            "value_mitigate_risk": "Mengurangkan Risiko",
            "value_protect_revenue": "Melindungi Hasil",
            "value_align_to_regulations": "Mematuhi Peraturan",
            "value_improve_customer_experience": "Meningkatkan Pengalaman Pelanggan",
            "value_enable_data_driven_decisions": "Membolehkan Keputusan Berasaskan Data",
            "value_optimize_operations": "Mengoptimumkan Operasi",
            "value_empower_talent": "Memperkasakan Bakat",
            "value_enhance_experience": "Meningkatkan Pengalaman",
            "value_drive_innovation": "Memacu Inovasi",
            "value_achieve_esg": "Mencapai ESG",
            "value_execute_strategy": "Melaksanakan Strategi",
            "value_forecasting": "Ramalan",
            "value_classification": "Pengelasan",
            "value_anomaly_detection": "Pengesanan Anomali",
            "value_cohort_analysis": "Analisis Kohort",
            "value_segmentation": "Segmentasi",
            "value_sentiment_analysis": "Analisis Sentimen",
            "value_trend_analysis": "Analisis Trend",
            "value_prescriptive_analytics": "Analitik Preskriptif",
            "value_root_cause_analysis": "Analisis Punca Masalah",
            "value_optimization": "Pengoptimuman",
            "value_recommendation": "Cadangan",
            "value_time_series_analysis": "Analisis Siri Masa",
            "value_predictive_analytics": "Analitik Ramalan",
            "value_descriptive_analytics": "Analitik Deskriptif",
        },
        "Tamil": {
            "main_title": "Databricks Genie Code மூலம் இயக்கப்படும் பயன்பாட்டு வழக்கு உருவாக்கி",
            "intro": "இந்த நோட்புக் உங்கள் திட்டமைப்புகளின் அடிப்படையில் AI உருவாக்கிய பயன்பாட்டு வழக்குகளைக் கொண்டுள்ளது. கீழே வணிக துறை வாரியாக உருவாக்கப்பட்ட காட்சிகளின் சுருக்கம் உள்ளது.",
            "domain": "வணிக துறை",
            "total": "மொத்த பயன்பாட்டு வழக்குகள்",
            "summaries": "பயன்பாட்டு வழக்கு சுருக்கங்கள்",
            "sum_id": "ID",
            "sum_name": "பெயர்",
            "sum_value": "வணிக மதிப்பு",
            "sum_outcome": "எதிர்பார்க்கப்படும் முடிவு",
            "warning_header": "எச்சரிக்கை",
            "warning_body": "இந்த நோட்புக்கை இயக்க வேண்டாம். இது ஆர்ப்பாட்டம் மற்றும் பட்டியலிடல் நோக்கங்களுக்கு மட்டுமே.",
            "disclaimer": "இந்த உள்ளடக்கம் AI ஆல் உருவாக்கப்பட்டது மற்றும் உற்பத்தியில் பயன்படுத்துவதற்கு முன் தகுதியான பொறியாளரால் மதிப்பாய்வு செய்யப்பட வேண்டும். இந்த உள்ளடக்கத்தின் பயன்பாட்டினால் எழும் எந்தப் பிரச்சனைகளுக்கும் Databricks பொறுப்பல்ல.",
            "detailed_scenarios": "பயன்பாட்டு வழக்கு விவரங்கள்",
            "aspect": "அம்சம்",
            "description": "விளக்கம்",
            "aspect_domain": "வணிக துறை",
            "type": "வகை",
            "analytics_technique": "பகுப்பாய்வு நுட்பம்",
            "primary_table": "முதன்மை அட்டவணை",
            "quality": "தரம்",
            "priority": "முன்னுரிமை",
            "usecase": "பயன்பாட்டு வழக்கு",
            "description_label": "விளக்கம்",
            "value_type_problem": "சிக்கல்",
            "value_type_risk": "அபாயம்",
            "value_type_opportunity": "வாய்ப்பு",
            "value_type_improvement": "மேம்பாடு",
            "value_priority_ultra_high": "மிக மிக உயர்ந்த",
            "value_priority_very_high": "மிக உயர்ந்த",
            "value_priority_high": "உயர்ந்த",
            "value_priority_medium": "நடுத்தர",
            "value_priority_low": "குறைந்த",
            "value_priority_very_low": "மிக குறைந்த",
            "value_priority_ultra_low": "மிக மிக குறைந்த",
            "statement": "அறிக்கை",
            "solution": "தீர்வு",
            "aspect_beneficiary": "பயனாளி",
            "beneficiary": "பயனாளி",
            "aspect_sponsor": "ஆதரவாளர்",
            "sponsor": "ஆதரவாளர்",
            "business_priority_alignment": "வணிக முன்னுரிமை சீரமைப்பு",
            "strategic_goals_alignment": "மூலோபாய இலக்கு சீரமைப்பு",
            "subdomain": "துணை துறை",
            "aspect_value": "வணிக மதிப்பு",
            "business_value": "வணிக மதிப்பு",
            "aspect_tables": "தொடர்புடைய அட்டவணைகள்",
            "quality_reasons": "தர காரணங்கள்",
            "aspect_ai_function": "AI செயல்பாடு",
            "aspect_analytics_technique": "பகுப்பாய்வு நுட்பம்",
            "aspect_primary_table": "முதன்மை அட்டவணை",
            "aspect_quality": "தரம்",
            "aspect_priority": "முன்னுரிமை",
            "strategic_alignment": "மூலோபாய சீரமைப்பு",
            "return_on_investment": "முதலீட்டு வருவாய்",
            "reusability": "மறுபயன்பாட்டுத் திறன்",
            "time_to_value": "மதிப்புக்கான நேரம்",
            "data_availability": "தரவு கிடைக்கும் தன்மை",
            "data_accessibility": "தரவு அணுகல்",
            "architecture_fitness": "கட்டமைப்பு பொருத்தம்",
            "team_skills": "குழு திறன்கள்",
            "domain_knowledge": "துறை அறிவு",
            "people_allocation": "பணியாளர் ஒதுக்கீடு",
            "budget_allocation": "பட்ஜெட் ஒதுக்கீடு",
            "time_to_production": "உற்பத்திக்கான நேரம்",
            "value_score": "மதிப்பு மதிப்பெண்",
            "feasibility_score": "சாத்தியக்கூறு மதிப்பெண்",
            "priority_score": "முன்னுரிமை மதிப்பெண்",
            "inspire_score": "Inspire மதிப்பெண்",
            "quality_score_header": "தர மதிப்பெண்",
            "pdf_title": "Databricks Genie Code மூலம் இயக்கப்படும் மூலோபாய AI பயன்பாட்டு வழக்குகள்",
            "pdf_for": "இதற்காக",
            "pdf_exec_summary": "நிர்வாக சுருக்கம்",
            "pdf_toc_title": "பயன்பாட்டு வழக்கு துறைகள்",
            "pdf_detailed_view": "விரிவான பயன்பாட்டு வழக்கு பட்டியல்",
            "pdf_disclaimer_title": "மறுப்பு",
            "pdf_fallback_summary_p1": "இந்த ஆவணம் {business_name} க்காக அடையாளம் காணப்பட்ட {total_cases} உயர் மதிப்புள்ள பகுப்பாய்வு பயன்பாட்டு வழக்குகளை விவரிக்கிறது.",
            "pdf_fallback_summary_p2": "அடுத்த பக்கங்கள் வணிக துறை வாரியாக வகைப்படுத்தப்பட்ட இந்த வாய்ப்புகளின் விரிவான பகுப்பாய்வை வழங்குகின்றன.",
            "pptx_main_title": "Databricks Genie Code மூலம் இயக்கப்படும் மூலோபாய AI பயன்பாட்டு வழக்குகள்",
            "pptx_for": "இதற்காக",
            "pptx_disclaimer_title": "மறுப்பு",
            "pptx_domain_suffix": "பயன்பாட்டு வழக்குகள்",
            "example_results": "எடுத்துக்காட்டு முடிவுகள்",
            "error_no_results": "முடிவுகளை உருவாக்க இயலவில்லை. நோட்புக்: {notebook_name} மற்றும் பயன்பாட்டு வழக்கு {use_case_id} ஐ சரிபாருங்கள்",
            "input_data_original": "உள்ளீட்டு தரவு (அசல் மதிப்புகள்)",
            "ai_generated_output": "AI உருவாக்கிய முடிவுகள்",
            "column": "நெடுவரிசை",
            "value": "மதிப்பு",
            "executive_summary_not_available": "நிர்வாக சுருக்கம் கிடைக்கவில்லை.",
            "domain_summary_not_available": "துறை சுருக்கம் கிடைக்கவில்லை.",
            "summary_not_available": "சுருக்கம் கிடைக்கவில்லை.",
            "value_general_improvement": "பொது மேம்பாடு",
            "value_reduce_cost": "செலவைக் குறைத்தல்",
            "value_increase_revenue": "வருவாயை அதிகரித்தல்",
            "value_boost_productivity": "உற்பத்தித்திறனை அதிகரித்தல்",
            "value_mitigate_risk": "அபாயத்தைக் குறைத்தல்",
            "value_protect_revenue": "வருவாயைப் பாதுகாத்தல்",
            "value_align_to_regulations": "விதிமுறைகளுக்கு இணங்குதல்",
            "value_improve_customer_experience": "வாடிக்கையாளர் அனுபவத்தை மேம்படுத்துதல்",
            "value_enable_data_driven_decisions": "தரவு அடிப்படையிலான முடிவுகளை செயல்படுத்துதல்",
            "value_optimize_operations": "செயல்பாடுகளை மேம்படுத்துதல்",
            "value_empower_talent": "திறமையை வலுப்படுத்துதல்",
            "value_enhance_experience": "அனுபவத்தை மேம்படுத்துதல்",
            "value_drive_innovation": "புதுமையை உந்துதல்",
            "value_achieve_esg": "ESG ஐ அடைதல்",
            "value_execute_strategy": "மூலோபாயத்தை செயல்படுத்துதல்",
            "value_forecasting": "முன்கணிப்பு",
            "value_classification": "வகைப்பாடு",
            "value_anomaly_detection": "அசாதாரண கண்டறிதல்",
            "value_cohort_analysis": "குழு பகுப்பாய்வு",
            "value_segmentation": "பிரிவுபடுத்தல்",
            "value_sentiment_analysis": "உணர்வு பகுப்பாய்வு",
            "value_trend_analysis": "போக்கு பகுப்பாய்வு",
            "value_prescriptive_analytics": "பரிந்துரை பகுப்பாய்வு",
            "value_root_cause_analysis": "மூல காரண பகுப்பாய்வு",
            "value_optimization": "உகப்பாக்கம்",
            "value_recommendation": "பரிந்துரை",
            "value_time_series_analysis": "நேரத் தொடர் பகுப்பாய்வு",
            "value_predictive_analytics": "முன்கணிப்பு பகுப்பாய்வு",
            "value_descriptive_analytics": "விளக்க பகுப்பாய்வு",
        },
    }

    def _apply_translation_fallbacks(self, translations: dict, target_language: str) -> dict:
        """
        Applies known fallback translations for commonly untranslated terms.
        This fixes cases where the LLM fails to translate certain terms.
        ALWAYS applies fallbacks for keys that are missing OR still have English values.
        """
        if target_language not in self.TRANSLATION_FALLBACKS:
            return translations
        
        fallbacks = self.TRANSLATION_FALLBACKS[target_language]
        english_values = list(self.ENGLISH_TRANSLATIONS.values())
        english_values_lower = [v.lower() for v in english_values]
        
        applied_count = 0
        for key, fallback_value in fallbacks.items():
            should_apply = False
            current_value = translations.get(key, None)
            
            # Apply fallback if: key is missing, value is empty, or value is still in English
            if key not in translations:
                should_apply = True
                self.logger.debug(f"Translation MISSING for '{key}', applying fallback: '{fallback_value}'")
            elif not current_value or (isinstance(current_value, str) and not current_value.strip()):
                should_apply = True
                self.logger.debug(f"Translation EMPTY for '{key}', applying fallback: '{fallback_value}'")
            elif isinstance(current_value, str) and current_value.lower() in english_values_lower:
                should_apply = True
                self.logger.debug(f"Translation still ENGLISH for '{key}': '{current_value}' → '{fallback_value}'")
            
            if should_apply:
                translations[key] = fallback_value
                applied_count += 1
        
        if applied_count > 0:
            self.logger.info(f"Applied {applied_count} fallback translations for {target_language}")
        
        return translations

    def _validate_translations(self, translations: dict, target_language: str) -> bool:
        """
        Validates that critical fields are actually translated and not left in English.
        Returns True if translations are valid, False otherwise.
        Handles cognates/loanwords that are legitimately the same across languages
        (e.g., 'Sponsor' in German/French/Italian, 'Problem' in German/Dutch/Swedish).
        """
        critical_keys = [
            "type", "subdomain", "analytics_technique", "primary_table", "priority", 
            "aspect_analytics_technique", "aspect_primary_table", "aspect_priority",
            "statement", "solution", "business_value", "beneficiary", "sponsor",
            "strategic_goals_alignment",
            "pdf_detailed_view", "pptx_domain_suffix", "domain",
            "value_type_opportunity", "value_type_problem", "value_type_risk", "value_type_improvement",
            "value_priority_very_high", "value_priority_high", "value_priority_medium", "value_priority_low", "value_priority_very_low",
            "example_results", "column", "value"
        ]
        
        fallbacks = self.TRANSLATION_FALLBACKS.get(target_language, {})
        
        failed_count = 0
        for key in critical_keys:
            if key not in translations:
                continue
            value = translations[key]
            if not isinstance(value, str):
                continue
            english_value = self.ENGLISH_TRANSLATIONS.get(key, "")
            if not english_value:
                continue
            if value.lower() != english_value.lower():
                continue
            if key in fallbacks and fallbacks[key].lower() == english_value.lower():
                continue
            failed_count += 1
            self.logger.warning(f"Translation validation: Key '{key}' still has English value '{value}' for {target_language}")
        
        if failed_count > 3:
            self.logger.warning(f"Translation validation FAILED for {target_language}: {failed_count} keys still in English")
            return False
        
        self.logger.info(f"Translation validation PASSED for {target_language} ({failed_count} cognate/loanword keys accepted)")
        return True

    def get_translations(self, target_language: str) -> dict:
        """
        Gets translations for UI elements for a given language.
        Uses AI agent and caches the result.
        Supports retry logic (max 2 attempts).
        """
        if target_language == "English":
            self.logger.info("Using default English UI translations.")
            return self.ENGLISH_TRANSLATIONS
        
        with self._translation_cache_lock:
            if target_language in self.translation_cache:
                cached = self.translation_cache[target_language]
                missing_keys = set(self.ENGLISH_TRANSLATIONS.keys()) - set(cached.keys())
                if not missing_keys:
                    if self._validate_translations(cached, target_language):
                        self.logger.info(f"Using cached UI translations for {target_language}.")
                        return cached
                    else:
                        self.logger.warning(f"Cached translations for {target_language} contain English values. Forcing re-translation...")
                        del self.translation_cache[target_language]
                else:
                    self.logger.info(f"Cache for {target_language} is outdated (missing {len(missing_keys)} keys). Refreshing...")
                    del self.translation_cache[target_language]

        self.logger.debug(f"Calling LLM to get UI translations for {target_language}...")
        
        # Retry loop: up to 2 attempts
        for attempt in range(1, 3):
            try:
                if attempt > 1:
                    self.logger.info(f"Retrying UI translation for {target_language} (Attempt {attempt}/2)...")
                
                # Escape braces in JSON payload so they don't interfere with .format()
                json_str = json.dumps(self.ENGLISH_TRANSLATIONS, indent=2)
                # Replace { with {{ and } with }} to escape them for .format()
                json_str_escaped = json_str.replace('{', '{{').replace('}', '}}')
                
                prompt_vars = {
                    "json_payload": json_str_escaped,
                    "target_language": target_language
                }
                
                self.logger.info(f"⏳ Waiting for LLM response (translating UI to {target_language})...")
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Translate_UI_{target_language}_Attempt{attempt}",
                    worker_prompt_path="KEYWORDS_TRANSLATE_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    skip_cache=(attempt > 1)
                )
                self.logger.info(f"✅ Received LLM response, parsing translations...")
                
                response_clean = clean_json_response(response_raw)
                translated_dict = json.loads(response_clean)
                
                final_translations = self.ENGLISH_TRANSLATIONS.copy()
                final_translations.update(translated_dict) 
                
                # Apply known fallback fixes for commonly untranslated terms
                final_translations = self._apply_translation_fallbacks(final_translations, target_language)
                
                # Validate translations before caching
                if not self._validate_translations(final_translations, target_language):
                    if attempt < 2:
                        self.logger.warning(f"Translation validation failed on attempt {attempt}. Will retry...")
                        continue  # Retry
                    else:
                        # Apply fallbacks one more time before giving up
                        final_translations = self._apply_translation_fallbacks(final_translations, target_language)
                        if self._validate_translations(final_translations, target_language):
                            self.logger.info(f"Fallback translations fixed the issue. Using fallbacks.")
                            with self._translation_cache_lock:
                                self.translation_cache[target_language] = final_translations
                            return final_translations
                        self.logger.error(f"Translation validation failed on final attempt. Falling back to English.")
                        return self.ENGLISH_TRANSLATIONS
                
                self.logger.info(f"Successfully fetched and cached UI translations for {target_language} on attempt {attempt}.")
                with self._translation_cache_lock:
                    self.translation_cache[target_language] = final_translations
                return final_translations

            except Exception as e:
                self.logger.error(f"Failed to get UI translations for {target_language} (Attempt {attempt}/2): {get_clean_error_message(e)}")
                if attempt == 2:
                    self.logger.error(f"All retry attempts exhausted for UI translations. Falling back to English.")
                    return self.ENGLISH_TRANSLATIONS

    def translate_use_case_list(self, english_use_cases: list, target_language: str, max_parallelism: int = 20, enable_parallelization: bool = True) -> list:
        """
        Translates the 9 key fields for a list of use case dictionaries.
        Uses dynamic batch sizing to maximize context utilization.
        
        Args:
            english_use_cases: List of use case dictionaries to translate
            target_language: Target language name
            max_parallelism: Maximum parallel workers (only used if enable_parallelization=True)
            enable_parallelization: If False, processes batches sequentially to avoid nested ThreadPoolExecutors
        """
        if target_language == "English":
            return english_use_cases
        
        if not english_use_cases:
             self.logger.warning("No use cases provided to translate.")
             return []

        self.logger.info(f"Starting translation for {target_language}... (parallelization: {'enabled' if enable_parallelization else 'disabled'})")
        
        # FIXED BATCH SIZE: 3 use cases per batch for ALL languages
        # This prevents LLM output truncation issues with long SQL queries (8-10K chars each)
        # Optimal balance between translation speed and reliability
        batch_size = 3
        
        self.logger.info(f"Translation batch sizing: {batch_size} use cases per batch (FIXED for all languages to prevent truncation)")
        
        batches = [english_use_cases[i:i + batch_size] for i in range(0, len(english_use_cases), batch_size)]
        translated_use_cases = []
        
        if enable_parallelization:
            # Parallel processing (used when NOT called from within another ThreadPoolExecutor)
            with _SafeExecutorContext(max_workers=max_parallelism, thread_name_prefix="Translator", logger=self.logger, name="Translation") as ctx:
                futures = [ctx.submit(self._translate_batch, batch, target_language, i) for i, batch in enumerate(batches)]
                
                total_timeout = (len(batches) * 600) // max_parallelism + 300
                self.logger.info(f"Translation timeout set to {total_timeout}s for {len(batches)} batches")
                
                try:
                    for future in safe_as_completed(futures, timeout=total_timeout):
                        try:
                            translated_batch = future.result(timeout=600)
                            if translated_batch:
                                translated_use_cases.extend(translated_batch)
                        except concurrent.futures.TimeoutError:
                            self.logger.error(f"Translation batch timed out after 10 minutes")
                        except Exception as e:
                            self.logger.error(f"A translation future failed unexpectedly: {get_clean_error_message(e)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Overall translation timeout reached ({total_timeout}s). Proceeding with {len(translated_use_cases)} translated use cases.")
        else:
            # Sequential processing (used when called from within another ThreadPoolExecutor to avoid nesting)
            self.logger.info(f"Processing {len(batches)} translation batches sequentially to avoid nested ThreadPoolExecutors...")
            for i, batch in enumerate(batches):
                try:
                    translated_batch = self._translate_batch(batch, target_language, i)
                    if translated_batch:
                        translated_use_cases.extend(translated_batch)
                except Exception as e:
                    self.logger.error(f"Translation batch {i} failed: {get_clean_error_message(e)}")
        
        if len(translated_use_cases) != len(english_use_cases):
            self.logger.warning(f"Translation mismatch: expected {len(english_use_cases)} use cases, translated {len(translated_use_cases)}. Some batches may have failed and reverted to English.")
        
        # Gap G6 Q1d identifier-fidelity: translated UCs must preserve Primary Table / Tables Involved / result_table verbatim.
        try:
            eng_by_no = {str(u.get("No", "")): u for u in english_use_cases if u.get("No") is not None}
            drift_count = 0
            for tu in translated_use_cases:
                no = str(tu.get("No", ""))
                orig = eng_by_no.get(no)
                if not orig:
                    continue
                for fld in ("Primary Table", "Tables Involved", "result_table"):
                    e_val = str(orig.get(fld, "") or "").strip()
                    t_val = str(tu.get(fld, "") or "").strip()
                    if e_val and t_val and e_val != t_val:
                        drift_count += 1
                        try:
                            NEXT_ACTIONS.add(
                                rule_id="Q1D_TRANSLATION_IDENT_DRIFT", severity="HIGH",
                                phase="Q1d", step=f"translate[{target_language}]",
                                evidence=f"UC {no} field {fld!r}: en={e_val[:80]!r} vs {target_language}={t_val[:80]!r}",
                                suggested_user_action="Review translation prompt; identifier fields must be copied verbatim.",
                            )
                        except Exception:
                            pass
                        tu[fld] = orig.get(fld)
            if drift_count:
                self.logger.warning(f"Q1d: repaired {drift_count} identifier drift(s) in {target_language} translations")
        except Exception as _qd:
            self.logger.debug(f"Q1d fidelity check skipped (non-fatal): {_qd}")
        
        self.logger.info(f"Translation completed for {target_language}.")
        return translated_use_cases

    def _translate_batch(self, use_case_batch: list, target_language: str, batch_num: int, attempt: int = 1, split_attempt: int = 0) -> list:
        """
        Private method to translate a single batch of use cases.
        Returns the original English batch on any failure.
        Supports retry logic with automatic batch splitting on truncation (up to 3 split attempts).
        
        Args:
            use_case_batch: List of use case dictionaries to translate
            target_language: Target language for translation
            batch_num: Batch number for logging
            attempt: Current attempt number (1 or 2)
            split_attempt: Number of times batch has been split (0-3)
        """
        self.logger.debug(f"Translating batch #{batch_num} (Attempt {attempt}/2, Split {split_attempt}/3) ({len(use_case_batch)} use cases) to {target_language}...")
        
        try:
            _TRANSLATION_FIELDS = {
                "No", "Name", "Business Domain", "Subdomain", "type",
                "Analytics Technique", "Statement", "Solution",
                "Business Value", "Beneficiary", "Sponsor",
                "Business Priority Alignment", "Strategic Goals Alignment",
                "Tables Involved", "Quality",
            }
            batch_for_translation = []
            for uc in use_case_batch:
                uc_slim = {k: v for k, v in uc.items() if k in _TRANSLATION_FIELDS}
                batch_for_translation.append(uc_slim)
            
            json_str = json.dumps(batch_for_translation, indent=2)
            json_str_escaped = json_str.replace('{', '{{').replace('}', '}}')
            
            prompt_vars = {
                "json_payload": json_str_escaped,
                "target_language": target_language
            }
            
            self.logger.info(f"⏳ [Batch {batch_num}] Waiting for LLM response (translating {len(use_case_batch)} use cases to {target_language})...")
            response_raw = self.ai_agent.run_worker(
                step_name=f"Translate_UseCases_{target_language}_Batch_{batch_num}_Attempt{attempt}_Split{split_attempt}",
                worker_prompt_path="USE_CASE_TRANSLATE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                skip_cache=(attempt > 1)
            )
            self.logger.info(f"✅ [Batch {batch_num}] Received LLM response, parsing translated use cases...")
            
            # Parse CSV response
            translated_batch = self._parse_translation_csv(response_raw, batch_for_translation, batch_num, target_language)
            
            if translated_batch and len(translated_batch) == len(use_case_batch):
                self.logger.debug(f"Successfully translated batch #{batch_num} on attempt {attempt}, split {split_attempt}.")
                return translated_batch
            else:
                raise ValueError(f"Translation returned {len(translated_batch) if translated_batch else 0} rows, expected {len(use_case_batch)}")

        except InputTooLongError as e:
            # Handle context limit exceeded - split batch recursively
            if len(use_case_batch) <= 1:
                # Cannot split further - single use case is too large
                self.logger.error(f"Batch #{batch_num} has single use case that's too large for translation. Reverting to English.")
                return use_case_batch
            
            # Split into 2 sub-batches
            self.logger.warning(f"Batch #{batch_num} exceeds context limit ({str(e)[:100]}). Splitting into smaller sub-batches...")
            mid = len(use_case_batch) // 2
            sub_batch_1 = use_case_batch[:mid]
            sub_batch_2 = use_case_batch[mid:]
            
            # Recursively translate sub-batches (with attempt=1 for fresh retries, increment split_attempt)
            translated_1 = self._translate_batch(sub_batch_1, target_language, f"{batch_num}a", attempt=1, split_attempt=split_attempt + 1)
            translated_2 = self._translate_batch(sub_batch_2, target_language, f"{batch_num}b", attempt=1, split_attempt=split_attempt + 1)
            
            # Combine results
            combined = translated_1 + translated_2
            if len(combined) == len(use_case_batch):
                self.logger.info(f"Successfully translated batch #{batch_num} after splitting into sub-batches")
                return combined
            else:
                self.logger.warning(f"Sub-batch splitting failed for batch #{batch_num}. Reverting to English.")
                return use_case_batch
        
        except Exception as e:
            error_msg = str(e)
            is_truncation = "TRUNCATED" in error_msg
            
            self.logger.error(f"Failed to translate use case batch #{batch_num} (Attempt {attempt}/2, Split {split_attempt}/3) for {target_language}: {get_clean_error_message(e)}")
            
            # If truncation detected and batch has more than 1 item, try splitting it (up to 3 times)
            if is_truncation and len(use_case_batch) > 1 and split_attempt < 3:
                self.logger.warning(f"Batch #{batch_num} appears truncated. Reducing batch size (split attempt {split_attempt + 1}/3)...")
                
                # Calculate smaller batch size (split into 2 or 3 pieces depending on batch size)
                if len(use_case_batch) >= 3:
                    # Split into 3 smaller pieces for better chance of success
                    third = len(use_case_batch) // 3
                    sub_batch_1 = use_case_batch[:third]
                    sub_batch_2 = use_case_batch[third:2*third]
                    sub_batch_3 = use_case_batch[2*third:]
                    
                    # Recursively translate sub-batches with incremented split_attempt
                    translated_1 = self._translate_batch(sub_batch_1, target_language, f"{batch_num}a", attempt=1, split_attempt=split_attempt + 1)
                    translated_2 = self._translate_batch(sub_batch_2, target_language, f"{batch_num}b", attempt=1, split_attempt=split_attempt + 1)
                    translated_3 = self._translate_batch(sub_batch_3, target_language, f"{batch_num}c", attempt=1, split_attempt=split_attempt + 1)
                    
                    # Combine results
                    combined = translated_1 + translated_2 + translated_3
                else:
                    # Split into 2 pieces
                    mid = len(use_case_batch) // 2
                    sub_batch_1 = use_case_batch[:mid]
                    sub_batch_2 = use_case_batch[mid:]
                    
                    translated_1 = self._translate_batch(sub_batch_1, target_language, f"{batch_num}a", attempt=1, split_attempt=split_attempt + 1)
                    translated_2 = self._translate_batch(sub_batch_2, target_language, f"{batch_num}b", attempt=1, split_attempt=split_attempt + 1)
                    
                    combined = translated_1 + translated_2
                
                if len(combined) == len(use_case_batch):
                    self.logger.info(f"Successfully translated batch #{batch_num} after reducing batch size (split {split_attempt + 1})")
                    return combined
                else:
                    self.logger.warning(f"Batch size reduction failed for batch #{batch_num}. Trying standard retry...")
                    # Fall through to standard retry logic
            
            # Standard retry logic: Try one more time if this is the first attempt and we haven't exceeded split attempts
            if attempt < 2 and not (is_truncation and split_attempt >= 3):
                self.logger.info(f"Retrying batch #{batch_num} for {target_language} (Attempt 2/2)...")
                return self._translate_batch(use_case_batch, target_language, batch_num, attempt=2, split_attempt=split_attempt)
            else:
                # All attempts exhausted
                if is_truncation and split_attempt >= 3:
                    self.logger.error(f"Batch #{batch_num} still truncated after {split_attempt} split attempts. Proceeding with English.")
                else:
                    self.logger.warning(f"All retry attempts exhausted for batch #{batch_num}. Reverting to English for this batch.")
                return use_case_batch
    
    def _parse_translation_csv(self, csv_response: str, original_batch: list, batch_num: int, target_language: str) -> list:
        """
        Parse CSV translation response and return list of translated use case dictionaries.
        Only returns rows that match the original batch by 'No' field to prevent incorrect row counts.
        """
        import re
        
        try:
            # VALIDATION 1: Check if response is empty or too short
            if not csv_response or len(csv_response.strip()) < 100:
                raise ValueError(f"Response is empty or too short (len={len(csv_response) if csv_response else 0})")
            
            # VALIDATION 2: Check if response contains obvious non-CSV content
            first_100_chars = csv_response[:100].lower()
            forbidden_starts = ['here is', 'i have', 'i\'ve', 'the translation', 'below is', 'sure,', 'of course']
            if any(first_100_chars.startswith(phrase) for phrase in forbidden_starts):
                self.logger.error(f"Batch #{batch_num}: Response starts with conversational text instead of CSV header")
                raise ValueError("Response contains conversational text instead of pure CSV")
            
            # Clean response - remove markdown fences if present
            csv_clean = csv_response.strip()
            if csv_clean.startswith('```'):
                csv_clean = re.sub(r'^```[a-z]*\n', '', csv_clean)
                csv_clean = re.sub(r'\n```$', '', csv_clean)
            
            # VALIDATION 3: Check for SQL code blocks that shouldn't be there
            # Note: This validation is informational only and doesn't block processing
            sql_pattern_count = csv_clean.count('SELECT ') + csv_clean.count('WITH ') + csv_clean.count('FROM ')
            # SQL should only appear in the SQL column, so count should be reasonable (approx. number of rows)
            expected_sql_mentions = len(original_batch)
            if sql_pattern_count > (expected_sql_mentions * 3):  # Allow some margin
                # Debug-level logging instead of warning (reduces noise)
                self.logger.debug(f"Batch #{batch_num}: Response contains {sql_pattern_count} SQL keywords (expected ~{expected_sql_mentions}). This is normal for complex queries.")
            
            header_patterns = [
                (r'"No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Strategic Goals Alignment","Tables Involved","Quality"', "15-col quoted with Strategic Goals"),
                (r'No,Name,Business Domain,Subdomain,type,Analytics Technique,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Strategic Goals Alignment,Tables Involved,Quality', "15-col unquoted with Strategic Goals"),
                (r'"No","Name","Business Domain","Subdomain","type","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Strategic Goals Alignment","Tables Involved","Quality"', "14-col quoted Quality with Strategic Goals (no Analytics Technique)"),
                (r'No,Name,Business Domain,Subdomain,type,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Strategic Goals Alignment,Tables Involved,Quality', "14-col unquoted Quality with Strategic Goals (no Analytics Technique)"),
                (r'"No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Tables Involved","Quality"', "14-col quoted Quality"),
                (r'No,Name,Business Domain,Subdomain,type,Analytics Technique,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Tables Involved,Quality', "14-col unquoted Quality"),
                (r'"No","Name","Business Domain","Subdomain","type","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Tables Involved","Quality"', "13-col quoted Quality (no Analytics Technique)"),
                (r'No,Name,Business Domain,Subdomain,type,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Tables Involved,Quality', "13-col unquoted Quality (no Analytics Technique)"),
                (r'"No","Name","Business Domain","Subdomain","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Tables Involved","Priority"', "14-col quoted Priority"),
                (r'No,Name,Business Domain,Subdomain,type,Analytics Technique,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Tables Involved,Priority', "14-col unquoted Priority"),
                (r'"No","Name","Business Domain","Subdomain","type","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Strategic Goals Alignment","Tables Involved","Priority"', "14-col quoted Priority with Strategic Goals (no Analytics Technique)"),
                (r'No,Name,Business Domain,Subdomain,type,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Strategic Goals Alignment,Tables Involved,Priority', "14-col unquoted Priority with Strategic Goals (no Analytics Technique)"),
                (r'"No","Name","Business Domain","Subdomain","type","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Tables Involved","Priority"', "13-col quoted Priority (no Analytics Technique)"),
                (r'No,Name,Business Domain,Subdomain,type,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Tables Involved,Priority', "13-col unquoted Priority (no Analytics Technique)"),
                (r'"No","Name","Business Domain","Subdomain","type","Statement","Solution","Business Value","Beneficiary","Sponsor","Business Priority Alignment","Tables Involved","Complexity","Quality"', "legacy 14-col Complexity+Quality"),
                (r'No,Name,Business Domain,Subdomain,type,Statement,Solution,Business Value,Beneficiary,Sponsor,Business Priority Alignment,Tables Involved,Complexity,Quality', "legacy 14-col unquoted Complexity+Quality"),
            ]
            header_match = None
            matched_format = None
            for pattern, fmt_label in header_patterns:
                header_match = re.search(pattern, csv_clean)
                if header_match:
                    matched_format = fmt_label
                    self.logger.debug(f"Batch #{batch_num}: Matched header format: {fmt_label}")
                    break
            if not header_match:
                first_line = csv_clean.split('\n')[0][:200] if csv_clean else "Empty"
                self.logger.error(f"Batch #{batch_num}: CSV header not found. First line: {first_line}")
                raise ValueError(f"Could not find CSV header. First line was: {first_line}")
            
            # Extract CSV starting from header
            csv_data = csv_clean[header_match.start():]
            
            # Build a set of expected IDs from the original batch
            expected_ids = {uc.get('No', '').strip() for uc in original_batch}
            self.logger.debug(f"Expected IDs for batch #{batch_num}: {expected_ids}")
            
            # Parse CSV using centralized utility
            parsed_rows = CSVParser.parse_csv_string(
                csv_data,
                logger=self.logger,
                context=f"Batch #{batch_num}",
                quoting=csv.QUOTE_ALL
            )
            all_parsed_rows = []
            translated_rows = []
            
            for row_dict in parsed_rows:
                # Clean up field values - handle both string and list values
                cleaned_row = {}
                for k, v in row_dict.items():
                    # Ensure k is a string (handle edge cases where CSV parser returns unexpected types)
                    key = str(k) if not isinstance(k, str) else k
                    
                    # Handle different value types robustly
                    if v is None:
                        cleaned_row[key] = ""
                    elif isinstance(v, list):
                        # If value is a list (shouldn't happen but handle it), join it
                        cleaned_row[key] = ', '.join(str(item) for item in v)
                    elif isinstance(v, str):
                        cleaned_row[key] = v.strip()
                    else:
                        cleaned_row[key] = str(v)
                
                row_id = cleaned_row.get('No', '').strip()
                
                if not row_id:
                    self.logger.debug(f"Batch #{batch_num}: Skipping row with empty ID")
                    continue
                
                all_parsed_rows.append(cleaned_row)
                
                if expected_ids and row_id not in expected_ids:
                    row_id_preview = row_id[:50] if len(row_id) > 50 else row_id
                    self.logger.warning(f"Batch #{batch_num}: Skipping unexpected row with ID '{row_id_preview}' (not in batch)")
                    continue
                
                translated_rows.append(cleaned_row)
            
            if not translated_rows and all_parsed_rows:
                has_analytics_technique = matched_format and 'Analytics Technique' in matched_format
                if has_analytics_technique:
                    _COLUMN_SHIFT_MAP = {
                        'No': 'Name',
                        'Name': 'Business Domain',
                        'Business Domain': 'Subdomain',
                        'Subdomain': 'type',
                        'type': 'Analytics Technique',
                        'Analytics Technique': 'Statement',
                        'Statement': 'Solution',
                        'Solution': 'Business Value',
                        'Business Value': 'Beneficiary',
                        'Beneficiary': 'Sponsor',
                        'Sponsor': 'Business Priority Alignment',
                        'Business Priority Alignment': 'Strategic Goals Alignment',
                        'Strategic Goals Alignment': 'Tables Involved',
                        'Tables Involved': 'Quality',
                    }
                else:
                    _COLUMN_SHIFT_MAP = {
                        'No': 'Name',
                        'Name': 'Business Domain',
                        'Business Domain': 'Subdomain',
                        'Subdomain': 'type',
                        'type': 'Statement',
                        'Statement': 'Solution',
                        'Solution': 'Business Value',
                        'Business Value': 'Beneficiary',
                        'Beneficiary': 'Sponsor',
                        'Sponsor': 'Business Priority Alignment',
                        'Business Priority Alignment': 'Strategic Goals Alignment',
                        'Strategic Goals Alignment': 'Tables Involved',
                        'Tables Involved': 'Quality',
                    }

                shift_candidates = [
                    row for row in all_parsed_rows
                    if row.get('No', '').strip().lower() == 'no'
                    and row.get('Name', '').strip() in expected_ids
                ]

                if shift_candidates:
                    self.logger.warning(
                        f"Batch #{batch_num}: Detected column shift in {len(shift_candidates)} rows "
                        f"(LLM put literal 'No' in ID column). Correcting..."
                    )
                    orig_lookup = {uc.get('No', '').strip(): uc for uc in original_batch}

                    for row in shift_candidates:
                        corrected = {}
                        for target_col, source_col in _COLUMN_SHIFT_MAP.items():
                            val = row.get(source_col, '')
                            corrected[target_col] = val.strip() if isinstance(val, str) else str(val)

                        corrected_id = corrected.get('No', '').strip()
                        if corrected_id in expected_ids:
                            orig_uc = orig_lookup.get(corrected_id, {})
                            corrected['Quality'] = orig_uc.get('Quality', 'High')
                            corrected['Tables Involved'] = orig_uc.get('Tables Involved', corrected.get('Tables Involved', ''))
                            corrected['Analytics Technique'] = orig_uc.get('Analytics Technique', corrected.get('Analytics Technique', ''))
                            translated_rows.append(corrected)
                            self.logger.info(f"Batch #{batch_num}: Corrected column shift for row {corrected_id}")
                        else:
                            self.logger.warning(f"Batch #{batch_num}: Column shift correction failed for row (corrected ID: '{corrected_id}')")

            if len(all_parsed_rows) > len(original_batch):
                self.logger.warning(f"Batch #{batch_num}: CSV response contained {len(all_parsed_rows)} rows, but only {len(translated_rows)} matched the expected batch IDs. "
                                  f"LLM may have returned extra rows - filtered to correct batch.")
            
            translated_ids = {row.get('No', '').strip() for row in translated_rows}
            missing_ids = expected_ids - translated_ids
            if missing_ids:
                self.logger.error(f"Batch #{batch_num}: Missing translations for IDs: {missing_ids}")
                
                # Check if response appears truncated (ends mid-sentence or mid-field)
                last_100_chars = csv_response[-100:].strip() if csv_response else ""
                is_truncated = (
                    not last_100_chars.endswith('"') or  # Doesn't end with closing quote
                    '","' in last_100_chars[-20:] or  # Ends mid-field
                    len(last_100_chars) < 50  # Response is suspiciously short
                )
                
                if is_truncated:
                    self.logger.error(f"Batch #{batch_num}: Response appears TRUNCATED. Last 100 chars: ...{last_100_chars}")
                    raise ValueError(f"Translation response was TRUNCATED - missing {len(missing_ids)} rows. Reduce batch size or simplify content.")
                else:
                    raise ValueError(f"Translation missing {len(missing_ids)} expected rows: {missing_ids}")
            
            _quality_labels = {'very high', 'high', 'medium', 'low', 'very low'}
            for row in translated_rows:
                if 'Priority' in row and 'Quality' not in row:
                    pval = str(row['Priority']).strip().lower()
                    if pval in _quality_labels:
                        row['Quality'] = row.pop('Priority')
                        self.logger.debug(f"Batch #{batch_num}: Remapped 'Priority' → 'Quality' for row {row.get('No', '?')}")
            
            _NON_LATIN_LANGUAGES = {'Arabic', 'Chinese (Mandarin)', 'Chinese (Simplified)', 'Japanese', 'Korean', 'Hindi', 'Russian'}
            if target_language in _NON_LATIN_LANGUAGES:
                _MUST_TRANSLATE_FIELDS = ['Statement', 'Solution', 'Business Value', 'Beneficiary', 'Strategic Goals Alignment']
                for row in translated_rows:
                    row_id = row.get('No', '?')
                    for field in _MUST_TRANSLATE_FIELDS:
                        val = row.get(field, '')
                        if val and len(val) > 10:
                            ascii_ratio = sum(1 for c in val if ord(c) < 128) / len(val)
                            if ascii_ratio > 0.85:
                                self.logger.warning(f"Batch #{batch_num}: Row {row_id} field '{field}' appears untranslated for {target_language} (ASCII ratio: {ascii_ratio:.0%}). Content: {val[:80]}...")
            
            self.logger.debug(f"Parsed {len(translated_rows)} translated rows from CSV for batch #{batch_num} (filtered from {len(all_parsed_rows)} total rows)")
            return translated_rows
            
        except Exception as e:
            error_msg = str(e)
            self.logger.error(f"Failed to parse translation CSV for batch #{batch_num}: {get_clean_error_message(e)}")
            snippet = csv_response[:500] if csv_response else "Empty response"
            self.logger.error(f"CSV response snippet: {snippet}")
            if "TRUNCATED" in error_msg:
                raise
            return []

# COMMAND ----------

# DBTITLE 1,Inspire
# ==============================================================================
# 2. MAIN APPLICATION CLASS (MODIFIED FOR STRICT ORDERING)
# ==============================================================================

# NOTE: Global dependency checks have been removed per user request.
# Dependencies will be checked and installed on-demand.

# ==============================================================================
# MEMORY-EFFICIENT STORAGE MANAGER
# ==============================================================================
class IntermediateStorageManager:
    """
    Manages file-based intermediate storage for large datasets to prevent memory explosion.
    Stores batch results on disk and provides memory-efficient iteration.
    """
    
    def __init__(self, base_path="/tmp", logger=None):
        """
        Initialize the storage manager.
        
        Args:
            base_path: Base path for temporary storage (default: /tmp)
            logger: Logger instance
        """
        self.logger = logger or logging.getLogger(self.__class__.__name__)
        self.base_path = base_path
        self.temp_dir = None
        self.batch_files = []
        self.initialized = False
        
    def initialize(self):
        """Create temporary directory for intermediate storage."""
        if not self.initialized:
            self.temp_dir = tempfile.mkdtemp(prefix="inspire_", dir=self.base_path)
            self.logger.info(f"📁 Initialized intermediate storage at: {self.temp_dir}")
            self.initialized = True
            
    def save_batch(self, batch_num, data):
        """
        Save batch data to disk.
        
        Args:
            batch_num: Batch identifier (can be int or str)
            data: List of use case dictionaries
        """
        if not self.initialized:
            self.initialize()
            
        # Handle both int and string batch_num
        if isinstance(batch_num, int):
            batch_file = os.path.join(self.temp_dir, f"batch_{batch_num:04d}.json")
        else:
            batch_file = os.path.join(self.temp_dir, f"batch_{batch_num}.json")
        with open(batch_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        self.batch_files.append(batch_file)
        file_size = os.path.getsize(batch_file) / (1024 * 1024)  # Size in MB
        self.logger.debug(f"💾 Saved batch {batch_num} to disk ({len(data)} use cases, {file_size:.2f} MB)")
        
    def load_batch(self, batch_file):
        """Load a single batch from disk."""
        with open(batch_file, 'r', encoding='utf-8') as f:
            return json.load(f)
            
    def iter_batches(self):
        """
        Iterator over all batches.
        Memory-efficient: loads one batch at a time.
        """
        for batch_file in self.batch_files:
            yield self.load_batch(batch_file)
            
    def iter_all_use_cases(self):
        """
        Iterator over all use cases across all batches.
        Memory-efficient: loads one batch at a time and yields individual use cases.
        """
        for batch in self.iter_batches():
            for use_case in batch:
                yield use_case
                
    def load_all_use_cases(self):
        """
        Load all use cases into memory.
        Use this only when necessary (e.g., for deduplication).
        """
        all_use_cases = []
        for batch in self.iter_batches():
            all_use_cases.extend(batch)
        return all_use_cases
        
    def get_total_count(self):
        """Get total count of use cases without loading all into memory."""
        count = 0
        for batch in self.iter_batches():
            count += len(batch)
        return count
        
    def cleanup(self):
        """Remove temporary directory and all files."""
        if self.temp_dir and os.path.exists(self.temp_dir):
            try:
                shutil.rmtree(self.temp_dir)
                self.logger.info(f"🧹 Cleaned up intermediate storage: {self.temp_dir}")
            except Exception as e:
                self.logger.warning(f"Failed to cleanup temp directory {self.temp_dir}: {get_clean_error_message(e)}")
            finally:
                self.temp_dir = None
                self.batch_files = []
                self.initialized = False
                
    def get_stats(self):
        """Get storage statistics."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return {"initialized": False}
            
        total_size = 0
        for batch_file in self.batch_files:
            if os.path.exists(batch_file):
                total_size += os.path.getsize(batch_file)
                
        return {
            "initialized": True,
            "temp_dir": self.temp_dir,
            "num_batches": len(self.batch_files),
            "total_size_mb": total_size / (1024 * 1024),
            "use_case_count": self.get_total_count()
        }
    
    def save_schema_details(self, data: list, name: str = "all_columns_for_sql"):
        """Save schema column details to disk to free memory."""
        if not self.initialized:
            self.initialize()
        schema_file = os.path.join(self.temp_dir, f"{name}.json")
        serializable = [list(t) for t in data]
        with open(schema_file, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, ensure_ascii=False)
        size_mb = os.path.getsize(schema_file) / (1024 * 1024)
        self.logger.debug(f"💾 Saved {name} to disk ({len(data)} rows, {size_mb:.2f} MB)")
    
    def load_schema_details(self, name: str = "all_columns_for_sql") -> list:
        """Load schema column details from disk."""
        if not self.temp_dir:
            return []
        schema_file = os.path.join(self.temp_dir, f"{name}.json")
        if not os.path.exists(schema_file):
            return []
        with open(schema_file, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        return [tuple(t) for t in raw]

    def save_scored_use_cases(self, data, tag: str = "combined_scored"):
        if not self.initialized:
            self.initialize()
        path = os.path.join(self.temp_dir, f"{tag}.json")
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=1)
        self.logger.debug(f"Saved {len(data)} scored use cases to disk ({tag})")

    def load_scored_use_cases(self, tag: str = "combined_scored"):
        if not self.temp_dir:
            return None
        path = os.path.join(self.temp_dir, f"{tag}.json")
        if not os.path.exists(path):
            return None
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def save_schema_markdown(self, markdown: str, tag: str = "schema_markdown"):
        if not self.initialized:
            self.initialize()
        path = os.path.join(self.temp_dir, f"{tag}.txt")
        with open(path, 'w', encoding='utf-8') as f:
            f.write(markdown)
        self.logger.debug(f"Saved schema markdown to disk ({len(markdown):,} chars)")

    def load_schema_markdown(self, tag: str = "schema_markdown"):
        if not self.temp_dir:
            return None
        path = os.path.join(self.temp_dir, f"{tag}.txt")
        if not os.path.exists(path):
            return None
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()

    def save_column_tracking(self, fqtn: str, column_names: list):
        """
        Save column tracking for a table to disk.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            column_names: List of column names that were kept
        """
        if not self.initialized:
            self.initialize()
        
        # Create column_tracking subdirectory if it doesn't exist
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        os.makedirs(tracking_dir, exist_ok=True)
        
        # Use a safe filename (replace dots and special chars)
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        
        with open(tracking_file, 'w', encoding='utf-8') as f:
            json.dump({
                "fqtn": fqtn,
                "columns": column_names,
                "timestamp": datetime.datetime.now().isoformat()
            }, f, ensure_ascii=False, indent=2)
        
        self.logger.debug(f"💾 Saved column tracking for {fqtn}: {len(column_names)} columns")
    
    def load_column_tracking(self, fqtn: str) -> list:
        """
        Load column tracking for a table from disk.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            
        Returns:
            List of column names that were kept, or None if no tracking exists
        """
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return None
        
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        if not os.path.exists(tracking_dir):
            return None
        
        # Use a safe filename (replace dots and special chars)
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        
        if not os.path.exists(tracking_file):
            return None
        
        try:
            with open(tracking_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data.get("columns", None)
        except Exception as e:
            self.logger.warning(f"Failed to load column tracking for {fqtn}: {get_clean_error_message(e)}")
            return None
    
    def has_column_tracking(self, fqtn: str) -> bool:
        """
        Check if column tracking exists for a table.
        
        Args:
            fqtn: Fully qualified table name (catalog.schema.table)
            
        Returns:
            True if tracking exists, False otherwise
        """
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return False
        
        tracking_dir = os.path.join(self.temp_dir, "column_tracking")
        if not os.path.exists(tracking_dir):
            return False
        
        safe_filename = fqtn.replace('.', '_').replace('`', '') + ".json"
        tracking_file = os.path.join(tracking_dir, safe_filename)
        return os.path.exists(tracking_file)
    
    def save_pass1_ids(self, use_case_ids: list):
        """Save PASS 1 use case IDs to disk for memory-efficient comparison."""
        if not self.initialized:
            self.initialize()
        ids_file = os.path.join(self.temp_dir, "pass1_ids.json")
        with open(ids_file, 'w', encoding='utf-8') as f:
            json.dump(use_case_ids, f)
        self.logger.debug(f"💾 Saved {len(use_case_ids)} PASS 1 IDs to disk")
    
    def load_pass1_ids(self) -> set:
        """Load PASS 1 use case IDs from disk as a set for O(1) lookup."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return set()
        ids_file = os.path.join(self.temp_dir, "pass1_ids.json")
        if not os.path.exists(ids_file):
            return set()
        try:
            with open(ids_file, 'r', encoding='utf-8') as f:
                return set(json.load(f))
        except Exception as e:
            self.logger.warning(f"Failed to load PASS 1 IDs: {get_clean_error_message(e)}")
            return set()
    
    def save_feedback_file(self, feedback_lines: list):
        """Save feedback string to disk to avoid keeping large string in memory."""
        if not self.initialized:
            self.initialize()
        feedback_file = os.path.join(self.temp_dir, "pass1_feedback.txt")
        with open(feedback_file, 'w', encoding='utf-8') as f:
            f.write("\n".join(feedback_lines))
        self.logger.debug(f"💾 Saved feedback to disk ({len(feedback_lines)} lines)")
    
    def load_feedback_file(self) -> str:
        """Load feedback string from disk."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return ""
        feedback_file = os.path.join(self.temp_dir, "pass1_feedback.txt")
        if not os.path.exists(feedback_file):
            return ""
        try:
            with open(feedback_file, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            self.logger.warning(f"Failed to load feedback file: {get_clean_error_message(e)}")
            return ""
    
    def iter_pass1_use_cases_for_feedback(self, limit: int = 200):
        """
        Memory-efficient iterator for building feedback.
        Yields (idx, name, tables) tuples from PASS 1 batches without loading all into memory.
        """
        idx = 0
        for batch in self.iter_batches():
            for uc in batch:
                if idx >= limit:
                    return
                idx += 1
                name = str(uc.get('Name', '')).replace('|', '-')[:80]
                tables = str(uc.get('Tables Involved', '')).replace('|', '-')[:60]
                yield (idx, name, tables)
    
    def save_domain_results(self, domain_name: str, use_cases: list):
        """Save completed domain use cases to disk to free memory during domain-by-domain processing."""
        if not self.initialized:
            self.initialize()
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        os.makedirs(domain_dir, exist_ok=True)
        safe_name = re.sub(r'[^\w\-]', '_', domain_name)
        domain_file = os.path.join(domain_dir, f"{safe_name}.json")
        with open(domain_file, 'w', encoding='utf-8') as f:
            json.dump(use_cases, f, ensure_ascii=False)
        size_mb = os.path.getsize(domain_file) / (1024 * 1024)
        self.logger.debug(f"💾 Saved domain '{domain_name}' results to disk ({len(use_cases)} use cases, {size_mb:.2f} MB)")

    def load_domain_results(self, domain_name: str) -> list:
        """Load domain use cases from disk."""
        if not self.temp_dir:
            return []
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        safe_name = re.sub(r'[^\w\-]', '_', domain_name)
        domain_file = os.path.join(domain_dir, f"{safe_name}.json")
        if not os.path.exists(domain_file):
            return []
        with open(domain_file, 'r', encoding='utf-8') as f:
            return json.load(f)

    def load_all_domain_results(self) -> list:
        """Load all domain results from disk in one pass."""
        if not self.temp_dir:
            return []
        domain_dir = os.path.join(self.temp_dir, "domain_results")
        if not os.path.exists(domain_dir):
            return []
        all_results = []
        for filename in sorted(os.listdir(domain_dir)):
            if filename.endswith('.json'):
                filepath = os.path.join(domain_dir, filename)
                with open(filepath, 'r', encoding='utf-8') as f:
                    all_results.extend(json.load(f))
        return all_results

    def save_phase_data(self, phase_name: str, data):
        """Save arbitrary phase data to disk for cross-phase handoff without holding in memory."""
        if not self.initialized:
            self.initialize()
        phase_file = os.path.join(self.temp_dir, f"phase_{phase_name}.json")
        with open(phase_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)

    def load_phase_data(self, phase_name: str):
        """Load phase data from disk."""
        if not self.temp_dir:
            return None
        phase_file = os.path.join(self.temp_dir, f"phase_{phase_name}.json")
        if not os.path.exists(phase_file):
            return None
        with open(phase_file, 'r', encoding='utf-8') as f:
            return json.load(f)

    def save_id_maps(self, column_id_map: dict, id_column_map: dict, table_id_map: dict, id_table_map: dict):
        """Save column/table ID maps to disk to reduce memory footprint."""
        if not self.initialized:
            self.initialize()
        maps_file = os.path.join(self.temp_dir, "id_maps.json")
        data = {
            "column_id_map": column_id_map,
            "id_column_map": id_column_map,
            "table_id_map": table_id_map,
            "id_table_map": id_table_map
        }
        with open(maps_file, 'w', encoding='utf-8') as f:
            json.dump(data, f)
        self.logger.debug(f"💾 Saved ID maps to disk (columns: {len(column_id_map)}, tables: {len(table_id_map)})")
    
    def load_id_maps(self) -> tuple:
        """Load ID maps from disk. Returns (column_id_map, id_column_map, table_id_map, id_table_map)."""
        if not self.temp_dir or not os.path.exists(self.temp_dir):
            return {}, {}, {}, {}
        maps_file = os.path.join(self.temp_dir, "id_maps.json")
        if not os.path.exists(maps_file):
            return {}, {}, {}, {}
        try:
            with open(maps_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return (data.get("column_id_map", {}), data.get("id_column_map", {}),
                    data.get("table_id_map", {}), data.get("id_table_map", {}))
        except Exception as e:
            self.logger.warning(f"Failed to load ID maps: {get_clean_error_message(e)}")
            return {}, {}, {}, {}

class AtomicWriter:
    def __init__(self, spark, logger, inspire_database: str, session_id: int, widget_values: dict = None):
        import os
        import threading
        import tempfile
        self.spark = spark
        self.logger = logger
        self.inspire_database = (inspire_database or "").strip()
        self.session_id = int(session_id)
        self.widget_values = widget_values or {}
        self.enabled = bool(self.inspire_database)
        self.session_table = f"{self.inspire_database}.__inspire_session" if self.enabled else ""
        self.steps_table = f"{self.inspire_database}.__inspire_step" if self.enabled else ""
        self.lock_table = f"{self.inspire_database}.__inspire_lock" if self.enabled else ""
        self._lock = threading.Lock()
        self._flush_lock = threading.Lock()
        self._flush_thread = None
        self._last_flush_attempt_epoch = 0.0
        self._flush_interval_seconds = 2.0
        self._step_counter = 0
        self.console_step_logging_enabled = str(self.widget_values.get("atomic_writer_console_logs", "false")).strip().lower() in {"1", "true", "yes", "y"}
        spool_dir = os.path.join(tempfile.gettempdir(), "inspire_atomic_writer")
        os.makedirs(spool_dir, exist_ok=True)
        self._spool_path = os.path.join(spool_dir, f"session_{self.session_id}.jsonl")
        self._spool_offset = 0
        if os.path.exists(self._spool_path):
            self._spool_offset = os.path.getsize(self._spool_path)

    def _console_log(self, message: str, level: str = "INFO", force: bool = False):
        if not force and level not in {"WARNING", "ERROR"} and not self.console_step_logging_enabled:
            return
        try:
            log_print(f"[AtomicWriter][session={self.session_id}] {message}", level=level)
        except Exception:
            pass

    def _utc_now_str(self):
        from datetime import datetime, timezone
        return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    def _safe_json_str(self, payload):
        import json
        try:
            return json.dumps(payload if payload is not None else {}, ensure_ascii=True)
        except Exception:
            return "{}"

    def _escape_sql_str(self, value: str) -> str:
        return str(value).replace("\\", "\\\\").replace("'", "''")

    def _next_step_id(self):
        import time
        with self._lock:
            self._step_counter += 1
            return int((time.time() * 1000) + self._step_counter)

    @staticmethod
    def _normalize_progress_increment(progress_increment: float) -> float:
        inc = float(progress_increment or 0.0)
        if inc <= 0.0:
            return 1.0
        return float(max(1, int(round(inc))))

    def _ensure_tables(self):
        if not self.enabled:
            return
        session_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.session_table} (
            session_id BIGINT,
            business_name STRING,
            inspire_database_name STRING,
            operation_mode STRING,
            table_election_mode STRING,
            use_cases_quality STRING,
            strategic_goals STRING,
            business_priorities STRING,
            business_domains STRING,
            catalogs STRING,
            schemas_str STRING,
            tables_str STRING,
            generate_choices STRING,
            generation_path STRING,
            output_language STRING,
            sql_generation_per_domain STRING,
            technical_exclusion_strategy STRING,
            json_file_path STRING,
            processing_status STRING,
            completed_percent DOUBLE,
            create_at TIMESTAMP,
            last_updated TIMESTAMP,
            completed_on TIMESTAMP,
            inspire_json VARIANT,
            results_json VARIANT,
            enriched_business_context STRING,
            enriched_strategic_goals STRING,
            enriched_business_priorities STRING,
            enriched_strategic_initiative STRING,
            enriched_value_chain STRING,
            enriched_revenue_model STRING,
            generation_instructions_section STRING
        ) USING DELTA
        """
        steps_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.steps_table} (
            session_id BIGINT,
            step_id BIGINT,
            last_updated TIMESTAMP,
            stage_name STRING,
            step_name STRING,
            sub_step_name STRING,
            progress_increment DOUBLE,
            message STRING,
            status STRING,
            result_json VARIANT
        ) USING DELTA
        """
        lock_sql = f"""
        CREATE TABLE IF NOT EXISTS {self.lock_table} (
            business_name STRING,
            session_id BIGINT,
            acquired_at TIMESTAMP,
            expires_at TIMESTAMP,
            holder_note STRING
        ) USING DELTA
        """
        self.spark.sql(session_sql)
        self.spark.sql(steps_sql)
        try:
            self.spark.sql(lock_sql)
        except Exception as _le:
            self.logger.debug(f"Lock table create failed (non-critical): {_le}")
        self._migrate_session_table_schema()

    def acquire_lock(self, business_name: str, ttl_seconds: int = 7200, holder_note: str = "") -> bool:
        """Try to acquire a per-business lock. Returns True on success, False if held."""
        if not self.enabled or not (business_name or "").strip():
            return True
        try:
            # Purge expired first
            self.spark.sql(f"DELETE FROM {self.lock_table} WHERE expires_at < current_timestamp()")
            held = self.spark.sql(
                f"SELECT COUNT(*) AS c FROM {self.lock_table} "
                f"WHERE business_name = '{str(business_name).replace(chr(39), chr(39)+chr(39))}' "
                f"AND session_id != {int(self.session_id)}"
            ).collect()
            if held and int(held[0][0] or 0) > 0:
                return False
            # Idempotent: purge any self-owned rows then insert fresh
            self.spark.sql(
                f"DELETE FROM {self.lock_table} "
                f"WHERE business_name = '{str(business_name).replace(chr(39), chr(39)+chr(39))}' "
                f"AND session_id = {int(self.session_id)}"
            )
            self.spark.sql(
                f"INSERT INTO {self.lock_table} VALUES ("
                f"'{str(business_name).replace(chr(39), chr(39)+chr(39))}', "
                f"{int(self.session_id)}, current_timestamp(), "
                f"current_timestamp() + INTERVAL {int(ttl_seconds)} SECONDS, "
                f"'{str(holder_note).replace(chr(39), chr(39)+chr(39))[:200]}')"
            )
            return True
        except Exception as _err:
            self.logger.warning(f"acquire_lock failed (non-fatal, proceeding): {_err}")
            return True  # fail-open: never block pipeline on lock infra

    def release_lock(self, business_name: str) -> None:
        if not self.enabled or not (business_name or "").strip():
            return
        try:
            self.spark.sql(
                f"DELETE FROM {self.lock_table} "
                f"WHERE business_name = '{str(business_name).replace(chr(39), chr(39)+chr(39))}' "
                f"AND session_id = {int(self.session_id)}"
            )
        except Exception as _err:
            self.logger.debug(f"release_lock failed (non-critical): {_err}")
    
    def _migrate_session_table_schema(self):
        """Idempotently add the 7 per-session enriched-context columns.
        Logs at WARNING level on failure (was DEBUG — silently hid the bug where
        columns never got added and enriched context was lost on every run)."""
        if not self.enabled:
            return
        existing = set()
        try:
            def _row_col(r):
                try:
                    return r["col_name"]
                except Exception:
                    try:
                        return getattr(r, "col_name", None)
                    except Exception:
                        return None
            _rows = self.spark.sql(f"DESCRIBE TABLE {self.session_table}").collect()
            existing = {
                str(_row_col(r)).lower() for r in _rows
                if _row_col(r) and not str(_row_col(r)).startswith("#")
            }
        except Exception as e:
            self.logger.warning(f"Session schema DESCRIBE failed: {get_clean_error_message(e)} — migration will skip. "
                                f"Enriched context UPDATE will fail downstream.")
            return
        add_if_missing = [
            ("enriched_business_context", "STRING"),
            ("enriched_strategic_goals", "STRING"),
            ("enriched_business_priorities", "STRING"),
            ("enriched_strategic_initiative", "STRING"),
            ("enriched_value_chain", "STRING"),
            ("enriched_revenue_model", "STRING"),
            ("generation_instructions_section", "STRING"),
        ]
        added = []
        already = []
        failed = []
        for col_name, col_type in add_if_missing:
            if col_name.lower() in existing:
                already.append(col_name)
                continue
            try:
                self.spark.sql(f"ALTER TABLE {self.session_table} ADD COLUMN {col_name} {col_type}")
                added.append(col_name)
            except Exception as e:
                msg = str(e).lower()
                if "already exists" in msg or "duplicate" in msg:
                    already.append(col_name)
                    continue
                failed.append((col_name, get_clean_error_message(e)[:200]))
        if added:
            self.logger.info(f"Session schema migrate: added {len(added)} columns: {added}")
        if failed:
            self.logger.warning(
                f"Session schema migrate: {len(failed)}/{len(add_if_missing)} ALTER TABLE ADD COLUMN failed. "
                f"Enriched context UPDATEs will fail for these columns. Details: {failed}"
            )

    def _widget_to_columns_map(self) -> dict:
        """Map widget_values dict keys to session table column names."""
        wv = self.widget_values or {}
        return {
            "business_name": str(wv.get("business", "") or ""),
            "inspire_database_name": str(wv.get("inspire_database", "") or ""),
            "operation_mode": str(wv.get("operation", wv.get("operation_mode", "")) or ""),
            "table_election_mode": str(wv.get("table_election_mode", "") or ""),
            "use_cases_quality": str(wv.get("use_cases_quality", "") or ""),
            "strategic_goals": str(wv.get("generation_instructions", "") or ""),
            "business_priorities": str(wv.get("business_priorities", "") or ""),
            "business_domains": str(wv.get("business_domains", "") or ""),
            "catalogs": str(wv.get("catalogs", "") or ""),
            "schemas_str": str(wv.get("schemas", "") or ""),
            "tables_str": str(wv.get("tables", "") or ""),
            "generate_choices": str(wv.get("generate", "") or ""),
            "generation_path": str(wv.get("generation_path", "") or ""),
            "output_language": str(wv.get("output_language", "") or ""),
            "sql_generation_per_domain": str(wv.get("sql_generation_per_domain", "") or ""),
            "technical_exclusion_strategy": str(wv.get("technical_exclusion_strategy", "") or ""),
            "json_file_path": str(wv.get("json_file_path", "") or ""),
        }

    def initialize_session(self):
        if not self.enabled:
            self._console_log("inspire_database not provided; running in console-only mode")
            return
        self._ensure_tables()
        ts = self._utc_now_str()
        inspire_json = self._escape_sql_str(self._safe_json_str({}))
        col_map = self._widget_to_columns_map()
        widget_col_select = ", ".join(
            f"'{self._escape_sql_str(v)}' AS {k}" for k, v in col_map.items()
        )
        widget_col_update = ", ".join(
            f"target.{k} = source.{k}" for k in col_map.keys()
        )
        widget_col_insert_names = ", ".join(col_map.keys())
        widget_col_insert_vals = ", ".join(f"source.{k}" for k in col_map.keys())

        merge_sql = f"""
        MERGE INTO {self.session_table} AS target
        USING (
            SELECT
                {self.session_id} AS session_id,
                {widget_col_select},
                'done' AS processing_status,
                1.0 AS completed_percent,
                TIMESTAMP('{ts}') AS create_at,
                TIMESTAMP('{ts}') AS last_updated,
                CAST(NULL AS TIMESTAMP) AS completed_on,
                parse_json('{inspire_json}') AS inspire_json,
                parse_json('{inspire_json}') AS results_json
        ) AS source
        ON target.session_id = source.session_id
        WHEN MATCHED THEN UPDATE SET
            {widget_col_update},
            target.processing_status = source.processing_status,
            target.completed_percent = source.completed_percent,
            target.create_at = source.create_at,
            target.last_updated = source.last_updated,
            target.completed_on = source.completed_on,
            target.inspire_json = source.inspire_json,
            target.results_json = source.results_json
        WHEN NOT MATCHED THEN INSERT (
            session_id, {widget_col_insert_names},
            processing_status, completed_percent, create_at, last_updated, completed_on,
            inspire_json, results_json
        ) VALUES (
            source.session_id, {widget_col_insert_vals},
            source.processing_status, source.completed_percent, source.create_at,
            source.last_updated, source.completed_on,
            source.inspire_json, source.results_json
        )
        """
        self.spark.sql(merge_sql)
        self._console_log("session initialized and persisted", force=True)
    
    def update_session_prompt_context(self, context_fields: dict):
        """Persist the 9 per-session Genie-prompt context fields onto the session row so
        Generate mode can rebuild prompts without re-running business-context extraction.
        Called once after business context is extracted. Idempotent (UPDATE statement)."""
        if not self.enabled or not context_fields:
            return
        allowed_cols = {
            "enriched_business_context",
            "enriched_strategic_goals",
            "enriched_business_priorities",
            "enriched_strategic_initiative",
            "enriched_value_chain",
            "enriched_revenue_model",
            "generation_instructions_section",
        }
        set_parts = []
        for col, val in context_fields.items():
            if col not in allowed_cols:
                continue
            escaped = self._escape_sql_str(str(val or ""))
            set_parts.append(f"{col} = '{escaped}'")
        if not set_parts:
            return
        try:
            set_clause = ", ".join(set_parts)
            self.spark.sql(
                f"UPDATE {self.session_table} SET {set_clause} WHERE session_id = {self.session_id}"
            )
            self._console_log(f"session prompt context updated ({len(set_parts)} fields)", force=False)
        except Exception as e:
            self._console_log(f"session prompt context update failed (non-critical): {e}", level="WARNING")

    def emit_step(self, stage_name: str, step_name: str, sub_step_name: str = "", progress_increment: float = 0.0,
                  message: str = "", status: str = "started", result_json=None, step_id: int = None):
        effective_step_id = int(step_id) if step_id is not None else self._next_step_id()
        from datetime import datetime, timezone
        payload = result_json if isinstance(result_json, dict) else {}
        payload.setdefault("event_at", datetime.now(timezone.utc).isoformat())
        payload.setdefault("entity_id", f"{self.session_id}:{stage_name}:{step_name}:{sub_step_name}")
        normalized_increment = self._normalize_progress_increment(progress_increment)
        if self.console_step_logging_enabled or status == "ended_error":
            self._console_log(
                f"{status} | stage='{stage_name}' step='{step_name}' sub_step='{sub_step_name}' progress={normalized_increment} | {message}",
                level="ERROR" if status == "ended_error" else "INFO"
            )
        if not self.enabled:
            return effective_step_id
        import json
        import time
        event = {
            "session_id": self.session_id,
            "step_id": effective_step_id,
            "stage_name": str(stage_name or ""),
            "step_name": str(step_name or ""),
            "sub_step_name": str(sub_step_name or ""),
            "progress_increment": normalized_increment,
            "message": str(message or ""),
            "status": str(status or "started"),
            "result_json": payload,
        }
        with self._lock:
            with open(self._spool_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(event, ensure_ascii=True) + "\n")
        now = time.time()
        if (now - self._last_flush_attempt_epoch) >= self._flush_interval_seconds:
            self._last_flush_attempt_epoch = now
            self._trigger_async_flush(force=False, finalize=False)
        return event["step_id"]

    def _trigger_async_flush(self, force: bool, finalize: bool):
        import threading
        with self._lock:
            if self._flush_thread and self._flush_thread.is_alive():
                return
            self._flush_thread = threading.Thread(
                target=self.flush_pending,
                kwargs={"force": force, "finalize": finalize},
                daemon=True,
                name=f"AtomicWriterFlush_{self.session_id}"
            )
            self._flush_thread.start()

    def _get_session_processing_status(self):
        if not self.enabled:
            return None
        try:
            rows = self.spark.sql(
                f"SELECT processing_status FROM {self.session_table} WHERE session_id = {self.session_id} LIMIT 1"
            ).collect()
            if not rows:
                return None
            return str(rows[0]["processing_status"]).lower() if rows[0]["processing_status"] is not None else None
        except Exception as e:
            self.logger.warning(f"AtomicWriter: failed reading processing_status: {get_clean_error_message(e)}")
            return None

    def _consume_pending_events_stream(self, ts: str, chunk_size: int = 300):
        import os
        import json
        inserted_count = 0
        progress_increment_sum = 0.0
        new_offset = self._spool_offset
        with self._lock:
            if not os.path.exists(self._spool_path):
                return inserted_count, new_offset, progress_increment_sum
            with open(self._spool_path, "r", encoding="utf-8") as f:
                f.seek(self._spool_offset)
                current_chunk = []
                while True:
                    line = f.readline()
                    if not line:
                        break
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        current_chunk.append(json.loads(line))
                    except Exception:
                        continue
                    if len(current_chunk) >= chunk_size:
                        progress_increment_sum += self._insert_steps_chunk(current_chunk, ts)
                        inserted_count += len(current_chunk)
                        current_chunk = []
                if current_chunk:
                    progress_increment_sum += self._insert_steps_chunk(current_chunk, ts)
                    inserted_count += len(current_chunk)
                new_offset = f.tell()
        return inserted_count, new_offset, progress_increment_sum

    def _insert_steps_chunk(self, rows: list, ts: str):
        if not rows:
            return 0.0
        import time
        from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
        schema = StructType([
            StructField("session_id", LongType(), False),
            StructField("step_id", LongType(), False),
            StructField("stage_name", StringType(), True),
            StructField("step_name", StringType(), True),
            StructField("sub_step_name", StringType(), True),
            StructField("progress_increment", DoubleType(), True),
            StructField("message", StringType(), True),
            StructField("status", StringType(), True),
            StructField("result_json_str", StringType(), True),
        ])
        payload_rows = []
        success_increment_sum = 0.0
        for row in rows:
            status = str(row.get("status", "") or "")
            inc = float(row.get("progress_increment", 0.0) or 0.0)
            if status == "ended_success":
                success_increment_sum += inc
            payload_rows.append((
                int(row.get("session_id", self.session_id)),
                int(row.get("step_id")),
                str(row.get("stage_name", "")),
                str(row.get("step_name", "")),
                str(row.get("sub_step_name", "")),
                inc,
                str(row.get("message", "")),
                status,
                self._safe_json_str(row.get("result_json", {}))
            ))
        df = self.spark.createDataFrame(payload_rows, schema=schema)
        view_name = f"_inspire_steps_queue_{self.session_id}_{int(time.time() * 1000)}"
        df.createOrReplaceTempView(view_name)
        insert_sql = f"""
        INSERT INTO {self.steps_table} (
            session_id, step_id, last_updated, stage_name, step_name, sub_step_name, progress_increment, message, status, result_json
        )
        SELECT
            session_id,
            step_id,
            TIMESTAMP('{ts}') AS last_updated,
            stage_name,
            step_name,
            sub_step_name,
            progress_increment,
            message,
            status,
            parse_json(result_json_str) AS result_json
        FROM {view_name}
        """
        self.spark.sql(insert_sql)
        self.spark.catalog.dropTempView(view_name)
        return success_increment_sum

    def _update_session_ready(self, ts: str, progress_increment_sum: float):
        update_sql = f"""
        UPDATE {self.session_table}
        SET
            completed_percent = LEAST(99.0, CAST(FLOOR(completed_percent + {float(progress_increment_sum)}) AS DOUBLE)),
            processing_status = 'ready',
            last_updated = TIMESTAMP('{ts}'),
            inspire_json = parse_json('{self._escape_sql_str(self._safe_json_str({"last_flush": ts}))}')
        WHERE session_id = {self.session_id}
        """
        self.spark.sql(update_sql)

    def _finalize_session(self, ts: str, final_results_json=None):
        final_payload = final_results_json if final_results_json is not None else {"completed": True, "completed_on": ts}
        final_payload_sql = self._escape_sql_str(self._safe_json_str(final_payload))
        finalize_sql = f"""
        UPDATE {self.session_table}
        SET
            completed_percent = 100.0,
            processing_status = 'ready',
            last_updated = TIMESTAMP('{ts}'),
            completed_on = TIMESTAMP('{ts}'),
            inspire_json = parse_json('{final_payload_sql}'),
            results_json = parse_json('{final_payload_sql}')
        WHERE session_id = {self.session_id}
        """
        self.spark.sql(finalize_sql)

    def flush_pending(self, force: bool = False, finalize: bool = False, final_results_json=None):
        if not self.enabled:
            return 0
        with self._flush_lock:
            if not force:
                processing_status = self._get_session_processing_status()
                if processing_status is None:
                    return 0
            ts = self._utc_now_str()
            inserted_count, new_offset, progress_increment_sum = self._consume_pending_events_stream(ts=ts, chunk_size=300)
            if inserted_count == 0 and not finalize:
                return 0
            with self._lock:
                self._spool_offset = max(self._spool_offset, new_offset)
            if finalize:
                self._finalize_session(ts, final_results_json=final_results_json)
            elif inserted_count > 0:
                self._update_session_ready(ts, progress_increment_sum)
            return inserted_count

    def finalize_pipeline(self, message: str = "Pipeline completed", final_results_json=None, pipeline_start_step_id: int = None):
        if not self.enabled:
            self._console_log(f"finalize (console-only): {message}")
            return
        self.emit_step(
            stage_name="Inspire Journey",
            step_name="Inspire pipeline completed",
            sub_step_name="Completed",
            progress_increment=1.0,
            message="Completed",
            status="ended_success",
            result_json={"session_id": self.session_id},
            step_id=pipeline_start_step_id
        )
        self.flush_pending(force=True, finalize=True, final_results_json=final_results_json)
        self._console_log("finalize completed with forced flush and session completion")


class DatabricksInspire:
    
    @staticmethod
    def _extract_primary_table(tables_involved: str) -> str:
        """
        Extracts the primary (first) table from the Tables Involved field.
        
        Args:
            tables_involved: Comma-separated string of table names
            
        Returns:
            The first table name, or 'N/A' if empty
        """
        if not tables_involved or not isinstance(tables_involved, str):
            return "N/A"
        
        # Split by comma and get the first table
        tables = [t.strip() for t in tables_involved.split(',') if t.strip()]
        if tables:
            return tables[0]
        return "N/A"

    @staticmethod
    def _get_enriched_field(ctx: dict, field_name: str, fallback: str, min_length: int = 10) -> str:
        raw = ctx.get(field_name, '') if isinstance(ctx, dict) else ''
        if isinstance(raw, list):
            value = ', '.join(str(item) for item in raw if item) if raw else ''
        else:
            value = str(raw).strip() if raw else ''
        if value and len(value) >= min_length:
            return value
        return fallback

    def _build_use_case_step_payload(self, use_case: dict) -> dict:
        uc = use_case
        def _s(key, default=''):
            v = uc.get(key, default)
            return str(v) if v is not None else str(default)
        def _n(key, default=0):
            v = uc.get(key, default)
            try:
                return float(v) if v is not None else float(default)
            except (TypeError, ValueError):
                return float(default)
        return {
            "use_case_id": _s('No'),
            "business_domain": _s('Business Domain'),
            "subdomain": _s('Subdomain'),
            "usecase": self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', '')),
            "name": _s('Name'),
            "generated": _s('generated'),
            "validated": _s('validated'),
            "type": _s('type'),
            "analytics_technique": _s('Analytics Technique'),
            "business_priority_alignment": _s('Business Priority Alignment'),
            "strategic_goals_alignment": _s('Strategic Goals Alignment'),
            "statement": _s('Statement'),
            "solution": _s('Solution'),
            "business_value": _s('Business Value'),
            "beneficiary": _s('Beneficiary'),
            "sponsor": _s('Sponsor'),
            "tables_involved": _s('Tables Involved'),
            "primary_table": _s('Primary Table'),
            "strategic_alignment": _n('Strategic Alignment'),
            "return_on_investment": _n('Return on Investment'),
            "reusability": _n('Reusability'),
            "time_to_value": _n('Time to Value'),
            "data_availability": _n('Data Availability'),
            "data_accessibility": _n('Data Accessibility'),
            "architecture_fitness": _n('Architecture Fitness'),
            "team_skills": _n('Team Skills'),
            "domain_knowledge": _n('Domain Knowledge'),
            "people_allocation": _n('People Allocation'),
            "budget_allocation": _n('Budget Allocation'),
            "time_to_production": _n('Time to Production'),
            "value_score": _n('Value'),
            "feasibility_score": _n('Feasibility'),
            "priority_score": _n('Priority'),
            "quality": _s('Quality'),
            "justification": _s('Justification'),
            "quality_summary": _s('Quality Summary'),
            "result_table": _s('result_table'),
            "notebook_path": _s('notebook_path'),
            "high_level_design": _s('High Level Design'),
            "genie_instruction": _s('genie_instruction'),
        }

    def _get_business_context_fallback(self) -> str:
        # Phase M: Grounded fallback only — NEVER invent scale, market position, or
        # firm-specific facts. If enriched context is unavailable, emit the literal
        # UNKNOWN_FROM_METADATA token and a strictly structural description that
        # downstream prompts must treat as a non-anchor (do-not-invent signal).
        ctx = getattr(self, 'merged_business_context', {})
        if isinstance(ctx, dict) and ctx.get('business_context') and len(ctx['business_context'].strip()) >= 80:
            return ctx['business_context']
        industries_str = ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else ""
        tables_hint = ""
        if hasattr(self, 'available_tables') and self.available_tables:
            sample_tables = list(self.available_tables)[:5]
            tables_hint = f" covering tables {', '.join(sample_tables)}"
        base = f"Business name: {getattr(self, 'business_name', 'UNKNOWN_FROM_METADATA')}."
        if industries_str:
            base += f" Industry: {industries_str}."
        else:
            base += " Industry: UNKNOWN_FROM_METADATA."
        base += f" Data scope{tables_hint}."
        base += (
            " UNKNOWN_FROM_METADATA: enriched business context was unavailable from the business-context LLM. "
            "Downstream consumers MUST NOT invent firm-specific figures (revenue, location counts, employee counts, "
            "customer counts, Fortune-rank, ISO certifications, market share) and MUST treat UNKNOWN_FROM_METADATA "
            "as a do-not-invent signal. Ground every claim in the supplied schema."
        )
        return base

    @staticmethod
    def _story_clean_text(value: str) -> str:
        import re
        text = str(value or "")
        text = text.replace("_PROMPT", "")
        text = text.replace("_", " ")
        text = re.sub(r"\s+", " ", text).strip()
        text = text.replace("LLM", "AI")
        text = text.replace("SQL", "implementation")
        return text

    def _story_status_line(self, status: str, message: str, fallback: str) -> str:
        context = self._story_clean_text(fallback or "")
        if status == "started":
            return f"In progress: {context.lower()}" if context else "In progress"
        if status == "ended_success":
            return f"Completed: {context.lower()}" if context else "Completed"
        if status == "ended_error":
            return f"Needs attention: {context.lower()}" if context else "Needs attention"
        if status == "ended_warning":
            return f"Completed with caution: {context.lower()}" if context else "Completed with caution"
        return context if context else "Progress update"

    def _story_prompt_labels(self, prompt_name: str, step_name: str, status: str, message: str):
        stage_name, step_title = PROMPT_STORY_TITLES.get(
            prompt_name,
            ("Inspire Progress", f"Processing {self._story_clean_text(prompt_name)}")
        )
        sub_step_name = self._story_status_line(status, message, step_name)
        return stage_name, step_title, sub_step_name

    def _story_prompt_message(self, prompt_name: str, status: str, step_title: str, sub_step_name: str) -> str:
        if status == "started":
            return PROMPT_STORY_STARTED_MESSAGES.get(
                prompt_name,
                f"Inspire has started {self._story_clean_text(step_title).lower()}"
            )
        if status == "ended_success":
            return f"Inspire has completed {self._story_clean_text(step_title).lower()}"
        if status == "ended_warning":
            return f"Inspire has completed {self._story_clean_text(step_title).lower()} with caution"
        if status == "ended_error":
            return f"Inspire needs attention while {self._story_clean_text(step_title).lower()}"
        return sub_step_name

    def _story_pipeline_labels(self, stage_name: str, step_name: str, sub_step_name: str, status: str, message: str):
        stage_map = {
            "Pipeline": "Inspire Journey",
            "Initialization": "Setup",
            "Business Context": "Business Understanding",
            "Document Generation": "Executive Readout",
            "Excel Generation": "Executive Readout",
            "Artifact Writing": "Executive Readout",
            "Summary": "Executive Readout",
            "Translation": "Localization",
            "Use Cases Scoring": "Use Case Prioritization",
            "Business Domains Clustering": "Domain Mapping",
            "Genie Code Instructions": "Implementation Planning",
        }
        base_stage = stage_map.get(stage_name, self._story_clean_text(stage_name or "Inspire Progress"))
        if stage_name == "Pipeline" and step_name == "Execution" and status == "started":
            base_step = "Inspire pipeline started"
        elif stage_name == "Pipeline" and step_name == "Execution" and status == "ended_success":
            base_step = "Inspire pipeline completed"
        elif stage_name == "Pipeline" and step_name == "Execution" and status == "ended_error":
            base_step = "Inspire pipeline needs attention"
        else:
            clean_step = self._story_clean_text(step_name)
            base_step = f"Inspire is working on {clean_step.lower()}" if clean_step else "Inspire is processing your request"
        live_text = self._story_status_line(status, message, sub_step_name)
        return base_stage, base_step, live_text

    def _story_pipeline_message(self, status: str, step_name: str, sub_step_name: str) -> str:
        clean_step = self._story_clean_text(step_name)
        if status == "started":
            return f"Inspire has started {clean_step.lower()}" if clean_step else "Inspire has started processing"
        if status == "ended_success":
            return f"Inspire has completed {clean_step.lower()}" if clean_step else "Inspire has completed processing"
        if status == "ended_warning":
            return f"Inspire has completed {clean_step.lower()} with caution" if clean_step else "Inspire has completed with caution"
        if status == "ended_error":
            return f"Inspire needs attention while {clean_step.lower()}" if clean_step else "Inspire needs attention"
        return sub_step_name

    def _emit_prompt_status(self, prompt_name: str, step_name: str, status: str, message: str = "", result_json=None, step_id: int = None):
        if not getattr(self, "atomic_writer", None):
            return None
        tracking_meta = PROMPT_TRACKING_CONFIG.get(prompt_name, None)
        if tracking_meta:
            total_prompt_budget = max(1, int(round(float(tracking_meta["progress_increment"]))))
        else:
            total_prompt_budget = 1

        payload = result_json if isinstance(result_json, dict) else {}
        entity_id = str(payload.get("entity_id") or f"{prompt_name}:{step_name}")

        progress_increment = 0
        if status == "started":
            with self._prompt_progress_lock:
                self._prompt_entities_seen.setdefault(prompt_name, set()).add(entity_id)
        elif status == "ended_success":
            with self._prompt_progress_lock:
                self._prompt_entities_seen.setdefault(prompt_name, set()).add(entity_id)
                entities_seen = self._prompt_entities_seen.get(prompt_name, set())
                emitted_so_far = int(self._prompt_progress_emitted.get(prompt_name, 0))
                remaining_budget = max(0, total_prompt_budget - emitted_so_far)
                if remaining_budget > 0:
                    if prompt_name in MULTI_ENTITY_PROGRESS_PROMPTS:
                        progress_increment = 1
                    else:
                        denominator = max(1, len(entities_seen))
                        progress_increment = max(1, remaining_budget // denominator)
                        progress_increment = min(progress_increment, remaining_budget)
                self._prompt_progress_emitted[prompt_name] = emitted_so_far + progress_increment

        story_stage_name, story_step_name, story_sub_step_name = self._story_prompt_labels(
            prompt_name=prompt_name,
            step_name=step_name,
            status=status,
            message=message
        )
        story_message = self._story_prompt_message(
            prompt_name=prompt_name,
            status=status,
            step_title=story_step_name,
            sub_step_name=story_sub_step_name
        )

        return self.atomic_writer.emit_step(
            stage_name=story_stage_name,
            step_name=story_step_name,
            sub_step_name=story_sub_step_name,
            progress_increment=progress_increment,
            message=story_message,
            status=status,
            result_json=result_json if result_json is not None else {},
            step_id=step_id
        )

    def _emit_pipeline_status(self, stage_name: str, step_name: str, sub_step_name: str, message: str, status: str,
                              progress_increment: float = 0.0, result_json=None, step_id: int = None):
        if not getattr(self, "atomic_writer", None):
            return None
        story_stage_name, story_step_name, story_sub_step_name = self._story_pipeline_labels(
            stage_name=stage_name,
            step_name=step_name,
            sub_step_name=sub_step_name,
            status=status,
            message=message
        )
        story_message = self._story_pipeline_message(
            status=status,
            step_name=story_step_name,
            sub_step_name=story_sub_step_name
        )
        emitted_step_id = self.atomic_writer.emit_step(
            stage_name=story_stage_name,
            step_name=story_step_name,
            sub_step_name=story_sub_step_name,
            progress_increment=progress_increment,
            message=story_message,
            status=status,
            result_json=result_json if result_json is not None else {},
            step_id=step_id
        )
        open_steps = getattr(self, "_open_pipeline_step_ids", None)
        if open_steps is not None:
            step_key = f"{stage_name}|{step_name}|{sub_step_name}"
            if status == "started":
                open_steps[step_key] = (emitted_step_id, stage_name, step_name, sub_step_name)
            elif status.startswith("ended_"):
                open_steps.pop(step_key, None)
        return emitted_step_id

    def finalize_atomic_writer(self, success: bool = True, error_message: str = ""):
        if not getattr(self, "atomic_writer", None):
            return
        final_results_json = getattr(self, "final_inspire_results_json", None)
        pipeline_start_step_id = getattr(self, "_pipeline_start_step_id", None)
        if not success:
            open_steps = getattr(self, "_open_pipeline_step_ids", {})
            for step_key, (open_step_id, stage, step, sub) in list(open_steps.items()):
                if open_step_id == pipeline_start_step_id:
                    continue
                try:
                    self._emit_pipeline_status(
                        stage_name=stage,
                        step_name=step,
                        sub_step_name=sub,
                        message=error_message or "Pipeline failed — closing open step",
                        status="ended_error",
                        progress_increment=0.0,
                        result_json={"error": error_message[:300] if error_message else "Pipeline terminated"},
                        step_id=open_step_id
                    )
                except Exception:
                    pass
            self._emit_pipeline_status(
                stage_name="Pipeline",
                step_name="Execution",
                sub_step_name="Finalize",
                message=error_message or "Pipeline failed",
                status="ended_error",
                progress_increment=1.0,
                result_json={"session_id": self.session_id},
                step_id=pipeline_start_step_id
            )
            self.atomic_writer.flush_pending(force=True, finalize=True, final_results_json=final_results_json)
        else:
            self.atomic_writer.finalize_pipeline(
                message="Pipeline completed",
                final_results_json=final_results_json,
                pipeline_start_step_id=pipeline_start_step_id
            )

    def __init__(self, **kwargs):
        self.spark = spark
        self.dbutils = dbutils
        # Initialize workspace client with timeout configuration to prevent indefinite hangs
        from databricks.sdk.config import Config
        config = Config()
        config.retry_timeout_seconds = 300  # 5 minute timeout for individual API calls
        self.workspace = WorkspaceClient(config=config)  # Changed from w_client to workspace
        self.w_client = self.workspace  # Keep w_client as alias for backward compatibility
        self._notebook_object_ids = {}

        # --- Store Widget Values ---
        self.business_name = kwargs.get("business", "gen")
        
        # --- Business Priorities (user-provided, multi-select) ---
        business_priorities_raw = kwargs.get("business_priorities", "")
        self.user_business_priorities = [goal.strip() for goal in business_priorities_raw.split(',') if goal.strip()]
        
        # --- Generation Instructions (user-provided, free-text) ---
        generation_instructions_raw = kwargs.get("generation_instructions", "")
        self.user_generation_instructions = generation_instructions_raw.strip() if generation_instructions_raw else ""
        if len(self.user_generation_instructions) > 2000:
            self.logger.warning(f"Generation Instructions exceeds 2000 char limit ({len(self.user_generation_instructions)} chars) -- truncating")
            self.user_generation_instructions = self.user_generation_instructions[:2000]
        self.user_strategic_goals = []
        self.parsed_generation_instructions = {}
        self._user_provided_session_id = bool(kwargs.get("_user_provided_session_id", False))
        
        # --- Business Domains (user-provided, comma-separated) ---
        business_domains_raw = kwargs.get("business_domains", "")
        self.user_business_domains = [domain.strip() for domain in business_domains_raw.split(',') if domain.strip()]
        
        self.additional_context = ""
        
        self.catalogs_str = kwargs.get("catalogs", "")
        self.schemas_str = kwargs.get("schemas", "")
        self.tables_str = kwargs.get("tables", "")
        self.generate_choices = [opt.strip() for opt in kwargs.get("generate", "use cases").split(',')]
        
        self.generation_path = kwargs.get("generation_path", "./")
        
        self.output_languages = [lang.strip() for lang in kwargs.get("output_language", "English").split(',') if lang.strip()]
        if not self.output_languages:
            self.output_languages = ["English"]
            
        _runtime = TECHNICAL_CONTEXT["runtime"]
        
        raw_max_parallelism = kwargs.get("max_parallelism", None)
        if raw_max_parallelism is None or str(raw_max_parallelism).strip() == "":
            self.max_parallelism = _runtime["default_max_parallelism"]
            self.auto_parallelism = True
        else:
            self.max_parallelism = min(int(raw_max_parallelism), 100)
            self.auto_parallelism = False
        self.scan_parallelism = _runtime["scan_parallelism"]
        self.cluster_memory_gb = int(kwargs.get("cluster_memory_gb", _runtime["cluster_memory_gb"]) or _runtime["cluster_memory_gb"])
        self.cluster_worker_count = int(kwargs.get("cluster_worker_count", _runtime["cluster_worker_count"]) or _runtime["cluster_worker_count"])
        
        raw_technical_exclusion_strategy = kwargs.get("technical_exclusion_strategy", "Aggressive")
        if not raw_technical_exclusion_strategy or str(raw_technical_exclusion_strategy).strip().lower() == "none":
            raw_technical_exclusion_strategy = "Aggressive"
        self.technical_exclusion_strategy = raw_technical_exclusion_strategy
        
        _raw_operation = kwargs.get("operation", "Discover Use Cases")
        if _raw_operation not in ("Discover Use Cases", "Generate Use Cases"):
            _raw_operation = "Discover Use Cases"
        self.operation = _raw_operation
        self.operation_mode = self.operation
        
        self.table_election_mode = kwargs.get("table_election_mode", "Let Inspire Decides")
        if self.table_election_mode not in ("Let Inspire Decides", "All Tables", "Transactional Only"):
            self.table_election_mode = "Let Inspire Decides"
        
        _raw_uc_quality = kwargs.get("use_cases_quality", "Balanced")
        _raw_uc_quality = USE_CASES_QUALITY_ALIASES.get(_raw_uc_quality, _raw_uc_quality)
        if _raw_uc_quality not in USE_CASES_QUALITY_VALID:
            _raw_uc_quality = "Balanced"
        self.use_cases_quality = _raw_uc_quality
        

        self.generate_genie_instructions = (self.operation == "Generate Use Cases")
        # v0.8.9 DRY: operation banner is emitted ONCE during widget validation
        # (~line 31905) — removed duplicate __init__ emit to keep the boot log clean.
        
        _quality_filter_map = {
            "Coverage Mode (All)": None,
            "Balanced": {'Medium', 'High', 'Very High', 'Ultra High'},
            "Strict Quality": {'High', 'Very High', 'Ultra High'},
        }
        self.quality_filter_acceptable = _quality_filter_map.get(self.use_cases_quality)

        self.min_use_cases_for_selection = max(1, _runtime.get("min_use_cases_for_selection", 10))

        self.llm_timeout_seconds = _runtime["llm_timeout_seconds"]
        self.max_retry_attempts = max(0, _runtime["max_retry_attempts"])
        
        # === NEW: Initialize merged business context ===
        self.merged_business_context = {}
        
        # === NEW: JSON file path for docs-only mode ===
        self.json_file_path = kwargs.get("json_file_path", None)
        self.final_inspire_results_json = {}
        self._pipeline_start_step_id = None
        
        # === Inspire Database (user-provided catalog.database for all result tables) ===
        self.inspire_database = kwargs.get("inspire_database", "")
        if self.inspire_database:
            self.inspire_database = self.inspire_database.strip()
        
        # === Auto-Genie scope ("all" or non-negative int as string) ===
        self.generate_genie_code_for = str(kwargs.get("generate_genie_code_for", "5") or "5").strip()

        # === Session ID (unique per inspire session for tracking) ===
        _user_session_id = kwargs.get("session_id", "")
        if _user_session_id:
            try:
                self.session_id = int(_user_session_id)
            except ValueError:
                import hashlib as _hashlib_mod
                _MAX_SIGNED_INT64 = 9223372036854775807
                _hash_int = int(_hashlib_mod.sha256(_user_session_id.encode()).hexdigest(), 16)
                self.session_id = _hash_int & _MAX_SIGNED_INT64
        else:
            import uuid as _uuid_mod
            _MAX_SIGNED_INT64 = 9223372036854775807
            self.session_id = (_uuid_mod.uuid4().int >> 64) & _MAX_SIGNED_INT64

        self._prompt_progress_lock = threading.Lock()
        self._prompt_entities_seen = {}
        self._prompt_progress_emitted = {}

        self.atomic_writer = AtomicWriter(
            spark=self.spark,
            logger=logging.getLogger("AtomicWriterBootstrap"),
            inspire_database=kwargs.get("inspire_database", ""),
            session_id=self.session_id,
            widget_values=dict(kwargs)
        )
        
        # === Use case tracking table infrastructure ===
        import threading as _threading_mod
        self._tracking_lock = _threading_mod.Lock()
        self._tracking_table_created = False
        self._tracking_table_name = f"{self.inspire_database}.__inspire_usecases" if self.inspire_database else ""

        self._cached_schema_markdown = None

        # --- Setup Logging & Output ---
        self.sanitized_customer_name = self._sanitize_name(self.business_name)
        
        # Keep log in /tmp during execution (always writable), will copy to output dir at end
        self.local_log_output_dir = f"/tmp/{self.sanitized_customer_name}_{self.session_id}"

        resolved_generation_path = self.generation_path
        if resolved_generation_path.startswith("./"):
            try:
                logical_notebook_path = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
                current_notebook_dir = os.path.dirname(logical_notebook_path)
                relative_path = resolved_generation_path[2:]
                resolved_generation_path = os.path.normpath(os.path.join(current_notebook_dir, relative_path))
                log_print(f"Resolved relative generation path to: {resolved_generation_path}")
            except Exception as _e:
                raise ValueError(
                    f"generation_path '{self.generation_path}' is relative and the notebook "
                    f"context could not be resolved to anchor it (root cause: {_e}). "
                    f"Please set generation_path to an absolute workspace path "
                    f"(e.g. '/Workspace/Users/<you>/inspire_output' or '/Volumes/<cat>/<sch>/<vol>/inspire_output')."
                )
        elif '/..' in resolved_generation_path or resolved_generation_path.startswith('..'):
            resolved_generation_path = os.path.normpath(resolved_generation_path)

        # Folder name = sanitized business_name (the user-supplied widget value).
        # Per CLAUDE.md §3c (User Vibes Are Supreme Authority), business_name is
        # the primary user-supplied identity for the run; the generation path
        # root MUST reflect it. Falls back to a sanitized form of the UC metadata
        # string only when business_name is empty (validation should have caught
        # this earlier, but keep a safe net).
        _folder_name = self.sanitized_customer_name
        if not _folder_name:
            _schema_src = (self.schemas_str or self.catalogs_str or "").strip()
            if "," in _schema_src:
                _schema_src = _schema_src.split(",")[0].strip()
            if "." in _schema_src:
                _schema_src = _schema_src.split(".")[-1].strip()
            _folder_name = re.sub(r'\s+', '_', _schema_src)
            _folder_name = re.sub(r'[^A-Za-z0-9._\-]', '_', _folder_name).strip("._-")
        self.base_output_dir = os.path.join(resolved_generation_path, _folder_name)
        # No session subfolder — each Discover run wipes and rewrites this schema's
        # folder. session_output_dir remains an alias so downstream code that
        # reads it still works; the existing Discover-mode wipe below deletes it.
        self.session_output_dir = self.base_output_dir
        self.notebook_output_dir = os.path.join(self.base_output_dir, "notebooks")
        self.docs_output_dir = os.path.join(self.base_output_dir, "docs")

        setup_logging(self.local_log_output_dir) 
        self.logger = logging.getLogger(self.__class__.__name__)
        _root_logger = logging.getLogger()
        if not any(isinstance(_f, DuplicateMessageFilter) for _f in _root_logger.filters):
            _root_logger.addFilter(DuplicateMessageFilter(window_seconds=1.5))
        if not any(isinstance(_f, DuplicateMessageFilter) for _f in self.logger.filters):
            self.logger.addFilter(DuplicateMessageFilter(window_seconds=1.5))
        
        # Initialize memory-efficient storage manager
        self.storage_manager = IntermediateStorageManager(base_path="/tmp", logger=self.logger)
        
        # Initialize honesty tracking for reporting what happened during processing
        self.processing_honesty = {
            'total_tables_discovered': 0,
            'total_tables_processed': 0,
            'total_batches_created': 0,
            'total_batch_splits': 0,
            'tables_with_columns_dropped': [],
            'batch_split_history': [],
            'tables_completely_processed': [],
            'tables_partially_processed': [],
            'tables_skipped': []
        }
        self.validation_timeouts_discarded = []
        
        max_parallelism_display = self.max_parallelism if not self.auto_parallelism else "auto"
        self.logger.info(f"DatabricksInspire initialized for business: {self.business_name}. Target Language(s): {self.output_languages}. Base Output Dir: {self.base_output_dir}. Notebooks Dir: {self.notebook_output_dir}. Docs Dir: {self.docs_output_dir}. Scan parallelism: {self.scan_parallelism}. Max parallelism: {max_parallelism_display}")
        
        try:
            if self.operation == "Discover Use Cases":
                self.logger.info(f"Discover mode: cleaning up this session's output directory only (session_id={self.session_id})...")
                try:
                    self.w_client.workspace.delete(self.session_output_dir, recursive=True)
                    self.logger.info(f"Deleted existing session directory: {self.session_output_dir}")
                except Exception as delete_error:
                    self.logger.debug(f"No existing session directory or deletion failed: {delete_error}")
            else:
                self.logger.info(f"Generate mode: preserving existing session directory {self.session_output_dir}.")
            
            self.logger.info(f"Ensuring workspace directories exist...")
            self.w_client.workspace.mkdirs(self.base_output_dir)
            self.w_client.workspace.mkdirs(self.session_output_dir)
            self.w_client.workspace.mkdirs(self.notebook_output_dir)
            self.w_client.workspace.mkdirs(self.docs_output_dir)
            self.logger.info(f"Successfully ensured all output directories.")
        except Exception as e:
            self.logger.error(f"Failed to create workspace directories: {get_clean_error_message(e)}")
        
        if 'PROMPT_TEMPLATES' not in globals():
            self.logger.critical("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.")
            raise NameError("PROMPT_TEMPLATES not found. Please run the cell that defines the prompt dictionary.")

        self.ai_agent = AIAgent(
            spark=self.spark,
            logger=self.logger,
            worker_llm_config=AI_MODEL_NAME,
            judge_llm_config=AI_MODEL_NAME,
            prompt_templates=PROMPT_TEMPLATES,
            default_timeout_seconds=self.llm_timeout_seconds,
            max_retry_attempts=self.max_retry_attempts,
            status_emitter=self._emit_prompt_status
        )
        self.ai_agent.enable_llm_cache(self.session_id)
        
        self.translation_service = TranslationService(ai_agent=self.ai_agent, logger=self.logger)
        
        # === NEW: Column Registry (Bitmap Technique) ===
        self.column_id_map = {}   # FQN -> ID
        self.id_column_map = {}   # ID -> {fqn, description}
        self.next_column_id = 1
        # === NEW: Table Registry (Bitmap Technique) ===
        self.table_id_map = {}    # Table FQN -> ID
        self.id_table_map = {}    # ID -> table FQN
        self.next_table_id = 1
        self.registry_lock = threading.Lock()
        
        self.data_loader = None
        # Skip data loader if we're in docs-only mode (JSON file path provided)
        # Note: "use cases" is always generated (removed from widget options)
        if not self.json_file_path:
            if ("PDF Catalog" in self.generate_choices or 
                "Use Cases Catalog PDF" in self.generate_choices or
                "Presentation" in self.generate_choices or
                "SQL Regeneration" not in self.generate_choices):
                
                if not (self.catalogs_str or self.schemas_str or self.tables_str):
                    self.logger.error("For 'use cases', 'PDF', or 'Presentation' you must provide at least one catalog, schema, or table.")
                else:
                    # === MEMORY-OPTIMIZED DATA LOADER ===
                    # Enable two-pass mode for intelligent batching based on table sizes
                    # Enable column sampling for tables with >250 columns
                    # Use streaming for schemas with >10K tables
                    self.data_loader = DataLoader(
                        catalogs=self.catalogs_str,
                        schemas=self.schemas_str,
                        tables=self.tables_str,
                        logger=self.logger,
                        enable_two_pass=True,           # Enable intelligent batching
                        enable_column_sampling=True,    # Sample wide tables
                        streaming_batch_size=1000,      # Stream in chunks of 1000 tables
                        max_parallelism=self.scan_parallelism,
                        schema_timeout_seconds=900      # 15 min timeout per schema
                    )
        
    def _ensure_inspire_database_exists(self):
        """Create the inspire database if it does not exist. Called once at the start of run()."""
        if not self.inspire_database:
            return
        try:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.inspire_database}")
            self.logger.info(f"✅ Ensured inspire database exists: {self.inspire_database}")
        except Exception as e:
            self.logger.error(f"Failed to create inspire database {self.inspire_database}: {get_clean_error_message(e)}")
            raise

    def _create_tracking_table(self):
        """Create the __inspire_usecases tracking table in the inspire database.
        Schema mirrors Excel catalog columns exactly (same order) plus:
          - session_id (first column)
          - updated_at (second-to-last)
          - notebook_path (stores workspace path of Genie Code notebook)
          - high_level_design (stores the high level design text)
          - genie_instruction (stores the generated Genie Code instruction)
          - generate_genie_code_instruction (Yes/No flag for on-demand Genie regeneration)
          - directly_involved_schema / foreign_key_relationships / available_tables_listing
            / technique_implementation_guidance (per-UC Genie prompt inputs cached at
            Discover time so Generate mode can rebuild the prompt without re-computing)
        Uses Delta format for MERGE support. Thread-safe via _tracking_lock.
        """
        if not self._tracking_table_name or self._tracking_table_created:
            return
        with self._tracking_lock:
            if self._tracking_table_created:
                return
            try:
                create_sql = f"""
                CREATE TABLE IF NOT EXISTS {self._tracking_table_name} (
                    session_id BIGINT,
                    id STRING,
                    business_domain STRING,
                    subdomain STRING,
                    use_case STRING,
                    description STRING,
                    inspire_score DOUBLE,
                    generated STRING,
                    type STRING,
                    analytics_technique STRING,
                    business_priority_alignment STRING,
                    strategic_goals_alignment STRING,
                    statement STRING,
                    solution STRING,
                    business_value STRING,
                    beneficiary STRING,
                    sponsor STRING,
                    tables_involved STRING,
                    primary_table STRING,
                    strategic_alignment DOUBLE,
                    roi DOUBLE,
                    reusability DOUBLE,
                    time_to_value DOUBLE,
                    data_availability DOUBLE,
                    data_accessibility DOUBLE,
                    architecture_fitness DOUBLE,
                    team_skills DOUBLE,
                    domain_knowledge DOUBLE,
                    people_allocation DOUBLE,
                    budget_allocation DOUBLE,
                    time_to_production DOUBLE,
                    value_score DOUBLE,
                    feasibility_score DOUBLE,
                    priority_score DOUBLE,
                    quality_score DOUBLE,
                    priority_reasons STRING,
                    quality_reasons STRING,
                    result_database STRING,
                    updated_at TIMESTAMP,
                    notebook_path STRING,
                    high_level_design STRING,
                    genie_instruction STRING,
                    generate_genie_code_instruction STRING,
                    has_genie_code STRING,
                    idea_theme STRING,
                    bob_score DOUBLE,
                    bob_tier1_score DOUBLE,
                    bob_tier2_score DOUBLE,
                    bob_tier3_score DOUBLE,
                    directly_involved_schema STRING,
                    foreign_key_relationships STRING,
                    available_tables_listing STRING,
                    technique_implementation_guidance STRING,
                    code_structure_mandate STRING
                ) USING DELTA
                """
                self.spark.sql(create_sql)
                self._migrate_tracking_table_schema()
                self._tracking_table_created = True
                self.logger.info(f"✅ Tracking table created/verified: {self._tracking_table_name}")
            except Exception as e:
                self.logger.error(f"Failed to create tracking table {self._tracking_table_name}: {get_clean_error_message(e)}")

    def _migrate_tracking_table_schema(self):
        """Migrate existing __inspire_usecases tables to the latest schema.
        Adds missing columns and renames legacy columns for backward compatibility.
        Safe to call repeatedly — each ALTER is idempotent (fails silently if column exists).
        """
        if not self._tracking_table_name:
            return
        rename_first = [
            (f"ALTER TABLE {self._tracking_table_name} RENAME COLUMN generated_sql TO notebook_path", "generated_sql→notebook_path"),
            (f"ALTER TABLE {self._tracking_table_name} RENAME COLUMN sql_generated TO generated", "sql_generated→generated"),
        ]
        for sql, desc in rename_first:
            try:
                self.spark.sql(sql)
                self.logger.info(f"   ✅ Renamed column: {desc}")
            except Exception:
                pass
        add_if_missing = [
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN notebook_path STRING", "notebook_path"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN high_level_design STRING", "high_level_design"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN genie_instruction STRING", "genie_instruction"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN generated STRING", "generated"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN inspire_score DOUBLE", "inspire_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN quality_score DOUBLE", "quality_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN generate_genie_code_instruction STRING", "generate_genie_code_instruction"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN has_genie_code STRING", "has_genie_code"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN idea_theme STRING", "idea_theme"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN bob_score DOUBLE", "bob_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN bob_tier1_score DOUBLE", "bob_tier1_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN bob_tier2_score DOUBLE", "bob_tier2_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN bob_tier3_score DOUBLE", "bob_tier3_score"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN directly_involved_schema STRING", "directly_involved_schema"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN foreign_key_relationships STRING", "foreign_key_relationships"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN available_tables_listing STRING", "available_tables_listing"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN technique_implementation_guidance STRING", "technique_implementation_guidance"),
            (f"ALTER TABLE {self._tracking_table_name} ADD COLUMN code_structure_mandate STRING", "code_structure_mandate"),
        ]
        for sql, col_name in add_if_missing:
            try:
                self.spark.sql(sql)
                self.logger.info(f"   ✅ Added column '{col_name}' to tracking table")
            except Exception:
                pass

    def _tracking_unique_view(self, prefix: str) -> str:
        """Generate a session-unique temp view name to avoid any cross-thread collision."""
        import random as _rnd
        return f"_inspire_{prefix}_{self.session_id}_{_rnd.randint(100000, 999999)}"

    def _tracking_save_to_disk(self, use_cases: list, tag: str) -> str:
        """Save use cases to a JSONL file on disk for memory-efficient streaming.
        Returns the file path. Each line is one JSON object.
        """
        import json as _json_mod
        _dir = os.path.join(tempfile.gettempdir(), f"inspire_{self.session_id}", "tracking_cache")
        os.makedirs(_dir, exist_ok=True)
        _path = os.path.join(_dir, f"{tag}.jsonl")
        with open(_path, 'w', encoding='utf-8') as f:
            for uc in use_cases:
                f.write(_json_mod.dumps(uc, ensure_ascii=False, default=str) + "\n")
        self.logger.debug(f"Saved {len(use_cases)} use cases to disk cache ({tag}): {_path}")
        return _path

    def _tracking_load_from_disk_chunked(self, tag: str, chunk_size: int = 500):
        """Generator that yields chunks of use cases from a JSONL disk cache file.
        Avoids loading the entire list into memory at once.
        """
        import json as _json_mod
        _path = os.path.join(tempfile.gettempdir(), f"inspire_{self.session_id}", "tracking_cache", f"{tag}.jsonl")
        if not os.path.exists(_path):
            self.logger.warning(f"Tracking disk cache not found: {_path}")
            return
        chunk = []
        with open(_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    chunk.append(_json_mod.loads(line))
                except Exception:
                    continue
                if len(chunk) >= chunk_size:
                    yield chunk
                    chunk = []
        if chunk:
            yield chunk

    @staticmethod
    def _uc_to_row(uc: dict, session_id: int, normalize_fn) -> tuple:
        """Convert a use case dict to a row tuple matching the full tracking table schema.
        Static helper to avoid closure over large objects.
        """
        def _sf(val, default=None):
            if val is None:
                return default
            try:
                return float(val)
            except (ValueError, TypeError):
                return default
        def _ss(val, default=''):
            if val is None:
                return default
            s = str(val).strip()
            return s if s else default

        uc_name = _ss(uc.get('Name'))
        uc_usecase = _ss(uc.get('usecase'))
        normalized = str(normalize_fn(uc_usecase, uc_name)) if (uc_usecase or uc_name) else ''
        return (
            session_id,
            _ss(uc.get('No')),
            _ss(uc.get('Business Domain')),
            _ss(uc.get('Subdomain')),
            normalized,
            uc_name,
            _sf(uc.get('Inspire Score')),
            _ss(uc.get('generated', 'N/A')),
            _ss(uc.get('type')),
            _ss(uc.get('Analytics Technique')),
            _ss(uc.get('Business Priority Alignment', 'General Improvement')),
            _ss(uc.get('Strategic Goals Alignment', 'General Improvement')),
            _ss(uc.get('Statement')),
            _ss(uc.get('Solution')),
            _ss(uc.get('Business Value')),
            _ss(uc.get('Beneficiary')),
            _ss(uc.get('Sponsor')),
            _ss(uc.get('Tables Involved')),
            _ss(uc.get('Primary Table')),
            _sf(uc.get('Strategic Alignment')),
            _sf(uc.get('Return on Investment')),
            _sf(uc.get('Reusability')),
            _sf(uc.get('Time to Value')),
            _sf(uc.get('Data Availability')),
            _sf(uc.get('Data Accessibility')),
            _sf(uc.get('Architecture Fitness')),
            _sf(uc.get('Team Skills')),
            _sf(uc.get('Domain Knowledge')),
            _sf(uc.get('People Allocation')),
            _sf(uc.get('Budget Allocation')),
            _sf(uc.get('Time to Production')),
            _sf(uc.get('Value')),
            _sf(uc.get('Feasibility')),
            _sf(uc.get('Priority')),
            _sf(uc.get('Quality Score')),
            _ss(uc.get('Justification')) if uc.get('Justification') else None,
            _ss(uc.get('Quality Summary')) if uc.get('Quality Summary') else None,
            _ss(uc.get('result_table')) if uc.get('result_table') else None,
            _ss(uc.get('notebook_path')) if uc.get('notebook_path') else None,
            _ss(uc.get('High Level Design')) if uc.get('High Level Design') else None,
            _ss(uc.get('genie_instruction')) if uc.get('genie_instruction') else None,
            _ss(uc.get('generate_genie_code_instruction')) if uc.get('generate_genie_code_instruction') else None,
            _ss(uc.get('has_genie_code')) if uc.get('has_genie_code') else None,
            _ss(uc.get('idea_theme')) if uc.get('idea_theme') else None,
            _sf(uc.get('bob_score')),
            _sf(uc.get('bob_tier1_score')),
            _sf(uc.get('bob_tier2_score')),
            _sf(uc.get('bob_tier3_score')),
            _ss(uc.get('directly_involved_schema')) if uc.get('directly_involved_schema') else None,
            _ss(uc.get('foreign_key_relationships')) if uc.get('foreign_key_relationships') else None,
            _ss(uc.get('available_tables_listing')) if uc.get('available_tables_listing') else None,
            _ss(uc.get('technique_implementation_guidance')) if uc.get('technique_implementation_guidance') else None,
        )

    _TRACKING_FULL_SCHEMA = None

    @classmethod
    def _get_tracking_full_schema(cls):
        if cls._TRACKING_FULL_SCHEMA is None:
            from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
            cls._TRACKING_FULL_SCHEMA = StructType([
                StructField("session_id", LongType(), False),
                StructField("id", StringType(), True),
                StructField("business_domain", StringType(), True),
                StructField("subdomain", StringType(), True),
                StructField("use_case", StringType(), True),
                StructField("description", StringType(), True),
                StructField("inspire_score", DoubleType(), True),
                StructField("generated", StringType(), True),
                StructField("type", StringType(), True),
                StructField("analytics_technique", StringType(), True),
                StructField("business_priority_alignment", StringType(), True),
                StructField("strategic_goals_alignment", StringType(), True),
                StructField("statement", StringType(), True),
                StructField("solution", StringType(), True),
                StructField("business_value", StringType(), True),
                StructField("beneficiary", StringType(), True),
                StructField("sponsor", StringType(), True),
                StructField("tables_involved", StringType(), True),
                StructField("primary_table", StringType(), True),
                StructField("strategic_alignment", DoubleType(), True),
                StructField("roi", DoubleType(), True),
                StructField("reusability", DoubleType(), True),
                StructField("time_to_value", DoubleType(), True),
                StructField("data_availability", DoubleType(), True),
                StructField("data_accessibility", DoubleType(), True),
                StructField("architecture_fitness", DoubleType(), True),
                StructField("team_skills", DoubleType(), True),
                StructField("domain_knowledge", DoubleType(), True),
                StructField("people_allocation", DoubleType(), True),
                StructField("budget_allocation", DoubleType(), True),
                StructField("time_to_production", DoubleType(), True),
                StructField("value_score", DoubleType(), True),
                StructField("feasibility_score", DoubleType(), True),
                StructField("priority_score", DoubleType(), True),
                StructField("quality_score", DoubleType(), True),
                StructField("priority_reasons", StringType(), True),
                StructField("quality_reasons", StringType(), True),
                StructField("result_database", StringType(), True),
                StructField("notebook_path", StringType(), True),
                StructField("high_level_design", StringType(), True),
                StructField("genie_instruction", StringType(), True),
                StructField("generate_genie_code_instruction", StringType(), True),
                StructField("has_genie_code", StringType(), True),
                StructField("idea_theme", StringType(), True),
                StructField("bob_score", DoubleType(), True),
                StructField("bob_tier1_score", DoubleType(), True),
                StructField("bob_tier2_score", DoubleType(), True),
                StructField("bob_tier3_score", DoubleType(), True),
                StructField("directly_involved_schema", StringType(), True),
                StructField("foreign_key_relationships", StringType(), True),
                StructField("available_tables_listing", StringType(), True),
                StructField("technique_implementation_guidance", StringType(), True),
            ])
        return cls._TRACKING_FULL_SCHEMA

    def _tracking_replace_session(self, use_cases: list):
        """UPDATE 1 of 2: Delete all rows for this session and re-insert with ALL columns
        matching Excel exactly. Called after Excel generation when all fields (scores, quality,
        primary table, result database) are available.
        Saves to disk first, then streams from disk in chunks to avoid memory overload.
        Thread-safe via _tracking_lock.
        """
        if not self._tracking_table_name or not use_cases:
            return
        self._create_tracking_table()
        self._tracking_save_to_disk(use_cases, "update1_post_excel")
        with self._tracking_lock:
            try:
                self.spark.sql(
                    f"DELETE FROM {self._tracking_table_name} WHERE session_id = {self.session_id}"
                )
                from pyspark.sql.functions import current_timestamp as _cts
                _schema = self._get_tracking_full_schema()
                total_inserted = 0
                for chunk in self._tracking_load_from_disk_chunked("update1_post_excel", chunk_size=500):
                    rows = [self._uc_to_row(uc, self.session_id, self._normalize_usecase) for uc in chunk]
                    if not rows:
                        continue
                    df = self.spark.createDataFrame(rows, schema=_schema)
                    df = df.withColumn("updated_at", _cts())
                    df.write.format("delta").mode("append").saveAsTable(self._tracking_table_name)
                    total_inserted += len(rows)
                    del rows, chunk, df
                self.logger.info(f"✅ [UPDATE 1/2] Replaced session: {total_inserted} use cases (all Excel columns)")
            except Exception as e:
                self.logger.error(f"Failed to replace session rows in tracking table: {get_clean_error_message(e)}")

    def _calculate_dynamic_parallelism(self, total_tables, total_schema_chars, safe_context_limit, base_prompt_size):
        memory_pool = max(1, self.cluster_memory_gb * max(self.cluster_worker_count, 1))
        max_by_memory = max(2, min(8, int(memory_pool // 8)))
        if total_tables <= 0:
            return max_by_memory, 0, 0, 0, max_by_memory
        avg_table_chars = max(1, int(total_schema_chars / total_tables))
        available_chars = max(1, safe_context_limit - base_prompt_size)
        tables_per_batch = max(1, int(available_chars // avg_table_chars))
        est_batches = int((total_tables + tables_per_batch - 1) // tables_per_batch) if tables_per_batch else total_tables
        size_factor = 1.0
        if avg_table_chars >= 24000:
            size_factor = 0.5
        elif avg_table_chars >= 18000:
            size_factor = 0.7
        elif avg_table_chars >= 12000:
            size_factor = 0.85
        recommended = int(max_by_memory * size_factor)
        if recommended < 2:
            recommended = 2
        if recommended > max_by_memory:
            recommended = max_by_memory
        if est_batches > 0 and recommended > est_batches:
            recommended = est_batches
        if recommended < 1:
            recommended = 1
        return recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory
        
    def _get_translations(self, language: str) -> dict:
        return self.translation_service.get_translations(language)

    def _report_processing_honesty(self):
        """
        Generates and displays a detailed report of what happened during processing.
        Reports on batch splits, column drops, and whether all data was processed completely.
        """
        log_print(f"\n{'='*80}")
        log_print(f"📊 PROCESSING HONESTY REPORT")
        log_print(f"{'='*80}\n")
        
        h = self.processing_honesty
        
        log_print(f"📈 Overall Statistics:")
        log_print(f"   • Total tables discovered: {h['total_tables_discovered']}")
        log_print(f"   • Total tables processed: {h['total_tables_processed']}")
        log_print(f"   • Total batches created: {h['total_batches_created']}")
        log_print(f"   • Total batch splits performed: {h['total_batch_splits']}")
        
        if h['batch_split_history']:
            proactive_splits = [s for s in h['batch_split_history'] if s.get('split_type') == 'Proactive']
            reactive_splits = [s for s in h['batch_split_history'] if s.get('split_type') == 'Reactive']
            
            log_print(f"\n⚡ Batch Splitting Details:")
            log_print(f"   {h['total_batch_splits']} batch(es) were split to fit LLM context limits:")
            log_print(f"   • Proactive splits (detected before LLM call): {len(proactive_splits)}")
            log_print(f"   • Reactive splits (after LLM error): {len(reactive_splits)}")
            
            for idx, split in enumerate(h['batch_split_history'], 1):
                split_icon = "🔮" if split.get('split_type') == 'Proactive' else "⚡"
                log_print(f"   {idx}. {split_icon} {split.get('split_type', 'Unknown')} | Batch '{split['batch']}': {split['original_tables']} tables → {split['split_into']} sub-batches")
                log_print(f"      - Sub-batch 1: {split['sub_batch_1_tables']} tables")
                log_print(f"      - Sub-batch 2: {split['sub_batch_2_tables']} tables")
        else:
            log_print(f"\n✅ No batch splitting was required - all batches fit within LLM context limits")
        
        if h['tables_with_columns_dropped']:
            log_print(f"\n⚠️  Column Dropping Details:", level="WARNING")
            log_print(f"   {len(h['tables_with_columns_dropped'])} table(s) had columns dropped to fit LLM limits:")
            for idx, drop_info in enumerate(h['tables_with_columns_dropped'], 1):
                business_tag = "🔵 BUSINESS" if drop_info['is_business'] else "⚪ non-business"
                log_print(f"   {idx}. {business_tag} | {drop_info['table']}")
                log_print(f"      - Original columns: {drop_info['original_columns']}")
                log_print(f"      - Kept columns: {drop_info['kept_columns']} ({drop_info['drop_percentage']:.1f}% dropped)")
                log_print(f"      - Dropped columns: {drop_info['dropped_columns']}")
        else:
            log_print(f"\n✅ No columns were dropped - all tables processed with full schema")
        
        log_print(f"\n{'='*80}")
        log_print(f"🎯 HONESTY ASSESSMENT:")
        log_print(f"{'='*80}")
        
        honesty_percentage = 100.0
        issues = []
        
        if h['batch_split_history']:
            issues.append(f"• {h['total_batch_splits']} batch split(s) occurred (but all tables were still processed)")
        
        if h['tables_with_columns_dropped']:
            total_columns_dropped = sum(d['dropped_columns'] for d in h['tables_with_columns_dropped'])
            total_columns_original = sum(d['original_columns'] for d in h['tables_with_columns_dropped'])
            drop_percentage = (total_columns_dropped / total_columns_original * 100) if total_columns_original > 0 else 0
            
            issues.append(f"• {len(h['tables_with_columns_dropped'])} table(s) had columns dropped ({drop_percentage:.1f}% of columns from affected tables)")
            
            honesty_percentage -= min(drop_percentage / 2, 30)
        
        if not issues:
            log_print(f"\n✅ 100% HONEST - All tables processed completely with all columns")
            log_print(f"   No compromises were made during processing.")
        else:
            log_print(f"\n📊 Honesty Score: {honesty_percentage:.1f}%")
            log_print(f"\n   Processing required the following compromises:")
            for issue in issues:
                log_print(f"   {issue}")
            log_print(f"\n   ℹ️  Note: Batch splits don't affect completeness - all tables are still processed.")
            log_print(f"   ⚠️  Column drops DO affect completeness - some schema details were omitted.", level="WARNING")
        
        log_print(f"\n{'='*80}\n")
    
    def _get_lang_abbr(self, language: str) -> str:
        """Returns a 2-letter abbreviation for a language."""
        lang_map = {
            "english": "en",
            "arabic": "ar",
            "french": "fr",
            "spanish": "es",
            "german": "de",
            "portuguese": "pt",
            "italian": "it",
            "japanese": "ja",
            "korean": "ko",
            "chinese": "zh"
        }
        return lang_map.get(language.lower(), language.lower()[:2])

    # === BUSINESS CONTEXT AND SPONSOR MAPPING ===
    def _interpret_generation_instructions(self) -> dict:
        """Parse user generation instructions into structured directives using LLM.
        Returns a dict with table_joins, goal_setting, forced_techniques, table_ignores, per_prompt_instructions.
        """
        raw_instructions = getattr(self, "user_generation_instructions", "") or ""
        default_result = {
            "table_joins": [],
            "goal_setting": {"has_goal": False, "goal_description": "", "goal_summary_sentence": "", "strategic_goal_match": ""},
            "forced_techniques": [],
            "table_ignores": [],
            "per_prompt_instructions": {
                "FILTER_BUSINESS_TABLES_PROMPT": "",
                "BASE_USE_CASE_GEN_PROMPT": "",
                "USE_CASE_VALUE_SCORE_PROMPT": "",
                "USE_CASE_QUALITY_SCORE_PROMPT": "",
                "COMBINED_VALUE_QUALITY_SCORE_PROMPT": "",
                "REVIEW_USE_CASES_PROMPT": "",
                "DOMAIN_FINDER_PROMPT": "",
                "SUBDOMAIN_DETECTOR_PROMPT": "",
                "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT": "",
                "SUMMARY_GEN_PROMPT": "",
            }
        }
        if not raw_instructions.strip():
            self.logger.info("No generation instructions provided - skipping interpretation")
            self.parsed_generation_instructions = default_result
            return default_result
        
        self.logger.info(f"Interpreting generation instructions ({len(raw_instructions)} chars)...")
        
        def _extract_json_robust(response_text: str) -> dict:
            """Multi-strategy JSON extraction that handles thinker model quirks."""
            if not response_text:
                return None
            
            strategies = []
            
            cleaned = clean_json_response(response_text)
            strategies.append(("clean_json_response", cleaned))
            
            single_to_double = cleaned.replace("'", '"')
            strategies.append(("single_to_double_quotes", single_to_double))
            
            json_block_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response_text)
            if json_block_match:
                strategies.append(("markdown_fence_extract", json_block_match.group(1).strip()))
            
            brace_match = re.search(r'(\{[\s\S]*"per_prompt_instructions"[\s\S]*\})', response_text)
            if brace_match:
                strategies.append(("per_prompt_instructions_anchor", brace_match.group(1)))
            
            outermost_match = re.search(r'(\{[\s\S]*\})\s*$', response_text)
            if outermost_match:
                strategies.append(("outermost_brace", outermost_match.group(1)))
            
            for strategy_name, candidate in strategies:
                try:
                    result = json.loads(candidate)
                    if isinstance(result, dict):
                        if "data" in result:
                            result = result["data"]
                        self.logger.info(f"JSON parsed successfully using strategy: {strategy_name}")
                        return result
                except (json.JSONDecodeError, TypeError):
                    continue
            
            try:
                fixed = re.sub(r"(?<!\\)'", '"', response_text)
                fixed = re.sub(r',\s*}', '}', fixed)
                fixed = re.sub(r',\s*]', ']', fixed)
                fixed_clean = clean_json_response(fixed)
                result = json.loads(fixed_clean)
                if isinstance(result, dict):
                    if "data" in result:
                        result = result["data"]
                    self.logger.info("JSON parsed with aggressive quote/trailing-comma fix")
                    return result
            except (json.JSONDecodeError, TypeError):
                pass
            
            self.logger.error(f"All JSON extraction strategies failed. Response preview: {response_text[:500]}")
            return None
        
        max_attempts = 2
        for attempt in range(1, max_attempts + 1):
            try:
                prompt_vars = {
                    "generation_instructions": raw_instructions,
                    "business_name": self.business_name,
                }
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Generation_Instructions_Interpreter_Attempt{attempt}",
                    worker_prompt_path="GENERAL_INSTRUCTION_INTERPRETER_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    timeout_override=120,
                    skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                )
                if not response_raw:
                    self.logger.warning(f"Interpreter attempt {attempt} returned empty response")
                    continue
                
                parsed = _extract_json_robust(response_raw)
                if parsed is None:
                    self.logger.warning(f"Interpreter attempt {attempt} returned unparseable JSON - will retry")
                    continue
                
                for key in default_result:
                    if key not in parsed:
                        parsed[key] = default_result[key]
                
                if not isinstance(parsed.get("per_prompt_instructions"), dict):
                    parsed["per_prompt_instructions"] = default_result["per_prompt_instructions"]
                for pname in default_result["per_prompt_instructions"]:
                    if pname not in parsed["per_prompt_instructions"]:
                        parsed["per_prompt_instructions"][pname] = ""
                
                goal = parsed.get("goal_setting", {})
                if isinstance(goal, dict) and goal.get("has_goal"):
                    goal_match = goal.get("strategic_goal_match", "")
                    if goal_match:
                        self.user_strategic_goals = [goal_match]
                        self.logger.info(f"✅ Goal extracted from generation instructions: {goal_match}")
                    goal_desc = goal.get("goal_description", "")
                    if goal_desc:
                        self.logger.info(f"✅ Goal description: {goal_desc}")
                
                joins = parsed.get("table_joins", [])
                if joins:
                    self.logger.info(f"✅ Table joins extracted: {len(joins)} join(s)")
                    for j in joins:
                        self.logger.info(f"  JOIN: {j.get('left_table', '?')} <-> {j.get('right_table', '?')} ON {j.get('join_columns', [])}")
                
                ignores = parsed.get("table_ignores", [])
                if ignores:
                    self.logger.info(f"✅ Table ignore patterns extracted: {len(ignores)} pattern(s)")
                    for ig in ignores:
                        self.logger.info(f"  IGNORE: {ig.get('pattern', '?')} (type: {ig.get('type', '?')})")
                
                techniques = parsed.get("forced_techniques", [])
                if techniques:
                    self.logger.info(f"✅ Forced techniques extracted: {len(techniques)} technique(s)")
                    for t in techniques:
                        self.logger.info(f"  TECHNIQUE: {t.get('technique', '?')} ({t.get('context', 'all use cases')})")
                
                has_instructions = any(
                    v.strip() for v in parsed.get("per_prompt_instructions", {}).values() if isinstance(v, str)
                )
                if has_instructions:
                    self.logger.info(f"✅ Per-prompt instructions generated for {sum(1 for v in parsed['per_prompt_instructions'].values() if isinstance(v, str) and v.strip())} prompts")
                else:
                    self.logger.warning("⚠️ Interpreter returned parsed JSON but no per-prompt instructions were generated")
                
                self.parsed_generation_instructions = parsed
                self.logger.info("✅ Generation instructions interpreted successfully")
                return parsed
            except Exception as e:
                self.logger.error(f"Interpreter attempt {attempt} failed: {get_clean_error_message(e)}")
                if attempt < max_attempts:
                    self.logger.info(f"Retrying interpretation (attempt {attempt + 1}/{max_attempts})...")
        
        self.logger.error("All interpreter attempts failed - using defaults. USER INSTRUCTIONS WILL NOT BE APPLIED.")
        self.parsed_generation_instructions = default_result
        return default_result
    
    def _get_generation_instruction_for_prompt(self, prompt_name: str) -> str:
        """Get the formatted generation instruction section for a specific prompt."""
        parsed = getattr(self, "parsed_generation_instructions", {}) or {}
        per_prompt = parsed.get("per_prompt_instructions", {})
        instruction = str(per_prompt.get(prompt_name, "") or "").strip()
        if instruction:
            return f"""### NON-NEGOTIABLE GENERATION INSTRUCTIONS (MUST FOLLOW STRICTLY)

The following instructions were provided by the user and MUST be followed without exception. Violating any of these instructions is an IMPLEMENTATION FAILURE.

{instruction}"""
        return "*(No additional generation instructions provided)*"
    
    def _filter_use_cases_by_goal_alignment(self, use_cases: list) -> list:
        """Hard filter: remove use cases that do not align with the user's stated goal.
        Only triggered when the interpreter extracted has_goal=True.
        Returns the filtered list.
        """
        parsed = getattr(self, "parsed_generation_instructions", {}) or {}
        goal = parsed.get("goal_setting", {})
        if not (isinstance(goal, dict) and goal.get("has_goal")):
            return use_cases

        goal_desc = goal.get("goal_description", "") or ""
        if not goal_desc.strip():
            self.logger.warning("has_goal is True but goal_description is empty - skipping goal filter")
            return use_cases

        if not use_cases:
            return use_cases

        _sep = "=" * 80
        self.logger.info(f"\n{_sep}")
        self.logger.info(f"🎯 GOAL ALIGNMENT FILTER: Removing use cases that do not match user goal")
        self.logger.info(f"{_sep}")
        self.logger.info(f"Goal: {goal_desc}")
        self.logger.info(f"Use cases to classify: {len(use_cases)}")
        _goal_preview = goal_desc[:100] + "..." if len(goal_desc) > 100 else goal_desc
        log_print(f"\n🎯 GOAL ALIGNMENT FILTER: Checking {len(use_cases)} use cases against goal: \"{_goal_preview}\"")

        context_limit = get_max_context_chars("English", "GOAL_ALIGNMENT_FILTER_PROMPT")
        base_prompt_size = len(goal_desc) + 500

        batches = []
        current_batch = []
        current_size = base_prompt_size

        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            uc_name = str(uc.get("Name", ""))
            uc_desc = str(uc.get("Description", "") or uc.get("Statement", "") or "")
            row_text = f"| {uc_id} | {uc_name} | {uc_desc[:200]} |\n"
            row_size = len(row_text)

            if current_size + row_size > context_limit * 0.85 and current_batch:
                batches.append(current_batch)
                current_batch = [uc]
                current_size = base_prompt_size + row_size
            else:
                current_batch.append(uc)
                current_size += row_size

        if current_batch:
            batches.append(current_batch)

        self.logger.info(f"Batching: {len(batches)} batch(es) for goal alignment classification")

        aligned_ids = set()
        not_aligned_ids = set()

        def _classify_batch(batch: list, batch_idx: int) -> None:
            md_rows = ["| ID | Name | Description |\n|---|---|---|\n"]
            for uc in batch:
                _id = str(uc.get("No", "")).replace("|", "/")
                _name = str(uc.get("Name", "")).replace("|", "/")
                _desc = str(uc.get("Description", "") or uc.get("Statement", "") or "").replace("|", "/")[:200]
                md_rows.append(f"| {_id} | {_name} | {_desc} |\n")
            use_cases_table = "".join(md_rows)

            prompt_vars = {
                "goal_description": goal_desc,
                "use_cases_table": use_cases_table,
            }

            response_raw = self.ai_agent.run_worker(
                step_name=f"Goal_Alignment_Filter_Batch_{batch_idx}",
                worker_prompt_path="GOAL_ALIGNMENT_FILTER_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=120
            )

            if not response_raw:
                self.logger.warning(f"[Goal Filter Batch {batch_idx}] Empty response - keeping all use cases in this batch")
                for uc in batch:
                    aligned_ids.add(str(uc.get("No", "")))
                return

            for line in response_raw.strip().split("\n"):
                line = line.strip()
                if not line or line.startswith("|") or line.startswith("ID"):
                    continue
                parts = [p.strip() for p in line.split(",")]
                if len(parts) >= 2:
                    uc_id = parts[0]
                    verdict = parts[1].upper().strip()
                    if verdict == "YES":
                        aligned_ids.add(uc_id)
                    elif verdict == "NO":
                        not_aligned_ids.add(uc_id)
                    else:
                        aligned_ids.add(uc_id)

        if len(batches) == 1:
            _classify_batch(batches[0], 1)
        else:
            import concurrent.futures as _goal_futures
            workers = min(len(batches), 6)
            self.logger.info(f"Running {len(batches)} goal alignment batches with {workers} workers")
            with _goal_futures.ThreadPoolExecutor(max_workers=workers, thread_name_prefix="GoalFilter") as executor:
                futures = []
                for idx, batch in enumerate(batches):
                    futures.append(executor.submit(_classify_batch, batch, idx + 1))
                for f in _goal_futures.as_completed(futures):
                    try:
                        f.result(timeout=180)
                    except Exception as e:
                        self.logger.error(f"Goal filter batch failed: {get_clean_error_message(e)} - keeping unclassified use cases")

        unclassified = []
        for uc in use_cases:
            uc_id = str(uc.get("No", ""))
            if uc_id not in aligned_ids and uc_id not in not_aligned_ids:
                unclassified.append(uc_id)
                aligned_ids.add(uc_id)

        if unclassified:
            self.logger.warning(f"⚠️ {len(unclassified)} use cases were not classified by LLM - keeping them (fail-safe)")

        filtered = [uc for uc in use_cases if str(uc.get("No", "")) in aligned_ids]
        removed = [uc for uc in use_cases if str(uc.get("No", "")) in not_aligned_ids]

        if removed:
            self.logger.info(f"\n🎯 GOAL FILTER RESULTS:")
            self.logger.info(f"   ✅ ALIGNED (kept): {len(filtered)} use cases")
            self.logger.info(f"   ❌ NOT ALIGNED (removed): {len(removed)} use cases")
            log_print(f"\n🎯 GOAL FILTER RESULTS:")
            log_print(f"   ✅ ALIGNED with goal: {len(filtered)} use cases kept")
            log_print(f"   ❌ NOT ALIGNED with goal: {len(removed)} use cases removed:")
            for uc in removed:
                _uc_no = uc.get('No', '?')
                _uc_name = uc.get('Name', '?')
                self.logger.info(f"      ❌ {_uc_no} - {_uc_name}")
                log_print(f"      ❌ {_uc_no} - {_uc_name}")
        else:
            self.logger.info(f"✅ All {len(filtered)} use cases are aligned with the goal")
            log_print(f"✅ All {len(filtered)} use cases are aligned with the goal")

        if not filtered:
            self.logger.error("🚨 GOAL FILTER removed ALL use cases! Falling back to keeping all.")
            log_print("🚨 GOAL FILTER removed ALL use cases! Keeping all as fallback.")
            return use_cases

        self.logger.info(f"{'=' * 80}\n")
        return filtered

    def _get_business_context_from_llm(self) -> dict:
        """
        Calls the BUSINESS_CONTEXT_WORKER_PROMPT to get comprehensive business context.
        Returns a dict with 15 business context fields.
        """
        self.logger.info(f"🔍 Calling Business Context Worker for: {self.business_name}")
        
        try:
            # First, determine the industry using a simple analysis
            industry = self.business_name  # Default to business name
            
            prompt_vars = {
                "name": self.business_name,
                "industry": industry,
                "type_description": "a business entity or organization",
                "type_label": "organization"
            }
            
            self.logger.info(f"⏳ Waiting for LLM response (Business Context extraction)...")
            response_json = self.ai_agent.run_worker(
                step_name="Business_Context_Extraction",
                worker_prompt_path="BUSINESS_CONTEXT_WORKER_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None
            )
            self.logger.info(f"✅ Received LLM response, parsing business context...")
            
            context_data = None
            strategies = [
                ("clean_json", clean_json_response(response_json)),
            ]
            fence_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response_json)
            if fence_match:
                strategies.append(("markdown_fence", fence_match.group(1).strip()))
            last_brace = re.search(r'(\{[\s\S]*\})\s*$', response_json)
            if last_brace:
                strategies.append(("last_json_object", last_brace.group(1)))
            strategies.append(("single_to_double", clean_json_response(response_json).replace("'", '"')))
            fixed_resp = re.sub(r',\s*}', '}', re.sub(r',\s*]', ']', re.sub(r"(?<!\\\\)'", '"', response_json)))
            strategies.append(("aggressive_fix", clean_json_response(fixed_resp)))
            for strat_name, candidate in strategies:
                try:
                    result = json.loads(candidate)
                    if isinstance(result, dict):
                        context_data = result.get("data", result) if "data" in result else result
                        self.logger.info(f"Business context JSON parsed via strategy: {strat_name}")
                        break
                except (json.JSONDecodeError, TypeError):
                    continue
            if context_data is None:
                self.logger.error(f"All JSON strategies failed for business context. Preview: {str(response_json)[:500]}")
                return {}
            
            self.logger.info(f"✅ Business Context Worker extracted {len(context_data)} fields")
            return context_data
            
        except Exception as e:
            self.logger.error(f"Failed to get business context from LLM: {get_clean_error_message(e)}")
            return {}
    
    def _merge_business_contexts(self, llm_context: dict, user_context_str: str) -> dict:
        """
        Merges LLM-generated business context with user-provided context.
        User context ALWAYS takes precedence and overrides LLM context.
        
        Args:
            llm_context: Dictionary from Business Context Worker (15 fields)
            user_context_str: Comma-separated string from user (Business Context widget)
            
        Returns:
            Merged dictionary with user values taking precedence
        """
        merged_context = llm_context.copy()
        
        if not user_context_str or not user_context_str.strip():
            self.logger.info("No user-provided business context. Using LLM context only.")
            return merged_context
        
        # Parse user context - assume it's comma-separated values that override specific fields
        # For simplicity, we'll treat the entire user context as additional focus areas
        user_focus_areas = [area.strip() for area in user_context_str.split(',') if area.strip()]
        
        if user_focus_areas:
            self.logger.info(f"✅ User provided {len(user_focus_areas)} business context items - these will take precedence")
            
            # Store user focus areas separately for later use
            # We'll use them to override or extend LLM context
            merged_context['user_focus_areas'] = ', '.join(user_focus_areas)
            
            # Also update relevant fields with user context
            if 'business_units_divisions_domains_subdomains' in merged_context:
                # Prepend user areas to ensure they're used
                existing = merged_context['business_units_divisions_domains_subdomains']
                merged_context['business_units_divisions_domains_subdomains'] = ', '.join(user_focus_areas) + ', ' + existing
            else:
                merged_context['business_units_divisions_domains_subdomains'] = ', '.join(user_focus_areas)
        
        return merged_context
    
    # === USER DOMAIN ASSIGNMENT ===
    def _assign_to_user_domains(self, use_cases: list, user_domains: list, language: str) -> list:
        """
        Assigns use cases to user-provided business domains using LLM, 
        then discovers subdomains within each domain.
        
        This method is called when user provides specific business domains.
        The user provides TOP-LEVEL DOMAINS ONLY - Inspire discovers subdomains automatically.
        
        Args:
            use_cases: List of use case dictionaries
            user_domains: List of user-provided domain names (top-level only)
            language: Output language
            
        Returns:
            List of use cases with Business Domain and Subdomain properly assigned
        """
        import io
        import csv
        from collections import defaultdict, Counter
        from concurrent.futures import ThreadPoolExecutor
        import concurrent.futures
        
        self.logger.info(f"📍 Assigning {len(use_cases)} use cases to {len(user_domains)} user-provided domains...")
        self.logger.info(f"📍 User provided TOP-LEVEL domains only: {', '.join(user_domains)}")
        self.logger.info(f"📍 Inspire will discover SUBDOMAINS within each domain automatically")
        
        if not use_cases or not user_domains:
            return use_cases
        
        # Build a simple prompt for domain assignment
        domain_list_str = ", ".join([f'"{d}"' for d in user_domains])
        
        # Convert use cases to CSV for LLM
        output = io.StringIO()
        fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                     'Business Value', 'Beneficiary', 'Sponsor', 'Tables Involved']
        writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(use_cases)
        use_cases_csv = output.getvalue()
        
        # Build custom prompt for user domain assignment
        assignment_prompt = f"""You are an expert business analyst. Your task is to assign each use case to ONE of the following EXACT business domains:

**ALLOWED DOMAINS (USE EXACTLY AS PROVIDED)**: {domain_list_str}

**🚨 CRITICAL RULES 🚨**:
1. You MUST assign EVERY use case to EXACTLY ONE domain from the list above
2. You MUST use the domain names EXACTLY as provided - no modifications
3. You MUST NOT create or invent any new domain names
4. Every use case MUST have a domain assigned

**USE CASES TO ASSIGN**:
{use_cases_csv}

**OUTPUT FORMAT** (CSV only, no explanation):
```csv
use_case_id,domain
```

For each use case, output ONE row with the use case ID and the assigned domain.
Start your response with the CSV header line: use_case_id,domain
"""

        try:
            # Call LLM for domain assignment using _call_ai_query directly
            response_raw = self.ai_agent._call_ai_query(
                prompt=assignment_prompt,
                prompt_name=f"Assign_User_Domains_{language}",
                response_schema=None,
                display_name=f"Assign_User_Domains_{language}"
            )
            
            # Clean response
            response_clean = clean_json_response(response_raw)
            
            # Parse CSV response
            csv_rows = CSVParser.parse_csv_string(
                response_clean,
                logger=self.logger,
                context="User domain assignment"
            )
            
            # Build domain assignment map
            domain_map = {}
            for row in csv_rows:
                uc_id = row.get('use_case_id', '').strip()
                domain = row.get('domain', '').strip()
                if uc_id and domain:
                    # Validate domain is in user list
                    if domain in user_domains:
                        domain_map[uc_id] = domain
                    else:
                        # Try case-insensitive match
                        for ud in user_domains:
                            if ud.lower() == domain.lower():
                                domain_map[uc_id] = ud
                                break
                        else:
                            # Assign to first domain as fallback
                            domain_map[uc_id] = user_domains[0]
                            self.logger.warning(f"Use case {uc_id} assigned invalid domain '{domain}', defaulting to '{user_domains[0]}'")
            
            # Apply domain assignments (only domain, NOT subdomain yet)
            assigned_count = 0
            for uc in use_cases:
                uc_id = uc.get('No', '')
                if uc_id in domain_map:
                    uc['Business Domain'] = domain_map[uc_id]
                    assigned_count += 1
                else:
                    # Fallback: assign to first user domain
                    uc['Business Domain'] = user_domains[0]
                    self.logger.warning(f"Use case {uc_id} not in LLM response, defaulting to '{user_domains[0]}'")
            
            self.logger.info(f"✅ Successfully assigned {assigned_count}/{len(use_cases)} use cases to user domains")
            
            # Log domain distribution
            domain_counts = Counter(uc.get('Business Domain', 'Unknown') for uc in use_cases)
            for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
                self.logger.info(f"   📁 {domain}: {count} use cases")
            
            # === NOW DISCOVER SUBDOMAINS WITHIN EACH DOMAIN ===
            self.logger.info(f"📍 STEP 2: Discovering subdomains within each user-provided domain...")
            
            # Group use cases by domain
            domain_usecases_map = defaultdict(list)
            for uc in use_cases:
                domain = uc.get('Business Domain', '').strip()
                if domain:
                    domain_usecases_map[domain].append(uc)
            
            # ADAPTIVE PARALLELISM: Calculate based on number of domains and use cases
            subdomain_parallelism, reason = calculate_adaptive_parallelism(
                "subdomain_detection", self.max_parallelism,
                num_items=len(use_cases),
                num_domains=len(domain_usecases_map),
                is_llm_operation=True, logger=self.logger
            )
            log_adaptive_parallelism_decision("subdomain_detection", subdomain_parallelism, self.max_parallelism, reason)
            self.logger.info(f"Processing {len(domain_usecases_map)} domains for subdomain discovery...")
            
            final_use_cases_with_subdomains = []
            
            subdomain_workers = min(subdomain_parallelism, len(domain_usecases_map))
            self.logger.info(f"🚀 Processing {len(domain_usecases_map)} domains in parallel (workers={subdomain_workers}) for subdomain discovery...")
            import time as _time
            
            domain_items = list(domain_usecases_map.items())
            total_d = len(domain_items)
            
            def _detect_subdomain_task(args):
                idx, (domain_name, domain_use_cases) = args
                domain_start = _time.time()
                self.logger.info(f"🔄 [{idx}/{total_d}] Domain '{domain_name}' ({len(domain_use_cases)} use cases)...")
                log_print(f"🔄 [{idx}/{total_d}] Subdomain discovery: '{domain_name}'")
                try:
                    result = self._detect_subdomains_for_domain(domain_name, domain_use_cases, language)
                    elapsed = _time.time() - domain_start
                    if result:
                        self.logger.info(f"✅ [{idx}/{total_d}] Domain '{domain_name}' complete in {elapsed:.1f}s ({len(result)} use cases)")
                        return result
                    else:
                        fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                        self.logger.warning(f"⚠️ [{idx}/{total_d}] Domain '{domain_name}': empty after {elapsed:.1f}s, used defaults")
                        return fallback
                except Exception as e:
                    elapsed = _time.time() - domain_start
                    self.logger.error(f"❌ [{idx}/{total_d}] Domain '{domain_name}' failed after {elapsed:.1f}s: {get_clean_error_message(e)}")
                    fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                    return fallback
            
            _subdomain_total_timeout = self.llm_timeout_seconds * 3 * len(domain_items) + 120
            with _SafeExecutorContext(max_workers=subdomain_workers, thread_name_prefix="subdomain", logger=self.logger, name="SubdomainDetection") as ctx:
                futures = {ctx.submit(_detect_subdomain_task, (i, item)): i 
                           for i, item in enumerate(domain_items, 1)}
                try:
                    for future in safe_as_completed(futures, timeout=_subdomain_total_timeout):
                        try:
                            result_ucs = future.result(timeout=self.llm_timeout_seconds * 3)
                            if result_ucs:
                                final_use_cases_with_subdomains.extend(result_ucs)
                        except Exception as exc:
                            self.logger.error(f"Subdomain detection task failed: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Subdomain detection timed out globally ({_subdomain_total_timeout}s)")
            
            self.logger.info(f"✅ Domain assignment and subdomain discovery complete! {len(final_use_cases_with_subdomains)} use cases processed")
            
            self.logger.info(f"📍 STEP 2.5: Cross-domain subdomain deduplication (user-domain path)...")
            final_use_cases_with_subdomains = self._merge_duplicate_subdomains(final_use_cases_with_subdomains)
            final_use_cases_with_subdomains = self._rename_placeholder_domains_from_subdomains(final_use_cases_with_subdomains)
            
            subdomain_counts = Counter(uc.get('Subdomain', 'Unknown') for uc in final_use_cases_with_subdomains)
            self.logger.info(f"📊 Subdomain distribution:")
            for subdomain, count in sorted(subdomain_counts.items(), key=lambda x: -x[1]):
                self.logger.info(f"   📁 {subdomain}: {count} use cases")
            
            return final_use_cases_with_subdomains
            
        except Exception as e:
            self.logger.error(f"Failed to assign user domains via LLM: {get_clean_error_message(e)}. Using fallback distribution with subdomain discovery.")
            
            # Fallback: distribute use cases evenly across user domains
            for idx, uc in enumerate(use_cases):
                domain_idx = idx % len(user_domains)
                uc['Business Domain'] = user_domains[domain_idx]
            
            # Even on fallback, try to discover subdomains
            self.logger.info(f"📍 Fallback: Attempting subdomain discovery for distributed domains...")
            
            domain_usecases_map = defaultdict(list)
            for uc in use_cases:
                domain = uc.get('Business Domain', '').strip()
                if domain:
                    domain_usecases_map[domain].append(uc)
            
            final_use_cases = []
            fallback_workers = min(4, len(domain_usecases_map))
            
            def _fallback_subdomain_task(domain_name, domain_use_cases):
                try:
                    return self._detect_subdomains_for_domain(domain_name, domain_use_cases, language)
                except Exception as sub_e:
                    self.logger.error(f"Subdomain detection failed for domain '{domain_name}': {sub_e}")
                    for uc in domain_use_cases:
                        uc['Subdomain'] = f"General {domain_name}"
                    return domain_use_cases
            
            _fb_total_timeout = self.llm_timeout_seconds * 3 * len(domain_usecases_map) + 120
            with _SafeExecutorContext(max_workers=fallback_workers, thread_name_prefix="fb_subdomain", logger=self.logger, name="FallbackSubdomain") as ctx:
                fb_futures = {ctx.submit(_fallback_subdomain_task, dn, ducs): dn 
                              for dn, ducs in domain_usecases_map.items()}
                try:
                    for future in safe_as_completed(fb_futures, timeout=_fb_total_timeout):
                        try:
                            result = future.result(timeout=self.llm_timeout_seconds * 3)
                            if result:
                                final_use_cases.extend(result)
                        except Exception as exc:
                            self.logger.error(f"Fallback subdomain task failed: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    ctx.mark_timed_out()
                    self.logger.error(f"Fallback subdomain detection timed out globally ({_fb_total_timeout}s)")
            
            return final_use_cases if final_use_cases else use_cases
    
    # === HELPER FUNCTIONS ===
    def _apply_domain_mapping_flat(self, all_use_cases: list, domain_mapping: dict) -> list:
        """Applies a domain mapping to a flat list[dict] of use cases."""
        modified_use_cases = []
        for uc in all_use_cases:
            original_domain = uc.get('Business Domain', 'Other')
            new_domain = domain_mapping.get(original_domain, original_domain)
            uc['Business Domain'] = new_domain
            modified_use_cases.append(uc)
        return modified_use_cases
    
    def _group_use_cases_by_domain_flat(self, all_use_cases: list) -> dict:
        """
        Groups a flat list[dict] of use cases by their 'Business Domain'.
        Use cases within each domain are sorted by priority descending (Very High first).
        
        NOTE: This sorting is used for PDF/PPT/XLS outputs. Notebooks use ID-based sorting instead.
        """
        # Sort by priority first (for PDF/PPT/XLS outputs)
        all_use_cases = sorted(all_use_cases, key=self._priority_sort_key)
        grouped_by_domain = defaultdict(list)
        for uc in all_use_cases:
            domain = uc.get('Business Domain') or 'Other'
            grouped_by_domain[domain].append(uc)
        return grouped_by_domain

    def _align_translated_data(self, master_grouped_data: dict, translated_flat_list: list) -> dict:
        """
        Creates a grouped dictionary for the target language that strictly follows 
        the ordering and categorization of the master (English) data.
        
        Merges translated fields ON TOP of the English dict so that non-translated
        fields (SQL, usecase, Strategic Goals Alignment, scoring fields, etc.) are
        preserved from the original.
        """
        translated_map = {uc['No']: uc for uc in translated_flat_list}
        aligned_data = {}
        for en_domain in master_grouped_data.keys(): 
            english_cases = master_grouped_data[en_domain]
            translated_domain_name = en_domain
            if english_cases:
                first_id = english_cases[0]['No']
                if first_id in translated_map:
                    translated_domain_name = translated_map[first_id].get('Business Domain', en_domain)
            
            translated_group = []
            for en_uc in english_cases:
                uc_id = en_uc['No']
                if uc_id in translated_map:
                    merged = en_uc.copy()
                    merged.update(translated_map[uc_id])
                    translated_group.append(merged)
                else:
                    translated_group.append(en_uc)
            aligned_data[translated_domain_name] = translated_group
        return aligned_data
    # === END HELPER FUNCTIONS ===


    def _build_dedup_markdown(self, use_cases: list) -> str:
        """
        Build a markdown table for deduplication including ID, Name, Domain,
        Analytics Technique, Tables Involved (table names only), and a
        1-line Statement. The extra fields are required to activate Layer 4
        (technique swap) and Layer 5 (table + intent overlap) duplicate
        detection. Scores and activation suffixes are intentionally excluded
        to keep the LLM focused on business intent + data grounding.
        Shared by global and per-domain dedup (DRY).
        """
        pipe_esc = r'\|'
        header = (
            "| ID | Name | Domain | Technique | Tables | Statement |\n"
            "|---|---|---|---|---|---|\n"
        )
        def _sanitize(value: str) -> str:
            return str(value or '').replace('|', pipe_esc).replace('\n', ' ').strip()
        def _compact_tables(raw: str) -> str:
            parts = []
            for token in _sanitize(raw).split(','):
                token = token.strip().strip('`')
                if not token:
                    continue
                parts.append(token.split('.')[-1])
            return ', '.join(parts) if parts else '—'
        md_parts = [header]
        for uc in use_cases:
            name = _sanitize(uc.get('Name', ''))
            domain = _sanitize(uc.get('Business Domain', ''))
            technique = _sanitize(uc.get('Analytics Technique', '')) or '—'
            tables = _compact_tables(uc.get('Tables Involved', ''))
            statement = _sanitize(uc.get('Statement', ''))
            if len(statement) > 200:
                statement = statement[:197] + '...'
            if not statement:
                statement = '—'
            md_parts.append(
                f"| {uc['No']} | {name} | {domain} | {technique} | {tables} | {statement} |\n"
            )
        return "".join(md_parts)

    def _parse_dedup_response(self, response_raw: str, all_use_cases: list, context_label: str) -> list:
        """
        Parse LLM dedup response CSV and return filtered use cases.
        Shared by global and per-domain dedup (DRY).
        Returns the filtered list, or raises on failure.
        """
        response_clean = response_raw.strip()
        response_clean = re.sub(r'^```(?:csv|json)?\s*', '', response_clean, flags=re.IGNORECASE | re.MULTILINE)
        response_clean = re.sub(r'\s*```\s*$', '', response_clean, flags=re.IGNORECASE | re.MULTILINE)
        response_clean = response_clean.strip()
        
        csv_rows = CSVParser.parse_csv_string(
            response_clean,
            logger=self.logger,
            context=context_label
        )
        ids_to_keep = set()
        id_to_flags = {}
        _KNOWN_FLAG_TOKENS = ("_FLAG_DUPLICATE_SUSPECT", "_TRIVIAL_WRAPPER", "_TECH_ONLY")
        
        for row in csv_rows:
            uc_id = row.get('use_case_id', '').strip()
            if not uc_id:
                continue
            ids_to_keep.add(uc_id)
            _justification = (row.get('honesty_justification') or '').strip()
            if _justification:
                _hits = [tok for tok in _KNOWN_FLAG_TOKENS if tok in _justification]
                if _hits:
                    id_to_flags[uc_id] = {"tokens": _hits, "note": _justification}
        
        if not ids_to_keep:
            raise ValueError(f"[{context_label}] CSV contains no use case IDs")
        
        removed_count = len(all_use_cases) - len(ids_to_keep)
        removal_pct = (removed_count / len(all_use_cases)) * 100 if all_use_cases else 0
        
        self.logger.info(f"[{context_label}] Retained {len(ids_to_keep)} use cases, removed {removed_count} ({removal_pct:.1f}%)")
        
        if id_to_flags:
            _suspect_ids = sorted(uid for uid, f in id_to_flags.items() if "_FLAG_DUPLICATE_SUSPECT" in f["tokens"])
            _trivial_ids = sorted(uid for uid, f in id_to_flags.items() if "_TRIVIAL_WRAPPER" in f["tokens"])
            _tech_ids = sorted(uid for uid, f in id_to_flags.items() if "_TECH_ONLY" in f["tokens"])
            if _suspect_ids:
                self.logger.warning(
                    f"[{context_label}] {len(_suspect_ids)} kept use case(s) flagged as SUSPECTED DUPLICATES "
                    f"(kept for human review): {', '.join(_suspect_ids[:20])}{'...' if len(_suspect_ids) > 20 else ''}"
                )
            if _trivial_ids:
                self.logger.info(f"[{context_label}] {len(_trivial_ids)} kept use case(s) flagged as trivial-wrapper framing")
            if _tech_ids:
                self.logger.info(f"[{context_label}] {len(_tech_ids)} kept use case(s) flagged as technique-only framing")
        
        if removed_count > 0:
            from collections import defaultdict
            domain_removal_counts = defaultdict(int)
            for uc in all_use_cases:
                if uc['No'] not in ids_to_keep:
                    domain_removal_counts[uc.get('Business Domain', 'Unknown')] += 1
            
            for domain in sorted(domain_removal_counts.keys()):
                self.logger.info(f"  - {domain}: {domain_removal_counts[domain]} use case(s) removed")
        
        kept_use_cases = []
        for uc in all_use_cases:
            if uc['No'] not in ids_to_keep:
                continue
            _flags = id_to_flags.get(uc['No'])
            if _flags:
                _existing = uc.get('Dedup Flags')
                _merged_tokens = list(_flags["tokens"])
                _note = _flags["note"]
                if _existing and isinstance(_existing, dict):
                    for _tok in _existing.get("tokens", []):
                        if _tok not in _merged_tokens:
                            _merged_tokens.append(_tok)
                    _prev_note = _existing.get("note", "")
                    if _prev_note and _prev_note != _note:
                        _note = f"{_prev_note} | {_note}"
                uc['Dedup Flags'] = {"tokens": _merged_tokens, "note": _note}
            kept_use_cases.append(uc)
        return kept_use_cases

    def _deduplicate_use_cases(self, all_use_cases: list) -> list:
        """
        Global deduplication: attempts to deduplicate ALL use cases in one LLM call
        using the thinker model (highest-capability model) for best cross-domain
        duplicate detection.
        
        Sends full Name, Statement, and Tables for each use case so the LLM can
        assess table-based viability when choosing which duplicate to keep.
        
        Falls back to per-domain parallel deduplication on:
        - Context window overflow (too many use cases for one prompt)
        - Any LLM or parsing failure
        """
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Global Semantic Dedup",
                sub_step_name="Started", status="started",
                message=f"Running one-shot dedup for {len(all_use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info(f"🌐 Attempting GLOBAL one-shot deduplication for {len(all_use_cases)} use cases (thinker model)...")
        
        if len(all_use_cases) < 2:
            self.logger.debug("Skipping deduplication, not enough use cases to compare.")
            return all_use_cases
        
        try:
            use_case_markdown = self._build_dedup_markdown(all_use_cases)
            self.logger.debug(f"Built global dedup markdown table: {len(use_case_markdown):,} chars, {len(all_use_cases)} use cases")
        except Exception as e:
            self.logger.error(f"Failed to create markdown for global dedup: {get_clean_error_message(e)}")
            self.logger.info("⬇️  Falling back to per-domain parallel deduplication...")
            return self._deduplicate_use_cases_by_domain_parallel(all_use_cases)
        
        try:
            thinker_cascade = get_models_by_type("thinker")
            if not thinker_cascade:
                self.logger.warning("No thinker models configured. Falling back to per-domain dedup...")
                return self._deduplicate_use_cases_by_domain_parallel(all_use_cases)
            
            thinker_model = thinker_cascade[0]
            thinker_endpoint = thinker_model["llm_endpoint_name"]
            thinker_context_limit = get_context_limit_chars_for_model(
                thinker_model, getattr(self, 'current_language', 'English')
            )
            
            prompt_template = self.ai_agent.prompt_templates.get("REVIEW_USE_CASES_PROMPT", "")
            estimated_prompt_size = len(prompt_template) + len(use_case_markdown) + 1000
            
            if estimated_prompt_size > thinker_context_limit:
                self.logger.warning(
                    f"Global dedup prompt ({estimated_prompt_size:,} chars) exceeds thinker model limit "
                    f"({thinker_context_limit:,} chars for {thinker_model.get('name', thinker_endpoint)}). "
                    f"Falling back to per-domain parallel deduplication..."
                )
                return self._deduplicate_use_cases_by_domain_parallel(all_use_cases)
            
            prompt_vars = {
                "use_case_markdown": use_case_markdown,
                "total_count": len(all_use_cases),
                "generation_instructions_section": self._get_generation_instruction_for_prompt("REVIEW_USE_CASES_PROMPT"),
            }
            
            self.logger.info(
                f"⏳ Sending {len(all_use_cases)} use cases to thinker model "
                f"({thinker_model.get('name', thinker_endpoint)}, {estimated_prompt_size:,} chars)..."
            )
            
            response_raw = self.ai_agent.run_worker(
                step_name="Deduplicate_Global",
                worker_prompt_path="REVIEW_USE_CASES_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                model_override=thinker_endpoint
            )
            
            self.logger.info(f"✅ Received global dedup response, parsing results...")
            
            unique_use_cases = self._parse_dedup_response(response_raw, all_use_cases, "Global Deduplication")
            return unique_use_cases
            
        except Exception as e:
            self.logger.error(
                f"🌐 Global one-shot deduplication failed: {get_clean_error_message(e)}. "
                f"Falling back to per-domain parallel deduplication..."
            )
            return self._deduplicate_use_cases_by_domain_parallel(all_use_cases)

    def _deduplicate_use_cases_by_domain_parallel(self, all_use_cases: list) -> list:
        """
        Deduplicate use cases at domain level in parallel.
        Uses the same REVIEW_USE_CASES_PROMPT and shared markdown builder as global dedup (DRY).
        
        Args:
            all_use_cases: List of all use case dictionaries (should be scored first)
            
        Returns:
            List of deduplicated use cases
        """
        from collections import defaultdict
        import concurrent.futures
        import time
        
        self.logger.info(f"🔄 Starting domain-level parallel deduplication for {len(all_use_cases)} use cases...")
        
        domain_use_cases = defaultdict(list)
        for uc in all_use_cases:
            domain = uc.get('Business Domain', 'Unknown')
            domain_use_cases[domain].append(uc)
        
        self.logger.info(f"📊 Grouped use cases into {len(domain_use_cases)} domains")
        
        deduplicated_results = []
        
        def dedupe_domain(domain_name, domain_ucs):
            """Deduplicate a single domain's use cases using the shared prompt."""
            if len(domain_ucs) < 2:
                self.logger.debug(f"[{domain_name}] Only {len(domain_ucs)} use case(s), skipping dedup.")
                return domain_ucs
            try:
                self.logger.info(f"[{domain_name}] Deduplicating {len(domain_ucs)} use cases...")
                
                use_case_markdown = self._build_dedup_markdown(domain_ucs)
                
                review_context_limit = get_max_context_chars("English", "REVIEW_USE_CASES_PROMPT")
                prompt_template = self.ai_agent.prompt_templates.get("REVIEW_USE_CASES_PROMPT", "")
                estimated_size = len(prompt_template) + len(use_case_markdown) + 1000
                
                if estimated_size > review_context_limit:
                    self.logger.warning(f"[{domain_name}] Domain too large ({estimated_size:,} chars). Keeping all {len(domain_ucs)} use cases.")
                    return domain_ucs
                
                prompt_vars = {
                    "use_case_markdown": use_case_markdown,
                    "total_count": len(domain_ucs),
                    "generation_instructions_section": self._get_generation_instruction_for_prompt("REVIEW_USE_CASES_PROMPT"),
                }
                
                self.logger.info(f"⏳ [{domain_name}] Waiting for LLM response...")
                
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Deduplicate_Domain_{domain_name}",
                    worker_prompt_path="REVIEW_USE_CASES_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None
                )
                
                deduplicated = self._parse_dedup_response(response_raw, domain_ucs, f"Domain:{domain_name}")
                return deduplicated
                
            except Exception as e:
                self.logger.error(f"[{domain_name}] Deduplication failed: {get_clean_error_message(e)}. Keeping all {len(domain_ucs)} use cases.")
                return domain_ucs
        
        # ADAPTIVE PARALLELISM: Calculate based on domains and total use cases
        total_use_cases = sum(len(ucs) for ucs in domain_use_cases.values())
        num_domains = len(domain_use_cases)
        
        dedup_parallelism, reason = calculate_adaptive_parallelism(
            "deduplication", self.max_parallelism,
            num_items=total_use_cases,
            num_domains=num_domains,
            is_llm_operation=True, logger=self.logger
        )
        
        # Calculate timeout: 5 minutes per domain + 5 minutes buffer
        overall_timeout = (num_domains * 300 // dedup_parallelism) + 300
        
        log_print(f"\n{'='*80}")
        log_print(f"🔄 DEDUPLICATION: Processing {num_domains} domains ({total_use_cases} total use cases)")
        log_print(f"{'='*80}")
        log_adaptive_parallelism_decision("deduplication", dedup_parallelism, self.max_parallelism, reason)
        log_print(f"Overall timeout: {overall_timeout}s ({overall_timeout//60} min)")
        log_print(f"{'='*80}\n")
        
        with _SafeExecutorContext(max_workers=dedup_parallelism, thread_name_prefix="DomainDedupe", logger=self.logger, name="DomainDedupe") as ctx:
            future_to_domain = {}
            for domain, domain_ucs in domain_use_cases.items():
                future = ctx.submit(dedupe_domain, domain, domain_ucs)
                future_to_domain[future] = domain
            
            completed_count = 0
            completed_domains = set()
            start_time = time.time()
            
            try:
                for future in safe_as_completed(future_to_domain, timeout=overall_timeout):
                    domain = future_to_domain[future]
                    elapsed = time.time() - start_time
                    try:
                        domain_deduplicated = future.result(timeout=30)
                        deduplicated_results.extend(domain_deduplicated)
                        completed_count += 1
                        completed_domains.add(domain)
                        log_print(f"[Deduplication] ✓ Domain {completed_count}/{len(domain_use_cases)} complete: {domain} ({elapsed:.1f}s elapsed)")
                    except concurrent.futures.TimeoutError:
                        self.logger.error(f"[{domain}] Result collection timed out - keeping original use cases")
                        deduplicated_results.extend(domain_use_cases.get(domain, []))
                        completed_domains.add(domain)
                    except Exception as e:
                        self.logger.error(f"[{domain}] Failed to collect results: {get_clean_error_message(e)} - keeping original use cases")
                        deduplicated_results.extend(domain_use_cases.get(domain, []))
                        completed_domains.add(domain)
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⚠️  Overall deduplication timeout reached ({overall_timeout}s). {completed_count}/{len(domain_use_cases)} domains completed.")
                log_print(f"[Deduplication] ⚠️  TIMEOUT - keeping original use cases for incomplete domains", level="WARNING")
                for domain, domain_ucs in domain_use_cases.items():
                    if domain not in completed_domains:
                        self.logger.warning(f"[{domain}] Timed out - keeping all {len(domain_ucs)} original use cases")
                        deduplicated_results.extend(domain_ucs)
        
        del domain_use_cases
        del future_to_domain
        _force_gc(self.logger, "after deduplication")
        
        total_removed = len(all_use_cases) - len(deduplicated_results)
        removal_pct = (total_removed / len(all_use_cases)) * 100 if all_use_cases else 0
        
        self.logger.info(f"✅ Domain-level deduplication complete: Retained {len(deduplicated_results)} use cases, removed {total_removed} ({removal_pct:.1f}%)")
        
        return deduplicated_results

    def _score_use_cases_global(self, all_use_cases: list, business_context: str = "",
                                strategic_goals: list = None, business_priorities: list = None,
                                strategic_initiative: str = "", value_chain: str = "",
                                revenue_model: str = "") -> list:
        """
        Try to score ALL use cases together in one prompt using minimal context (ID, Name, Statement, Business Value).
        Returns None on failure so callers can fall back to domain-based scoring.
        """
        if not all_use_cases:
            return all_use_cases

        try:
            md_parts = ["| No | Name | Statement | Business Value |\n|---|---|---|---|\n"]
            pipe_escape = r'\|'
            for uc in all_use_cases:
                no_val = str(uc.get('No', '')).replace('|', pipe_escape)
                name_val = str(uc.get('Name', '')).replace('|', pipe_escape)
                stmt_val = str(uc.get('Statement', '')).replace('|', pipe_escape)
                bv_val = str(uc.get('Business Value', '')).replace('|', pipe_escape)
                md_parts.append(f"| {no_val} | {name_val} | {stmt_val} | {bv_val} |\n")
            use_case_markdown = "".join(md_parts)

            prompt_vars = {
                "use_case_markdown": use_case_markdown,
                "business_context": business_context or self._get_business_context_fallback(),
                "strategic_goals": "\n".join([f"- {goal}" for goal in (strategic_goals or [])]) or "- Improve operational efficiency",
                "business_priorities": "\n".join([f"- {priority}" for priority in (business_priorities or [])]) or "- Optimize costs",
                "strategic_initiative": strategic_initiative or "Data-driven transformation program",
                "value_chain": value_chain or "Standard operations",
                "revenue_model": revenue_model or "Products and services",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_VALUE_SCORE_PROMPT"),
            }

            self.logger.info(f"✅ Attempting GLOBAL scoring for {len(all_use_cases)} use cases (minimal fields to fit context)")
            response_raw = self.ai_agent.run_worker(
                step_name="Score_All_Use_Cases_Global",
                worker_prompt_path="USE_CASE_VALUE_SCORE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None
            )

            response_clean = clean_json_response(response_raw)
            scoring_data = CSVParser.parse_csv_string(
                response_clean,
                logger=self.logger,
                context="GLOBAL scoring"
            )

            scoring_by_no = {s.get('No'): s for s in scoring_data}
            scored_use_cases = []
            for uc in all_use_cases:
                no = uc.get('No')
                scores = scoring_by_no.get(no, {})
                uc_copy = uc.copy()
                for key, value in scores.items():
                    if key != 'No':
                        uc_copy[key] = value

                try:
                    strategic_alignment = float(uc_copy.get('Strategic Alignment', 3.5))
                    roi = float(uc_copy.get('Return on Investment', 3.5))
                    reusability = float(uc_copy.get('Reusability', 3.5))
                    time_to_value = float(uc_copy.get('Time to Value', 3.5))

                    data_availability = float(uc_copy.get('Data Availability', 3.5))
                    data_accessibility = float(uc_copy.get('Data Accessibility', 3.5))
                    architecture_fitness = float(uc_copy.get('Architecture Fitness', 3.5))
                    team_skills = float(uc_copy.get('Team Skills', 3.5))
                    domain_knowledge = float(uc_copy.get('Domain Knowledge', 3.5))
                    people_allocation = float(uc_copy.get('People Allocation', 3.5))
                    budget_allocation = float(uc_copy.get('Budget Allocation', 3.5))
                    time_to_production = float(uc_copy.get('Time to Production', 3.5))

                    value_score = (
                        (roi * 0.60)
                        + (strategic_alignment * 0.25)
                        + (time_to_value * 0.075)
                        + (reusability * 0.075)
                    )

                    feasibility_inputs = [
                        data_availability,
                        data_accessibility,
                        architecture_fitness,
                        team_skills,
                        domain_knowledge,
                        people_allocation,
                        budget_allocation,
                        time_to_production,
                    ]
                    feasibility_score = sum(feasibility_inputs) / len(feasibility_inputs)

                    priority_score = (value_score * 1.5) + (feasibility_score * 0.5)

                    uc_copy['Value'] = round(value_score, 2)
                    uc_copy['Feasibility'] = round(feasibility_score, 2)
                    uc_copy['Priority'] = round(priority_score, 2)
                    
                    # Ensure AI_Confidence and AI_Feedback are set with defaults if missing
                    if 'AI_Confidence' not in uc_copy or not uc_copy.get('AI_Confidence'):
                        uc_copy['AI_Confidence'] = 0.5
                    if 'AI_Feedback' not in uc_copy or not uc_copy.get('AI_Feedback'):
                        uc_copy['AI_Feedback'] = 'No feedback provided by AI scoring.'

                except Exception:
                    pass
                    uc_copy['AI_Confidence'] = uc_copy.get('AI_Confidence', 0.5)
                    uc_copy['AI_Feedback'] = uc_copy.get('AI_Feedback', 'Scoring exception occurred.')
                scored_use_cases.append(uc_copy)

            self.logger.info(f"✅ GLOBAL scoring succeeded for {len(scored_use_cases)} use cases")
            return scored_use_cases
        except Exception as e:
            self.logger.error(f"Global scoring failed, will fall back to domain-based scoring: {get_clean_error_message(e)}")
            return None

    def _score_per_domain_parallel(self, all_use_cases: list, business_context: str = "",
                                   strategic_goals: list = None, business_priorities: list = None,
                                   strategic_initiative: str = "", value_chain: str = "",
                                   revenue_model: str = "") -> list:
        """
        Score use cases per domain in parallel (Phase 1).
        
        Each domain is scored in its own thread, all domains run in parallel.
        After this, ALL scored use cases are returned for documentation and Genie Code generation.
        
        STABILITY FIX: Uses adaptive parallelism to prevent LLM API rate limiting 
        and adds heartbeat + total timeout to prevent hangs.
        """
        import time
        from collections import defaultdict
        from concurrent.futures import TimeoutError as FuturesTimeoutError
        
        # Group use cases by domain first to calculate adaptive parallelism
        domain_groups = defaultdict(list)
        for uc in all_use_cases:
            domain = uc.get('Business Domain', 'Other')
            domain_groups[domain].append(uc)
        
        num_use_cases = len(all_use_cases)
        num_domains = len(domain_groups)
        
        # ADAPTIVE PARALLELISM: Calculate based on use cases and domains
        scoring_parallelism, reason = calculate_adaptive_parallelism(
            "scoring", self.max_parallelism,
            num_items=num_use_cases,
            num_domains=num_domains,
            is_llm_operation=True, logger=self.logger
        )
        
        # Dynamic timeouts based on workload
        # More use cases = more time needed
        base_timeout_per_uc = 30  # seconds per use case
        TOTAL_SCORING_TIMEOUT = max(1800, min(3600, num_use_cases * base_timeout_per_uc))  # 30-60 min
        HEARTBEAT_INTERVAL = 60  # Log progress every 60 seconds
        PER_DOMAIN_TIMEOUT = max(600, min(1200, (num_use_cases // num_domains) * 60))  # 10-20 min per domain
        
        self.logger.info(f"📊 PHASE 1: Scoring {num_use_cases} use cases per domain in parallel")
        self.logger.info(f"📊 Grouped into {num_domains} domains")
        
        log_print(f"\n{'='*80}")
        log_print(f"📊 PHASE 1: SCORING PER DOMAIN (PARALLEL)")
        log_print(f"{'='*80}")
        log_print(f"Total use cases: {num_use_cases}")
        log_print(f"Total domains: {num_domains}")
        log_adaptive_parallelism_decision("scoring", scoring_parallelism, self.max_parallelism, reason)
        log_print(f"Total timeout: {TOTAL_SCORING_TIMEOUT}s ({TOTAL_SCORING_TIMEOUT//60} min)")
        log_print(f"{'='*80}\n")
        
        for domain, use_cases in sorted(domain_groups.items()):
            self.logger.info(f"   - {domain}: {len(use_cases)} use cases")
        
        def score_domain(domain_name, domain_use_cases):
            """Score one domain's use cases"""
            try:
                self.logger.info(f"📊 [{domain_name}] Scoring {len(domain_use_cases)} use cases...")
                
                scored = self._score_use_cases(
                    domain_use_cases,
                    business_context=business_context,
                    strategic_goals=strategic_goals,
                    business_priorities=business_priorities,
                    strategic_initiative=strategic_initiative,
                    value_chain=value_chain,
                    revenue_model=revenue_model
                )
                
                self.logger.info(f"✅ [{domain_name}] Scoring complete")
                return (domain_name, scored)
                
            except Exception as e:
                self.logger.error(f"❌ [{domain_name}] Scoring failed: {get_clean_error_message(e)}")
                # Return with default scores - fix bug where 'Pending' priority was kept
                for uc in domain_use_cases:
                    if uc.get('Quality') in (None, '', 'Pending'):
                        uc['Quality'] = 'Medium'
                    uc['Priority'] = uc.get('Priority', 5.0)
                    uc['Value'] = uc.get('Value', 3.5)
                    uc['Feasibility'] = uc.get('Feasibility', 3.5)
                return (domain_name, domain_use_cases)
        
        # Score all domains in parallel with reduced parallelism
        all_scored_use_cases = []
        
        with _SafeExecutorContext(max_workers=scoring_parallelism, thread_name_prefix="DomainScoring", logger=self.logger, name="DomainScoring") as ctx:
            future_to_domain = {}
            for domain, domain_use_cases in domain_groups.items():
                future = ctx.submit(score_domain, domain, domain_use_cases)
                future_to_domain[future] = domain
            
            completed_domains = 0
            failed_domains = 0
            total_domains = len(domain_groups)
            start_time = time.time()
            last_heartbeat = start_time
            
            try:
                for future in safe_as_completed(future_to_domain, timeout=TOTAL_SCORING_TIMEOUT):
                    domain = future_to_domain[future]
                    current_time = time.time()
                    
                    if current_time - last_heartbeat >= HEARTBEAT_INTERVAL:
                        elapsed = current_time - start_time
                        pending = total_domains - completed_domains - failed_domains
                        log_print(f"⏳ Scoring progress: {completed_domains}/{total_domains} done, {pending} pending ({elapsed:.0f}s elapsed)")
                        self.logger.info(f"⏳ Scoring heartbeat: {completed_domains}/{total_domains} domains complete, {pending} pending")
                        last_heartbeat = current_time
                    
                    try:
                        domain_name, scored_use_cases = future.result(timeout=PER_DOMAIN_TIMEOUT)
                        all_scored_use_cases.extend(scored_use_cases)
                        completed_domains += 1
                        
                        log_print(f"✓ Scored domain {completed_domains}/{total_domains}: {domain_name} ({len(scored_use_cases)} use cases)")
                        self.logger.info(f"✓ Domain {completed_domains}/{total_domains} scoring complete: {domain_name}")
                        
                    except FuturesTimeoutError:
                        failed_domains += 1
                        self.logger.error(f"❌ [{domain}] Scoring timed out after {PER_DOMAIN_TIMEOUT}s")
                        log_print(f"✗ Timeout domain {completed_domains + failed_domains}/{total_domains}: {domain}", level="ERROR")
                    except Exception as e:
                        failed_domains += 1
                        self.logger.error(f"❌ [{domain}] Failed to collect scoring results: {get_clean_error_message(e)}")
                        log_print(f"✗ Failed domain {completed_domains + failed_domains}/{total_domains}: {domain}", level="ERROR")
                        
            except FuturesTimeoutError:
                ctx.mark_timed_out()
                elapsed = time.time() - start_time
                pending = total_domains - completed_domains - failed_domains
                self.logger.error(f"⚠️ TOTAL SCORING TIMEOUT reached after {elapsed:.0f}s. {completed_domains}/{total_domains} domains completed, {pending} still pending.")
                log_print(f"⚠️ SCORING TIMEOUT: {completed_domains}/{total_domains} domains completed after {elapsed:.0f}s", level="WARNING")
                log_print(f"   Proceeding with {len(all_scored_use_cases)} scored use cases", level="WARNING")
        
        log_print(f"\n{'='*80}")
        log_print(f"✅ PHASE 1 COMPLETE: ALL DOMAINS SCORED")
        log_print(f"{'='*80}")
        log_print(f"Total scored use cases: {len(all_scored_use_cases)}")
        log_print(f"{'='*80}\n")
        
        self.logger.info(f"✅ Phase 1 complete: {len(all_scored_use_cases)} use cases scored across all domains")
        
        del domain_groups
        del future_to_domain
        _force_gc(self.logger, "after scoring phase")
        
        normalized_use_cases = self._normalize_priority_scores(all_scored_use_cases)
        self.logger.info(f"✅ Phase 1 normalized across {len(normalized_use_cases)} use cases")
        
        return normalized_use_cases

    def _normalize_priority_scores(self, use_cases: list) -> list:
        if not use_cases:
            return use_cases
        
        scores = []
        for uc in use_cases:
            try:
                scores.append(float(uc.get('Priority', 0)))
            except (TypeError, ValueError):
                continue
        
        max_score = max(scores) if scores else 0.0
        if max_score <= 0:
            return use_cases
        
        target_max = 9.5  # B3: deterministic cap; previously random.uniform(9.5, 9.95)
        scale = target_max / max_score
        
        for uc in use_cases:
            try:
                priority_score = float(uc.get('Priority', 0))
            except (TypeError, ValueError):
                priority_score = 0.0
            try:
                value_score = float(uc.get('Value', 0))
            except (TypeError, ValueError):
                value_score = 0.0
            try:
                feasibility_score = float(uc.get('Feasibility', 0))
            except (TypeError, ValueError):
                feasibility_score = 0.0
            
            priority_scaled = min(priority_score * scale, target_max)
            value_scaled = min(value_score * scale, target_max)
            feasibility_scaled = min(feasibility_score * scale, target_max)
            
            uc['Priority'] = round(priority_scaled, 2)
            uc['Value'] = round(value_scaled, 2)
            uc['Feasibility'] = round(feasibility_scaled, 2)
            
        self.logger.info(f"Applied priority normalization scale factor {scale:.2f} with target max {target_max:.2f}")
        return use_cases
    
    def _score_use_cases_batched(self, use_cases_to_score, prompt_vars, domain_name, context_limit, start_from_model=None):
        """
        Score use cases in batches that fit within the given context limit.
        Handles CascadeRebatchError by preserving partial progress and re-batching
        remaining use cases for the smaller model.
        
        Returns (all_scoring_rows, last_model_used) where last_model_used can be
        passed as start_from_model for subsequent retry calls.
        """
        MAX_CASCADE_RETRIES = 3
        remaining = list(use_cases_to_score)
        all_rows = []
        effective_limit = context_limit
        effective_model = start_from_model

        for cascade_attempt in range(MAX_CASCADE_RETRIES):
            if not remaining:
                break

            prompt_template_text = self.ai_agent.prompt_templates.get("USE_CASE_VALUE_SCORE_PROMPT", "")
            prompt_overhead = len(prompt_template_text) + 2000
            available_chars = effective_limit - prompt_overhead

            total_md_len = len("| No | Name | Business Value |\n|---|---|---|\n")
            for uc in remaining:
                total_md_len += len(f"| {uc.get('No','')} | {uc.get('Name','')} | {uc.get('Business Value','')} |\n")
            chars_per_uc = max(100, total_md_len // max(1, len(remaining)))
            batch_size = max(10, int(available_chars / chars_per_uc * 0.7))

            total_batches = (len(remaining) + batch_size - 1) // batch_size
            self.logger.info(
                f"📦 [{domain_name}] Scoring {len(remaining)} use cases in ~{total_batches} batches of ~{batch_size} "
                f"(limit: {effective_limit:,} chars, model: {effective_model or 'default cascade'}, "
                f"already scored: {len(all_rows)})"
            )

            cascade_triggered = False
            cascade_error = None

            for sb_idx in range(0, len(remaining), batch_size):
                sb_use_cases = remaining[sb_idx:sb_idx + batch_size]
                sb_num = (sb_idx // batch_size) + 1

                sb_md_parts = ["| No | Name | Business Value |\n|---|---|---|\n"]
                for uc in sb_use_cases:
                    no = str(uc.get('No', '')).replace('|', r'\|')
                    name = str(uc.get('Name', '')).replace('|', r'\|')
                    bv = str(uc.get('Business Value', '')).replace('|', r'\|')
                    sb_md_parts.append(f"| {no} | {name} | {bv} |\n")

                sb_prompt_vars = dict(prompt_vars)
                sb_prompt_vars["use_case_markdown"] = "".join(sb_md_parts)

                try:
                    sb_response = self.ai_agent.run_worker(
                        step_name=f"Score_Use_Cases_{domain_name}_Batch{sb_num}of{total_batches}",
                        worker_prompt_path="USE_CASE_VALUE_SCORE_PROMPT",
                        prompt_vars=sb_prompt_vars,
                        response_schema=None,
                        start_from_model=effective_model
                    )
                    if sb_response and sb_response.strip():
                        sb_clean = clean_json_response(sb_response)
                        sb_rows = CSVParser.parse_csv_string(
                            sb_clean,
                            logger=self.logger,
                            context=f"Scoring batch {sb_num}/{total_batches} for {domain_name}"
                        )
                        all_rows.extend(sb_rows)
                        self.logger.info(
                            f"✅ [{domain_name}] Scoring batch {sb_num}/{total_batches} "
                            f"({len(sb_rows)} scores, {len(all_rows)} total)"
                        )
                except CascadeRebatchError as cre:
                    self.logger.warning(
                        f"🔄 [{domain_name}] Scoring batch {sb_num} triggered cascade rebatch → "
                        f"'{cre.target_model_name}' ({cre.target_context_limit_chars:,} chars). "
                        f"Preserving {len(all_rows)} scored, rebatching remaining."
                    )
                    cascade_triggered = True
                    cascade_error = cre
                    break
                except Exception as sb_err:
                    self.logger.error(f"❌ [{domain_name}] Scoring batch {sb_num}/{total_batches} failed: {sb_err}")

            if cascade_triggered and cascade_error:
                scored_nos = {row.get('No', '') for row in all_rows}
                remaining = [uc for uc in remaining if str(uc.get('No', '')) not in scored_nos]
                effective_limit = cascade_error.target_context_limit_chars
                effective_model = cascade_error.target_model_endpoint
                if not remaining:
                    self.logger.info(f"📦 [{domain_name}] All use cases scored despite cascade trigger")
                    break
                continue

            break

        return all_rows, effective_model

    def _score_use_cases(self, all_use_cases: list, business_context: str = "", strategic_goals: list = None,
                         business_priorities: list = None, strategic_initiative: str = "",
                         value_chain: str = "", revenue_model: str = "") -> list:
        """
        Calls an LLM to score use cases across 13 different factors.
        
        🚨 NEW SCORING STRATEGY (PRIORITIZED):
        1. PRIMARY APPROACH: Score ALL use cases in ONE prompt (preferred for consistency)
        2. FALLBACK APPROACH: If all use cases don't fit in one prompt, score by domain in parallel
        
        🚨 WEIGHTING: Priority Score = (Value × 1.5) + (Feasibility × 0.5)
        - Value = (ROI × 0.60) + (Strategic Alignment × 0.25) + (Time to Value × 0.075) + (Reusability × 0.075)
        
        🚨 NEW: If use cases exceed 2048, implements 2-pass scoring:
        - Pass 1: Score all use cases and select top 2048 by priority
        - Pass 2: Re-score the top 2048 for final ranking
        
        Args:
            all_use_cases: List of use case dictionaries to score
            business_context: Business context description
            strategic_goals: List of strategic goals
            business_priorities: List of business priorities
            strategic_initiative: Description of strategic initiative
            value_chain: Description of value chain
            revenue_model: Description of revenue model
            
        Returns:
            List of scored use cases (top 2048 if input >2048, otherwise all)
        """
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Scoring Phase 1",
                sub_step_name="Started", status="started",
                message=f"Scoring {len(all_use_cases)} use cases"
            )
        except Exception:
            pass
        self.logger.info(f"Starting LLM-based scoring for {len(all_use_cases)} use cases...")
        
        if not all_use_cases:
            self.logger.warning("No use cases to score.")
            return all_use_cases
        
        # Check if we need 2-pass scoring
        needs_two_pass = len(all_use_cases) > 2048
        
        if needs_two_pass:
            self.logger.warning(f"⚠️ Use case count ({len(all_use_cases)}) exceeds 2048. Implementing 2-PASS SCORING process...")
            log_print(f"\n{'='*80}", level="WARNING")
            log_print(f"⚠️  LARGE USE CASE SET DETECTED: {len(all_use_cases)} use cases", level="WARNING")
            log_print(f"{'='*80}")
            log_print(f"Implementing 2-PASS SCORING to select top 2048 use cases:")
            log_print(f"  PASS 1: Score all {len(all_use_cases)} use cases → Select top 2048")
            log_print(f"  PASS 2: Re-score top 2048 for final ranking")
            log_print(f"{'='*80}\n")
        
        # Format business context variables for the prompt
        strategic_goals_text = "\n".join([f"- {goal}" for goal in (strategic_goals or [])])
        business_priorities_text = "\n".join([f"- {priority}" for priority in (business_priorities or [])])
        
        domain_name = all_use_cases[0].get('Business Domain', 'Domain') if all_use_cases else "Domain"
        
        try:
            # Create markdown table for this domain (minimal fields to fit context)
            md_parts = ["| No | Name | Business Value |\n|---|---|---|\n"]
            for uc in all_use_cases:
                no = str(uc.get('No', '')).replace('|', r'\|')
                name = str(uc.get('Name', '')).replace('|', r'\|')
                business_value = str(uc.get('Business Value', '')).replace('|', r'\|')
                md_parts.append(f"| {no} | {name} | {business_value} |\n")
            use_case_markdown = "".join(md_parts)
            
            prompt_vars = {
                "use_case_markdown": use_case_markdown,
                "business_context": business_context or self._get_business_context_fallback(),
                "strategic_goals": strategic_goals_text or "- Maximize operational efficiency\n- Improve customer satisfaction",
                "business_priorities": business_priorities_text or "- Digital transformation\n- Cost optimization",
                "strategic_initiative": strategic_initiative or "Data-driven transformation program",
                "value_chain": value_chain or "Standard business operations",
                "revenue_model": revenue_model or "Product and service sales",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_VALUE_SCORE_PROMPT"),
            }
            
            self.logger.info(f"⏳ [{domain_name}] Waiting for LLM response (scoring {len(all_use_cases)} use cases)...")
            
            scoring_start_from_model = None
            scoring_data = None
            
            scoring_cascade = get_model_cascade_for_prompt("USE_CASE_VALUE_SCORE_PROMPT")
            prompt_template_text = self.ai_agent.prompt_templates.get("USE_CASE_VALUE_SCORE_PROMPT", "")
            estimated_prompt_size = len(prompt_template_text) + len(use_case_markdown) + 2000
            primary_model_limit = get_context_limit_chars_for_model(
                scoring_cascade[0], getattr(self, 'current_language', 'English')
            ) if scoring_cascade else 500000

            if estimated_prompt_size > primary_model_limit:
                best_model = max(scoring_cascade, key=lambda m: m.get("llm_input_context_tokens_count", 0)) if scoring_cascade else None
                best_limit = get_context_limit_chars_for_model(
                    best_model, getattr(self, 'current_language', 'English')
                ) if best_model else primary_model_limit

                if estimated_prompt_size > best_limit:
                    self.logger.warning(
                        f"📦 [{domain_name}] Scoring prompt ({estimated_prompt_size:,} chars) exceeds ALL models' "
                        f"limits (best: {best_limit:,} chars). Proactively batching {len(all_use_cases)} use cases."
                    )
                    batch_rows, scoring_start_from_model = self._score_use_cases_batched(
                        all_use_cases, prompt_vars, domain_name, best_limit
                    )
                    if batch_rows:
                        scoring_data = batch_rows
                        response_raw = None
                else:
                    self.logger.info(
                        f"📊 [{domain_name}] Scoring prompt ({estimated_prompt_size:,} chars) exceeds primary model "
                        f"({primary_model_limit:,} chars) but fits in cascade ({best_limit:,} chars). Using normal cascade."
                    )

            if scoring_data is None:
                try:
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Score_Use_Cases_{domain_name}",
                        worker_prompt_path="USE_CASE_VALUE_SCORE_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None
                    )
                except CascadeRebatchError as cre:
                    self.logger.warning(
                        f"🔄 [{domain_name}] Scoring cascade rebatch needed after '{cre.failed_model_name}' failed. "
                        f"Splitting {len(all_use_cases)} use cases into smaller scoring batches for "
                        f"model '{cre.target_model_name}' (limit: {cre.target_context_limit_chars:,} chars)"
                    )
                    batch_rows, scoring_start_from_model = self._score_use_cases_batched(
                        all_use_cases, prompt_vars, domain_name,
                        cre.target_context_limit_chars, cre.target_model_endpoint
                    )
                    if batch_rows:
                        scoring_data = batch_rows
                        response_raw = None
                    else:
                        raise Exception(f"All scoring batches failed for domain '{domain_name}' after cascade rebatch")

            if scoring_data is None:
                self.logger.info(f"✅ [{domain_name}] Received LLM response, processing results...")
                response_clean = clean_json_response(response_raw)
                scoring_data = CSVParser.parse_csv_string(
                    response_clean,
                    logger=self.logger,
                    context=f"Scoring for domain {domain_name}"
                )

            scoring_map = {item['No']: item for item in scoring_data}
            
            self.logger.info(f"Received scoring for {len(scoring_map)} use cases from LLM for domain '{domain_name}'")
            
            missing_ids = [uc['No'] for uc in all_use_cases if uc['No'] not in scoring_map]
            _rt = TECHNICAL_CONTEXT["runtime"]
            MAX_RETRY_ROUNDS = _rt["value_scoring_max_retry_rounds"]
            BATCH_SIZE_FOR_RETRY = _rt["value_scoring_retry_batch_size"]
            
            retry_round = 0
            while missing_ids and retry_round < MAX_RETRY_ROUNDS:
                retry_round += 1
                missing_ucs = [uc for uc in all_use_cases if uc['No'] in missing_ids]
                
                # If many missing, split into smaller batches for better success rate
                if len(missing_ucs) > BATCH_SIZE_FOR_RETRY:
                    self.logger.warning(f"⚠️ [{domain_name}] Round {retry_round}: {len(missing_ids)} use cases missing scores. Splitting into batches of {BATCH_SIZE_FOR_RETRY}...")
                    batches = [missing_ucs[i:i + BATCH_SIZE_FOR_RETRY] for i in range(0, len(missing_ucs), BATCH_SIZE_FOR_RETRY)]
                else:
                    self.logger.warning(f"⚠️ [{domain_name}] Round {retry_round}: Retrying {len(missing_ids)} missing use cases...")
                    batches = [missing_ucs]
                
                _scoring_lock = threading.Lock()
                num_retry_batches = len(batches)
                
                def _score_retry_batch(batch_idx, batch_ucs):
                    retry_md_parts = ["| No | Name | Business Value |\n|---|---|---|\n"]
                    for uc in batch_ucs:
                        no = str(uc.get('No', '')).replace('|', r'\|')
                        name = str(uc.get('Name', '')).replace('|', r'\|')
                        business_value = str(uc.get('Business Value', '')).replace('|', r'\|')
                        retry_md_parts.append(f"| {no} | {name} | {business_value} |\n")
                    retry_use_case_markdown = "".join(retry_md_parts)
                    
                    retry_prompt_vars = {
                        "use_case_markdown": retry_use_case_markdown,
                        "business_context": business_context or self._get_business_context_fallback(),
                        "strategic_goals": strategic_goals_text or "- Maximize operational efficiency\n- Improve customer satisfaction",
                        "business_priorities": business_priorities_text or "- Digital transformation\n- Cost optimization",
                        "strategic_initiative": strategic_initiative or "Data-driven transformation program",
                        "value_chain": value_chain or "Standard business operations",
                        "revenue_model": revenue_model or "Product and service sales",
                        "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_VALUE_SCORE_PROMPT"),
                    }
                    
                    try:
                        batch_label = f"Batch {batch_idx+1}/{num_retry_batches}" if num_retry_batches > 1 else ""
                        self.logger.info(f"⏳ [{domain_name}] Round {retry_round} {batch_label}: Scoring {len(batch_ucs)} use cases...")
                        retry_response_raw = self.ai_agent.run_worker(
                            step_name=f"Score_Use_Cases_{domain_name}_Retry{retry_round}_B{batch_idx+1}",
                            worker_prompt_path="USE_CASE_VALUE_SCORE_PROMPT",
                            prompt_vars=retry_prompt_vars,
                            response_schema=None,
                            start_from_model=scoring_start_from_model,
                            skip_cache=True  # v0.7.11: retries MUST bypass cache — initial bad response must not be served from cache on retry
                        )
                        retry_response_clean = clean_json_response(retry_response_raw)
                        retry_scoring_data = CSVParser.parse_csv_string(
                            retry_response_clean,
                            logger=self.logger,
                            context=f"Retry scoring for domain {domain_name} round {retry_round} batch {batch_idx+1}"
                        )
                        
                        new_scores = 0
                        with _scoring_lock:
                            for item in retry_scoring_data:
                                if item.get('No') and item['No'] not in scoring_map:
                                    scoring_map[item['No']] = item
                                    new_scores += 1
                        
                        if new_scores > 0:
                            self.logger.info(f"✅ [{domain_name}] Round {retry_round} {batch_label}: Got {new_scores} new scores (total: {len(scoring_map)})")
                    except Exception as retry_err:
                        batch_label = f"Batch {batch_idx+1}/{num_retry_batches}" if num_retry_batches > 1 else ""
                        self.logger.warning(f"[{domain_name}] Round {retry_round} {batch_label} failed: {str(retry_err)[:100]}")
                
                retry_workers = min(4, num_retry_batches)
                if num_retry_batches <= 1:
                    _score_retry_batch(0, batches[0])
                else:
                    _sr_total_timeout = self.llm_timeout_seconds * 2 * len(batches) + 120
                    with _SafeExecutorContext(max_workers=retry_workers, thread_name_prefix="score_retry", logger=self.logger, name="ScoreRetry") as ctx:
                        sr_futures = [ctx.submit(_score_retry_batch, i, b) for i, b in enumerate(batches)]
                        try:
                            for f in safe_as_completed(sr_futures, timeout=_sr_total_timeout):
                                try:
                                    f.result(timeout=self.llm_timeout_seconds * 2)
                                except Exception as exc:
                                    self.logger.warning(f"[{domain_name}] Retry scoring task failed: {get_clean_error_message(exc)}")
                        except concurrent.futures.TimeoutError:
                            ctx.mark_timed_out()
                            self.logger.warning(f"[{domain_name}] Score retry timed out globally ({_sr_total_timeout}s)")
                
                # Update missing_ids for next round
                missing_ids = [uc['No'] for uc in all_use_cases if uc['No'] not in scoring_map]
                if missing_ids:
                    self.logger.info(f"[{domain_name}] After round {retry_round}: Still missing {len(missing_ids)} scores")
            
            if missing_ids:
                self.logger.warning(f"⚠️ [{domain_name}] After {MAX_RETRY_ROUNDS} retry rounds, {len(missing_ids)} use cases still missing scores. Using defaults.")
            
            # Add scoring data to use cases and compute Value/Feasibility/Priority in code
            scored_use_cases = []
            for uc in all_use_cases:
                uc_id = uc['No']
                if uc_id in scoring_map:
                    scores = scoring_map[uc_id]
                    def _safe_score(v, default=3.5):
                        if v is None or v == "":
                            return float(default)
                        try:
                            return float(v)
                        except (ValueError, TypeError):
                            return float(default)
                    
                    uc['Strategic Alignment'] = _safe_score(scores.get('Strategic Alignment', 3.5))
                    uc['Return on Investment'] = _safe_score(scores.get('Return on Investment', 3.5))
                    uc['Reusability'] = _safe_score(scores.get('Reusability', 3.5))
                    uc['Time to Value'] = _safe_score(scores.get('Time to Value', 3.5))
                    uc['Data Availability'] = _safe_score(scores.get('Data Availability', 3.5))
                    uc['Data Accessibility'] = _safe_score(scores.get('Data Accessibility', 3.5))
                    uc['Architecture Fitness'] = _safe_score(scores.get('Architecture Fitness', 3.5))
                    uc['Team Skills'] = _safe_score(scores.get('Team Skills', 3.5))
                    uc['Domain Knowledge'] = _safe_score(scores.get('Domain Knowledge', 3.5))
                    uc['People Allocation'] = _safe_score(scores.get('People Allocation', 3.5))
                    uc['Budget Allocation'] = _safe_score(scores.get('Budget Allocation', 3.5))
                    uc['Time to Production'] = _safe_score(scores.get('Time to Production', 3.5))
                    uc['Business Priority Alignment'] = scores.get('Business Priority Alignment', 'General Improvement')
                    uc['Strategic Goals Alignment'] = scores.get('Strategic Goals Alignment', 'General Improvement')

                    value_score = (
                        (uc['Return on Investment'] * 0.60)
                        + (uc['Strategic Alignment'] * 0.25)
                        + (uc['Time to Value'] * 0.075)
                        + (uc['Reusability'] * 0.075)
                    )

                    feasibility_inputs = [
                        uc['Data Availability'],
                        uc['Data Accessibility'],
                        uc['Architecture Fitness'],
                        uc['Team Skills'],
                        uc['Domain Knowledge'],
                        uc['People Allocation'],
                        uc['Budget Allocation'],
                        uc['Time to Production'],
                    ]
                    feasibility_score = sum(feasibility_inputs) / len(feasibility_inputs)

                    priority_score = (value_score * 1.5) + (feasibility_score * 0.5)

                    uc['Value'] = round(value_score, 2)
                    uc['Feasibility'] = round(feasibility_score, 2)
                    uc['Priority'] = round(priority_score, 2)
                    
                    if 'Justification' in scores:
                        uc['Justification'] = scores.get('Justification', '')
                    
                    uc['AI_Confidence'] = scores.get('AI_Confidence', 0.5)
                    uc['AI_Feedback'] = scores.get('AI_Feedback', 'No feedback provided by AI scoring.')
                    
                    self.logger.debug(f"Scored {uc_id}: Value={uc['Value']}, Feasibility={uc['Feasibility']}, Priority={priority_score}")
                else:
                    self.logger.warning(f"No scoring data received for use case {uc_id}, using defaults")
                    uc['Strategic Alignment'] = 3.5
                    uc['Return on Investment'] = 3.5
                    uc['Reusability'] = 3.5
                    uc['Time to Value'] = 3.5
                    uc['Data Availability'] = 3.5
                    uc['Data Accessibility'] = 3.5
                    uc['Architecture Fitness'] = 3.5
                    uc['Team Skills'] = 3.5
                    uc['Domain Knowledge'] = 3.5
                    uc['People Allocation'] = 3.5
                    uc['Budget Allocation'] = 3.5
                    uc['Time to Production'] = 3.5
                    uc['Value'] = 3.5
                    uc['Feasibility'] = 3.5
                    uc['Priority'] = 7.0
                    uc['AI_Confidence'] = 0.5
                    uc['AI_Feedback'] = 'Default scoring applied - no LLM scoring data received.'
                
                scored_use_cases.append(uc)
            
            self.logger.info(f"Pass 1 scoring complete for {len(scored_use_cases)} use cases in domain '{domain_name}'")
            
            if needs_two_pass and len(scored_use_cases) > 2048:
                self.logger.warning(f"🔄 Starting PASS 2: Selecting top 2048 use cases and re-scoring...")
                log_print(f"\n{'='*80}")
                log_print(f"🔄 PASS 1 COMPLETE: {len(scored_use_cases)} use cases scored")
                log_print(f"{'='*80}")
                log_print(f"Selecting top 2048 use cases by Priority for PASS 2...")
                
                scored_use_cases.sort(key=lambda x: x.get('Priority', 0), reverse=True)
                top_2048 = scored_use_cases[:2048]
                excluded_count = len(scored_use_cases) - 2048
                
                self.logger.info(f"Selected top 2048 use cases. Excluded {excluded_count} lower-priority use cases.")
                log_print(f"✓ Selected top 2048 use cases")
                log_print(f"✗ Excluded {excluded_count} lower-priority use cases")
                log_print(f"\nStarting PASS 2: Re-scoring top 2048 use cases for final ranking...")
                log_print(f"{'='*80}\n")
                
                final_scored = self._score_use_cases(
                    top_2048,
                    business_context=business_context,
                    strategic_goals=strategic_goals,
                    business_priorities=business_priorities,
                    strategic_initiative=strategic_initiative,
                    value_chain=value_chain,
                    revenue_model=revenue_model
                )
                
                self.logger.info(f"✅ PASS 2 complete. Final set: {len(final_scored)} use cases")
                log_print(f"\n{'='*80}")
                log_print(f"✅ 2-PASS SCORING COMPLETE")
                log_print(f"{'='*80}")
                log_print(f"Final use case count: {len(final_scored)}")
                log_print(f"{'='*80}\n")
                
                return final_scored
            
            self.logger.info(f"Scoring complete for {len(scored_use_cases)} use cases in domain '{domain_name}'")
            return scored_use_cases
            
        except Exception as e:
            self.logger.error(f"Use case scoring failed: {get_clean_error_message(e)}. Proceeding without LLM scoring.")
            for uc in all_use_cases:
                uc['Strategic Alignment'] = 3.5
                uc['Return on Investment'] = 3.5
                uc['Reusability'] = 3.5
                uc['Time to Value'] = 3.5
                uc['Data Availability'] = 3.5
                uc['Data Accessibility'] = 3.5
                uc['Architecture Fitness'] = 3.5
                uc['Team Skills'] = 3.5
                uc['Domain Knowledge'] = 3.5
                uc['People Allocation'] = 3.5
                uc['Budget Allocation'] = 3.5
                uc['Time to Production'] = 3.5
                uc['Value'] = 3.5
                uc['Feasibility'] = 3.5
                uc['Priority'] = 7.0
            return all_use_cases

    def _translate_and_prepare_language_pack(self, lang: str, flat_english_use_cases: list, english_grouped_data: dict, business_name: str) -> tuple:
        """
        A single-function wrapper to run all data-gathering for a language.
        Designed to be run in a ThreadPoolExecutor.
        """
        try:
            # Set language for context limit calculations
            self.ai_agent.set_language(lang)
            
            self.logger.info(f"Starting translation & summary pack for {lang}...")
            lang_abbr = self._get_lang_abbr(lang)
            lang_translations = self.translation_service.get_translations(lang)
            # Disable parallelization to avoid nested ThreadPoolExecutors (this function is already called in parallel)
            lang_use_cases_translated = self.translation_service.translate_use_case_list([uc.copy() for uc in flat_english_use_cases], lang, max_parallelism=self.max_parallelism, enable_parallelization=False)
            lang_grouped_data = self._align_translated_data(english_grouped_data, lang_use_cases_translated)
            (lang_summary_dict, transliterated_name) = self._get_salesy_summary(lang_grouped_data, business_name, lang, lang_translations)
            
            self.logger.debug(f"Successfully processed all data for {lang}.")
            return (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name)
        except Exception as e:
            self.logger.error(f"Failed to process translation artifacts for {lang}: {get_clean_error_message(e)}")
            return (lang, lang_abbr, None, None, None, None) # Return Nones to signal failure

    def _generate_documents_for_all_languages(self, final_consolidated_use_cases: list, english_grouped_data: dict = None, summary_dict: dict = None, languages: list = None, skip_excel_langs: list = None):
        """
        Generates PDF/PPTX/Excel documents for all languages.
        Can be called from normal path (after use case generation) or docs-only path (from JSON).
        
        Args:
            final_consolidated_use_cases: List of use cases (flat)
            english_grouped_data: Optional, will be computed if not provided
            summary_dict: Optional, will be computed if not provided
            languages: Optional list of languages to generate (default: self.output_languages)
        """
        target_languages = languages if languages else self.output_languages
        skip_excel_langs = set(skip_excel_langs or [])
        self.logger.info(f"--- Starting Document Generation for Languages: {target_languages} ---")
        
        # CRITICAL: Install all required dependencies BEFORE starting translations
        # This prevents wasting time on translations if dependencies are missing
        self.logger.info("Checking and installing required dependencies before starting translations...")
        dependencies_ok = True
        
        if "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices:
            self.logger.info("Checking PDF dependencies (weasyprint)...")
            try:
                import weasyprint
                self.logger.info("✓ PDF package (weasyprint) already installed.")
            except ImportError:
                self.logger.info("PDF package (weasyprint) not found. Installing...")
                try:
                    import subprocess, sys
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "weasyprint"])
                    import weasyprint
                    self.logger.info("✓ PDF package (weasyprint) installed successfully.")
                except Exception as e:
                    self.logger.error(f"✗ Failed to install PDF dependencies: {get_clean_error_message(e)}")
                    dependencies_ok = False
        
        if "Presentation" in self.generate_choices:
            self.logger.info("Checking PPTX dependencies (python-pptx)...")
            try:
                import pptx
                self.logger.info("✓ PPTX package (python-pptx) already installed.")
            except ImportError:
                self.logger.info("PPTX package (python-pptx) not found. Installing...")
                try:
                    import subprocess, sys
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx"])
                    import pptx
                    self.logger.info("✓ PPTX package (python-pptx) installed successfully.")
                except Exception as e:
                    self.logger.error(f"✗ Failed to install PPTX dependencies: {get_clean_error_message(e)}")
                    dependencies_ok = False
        
        # Always check Excel dependencies as they're used for all artifact generation
        self.logger.info("Checking Excel dependencies (pandas, openpyxl)...")
        try:
            import pandas, openpyxl
            self.logger.info("✓ Excel packages (pandas, openpyxl) already installed.")
        except ImportError:
            self.logger.info("Excel packages not found. Installing...")
            try:
                import subprocess, sys
                subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "openpyxl"])
                import pandas, openpyxl
                self.logger.info("✓ Excel packages (pandas, openpyxl) installed successfully.")
            except Exception as e:
                self.logger.error(f"✗ Failed to install Excel dependencies: {get_clean_error_message(e)}")
                dependencies_ok = False
        
        if not dependencies_ok:
            self.logger.warning("⚠️ Some dependencies failed to install. PDF/Excel generation may fail, but .md and .csv files will still be generated.")
        
        self.logger.info("✓ Proceeding with translations and artifact generation (fallback to .md/.csv if needed)...")
        
        # Prepare English data if not provided
        flat_english_use_cases = final_consolidated_use_cases
        if english_grouped_data is None:
            _unsorted_grouped = self._group_use_cases_by_domain_flat(flat_english_use_cases)
            english_grouped_data = {k: _unsorted_grouped[k] for k in sorted(_unsorted_grouped.keys())}
        
        # Get English translations
        english_translations = self.translation_service.get_translations("English")
        
        # Get summary if not provided
        if summary_dict is None:
            (summary_dict, transliterated_name) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
        
        # STEP 1: Run translations in parallel (no nesting)
        # ADAPTIVE PARALLELISM: Calculate based on languages and use cases
        num_languages = len([l for l in target_languages if l != "English"])
        num_use_cases = len(flat_english_use_cases)
        
        translation_parallelism, reason = calculate_adaptive_parallelism(
            "translation", self.max_parallelism,
            num_items=num_use_cases,
            num_domains=num_languages,
            is_llm_operation=True, logger=self.logger,
            estimated_mb_per_worker=15.0
        )
        log_adaptive_parallelism_decision("translation", translation_parallelism, self.max_parallelism, reason)
        
        translation_futures = []
        translation_results = {}
        
        _lang_timed_out = False
        with _SafeExecutorContext(max_workers=translation_parallelism, thread_name_prefix="Translator", logger=self.logger, name="LangPack") as ctx:
            for lang in target_languages:
                if lang == "English":
                    self.logger.info("Preparing English artifacts (no translation needed).")
                    (summary_dict_en, transliterated_name_en) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
                    lang_abbr = self._get_lang_abbr("English")
                    translation_results["English"] = ("English", lang_abbr, english_translations, english_grouped_data, summary_dict_en, transliterated_name_en)
                    continue

                self.logger.info(f"Submitting translation & summary pack job for {lang}...")
                f = ctx.submit(
                    self._translate_and_prepare_language_pack,
                    lang, flat_english_use_cases, english_grouped_data, self.business_name
                )
                translation_futures.append((f, lang))

            self.logger.info(f"Waiting for {len(translation_futures)} language packs to complete...")
            total_timeout = len(translation_futures) * 1800
            self.logger.info(f"Language pack processing timeout set to {total_timeout}s ({total_timeout//60} minutes)")
            
            for future, lang in translation_futures:
                try:
                    result = _safe_future_result(future, timeout=1800)
                    (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name) = result
                    
                    if lang_translations is None:
                        self.logger.warning(f"Skipping artifact generation for {lang} due to translation/summary failure.")
                        continue

                    self.logger.info(f"Translation pack for {lang} complete.")
                    translation_results[lang] = result
                except (concurrent.futures.TimeoutError, TimeoutError):
                    _lang_timed_out = True
                    ctx.mark_timed_out()
                    self.logger.error(f"Language pack processing timed out after 30 minutes for {lang}")
                except Exception as e:
                    self.logger.error(f"Language pack processing failed for {lang}: {get_clean_error_message(e)}")
        
        # STEP 2: Run artifact writing in parallel (separate, not nested)
        # ADAPTIVE PARALLELISM: Calculate based on number of artifacts to write
        num_artifacts = len(translation_results) * 3  # Roughly: PDF, PPTX, Excel per language
        
        writing_parallelism, reason = calculate_adaptive_parallelism(
            "artifact_writing", self.max_parallelism,
            num_items=num_artifacts,
            is_llm_operation=False, logger=self.logger
        )
        log_adaptive_parallelism_decision("artifact_writing", writing_parallelism, self.max_parallelism, reason)
        self.logger.info(f"Translations complete. Starting artifact generation for {len(translation_results)} languages...")
        
        with _SafeExecutorContext(max_workers=writing_parallelism, thread_name_prefix="Writer", logger=self.logger, name="ArtifactWriter") as writer_ctx:
            writing_futures = []
            
            for lang, result in translation_results.items():
                (lang, lang_abbr, lang_translations, lang_grouped_data, lang_summary_dict, transliterated_name) = result
                
                self.logger.info(f"Submitting writing jobs for {lang}...")
                
                if lang == "English":
                    f = writer_ctx.submit(self._generate_markdown_catalog, lang, lang_abbr, lang_grouped_data, lang_summary_dict, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="Markdown Catalog",
                        sub_step_name=lang,
                        message=f"{lang} markdown generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} Markdown", step_id, "Markdown Catalog", lang))
                    f = writer_ctx.submit(self._generate_csv_catalog, lang, lang_abbr, lang_grouped_data)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="CSV Catalog",
                        sub_step_name=lang,
                        message=f"{lang} CSV generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} CSV", step_id, "CSV Catalog", lang))
                
                if "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices:
                    f = writer_ctx.submit(self.generate_catalog_pdf, lang, lang_abbr, lang_translations, lang_summary_dict, lang_grouped_data, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Document Generation",
                        step_name="PDF Catalog",
                        sub_step_name=lang,
                        message=f"{lang} PDF generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} PDF", step_id, "PDF Catalog", lang))
                if "Presentation" in self.generate_choices:
                    f = writer_ctx.submit(self.generate_presentation_pptx, lang, lang_abbr, lang_translations, lang_summary_dict, lang_grouped_data, transliterated_name)
                    step_id = self._emit_pipeline_status(
                        stage_name="Document Generation",
                        step_name="Presentation",
                        sub_step_name=lang,
                        message=f"{lang} presentation generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} PPTX", step_id, "Presentation", lang))
                if lang == "English" and lang not in skip_excel_langs:
                    f = writer_ctx.submit(self._generate_use_case_excel, lang, lang_abbr, lang_grouped_data)
                    step_id = self._emit_pipeline_status(
                        stage_name="Excel Generation",
                        step_name="Use Case Excel",
                        sub_step_name=lang,
                        message=f"{lang} Excel generation started",
                        status="started",
                        progress_increment=1.0
                    )
                    writing_futures.append((f, f"{lang} Excel", step_id, "Use Case Excel", lang))
                elif lang == "English":
                    self.logger.info(f"Skipping Excel generation for {lang} (already generated).")
                else:
                    self.logger.info(f"Skipping Excel generation for {lang} (English only).")
            
            for future, job_name, start_step_id, step_name, lang in writing_futures:
                try:
                    _safe_future_result(future, timeout=600)
                    self.logger.info(f"✓ {job_name} completed")
                    self._emit_pipeline_status(
                        stage_name="Artifact Writing",
                        step_name=step_name,
                        sub_step_name=lang,
                        message=f"{job_name} completed",
                        status="ended_success",
                        progress_increment=1.0,
                        result_json={"job_name": job_name, "artifact_type": step_name, "language": lang},
                        step_id=start_step_id
                    )
                except Exception as e:
                    self.logger.error(f"✗ {job_name} failed: {get_clean_error_message(e)}")
                    self._emit_pipeline_status(
                        stage_name="Artifact Writing",
                        step_name=step_name,
                        sub_step_name=lang,
                        message=f"{job_name} failed",
                        status="ended_error",
                        progress_increment=1.0,
                        result_json={"job_name": job_name, "artifact_type": step_name, "language": lang, "error": get_clean_error_message(e, max_chars=300)},
                        step_id=start_step_id
                    )
        
        self.logger.info("All artifact writing jobs completed.")

    def _resolve_live_log_target(self):
        """Pick a destination for the live log file. Priority (v0.8.5):
          1. UC Volume derived from inspire_database (best: external `databricks fs cat` tailing)
          2. Fallback: `<base_output_dir>/.inspire_logs/session_<sid>/log.log` under the
             user-provided generation_path (works for both /Volumes/... and /Workspace/...)
          3. None → caller logs a loud warning and bails.

        Returns dict with keys: dst_path, kind ('volume'|'workspace'), parent_dir.
        Returns None when no destination can be resolved (caller logs guidance)."""
        import os, re
        biz = re.sub(r"[^A-Za-z0-9._-]+", "_", str(getattr(self, "business_name", "unknown") or "unknown"))
        sid = str(getattr(self, "session_id", "unknown") or "unknown")

        db = (getattr(self, "inspire_database", None) or "").strip()
        parts = [p.strip("`") for p in db.split(".")] if db else []
        if len(parts) >= 2 and parts[0] and parts[1]:
            catalog, schema = parts[0], parts[1]
            try:
                self.spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.vol_root")
                vol_base = f"/Volumes/{catalog}/{schema}/vol_root/logs/{biz}/session_{sid}"
                os.makedirs(vol_base, exist_ok=True)
                return {"dst_path": os.path.join(vol_base, "log.log"), "kind": "volume", "parent_dir": vol_base}
            except Exception as _cve:
                self.logger.warning(f"UC volume target unavailable, falling back to generation_path: {get_clean_error_message(_cve)}")

        base = (getattr(self, "base_output_dir", None) or "").strip()
        if base:
            parent = os.path.join(base, ".inspire_logs", f"session_{sid}")
            kind = "workspace" if (base.startswith("/Workspace/") or "@" in base) else "volume"
            if kind == "volume":
                try:
                    os.makedirs(parent, exist_ok=True)
                except Exception as _me:
                    self.logger.warning(f"Live log fallback parent dir create failed: {get_clean_error_message(_me)}")
                    return None
            else:
                # v0.8.6 fix: workspace.import_ does NOT auto-create parents —
                # without this mkdirs every 10s flush silently fails with
                # "parent folder does not exist" (debug-level log).
                try:
                    self.w_client.workspace.mkdirs(parent)
                except Exception as _wme:
                    self.logger.warning(f"Live log workspace mkdirs failed: {get_clean_error_message(_wme)}")
                    return None
            return {"dst_path": os.path.join(parent, "log.log"), "kind": kind, "parent_dir": parent}

        return None

    def _start_live_log_flush_thread(self):
        """Copy local log.txt to a live-tailable destination every 10s so external
        observers (humans, agents, CI) can read the run line-by-line without
        waiting for `_upload_log_file` at the very end.

        v0.8.5: now ALWAYS fires (was: only when inspire_database widget set).
        Destination resolution order in `_resolve_live_log_target`:
          1. UC Volume from inspire_database (preferred — external SQL/CLI tail)
          2. `<generation_path>/.inspire_logs/session_<sid>/log.log` (workspace OR volume)
        Cadence: 10s (was 60s in v0.8.3).

        Backward-compat: keeps writing to `_volume_log_*` instance attrs so the
        teardown in `_upload_log_file` works unchanged."""
        import threading, os, shutil, base64
        try:
            target = self._resolve_live_log_target()
            if not target:
                self.logger.warning(
                    "🔇 Live log streaming DISABLED — no inspire_database widget set AND "
                    "no usable generation_path. External tailers will see nothing until "
                    "_upload_log_file runs at the very end of the pipeline. To enable "
                    "live monitoring, set 02_inspire_database (preferred) or ensure "
                    "11_generation_path is writable."
                )
                return
            dst_log = target["dst_path"]
            kind = target["kind"]
            src_log = os.path.join(self.local_log_output_dir, "log.txt")
            self._volume_log_dst = dst_log
            self._volume_log_kind = kind
            self._volume_log_stop = threading.Event()

            def _flush_once():
                if not os.path.exists(src_log):
                    return
                if kind == "workspace":
                    with open(src_log, "rb") as _f:
                        _b64 = base64.b64encode(_f.read()).decode()
                    self.w_client.workspace.import_(
                        path=dst_log, content=_b64,
                        format=workspace.ImportFormat.AUTO, overwrite=True
                    )
                else:
                    shutil.copyfile(src_log, dst_log)

            self._live_log_fail_count = 0
            def _loop():
                while not self._volume_log_stop.is_set():
                    try:
                        _flush_once()
                        self._live_log_fail_count = 0
                    except Exception as _ce:
                        try:
                            # v0.8.6: first failure at WARNING so silent breakage
                            # is visible in the cluster log; subsequent failures
                            # at DEBUG to avoid noise.
                            self._live_log_fail_count += 1
                            _msg = f"Live log flush failed (n={self._live_log_fail_count}): {get_clean_error_message(_ce)}"
                            if self._live_log_fail_count == 1:
                                self.logger.warning(_msg)
                            else:
                                self.logger.debug(_msg)
                        except Exception:
                            pass
                    if self._volume_log_stop.wait(timeout=10.0):
                        break
                try:
                    _flush_once()
                except Exception:
                    pass

            t = threading.Thread(target=_loop, name=f"LiveLogFlush_{getattr(self, 'session_id', 'x')}", daemon=True)
            t.start()
            self._volume_log_thread = t
            self.logger.info(f"📝 Live log streaming enabled (kind={kind}, 10s flush) → {dst_log}")
        except Exception as _e:
            try:
                self.logger.warning(f"Live log streaming setup failed: {get_clean_error_message(_e)}")
            except Exception:
                pass

    # v0.8.4 alias — keep old name for any internal caller that referenced it.
    def _start_volume_log_flush_thread(self):
        return self._start_live_log_flush_thread()

    def run(self):
        import time
        self._run_start_time = time.time()
        self.logger.info(f"Starting tasks: {self.generate_choices}, Operation Mode: {self.operation_mode}")
        self.logger.info(f"🔑 Session ID: {self.session_id}")
        log_print(f"🔑 Inspire Session ID: {self.session_id}")
        if self.inspire_database:
            self._ensure_inspire_database_exists()
            self._create_tracking_table()
        try:
            self._start_live_log_flush_thread()
        except Exception as _vle:
            try:
                self.logger.warning(f"Live log streaming failed to start (non-critical): {get_clean_error_message(_vle)}")
            except Exception:
                pass
        if self.atomic_writer:
            self.atomic_writer.logger = self.logger
            self.atomic_writer.initialize_session()
            self._pipeline_start_step_id = self._emit_pipeline_status(
                stage_name="Pipeline",
                step_name="Execution",
                sub_step_name="Start",
                message="Inspire pipeline started",
                status="started",
                progress_increment=1.0,
                result_json={"operation_mode": self.operation_mode, "session_id": self.session_id}
            )
        if self.inspire_database:
            _db_setup_step_id = self._emit_pipeline_status(
                stage_name="Initialization",
                step_name="Database Setup",
                sub_step_name=self.inspire_database,
                message="Setting up Inspire database",
                status="started",
                progress_increment=0.0
            )
            self._emit_pipeline_status(
                stage_name="Initialization",
                step_name="Database Setup",
                sub_step_name=self.inspire_database,
                message="Inspire database and tracking structures are ready",
                status="ended_success",
                progress_increment=1.0,
                result_json={"database": self.inspire_database},
                step_id=_db_setup_step_id
            )
        
        if 'PROMPT_TEMPLATES' not in globals():
            self.logger.critical("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.")
            log_print("CRITICAL ERROR: 'PROMPT_TEMPLATES' dictionary is not defined. Please run the cell defining it.", level="CRITICAL")
            raise NameError("PROMPT_TEMPLATES not found. Please run the cell that defines the prompt dictionary.")
        
        if self.auto_parallelism and self.max_parallelism <= 0:
            (recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory) = self._calculate_dynamic_parallelism(
                0,
                0,
                0,
                0
            )
            self.max_parallelism = recommended
            self.logger.info(f"Dynamic parallelism set to {self.max_parallelism} (memory_cap={max_by_memory})")
            log_print(f"✅ Dynamic parallelism: {self.max_parallelism}")

        english_translations = self.translation_service.get_translations("English")
        final_consolidated_use_cases = []
        
        if self.json_file_path:
            self.logger.info(f"🚀 JSON MODE: Loading use cases from JSON file: {self.json_file_path}")
            log_print(f"\n{'='*80}")
            log_print(f"🚀 JSON MODE ACTIVATED")
            log_print(f"{'='*80}")
            log_print(f"📁 JSON File: {self.json_file_path}")
            log_print(f"📋 Languages: {', '.join(self.output_languages)}")
            log_print(f"\n⚠️  SKIPPING:", level="WARNING")
            log_print(f"   ❌ Use Case Generation (using existing data from JSON)")
            log_print(f"\n✅ GENERATING:")
            log_print(f"   📓 Notebooks (always generated)")
            log_print(f"   📄 PDF Catalogs" if "PDF Catalog" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices else "   (PDF generation not selected)")
            log_print(f"   📊 Presentations" if "Presentation" in self.generate_choices else "   (Presentation generation not selected)")
            log_print(f"{'='*80}\n")
            
            try:
                # Load the JSON catalog
                (final_consolidated_use_cases, summary_dict, english_grouped_data) = self._load_usecases_catalog_json(self.json_file_path)
                
                # 1. Generate documents for all languages (Excel deferred until after notebooks)
                if "PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices or "Use Cases Catalog PDF" in self.generate_choices:
                    self._generate_documents_for_all_languages(
                        final_consolidated_use_cases,
                        english_grouped_data=english_grouped_data,
                        summary_dict=summary_dict,
                        skip_excel_langs=["English"]
                    )
                
                # Report table inclusion/exclusion statistics
                if final_consolidated_use_cases:
                    self._report_table_statistics(final_consolidated_use_cases)
                
                # 2. Generate Genie Code instructions (if selected, JSON mode)
                if self.generate_genie_instructions and final_consolidated_use_cases:
                    self.logger.info("🤖 Genie Code Instructions selected - generating from JSON data...")
                    log_print(f"\n{'='*80}")
                    log_print(f"🤖 GENIE CODE INSTRUCTIONS (from JSON)")
                    log_print(f"{'='*80}")
                    _json_schema_details = self._build_schema_details_from_use_cases(final_consolidated_use_cases)
                    try:
                        self._generate_genie_instructions_and_notebooks(
                            final_consolidated_use_cases,
                            _json_schema_details,
                            "",
                            english_translations,
                            summary_dict
                        )
                    except Exception as e:
                        self.logger.error(f"Genie Code instruction generation failed: {get_clean_error_message(e)}")
                        log_print(f"⚠️ Genie Code instruction generation encountered an error: {str(e)[:200]}", level="WARNING")
                    finally:
                        del _json_schema_details
                elif (not self.generate_genie_instructions) and final_consolidated_use_cases and getattr(self, 'operation', '') == 'Discover Use Cases':
                    self.logger.info("🔎 Discover mode (JSON): Capturing per-UC prompt fields + generating skeleton notebooks...")
                    _json_schema_details = self._build_schema_details_from_use_cases(final_consolidated_use_cases)
                    try:
                        final_consolidated_use_cases = self._run_discover_skeleton_phase(
                            final_consolidated_use_cases,
                            _json_schema_details
                        )
                    except Exception as e:
                        self.logger.error(f"Discover skeleton phase failed (JSON): {get_clean_error_message(e)}")
                        log_print(f"⚠️ Discover skeleton phase failed: {str(e)[:200]}", level="WARNING")
                    finally:
                        del _json_schema_details
                
                # 3. Generate Notebooks (batch fallback — only if NOT already streamed inline during genie phase)
                _notebooks_done_inline_json = getattr(self, '_notebooks_generated_inline', False)
                if final_consolidated_use_cases and not _notebooks_done_inline_json:
                    self.logger.info("📓 Generating per-use-case notebooks from JSON data (batch fallback)...")
                    self.assemble_use_case_notebooks(final_consolidated_use_cases, english_translations, summary_dict)
                elif _notebooks_done_inline_json:
                    self.logger.info("📓 Notebooks already generated inline during genie streaming phase — skipping batch generation")
                else:
                    self.logger.warning("No use cases found in JSON, skipping notebook creation.")
                
                # 4. Generate English Excel AFTER notebooks exist so Use Case ID hyperlinks resolve
                if final_consolidated_use_cases:
                    self.logger.info("Generating English Excel (post-notebook, with hyperlinks)...")
                    lang_abbr_en = self._get_lang_abbr("English")
                    try:
                        self._generate_use_case_excel("English", lang_abbr_en, english_grouped_data)
                    except Exception as e:
                        self.logger.error(f"Failed to generate English Excel: {get_clean_error_message(e)}")

                # Upload log file and show summary
                self.logger.info(f"✅ All artifacts for {self.business_name} generated successfully from JSON")
                self.logger.info("Uploading log file...")
                self._upload_log_file()
                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                
                # Final success message
                log_print(f"\n{'='*80}")
                log_print(f"✅ SUCCESS: All artifacts generated successfully from JSON")
                log_print(f"{'='*80}\n")
                return
                
            except Exception as e:
                self.logger.critical(f"Failed to process in docs-only mode: {get_clean_error_message(e)}")
                log_print(f"\n❌ FATAL: Failed to process in docs-only mode: {get_clean_error_message(e, max_chars=200)}")
                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                raise
        
        # === GENERATE USE CASES (Operation == 'Generate Use Cases'): short-circuit. ===
        # Scans __inspire_usecases (Route 1) then workspace notebooks (Route 2) for flagged
        # use cases and regenerates their Genie Code Instruction only. Entire use-case
        # discovery pipeline is skipped.
        if getattr(self, 'operation', '') == 'Generate Use Cases':
            try:
                self._run_generate_mode()
            except Exception as _gen_err:
                self.logger.critical(f"Generate Use Cases mode failed: {get_clean_error_message(_gen_err)}")
                log_print(f"\n❌ FATAL: Generate Use Cases mode failed: {get_clean_error_message(_gen_err, max_chars=200)}")
                AIAgent.get_summary_report()
                self.ai_agent.get_manager_stats_report()
                raise
            AIAgent.get_summary_report()
            self.ai_agent.get_manager_stats_report()
            return
        
        # === NORMAL PATH: Generate use cases from metadata ===
        
        # === NEW: Call Business Context Worker first ===
        self.logger.info("=" * 80)
        self.logger.info("🚀 STEP 1: EXTRACTING BUSINESS CONTEXT, STRATEGIC GOALS, AND PRIORITIES")
        self.logger.info("=" * 80)
        _ctx_extraction_step_id = self._emit_pipeline_status(
            stage_name="Business Context",
            step_name="Context Extraction",
            sub_step_name="LLM",
            message="Business context extraction started",
            status="started",
            progress_increment=1.0
        )
        
        try:
            # Prepare user context string early - now uses business_domains instead of use_cases_focus
            user_domains_str = ', '.join(self.user_business_domains) if self.user_business_domains else ''
            
            # Extract business context AND interpret generation instructions in parallel
            import concurrent.futures as _ctx_futures
            with _ctx_futures.ThreadPoolExecutor(max_workers=2, thread_name_prefix="CtxInit") as _ctx_executor:
                _ctx_future = _ctx_executor.submit(self._get_business_context_from_llm)
                _instr_future = _ctx_executor.submit(self._interpret_generation_instructions)
                llm_business_context = _ctx_future.result(timeout=300)
                _parsed_instructions = _instr_future.result(timeout=120)
            self.logger.info("✅ Business context extraction and generation instructions interpretation completed (parallel)")
            
            # Merge with user-provided domains (user takes precedence)
            merged_business_context = self._merge_business_contexts(llm_business_context, user_domains_str)
            
            # Handle user-provided strategic goals (hard focus)
            if self.user_strategic_goals:
                self.logger.info(f"✅ User provided {len(self.user_strategic_goals)} strategic goals - ONLY these will be used")
                merged_business_context["strategic_goals"] = self.user_strategic_goals
                self.logger.info(f"   Strategic Goals: {', '.join(self.user_strategic_goals)}")
            else:
                self.logger.info("ℹ️ No user strategic goals provided - will generate goals based on business context")
                llm_goals = merged_business_context.get("strategic_goals", [])
                if isinstance(llm_goals, str):
                    llm_goals = [g.strip() for g in llm_goals.split(",") if g.strip()]
                merged_business_context["strategic_goals"] = llm_goals
            
            # Handle user-provided business priorities
            if self.user_business_priorities:
                self.logger.info(f"✅ User provided {len(self.user_business_priorities)} business priorities - these will drive ranking")
                merged_business_context["business_priorities"] = self.user_business_priorities
                self.logger.info(f"   Business Priorities: {', '.join(self.user_business_priorities)}")
            
            # Handle user-provided business domains
            if self.user_business_domains:
                self.logger.info(f"✅ User provided {len(self.user_business_domains)} business domains - use cases will be aligned to these domains ONLY")
                merged_business_context["user_business_domains"] = self.user_business_domains
                self.logger.info(f"   Business Domains: {', '.join(self.user_business_domains)}")
            else:
                self.logger.info("ℹ️ No user business domains provided - domains will be inferred from data")
            
            self.logger.info("✅ Business context extracted and merged.")
            self.logger.info("=" * 80)
            _ctx_result = {
                "business_domains_count": len(merged_business_context.get("user_business_domains", []) or []),
                "strategic_goals_count": len(merged_business_context.get("strategic_goals", []) or []),
                "business_context": str(merged_business_context.get("business_context", ""))[:500],
                "strategic_goals": merged_business_context.get("strategic_goals", []),
                "business_priorities": merged_business_context.get("business_priorities", []),
                "user_business_domains": merged_business_context.get("user_business_domains", []),
                "strategic_initiative": str(merged_business_context.get("strategic_initiative", ""))[:300],
                "revenue_model": str(merged_business_context.get("revenue_model", ""))[:300],
            }
            self._emit_pipeline_status(
                stage_name="Business Context",
                step_name="Context Extraction",
                sub_step_name="LLM",
                message="Business context extraction completed",
                status="ended_success",
                progress_increment=3.0,
                result_json=_ctx_result,
                step_id=_ctx_extraction_step_id
            )
        except Exception as _ctx_err:
            self._emit_pipeline_status(
                stage_name="Business Context",
                step_name="Context Extraction",
                sub_step_name="LLM",
                message=f"Business context extraction failed: {get_clean_error_message(_ctx_err, max_chars=200)}",
                status="ended_error",
                progress_increment=1.0,
                result_json={"error": get_clean_error_message(_ctx_err, max_chars=300)},
                step_id=_ctx_extraction_step_id
            )
            raise
        
        # Store merged context for use in prompt generation
        self.merged_business_context = merged_business_context
        self._persist_session_prompt_context()
        
        # Extract business context fields for prompts
        ctx_business_context = merged_business_context.get("business_context", "")
        ctx_strategic_goals = merged_business_context.get("strategic_goals", [])
        # Handle if strategic_goals is a string (comma separated) or list
        if isinstance(ctx_strategic_goals, str):
            ctx_strategic_goals = [s.strip() for s in ctx_strategic_goals.split(",") if s.strip()]
        
        ctx_business_priorities = merged_business_context.get("business_priorities", [])
        if isinstance(ctx_business_priorities, str):
            ctx_business_priorities = [s.strip() for s in ctx_business_priorities.split(",") if s.strip()]
        ctx_strategic_initiative = merged_business_context.get("strategic_initiative", "")
        ctx_value_chain = merged_business_context.get("value_chain", "")
        ctx_revenue_model = merged_business_context.get("revenue_model", "")
        
        
        try:
            if self.data_loader:
                self.logger.debug("Data loader found. Starting batched table processing with MAX_CONTEXT_CHARS management...")
                
                document_context_markdown = ""
                strategic_goals = ctx_strategic_goals
                business_context = ""
                business_priorities = []
                strategic_initiative = ""
                value_chain = ""
                revenue_model = ""

                # Now process tables in batches with model-specific context limits from TECHNICAL_CONTEXT
                use_case_gen_context_limit = get_max_context_chars("English", "BASE_USE_CASE_GEN_PROMPT")
                safe_context_limit = get_safe_context_limit("English", buffer_percent=0.9, prompt_name="BASE_USE_CASE_GEN_PROMPT")
                batch_num = 1
                accumulated_columns = []
                accumulated_schema_size = 0
                batches_to_process = []  # List of (batch_num, column_details) tuples
                
                base_prompt_template = PROMPT_TEMPLATES.get("BASE_USE_CASE_GEN_PROMPT", "")
                _quality_gate_size = len(QUALITY_GATE_GENERATION_BLOCKS.get(self.use_cases_quality, QUALITY_GATE_GOOD))
                base_template_size = len(base_prompt_template) + len(document_context_markdown) + _quality_gate_size
                base_prompt_size = base_template_size + 2000
                self.logger.debug(f"Base prompt template size: {base_template_size} chars, context limit: {use_case_gen_context_limit}")
                
                # CRITICAL FIX: Keep pulling tables until we fill the model's context limit
                self.logger.info("Collecting batches for parallel processing (MAXIMIZING context utilization)...")
                while True:
                    # Keep pulling table batches until we hit the context limit
                    while True:
                        batch_columns = self.data_loader.getNextTables(self.scan_parallelism)
                        
                        if batch_columns is None:
                            # No more tables available
                            break

                        batch_columns = self._augment_columns_with_foreign_keys(batch_columns)
                        
                        batch_schema_size = self._estimate_schema_markdown_size(batch_columns)
                        estimated_prompt_size = base_prompt_size + accumulated_schema_size + batch_schema_size
                        
                        self.logger.debug(f"Batch {batch_num}: Got {len(batch_columns)} columns. "
                                       f"Total accumulated: {len(accumulated_columns) + len(batch_columns)} columns. "
                                       f"Estimated prompt size: {estimated_prompt_size}/{safe_context_limit} chars")
                        
                        if estimated_prompt_size > safe_context_limit:
                            if not accumulated_columns:
                                self.logger.warning(f"Initial table pull exceeds context limit ({estimated_prompt_size} chars). Splitting into sub-batches.")
                                split_batches = self._split_columns_to_fit_context(batch_columns, base_prompt_size, safe_context_limit)
                                for idx, split_columns in enumerate(split_batches, start=1):
                                    table_count = len({(c[0], c[1], c[2]) for c in split_columns})
                                    self.logger.warning(f"   Sub-batch {idx}: {table_count} tables ({len(split_columns)} columns)")
                                    batches_to_process.append((batch_num, split_columns))
                                    batch_num += 1
                                accumulated_columns = []
                                accumulated_schema_size = 0
                                break
                            else:
                                # Would exceed limit - save current accumulator as a batch
                                self.logger.info(f"Context limit reached. Saving batch {batch_num} with {len(accumulated_columns)} columns ({base_prompt_size + accumulated_schema_size} chars)")
                                batches_to_process.append((batch_num, accumulated_columns))
                                batch_num += 1
                                
                                # Start new accumulator with the current batch
                                accumulated_columns = batch_columns
                                accumulated_schema_size = batch_schema_size
                                break
                        else:
                            # Fits - keep accumulating and PULL MORE TABLES
                            accumulated_columns.extend(batch_columns)
                            accumulated_schema_size += batch_schema_size
                            # Continue inner loop to pull more tables
                    
                    # Check if we're done (no more tables)
                    if batch_columns is None:
                        # Add any remaining accumulated columns to batches
                        if accumulated_columns:
                            self.logger.info(f"Adding final batch {batch_num} with {len(accumulated_columns)} columns")
                            batches_to_process.append((batch_num, accumulated_columns))
                        break
                
                if not batches_to_process:
                    self.logger.warning("No batches to process. Skipping all generation.")
                    return
                
                total_tables = len({(c, s, t) for _, cols in batches_to_process for (c, s, t, _, _, _) in cols})
                tables_per_call = self._determine_tables_per_call(total_tables)
                if tables_per_call > 0:
                    adjusted_batches = []
                    next_batch_num = 1
                    for _, batch_columns in batches_to_process:
                        grouped = self._split_by_table_limit(batch_columns, tables_per_call)
                        for group in grouped:
                            estimated_prompt_size = base_prompt_size + self._estimate_schema_markdown_size(group)
                            if estimated_prompt_size > safe_context_limit:
                                singles = self._split_by_table_limit(group, 1)
                                for single in singles:
                                    adjusted_batches.append((next_batch_num, single))
                                    next_batch_num += 1
                            else:
                                adjusted_batches.append((next_batch_num, group))
                                next_batch_num += 1
                    batches_to_process = adjusted_batches
                    self.logger.info(f"Table-based batching: {tables_per_call} tables per call, {len(batches_to_process)} batches")
                

                # === TABLE JOIN MERGE: Combine batches containing user-specified joined tables ===
                table_joins = self.parsed_generation_instructions.get("table_joins", [])
                if table_joins:
                    self.logger.info(f"🔗 Processing {len(table_joins)} table join instruction(s)...")
                    for join_spec in table_joins:
                        left_tbl = join_spec.get("left_table", "").lower().strip()
                        right_tbl = join_spec.get("right_table", "").lower().strip()
                        if not left_tbl or not right_tbl:
                            continue
                        left_batch_idx = None
                        right_batch_idx = None
                        for bidx, (bnum, bcols) in enumerate(batches_to_process):
                            batch_fqns = {f"{c[0]}.{c[1]}.{c[2]}".lower() for c in bcols}
                            if left_tbl in batch_fqns:
                                left_batch_idx = bidx
                            if right_tbl in batch_fqns:
                                right_batch_idx = bidx
                        if left_batch_idx is not None and right_batch_idx is not None and left_batch_idx != right_batch_idx:
                            left_bnum, left_cols = batches_to_process[left_batch_idx]
                            right_bnum, right_cols = batches_to_process[right_batch_idx]
                            merged_cols = left_cols + [c for c in right_cols if (c[0], c[1], c[2]) not in {(lc[0], lc[1], lc[2]) for lc in left_cols}]
                            self.logger.info(f"  🔗 Merged batch {left_bnum} ({left_tbl}) + batch {right_bnum} ({right_tbl}) into single batch")
                            batches_to_process[left_batch_idx] = (left_bnum, merged_cols)
                            batches_to_process.pop(right_batch_idx)
                            for k in range(len(batches_to_process)):
                                batches_to_process[k] = (k + 1, batches_to_process[k][1])
                        elif left_batch_idx is not None and right_batch_idx is not None:
                            self.logger.info(f"  ✅ Tables {left_tbl} and {right_tbl} already in the same batch")
                        else:
                            missing = []
                            if left_batch_idx is None:
                                missing.append(left_tbl)
                            if right_batch_idx is None:
                                missing.append(right_tbl)
                            self.logger.warning(f"  ⚠️ Could not find table(s) {missing} in any batch for join")


                # === POST-MERGE CONTEXT SIZE VALIDATION ===
                if table_joins:
                    overflow_batches = []
                    for bidx, (bnum, bcols) in enumerate(batches_to_process):
                        merged_size = base_prompt_size + self._estimate_schema_markdown_size(bcols)
                        if merged_size > safe_context_limit:
                            overflow_batches.append(bidx)
                    if overflow_batches:
                        self.logger.warning(f"  ⚠️ {len(overflow_batches)} merged batch(es) exceed context limit -- splitting into single-table sub-batches")
                        new_batches = []
                        next_num = 1
                        for bidx, (bnum, bcols) in enumerate(batches_to_process):
                            if bidx in overflow_batches:
                                singles = self._split_by_table_limit(bcols, 1)
                                for single in singles:
                                    new_batches.append((next_num, single))
                                    next_num += 1
                            else:
                                new_batches.append((next_num, bcols))
                                next_num += 1
                        batches_to_process = new_batches
                        self.logger.info(f"  📦 Post-merge re-split: {len(batches_to_process)} batches")

                # === NEW: Filter business vs technical tables ===
                try:
                    self._emit_pipeline_status(
                        stage_name="Data Preparation", step_name="Filter Business Tables",
                        sub_step_name="Started", status="started",
                        message="Classifying tables as BUSINESS vs TECHNICAL"
                    )
                except Exception:
                    pass
                self.logger.info("🔍 Filtering tables into BUSINESS vs TECHNICAL categories...")
                log_print(f"\n{'='*80}")
                log_print(f"🔍 FILTERING TABLES: Business Data vs Technical/Metadata")
                log_print(f"{'='*80}\n")
                
                # Collect all columns from all batches for filtering
                all_batch_columns = []
                for batch_num, batch_columns in batches_to_process:
                    all_batch_columns.extend(batch_columns)

                # Early FK inference (pre-UC-gen): populate foreign_key_graph with
                # column-name-inferred FKs BEFORE UC batches run their declared-FK lookup.
                # Without this, the per-batch fk_relationships_text is always "None" when
                # UC has no explicit FKs, and LLM misses multi-table UC opportunities.
                try:
                    if self.data_loader and all_batch_columns:
                        _pre_inferred = self.data_loader.merge_inferred_foreign_keys(all_batch_columns)
                        if _pre_inferred:
                            self.logger.info(f"🔗 Early FK inference: {_pre_inferred} relationships available for UC generation")
                except Exception as _pre_fk_err:
                    self.logger.debug(f"Early FK inference skipped: {_pre_fk_err}")
                
                if self.auto_parallelism:
                    total_tables = len({(c, s, t) for (c, s, t, _, _, _) in all_batch_columns})
                    total_schema_chars = self._estimate_schema_markdown_size(all_batch_columns)
                    (recommended, tables_per_batch, est_batches, avg_table_chars, max_by_memory) = self._calculate_dynamic_parallelism(
                        total_tables,
                        total_schema_chars,
                        safe_context_limit,
                        base_prompt_size
                    )
                    self._batch_parallelism_override = recommended
                    if recommended < self.max_parallelism:
                        self.logger.info(
                            f"Batch-specific parallelism set to {recommended} (est_batches={est_batches}), "
                            f"global max_parallelism stays at {self.max_parallelism} (memory_cap={max_by_memory})"
                        )
                    else:
                        self.max_parallelism = recommended
                        self.logger.info(f"Dynamic parallelism set to {self.max_parallelism} (tables={total_tables}, avg_table_chars={avg_table_chars}, tables_per_batch={tables_per_batch}, est_batches={est_batches}, memory_cap={max_by_memory})")
                    log_print(f"✅ Dynamic parallelism: {self.max_parallelism} (tables={total_tables}, est_batches={est_batches})")
                
                # Get industry from business context if available
                industry = ""
                if 'business_context' in locals() and business_context:
                    # Extract industry info (simplified - you could enhance this)
                    industry = business_context.split('\n')[0] if business_context else ""
                
                # Filter tables
                (business_details, technical_details, business_tables, technical_tables, business_scores, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set) = self._filter_business_tables(
                    all_batch_columns,
                    business_context=business_context if 'business_context' in locals() else "",
                    industry=industry,
                    exclusion_strategy=self.technical_exclusion_strategy
                )
                
                # === MEMORY OPTIMIZATION: Clear all_batch_columns after filtering ===
                del all_batch_columns
                gc.collect()
                self.logger.debug("🧹 Cleared all_batch_columns from memory")
                
                self._business_column_details_global = business_details
                self.global_table_names = {f"{c}.{s}.{t}" for (c, s, t, _, _, _) in business_details}
                self._known_hallucinated_tables = set()
                
                self.business_scores = business_scores
                self.data_category_map = data_category_map
                
                if not business_details:
                    self.logger.warning("No business tables found after filtering. All tables were classified as technical/metadata.")
                    log_print("⚠️ No business tables found. All tables appear to be technical/metadata.", level="WARNING")
                    return
                
                self.logger.info(f"✅ Filtering complete: Proceeding with {len(business_tables)} business tables, "
                               f"excluding {len(technical_tables)} technical/metadata tables, "
                               f"including {len(reference_tables_set)} reference tables as joinable dims")
                
                master_details = []
                transactional_details = []
                for detail in business_details:
                    (catalog, schema, table, _, _, _) = detail
                    fqtn = f"{catalog}.{schema}.{table}"
                    if fqtn in transactional_tables_set:
                        transactional_details.append(detail)
                    else:
                        master_details.append(detail)
                
                _rt = TECHNICAL_CONTEXT["runtime"]
                _table_election_threshold = _rt["table_election_threshold"]
                _table_election_tx_min = _rt["table_election_transactional_tables_min"]
                total_business_tables = len(business_tables)
                tx_count = len(transactional_tables_set)
                
                elected_master_details = master_details
                elected_transactional_details = transactional_details
                
                if self.table_election_mode == "Let Inspire Decides":
                    if total_business_tables > _table_election_threshold:
                        if tx_count >= _table_election_tx_min:
                            elected_master_details = []
                            log_print(f"🗳️ Table Election: {total_business_tables} tables > threshold ({_table_election_threshold}), "
                                      f"using TRANSACTIONAL tables only ({tx_count} tables)")
                            self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} > threshold={_table_election_threshold}, "
                                           f"tx={tx_count} >= min={_table_election_tx_min} → transactional only")
                        else:
                            log_print(f"🗳️ Table Election: {total_business_tables} tables > threshold ({_table_election_threshold}), "
                                      f"but transactional tables ({tx_count}) < min ({_table_election_tx_min}), "
                                      f"including MASTER tables as well")
                            self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} > threshold={_table_election_threshold}, "
                                           f"tx={tx_count} < min={_table_election_tx_min} → transactional + master")
                    else:
                        log_print(f"🗳️ Table Election: {total_business_tables} tables <= threshold ({_table_election_threshold}), "
                                  f"using ALL business tables (master + transactional)")
                        self.logger.info(f"Table Election [Let Inspire Decides]: total={total_business_tables} <= threshold={_table_election_threshold} → all tables")
                elif self.table_election_mode == "Transactional Only":
                    elected_master_details = []
                    if not elected_transactional_details:
                        self.logger.warning("Table Election [Transactional Only]: No transactional tables found. Falling back to all business tables.")
                        log_print("⚠️ Table Election: No transactional tables found. Falling back to all business tables.", level="WARNING")
                        elected_master_details = master_details
                    else:
                        log_print(f"🗳️ Table Election: TRANSACTIONAL ONLY mode — using {tx_count} transactional tables, "
                                  f"excluding {len(master_tables_set)} master tables")
                        self.logger.info(f"Table Election [Transactional Only]: tx={tx_count}, excluded master={len(master_tables_set)}")
                else:
                    log_print(f"🗳️ Table Election: ALL TABLES mode — using all {total_business_tables} business tables")
                    self.logger.info(f"Table Election [All Tables]: using all {total_business_tables} business tables")
                
                master_details = elected_master_details
                transactional_details = elected_transactional_details
                
                master_tables_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in master_details}) if master_details else 0
                tables_per_call = self._determine_tables_per_call(master_tables_count)
                
                adjusted_batches = []
                next_batch_num = 1
                
                if master_details:
                    grouped_master = self._split_by_table_limit(master_details, tables_per_call)
                    for group in grouped_master:
                        estimated_prompt_size = base_prompt_size + self._estimate_schema_markdown_size(group)
                        if estimated_prompt_size > safe_context_limit:
                            singles = self._split_by_table_limit(group, 1)
                            for single in singles:
                                adjusted_batches.append((next_batch_num, single))
                                next_batch_num += 1
                        else:
                            adjusted_batches.append((next_batch_num, group))
                            next_batch_num += 1
                
                if transactional_details:
                    grouped_tx = self._split_by_table_limit(transactional_details, 1)
                    for group in grouped_tx:
                        adjusted_batches.append((next_batch_num, group))
                        next_batch_num += 1
                
                augmented_batches = []
                for batch_num, batch_columns in adjusted_batches:
                    augmented = self._augment_columns_with_related_tables(batch_columns)
                    augmented_batches.append((batch_num, augmented))
                batches_to_process = augmented_batches
                elected_master_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in master_details}) if master_details else 0
                elected_tx_count = len({f"{c}.{s}.{t}" for (c, s, t, _, _, _) in transactional_details}) if transactional_details else 0
                elected_total = elected_master_count + elected_tx_count
                log_print(f"✅ Business tables (discovered): {len(business_tables)}")
                log_print(f"   📊 Master Data tables: {len(master_tables_set)}")
                log_print(f"   📈 Transactional tables: {len(transactional_tables_set)}")
                log_print(f"🗳️ Elected for generation: {elected_total} (master={elected_master_count}, transactional={elected_tx_count}) [{self.table_election_mode}]")
                log_print(f"🟡 Reference tables (excluded): {len(reference_tables_set)}")
                log_print(f"❌ Technical tables (excluded): {len(technical_tables)}")
                log_print(f"{'='*80}\n")

                # Update honesty tracking
                self.processing_honesty['total_tables_discovered'] = len(business_tables) + len(technical_tables) + len(reference_tables_set)
                self.processing_honesty['total_tables_processed'] = elected_total
                self.processing_honesty['total_batches_created'] = len(batches_to_process)

                # Process each batch once (deduplication handles redundancy)
                # MEMORY OPTIMIZATION: Use file-based storage instead of keeping everything in memory
                # ADAPTIVE PARALLELISM: Calculate based on batches, tables and columns
                total_batch_columns = sum(len(cols) for _, cols in batches_to_process)
                avg_prompt_chars = total_batch_columns * 100  # Estimate ~100 chars per column
                
                batch_parallelism, reason = calculate_adaptive_parallelism(
                    "use_case_generation", self.max_parallelism,
                    num_items=len(batches_to_process),
                    total_columns=total_batch_columns,
                    avg_prompt_chars=avg_prompt_chars,
                    is_llm_operation=True, logger=self.logger,
                    estimated_mb_per_worker=10.0
                )
                
                log_print(f"\n{'='*80}")
                log_print(f"🔄 USE CASE GENERATION: SERIAL ENSEMBLE (2 PASSES)")
                log_print(f"{'='*80}")
                log_print(f"📋 PASS 1: Generate initial use cases from {len(batches_to_process)} batch(es)")
                log_print(f"📋 PASS 2: Generate NEW use cases not in PASS 1 (with feedback)")
                log_print("💾 Using file-based intermediate storage to prevent memory explosion")
                log_print(f"{'='*80}\n")
                
                self.storage_manager.initialize()
                
                # === PASS 1: Generate initial use cases (parallel within pass) ===
                self.logger.info("🔄 PASS 1: Generating initial use cases...")
                log_print(f"\n{'='*60}")
                log_print(f"🔄 PASS 1: Initial Use Case Generation")
                log_print(f"{'='*60}")
                
                _bp_timeout = TECHNICAL_CONTEXT["runtime"]["batch_processing_timeout"]
                total_submissions = len(batches_to_process)
                total_timeout = (total_submissions * _bp_timeout) // max(1, self.max_parallelism) + 600
                
                _p1_bc = business_context if 'business_context' in locals() else ""
                _p1_bp = business_priorities if 'business_priorities' in locals() else ""
                _p1_si = strategic_initiative if 'strategic_initiative' in locals() else ""
                _p1_vc = value_chain if 'value_chain' in locals() else ""
                _p1_rm = revenue_model if 'revenue_model' in locals() else ""
                _p1_max_attempts = TECHNICAL_CONTEXT["runtime"]["batch_processing_max_attempts"]
                
                _pass1_temp = TECHNICAL_CONTEXT["runtime"].get("pass1_temperature", 0.25)
                _pass2_temp = TECHNICAL_CONTEXT["runtime"].get("pass2_temperature", 0.5)
                self.logger.info(f"🌡️ Temperature config: Pass 1={_pass1_temp}, Pass 2={_pass2_temp}")
                log_print(f"🌡️ LLM Temperature: Pass 1={_pass1_temp} (focused), Pass 2={_pass2_temp} (table-grounded gap fill)")

                def _pass1_worker(batch_num_inner, column_details_inner):
                    unique_batch_id = f"P1_{batch_num_inner}"
                    _n_cols = len(column_details_inner) if column_details_inner else 0
                    _n_tables = len({(c[0], c[1], c[2]) for c in column_details_inner}) if column_details_inner else 0
                    import time as _t
                    _t_start = _t.time()
                    try:
                        self._emit_pipeline_status(
                            stage_name="Use Cases Generation",
                            step_name=f"UC Generation Batch {batch_num_inner}",
                            sub_step_name="Started",
                            status="started",
                            message=f"Pass 1 batch {batch_num_inner}: {_n_tables} tables / {_n_cols} columns -> LLM"
                        )
                    except Exception:
                        pass
                    use_cases = self._process_batch_with_retry(
                        column_details_inner,
                        unique_batch_id,
                        document_context_markdown,
                        strategic_goals,
                        _p1_bc, _p1_bp, _p1_si, _p1_vc, _p1_rm,
                        _p1_max_attempts,
                        "",
                        temperature_override=_pass1_temp
                    )
                    if use_cases:
                        self.storage_manager.save_batch(unique_batch_id, use_cases)
                    try:
                        _elapsed = _t.time() - _t_start
                        self._emit_pipeline_status(
                            stage_name="Use Cases Generation",
                            step_name=f"UC Generation Batch {batch_num_inner}",
                            sub_step_name="Complete",
                            status="ended_success" if use_cases else "ended_warning",
                            message=f"Pass 1 batch {batch_num_inner}: emitted {len(use_cases) if use_cases else 0} UCs in {_elapsed:.1f}s",
                            result_json={"batch_num": batch_num_inner, "n_tables": _n_tables,
                                         "n_columns": _n_cols, "n_use_cases": len(use_cases) if use_cases else 0,
                                         "elapsed_sec": round(_elapsed, 1)}
                        )
                    except Exception:
                        pass
                    return (unique_batch_id, use_cases)
                
                tasks = [(_pass1_worker, (bn, cd)) for bn, cd in batches_to_process]
                for bn, _ in batches_to_process:
                    self.logger.info(f"✓ [PASS 1] Submitted batch {bn}")
                
                pass1_results = ParallelExecutor.execute_parallel(
                    tasks=tasks,
                    max_workers=batch_parallelism,
                    task_name="Pass1Batch",
                    logger=self.logger,
                    timeout_per_task=_bp_timeout,
                    total_timeout=total_timeout,
                    thread_name_prefix="Pass1Batch",
                    return_exceptions=True
                )
                
                batches_completed = 0
                for result in pass1_results:
                    if isinstance(result, Exception):
                        self.logger.error(f"❌ [PASS 1] Batch failed: {get_clean_error_message(result)}")
                        continue
                    unique_batch_id, use_cases = result
                    if use_cases:
                        batches_completed += 1
                        self.logger.info(f"✓ [PASS 1] Batch {unique_batch_id}: {len(use_cases)} use cases ({batches_completed}/{total_submissions})")
                        log_print(f"✓ [PASS 1] Batch complete ({batches_completed}/{total_submissions})")
                
                # === MEMORY OPTIMIZATION: Save PASS 1 results to disk immediately ===
                # Count use cases and save IDs without loading all into memory
                pass1_count = self.storage_manager.get_total_count()
                self.logger.info(f"✅ PASS 1 complete: Generated {pass1_count} use cases")
                log_print(f"✅ PASS 1 complete: {pass1_count} use cases generated")
                
                # Save PASS 1 IDs to disk for later comparison (memory-efficient)
                pass1_ids = []
                for batch in self.storage_manager.iter_batches():
                    for uc in batch:
                        pass1_ids.append(uc.get('No', ''))
                self.storage_manager.save_pass1_ids(pass1_ids)
                del pass1_ids  # Free memory immediately
                
                # Force garbage collection after PASS 1
                gc.collect()
                self.logger.debug("🧹 Memory cleanup after PASS 1")
                
                # === PASS 2: ALWAYS run to find gap use cases with higher temperature (creative diversity pass) ===
                transactional_batches = []
                seen_tx_fqtns = set()
                unique_tx_columns_by_fqtn = {}
                for _, column_details in batches_to_process:
                    for col in column_details:
                        fqtn = f"{col[0]}.{col[1]}.{col[2]}"
                        if fqtn not in seen_tx_fqtns:
                            if fqtn not in unique_tx_columns_by_fqtn:
                                unique_tx_columns_by_fqtn[fqtn] = []
                            unique_tx_columns_by_fqtn[fqtn].append(col)
                    for fqtn in unique_tx_columns_by_fqtn:
                        seen_tx_fqtns.add(fqtn)
                p2_batch_num = 1
                for fqtn, tx_cols in unique_tx_columns_by_fqtn.items():
                    augmented_tx = self._augment_columns_with_related_tables(tx_cols)
                    transactional_batches.append((p2_batch_num, augmented_tx))
                    p2_batch_num += 1
                if len(transactional_batches) > 0:
                    self.logger.info(
                        f"PASS 2 batching: {len(unique_tx_columns_by_fqtn)} unique tables "
                        f"→ {len(transactional_batches)} batches (diversity pass with temp={_pass2_temp})"
                    )
                del seen_tx_fqtns, unique_tx_columns_by_fqtn
                
                if pass1_count > 0 and transactional_batches:
                    self.logger.info("🔄 PASS 2: Generating NEW use cases from TRANSACTIONAL TABLES (with PASS 1 feedback)...")
                    log_print(f"\n{'='*60}")
                    log_print(f"🔄 PASS 2: Ensemble - TRANSACTIONAL TABLES ONLY")
                    log_print(f"{'='*60}")
                    log_print(f"📋 Feedback: {pass1_count} use cases from PASS 1")
                    log_print(f"📊 Transactional batches: {len(transactional_batches)} (focusing on event/transaction data)")
                    log_print(f"🎯 Goal: Find NEW use cases from transactional data NOT covered in PASS 1")
                    
                    # === MEMORY OPTIMIZATION: Build feedback iteratively and save to disk ===
                    feedback_lines = ["**🚀 PASS 2: YOUR MISSION IS TO FIND EVEN MORE VALUE! 🚀**\n"]
                    feedback_lines.append("Pass 1 generated some use cases, but there is MUCH MORE VALUE to extract from the transactional data!")
                    feedback_lines.append("Your job is to find DIFFERENT, COMPLEMENTARY use cases that Pass 1 missed.\n")
                    feedback_lines.append("**Already generated (reference only - avoid exact duplicates, but EXPLORE VARIATIONS):**")
                    feedback_lines.append("| No | Name | Tables Involved |")
                    feedback_lines.append("|---|---|---|")
                    
                    # Use memory-efficient iterator (doesn't load all at once)
                    feedback_count = 0
                    for idx, name, tables in self.storage_manager.iter_pass1_use_cases_for_feedback(limit=200):
                        feedback_lines.append(f"| {idx} | {name} | {tables} |")
                        feedback_count = idx
                    
                    if pass1_count > 200:
                        feedback_lines.append(f"\n*... and {pass1_count - 200} more use cases (not shown)*")
                    
                    feedback_lines.append("\n**⚠️ ALREADY-CLAIMED CORE BUSINESS PROBLEMS (normalized — ANY use case with these core phrases is a DUPLICATE):**")
                    feedback_lines.append("The following are the core phrases extracted from Pass 1 use cases by stripping the 'with...' activation suffix.")
                    feedback_lines.append("If your new use case addresses the SAME core business problem as ANY of these phrases, it is a DUPLICATE — DO NOT GENERATE IT.")
                    feedback_lines.append("")
                    import re as _re
                    seen_cores = set()
                    for _, name, _ in self.storage_manager.iter_pass1_use_cases_for_feedback(limit=200):
                        core = _re.sub(r'\s+with\s+.*$', '', name, flags=_re.IGNORECASE).strip()
                        if core and core not in seen_cores:
                            seen_cores.add(core)
                    for core_phrase in sorted(seen_cores):
                        feedback_lines.append(f"- {core_phrase}")
                    feedback_lines.append("")
                    
                    feedback_lines.append("\n**🔥 PASS 2 MISSION: FIND HIGH-VALUE USE CASES THAT PASS 1 MISSED 🔥**")
                    feedback_lines.append("")
                    feedback_lines.append("**VALUE-FOCUSED EXPLORATION:**")
                    feedback_lines.append("Pass 1 covered some ground, but transactional data often hides MASSIVE business value.")
                    feedback_lines.append("Your job is to find HIGH-ROI use cases that Pass 1 missed - NOT to generate filler.")
                    feedback_lines.append("")
                    feedback_lines.append("**WHAT TO LOOK FOR (only if they deliver REAL business value):**")
                    feedback_lines.append("- 💰 **Revenue opportunities**: Patterns that could increase sales, reduce churn, optimize pricing")
                    feedback_lines.append("- 🛡️ **Risk signals**: Fraud patterns, compliance risks, operational anomalies worth preventing")
                    feedback_lines.append("- 📊 **Strategic insights**: Cross-table relationships that reveal hidden business drivers")
                    feedback_lines.append("- ⚡ **Efficiency gains**: Process bottlenecks, resource optimization opportunities")
                    feedback_lines.append("")
                    feedback_lines.append("**🚨 QUALITY RULES (EXTREME — ZERO TOLERANCE — violations cause IMMEDIATE rejection) 🚨**")
                    feedback_lines.append("")
                    feedback_lines.append("**5-LAYER DUPLICATE DETECTION (match on ANY layer = DO NOT GENERATE):**")
                    feedback_lines.append("- Layer 1: Normalize verb (Forecast/Predict→PREDICT, Detect/Identify→DETECT, etc.) + strip 'with...' activation suffix. Core phrase match → DUPLICATE")
                    feedback_lines.append("- Layer 2: Also replace noun synonyms (Revenue=Income=Sales, Churn=Attrition, Cost=Expense, Customer=Client, etc.). Match → DUPLICATE")
                    feedback_lines.append("- Layer 3: Extract entity + metric. Same entity + same metric as ANY Pass 1 use case → DUPLICATE")
                    feedback_lines.append("- Layer 4: Technique swap on SAME QUESTION only → DUPLICATE ('Forecast Churn' vs 'Predict Churn' = duplicate; 'Forecast Revenue' vs 'Detect Revenue Anomalies' = NOT duplicate — different questions)")
                    feedback_lines.append("- Layer 5: Same tables + similar business question → DUPLICATE")
                    feedback_lines.append("")
                    feedback_lines.append("**ALSO CHECK the ALREADY-CLAIMED CORE PHRASES list above — if your normalized core matches ANY entry, it is a DUPLICATE.**")
                    feedback_lines.append("")
                    feedback_lines.append("**ADDITIONAL REJECTION CRITERIA:**")
                    feedback_lines.append("- ❌ STRIP TEST: Remove AI functions mentally. If what remains is a WHERE clause, threshold, GROUP BY, or date check → REJECT (trivial)")
                    feedback_lines.append("- ❌ DELIVERABLE TEST: Does your 'with [...]' suffix describe a business DELIVERABLE (recommendation, action plan, queue, strategy)? If it describes a technical method, data source, or analytical technique → REJECT. Also check: does the action part (before 'with') contain any technical jargon, domain acronyms, or statistical terms? If yes → REJECT.")
                    feedback_lines.append("- ❌ Single-table use cases when FK relationships exist → REJECT unless there's a genuinely unique single-table insight")
                    feedback_lines.append("- ✅ Genuinely DIFFERENT business entities and metrics using DIFFERENT data dimensions ARE encouraged")
                    feedback_lines.append("- ✅ Cross-table joins with 2+ tables unlock the highest value — prioritize these")
                    feedback_lines.append("- ✅ Use UNDER-REPRESENTED techniques (check Pass 1 technique distribution and balance)")
                    feedback_lines.append("")
                    feedback_lines.append("**SELF-CHECK before EACH use case**: (1) 'Would a skeptical CFO fund this?' (2) 'Does this pass ALL 5 duplicate layers against ALL Pass 1 use cases?' (3) 'Does this REQUIRE genuine multi-step analysis, ML modeling, or advanced statistical reasoning — or is it just basic filtering/aggregation?' (4) 'Does my activation suffix describe a business DELIVERABLE (recommendation, queue, strategy, action plan) rather than a technical method?' (5) 'Does my action part contain ANY technical jargon, domain acronyms, or statistical terms? If yes, rewrite in plain business language.' (6) 'Have I chosen the BEST algorithm for this problem — SQL, ML model, AI function, or hybrid?'")
                    
                    # Save feedback to disk and clear from memory
                    self.storage_manager.save_feedback_file(feedback_lines)
                    previous_use_cases_feedback = "\n".join(feedback_lines)
                    del feedback_lines  # Free memory
                    gc.collect()
                    
                    pass2_submissions = len(transactional_batches)
                    
                    _p2_bc = business_context if 'business_context' in locals() else ""
                    _p2_bp = business_priorities if 'business_priorities' in locals() else ""
                    _p2_si = strategic_initiative if 'strategic_initiative' in locals() else ""
                    _p2_vc = value_chain if 'value_chain' in locals() else ""
                    _p2_rm = revenue_model if 'revenue_model' in locals() else ""
                    _p2_max_attempts = TECHNICAL_CONTEXT["runtime"]["batch_processing_max_attempts"]
                    _p2_feedback = previous_use_cases_feedback
                    
                    def _pass2_worker(batch_num_inner, column_details_inner):
                        unique_batch_id = f"P2_{batch_num_inner}"
                        _n_cols2 = len(column_details_inner) if column_details_inner else 0
                        _n_tabs2 = len({(c[0], c[1], c[2]) for c in column_details_inner}) if column_details_inner else 0
                        import time as _t2
                        _t2_start = _t2.time()
                        try:
                            self._emit_pipeline_status(
                                stage_name="Use Cases Generation",
                                step_name=f"UC Generation Batch P2_{batch_num_inner}",
                                sub_step_name="Started", status="started",
                                message=f"Pass 2 batch {batch_num_inner}: {_n_tabs2} tables / {_n_cols2} columns -> LLM"
                            )
                        except Exception:
                            pass
                        use_cases = self._process_batch_with_retry(
                            column_details_inner,
                            unique_batch_id,
                            document_context_markdown,
                            strategic_goals,
                            _p2_bc, _p2_bp, _p2_si, _p2_vc, _p2_rm,
                            _p2_max_attempts,
                            _p2_feedback,
                            temperature_override=_pass2_temp
                        )
                        if use_cases:
                            self.storage_manager.save_batch(unique_batch_id, use_cases)
                        return (unique_batch_id, use_cases)
                    
                    pass2_tasks = [(_pass2_worker, (bn, cd)) for bn, cd in transactional_batches]
                    for bn, _ in transactional_batches:
                        self.logger.info(f"✓ [PASS 2] Submitted transactional batch {bn}")
                    
                    pass2_results = ParallelExecutor.execute_parallel(
                        tasks=pass2_tasks,
                        max_workers=batch_parallelism,
                        task_name="Pass2Batch",
                        logger=self.logger,
                        timeout_per_task=_bp_timeout,
                        total_timeout=total_timeout,
                        thread_name_prefix="Pass2Batch",
                        return_exceptions=True
                    )
                    
                    batches_completed = 0
                    for result in pass2_results:
                        if isinstance(result, Exception):
                            self.logger.error(f"❌ [PASS 2] Batch failed: {get_clean_error_message(result)}")
                            continue
                        unique_batch_id, use_cases = result
                        if use_cases:
                            batches_completed += 1
                            self.logger.info(f"✓ [PASS 2] Batch {unique_batch_id}: {len(use_cases)} NEW use cases ({batches_completed}/{pass2_submissions})")
                            log_print(f"✓ [PASS 2] Batch complete ({batches_completed}/{pass2_submissions})")
                    
                    # === MEMORY OPTIMIZATION: Count PASS 2 results without loading all into memory ===
                    total_after_pass2 = self.storage_manager.get_total_count()
                    pass2_new_count = total_after_pass2 - pass1_count
                    self.logger.info(f"✅ PASS 2 complete: Generated {pass2_new_count} additional NEW use cases from transactional tables")
                    log_print(f"✅ PASS 2 complete: {pass2_new_count} NEW use cases generated")
                    
                    # Clean up PASS 2 variables to free memory
                    del previous_use_cases_feedback
                    del transactional_batches
                    gc.collect()
                    self.logger.debug("🧹 Memory cleanup after PASS 2")
                elif not transactional_batches and not _pass1_already_tx_only:
                    self.logger.info("⚠️ PASS 2 skipped: No transactional tables found for ensemble pass")
                
                log_print(f"\n{'='*60}")
                log_print(f"✅ SERIAL ENSEMBLE COMPLETE")
                log_print(f"{'='*60}")
                
                # Check if any batches were successfully processed
                storage_stats = self.storage_manager.get_stats()
                if storage_stats['num_batches'] == 0:
                    self.logger.warning("No use cases were generated from any batch. Skipping all generation.")
                    self.storage_manager.cleanup()
                    self.ai_agent.cleanup_llm_cache()
                    return
                
                self.logger.info(f"📊 Batch processing complete. Storage stats: {storage_stats['num_batches']} batches, "
                               f"{storage_stats['use_case_count']} use cases, {storage_stats['total_size_mb']:.2f} MB on disk")
                
                # MEMORY OPTIMIZATION: Load use cases from disk for deduplication only when needed
                gc.collect()
                try:
                    import psutil, os as _os
                    _proc = psutil.Process(_os.getpid())
                    _mem_mb = _proc.memory_info().rss / (1024 * 1024)
                    self.logger.info(f"📊 MEMORY before loading use cases: {_mem_mb:.0f} MB (RSS)")
                    log_print(f"📊 Memory before dedup: {_mem_mb:.0f} MB")
                except ImportError:
                    import resource
                    _mem_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
                    self.logger.info(f"📊 MEMORY before loading use cases: {_mem_mb:.0f} MB (maxrss)")
                    _proc = None
                
                self.logger.info("Loading use cases from disk for deduplication...")
                all_use_cases = self.storage_manager.load_all_use_cases()
                
                # Filter out use cases without valid tables (before deduplication)
                # Keep use cases that have either:
                # 1. Valid table references (non-empty and not just catalog/schema prefix)
                # 2. Volume paths (for document data use cases)
                pre_filter_count = len(all_use_cases)
                all_use_cases = [
                    uc for uc in all_use_cases 
                    if (uc.get('Tables Involved', '').strip() and (
                        uc.get('Tables Involved', '').startswith('/Volumes') or 
                        '.' in uc.get('Tables Involved', '')  # Has at least one dot (catalog.schema.table format)
                    ))
                ]
                filtered_count = pre_filter_count - len(all_use_cases)
                if filtered_count > 0:
                    self.logger.warning(f"⚠️ Filtered out {filtered_count} use cases without valid tables before deduplication")

                self.logger.debug(f"Total use cases generated (pre-deduplication): {len(all_use_cases)}")
                unique_use_cases = all_use_cases

                # === GOAL ALIGNMENT HARD FILTER ===
                # Only triggers when the interpreter extracted a user goal (has_goal=True)
                # Removes use cases that do not align with the user's stated goal
                pre_goal_filter_count = len(unique_use_cases)
                unique_use_cases = self._filter_use_cases_by_goal_alignment(unique_use_cases)
                if len(unique_use_cases) < pre_goal_filter_count:
                    self.logger.info(f"🎯 Goal filter removed {pre_goal_filter_count - len(unique_use_cases)} use cases "
                                     f"({len(unique_use_cases)} remaining)")

                
                all_columns_for_sql = []
                _seen_detail_keys = set()
                def _dedupe_add(_detail, _bucket):
                    try:
                        (_c, _s, _t, _col, _, _) = _detail
                    except Exception:
                        return
                    _key = (str(_c or ""), str(_s or ""), str(_t or ""), str(_col or ""))
                    if _key in _seen_detail_keys:
                        return
                    _seen_detail_keys.add(_key)
                    _bucket.append(_detail)
                for batch_num, batch_columns in batches_to_process:
                    for detail in batch_columns:
                        # Keep REFERENCE tables available for joins so downstream UCs
                        # may cite them (e.g., nation, region, category, calendar dims)
                        # without triggering the hallucinated-table check. REFERENCE
                        # tables remain flagged via reference_tables_set for classification.
                        _dedupe_add(detail, all_columns_for_sql)
                # Per north-star rule: classified-out tables must remain joinable and
                # visible in downstream grounding. Merge TECHNICAL and REFERENCE details
                # so quality validators see the full discovered universe instead of
                # flagging legit joins (ship_mode, reason, household_demographics,
                # income_band) as HALLUCINATION.
                try:
                    for _detail in (technical_details or []):
                        _dedupe_add(_detail, all_columns_for_sql)
                    _ref_details_for_merge = getattr(self, "_reference_details_last", None) or []
                    for _detail in _ref_details_for_merge:
                        _dedupe_add(_detail, all_columns_for_sql)
                    try:
                        self.logger.info(
                            f"📘 Schema-for-validation: {len(all_columns_for_sql)} columns "
                            f"({len(_seen_detail_keys)} unique; business + "
                            f"{len(technical_details or [])} technical + "
                            f"{len(_ref_details_for_merge)} reference merged)"
                        )
                    except Exception:
                        pass
                except Exception as _tech_err:
                    try:
                        self.logger.debug(f"Could not merge technical/reference details into all_columns_for_sql: {_tech_err}")
                    except Exception:
                        pass
                
                # === INFER FK RELATIONSHIPS FROM COLUMN NAMING PATTERNS ===
                # This adds relationships like customer_id -> customers.id when no explicit FK exists
                if self.data_loader and all_columns_for_sql:
                    self.logger.info("🔗 Analyzing column names to infer PK-FK relationships...")
                    inferred_count = self.data_loader.merge_inferred_foreign_keys(all_columns_for_sql)
                    if inferred_count > 0:
                        log_print(f"🔗 Inferred {inferred_count} table relationships from column naming patterns")
                
                # === MEMORY OPTIMIZATION: Save schema to disk, clear from memory ===
                self.storage_manager.save_schema_details(all_columns_for_sql, "all_columns_for_sql")
                del batches_to_process
                gc.collect()
                self.logger.debug("🧹 Cleared batches_to_process from memory")
                
                # === NEW: Catch uncovered tables BEFORE clustering/scoring/SQL ===
                self.logger.info("🔍 Checking table coverage before clustering and scoring...")
                catchall_rounds = 3
                for round_idx in range(catchall_rounds):
                    pre_scoring_retry = self._retry_missing_table_coverage(
                        unique_use_cases,
                        all_columns_for_sql,
                        document_context_markdown,
                        strategic_goals,
                        include_business_catchall=True
                    )
                    if not pre_scoring_retry:
                        if round_idx == 0:
                            self.logger.info("✅ All tables covered by initial use cases")
                        else:
                            self.logger.info(f"✅ All tables covered after catch-all pass {round_idx}")
                        break
                    self.logger.info(f"✅ Generated {len(pre_scoring_retry)} additional use cases for uncovered tables (pass {round_idx + 1})")
                    pre_scoring_retry = [
                        uc for uc in pre_scoring_retry
                        if uc.get('Tables Involved', '').strip() and not uc.get('Tables Involved', '').startswith('/Volumes')
                    ]
                    self.logger.info(f"✅ After filtering, {len(pre_scoring_retry)} pre-scoring retry use cases have valid tables")
                    if not pre_scoring_retry:
                        continue
                    unique_use_cases.extend(pre_scoring_retry)
                    self.logger.info(f"📊 Total use cases after pre-scoring coverage pass {round_idx + 1}: {len(unique_use_cases)}")
                else:
                    self.logger.info("⚠️ Catch-all reached maximum retries")

                del all_columns_for_sql
                gc.collect()
                self.logger.debug("🧹 Freed all_columns_for_sql from memory (saved to disk)")
                
                # === PHASE 1: Cluster domains/subdomains FIRST (before scoring) ===
                self.logger.info(f"📊 Clustering {len(unique_use_cases)} deduplicated use cases into domains and subdomains...")

                # === GOAL ALIGNMENT FILTER (SECOND PASS) ===
                # Re-run after table coverage retry to filter any newly-generated non-aligned use cases
                _pre_2nd_goal = len(unique_use_cases)
                unique_use_cases = self._filter_use_cases_by_goal_alignment(unique_use_cases)
                if len(unique_use_cases) < _pre_2nd_goal:
                    self.logger.info(f"🎯 Goal filter (2nd pass) removed {_pre_2nd_goal - len(unique_use_cases)} additional use cases")

                clustered_use_cases = self._cluster_domains_and_subdomains(unique_use_cases, "English")
                
                del unique_use_cases
                del all_use_cases
                gc.collect()
                if _proc:
                    _mem_mb = _proc.memory_info().rss / (1024 * 1024)
                    self.logger.info(f"📊 MEMORY after clustering: {_mem_mb:.0f} MB (RSS)")
                
                # === PHASE 2: Score per domain in parallel FIRST (before deduplication) ===
                self.logger.info(f"🔄 PHASE 1: Scoring use cases per domain in parallel...")
                unique_use_cases_scored = self._score_per_domain_parallel(
                    clustered_use_cases, # Use clustered cases here (not deduped yet)
                    business_context=ctx_business_context,
                    strategic_goals=ctx_strategic_goals,
                    business_priorities=ctx_business_priorities,
                    strategic_initiative=ctx_strategic_initiative,
                    value_chain=ctx_value_chain,
                    revenue_model=ctx_revenue_model
                )
                self.logger.info("✅ Phase 1 complete: All use cases scored")
                
                del clustered_use_cases
                _force_gc(self.logger, "after scoring, before dedup")
                
                # === PHASE 3: Intelligent Deduplication (Global first, per-domain fallback) ===
                self.logger.info("🔄 Starting INTELLIGENT deduplication (global attempt → per-domain fallback)...")
                final_deduplicated_use_cases = self._deduplicate_use_cases(unique_use_cases_scored)
                
                del unique_use_cases_scored
                _force_gc(self.logger, "after dedup")
                
                final_consolidated_use_cases = final_deduplicated_use_cases
                del final_deduplicated_use_cases
                
                # Re-number use case IDs to match the domain-based notebook prefixes (C01, C02, etc.)
                self.logger.debug("Re-numbering use case IDs to match domain-based notebook prefixes...")
                grouped_by_domain = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)
                
                # Sort domains by impact (most impactful first)
                domain_impact_scores = {domain: self._calculate_domain_impact_score(use_cases) 
                                       for domain, use_cases in grouped_by_domain.items()}
                sorted_domain_names = sorted(grouped_by_domain.keys(), 
                                            key=lambda d: domain_impact_scores[d], 
                                            reverse=True)
                
                renumbered_use_cases = []
                for domain_idx, domain_name in enumerate(sorted_domain_names):
                    domain_use_cases = grouped_by_domain[domain_name]
                    domain_prefix = f"N{domain_idx+1:02d}"
                    
                    for uc_idx, uc in enumerate(domain_use_cases, start=1):
                        new_id = f"{domain_prefix}-U{uc_idx:02d}"
                        uc['No'] = new_id
                        renumbered_use_cases.append(uc)
                
                self.logger.debug(f"Re-numbered {len(renumbered_use_cases)} use cases to match notebook prefixes.")
                del grouped_by_domain, domain_impact_scores, sorted_domain_names
                
                # Set final_consolidated_use_cases before quality scoring
                final_consolidated_use_cases = renumbered_use_cases

                # === PHASE G + Q GATES: quality checks, grounding, canonical-sig dedup ===
                try:
                    self._emit_pipeline_status(
                        stage_name="Quality Gates", step_name="G+Q Integrity Checks",
                        sub_step_name="Started", status="started",
                        message="Running Phase G quality gates + Phase Q integrity gates (grounding, dedup, relevance, recall)"
                    )
                except Exception:
                    pass
                try:
                    _q_schema = self.storage_manager.load_schema_details("all_columns_for_sql")
                    if _q_schema and final_consolidated_use_cases:
                        before_n = len(final_consolidated_use_cases)
                        # Phase G first (cheap deterministic filters)
                        final_consolidated_use_cases = self._phaseG_run_quality_gates(
                            final_consolidated_use_cases, _q_schema
                        )
                        # Phase Q (grounding + dedup) over the G-filtered portfolio
                        final_consolidated_use_cases = self._phaseQ_run_integrity_gates(
                            final_consolidated_use_cases, _q_schema
                        )
                        self.logger.info(f"G+Q gates complete: {before_n} -> {len(final_consolidated_use_cases)} use cases")
                        try:
                            self._emit_pipeline_status(
                                stage_name="Quality Gates", step_name="G+Q Integrity Checks",
                                sub_step_name="Complete", status="ended_success",
                                message=f"Quality gates complete: {before_n} -> {len(final_consolidated_use_cases)} use cases kept",
                                result_json={"before": before_n, "after": len(final_consolidated_use_cases)}
                            )
                        except Exception:
                            pass
                        del _q_schema
                        import gc as _gc
                        _gc.collect()
                except Exception as _gq_err:
                    self.logger.warning(f"Phase G+Q gates failed non-fatally: {get_clean_error_message(_gq_err)}")

                # === PHASE 4.5: VALUE+QUALITY SCORING ===
                _schema_for_quality = self.storage_manager.load_schema_details("all_columns_for_sql")
                if renumbered_use_cases and _schema_for_quality:
                    _ctx_bc = getattr(self, 'merged_business_context', {}).get('business_context', business_context or '')
                    _ctx_ind = getattr(self, 'merged_business_context', {}).get('industry', '')
                    
                    _use_combined = TECHNICAL_CONTEXT["runtime"].get("use_combined_scoring", True)
                    _cache_tag = "combined_scored" if _use_combined else "quality_scored"
                    
                    _cached_scored = self.storage_manager.load_scored_use_cases(_cache_tag)
                    if _cached_scored and len(_cached_scored) == len(renumbered_use_cases):
                        self.logger.info(f"Loaded {len(_cached_scored)} scored use cases from disk cache ({_cache_tag})")
                        final_consolidated_use_cases = _cached_scored
                        del _schema_for_quality
                    elif _use_combined:
                        final_consolidated_use_cases = self._score_combined_value_and_quality(
                            use_cases=renumbered_use_cases,
                            full_schema_details=_schema_for_quality,
                            business_context=_ctx_bc,
                            industry=_ctx_ind,
                            strategic_goals=ctx_strategic_goals,
                            business_priorities=ctx_business_priorities,
                            strategic_initiative=ctx_strategic_initiative,
                            value_chain=ctx_value_chain,
                            revenue_model=ctx_revenue_model
                        )
                        self.storage_manager.save_scored_use_cases(final_consolidated_use_cases, _cache_tag)
                        del _schema_for_quality
                    else:
                        final_consolidated_use_cases = self._score_use_case_data_quality(
                            use_cases=renumbered_use_cases,
                            full_schema_details=_schema_for_quality,
                            business_context=_ctx_bc,
                            industry=_ctx_ind
                        )
                        self.storage_manager.save_scored_use_cases(final_consolidated_use_cases, _cache_tag)
                        del _schema_for_quality
                    gc.collect()
                    
                    quality_distribution = {}
                    for uc in final_consolidated_use_cases:
                        label = uc.get('Quality', 'Medium')
                        quality_distribution[label] = quality_distribution.get(label, 0) + 1
                    
                    log_print(f"\n📊 DATA QUALITY DISTRIBUTION:")
                    for label in ['Ultra High', 'Very High', 'High', 'Medium', 'Low', 'Very Low', 'Ultra Low']:
                        count = quality_distribution.get(label, 0)
                        if count > 0:
                            log_print(f"   {label}: {count} use cases")
                    
                    # === COMPUTE INSPIRE SCORE (configurable weighted combination) ===
                    _quality_weight = TECHNICAL_CONTEXT["runtime"].get("inspire_score_quality_weight", 0.65)
                    _priority_weight = TECHNICAL_CONTEXT["runtime"].get("inspire_score_priority_weight", 0.35)
                    for uc in final_consolidated_use_cases:
                        raw_quality = float(uc.get('Quality Score', 3.0) or 3.0)
                        quality_normalized = min(10.0, max(0.0, raw_quality * 2.0))
                        raw_priority = float(uc.get('Priority', 5.0) or 5.0)
                        priority_normalized = min(10.0, max(0.0, raw_priority))
                        inspire_score = round(_quality_weight * quality_normalized + _priority_weight * priority_normalized, 2)
                        uc['Inspire Score'] = inspire_score
                    
                    log_print(f"\n📊 INSPIRE SCORES COMPUTED (weights: quality={_quality_weight}, priority={_priority_weight})")
                    self.logger.info(f"✅ Inspire Scores computed for {len(final_consolidated_use_cases)} use cases (quality_weight={_quality_weight}, priority_weight={_priority_weight})")
                    
                    _inspire_scores = [float(uc.get('Inspire Score', 0)) for uc in final_consolidated_use_cases]
                    _inspire_step_result = {
                        "total_use_cases": len(final_consolidated_use_cases),
                        "quality_weight": _quality_weight,
                        "priority_weight": _priority_weight,
                        "quality_distribution": quality_distribution,
                        "inspire_score_min": round(min(_inspire_scores), 2) if _inspire_scores else 0,
                        "inspire_score_max": round(max(_inspire_scores), 2) if _inspire_scores else 0,
                        "inspire_score_avg": round(sum(_inspire_scores) / len(_inspire_scores), 2) if _inspire_scores else 0,
                    }
                    self._emit_pipeline_status(
                        stage_name="Scoring",
                        step_name="Inspire Score",
                        sub_step_name="Computation",
                        message=f"Inspire Scores computed for {len(final_consolidated_use_cases)} use cases (avg={_inspire_step_result['inspire_score_avg']}, min={_inspire_step_result['inspire_score_min']}, max={_inspire_step_result['inspire_score_max']})",
                        status="ended_success",
                        progress_increment=1.0,
                        result_json=_inspire_step_result,
                    )
                    del _inspire_scores, _inspire_step_result
                    
                    # === USE CASES QUALITY FILTER: Apply filtering based on widget selection ===
                    if self.quality_filter_acceptable is not None:
                        acceptable_labels_str = ', '.join(sorted(self.quality_filter_acceptable))
                        log_print(f"\n🔴 QUALITY FILTER [{self.use_cases_quality}]: Keeping only {acceptable_labels_str}...")
                        self.logger.info(f"🔴 Quality filter [{self.use_cases_quality}]: Acceptable labels = {acceptable_labels_str}")
                        
                        accepted_use_cases = []
                        rejected_use_cases = []
                        
                        for uc in final_consolidated_use_cases:
                            quality = uc.get('Quality', 'Medium')
                            if quality in self.quality_filter_acceptable:
                                accepted_use_cases.append(uc)
                            else:
                                rejected_use_cases.append(uc)
                        
                        if rejected_use_cases:
                            log_print(f"\n⚠️ REJECTED USE CASES ({len(rejected_use_cases)} total - below {self.use_cases_quality} threshold):")
                            self.logger.info(f"⚠️ REJECTED USE CASES ({len(rejected_use_cases)} total):")
                            for uc in rejected_use_cases:
                                uc_id = uc.get('No', 'Unknown')
                                uc_name = uc.get('Name', 'Unknown')
                                uc_quality = uc.get('Quality', 'Unknown')
                                uc_summary = uc.get('Quality Summary', 'No summary')
                                log_print(f"   ⚠️ {uc_id}: {uc_name}")
                                log_print(f"      Quality: {uc_quality}")
                                log_print(f"      Reason: {uc_summary[:200]}..." if len(uc_summary) > 200 else f"      Reason: {uc_summary}")
                                self.logger.info(f"   ⚠️ {uc_id}: {uc_name} | Quality: {uc_quality} | {uc_summary}")
                        
                        if not accepted_use_cases:
                            log_print(f"\n{'='*80}")
                            log_print(f"⚠️⚠️⚠️  ZERO USE CASES PASSED QUALITY FILTER [{self.use_cases_quality}]")
                            log_print(f"⚠️⚠️⚠️  FALLBACK: Pulling the next best quality tier forward so you get results")
                            log_print(f"{'='*80}")
                            self.logger.warning(f"⚠️ Quality filter fallback: 0 use cases passed [{self.use_cases_quality}]")

                            fallback_order = ['Medium', 'Low', 'Very Low', 'Ultra Low']
                            fallback_use_cases = []
                            fallback_label = None
                            for fb_label in fallback_order:
                                fallback_use_cases = [uc for uc in rejected_use_cases if uc.get('Quality', '') == fb_label]
                                if fallback_use_cases:
                                    fallback_label = fb_label
                                    break

                            if fallback_use_cases:
                                log_print(f"\n🔄 FALLBACK RECOVERED: {len(fallback_use_cases)} '{fallback_label}' quality use cases pulled forward")
                                self.logger.warning(f"🔄 FALLBACK RECOVERED: {len(fallback_use_cases)} '{fallback_label}' quality use cases")
                                for uc in fallback_use_cases:
                                    log_print(f"   🔄 {uc.get('No', 'Unknown')}: {uc.get('Name', 'Unknown')} (Quality: {fallback_label})")
                                accepted_use_cases = fallback_use_cases
                                still_rejected = [uc for uc in rejected_use_cases if uc.get('Quality', '') != fallback_label]
                                if still_rejected:
                                    log_print(f"\n⚠️ STILL REJECTED ({len(still_rejected)} use cases - Quality < {fallback_label}):")
                                    for uc in still_rejected:
                                        log_print(f"   ⚠️ {uc.get('No', 'Unknown')}: {uc.get('Name', 'Unknown')} (Quality: {uc.get('Quality', 'Unknown')})")
                                del still_rejected
                            else:
                                log_print(f"\n🚨🚨🚨  CRITICAL: No use cases available at any quality level!")
                                log_print(f"🚨🚨🚨  ALL {len(rejected_use_cases)} use cases were rejected")
                                log_print(f"🚨🚨🚨  Pipeline cannot produce any output. Check your data quality.")
                                log_print(f"{'='*80}")
                                _rejected_count = len(rejected_use_cases)
                                self.logger.critical(f"🚨 Quality filter fallback FAILED: All {_rejected_count} use cases rejected. No output possible.")
                                del rejected_use_cases, accepted_use_cases
                                self.storage_manager.cleanup()
                                self.ai_agent.cleanup_llm_cache()
                                AIAgent.get_summary_report()
                                self.ai_agent.get_manager_stats_report()
                                raise RuntimeError(f"Quality filter rejected all {_rejected_count} use cases. No output possible. Check your data quality.")

                        final_consolidated_use_cases = accepted_use_cases

                        # ---- TARGET-ZONE VETTING (bidirectional) ----
                        # Keep portfolio between target_min and target_max (sweet spot 50-100).
                        #   count > target_max  -> raise Inspire Score floor, drop weakest
                        #   count < target_min  -> recover best-scored from rejection pool
                        #   in-zone             -> no-op
                        _vet_threshold = TECHNICAL_CONTEXT["runtime"].get("aggressive_prune_threshold", 100)
                        _elite_floor = TECHNICAL_CONTEXT["runtime"].get("aggressive_vetting_score_floor", 8.5)
                        _target_min = TECHNICAL_CONTEXT["runtime"].get("target_min_use_cases", 50)
                        # Recovery: if count < target_min, pull best-scored rejects
                        if len(final_consolidated_use_cases) < _target_min and rejected_use_cases:
                            try:
                                def _sc_rec(u):
                                    try:
                                        return float(u.get("Inspire Score", u.get("Priority", 0)) or 0)
                                    except Exception:
                                        return 0.0
                                _kept_ids = {u.get("No") for u in final_consolidated_use_cases}
                                _rej_pool = sorted(
                                    [u for u in rejected_use_cases if u.get("No") not in _kept_ids],
                                    key=_sc_rec, reverse=True
                                )
                                _need = _target_min - len(final_consolidated_use_cases)
                                _recovered = _rej_pool[:_need]
                                if _recovered:
                                    _prev = len(final_consolidated_use_cases)
                                    final_consolidated_use_cases = final_consolidated_use_cases + _recovered
                                    log_print(
                                        f"RECOVERY: {_prev} < target_min {_target_min}. "
                                        f"Recovered {len(_recovered)} best-scored UCs from rejection pool. "
                                        f"Final: {len(final_consolidated_use_cases)} UCs."
                                    )
                                    self.logger.info(
                                        f"Target-zone recovery +{len(_recovered)} UCs (target_min={_target_min})"
                                    )
                                    try:
                                        self._emit_pipeline_status(
                                            stage_name="Use Case Prioritization",
                                            step_name="Recovery Vetting",
                                            sub_step_name="Complete", status="ended_success",
                                            message=f"Recovered {len(_recovered)} UCs",
                                            result_json={"recovered": len(_recovered),
                                                         "target_min": _target_min,
                                                         "final_count": len(final_consolidated_use_cases)}
                                        )
                                    except Exception:
                                        pass
                            except Exception as _rec_err:
                                self.logger.warning(
                                    f"Recovery vetting failed non-fatally: {get_clean_error_message(_rec_err)}"
                                )
                        # Sweet-spot target: when portfolio > 75 trigger Best-of-Best ranking +
                        # stratified-by-idea-theme trim to settle in the 50-100 zone with full idea coverage.
                        _consolidation_trigger = TECHNICAL_CONTEXT["runtime"].get("aggressive_prune_threshold", 75)
                        _bob_min_threshold = TECHNICAL_CONTEXT["runtime"].get("bob_min_threshold", 20)
                        if len(final_consolidated_use_cases) >= _bob_min_threshold:
                            _before = len(final_consolidated_use_cases)
                            _target_max_cfg = TECHNICAL_CONTEXT["runtime"].get("target_max_use_cases", 90)
                            _target_max_jitter = TECHNICAL_CONTEXT["runtime"].get("target_max_jitter", 20)
                            import random as _rnd_jitter
                            _target_max_llm = _target_max_cfg + _rnd_jitter.randint(1, max(1, _target_max_jitter))
                            _theme_cap = TECHNICAL_CONTEXT["runtime"].get("bob_theme_max_share", 0.15)
                            _will_trim = _before > _consolidation_trigger
                            try:
                                if _will_trim:
                                    log_print(
                                        f"\U0001F3C6 BEST-OF-BEST + STRATIFIED TRIM: {_before} UCs > prune threshold {_consolidation_trigger}. "
                                        f"Scoring each UC on 19 gates (0-10) + idea theme, then trimming per theme to ~{_target_max_llm}."
                                    )
                                else:
                                    log_print(
                                        f"\U0001F3C6 BEST-OF-BEST SCORING ONLY: {_before} UCs \u2265 bob_min {_bob_min_threshold} "
                                        f"but \u2264 prune threshold {_consolidation_trigger} \u2014 scoring without trim so bob_score is populated for every UC."
                                    )
                                _schema_for_bob = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                                final_consolidated_use_cases = self._score_best_of_best(
                                    final_consolidated_use_cases, _schema_for_bob,
                                    business_context=getattr(self, "_business_context_cached", ""),
                                    industry=getattr(self, "_industry_cached", ""),
                                    strategic_goals=getattr(self, "_strategic_goals_cached", None),
                                    business_priorities=getattr(self, "_business_priorities_cached", None),
                                )
                                if _will_trim:
                                    _reviewed = self._stratified_trim_by_theme(
                                        final_consolidated_use_cases,
                                        target_max=_target_max_llm,
                                        theme_max_share=_theme_cap,
                                    )
                                    if _reviewed and len(_reviewed) < _before:
                                        final_consolidated_use_cases = _reviewed
                                        log_print(
                                            f"\U0001F50D LLM consolidation: {_before} \u2192 {len(_reviewed)} UCs (LLM-based dedup re-review)"
                                        )
                                        self.logger.info(
                                            f"LLM aggressive consolidation: {_before} \u2192 {len(_reviewed)}"
                                        )
                                        try:
                                            self._emit_pipeline_status(
                                                stage_name="Use Case Prioritization",
                                                step_name="LLM Aggressive Consolidation",
                                                sub_step_name="Complete", status="ended_success",
                                                message=f"LLM re-review: {_before} \u2192 {len(_reviewed)}",
                                                result_json={"before": _before, "after": len(_reviewed),
                                                             "target_max": _target_max_llm}
                                            )
                                        except Exception:
                                            pass
                                    else:
                                        self.logger.info(
                                            f"LLM re-review found no further consolidation opportunity ({_before} UCs kept)"
                                        )
                            except Exception as _ap_err:
                                self.logger.warning(
                                    f"BoB scoring/trim failed non-fatally: {get_clean_error_message(_ap_err)}"
                                )
                        else:
                            self.logger.info(
                                f"BoB scoring skipped: portfolio has {len(final_consolidated_use_cases)} UCs < bob_min_threshold {_bob_min_threshold}; "
                                f"bob_score will be NULL for this run (insufficient UCs to discriminate)."
                            )
                        # ---- end BoB scoring + optional aggressive consolidation ----

                        # ---- post-BoB safety net: canonical-title dedup ----
                        # The LLM-based dedup (line ~18481) is conservative and may keep verbatim duplicate
                        # titles when their statements differ slightly. This deterministic pass removes any
                        # duplicates that share the SAME canonical title after `_normalize_usecase` is applied
                        # — the same function the persist + display layer uses. That function truncates at
                        # ' with ' and title-cases, so two raw Name fields that differ only after the ' with '
                        # qualifier collapse to one displayed/persisted title. Without this safety net those
                        # twins would surface as identical rows in __inspire_usecases.
                        # Keeps the higher bob_score (tie-break: higher Priority). Idempotent on clean portfolios.
                        try:
                            _seen_titles = {}
                            _dedup_dropped = []
                            import re as _re_dedup_local
                            def _norm_title(uc):
                                _raw_uc = uc.get("usecase") or ""
                                _raw_nm = uc.get("Name") or uc.get("use_case") or ""
                                try:
                                    _canon = self._normalize_usecase(_raw_uc, _raw_nm, "English")
                                except Exception:
                                    _canon = _raw_nm or _raw_uc or ""
                                if not _canon or _canon == "N/A":
                                    return ""
                                return _re_dedup_local.sub(r"\s+", " ", str(_canon)).strip().lower()
                            def _dedup_rank(uc):
                                _b = uc.get("bob_score")
                                _p = uc.get("Priority", 0) or 0
                                try:
                                    _b_f = float(_b) if _b is not None else -1.0
                                except Exception:
                                    _b_f = -1.0
                                try:
                                    _p_f = float(_p)
                                except Exception:
                                    _p_f = 0.0
                                return (_b_f, _p_f)
                            for uc in final_consolidated_use_cases:
                                _key = _norm_title(uc)
                                if not _key:
                                    continue
                                _existing = _seen_titles.get(_key)
                                if _existing is None:
                                    _seen_titles[_key] = uc
                                else:
                                    if _dedup_rank(uc) > _dedup_rank(_existing):
                                        _dedup_dropped.append(_existing)
                                        _seen_titles[_key] = uc
                                    else:
                                        _dedup_dropped.append(uc)
                            if _dedup_dropped:
                                _kept_set = {id(v) for v in _seen_titles.values()}
                                _new_list = [uc for uc in final_consolidated_use_cases if id(uc) in _kept_set]
                                _dropped_summary = [
                                    f"{uc.get('No','?')}:{(uc.get('Name') or uc.get('use_case') or '')[:60]}"
                                    for uc in _dedup_dropped[:5]
                                ]
                                log_print(
                                    f"\U0001F9F9 CANONICAL-TITLE DEDUP SAFETY NET: dropped {len(_dedup_dropped)} duplicate-title UC(s). "
                                    f"Kept {len(_new_list)}/{len(final_consolidated_use_cases)}. Sample drops: {_dropped_summary}"
                                )
                                self.logger.warning(
                                    f"Canonical-title dedup safety net dropped {len(_dedup_dropped)} UC(s) the LLM dedup missed: "
                                    f"{[uc.get('No') for uc in _dedup_dropped]}"
                                )
                                final_consolidated_use_cases = _new_list
                            else:
                                self.logger.info(
                                    f"Canonical-title dedup safety net: 0 duplicate-title UCs found across {len(final_consolidated_use_cases)} UCs (clean)."
                                )
                        except Exception as _ddp_err:
                            self.logger.warning(
                                f"Canonical-title dedup safety net failed non-fatally: {get_clean_error_message(_ddp_err)}"
                            )
                        # ---- end canonical-title dedup safety net ----

                        if len(final_consolidated_use_cases) < self.min_use_cases_for_selection:
                            pool = [uc for uc in rejected_use_cases if uc not in final_consolidated_use_cases]
                            if pool:
                                _quality_order = {'Ultra High': 7, 'Very High': 6, 'High': 5, 'Medium': 4, 'Low': 3, 'Very Low': 2, 'Ultra Low': 1}
                                def _sort_key(uc):
                                    p = uc.get('Priority', 0)
                                    try:
                                        pval = float(p)
                                    except (TypeError, ValueError):
                                        pval = 0
                                    q = uc.get('Quality', 'Medium')
                                    qval = _quality_order.get(q, 0)
                                    return (-pval, -qval)
                                pool_sorted = sorted(pool, key=_sort_key)
                                need = self.min_use_cases_for_selection - len(final_consolidated_use_cases)
                                to_add = pool_sorted[:need]
                                for uc in to_add:
                                    final_consolidated_use_cases.append(uc)
                                log_print(f"\n📈 MIN USE CASES TOP-UP [{self.min_use_cases_for_selection}]: Added {len(to_add)} use cases from below-threshold pool (sorted by Priority desc, Quality tie-break) to ensure minimum selection size")
                                self.logger.info(f"📈 Min use cases top-up: added {len(to_add)} from rejected pool to reach {self.min_use_cases_for_selection}")
                        
                        log_print(f"\n✅ ACCEPTED USE CASES: {len(final_consolidated_use_cases)} (passed {self.use_cases_quality} filter)")
                        log_print(f"⚠️ REJECTED USE CASES: {len(rejected_use_cases)} (below threshold)")
                        self.logger.info(f"✅ Quality filter [{self.use_cases_quality}]: {len(final_consolidated_use_cases)} accepted, {len(rejected_use_cases)} rejected")
                        del rejected_use_cases, accepted_use_cases
                        
                        quality_distribution = {}
                        for uc in final_consolidated_use_cases:
                            label = uc.get('Quality', 'Medium')
                            quality_distribution[label] = quality_distribution.get(label, 0) + 1
                        
                        log_print(f"\n📊 FINAL QUALITY DISTRIBUTION (after filtering):")
                        for label in ['Ultra High', 'Very High', 'High', 'Medium', 'Low', 'Very Low', 'Ultra Low']:
                            count = quality_distribution.get(label, 0)
                            if count > 0:
                                log_print(f"   {label}: {count} use cases")
                    else:
                        log_print(f"\nℹ️ Quality Filter: Coverage Mode (All) - All scored use cases retained (no post-scoring filtering)")
                        self.logger.info(f"ℹ️ Quality Filter: Coverage Mode (All) - No quality filtering applied")
                
            else:
                 self.logger.warning("No data loader. Skipping use case, PDF, and Presentation generation.")
                 return
        except Exception as e:
            self.logger.critical(f"A critical error occurred during English generation: {get_clean_error_message(e)}")
            log_print(f"\n❌ FATAL: {get_clean_error_message(e)}")
            self.storage_manager.cleanup()
            self.ai_agent.cleanup_llm_cache()
            AIAgent.get_summary_report()
            self.ai_agent.get_manager_stats_report()
            raise

        if final_consolidated_use_cases:
            _missing_inspire = sum(1 for uc in final_consolidated_use_cases if 'Inspire Score' not in uc)
            if _missing_inspire > 0:
                self.logger.warning(f"⚠️ {_missing_inspire}/{len(final_consolidated_use_cases)} use cases missing Inspire Score - computing defaults")
                _qw = TECHNICAL_CONTEXT["runtime"].get("inspire_score_quality_weight", 0.65)
                _pw = TECHNICAL_CONTEXT["runtime"].get("inspire_score_priority_weight", 0.35)
                for uc in final_consolidated_use_cases:
                    if 'Inspire Score' not in uc:
                        raw_q = float(uc.get('Quality Score', 3.0) or 3.0)
                        q_norm = min(10.0, max(0.0, raw_q * 2.0))
                        raw_p = float(uc.get('Priority', 5.0) or 5.0)
                        p_norm = min(10.0, max(0.0, raw_p))
                        uc['Inspire Score'] = round(_qw * q_norm + _pw * p_norm, 2)

        english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases) if final_consolidated_use_cases else {}
        summary_dict = None

        if final_consolidated_use_cases:
            for uc in final_consolidated_use_cases:
                if not uc.get('Analytics Technique') or uc.get('Analytics Technique') == 'N/A':
                    uc['Analytics Technique'] = 'AI Analysis'
                tables_involved = uc.get('Tables Involved', '')
                uc['Primary Table'] = self._extract_primary_table(tables_involved)
                if 'result_table' not in uc or not uc.get('result_table'):
                    uc['result_table'] = compute_result_table_name(
                        uc.get('No', 'unknown'), uc.get('Name', 'unknown'),
                        tables_involved, inspire_database=self.inspire_database
                    )
            self.logger.info(f"✅ Pre-computed Primary Table and Result Database for {len(final_consolidated_use_cases)} use cases")
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)
            # Phase L: advisory final-gate run AFTER Primary Table derivation (UC dicts now complete).
            try:
                self._emit_pipeline_status(
                    stage_name="Final Gate", step_name="Phase L Production Checks",
                    sub_step_name="Started", status="started",
                    message="Running Phase L 19-check production-readiness gate"
                )
            except Exception:
                pass
            try:
                _q_schema_L = self.storage_manager.load_schema_details("all_columns_for_sql") or []
                _L_results = self._phaseL_final_gate(final_consolidated_use_cases, _q_schema_L)
                _L_fails = [k for k, v in _L_results.items() if not v.get("pass")]
                self.logger.info(f"Phase L final-gate: {len(_L_results)-len(_L_fails)}/{len(_L_results)} passed; failures={_L_fails}")
                try:
                    self._emit_pipeline_status(
                        stage_name="Final Gate", step_name="Phase L Production Checks",
                        sub_step_name="Complete",
                        status="ended_success" if not _L_fails else "ended_warning",
                        message=f"Phase L: {len(_L_results)-len(_L_fails)}/{len(_L_results)} checks passed",
                        result_json={"passed": len(_L_results)-len(_L_fails), "total": len(_L_results), "failures": _L_fails}
                    )
                except Exception:
                    pass
                del _q_schema_L
            except Exception as _L_err:
                self.logger.warning(f"Phase L final-gate failed non-fatally: {get_clean_error_message(_L_err)}")
                try:
                    self._emit_pipeline_status(
                        stage_name="Final Gate", step_name="Phase L Production Checks",
                        sub_step_name="Error", status="ended_warning",
                        message=f"Phase L failed non-fatally: {get_clean_error_message(_L_err)}"
                    )
                except Exception:
                    pass

        if final_consolidated_use_cases:
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)

        if final_consolidated_use_cases:
            self.logger.info("Generating English CSV...")
            lang_abbr_en = self._get_lang_abbr("English")
            try:
                self._generate_csv_catalog("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to generate English CSV: {get_clean_error_message(e)}")
            try:
                self._tracking_replace_session(final_consolidated_use_cases)
            except Exception as _track_err:
                self.logger.error(f"⚠️ Tracking table UPDATE 1/2 (post-CSV) failed (non-fatal): {_track_err}")

        
        # === PHASE 2: GENERATE SUMMARY & DOCUMENTATION ===
        if final_consolidated_use_cases and not self.json_file_path:
            if summary_dict is None:
                self.logger.info("Generating executive summaries...")
                (summary_dict, _) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
        
        # === START DOCUMENTATION GENERATION ===
        doc_generation_future = None
        doc_generation_executor = None
        remaining_langs = [lang for lang in self.output_languages if lang != "English"]
        target_langs = ["English"] + remaining_langs if "English" in self.output_languages else remaining_langs
        
        if final_consolidated_use_cases and target_langs and ("PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices):
            self.logger.info("🚀 Starting PDF/PPTX documentation generation...")
            log_print(f"📄 Documentation generation starting (languages: {', '.join(target_langs)})")
            doc_generation_executor = concurrent.futures.ThreadPoolExecutor(max_workers=1, thread_name_prefix="DocGen")
            doc_generation_future = doc_generation_executor.submit(
                self._generate_documents_for_all_languages,
                final_consolidated_use_cases,
                english_grouped_data,
                summary_dict,
                target_langs,
                ["English"]  # skip_excel_langs - already generated
            )
        
        _force_gc(self.logger, "pre-Genie-generation phase cleanup")
        
        # === PHASE 2: GENIE CODE INSTRUCTIONS GENERATION ===
        if self.generate_genie_instructions and final_consolidated_use_cases:
            self.logger.info(f"🤖 PHASE 2: Genie Code instructions generation...")
            _schema_for_genie = self.storage_manager.load_schema_details("all_columns_for_sql")
            try:
                final_consolidated_use_cases = self._generate_genie_instructions_and_notebooks(
                    final_consolidated_use_cases,
                    _schema_for_genie,
                    document_context_markdown,
                    english_translations,
                    summary_dict
                )
            except Exception as e:
                self.logger.error(f"❌ Genie Code instruction generation failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Genie Code instruction generation encountered an error: {str(e)[:200]}", level="WARNING")
                log_print(f"ℹ️ Proceeding with remaining artifacts...")
            finally:
                del _schema_for_genie
            _force_gc(self.logger, "after Genie Code generation phase")
            self.logger.info("✅ Phase 2 complete: Genie Code instructions generated")
        elif (not self.generate_genie_instructions) and final_consolidated_use_cases and getattr(self, 'operation', '') == 'Discover Use Cases':
            self.logger.info(f"🔎 PHASE 2 (Discover mode): Capturing per-UC Genie prompt fields + generating skeleton notebooks...")
            _schema_for_genie = self.storage_manager.load_schema_details("all_columns_for_sql")
            try:
                final_consolidated_use_cases = self._run_discover_skeleton_phase(
                    final_consolidated_use_cases,
                    _schema_for_genie
                )
            except Exception as e:
                self.logger.error(f"❌ Discover skeleton phase failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Discover skeleton generation encountered an error: {str(e)[:200]}", level="WARNING")
            finally:
                del _schema_for_genie
            _force_gc(self.logger, "after Discover skeleton phase")
            self.logger.info("✅ Phase 2 complete (Discover): skeleton notebooks generated, prompt fields captured")
        
        if final_consolidated_use_cases and self._tracking_table_name:
            try:
                self._tracking_replace_session(final_consolidated_use_cases)
                self.logger.info("✅ [UPDATE 2/2] Tracking table re-synced post-Phase-2 (notebook_path, genie_instruction, discover fields)")
            except Exception as _track_err2:
                self.logger.error(f"⚠️ Tracking table UPDATE 2/2 (post-Phase-2) failed (non-fatal): {_track_err2}")
        
        # === GENERATE USE CASE NOTEBOOKS (batch fallback — only if NOT already streamed inline during genie phase) ===
        _notebooks_done_inline = getattr(self, '_notebooks_generated_inline', False)
        if final_consolidated_use_cases and not self.json_file_path and not _notebooks_done_inline:
            self.logger.info("📓 Generating per-use-case notebooks (batch fallback — no genie streaming)...")
            log_print(f"\n📓 Generating per-use-case notebooks for {len(final_consolidated_use_cases)} use cases...")
            try:
                self.assemble_use_case_notebooks(final_consolidated_use_cases, english_translations, summary_dict)
            except Exception as e:
                self.logger.error(f"❌ Notebook generation failed: {get_clean_error_message(e)}")
                log_print(f"⚠️ Notebook generation failed: {str(e)[:200]}", level="WARNING")
        elif _notebooks_done_inline:
            self.logger.info("📓 Notebooks already generated inline during genie streaming phase — skipping batch generation")

        if final_consolidated_use_cases:
            self.logger.info("Generating English Excel (post-notebook, with hyperlinks)...")
            lang_abbr_en = self._get_lang_abbr("English")
            english_grouped_data = self._group_use_cases_by_domain_flat(final_consolidated_use_cases)
            try:
                self._generate_use_case_excel("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to generate English Excel: {get_clean_error_message(e)}")
            try:
                self._generate_csv_catalog("English", lang_abbr_en, english_grouped_data)
            except Exception as e:
                self.logger.error(f"Failed to regenerate English CSV (post-Auto-Genie): {get_clean_error_message(e)}")

        # === WAIT FOR DOCUMENTATION GENERATION TO COMPLETE ===
        _parallel_doc_gen_was_started = doc_generation_future is not None
        if doc_generation_future:
            try:
                self.logger.info("⏳ Waiting for documentation generation to complete...")
                _safe_future_result(doc_generation_future, timeout=1800)
                self.logger.info("✅ Documentation generation completed")
                log_print("✅ Documentation generation completed")
                try:
                    # Phase O persist removed per user request — next_actions.jsonl,
                    # manifest.json, NEXT_ACTIONS_REPORT.md no longer written.
                    pass
                    try:
                        self._emit_pipeline_status(
                            stage_name="Artifact Writing", step_name="Phase O Run-Artifact Persist",
                            sub_step_name="Complete", status="ended_success",
                            message="Run artifacts persisted"
                        )
                    except Exception:
                        pass
                except Exception as _o_err:
                    self.logger.warning(f"Phase O persistence failed non-fatally: {get_clean_error_message(_o_err)}")
            except concurrent.futures.TimeoutError:
                self.logger.warning("⚠️ Documentation generation timed out after 30 minutes")
                log_print("⚠️ Documentation generation timed out", level="WARNING")
            except Exception as e:
                self.logger.error(f"❌ Documentation generation failed: {get_clean_error_message(e)}")
                log_print(f"❌ Documentation generation failed: {str(e)[:100]}", level="ERROR")
            finally:
                if doc_generation_executor:
                    try:
                        doc_generation_executor.shutdown(wait=False, cancel_futures=True)
                    except TypeError:
                        doc_generation_executor.shutdown(wait=False)
                del doc_generation_future
                doc_generation_future = None
        _force_gc(self.logger, "after documentation generation")

        # === SAVE JSON CATALOG (POST-SQL) ===
        if final_consolidated_use_cases and not self.json_file_path:
            self.logger.info("Saving JSON Catalog...")
            summary_dict = self._save_usecases_catalog_json(final_consolidated_use_cases, english_translations, summary_dict)
        
        # Note: PDF/PPTX documentation generation was started in parallel earlier
        # Only generate here if parallel generation was NOT started (no PDF/Presentation selected initially)
        if final_consolidated_use_cases and not _parallel_doc_gen_was_started:
            remaining_langs = [lang for lang in self.output_languages if lang != "English"]
            target_langs = ["English"] + remaining_langs if "English" in self.output_languages else remaining_langs
            if target_langs and ("PDF Catalog" in self.generate_choices or "Presentation" in self.generate_choices):
                if summary_dict is None:
                    (summary_dict, _) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
                self.logger.info("Generating PDFs/Presentations and translations...")
                self._generate_documents_for_all_languages(
                    final_consolidated_use_cases,
                    english_grouped_data=english_grouped_data,
                    summary_dict=summary_dict,
                    languages=target_langs,
                    skip_excel_langs=["English"]
                )
        del english_grouped_data
        _force_gc(self.logger, "freed english_grouped_data after all doc gen")
        
        # Report table inclusion/exclusion statistics
        if final_consolidated_use_cases and self.data_loader:
            self._report_table_statistics(final_consolidated_use_cases)
        
        self.storage_manager.cleanup()
        self.ai_agent.cleanup_llm_cache()
        
        # Upload log file, honesty report, and final success BEFORE AI usage summary
        self.logger.info(f"✅ All Use cases for {self.business_name} generated successfully")
        self.logger.info("Uploading log file...")
        self._upload_log_file()
        
        # Show processing honesty report
        self._report_processing_honesty()
        
        # Final success message first; AI usage summary must be the final output block
        log_print(f"✅ All Use cases for {self.business_name} generated successfully")
        
        # Show AI usage summary as the very last run output
        AIAgent.get_summary_report()
        self.ai_agent.get_manager_stats_report()
            

    def _get_salesy_summary(self, grouped_data: dict, business_name: str, language: str, translations: dict) -> tuple:
        self.logger.debug(f"Calling LLM for executive and domain summaries in {language}...")
        t = translations
        summary_dict = {}
        transliterated_name = business_name # Default
        try:
            domain_list_for_prompt = "\n".join([f"- {domain}" for domain in grouped_data.keys()])
            total_cases = sum(len(cases) for cases in grouped_data.values())
            
            highlights_parts = []
            for domain, cases in grouped_data.items():
                sorted_cases = sorted(cases, key=lambda c: float(c.get('Priority', 0) or 0), reverse=True)
                top_cases = sorted_cases[:3]
                case_bullets = []
                for c in top_cases:
                    name = c.get('Name', 'Unnamed')
                    bv = c.get('Business Value', '')
                    bv_snippet = (bv[:120] + '...') if len(bv) > 120 else bv
                    case_bullets.append(f"  - {name}: {bv_snippet}" if bv_snippet else f"  - {name}")
                highlights_parts.append(f"**{domain}** ({len(cases)} use cases):\n" + "\n".join(case_bullets))
            domain_use_case_highlights = "\n\n".join(highlights_parts)
            
            prompt_vars = {
                "business_name": business_name, "total_cases": str(total_cases),
                "domain_list": domain_list_for_prompt, "output_language": language,
                "domain_use_case_highlights": domain_use_case_highlights,
                "generation_instructions_section": self._get_generation_instruction_for_prompt("SUMMARY_GEN_PROMPT"),
            }
            summary_csv_raw = self.ai_agent.run_worker(
                step_name=f"Executive_Summary_{language}",
                worker_prompt_path="SUMMARY_GEN_PROMPT",
                prompt_vars=prompt_vars, response_schema=None
            )
            self.logger.info(f"LLM summaries (CSV) received for {language}.")
            
            # === ROBUST CSV PARSING (Req 2) ===
            # Support both quoted and unquoted headers from LLM
            header_3_col_quoted = '"Type","Summary","TransliteratedBusinessName"'
            header_3_col_unquoted = 'Type,Summary,TransliteratedBusinessName'
            header_2_col_quoted = '"Type","Summary"'
            header_2_col_unquoted = 'Type,Summary'
            
            header_start_index = summary_csv_raw.find(header_3_col_quoted)
            is_3_col = True
            
            if header_start_index == -1:
                header_start_index = summary_csv_raw.find(header_3_col_unquoted)
            
            if header_start_index == -1:
                self.logger.warning(f"Could not find 3-column CSV header in LLM response for {language}. Attempting 2-col parse.")
                is_3_col = False
                header_start_index = summary_csv_raw.find(header_2_col_quoted)
                if header_start_index == -1:
                    header_start_index = summary_csv_raw.find(header_2_col_unquoted)
                if header_start_index == -1:
                    self.logger.error(f"Could not find 2-column or 3-column CSV header. Aborting summary parse. Response: {summary_csv_raw[:200]}")
                    raise ValueError(f"Could not parse summary CSV for {language}")

            self.logger.info(f"Found CSV header at index {header_start_index}. Parsing as {3 if is_3_col else 2}-column.")
            summary_csv_clean = summary_csv_raw[header_start_index:]
            # ==================================

            # Use centralized CSV parser
            csv_rows = CSVParser.parse_csv_list(
                summary_csv_clean,
                logger=self.logger,
                context="Domain summary",
                delimiter=',',
                quotechar='"',
                quoting=csv.QUOTE_ALL,
                skipinitialspace=True
            )
            if csv_rows:
                header = csv_rows[0]  # First row is header
                csv_reader = csv_rows[1:]  # Rest are data rows
            else:
                csv_reader = []
            
            if is_3_col:
                for row in csv_reader:
                    # Handle rows that may span multiple lines or have embedded content
                    if len(row) >= 3:
                        row_type = row[0].strip()
                        summary_dict[row_type] = row[1].strip()
                        if row_type == "Executive" and row[2].strip():
                            transliterated_name = row[2].strip()
                    elif len(row) > 0:
                        # Partial row - try to accumulate
                        self.logger.debug(f"Partial 3-col row (len={len(row)}): {row[:1]}")
                        # Skip partial rows gracefully without warning
            else: # 2-col fallback
                for row in csv_reader:
                    if len(row) >= 2:
                        summary_dict[row[0].strip()] = row[1].strip()
                    elif len(row) > 0:
                        self.logger.debug(f"Partial 2-col row (len={len(row)}): {row[:1]}")
            transliterated_name = business_name if not is_3_col else transliterated_name
            
            if "Executive" not in summary_dict:
                self.logger.error("Failed to parse 'Executive' summary from LLM response.")
                raise ValueError("Missing 'Executive' summary")
                
            self.logger.info(f"Successfully parsed {len(summary_dict)} summaries for {language}. Transliterated name: {transliterated_name}")
            return summary_dict, transliterated_name
            
        except Exception as e:
            self.logger.error(f"LLM summary generation failed for {language}: {get_clean_error_message(e)}. Using default text.")
            fallback_dict = {}
            total_cases_fallback = sum(len(cases) for cases in grouped_data.values())
            p1 = t['pdf_fallback_summary_p1'].format(total_cases=total_cases_fallback, business_name=business_name)
            p2 = t['pdf_fallback_summary_p2']
            fallback_dict["Executive"] = f"<p>{p1}</p><p>{p2}</p>"
            for domain in grouped_data.keys():
                fallback_dict[domain] = "<p>Summary generation failed. This domain's key responsibilities and opportunities have been identified for AI transformation.</p>"
            return fallback_dict, business_name # Return default name

    def _merge_small_domains(self, use_cases: list, min_cases_per_domain: int = 4) -> list:
        """
        Merges business domains that have fewer than min_cases_per_domain use cases.
        Uses adaptive thresholds based on total UC count and enforces anti-mega-domain constraints.
        """
        if not use_cases:
            return use_cases
        
        total_ucs = len(use_cases)
        
        if total_ucs < 30:
            min_cases_per_domain = 2
        elif total_ucs < 60:
            min_cases_per_domain = 3
        else:
            min_cases_per_domain = 4
        
        max_domain_share = 0.35
        max_ucs_per_domain = max(int(total_ucs * max_domain_share), min_cases_per_domain + 1)
        
        import math
        target_domain_count = max(3, min(round(math.sqrt(total_ucs)), 25))
        
        self.logger.info(
            f"Domain merge config: {total_ucs} UCs, min_per_domain={min_cases_per_domain}, "
            f"max_per_domain={max_ucs_per_domain} ({max_domain_share:.0%} of total), "
            f"target_domains={target_domain_count}"
        )
        
        domain_counts = defaultdict(int)
        for uc in use_cases:
            domain = uc.get('Business Domain', 'Other')
            domain_counts[domain] += 1
        
        small_domains = {domain: count for domain, count in domain_counts.items() if count < min_cases_per_domain}
        
        if not small_domains:
            self.logger.info(f"All domains have at least {min_cases_per_domain} use cases. No merging needed.")
            return use_cases
        
        self.logger.debug(f"Found {len(small_domains)} domains with fewer than {min_cases_per_domain} use cases: {list(small_domains.keys())}")
        
        domain_info_lines = []
        for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
            pct = count / total_ucs * 100
            if count < min_cases_per_domain:
                status = f"❌ TOO SMALL (needs merging)"
            elif count > max_ucs_per_domain:
                status = f"⚠️ ALREADY LARGE ({pct:.0f}% of total — do NOT merge more into this)"
            else:
                status = f"✓ OK ({pct:.0f}% of total)"
            domain_info_lines.append(f"  - {domain}: {count} use cases {status}")
        
        domain_info_str = "\n".join(domain_info_lines)
        
        merge_prompt = f"""You are a business domain expert. Analyze this list of business domains and their use case counts:

Total use cases: {total_ucs}
Target domain count: {target_domain_count}

{domain_info_str}

**CRITICAL REQUIREMENTS**:
1. Domains with < {min_cases_per_domain} use cases MUST be merged into a semantically related domain
2. **ANTI-MEGA-DOMAIN RULE**: After merging, NO domain should have more than {max_ucs_per_domain} use cases ({max_domain_share:.0%} of total). If merging a small domain into a large one would exceed this limit, find a DIFFERENT target or merge multiple small domains TOGETHER into a new domain.
3. **SEMANTIC RELATEDNESS**: ONLY merge domains that are genuinely related. Do NOT merge semantically unrelated domains just because one is small. Fraud ≠ Engagement. Inventory ≠ Customers. Workforce ≠ Revenue.
4. After merging, you should still have at least {max(3, target_domain_count - 2)} domains. Do NOT over-merge.

**Your Task**:
1. Identify domains marked ❌ (too small)
2. For each, find the most semantically related domain to merge into
3. If no good semantic match exists, merge 2-3 small related domains TOGETHER into a new combined domain
4. NEVER merge a small domain into a domain marked ⚠️ ALREADY LARGE

**Output Format** (honesty-wrapped JSON):
Your response MUST be wrapped: {{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field maps old domain names to new domain names.
Only include domains that need renaming/merging.

**Rules**:
1. Every domain name must be EXACTLY ONE WORD (no spaces), unique core word, industry-specific
2. ❌ BANNED generic names: "Operations", "Engagement", "Management", "Services", "Activities", "General"
3. ✅ Use specific business nouns: "Revenue", "Customers", "Inventory", "Fraud", "Workforce", "Pricing", "Franchise", "Logistics"
4. Domains with ≥ {min_cases_per_domain} can stay as-is UNLESS you have a significantly better name

Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON
        
        try:
            # Use run_worker with a direct prompt
            merge_prompt_key = "DOMAINS_MERGER_PROMPT"
            _merger_instr = self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT")
            merge_prompt = f"{merge_prompt}\n\n{_merger_instr}\n"
            self.ai_agent.prompt_templates[merge_prompt_key] = merge_prompt
            
            response = self.ai_agent.run_worker(
                step_name=f"Merge_Small_Domains",
                worker_prompt_path=merge_prompt_key,
                prompt_vars={},
                response_schema=None
            )
            
            # Clean up temporary prompt template
            del self.ai_agent.prompt_templates[merge_prompt_key]
            
            cleaned = clean_json_response(response)
            if not cleaned or cleaned.strip() == '':
                self.logger.warning("Domain merge response is empty after cleaning. Skipping merge.")
                return use_cases
            merge_mapping = json.loads(cleaned)
            
            # Extract data from honesty-wrapped response if present
            if isinstance(merge_mapping, dict) and 'data' in merge_mapping:
                merge_mapping = merge_mapping['data']
            
            # Ensure merge_mapping is a dict with string keys/values
            if not isinstance(merge_mapping, dict):
                self.logger.warning(f"Domain merge response is not a dict: {type(merge_mapping)}. Skipping merge.")
                return use_cases
            
            # Sanitize: LLMs sometimes return nested dicts/lists as values instead of plain strings
            sanitized = {}
            for k, v in merge_mapping.items():
                if isinstance(k, str) and isinstance(v, str):
                    sanitized[k] = v
                else:
                    self.logger.warning(f"Domain merge: dropping non-string entry {type(k).__name__}={type(v).__name__} for key '{k}'")
            merge_mapping = sanitized
            
            # Resolve transitive merges (e.g., A->B, B->C  =>  A->C, B->C)
            for _ in range(5):
                updated = False
                for key, val in list(merge_mapping.items()):
                    if val in merge_mapping and merge_mapping[val] != val:
                         merge_mapping[key] = merge_mapping[val]
                         updated = True
                if not updated:
                    break
            
            projected_counts = defaultdict(int, domain_counts)
            rejected_merges = []
            for old_name, new_name in list(merge_mapping.items()):
                if old_name == new_name:
                    continue
                old_count = domain_counts.get(old_name, 0)
                target_current = projected_counts.get(new_name, 0)
                if target_current + old_count > max_ucs_per_domain:
                    self.logger.warning(
                        f"Rejecting merge '{old_name}' ({old_count} UCs) → '{new_name}' ({target_current} UCs): "
                        f"would create mega-domain with {target_current + old_count} UCs (max {max_ucs_per_domain})"
                    )
                    rejected_merges.append(old_name)
                    del merge_mapping[old_name]
                else:
                    projected_counts[new_name] = target_current + old_count
                    if old_name in projected_counts:
                        del projected_counts[old_name]
            
            if rejected_merges:
                self.logger.info(f"Rejected {len(rejected_merges)} merges that would create mega-domains: {rejected_merges}")
            
            merged_count = 0
            for uc in use_cases:
                old_domain = uc.get('Business Domain', 'Other')
                if old_domain in merge_mapping:
                    new_domain = merge_mapping[old_domain]
                    if old_domain != new_domain:
                        self.logger.info(f"Merging '{old_domain}' ({domain_counts[old_domain]} cases) → '{new_domain}'")
                        uc['Business Domain'] = new_domain
                        merged_count += 1
            
            self.logger.debug(f"Domain merging complete. {merged_count} use cases reassigned.")
            
            final_counts = defaultdict(int)
            for uc in use_cases:
                final_counts[uc.get('Business Domain', 'Other')] += 1
            
            remaining_small = [d for d, count in final_counts.items() if count < min_cases_per_domain]
            if remaining_small:
                self.logger.warning(f"WARNING: {len(remaining_small)} domains still have fewer than {min_cases_per_domain} use cases: {remaining_small}")
            
            mega_domains = {d: c for d, c in final_counts.items() if c > max_ucs_per_domain}
            if mega_domains:
                self.logger.warning(
                    f"⚠️ {len(mega_domains)} mega-domain(s) detected after merge: "
                    + ", ".join(f"'{d}' ({c} UCs, {c/total_ucs:.0%})" for d, c in mega_domains.items())
                    + f". Max allowed: {max_ucs_per_domain} UCs ({max_domain_share:.0%}). "
                    f"These will be split in the next step."
                )
            
            domain_count = len(final_counts)
            self.logger.info(
                f"Post-merge: {domain_count} domains (target: {target_domain_count}). "
                f"Distribution: {dict(sorted(final_counts.items(), key=lambda x: -x[1]))}"
            )
            
            return use_cases
            
        except Exception as e:
            self.logger.error(f"Failed to merge domains: {get_clean_error_message(e)}")
            return use_cases

    def _split_mega_domains(self, use_cases: list) -> list:
        """
        Post-merge step: detects domains that are too large (>35% of total UCs)
        and splits them into more specific sub-domains using an LLM.
        """
        if not use_cases:
            return use_cases
        
        total_ucs = len(use_cases)
        max_domain_share = 0.35
        max_ucs_per_domain = max(int(total_ucs * max_domain_share), 6)
        
        domain_counts = defaultdict(int)
        for uc in use_cases:
            domain = uc.get('Business Domain', 'Other')
            domain_counts[domain] += 1
        
        mega_domains = {d: c for d, c in domain_counts.items() if c > max_ucs_per_domain}
        
        if not mega_domains:
            self.logger.info(f"No mega-domains found (max allowed: {max_ucs_per_domain} UCs per domain). Skipping split.")
            return use_cases
        
        import math
        all_domain_names = set(domain_counts.keys())
        
        for mega_name, mega_count in mega_domains.items():
            target_splits = max(2, round(math.sqrt(mega_count)))
            
            self.logger.info(
                f"📍 Splitting mega-domain '{mega_name}' ({mega_count} UCs, {mega_count/total_ucs:.0%} of total) "
                f"into ~{target_splits} new domains..."
            )
            
            mega_ucs = [uc for uc in use_cases if uc.get('Business Domain', '').strip() == mega_name]
            
            uc_lines = []
            for uc in mega_ucs:
                uc_lines.append(
                    f"  - {uc.get('No', '')}: {uc.get('Name', '')} "
                    f"[{uc.get('type', '')}] [{uc.get('Analytics Technique', '')}] "
                    f"Beneficiary: {uc.get('Beneficiary', '')} | Tables: {uc.get('Tables Involved', '')}"
                )
            uc_str = "\n".join(uc_lines)
            
            other_domains = sorted(all_domain_names - {mega_name})
            other_domains_str = ", ".join(f'"{d}"' for d in other_domains) if other_domains else "(none)"
            
            split_prompt = f"""You are a business domain expert. The domain "{mega_name}" is too large and must be split.

**CURRENT SITUATION**:
- Domain "{mega_name}" has {mega_count} use cases ({mega_count/total_ucs:.0%} of all {total_ucs} UCs)
- Maximum allowed per domain: {max_ucs_per_domain} UCs ({max_domain_share:.0%})
- Other existing domains: {other_domains_str}

**USE CASES IN "{mega_name}"**:
{uc_str}

**YOUR TASK**: Split these {mega_count} use cases into {target_splits}-{target_splits+1} NEW domain names.

**Output Format** (honesty-wrapped JSON):
{{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field maps use_case_id → new_domain_name.
Example:
{{{{"honesty_score": 90, "honesty_justification": "Split based on functional areas.", "data": {{{{"N01-U03": "Revenue", "N01-U06": "Inventory", "N01-U01": "Workforce"}}}}}}}}

**Rules**:
1. Every domain name must be EXACTLY ONE WORD (no spaces), unique
2. ❌ BANNED: "Operations", "Engagement", "Management", "Services", "Activities", "General"
3. ✅ Use specific business nouns: "Revenue", "Inventory", "Fraud", "Workforce", "Franchise", "Pricing", "Customers"
4. New domain names MUST NOT match any existing domain: {other_domains_str}
5. Each new domain must have at least 2 use cases
6. Distribute use cases as evenly as possible across new domains
7. Every use case must be assigned to exactly one new domain

Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON

            try:
                split_prompt_key = "DOMAIN_SPLITTER_PROMPT"
                _splitter_instr = self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT")
                split_prompt = f"{split_prompt}\n\n{_splitter_instr}\n"
                self.ai_agent.prompt_templates[split_prompt_key] = split_prompt
                
                response = self.ai_agent.run_worker(
                    step_name=f"Split_Mega_Domain_{mega_name}",
                    worker_prompt_path=split_prompt_key,
                    prompt_vars={},
                    response_schema=None
                )
                
                del self.ai_agent.prompt_templates[split_prompt_key]
                
                cleaned = clean_json_response(response)
                if not cleaned or cleaned.strip() == '':
                    self.logger.warning(f"Domain split response is empty after cleaning. Keeping '{mega_name}' as-is.")
                    continue
                split_mapping = json.loads(cleaned)
                
                if isinstance(split_mapping, dict) and 'data' in split_mapping:
                    split_mapping = split_mapping['data']
                
                if not isinstance(split_mapping, dict):
                    self.logger.warning(f"Domain split response is not a dict. Keeping '{mega_name}' as-is.")
                    continue
                
                new_domain_counts = defaultdict(int)
                for uc_id, new_domain in split_mapping.items():
                    if isinstance(new_domain, str):
                        new_domain_counts[new_domain.strip()] += 1
                
                if len(new_domain_counts) < 2:
                    self.logger.warning(f"LLM only produced {len(new_domain_counts)} domain(s) for split. Keeping '{mega_name}' as-is.")
                    continue
                
                collisions = set(new_domain_counts.keys()) & (all_domain_names - {mega_name})
                if collisions:
                    self.logger.warning(f"LLM split produced domain names that collide with existing: {collisions}. Keeping '{mega_name}' as-is.")
                    continue
                
                split_count = 0
                for uc in mega_ucs:
                    uc_id = uc.get('No', '')
                    if uc_id in split_mapping:
                        new_domain = split_mapping[uc_id].strip() if isinstance(split_mapping[uc_id], str) else None
                        if new_domain:
                            uc['Business Domain'] = new_domain
                            split_count += 1
                
                all_domain_names = (all_domain_names - {mega_name}) | set(new_domain_counts.keys())
                
                self.logger.info(
                    f"✅ Split '{mega_name}' → {len(new_domain_counts)} new domains: "
                    + ", ".join(f"'{d}' ({c} UCs)" for d, c in sorted(new_domain_counts.items(), key=lambda x: -x[1]))
                )
                
            except Exception as e:
                self.logger.error(f"Failed to split mega-domain '{mega_name}': {get_clean_error_message(e)}. Keeping as-is.")
                continue
        
        final_counts = defaultdict(int)
        for uc in use_cases:
            final_counts[uc.get('Business Domain', 'Other')] += 1
        self.logger.info(
            f"Post-split: {len(final_counts)} domains. "
            f"Distribution: {dict(sorted(final_counts.items(), key=lambda x: -x[1]))}"
        )
        
        return use_cases

    def _merge_duplicate_subdomains(self, use_cases: list) -> list:
        """
        Step 2.5: Cross-domain subdomain deduplication.
        
        After per-domain subdomain detection (Step 2), different domains may have
        independently chosen identical subdomain names (e.g., "Transaction Monitoring"
        under both Fraud and Operations). This method detects such collisions and
        uses an LLM to rename them so each subdomain name is globally unique while
        remaining descriptive of its specific domain context.
        
        Handles edge cases:
        - 2-domain collisions (most common)
        - 3+ domain collisions (e.g., "Risk Management" in Fraud, Operations, and Pricing)
        - LLM renames that collide with existing non-duplicate subdomain names
        - LLM returning invalid/empty responses → deterministic fallback always resolves
        - Ensures all renamed subdomains remain valid 2-word business names
        
        Args:
            use_cases: List of use case dicts with 'Business Domain' and 'Subdomain' set
            
        Returns:
            Same list with duplicate subdomain names resolved (modified in-place)
        """
        if not use_cases:
            return use_cases
        
        from collections import defaultdict
        
        subdomain_to_domains = defaultdict(set)
        for uc in use_cases:
            domain = uc.get('Business Domain', '').strip()
            subdomain = uc.get('Subdomain', '').strip()
            if domain and subdomain:
                subdomain_to_domains[subdomain].add(domain)
        
        duplicates = {sd: sorted(domains) for sd, domains in subdomain_to_domains.items() if len(domains) > 1}
        
        if not duplicates:
            self.logger.info("✅ No cross-domain subdomain duplicates found. Skipping subdomain merger.")
            return use_cases
        
        total_collisions = sum(len(doms) for doms in duplicates.values())
        self.logger.warning(
            f"⚠️ Found {len(duplicates)} subdomain name(s) duplicated across {total_collisions} domain instances: "
            + ", ".join(f"'{sd}' in [{', '.join(doms)}]" for sd, doms in duplicates.items())
        )
        
        domain_subdomain_ucs = defaultdict(lambda: defaultdict(list))
        for uc in use_cases:
            domain = uc.get('Business Domain', '').strip()
            subdomain = uc.get('Subdomain', '').strip()
            if domain and subdomain:
                domain_subdomain_ucs[domain][subdomain].append(
                    uc.get('Name', uc.get('No', ''))
                )
        
        all_existing_names = set()
        for domain_sds in domain_subdomain_ucs.values():
            all_existing_names.update(domain_sds.keys())
        
        dup_details_lines = []
        for sd, domains in duplicates.items():
            dup_details_lines.append(f"\n  DUPLICATE SUBDOMAIN: \"{sd}\" (appears in {len(domains)} domains)")
            for domain in domains:
                uc_names = domain_subdomain_ucs[domain].get(sd, [])
                dup_details_lines.append(
                    f"    - Domain \"{domain}\": {len(uc_names)} use cases"
                    f" → {', '.join(uc_names[:5])}"
                    + (f" ... and {len(uc_names) - 5} more" if len(uc_names) > 5 else "")
                )
        dup_details_str = "\n".join(dup_details_lines)
        
        all_subdomains_lines = []
        for domain in sorted(domain_subdomain_ucs.keys()):
            sds = sorted(domain_subdomain_ucs[domain].keys())
            for sd in sds:
                uc_names = domain_subdomain_ucs[domain][sd]
                is_dup = "⚠️ DUPLICATE" if sd in duplicates else "✓"
                all_subdomains_lines.append(
                    f"  - Domain \"{domain}\" / Subdomain \"{sd}\": "
                    f"{len(uc_names)} use cases {is_dup}"
                    f"  (UCs: {', '.join(uc_names[:3])}{'...' if len(uc_names) > 3 else ''})"
                )
        all_subdomains_str = "\n".join(all_subdomains_lines)
        
        non_dup_names = sorted(all_existing_names - set(duplicates.keys()))
        reserved_names_str = ", ".join(f'"{n}"' for n in non_dup_names) if non_dup_names else "(none)"
        
        merge_prompt = f"""You are a business domain taxonomy expert. You must resolve DUPLICATE subdomain names that appear in multiple business domains.

**CURRENT SUBDOMAIN MAP** (all domains, subdomains, and their use cases):
{all_subdomains_str}

**DUPLICATES TO RESOLVE**:
{dup_details_str}

**RESERVED NAMES** (already used by non-duplicate subdomains — your renames MUST NOT match any of these):
{reserved_names_str}

**YOUR TASK**: For each duplicate subdomain name, rename ONE OR BOTH instances so that every subdomain name is globally unique across all domains. The new names must:
1. Be EXACTLY 2 words (same constraint as original). Not 1, not 3+. EXACTLY 2 words.
2. Clearly reflect the SPECIFIC domain context and the USE CASES within that subdomain
3. NOT match any existing subdomain name (including the reserved names above)
4. NOT match any OTHER renamed subdomain you produce in this response
5. Remain business-focused and professional — no generic names like "Other Services"

**RULES**:
- Only rename subdomains that are duplicated. Do NOT rename non-duplicated subdomains.
- Prefer renaming the subdomain in the domain where it is LESS central. Use the use case names to judge centrality.
- If both are equally central, rename both to be domain-specific.
- If a subdomain appears in 3+ domains, you MUST rename ALL instances to be unique.
- No two subdomain names (across ALL domains) may be identical after your renaming.
- Validate your output: every new name must be exactly 2 words and globally unique.

**Output Format** (honesty-wrapped JSON):
Your response MUST be wrapped: {{{{"honesty_score": XX, "honesty_justification": "...", "data": <your_mapping>}}}}

The "data" field must be a JSON object mapping:
  "OldDomain:::OldSubdomain" → "NewSubdomain"

Example (2-domain collision — rename the less central one):
{{{{
  "honesty_score": 90,
  "honesty_justification": "Renamed Operations instance as Transaction Monitoring is more central to Fraud domain.",
  "data": {{{{
    "Operations:::Transaction Monitoring": "Process Oversight"
  }}}}
}}}}

Example (3-domain collision — rename all):
{{{{
  "honesty_score": 88,
  "honesty_justification": "All three domains equally use this subdomain, renamed all with domain context.",
  "data": {{{{
    "Fraud:::Risk Management": "Fraud Mitigation",
    "Operations:::Risk Management": "Operational Safeguards",
    "Pricing:::Risk Management": "Pricing Controls"
  }}}}
}}}}

Only include entries for subdomains that need renaming. Start your response with: {{{{"honesty_score":
""" + HONESTY_CHECK_JSON

        try:
            merge_prompt_key = "SUBDOMAINS_MERGER_PROMPT"
            _subdomain_instr = self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT")
            merge_prompt = f"{merge_prompt}\n\n{_subdomain_instr}\n"
            self.ai_agent.prompt_templates[merge_prompt_key] = merge_prompt
            
            self.logger.info(
                f"📍 STEP 2.5: Resolving {len(duplicates)} cross-domain subdomain name collision(s) "
                f"across {total_collisions} domain instances..."
            )
            
            response = self.ai_agent.run_worker(
                step_name="Merge_Duplicate_Subdomains",
                worker_prompt_path=merge_prompt_key,
                prompt_vars={},
                response_schema=None
            )
            
            del self.ai_agent.prompt_templates[merge_prompt_key]
            
            rename_mapping = json.loads(clean_json_response(response))
            
            if isinstance(rename_mapping, dict) and 'data' in rename_mapping:
                rename_mapping = rename_mapping['data']
            
            if not isinstance(rename_mapping, dict):
                self.logger.warning(f"Subdomain merger response is not a dict: {type(rename_mapping)}. Falling back to deterministic rename.")
                return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)
            
            sanitized = {}
            for k, v in rename_mapping.items():
                if isinstance(k, str) and isinstance(v, str) and ':::' in k:
                    sanitized[k] = v.strip()
                else:
                    self.logger.warning(f"Subdomain merger: dropping invalid entry '{k}' → '{v}' (expected 'Domain:::Subdomain' → 'NewName')")
            rename_mapping = sanitized
            
            if not rename_mapping:
                self.logger.info("Subdomain merger returned no renames. Falling back to deterministic rename.")
                return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)
            
            proposed_new_names = set()
            collision_with_existing = []
            collision_with_proposed = []
            for key, new_name in list(rename_mapping.items()):
                domain_part = key.split(':::')[0] if ':::' in key else ''
                old_sd = key.split(':::')[1] if ':::' in key else ''
                
                if new_name in all_existing_names and new_name != old_sd:
                    collision_with_existing.append((key, new_name))
                    self.logger.warning(
                        f"LLM proposed rename '{old_sd}' → '{new_name}' in domain '{domain_part}' "
                        f"collides with existing subdomain. Will use deterministic fallback for this one."
                    )
                    del rename_mapping[key]
                elif new_name in proposed_new_names:
                    collision_with_proposed.append((key, new_name))
                    self.logger.warning(
                        f"LLM proposed rename '{old_sd}' → '{new_name}' in domain '{domain_part}' "
                        f"collides with another proposed rename. Will use deterministic fallback for this one."
                    )
                    del rename_mapping[key]
                else:
                    proposed_new_names.add(new_name)
            
            llm_rename_count = 0
            for uc in use_cases:
                domain = uc.get('Business Domain', '').strip()
                subdomain = uc.get('Subdomain', '').strip()
                key = f"{domain}:::{subdomain}"
                if key in rename_mapping:
                    new_name = rename_mapping[key]
                    if new_name != subdomain:
                        uc['Subdomain'] = new_name
                        llm_rename_count += 1
            
            unique_renames = len(set(rename_mapping.values()))
            self.logger.info(
                f"✅ LLM subdomain merger: {llm_rename_count} use cases renamed "
                f"across {unique_renames} unique subdomain renames"
            )
            if collision_with_existing:
                self.logger.info(f"   ⚠️ {len(collision_with_existing)} LLM renames rejected (collided with existing names)")
            if collision_with_proposed:
                self.logger.info(f"   ⚠️ {len(collision_with_proposed)} LLM renames rejected (collided with each other)")
            
            updated_existing = set()
            for uc in use_cases:
                s = uc.get('Subdomain', '').strip()
                if s:
                    updated_existing.add(s)
            
            remaining_dups = defaultdict(set)
            for uc in use_cases:
                d = uc.get('Business Domain', '').strip()
                s = uc.get('Subdomain', '').strip()
                if d and s:
                    remaining_dups[s].add(d)
            still_dup = {sd: sorted(doms) for sd, doms in remaining_dups.items() if len(doms) > 1}
            
            if still_dup:
                self.logger.warning(
                    f"⚠️ After LLM merger, {len(still_dup)} subdomain name(s) still duplicated: "
                    + ", ".join(f"'{sd}' in [{', '.join(doms)}]" for sd, doms in still_dup.items())
                    + ". Applying deterministic fallback for remaining collisions..."
                )
                return self._deterministic_subdomain_dedup(use_cases, still_dup, updated_existing)
            
            return use_cases
        
        except Exception as e:
            self.logger.error(f"Failed to merge duplicate subdomains via LLM: {get_clean_error_message(e)}. Applying deterministic fallback...")
            return self._deterministic_subdomain_dedup(use_cases, duplicates, all_existing_names)

    def _deterministic_subdomain_dedup(self, use_cases: list, duplicates: dict, all_existing_names: set) -> list:
        """
        Deterministic fallback for cross-domain subdomain deduplication.
        Guarantees resolution of all collisions without LLM, producing valid 2-word names.
        
        Strategy: For each duplicated subdomain name, iterate through the domains that share it.
        Keep the name as-is for the domain with the MOST use cases in that subdomain (most central).
        For all other domains, generate a domain-prefixed 2-word name.
        If the generated name also collides, append a numeric suffix to the domain word.
        
        Args:
            use_cases: List of use case dicts
            duplicates: Dict mapping subdomain_name → set/list of domain names that share it
            all_existing_names: Set of all currently used subdomain names (for collision avoidance)
        
        Returns:
            Modified use_cases list with all cross-domain subdomain duplicates resolved
        """
        from collections import defaultdict
        
        domain_sd_count = defaultdict(lambda: defaultdict(int))
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            if d and s:
                domain_sd_count[d][s] += 1
        
        used_names = set(all_existing_names)
        rename_plan = {}
        
        fallback_rename_count = 0
        for sd_name, domains in duplicates.items():
            domains_list = sorted(domains) if isinstance(domains, set) else list(domains)
            
            domain_uc_counts = [(d, domain_sd_count[d].get(sd_name, 0)) for d in domains_list]
            domain_uc_counts.sort(key=lambda x: -x[1])
            
            keep_domain = domain_uc_counts[0][0]
            rename_domains = [d for d, _ in domain_uc_counts[1:]]
            
            self.logger.info(
                f"Deterministic dedup '{sd_name}': keeping in '{keep_domain}' "
                f"({domain_uc_counts[0][1]} UCs), renaming in: "
                + ", ".join(f"'{d}' ({c} UCs)" for d, c in domain_uc_counts[1:])
            )
            
            for domain in rename_domains:
                sd_words = sd_name.split()
                domain_words = domain.split()
                domain_prefix = domain_words[0] if domain_words else "Alt"
                sd_suffix = sd_words[-1] if sd_words else "Area"
                
                if domain_prefix.lower() == sd_suffix.lower():
                    sd_suffix = sd_words[0] if len(sd_words) > 1 else "Focus"
                
                candidate = f"{domain_prefix} {sd_suffix}"
                
                attempt = 0
                while candidate.lower() in {n.lower() for n in used_names} or candidate.lower() in {v.lower() for v in rename_plan.values()}:
                    attempt += 1
                    if attempt <= len(sd_words) and attempt < len(sd_words):
                        alt_suffix = sd_words[-(attempt + 1)] if len(sd_words) > attempt else sd_suffix
                        candidate = f"{domain_prefix} {alt_suffix}"
                    else:
                        candidate = f"{domain_prefix}{attempt} {sd_suffix}"
                    if attempt > 10:
                        candidate = f"{domain_prefix} Area{attempt}"
                        break
                
                rename_plan[f"{domain}:::{sd_name}"] = candidate
                used_names.add(candidate)
                self.logger.info(f"   → Domain '{domain}': '{sd_name}' → '{candidate}'")
        
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            key = f"{d}:::{s}"
            if key in rename_plan:
                uc['Subdomain'] = rename_plan[key]
                fallback_rename_count += 1
        
        self.logger.info(f"✅ Deterministic subdomain dedup: {fallback_rename_count} use cases renamed across {len(rename_plan)} renames")
        
        final_check = defaultdict(set)
        for uc in use_cases:
            d = uc.get('Business Domain', '').strip()
            s = uc.get('Subdomain', '').strip()
            if d and s:
                final_check[s].add(d)
        final_still_dup = {sd: doms for sd, doms in final_check.items() if len(doms) > 1}
        if final_still_dup:
            self.logger.error(
                f"🚨 CRITICAL: After deterministic dedup, {len(final_still_dup)} collision(s) remain: "
                + ", ".join(f"'{sd}' in [{', '.join(sorted(doms))}]" for sd, doms in final_still_dup.items())
                + ". This should never happen — please report this bug."
            )
        else:
            self.logger.info("✅ All cross-domain subdomain name collisions fully resolved.")
        
        return use_cases

    def _cluster_domains_and_subdomains(self, use_cases: list, language: str) -> list:
        """
        Cluster use cases into appropriate domains and subdomains using LLM TWO-STEP approach:
        Step 1: Assign domains to all use cases
        Step 2: For each domain, assign subdomains in parallel
        
        Args:
            use_cases: List of use case dictionaries
            language: Output language
            
        Returns:
            List of use cases with updated domains and subdomains
        """
        from collections import defaultdict
        import io
        import csv
        from concurrent.futures import ThreadPoolExecutor
        import concurrent.futures
        
        try:
            self._emit_pipeline_status(
                stage_name="Domain Mapping", step_name="Domain/Subdomain Clustering",
                sub_step_name="Started", status="started",
                message=f"Two-step clustering started for {len(use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info(f"🎯 Starting TWO-STEP domain/subdomain clustering for {len(use_cases)} use cases...")
        
        if not use_cases:
            return use_cases
        
        # === USER-PROVIDED BUSINESS DOMAINS ENFORCEMENT ===
        # If user provided business domains, we MUST force all use cases to align to those domains only
        if self.user_business_domains and len(self.user_business_domains) > 0:
            self.logger.info(f"🚨 USER-PROVIDED DOMAINS DETECTED: Forcing use cases to align ONLY to: {', '.join(self.user_business_domains)}")
            log_print(f"\n🚨 USER-PROVIDED DOMAINS: Aligning all use cases to: {', '.join(self.user_business_domains)}")
            
            # Use LLM to intelligently assign use cases to user-provided domains
            return self._assign_to_user_domains(use_cases, self.user_business_domains, language)
        
        # === NOTE: Removed legacy fallback - always use two-step approach ===
        # The two-step approach (DOMAIN_FINDER_PROMPT + SUBDOMAIN_DETECTOR_PROMPT) 
        # provides better quality and more consistent results than the old single-step approach
        if len(use_cases) > 250:
            self.logger.info(
                f"📊 Large use case set detected ({len(use_cases)} use cases). "
                f"Using two-step clustering with parallel subdomain detection for optimal quality."
            )
        
        # === STEP 1: DOMAIN DETECTION ===
        
        try:
            # Convert use cases to CSV for LLM (without Business Domain and Subdomain since those will be detected)
            output = io.StringIO()
            if use_cases:
                fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                             'Business Value', 'Beneficiary', 'Sponsor', 
                             'Tables Involved']
                writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
                writer.writeheader()
                writer.writerows(use_cases)
            use_cases_csv = output.getvalue()
            
            # Check context size for domain detection (uses model-specific limits from TECHNICAL_CONTEXT)
            prompt_template = self.ai_agent.prompt_templates.get("DOMAIN_FINDER_PROMPT", "")
            estimated_size = len(prompt_template) + len(use_cases_csv) + 1000
            MAX_CONTEXT_CHARS = get_max_context_chars(language, "DOMAIN_FINDER_PROMPT")
            
            if estimated_size > MAX_CONTEXT_CHARS:
                # === BATCHED DOMAIN DETECTION: Process in smaller chunks ===
                self.logger.warning(
                    f"Domain detection prompt size ({estimated_size:,} chars) exceeds MAX_CONTEXT_CHARS ({MAX_CONTEXT_CHARS:,}). "
                    f"Using BATCHED domain detection to process {len(use_cases)} use cases in smaller chunks."
                )
                
                batched_domain_assignments = self._run_domain_detection_batched(
                    use_cases, fieldnames, language, prompt_template, MAX_CONTEXT_CHARS
                )
                
                domain_assignments = batched_domain_assignments if batched_domain_assignments else self._assign_default_domains(use_cases)
                
                # === STEP 1.5: DOMAIN MERGING (MERGE SMALL/SIMILAR DOMAINS) ===
                self.logger.info("📍 STEP 1.5: Merging small/similar domains from batched detection...")
                domain_assignments = self._merge_small_domains(domain_assignments)
                
                # === STEP 1.6: SPLIT MEGA-DOMAINS ===
                self.logger.info("📍 STEP 1.6: Splitting mega-domains (if any)...")
                domain_assignments = self._split_mega_domains(domain_assignments)
                
                # === STEP 2: SUBDOMAIN DETECTION (PARALLEL FOR EACH DOMAIN) ===
                self.logger.info(f"📍 STEP 2: Detecting subdomains for each domain in parallel...")
                
                # Group use cases by domain
                domain_usecases_map = defaultdict(list)
                for uc in domain_assignments:
                    domain = uc.get('Business Domain', '').strip()
                    if domain:
                        domain_usecases_map[domain].append(uc)
                
                # ADAPTIVE PARALLELISM: Calculate based on domains and use cases
                subdomain_parallelism, reason = calculate_adaptive_parallelism(
                    "subdomain_detection", self.max_parallelism,
                    num_items=len(domain_assignments),
                    num_domains=len(domain_usecases_map),
                    is_llm_operation=True, logger=self.logger
                )
                log_adaptive_parallelism_decision("subdomain_detection", subdomain_parallelism, self.max_parallelism, reason)
                self.logger.info(f"Processing {len(domain_usecases_map)} domains for subdomain detection...")
                
                final_use_cases_with_subdomains = []
                
                self.logger.info(f"🔄 Processing {len(domain_usecases_map)} domains sequentially for subdomain detection...")
                import time as _time
                for idx, (domain_name, domain_use_cases) in enumerate(domain_usecases_map.items(), 1):
                    domain_start = _time.time()
                    self.logger.info(f"🔄 [{idx}/{len(domain_usecases_map)}] Domain '{domain_name}' ({len(domain_use_cases)} use cases)...")
                    log_print(f"🔄 [{idx}/{len(domain_usecases_map)}] Subdomain detection: '{domain_name}'")
                    try:
                        result = self._detect_subdomains_for_domain(domain_name, domain_use_cases, language)
                        elapsed = _time.time() - domain_start
                        if result:
                            final_use_cases_with_subdomains.extend(result)
                            self.logger.info(f"✅ [{idx}/{len(domain_usecases_map)}] Domain '{domain_name}' complete in {elapsed:.1f}s ({len(result)} use cases)")
                        else:
                            fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                            final_use_cases_with_subdomains.extend(fallback)
                            self.logger.warning(f"⚠️ [{idx}/{len(domain_usecases_map)}] Domain '{domain_name}': empty after {elapsed:.1f}s, used defaults")
                    except Exception as e:
                        elapsed = _time.time() - domain_start
                        self.logger.error(f"❌ [{idx}/{len(domain_usecases_map)}] Domain '{domain_name}' failed after {elapsed:.1f}s: {get_clean_error_message(e)}")
                        fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                        final_use_cases_with_subdomains.extend(fallback)
                
                self.logger.info(f"📍 STEP 2.5: Cross-domain subdomain deduplication (batched path)...")
                final_use_cases_with_subdomains = self._merge_duplicate_subdomains(final_use_cases_with_subdomains)
                final_use_cases_with_subdomains = self._rename_placeholder_domains_from_subdomains(final_use_cases_with_subdomains)
                
                self.logger.info(f"✅ BATCHED two-step clustering complete! {len(final_use_cases_with_subdomains)} use cases processed across {len(domain_usecases_map)} domains")
                return final_use_cases_with_subdomains
            
            # Warn if prompt is very large (might cause slowness)
            if estimated_size > MAX_CONTEXT_CHARS * 0.7:
                self.logger.warning(
                    f"⚠️ Domain detection prompt is very large ({estimated_size:,} chars, {len(use_cases)} use cases). "
                    f"This may take 2-5 minutes to process. Please be patient..."
                )
                log_print(f"\n⏳ Processing {len(use_cases)} use cases for domain detection...")
                log_print(f"   Prompt size: {estimated_size:,} characters")
                log_print(f"   This may take 2-5 minutes. Please wait...\n")
            
            max_attempts = self.max_retry_attempts + 1
            prompt_vars = {
                "use_cases_csv": use_cases_csv,
                "output_language": language,
                "business_name": self.business_name,
                "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                "business_context": self._get_business_context_fallback(),
                "previous_violations": "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT"),
            }
            
            domain_assignments = None
            for attempt in range(1, max_attempts + 1):
                try:
                    self.logger.info(f"📍 STEP 1: Detecting domains for all {len(use_cases)} use cases ({estimated_size:,} chars) - {attempt}/{max_attempts}")
                    
                    import time
                    start_time = time.time()
                    
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Detect_Domains_{language}_Attempt{attempt}",
                        worker_prompt_path="DOMAIN_FINDER_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None,
                        skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                    )
                    
                    elapsed_time = time.time() - start_time
                    self.logger.info(f"✅ LLM response received in {elapsed_time:.1f} seconds")
                    
                    # Validate response is not empty
                    if not response_raw or len(response_raw.strip()) == 0:
                        raise ValueError("LLM returned empty response")
                    
                    # Clean response (remove markdown fences if present)
                    response_clean = clean_json_response(response_raw)
                    
                    # Validate cleaned response
                    if not response_clean or len(response_clean.strip()) == 0:
                        raise ValueError("Cleaned response is empty")
                    
                    # Parse CSV using centralized utility
                    try:
                        csv_rows = CSVParser.parse_csv_string(
                            response_clean,
                            logger=self.logger,
                            context="Domain detection"
                        )
                        domain_assignment_map = {}
                        row_count = 0
                        
                        for row in csv_rows:
                            row_count += 1
                            # Handle both possible column names - also handle None values
                            uc_id_raw = row.get('use_case_id', '') or ''
                            domain_raw = row.get('domain', '') or ''
                            uc_id = uc_id_raw.strip() if isinstance(uc_id_raw, str) else str(uc_id_raw).strip()
                            domain = domain_raw.strip() if isinstance(domain_raw, str) else str(domain_raw).strip()
                            
                            if uc_id and domain:
                                domain_assignment_map[uc_id] = domain
                        
                        if row_count == 0:
                            raise ValueError("CSV has no data rows")
                        
                        if not domain_assignment_map:
                            raise ValueError("No valid domain assignments found in CSV")
                            
                    except Exception as csv_err:
                        self.logger.error(f"CSV parsing failed. Raw response (first 500 chars): {response_raw[:500]}")
                        self.logger.error(f"Cleaned response (first 500 chars): {response_clean[:500]}")
                        raise ValueError(f"Failed to parse CSV: {csv_err}")
                    
                    # Apply domain assignments to use cases
                    domain_assigned_use_cases = []
                    for uc in use_cases:
                        uc_copy = uc.copy()
                        uc_id = uc_copy.get('No', '')
                        if uc_id in domain_assignment_map:
                            uc_copy['Business Domain'] = domain_assignment_map[uc_id]
                            # Clear subdomain for now (will be assigned in step 2)
                            uc_copy['Subdomain'] = ''
                        domain_assigned_use_cases.append(uc_copy)
                    
                    # Validate domain assignments
                    violations = []
                    
                    # Group by domain
                    domain_usecases = defaultdict(list)
                    
                    for uc in domain_assigned_use_cases:
                        domain = uc.get('Business Domain', '').strip()
                        if domain:
                            domain_usecases[domain].append(uc)
                    
                    # Separate HARD violations (blocking) from SOFT warnings (acceptable)
                    hard_violations = []
                    soft_warnings = []
                    
                    # Validate domain count (3-25) - HARD LIMIT on maximum
                    total_domains = len(domain_usecases)
                    if total_domains < 3:
                        hard_violations.append(f"Only {total_domains} domains, minimum required: 3")
                    if total_domains > 25:
                        hard_violations.append(f"🚨 CRITICAL: {total_domains} domains exceeds MAXIMUM 25 (HARD LIMIT) - THIS IS UNACCEPTABLE")
                    
                    # Validate domain naming (must be 1 word) - HARD REQUIREMENT
                    # Also check for shared words between domains
                    domain_words = {}
                    for domain in domain_usecases.keys():
                        word_count = len(domain.split())
                        if word_count != 1:
                            hard_violations.append(f"Domain '{domain}' has {word_count} word(s), must be exactly 1 word")
                        else:
                            # Track words used in each domain
                            domain_word = domain.lower().strip()
                            if domain_word in domain_words:
                                hard_violations.append(f"Domains share word '{domain_word}': '{domain_words[domain_word]}' and '{domain}' - merge these domains")
                            else:
                                domain_words[domain_word] = domain
                    
                    # Validate use cases per domain (4-80) - SOFT guideline for minimum
                    total_use_cases = len(use_cases)
                    small_domains = []  # Track domains with <4 use cases
                    for domain, ucs in domain_usecases.items():
                        count = len(ucs)
                        if count < 4:
                            # Only soft warning if total use cases is small
                            if total_use_cases < 50:
                                soft_warnings.append(f"Domain '{domain}' has {count} use case(s) (acceptable for small dataset)")
                            else:
                                hard_violations.append(f"Domain '{domain}' has only {count} use case(s), minimum required: 4")
                                small_domains.append((domain, count))
                        elif count > 80:
                            hard_violations.append(f"Domain '{domain}' has {count} use case(s), maximum allowed: 80")
                    
                    violations = hard_violations  # Only hard violations cause retries
                    
                    # Log soft warnings (informational only, doesn't block)
                    if soft_warnings:
                        self.logger.info(f"ℹ️ Soft warnings (acceptable for small datasets): {'; '.join(soft_warnings[:3])}")
                    
                    # If no HARD violations, domain assignment is successful!
                    if not violations:
                        self.logger.info(f"✅ Domain detection successful on attempt {attempt}! Created {total_domains} domains")
                        if soft_warnings:
                            self.logger.info(f"ℹ️ Note: {len(soft_warnings)} domains have <4 use cases (acceptable for dataset of {total_use_cases} total use cases)")
                        domain_assignments = domain_assigned_use_cases
                        break  # Exit retry loop
                    else:
                        self.logger.warning(f"⚠️ Domain detection attempt {attempt} has {len(violations)} HARD violations")
                        if attempt == max_attempts:
                            self.logger.error(f"❌ Max attempts reached with {len(violations)} HARD violations - This should not happen!")
                            self.logger.error(f"HARD Violations: {'; '.join(violations[:5])}")
                            # Still use it but log as error
                            domain_assignments = domain_assigned_use_cases
                            break
                        else:
                            self.logger.info(f"Retrying domain detection (attempt {attempt + 1}/{max_attempts})...")
                            
                            # Prepare violation summary with actionable suggestions for next attempt
                            violation_summary = "\n\n**🚨 PREVIOUS ATTEMPT VIOLATIONS - YOU MUST FIX THESE 🚨**:\n"
                            violation_summary += "\n".join([f"- {v}" for v in violations])
                            
                            # Add special guidance for small domains (only if it's a hard violation)
                            if small_domains and total_use_cases >= 50:
                                violation_summary += "\n\n**💡 SUGGESTION FOR SMALL DOMAINS**:\n"
                                violation_summary += "Domains with fewer than 4 use cases should be merged into larger related domains.\n"
                                for small_domain, count in small_domains:
                                    violation_summary += f"  - '{small_domain}' ({count} use cases) → Merge with a related domain\n"
                            
                            # Emphasize the HARD LIMIT on maximum domains
                            if total_domains > 25:
                                violation_summary += f"\n\n**🚨 CRITICAL: MAXIMUM 25 DOMAINS IS AN ABSOLUTE HARD LIMIT 🚨**\n"
                                violation_summary += f"You created {total_domains} domains. You MUST reduce to 25 or fewer.\n"
                                violation_summary += f"Merge related domains together to stay within the limit.\n"
                            
                            # Update prompt_vars for next attempt
                            prompt_vars['previous_violations'] = violation_summary
                            continue
                
                except CascadeRebatchError as cre:
                    self.logger.warning(
                        f"🔄 Domain detection cascade rebatch needed after '{cre.failed_model_name}' failed. "
                        f"Switching to batched mode for model '{cre.target_model_name}' "
                        f"(limit: {cre.target_context_limit_chars:,} chars)"
                    )
                    batched_result = self._run_domain_detection_batched(
                        use_cases, fieldnames, language, prompt_template,
                        cre.target_context_limit_chars, cre.target_model_endpoint
                    )
                    if batched_result:
                        domain_assignments = batched_result
                    break

                except Exception as e:
                    self.logger.error(f"Domain detection attempt {attempt} failed: {get_clean_error_message(e)}")
                    if attempt == max_attempts:
                        self.logger.error("Max attempts reached. Using DEFAULT domains as fallback...")
                        domain_assignments = self._assign_default_domains(use_cases)
                        self.logger.warning(f"✅ Fallback complete: Assigned {len(set(uc.get('Business Domain', '') for uc in domain_assignments))} default domains to {len(domain_assignments)} use cases")
                        break
            
            if not domain_assignments:
                self.logger.error("Domain detection failed. Using DEFAULT domains as fallback...")
                domain_assignments = self._assign_default_domains(use_cases)
                self.logger.warning(f"✅ Fallback complete: Assigned {len(set(uc.get('Business Domain', '') for uc in domain_assignments))} default domains to {len(domain_assignments)} use cases")
            
            # === STEP 1.5: DOMAIN MERGING (MERGE SMALL DOMAINS) ===
            self.logger.info("📍 STEP 1.5: Merging small domains (if any)...")
            domain_assignments = self._merge_small_domains(domain_assignments)
            
            # === STEP 1.6: SPLIT MEGA-DOMAINS ===
            self.logger.info("📍 STEP 1.6: Splitting mega-domains (if any)...")
            domain_assignments = self._split_mega_domains(domain_assignments)
            
            # === STEP 2: SUBDOMAIN DETECTION (PARALLEL WITH SERIAL FALLBACK) ===
            self.logger.info(f"📍 STEP 2: Detecting subdomains for each domain...")
            
            domain_usecases_map = defaultdict(list)
            for uc in domain_assignments:
                domain = uc.get('Business Domain', '').strip()
                if domain:
                    domain_usecases_map[domain].append(uc)
            
            subdomain_parallelism, reason = calculate_adaptive_parallelism(
                "subdomain_detection", self.max_parallelism,
                num_items=len(domain_assignments),
                num_domains=len(domain_usecases_map),
                is_llm_operation=True, logger=self.logger
            )
            log_adaptive_parallelism_decision("subdomain_detection", subdomain_parallelism, self.max_parallelism, reason)
            self.logger.info(f"Processing {len(domain_usecases_map)} domains for subdomain detection...")
            
            final_use_cases_with_subdomains = []
            parallel_succeeded = False
            
            gc.collect()
            try:
                import psutil, os as _os
                proc = psutil.Process(_os.getpid())
                mem_mb = proc.memory_info().rss / (1024 * 1024)
                self.logger.info(f"📊 MEMORY before subdomain detection: {mem_mb:.0f} MB (RSS)")
                log_print(f"📊 Memory usage: {mem_mb:.0f} MB")
            except ImportError:
                import resource
                mem_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
                self.logger.info(f"📊 MEMORY before subdomain detection: {mem_mb:.0f} MB (maxrss)")
            
            _thread_diag = []
            for t in threading.enumerate():
                _thread_diag.append(f"{t.name}(alive={t.is_alive()}, daemon={t.daemon})")
            self.logger.info(f"🧵 Thread dump before subdomain detection ({len(_thread_diag)} threads): {', '.join(_thread_diag[:20])}")
            
            num_domains = len(domain_usecases_map)
            per_domain_timeout = self.llm_timeout_seconds + 120
            
            if num_domains >= 2:
                subdomain_workers = min(subdomain_parallelism, num_domains)
                total_timeout = per_domain_timeout * ((num_domains + subdomain_workers - 1) // subdomain_workers)
                self.logger.info(
                    f"🔄 PARALLEL SUBDOMAIN DETECTION: {subdomain_workers} workers for {num_domains} domains "
                    f"(per-domain timeout: {per_domain_timeout}s, total timeout: {total_timeout}s)"
                )
                log_print(f"🔄 Subdomain detection: {num_domains} domains with {subdomain_workers} workers")
                
                executor = None
                try:
                    executor = ThreadPoolExecutor(max_workers=subdomain_workers, thread_name_prefix="SubdomainDetect")
                    future_to_domain = {}
                    for domain_name, domain_use_cases in domain_usecases_map.items():
                        self.logger.info(f"📤 Submitting domain '{domain_name}' ({len(domain_use_cases)} use cases) to thread pool...")
                        future = executor.submit(
                            self._detect_subdomains_for_domain,
                            domain_name,
                            domain_use_cases,
                            language
                        )
                        future_to_domain[future] = domain_name
                    
                    completed_domains = set()
                    import time as _time
                    try:
                        for future in safe_as_completed(future_to_domain, timeout=total_timeout):
                            domain_name = future_to_domain[future]
                            try:
                                use_cases_with_subdomains = future.result(timeout=per_domain_timeout)
                                if use_cases_with_subdomains:
                                    self.logger.info(f"✅ Domain '{domain_name}': {len(use_cases_with_subdomains)} use cases with subdomains")
                                    final_use_cases_with_subdomains.extend(use_cases_with_subdomains)
                                else:
                                    self.logger.warning(f"⚠️ Domain '{domain_name}': Empty result - assigning defaults")
                                    fallback = self._assign_default_subdomains(domain_usecases_map[domain_name], domain_name)
                                    final_use_cases_with_subdomains.extend(fallback)
                                completed_domains.add(domain_name)
                            except Exception as e:
                                self.logger.error(f"❌ Domain '{domain_name}': Failed: {get_clean_error_message(e)}")
                                fallback = self._assign_default_subdomains(domain_usecases_map[domain_name], domain_name)
                                final_use_cases_with_subdomains.extend(fallback)
                                completed_domains.add(domain_name)
                    except concurrent.futures.TimeoutError:
                        timed_out_domains = [d for d in domain_usecases_map if d not in completed_domains]
                        self.logger.error(
                            f"⏱️ Progressive parallel timed out after {total_timeout}s. "
                            f"Completed {len(completed_domains)}/{num_domains}. Timed-out: {timed_out_domains}"
                        )
                        _alive_threads = [(t.name, t.is_alive()) for t in threading.enumerate() if 'SubdomainDetect' in t.name or 'LLM_Query' in t.name]
                        self.logger.error(f"🧵 Stuck threads: {_alive_threads}")
                        
                        for future in future_to_domain:
                            if not future.done():
                                future.cancel()
                        
                        for domain_name in timed_out_domains:
                            self.logger.warning(f"⏱️ Domain '{domain_name}' timed out — assigning defaults")
                            fallback = self._assign_default_subdomains(domain_usecases_map[domain_name], domain_name)
                            final_use_cases_with_subdomains.extend(fallback)
                        parallel_succeeded = True
                    else:
                        parallel_succeeded = True
                except Exception as e:
                    self.logger.error(f"❌ Progressive parallel failed entirely: {get_clean_error_message(e)}")
                    _alive_threads = [(t.name, t.is_alive()) for t in threading.enumerate() if 'SubdomainDetect' in t.name or 'LLM_Query' in t.name]
                    self.logger.error(f"🧵 Thread state after failure: {_alive_threads}")
                    final_use_cases_with_subdomains = []
                else:
                    if completed_domains:
                        parallel_succeeded = True
                finally:
                    if executor:
                        try:
                            executor.shutdown(wait=False, cancel_futures=True)
                        except TypeError:
                            executor.shutdown(wait=False)
                        gc.collect()
            
            if not parallel_succeeded:
                self.logger.info(f"🔄 SEQUENTIAL FALLBACK: Processing {num_domains} domains one at a time...")
                log_print(f"🔄 Subdomain detection: switching to sequential mode ({num_domains} domains)")
                final_use_cases_with_subdomains = []
                
                import time as _time
                for idx, (domain_name, domain_use_cases) in enumerate(domain_usecases_map.items(), 1):
                    domain_start = _time.time()
                    self.logger.info(f"🔄 [{idx}/{num_domains}] Processing domain '{domain_name}' ({len(domain_use_cases)} use cases)...")
                    log_print(f"🔄 [{idx}/{num_domains}] Subdomain detection: '{domain_name}' ({len(domain_use_cases)} use cases)")
                    try:
                        result = self._detect_subdomains_for_domain(domain_name, domain_use_cases, language)
                        elapsed = _time.time() - domain_start
                        if result:
                            final_use_cases_with_subdomains.extend(result)
                            self.logger.info(f"✅ [{idx}/{num_domains}] Domain '{domain_name}' complete in {elapsed:.1f}s ({len(result)} use cases)")
                        else:
                            fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                            final_use_cases_with_subdomains.extend(fallback)
                            self.logger.warning(f"⚠️ [{idx}/{num_domains}] Domain '{domain_name}': empty result after {elapsed:.1f}s, used defaults")
                    except Exception as e:
                        elapsed = _time.time() - domain_start
                        self.logger.error(f"❌ [{idx}/{num_domains}] Domain '{domain_name}' failed after {elapsed:.1f}s: {get_clean_error_message(e)}")
                        fallback = self._assign_default_subdomains(domain_use_cases, domain_name)
                        final_use_cases_with_subdomains.extend(fallback)
            
            self.logger.info(f"📍 STEP 2.5: Cross-domain subdomain deduplication...")
            final_use_cases_with_subdomains = self._merge_duplicate_subdomains(final_use_cases_with_subdomains)
            
            self.logger.info(f"✅ Two-step clustering complete! {len(final_use_cases_with_subdomains)} use cases processed")
            return final_use_cases_with_subdomains
        
        except Exception as e:
            self.logger.error(f"Domain/subdomain clustering failed with error: {get_clean_error_message(e)}. Using DEFAULT domains/subdomains as fallback...")
            # FALLBACK: Assign default domains and subdomains on any error
            fallback_use_cases = self._assign_default_domains(use_cases)
            # Also assign default subdomains for each domain
            domain_map = {}
            for uc in fallback_use_cases:
                domain = uc.get('Business Domain', 'Domain1')
                if domain not in domain_map:
                    domain_map[domain] = []
                domain_map[domain].append(uc)
            
            final_use_cases = []
            for domain_name, domain_ucs in domain_map.items():
                subdomained_ucs = self._assign_default_subdomains(domain_ucs, domain_name)
                final_use_cases.extend(subdomained_ucs)
            
            self.logger.warning(f"✅ Complete fallback applied: {len(final_use_cases)} use cases with default domains and subdomains")
            return final_use_cases

    def _run_domain_detection_batched(self, use_cases, fieldnames, language, prompt_template, context_limit, start_from_model=None):
        """
        Run domain detection in batches mirroring the same grouping used during use case generation.
        
        Strategy: Group use cases by their table schema prefix (catalog.schema from "Tables Involved"),
        which mirrors how the DataLoader batches tables during generation. Each schema-group is processed
        as a separate domain detection call, then domains are merged across groups.
        
        If a schema-group is still too large for the context window, it is further split by individual
        table prefixes (catalog.schema.table) within that schema.
        
        Returns list of domain-assigned use cases, or None on total failure.
        """
        import io
        import csv
        from collections import defaultdict

        def _extract_schema_prefix(tables_involved_str):
            if not tables_involved_str or not isinstance(tables_involved_str, str):
                return '_no_tables_'
            first_table = tables_involved_str.split(',')[0].strip().strip('`')
            parts = first_table.split('.')
            if len(parts) >= 2:
                return f"{parts[0]}.{parts[1]}"
            return first_table

        def _build_batch_csv(batch_ucs):
            buf = io.StringIO()
            w = csv.DictWriter(buf, fieldnames=fieldnames, extrasaction='ignore')
            w.writeheader()
            w.writerows(batch_ucs)
            return buf.getvalue()

        def _estimate_prompt_size(batch_csv_text):
            return len(prompt_template) + len(batch_csv_text) + 5000

        def _detect_domains_for_group(group_ucs, group_label, effective_limit_chars, effective_model, discovered_domains):
            batch_csv = _build_batch_csv(group_ucs)
            prompt_size = _estimate_prompt_size(batch_csv)
            
            if prompt_size > effective_limit_chars:
                table_groups = defaultdict(list)
                for uc in group_ucs:
                    tables = uc.get('Tables Involved', '')
                    first_table = tables.split(',')[0].strip().strip('`') if tables else '_unknown_'
                    table_groups[first_table].append(uc)
                
                if len(table_groups) <= 1:
                    self.logger.warning(f"⚠️ [{group_label}] Single table group still exceeds context ({prompt_size:,} > {effective_limit_chars:,}). Processing anyway.")
                else:
                    self.logger.info(f"📦 [{group_label}] Group too large ({prompt_size:,} chars, {len(group_ucs)} UCs). Splitting into {len(table_groups)} table sub-groups.")
                    sub_results = []
                    for sub_idx, (table_key, sub_ucs) in enumerate(table_groups.items(), 1):
                        sub_label = f"{group_label}_T{sub_idx}"
                        sub_result = _detect_domains_for_group(sub_ucs, sub_label, effective_limit_chars, effective_model, discovered_domains)
                        sub_results.extend(sub_result)
                    return sub_results
            
            domain_context = ""
            if discovered_domains:
                domain_context = f"\n\n**PREVIOUSLY DISCOVERED DOMAINS (reuse these where appropriate):**\n{', '.join(sorted(discovered_domains))}\n"
            
            batch_prompt_vars = {
                "use_cases_csv": batch_csv,
                "output_language": language,
                "business_name": self.business_name,
                "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                "business_context": self._get_business_context_fallback() + domain_context,
                "previous_violations": "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("DOMAIN_FINDER_PROMPT"),
            }
            
            try:
                batch_response = self.ai_agent.run_worker(
                    step_name=f"Detect_Domains_{group_label}_{language}",
                    worker_prompt_path="DOMAIN_FINDER_PROMPT",
                    prompt_vars=batch_prompt_vars,
                    response_schema=None,
                    start_from_model=effective_model
                )
            except CascadeRebatchError as cre:
                self.logger.warning(
                    f"🔄 [{group_label}] Cascade rebatch → model '{cre.target_model_name}' "
                    f"(limit: {cre.target_context_limit_chars:,} chars). Re-splitting group."
                )
                return _detect_domains_for_group(
                    group_ucs, group_label, cre.target_context_limit_chars,
                    cre.target_model_endpoint, discovered_domains
                )
            except Exception as e:
                self.logger.error(f"❌ [{group_label}] Domain detection failed: {get_clean_error_message(e)}. Using 'Uncategorized'.")
                results = []
                for uc in group_ucs:
                    uc_copy = uc.copy()
                    uc_copy['Business Domain'] = 'Uncategorized'
                    uc_copy['Subdomain'] = ''
                    results.append(uc_copy)
                return results
            
            results = []
            if batch_response and batch_response.strip():
                batch_response_clean = clean_json_response(batch_response)
                batch_csv_rows = CSVParser.parse_csv_string(
                    batch_response_clean, logger=self.logger,
                    context=f"Domain detection {group_label}"
                )
                batch_domain_map = {}
                for row in batch_csv_rows:
                    uc_id_raw = row.get('use_case_id', '') or ''
                    domain_raw = row.get('domain', '') or ''
                    uc_id = uc_id_raw.strip() if isinstance(uc_id_raw, str) else str(uc_id_raw).strip()
                    domain = domain_raw.strip() if isinstance(domain_raw, str) else str(domain_raw).strip()
                    if uc_id and domain:
                        batch_domain_map[uc_id] = domain
                        discovered_domains.add(domain)
                
                for uc in group_ucs:
                    uc_copy = uc.copy()
                    uc_id = uc_copy.get('No', '')
                    if uc_id in batch_domain_map:
                        uc_copy['Business Domain'] = batch_domain_map[uc_id]
                    elif batch_domain_map:
                        from collections import Counter
                        most_common = Counter(batch_domain_map.values()).most_common(1)[0][0]
                        uc_copy['Business Domain'] = most_common
                    else:
                        uc_copy['Business Domain'] = 'Uncategorized'
                    uc_copy['Subdomain'] = ''
                    results.append(uc_copy)
                self.logger.info(f"✅ [{group_label}] Assigned domains to {len(group_ucs)} use cases ({len(set(batch_domain_map.values()))} domains found)")
            else:
                self.logger.warning(f"⚠️ [{group_label}] Empty LLM response. Using 'Uncategorized'.")
                for uc in group_ucs:
                    uc_copy = uc.copy()
                    uc_copy['Business Domain'] = 'Uncategorized'
                    uc_copy['Subdomain'] = ''
                    results.append(uc_copy)
            return results

        schema_groups = defaultdict(list)
        for uc in use_cases:
            prefix = _extract_schema_prefix(uc.get('Tables Involved', ''))
            schema_groups[prefix].append(uc)
        
        self.logger.info(
            f"📦 BATCHED DOMAIN DETECTION (generation-aligned): {len(use_cases)} use cases "
            f"across {len(schema_groups)} schema groups"
        )
        for prefix, group_ucs in sorted(schema_groups.items(), key=lambda x: len(x[1]), reverse=True):
            self.logger.info(f"   📁 {prefix}: {len(group_ucs)} use cases")

        all_discovered_domains = set()
        completed_assignments = []
        effective_limit_chars = context_limit
        effective_model = start_from_model
        
        for group_idx, (schema_prefix, group_ucs) in enumerate(sorted(schema_groups.items()), 1):
            group_label = f"Schema{group_idx}_{schema_prefix.replace('.', '_')}"
            self.logger.info(f"📍 GROUP {group_idx}/{len(schema_groups)}: '{schema_prefix}' ({len(group_ucs)} use cases)...")
            
            group_results = _detect_domains_for_group(
                group_ucs, group_label, effective_limit_chars, effective_model, all_discovered_domains
            )
            completed_assignments.extend(group_results)
        
        if completed_assignments:
            self.logger.info(
                f"📦 BATCHED DOMAIN DETECTION COMPLETE: {len(completed_assignments)} use cases "
                f"assigned to {len(all_discovered_domains)} domains"
            )
            return completed_assignments

        self.logger.error("All schema-group domain detection attempts failed")
        return None

    def _phaseG_validate_schema_completeness(self, use_cases: list) -> list:
        """Phase G2: drop UCs missing any required field."""
        required = ("Name", "Statement", "Solution", "Tables Involved", "Analytics Technique")
        kept = []
        dropped = 0
        for uc in use_cases:
            missing = [f for f in required if not (uc.get(f) or "").strip()]
            if missing:
                dropped += 1
                self.logger.info(f"G schema-completeness: dropping UC {uc.get('No','UNK')} — empty {missing}")
                try:
                    NEXT_ACTIONS.add("UC_SCHEMA_INCOMPLETE", severity="HIGH",
                                     phase="G2", step="validate_schema_completeness",
                                     evidence=f"UC {uc.get('No','UNK')} empty: {missing}")
                except Exception:
                    pass
                continue
            kept.append(uc)
        if dropped:
            self.logger.warning(f"G2 schema-completeness: dropped {dropped} incomplete UCs")
        return kept

    def _phaseG_validate_name_uniqueness(self, use_cases: list) -> list:
        """Phase G2: de-duplicate on UC name within session (keep highest Inspire Score)."""
        if not use_cases:
            return use_cases
        seen = {}
        for uc in use_cases:
            name = (uc.get("Name") or "").strip().lower()
            if not name:
                seen[id(uc)] = uc
                continue
            try:
                score = float(uc.get("Inspire Score", uc.get("Priority", 0)) or 0)
            except Exception:
                score = 0.0
            prev = seen.get(name)
            if prev is None:
                seen[name] = uc
            else:
                try:
                    prev_score = float(prev.get("Inspire Score", prev.get("Priority", 0)) or 0)
                except Exception:
                    prev_score = 0.0
                if score > prev_score:
                    seen[name] = uc
        kept = list(seen.values())
        dropped = len(use_cases) - len(kept)
        if dropped:
            self.logger.info(f"G2 name-uniqueness: dropped {dropped} UCs with duplicate names")
        return kept

    def _phaseG_validate_no_catchall_technique(self, use_cases: list) -> list:
        """Phase G2 / Rule 12 enforcement: drop UCs with catch-all technique names."""
        banned = {"ai analysis", "ml analysis", "general analytics",
                  "data science", "ai", "statistical analysis", "advanced analytics"}
        kept, dropped = [], 0
        for uc in use_cases:
            t = (uc.get("Analytics Technique") or "").strip().lower()
            if t in banned:
                dropped += 1
                self.logger.info(f"G2 catch-all: dropping UC {uc.get('No','UNK')} with technique='{t}'")
                try:
                    NEXT_ACTIONS.add("UC_CATCHALL_TECHNIQUE", severity="HIGH",
                                     phase="G2", step="validate_no_catchall_technique",
                                     evidence=f"UC {uc.get('No','UNK')} technique={t}")
                except Exception:
                    pass
                continue
            kept.append(uc)
        if dropped:
            self.logger.warning(f"G2 catch-all: dropped {dropped} UCs with umbrella technique names")
        return kept

    def _phaseG_validate_column_grounding(self, use_cases: list, full_schema_details: list) -> list:
        """Phase G2 / B10: verify every column referenced in Solution (backticked)
        exists on a table in Tables Involved. Does not drop UCs (informational);
        marks them with a HIGH NEXT_ACTIONS entry when a reference is ungrounded.

        Excludes backticked identifiers that are KNOWN table names (bare or
        FQN-suffix) — UCs legitimately cite tables in prose and those aren't
        column references."""
        if not use_cases or not full_schema_details:
            return use_cases
        cols_by_fqtn = {}
        known_tables = set()
        for detail in full_schema_details:
            try:
                (c, s, t, col, _, _) = detail
            except Exception:
                continue
            fqtn = f"{c}.{s}.{t}"
            cols_by_fqtn.setdefault(fqtn, set()).add(col)
            known_tables.add(t)
            known_tables.add(t.lower())
            known_tables.add(fqtn)
            known_tables.add(fqtn.lower())
        import re as _re
        SQL_KW = {"SELECT","FROM","WHERE","AND","OR","NOT","NULL","TRUE","FALSE","IS",
                  "ORDER","GROUP","BY","HAVING","LIMIT","CASE","WHEN","THEN","ELSE","END",
                  "JOIN","INNER","OUTER","LEFT","RIGHT","ON","AS","DISTINCT"}
        for uc in use_cases:
            tabs = [t.strip().strip("`") for t in (uc.get("Tables Involved","") or "").split(",") if t.strip()]
            allowed_cols = set()
            for t in tabs:
                allowed_cols |= cols_by_fqtn.get(t, set())
                allowed_cols |= cols_by_fqtn.get(t.lower(), set())
            if not allowed_cols:
                continue
            blob = " ".join(uc.get(f, "") or "" for f in ("Solution", "Statement", "Business Value"))
            cited = set(_re.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`", blob))
            ungrounded = []
            for c in cited:
                if c in allowed_cols:
                    continue
                if c.upper() in SQL_KW:
                    continue
                # Skip table names and FQN-like refs (they are legitimate prose)
                if c in known_tables or c.lower() in known_tables:
                    continue
                if "." in c:
                    # Possibly FQN form: ``sch.tbl`` or ``cat.sch.tbl`` — only flag
                    # if the LAST segment is not a known table and not a known column.
                    last = c.rsplit(".", 1)[-1]
                    if last in known_tables or last.lower() in known_tables:
                        continue
                    if last in allowed_cols:
                        continue
                ungrounded.append(c)
            if ungrounded:
                try:
                    NEXT_ACTIONS.add("UC_COLUMN_UNGROUNDED", severity="HIGH",
                                     phase="G2", step="validate_column_grounding",
                                     evidence=f"UC {uc.get('No','UNK')}: columns not in schema: {ungrounded[:5]}")
                except Exception:
                    pass
        return use_cases

    def _phaseE0_catalog_drift_check(self, session_id):
        """Phase E0 (Gap 4): at Generate entry, flag UCs whose cited tables no longer resolve."""
        if not self._tracking_table_name:
            return []
        try:
            uc_rows = self.spark.sql(
                f"SELECT id, use_case, tables_involved FROM {self._tracking_table_name} "
                f"WHERE session_id = {int(session_id)}"
            ).collect()
        except Exception as e:
            self.logger.warning(f"E0 drift check: failed to read UCs: {e}")
            return []
        drifted = []
        for r in uc_rows:
            tabs_str = (getattr(r, "tables_involved", "") or "").strip()
            if not tabs_str:
                continue
            for t in tabs_str.split(","):
                t = t.strip().strip("`")
                if not t:
                    continue
                parts = t.split(".")
                if len(parts) != 3:
                    continue
                catalog, schema, table = parts
                try:
                    count = self.spark.sql(
                        f"SELECT COUNT(*) AS c FROM {catalog}.information_schema.tables "
                        f"WHERE table_schema = '{schema}' AND table_name = '{table}'"
                    ).collect()[0]["c"]
                    if count == 0:
                        drifted.append({"id": r.id, "table": t})
                        try:
                            NEXT_ACTIONS.add(
                                "UC_STALE_CATALOG_DRIFT", severity="HIGH",
                                phase="E0", step="catalog_drift_check",
                                evidence=f"UC {r.id} references {t} which is no longer in schema",
                                suggested_user_action="Rerun Discover to refresh UCs."
                            )
                        except Exception:
                            pass
                        break
                except Exception as e:
                    self.logger.debug(f"E0 drift probe failed for {t}: {e}")
                    continue
        if drifted:
            self.logger.warning(f"E0 CATALOG DRIFT: {len(drifted)} UC(s) reference tables no longer in schema")
        else:
            self.logger.info("E0 CATALOG DRIFT: no drift detected")
        return drifted

    def _phaseO_persist_run_artifacts(self):
        """Phase O: disabled — artifacts removed per user request (next_actions.jsonl,
        manifest.json, NEXT_ACTIONS_REPORT.md were noise). No-op."""
        return
        try:
            import os as _jl_os, json as _jl_json, base64 as _jl_b64
            from databricks.sdk.service import workspace as _jl_ws
            docs_dir = getattr(self, "docs_output_dir", None)
            if not docs_dir:
                return
            na_entries = []
            try:
                na_entries = list(getattr(NEXT_ACTIONS, "_actions", [])) if NEXT_ACTIONS else []
            except Exception:
                na_entries = []
            manifest = {
                "session_id": getattr(self, "session_id", None),
                "business_name": getattr(self, "business_name", None),
                "inspire_database": getattr(self, "inspire_database", None),
                "run_ts": _utc_iso8601(),
                "next_actions_total": len(na_entries),
                "next_actions_by_severity": {},
            }
            try:
                manifest["q1_grounding_dropped"] = self.processing_honesty.get("q1_grounding_dropped")
                manifest["q2_dedup_dropped"] = self.processing_honesty.get("q2_dedup_dropped")
                manifest["q3_relevance_dropped"] = self.processing_honesty.get("q3_relevance_dropped")
                manifest["phase_L_results"] = self.processing_honesty.get("phase_L_results")
                manifest["phase_L_fail_count"] = self.processing_honesty.get("phase_L_fail_count")
            except Exception:
                pass
            for a in na_entries:
                sev = a.get("severity", "INFO")
                manifest["next_actions_by_severity"][sev] = manifest["next_actions_by_severity"].get(sev, 0) + 1
            manifest_bytes = _jl_json.dumps(manifest, ensure_ascii=False, indent=2, default=str).encode("utf-8")
            na_text = ""
            for a in na_entries:
                na_text += _jl_json.dumps(a, ensure_ascii=False, default=str) + "\n"
            na_bytes = na_text.encode("utf-8")
            def _put(path, payload):
                try:
                    self.w_client.workspace.mkdirs(docs_dir)
                    self.w_client.workspace.import_(
                        path=path, overwrite=True,
                        format=_jl_ws.ImportFormat.AUTO,
                        content=_jl_b64.b64encode(payload).decode()
                    )
                    self.logger.info(f"Phase O: persisted {path}")
                except Exception as _e:
                    self.logger.warning(f"Phase O: failed to persist {path}: {get_clean_error_message(_e)}")
            _put(_jl_os.path.join(docs_dir, "next_actions.jsonl"), na_bytes)
            _put(_jl_os.path.join(docs_dir, "manifest.json"), manifest_bytes)
            # Phase H7: human-readable NEXT_ACTIONS_REPORT.md
            md_lines = [f"# Inspire Run — Next Actions Report", ""]
            md_lines.append(f"- Session: `{getattr(self, 'session_id', 'UNKNOWN')}`")
            md_lines.append(f"- Business: `{getattr(self, 'business_name', 'UNKNOWN')}`")
            md_lines.append(f"- Generated: {manifest.get('run_ts','?')}")
            md_lines.append("")
            md_lines.append("## Severity summary")
            for _sev in ("BLOCKING", "HIGH", "SAFE_IGNORE", "INFO"):
                _c = manifest.get("next_actions_by_severity", {}).get(_sev, 0)
                md_lines.append(f"- **{_sev}**: {_c}")
            md_lines.append("")
            _blocking = [a for a in na_entries if a.get("severity") == "BLOCKING"]
            if _blocking:
                md_lines.append("## BLOCKING actions")
                for _a in _blocking:
                    md_lines.append(f"- `{_a.get('rule_id','?')}` ({_a.get('phase','?')}::{_a.get('step','?')}) — {(_a.get('evidence','') or '')[:200]}")
                    _sa = _a.get("suggested_user_action")
                    if _sa:
                        md_lines.append(f"    - Suggested: {_sa[:200]}")
                md_lines.append("")
            _high = [a for a in na_entries if a.get("severity") == "HIGH"]
            if _high:
                md_lines.append("## HIGH (investigate)")
                for _a in _high[:50]:
                    md_lines.append(f"- `{_a.get('rule_id','?')}` — {(_a.get('evidence','') or '')[:200]}")
                if len(_high) > 50:
                    md_lines.append(f"- ... +{len(_high)-50} more")
                md_lines.append("")
            md_lines.append("## Integrity gate summary")
            md_lines.append(f"- Q1 GROUNDING dropped: {manifest.get('q1_grounding_dropped', 'n/a')}")
            md_lines.append(f"- Q2 DEDUP dropped: {manifest.get('q2_dedup_dropped', 'n/a')}")
            md_lines.append("")
            md_lines.append("## Artifacts in this folder")
            md_lines.append("- `next_actions.jsonl` — raw action entries (one JSON per line)")
            md_lines.append("- `manifest.json` — run-level summary")
            md_lines.append("- `generation_log.txt` — full run log")
            md_bytes = ("\n".join(md_lines) + "\n").encode("utf-8")
            _put(_jl_os.path.join(docs_dir, "NEXT_ACTIONS_REPORT.md"), md_bytes)
        except Exception as _err:
            try:
                self.logger.warning(f"Phase O non-fatal: {get_clean_error_message(_err)}")
            except Exception:
                pass

    def _phaseG_run_quality_gates(self, use_cases: list, full_schema_details: list) -> list:
        """Run G2 deterministic quality gates. Idempotent. Non-fatal on exception."""
        if not use_cases:
            return use_cases
        before = len(use_cases)
        use_cases = self._phaseG_validate_schema_completeness(use_cases)
        use_cases = self._phaseG_validate_name_uniqueness(use_cases)
        use_cases = self._phaseG_validate_no_catchall_technique(use_cases)
        use_cases = self._phaseG_validate_column_grounding(use_cases, full_schema_details)
        self.logger.info(f"Phase G gates: {before} -> {len(use_cases)} use cases")
        return use_cases

    def _phaseQ1_validate_grounding(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q1 GROUNDING validator (deterministic).

        Returns (kept, dropped) where `dropped` is a list of
        (use_case, failure_reason) tuples and `kept` are UCs with every
        catalog.schema.table in Tables Involved resolving to the supplied
        schema. No LLM. Industry-agnostic. Idempotent.
        """
        if not use_cases or not full_schema_details:
            return use_cases, []
        available_tables = set()
        for detail in full_schema_details:
            try:
                (catalog, schema, table, _, _, _) = detail
            except Exception:
                continue
            fqn = f"{catalog}.{schema}.{table}"
            available_tables.add(fqn)
            available_tables.add(fqn.lower())
        global_tables = getattr(self, "global_table_names", set())
        for t in global_tables:
            available_tables.add(t); available_tables.add(t.lower())
        # Extend with the FULL discovered table universe (includes tables
        # filtered as TECHNICAL / not-elected for UC generation — such tables
        # are NOT forgotten; they remain joinable/referenceable). A UC that
        # joins against a reference/technical table is valid as long as that
        # table actually exists in the passed schema. Per user 2026-04-19.
        try:
            for tup in getattr(self.data_loader, "all_table_tuples", []) or []:
                try:
                    cat, sch, tbl = tup[0], tup[1], tup[2]
                    fqn = f"{cat}.{sch}.{tbl}"
                    available_tables.add(fqn); available_tables.add(fqn.lower())
                except Exception:
                    continue
        except Exception:
            pass
        kept, dropped = [], []
        for uc in use_cases:
            tabs_str = (uc.get("Tables Involved", "") or "").strip()
            if not tabs_str:
                dropped.append((uc, "EMPTY_TABLES_INVOLVED"))
                continue
            # Volume paths (ai_parse_document use cases) pass through
            if tabs_str.startswith("/Volumes"):
                kept.append(uc); continue
            missing = []
            for raw_t in tabs_str.split(","):
                t = raw_t.strip().strip("`")
                if not t: continue
                if t not in available_tables and t.lower() not in available_tables:
                    missing.append(t)
            if missing:
                dropped.append((uc, f"UC_NOT_GROUNDED: missing {missing[:3]}"))
            else:
                kept.append(uc)
        if dropped:
            self.logger.warning(
                f"Q1 GROUNDING: dropped {len(dropped)} of {len(use_cases)} use cases "
                f"with ungrounded tables"
            )
            for uc, reason in dropped[:5]:
                self.logger.info(f"  dropped {uc.get('No', 'UNK')}: {reason}")
        else:
            self.logger.info(
                f"Q1 GROUNDING: all {len(use_cases)} use cases grounded"
            )
        return kept, dropped

    def _phaseQ2_canonical_dedup(self, use_cases: list) -> tuple:
        """Phase Q2 Pass A — canonical-signature dedup (deterministic).

        Signature = (sorted Tables Involved, Analytics Technique lowercase,
        first 6 alphabetic tokens of Statement lowercase). Exact matches
        are reduced to the highest inspire_score, ties broken by earliest id.
        Returns (kept, merge_log).
        """
        import re as _re
        if not use_cases:
            return use_cases, []
        def sig(uc):
            tabs = tuple(sorted(
                t.strip() for t in (uc.get("Tables Involved", "") or "").split(",")
                if t.strip()
            ))
            # Technique deliberately EXCLUDED — same decision with different
            # techniques is a dup, and we elect the highest-scored one.
            nouns = tuple(
                _re.findall(r"[a-z]+", (uc.get("Statement", "") or "").lower())[:6]
            )
            return (tabs, nouns)
        groups = {}
        for uc in use_cases:
            groups.setdefault(sig(uc), []).append(uc)
        kept, merge_log = [], []
        for s, group in groups.items():
            if len(group) == 1:
                kept.append(group[0])
                continue
            # sort by inspire_score desc, then id asc
            def score_of(x):
                try:
                    return float(x.get("Inspire Score", x.get("Priority", 0)) or 0)
                except Exception:
                    return 0.0
            group.sort(key=lambda x: (-score_of(x), str(x.get("No", "zzz"))))
            winner = group[0]
            kept.append(winner)
            for loser in group[1:]:
                merge_log.append({
                    "kept": winner.get("No", "UNK"),
                    "dropped": loser.get("No", "UNK"),
                    "reason": "EXACT_CANONICAL_DUPLICATE",
                    "tables": list(s[0]),
                    "technique": s[1],
                })
        # Pass A complete — now run Q2 Pass B name-canonical dedup. Catches UCs that
        # survived Pass A because technique or statement differs, but share the same
        # normalized NAME + overlapping tables. Industry-agnostic — uses the same
        # verb-synonym map that appears in the prompt D8 Layer-1 normalization.
        def _normalize_name(name: str) -> str:
            name_lc = (name or "").strip().lower()
            if not name_lc:
                return ""
            # verb-synonym collapse (universal English business-analytics verbs)
            verb_map = {
                "forecast": "PREDICT", "predict": "PREDICT", "anticipate": "PREDICT",
                "envision": "PREDICT", "estimate": "PREDICT", "project": "PREDICT",
                "detect": "DETECT", "identify": "DETECT", "reveal": "DETECT",
                "find": "DETECT", "discover": "DETECT", "uncover": "DETECT",
                "expose": "DETECT", "spot": "DETECT", "flag": "DETECT",
                "classify": "CLASSIFY", "segment": "CLASSIFY", "categorize": "CLASSIFY",
                "tier": "CLASSIFY", "rank": "CLASSIFY", "group": "CLASSIFY",
                "simulate": "SIMULATE", "model": "SIMULATE",
                "optimize": "OPTIMIZE", "maximize": "OPTIMIZE", "minimize": "OPTIMIZE",
                "assess": "ASSESS", "evaluate": "ASSESS", "measure": "ASSESS",
                "gauge": "ASSESS", "quantify": "ASSESS",
                "analyze": "ANALYZE", "examine": "ANALYZE", "investigate": "ANALYZE",
                "track": "TRACK", "monitor": "TRACK", "observe": "TRACK",
            }
            tokens = _re.findall(r"[a-z]+", name_lc)
            # replace first verb if known; drop connector words
            stop = {"with","and","or","the","a","an","for","to","of","by","in","on",
                    "plan","strategy","recommendation","program","playbook","actions",
                    "through","via","across","improve","improving","improved","improves",
                    "reduce","reducing","reduced","increase","increasing","increased"}
            # light stemmer: strip trailing -ing, -ed, -s from content tokens so
            # 'reducing'/'reduced'/'reduces' all collapse to 'reduc' and collide.
            def _stem(t):
                for suf in ("ing", "ed", "ies", "s"):
                    if len(t) > len(suf) + 2 and t.endswith(suf):
                        return t[:-len(suf)]
                return t
            norm_tokens = []
            verb_replaced = False
            for t in tokens:
                if not verb_replaced and t in verb_map:
                    norm_tokens.append(verb_map[t]); verb_replaced = True
                    continue
                if t in stop:
                    continue
                norm_tokens.append(_stem(t))
            # Keep first 4 content tokens — captures the decision core
            # (verb + object + qualifier + metric). 3 over-collapses distinct
            # decisions; 4 preserves them while still catching re-wordings.
            return " ".join(norm_tokens[:4])

        def _tables_tuple(uc):
            return tuple(sorted(
                t.strip().lower() for t in (uc.get("Tables Involved", "") or "").split(",")
                if t.strip()
            ))

        pass_b_groups = {}
        for uc in kept:
            key = (_normalize_name(uc.get("Name", "")), _tables_tuple(uc))
            if not key[0]:
                continue  # UCs with no name skip Pass B
            pass_b_groups.setdefault(key, []).append(uc)
        kept_b = []
        seen_dropped = set()
        for key, group in pass_b_groups.items():
            if len(group) == 1:
                kept_b.append(group[0])
                continue
            # within-group dedup: keep highest score
            def _score(x):
                try:
                    return float(x.get("Inspire Score", x.get("Priority", 0)) or 0)
                except Exception:
                    return 0.0
            group.sort(key=lambda x: (-_score(x), str(x.get("No", "zzz"))))
            kept_b.append(group[0])
            for loser in group[1:]:
                seen_dropped.add(id(loser))
                merge_log.append({
                    "kept": group[0].get("No", "UNK"),
                    "dropped": loser.get("No", "UNK"),
                    "reason": "NAME_CANONICAL_DUPLICATE",
                    "normalized_name": key[0],
                    "tables": list(key[1]),
                })
        # kept_b plus any kept UC that didn't enter Pass B (no-name UCs)
        kept_b_ids = {id(u) for u in kept_b}
        for uc in kept:
            if id(uc) not in kept_b_ids and id(uc) not in seen_dropped:
                kept_b.append(uc)
        kept = kept_b

        if merge_log:
            rate = len(merge_log) / max(1, len(use_cases))
            level = "warning" if rate > 0.05 else "info"
            getattr(self.logger, level)(
                f"Q2 DEDUP: dropped {len(merge_log)} of {len(use_cases)} "
                f"as canonical duplicates (rate {rate*100:.1f}%)"
            )
            if rate > 0.05:
                self.logger.warning(
                    "Q2 DEDUP_RATE_HIGH (>5%) — indicates prompt regression; "
                    "investigate which technique/tables cluster is over-represented"
                )
        else:
            self.logger.info(
                f"Q2 DEDUP: all {len(use_cases)} use cases have distinct signatures"
            )
        return kept, merge_log

    def _phaseQ3_validate_relevance(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q3 R1+R2 RELEVANCE: drop UCs that (R1) cite ONLY technical tables
        AND drop UCs whose (R2) declared Analytics Technique is structurally
        unsupported by the schema of its cited tables."""
        import re as _re
        if not use_cases:
            return use_cases, []
        # Build structural maps per table
        cols_by = {}
        for detail in full_schema_details:
            try:
                (c, s, t, col, dtype, _) = detail
            except Exception:
                continue
            fqn = f"{c}.{s}.{t}"
            cols_by.setdefault(fqn.lower(), []).append({"name": col, "type": (dtype or "").upper()})
        # Simple TECHNICAL-name heuristic (agent already classifies; we re-check conservatively here)
        def _is_technical_fqn(fqn):
            low = fqn.lower()
            for pat in ("_log", "_audit", "_system", "_metadata", "_temp",
                        "_backup", "_snapshot", "_changelog"):
                if pat in low:
                    return True
            return False
        # R2 technique prerequisites — structural (industry-agnostic).
        # Types in dtype field come from the schema, but when tables use STRING for
        # everything (common with external/delta-share/synthetic schemas), the type
        # alone doesn't tell us a column's role. Fall back to column-name-stem
        # heuristics — universal English conventions for labeling date-like and
        # numeric-like columns; these stems are NOT business-vocabulary, they're
        # the structural roots ("date"/"amount"/etc.) that any English schema uses.
        def _technique_supported(tech, uc_cols):
            import re as _rX
            t = (tech or "").strip().lower()
            if not t:
                return True
            types = {c["type"] for c in uc_cols}
            names = {c["name"].lower() for c in uc_cols if c.get("name")}
            has_date = any(x.startswith(("DATE", "TIMESTAMP")) for x in types)
            has_numeric = any(x.startswith(("INT", "BIGINT", "DOUBLE", "FLOAT", "DECIMAL", "SMALLINT", "LONG", "NUMERIC", "REAL")) for x in types)
            # Name-stem fallback (applies to any schema where types are unhelpful).
            if not has_date:
                _date_re = _rX.compile(r"(^|_)(date|time|timestamp|year|yr|month|mo|day|dt)(_|$)")
                has_date = any(bool(_date_re.search(n)) for n in names)
            if not has_numeric:
                _num_re = _rX.compile(r"(^|_)(amount|count|qty|quantity|num|number|price|cost|charge|fee|rate|total|sum|avg|units|value|score|size|age|year|yr)(_|$)")
                has_numeric = any(bool(_num_re.search(n)) for n in names)
            # Quick pass-through for anything not in the map below
            requirements = {
                "forecasting": has_date and has_numeric,
                "time-series decomposition": has_date and has_numeric,
                "trend analysis": has_date and has_numeric,
                "period-over-period analysis": has_date and has_numeric,
                "cohort analysis": has_date,
                "survival analysis": has_date,
                "funnel analysis": False,  # needs stage columns — conservative: reject unless explicitly supported elsewhere
                "anomaly detection": has_numeric,
                "regression analysis": has_numeric,
                "ridge/lasso regression": has_numeric,
                "clustering": has_numeric,
                "k-means": has_numeric,
                "predictive modeling": has_numeric,
                "classification": True,  # can classify on most schemas
                "top-n ranking": has_numeric,
                "ranking": has_numeric,
            }
            for key, needed in requirements.items():
                if key in t:
                    if not needed:
                        return False
                    return True
            return True  # unknown technique — don't reject
        kept, dropped = [], []
        for uc in use_cases:
            tabs = [t.strip().strip("`").lower() for t in (uc.get("Tables Involved", "") or "").split(",") if t.strip()]
            if not tabs:
                kept.append(uc); continue
            # R1: at least one non-technical table
            if all(_is_technical_fqn(t) for t in tabs):
                dropped.append((uc, "UC_NOT_RELEVANT:R1_TECHNICAL_ONLY"))
                try:
                    NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH",
                                     phase="Q3", step="R1",
                                     evidence=f"UC {uc.get('No','UNK')}: only technical tables cited")
                except Exception:
                    pass
                continue
            # R2: collect columns across cited tables
            uc_cols = []
            for t in tabs:
                uc_cols.extend(cols_by.get(t, []))
            tech = uc.get("Analytics Technique", "")
            if not _technique_supported(tech, uc_cols):
                dropped.append((uc, f"UC_NOT_RELEVANT:R2_TECHNIQUE_DATA_MISMATCH:{tech}"))
                try:
                    NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH",
                                     phase="Q3", step="R2",
                                     evidence=f"UC {uc.get('No','UNK')}: {tech} unsupported by cited-table columns")
                except Exception:
                    pass
                continue
            kept.append(uc)
        # ===== Q3 R4-R11 (batched §4.11-derived structural checks) =====
        # These run over the R1+R2-surviving "kept" list and demote weak UCs to dropped.
        import re as _rr
        def _col_names_for_tables(tabs_lower):
            out = set()
            for t in tabs_lower:
                for c in cols_by.get(t, []):
                    out.add(c.get("name","").lower())
            return out
        data_cat_map = getattr(self, "data_category_map", {}) or {}
        # R6 helper: does the cited-column pool contain an identifier-like column for this subject?
        # Industry-agnostic — no hardcoded subject vocabulary. Matches any column whose name
        # contains the subject substring AND has an identifier suffix/prefix pattern.
        def _subject_has_id_column(subject_noun: str, col_names: set) -> bool:
            s = (subject_noun or "").strip().lower()
            if not s:
                return True
            sn = s.rstrip("s")  # simple singular form
            id_suffixes = ("_id","id","_key","key","_code","code","_sk","sk","_num","num")
            id_prefixes = ("id_","fk_")
            for c in col_names:
                cl = c.lower()
                if s in cl or sn in cl:
                    if any(cl.endswith(suf) for suf in id_suffixes) or any(cl.startswith(pre) for pre in id_prefixes):
                        return True
                    # or the column IS exactly the subject (e.g., `customer`)
                    if cl == s or cl == sn:
                        return True
            return False
        kept2 = []
        for uc in kept:
            uc_no = uc.get("No","UNK")
            tabs_raw = [t.strip().strip("`") for t in (uc.get("Tables Involved","") or "").split(",") if t.strip()]
            tabs_lower = [t.lower() for t in tabs_raw]
            sol = str(uc.get("Solution","") or "")
            stmt = str(uc.get("Statement","") or "")
            bv = str(uc.get("Business Value","") or "")
            tech = str(uc.get("Analytics Technique","") or "").strip().lower()
            uc_col_names = _col_names_for_tables(tabs_lower)
            drop_reason = None

            # R4 join_path_executable: multi-table UC must have an executable join path.
            # PRIMARY check: any pair of the UC's tables is connected in the FK graph
            # (handles Kimball prefix-per-table keys like cs_item_sk -> i_item_sk where
            # column names differ but the FK is valid).
            # FALLBACK: shared column name (for schemas with no FK graph).
            if drop_reason is None and len(tabs_lower) >= 2:
                fk_connected = False
                # Build lookup: table -> set of tables it references (both directions)
                try:
                    fk_graph = getattr(self.data_loader, "foreign_key_graph", None) or {}
                    # Build undirected adjacency from the graph
                    fk_adj = {}
                    for (cat, sch, tbl), rels in fk_graph.items():
                        src_fqn = f"{cat}.{sch}.{tbl}".lower()
                        for rel in rels:
                            # rel shape: (src_cat, src_sch, src_tbl, src_col, ref_cat, ref_sch, ref_tbl, ref_col)
                            try:
                                ref_fqn = f"{rel[4]}.{rel[5]}.{rel[6]}".lower()
                            except Exception:
                                continue
                            fk_adj.setdefault(src_fqn, set()).add(ref_fqn)
                            fk_adj.setdefault(ref_fqn, set()).add(src_fqn)
                    tabs_set = set(tabs_lower)
                    for t in tabs_lower:
                        reachable = fk_adj.get(t, set())
                        if reachable & (tabs_set - {t}):
                            fk_connected = True
                            break
                except Exception:
                    fk_connected = False
                if not fk_connected:
                    # Fallback: shared-column-name heuristic (pre-Kimball-fix behavior)
                    per_table_cols = {t: {c.get("name","").lower() for c in cols_by.get(t, [])} for t in tabs_lower}
                    names_seen = {}
                    for t, cs in per_table_cols.items():
                        for n in cs:
                            names_seen.setdefault(n, set()).add(t)
                    shared_cols = [n for n, ts in names_seen.items() if len(ts) >= 2]
                    if not shared_cols:
                        drop_reason = "Q3_R4_JOIN_PATH_UNEXECUTABLE"
                        try:
                            NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH", phase="Q3", step="R4",
                                             evidence=f"UC {uc_no}: multi-table UC lists {len(tabs_lower)} tables but no FK edge and no shared column name")
                        except Exception:
                            pass

            # R5 reference-table-as-fact guard: TREND/FORECAST/COHORT on REFERENCE-only primary table
            if drop_reason is None:
                heavy_temporal = any(k in tech for k in ("forecast","trend","cohort","survival","time-series","period-over-period"))
                if heavy_temporal and len(tabs_lower) == 1:
                    fqn = tabs_lower[0]
                    fqn_upper_check = fqn
                    # Try both lowercase and as-stored form
                    cat = data_cat_map.get(fqn_upper_check) or data_cat_map.get(fqn.upper()) or ""
                    if str(cat).upper() == "REFERENCE":
                        drop_reason = "Q3_R5_REFERENCE_AS_FACT"
                        try:
                            NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH", phase="Q3", step="R5",
                                             evidence=f"UC {uc_no}: {tech} on REFERENCE-only table {fqn}")
                        except Exception:
                            pass

            # R6 action_subject_grounded: if solution recommends per-<noun> action (industry-agnostic).
            # Detect the noun, then check if the cited tables expose a column identifying that noun.
            if drop_reason is None:
                blob_lower = (stmt + " " + sol + " " + bv).lower()
                # Capture the noun that follows "per", "per-", "per each", "targeted/specific/each"
                # and the noun that precedes "targeting/scheduling/incentive/assignment/prioritization".
                patterns = [
                    r"\bper[\s-]+([a-z][a-z_-]{2,20})\b",
                    r"\bper\s+each\s+([a-z][a-z_-]{2,20})\b",
                    r"\btargeted\s+([a-z][a-z_-]{2,20})\b",
                    r"\beach\s+([a-z][a-z_-]{2,20})\s+(incentive|targeting|scheduling|assignment|prioritization|routing|intervention)\b",
                    r"\b([a-z][a-z_-]{2,20})\s+(incentive|targeting|scheduling|assignment|prioritization|routing|intervention)\s+(plan|strategy|recommendation|program)\b",
                ]
                found_subject = None
                for pat in patterns:
                    m = _rr.search(pat, blob_lower)
                    if m:
                        candidate = m.group(1).strip("-_").rstrip("s")
                        if candidate and len(candidate) >= 3 and candidate not in (
                            "location","market","customer","data","performance","revenue","product",
                            "result","feature","column","table","metric","score","category","segment"
                        ):
                            # the exclusion list is for VERY generic nouns that usually are not
                            # meant as actionable subjects; keep as-is, but any column check below
                            # will still validate them if found.
                            found_subject = candidate
                            break
                        elif candidate and len(candidate) >= 3:
                            found_subject = candidate
                            break
                if found_subject:
                    if not _subject_has_id_column(found_subject, uc_col_names):
                        # R6 demoted to INFO-only flag — regex-based subject extraction
                        # has a high false-positive rate on generic prepositional nouns
                        # (per-mile, per-time, per-available, etc.). The prompt's §D
                        # STAKEHOLDER self-review handles this at generation time; this
                        # rule is a safety net that surfaces suspects without dropping.
                        try:
                            NEXT_ACTIONS.add("UC_ACTION_SUBJECT_SUSPECT", severity="INFO", phase="Q3", step="R6",
                                             evidence=f"UC {uc_no}: text suggests per-{found_subject} action; verify cited tables expose a {found_subject}-identifying column")
                        except Exception:
                            pass

            # R7 business_value_column_backed: BV claims metric-computation — must trace to >=1 cited column name.
            # Upgraded threshold: >=3 unsupported quantity mentions + no [ESTIMATE:] tags + no
            # cited column substring → DROP. 1-2 mentions stay INFO (may be legitimate language).
            if drop_reason is None:
                qty_hits = len(_rr.findall(r"\$|\b\d+\s?%|\bpercent|\bincreas|\breduc|\bsav", bv.lower()))
                bv_has_estimate = "[ESTIMATE:" in bv
                cited_col_in_bv = any((cn and cn in bv.lower()) for cn in uc_col_names if cn and len(cn) >= 4)
                if qty_hits >= 1 and not bv_has_estimate and not cited_col_in_bv:
                    if qty_hits >= 3:
                        drop_reason = f"Q3_R7_BV_UNGROUNDED_QUANTITIES:{qty_hits}"
                        try:
                            NEXT_ACTIONS.add("UC_BV_UNGROUNDED_NUMBER", severity="HIGH", phase="Q3", step="R7",
                                             evidence=f"UC {uc_no}: Business Value has {qty_hits} quantity claims with zero column backing or [ESTIMATE:] tags")
                        except Exception:
                            pass
                    else:
                        try:
                            NEXT_ACTIONS.add("UC_BV_UNGROUNDED_NUMBER", severity="INFO", phase="Q3", step="R7",
                                             evidence=f"UC {uc_no}: Business Value mentions {qty_hits} quantity without citing a column or tagging [ESTIMATE:]")
                        except Exception:
                            pass

            # R8 target_leakage: if Solution claims to predict/forecast COLUMN_X and the same column appears in the feature list.
            if drop_reason is None:
                # Match patterns like "predict <col>" or "forecast <col>"
                m_pred = _rr.search(r"(predict|forecast)\s+([a-z_][a-z0-9_]*)", sol.lower())
                if m_pred:
                    target = m_pred.group(2)
                    # Check if target appears ELSEWHERE in the sol as a feature-list entry
                    feats = set(_rr.findall(r"`([a-z_][a-z0-9_]*)`", sol.lower()))
                    if target in feats:
                        # target column is cited as a feature → tautological
                        # Only flag if target is actually a column in cited tables
                        if target in uc_col_names:
                            drop_reason = f"Q3_R8_TARGET_LEAKAGE:{target}"
                            try:
                                NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH", phase="Q3", step="R8",
                                                 evidence=f"UC {uc_no}: predicts `{target}` while listing `{target}` as feature")
                            except Exception:
                                pass

            # R9 no_simulated_inputs: Solution describes a required input that is NOT real data.
            # Generalized regex: any "(simulated|assumed|synthetic|hypothetical|placeholder|made-up|
            #  fictitious|imaginary|invented|fabricated|mock)\s+<noun>" where <noun> looks like a
            # data input (capacity/cost/constraint/threshold/volume/demand/ground truth/label/target/
            # parameter/weight/rate/budget). Industry-agnostic.
            if drop_reason is None:
                r9_pat = _rr.compile(
                    r"\b(simulated|assumed|synthetic|hypothetical|placeholder|made[-\s]?up|"
                    r"fictitious|imaginary|invented|fabricated|mock)\s+"
                    r"([a-z][a-z_-]{2,30})\b"
                )
                r9_bad_nouns = {
                    "capacity","cost","constraint","threshold","volume","demand","revenue","price",
                    "budget","rate","weight","parameter","ground","truth","label","target","score",
                    "baseline","benchmark"
                }
                m_sim = r9_pat.search(sol.lower())
                if m_sim and m_sim.group(2) in r9_bad_nouns:
                    drop_reason = f"Q3_R9_SIMULATED_INPUT:{m_sim.group(1)}_{m_sim.group(2)}"
                    try:
                        NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH", phase="Q3", step="R9",
                                         evidence=f"UC {uc_no}: Solution depends on {m_sim.group(1)} {m_sim.group(2)} (not from cited schema)")
                    except Exception:
                        pass

            # R10 cardinality_vs_technique: K-means/Clustering/RF on tiny REFERENCE-scale tables
            if drop_reason is None:
                technique_needs_rows = any(k in tech for k in (
                    "clustering","k-means","hierarchical","classification",
                    "predictive modeling","regression","random forest","xgboost","gradient boosting",
                    "isolation forest"
                ))
                if technique_needs_rows and len(tabs_lower) == 1:
                    fqn = tabs_lower[0]
                    cat = data_cat_map.get(fqn) or data_cat_map.get(fqn.upper()) or ""
                    if str(cat).upper() == "REFERENCE":
                        # REFERENCE tables are structurally small; flag as data-fit weak
                        try:
                            NEXT_ACTIONS.add("UC_DATA_FIT_WEAK", severity="HIGH", phase="Q3", step="R10",
                                             evidence=f"UC {uc_no}: {tech} on REFERENCE-scale table {fqn}")
                        except Exception:
                            pass
                        # We demote rather than drop: priority gets capped later.
                        try:
                            uc["_data_fit_weak"] = True
                        except Exception:
                            pass

            # R11 cohort_temporal_anchor: Cohort Analysis must use an EVENT-timestamp column, not a biographical one.
            if drop_reason is None and "cohort" in tech:
                # Cheap heuristic: if Solution references "birth" or "age" near cohort-forming language, flag.
                s_low = sol.lower()
                if _rr.search(r"(cohort)[^.]*?(birth|birthdate|birth_year|age)", s_low):
                    drop_reason = "Q3_R11_COHORT_BIOGRAPHICAL_ANCHOR"
                    try:
                        NEXT_ACTIONS.add("UC_NOT_RELEVANT", severity="HIGH", phase="Q3", step="R11",
                                         evidence=f"UC {uc_no}: Cohort analysis uses biographical (birth) column instead of acquisition timestamp")
                    except Exception:
                        pass

            # R13 solution_grounding_floor: Solution must cite at least 2 backticked
            # column identifiers that match columns in the cited tables. This catches
            # "invented externality" UCs whose Solution describes concepts that have
            # no column representation (e.g., "optimize energy demand" on a weather-
            # only schema). Structural — cited column names are validated against
            # the actual schema, not any vocabulary list.
            if drop_reason is None:
                # Extract bare backticked identifiers (not FQN) from Solution
                backticked = set(_rr.findall(r"`([A-Za-z_][A-Za-z0-9_.]*)`", sol))
                SQL_KW = {"SELECT","FROM","WHERE","AND","OR","NOT","NULL","TRUE","FALSE","IS",
                          "ORDER","GROUP","BY","HAVING","LIMIT","CASE","WHEN","THEN","ELSE","END",
                          "JOIN","INNER","OUTER","LEFT","RIGHT","ON","AS","DISTINCT"}
                grounded_count = 0
                for tok in backticked:
                    if tok.upper() in SQL_KW:
                        continue
                    # treat last path-segment as the column name (handles FQN)
                    last = tok.rsplit(".", 1)[-1].lower()
                    if last in uc_col_names:
                        grounded_count += 1
                # Require >= 2 grounded columns. Below that, most likely invented.
                # Exception: UCs that cite Volume paths (ai_parse_document use cases)
                # are exempt — they're file-level operations, not SQL.
                _tabs_str = (uc.get("Tables Involved", "") or "")
                if "/Volumes" in _tabs_str:
                    pass  # skip R13 for Volume-backed UCs
                elif grounded_count < 2:
                    drop_reason = f"Q3_R13_SOLUTION_UNDER_GROUNDED:{grounded_count}"
                    try:
                        NEXT_ACTIONS.add("UC_SOLUTION_UNDER_GROUNDED", severity="HIGH", phase="Q3", step="R13",
                                         evidence=f"UC {uc_no}: Solution cites only {grounded_count} column(s) from schema (need >=2); likely invented-externality")
                    except Exception:
                        pass

            # R14 data-fit scale: technique-specific structural prerequisites beyond R2's
            # presence checks. Each rule tests counts of column signals in the cited tables.
            # Purely structural — relies on column-name/type, not on business vocabulary.
            if drop_reason is None:
                _tl = tech.lower()
                # Build column-type signals for cited tables
                _date_cols = [n for n in uc_col_names if n and any(s in n for s in ("date","time","timestamp","year","yr","month","mo","day","dt"))]
                _id_cols = [n for n in uc_col_names if n and (n.endswith("_id") or n.endswith("id") or n.endswith("_key") or n.endswith("_sk") or n.endswith("_nk") or n in {"id","key","code"} or "_id_" in n or "_sk_" in n)]
                _num_cols = [n for n in uc_col_names if n and any(s in n for s in ("amount","count","qty","quantity","num","number","price","cost","charge","fee","rate","total","sum","avg","units","value","score","size","age","measure"))]

                _r14_fail = None
                # Classification / Predictive Modeling / Regression: require >=3 feature columns
                if any(k in _tl for k in ("classification","random forest","xgboost","gradient boosting","logistic","ridge","lasso","regression","predictive modeling")):
                    if len(uc_col_names) < 3:
                        _r14_fail = f"R14_CLASSIFICATION_UNDER_FEATURED:{len(uc_col_names)}_cols"
                # Clustering: require >=3 numeric-like columns
                elif any(k in _tl for k in ("clustering","k-means","hierarchical","dbscan","segmentation")):
                    if len(_num_cols) < 3 and len(uc_col_names) < 3:
                        _r14_fail = f"R14_CLUSTERING_UNDER_FEATURED:{len(uc_col_names)}_cols"
                # Market basket / Association: require a grouping id column
                elif any(k in _tl for k in ("market basket","association rules")):
                    if not _id_cols:
                        _r14_fail = "R14_BASKET_NO_GROUPING_ID"
                # Top-N / Ranking: require a dimension to rank over
                elif any(k in _tl for k in ("top-n","ranking")):
                    if len(uc_col_names) < 2:
                        _r14_fail = "R14_RANKING_NEEDS_DIM_AND_MEASURE"
                # Correlation: require >=2 numeric columns
                elif "correlation" in _tl:
                    if len(_num_cols) < 2 and len(uc_col_names) < 2:
                        _r14_fail = "R14_CORRELATION_UNDER_COLUMNED"
                # Pareto: require a category-like + numeric
                elif "pareto" in _tl:
                    if not _num_cols:
                        _r14_fail = "R14_PARETO_NO_MEASURE"
                # Recommendation: require subject+item+interaction signal (3 cols min)
                elif any(k in _tl for k in ("recommendation","collaborative filtering")):
                    if len(uc_col_names) < 3:
                        _r14_fail = "R14_RECO_UNDER_COLUMNED"
                # Text analytics: require at least one string-looking column via name-stem
                elif any(k in _tl for k in ("text analytics","sentiment","topic modeling","lda","summarization","entity extraction")):
                    _text_like = [n for n in uc_col_names if n and any(s in n for s in ("text","comment","review","description","note","feedback","transcript","body","message","title"))]
                    if not _text_like:
                        _r14_fail = "R14_TEXT_NO_TEXT_COLUMN"
                # Network Analysis: require 2+ id columns
                elif "network" in _tl:
                    if len(_id_cols) < 2:
                        _r14_fail = "R14_NETWORK_NEEDS_TWO_IDS"
                # Geospatial: require geographic signal
                elif "geospatial" in _tl:
                    _geo = [n for n in uc_col_names if n and any(s in n for s in ("lat","lon","long","latitude","longitude","zip","postal","city","state","country","region","address","geo"))]
                    if not _geo:
                        _r14_fail = "R14_GEO_NO_GEOGRAPHIC_COLUMN"

                if _r14_fail:
                    drop_reason = f"Q3_R14_DATA_FIT_SCALE:{_r14_fail}"
                    try:
                        NEXT_ACTIONS.add("UC_DATA_FIT_WEAK", severity="HIGH", phase="Q3", step="R14",
                                         evidence=f"UC {uc_no}: {tech} fails structural data-fit rule ({_r14_fail})")
                    except Exception:
                        pass

            if drop_reason is None:
                kept2.append(uc)
            else:
                dropped.append((uc, drop_reason))

        kept = kept2

        if dropped:
            self.logger.warning(f"Q3 RELEVANCE: dropped {len(dropped)} of {len(use_cases)} UCs (R1-R11)")
            for uc, reason in dropped[:10]:
                self.logger.info(f"  dropped {uc.get('No','UNK')}: {reason}")
        else:
            self.logger.info(f"Q3 RELEVANCE: all {len(use_cases)} UCs pass R1-R11")
        return kept, dropped

    def _phaseQ4_detect_operational_patterns(self, full_schema_details: list) -> set:
        """Phase Q4 TIER-1 detector. Returns set of patterns the schema
        STRUCTURALLY supports. Industry-agnostic: type + structure only,
        no English keyword matching."""
        patterns = set()
        if not full_schema_details:
            return patterns
        # Aggregate columns per table
        by_table = {}
        for detail in full_schema_details:
            try:
                (c, s, t, col, dtype, _) = detail
            except Exception:
                continue
            by_table.setdefault(f"{c}.{s}.{t}", []).append({"name": (col or ""), "type": (dtype or "").upper()})
        has_date_col_anywhere = False
        has_fact_like = False
        has_dim_like = False
        has_grouping_candidate = False
        has_status_like = False
        for fqn, cols in by_table.items():
            types = {c["type"] for c in cols}
            numeric_count = sum(1 for c in cols if c["type"].startswith(("INT", "BIGINT", "DOUBLE", "FLOAT", "DECIMAL", "SMALLINT", "LONG")))
            date_cols = [c for c in cols if c["type"].startswith(("DATE", "TIMESTAMP"))]
            if date_cols:
                has_date_col_anywhere = True
            # Fact-like: >= 3 numeric columns + some kind of date column
            if numeric_count >= 3 and date_cols:
                has_fact_like = True
            # Dim-like: small to moderate column count, has an id-like column (name ends with _id/_key/id/key)
            if any(c["name"].lower().endswith(("id", "_id", "key", "_key")) for c in cols) and len(cols) >= 3:
                has_dim_like = True
            # Grouping id candidate: a column that could group multiple rows (order_id / transaction_id / session_id pattern)
            if any(any(tok in c["name"].lower() for tok in ("order", "basket", "transaction", "session", "invoice", "visit")) for c in cols):
                has_grouping_candidate = True
            # Status-like: string column with limited cardinality (we can't probe cardinality here — use name heuristic as TIER-1 proxy)
            if any(c["type"].startswith("STRING") and any(tok in c["name"].lower() for tok in ("status", "state", "flag", "category", "type", "kind")) for c in cols):
                has_status_like = True
        if has_date_col_anywhere and has_fact_like:
            patterns.add("TREND_OR_PERIOD")
        if has_dim_like and has_fact_like:
            patterns.add("SUBJECT_360")
            patterns.add("MIX_OR_MARGIN")
        if has_fact_like and has_status_like and has_date_col_anywhere:
            patterns.add("SLA_OR_ANOMALY")
        if has_grouping_candidate:
            patterns.add("BASKET_OR_FUNNEL")
        self.logger.info(f"Q4 detect_operational_patterns: {sorted(patterns) or 'none (thin schema)'}")
        return patterns

    def _phaseQ5_decision_readiness(self, use_cases: list, full_schema_details: list) -> tuple:
        """Phase Q5 — deterministic decision-readiness gate (no LLM).

        Applies R15-R18. Industry-agnostic. Returns (kept, dropped).
        R15: decision-verb rejection
        R16: technique-data fit (sentiment/NLP only on genuine text cols)
        R17: score floor 7.0
        R18: technique-label consistency with HLD keyword
        """
        import re as _re
        if not use_cases:
            return use_cases, []
        # Collect real text columns from schema — any STRING column whose NAME contains
        # text-signal tokens. No hardcoded column names.
        text_col_signals = {"desc", "description", "comment", "review", "note",
                            "feedback", "body", "message", "transcript", "title",
                            "subject", "content", "text", "reason"}
        text_cols_present = set()
        try:
            for detail in (full_schema_details or []):
                try:
                    (_cat, _sch, _tbl, col_name, col_type, _cmt) = detail
                except Exception:
                    continue
                cn = str(col_name or "").lower()
                if "string" in str(col_type or "").lower() and any(sig in cn for sig in text_col_signals):
                    text_cols_present.add(cn)
        except Exception:
            pass

        decision_verbs = {
            "detect","reduce","increase","predict","prevent","optimize","classify","segment",
            "forecast","prioritize","recover","accelerate","extend","stabilize","rebalance",
            "eliminate","recommend","personalize","protect","identify","rationalize","shift",
            "surface","flag","route","allocate","approve","escalate","trigger","intervene",
            "grow","improve"  # 'improve' / 'grow' accepted as they imply lifting a metric
        }
        reject_verbs = {"map","analyze","understand","study","examine","investigate","explore",
                        "measure","observe","track","monitor","describe","review","assess"}

        technique_method_keywords = {
            "classification": ("classif","xgboost","random forest","lightgbm","logistic","ai_classify"),
            "regression": ("regress","ridge","lasso","elasticnet","glm","poisson","gamma"),
            "clustering": ("k-means","kmeans","dbscan","hierarchical","cluster"),
            "anomaly detection": ("isolation forest","lof","local outlier","one-class","z-score",
                                  "anomaly","mahalanobis"),
            "forecasting": ("prophet","arima","ai_forecast","time-series","exponential smoothing"),
            "trend analysis": ("trend","slope","linear fit","stl","seasonality","percent change"),
            "pareto analysis": ("pareto","cumulative","gini","lorenz","herfindahl","concentration"),
            "correlation analysis": ("correlation","pearson","spearman","covariance"),
            "cohort analysis": ("cohort","vintage"),
            "survival analysis": ("kaplan","survival","hazard","cox"),
            "segmentation": ("segment","persona"),
            "recommendation": ("recommendation","collaborative filter","content-based","als"),
            "sentiment analysis": ("sentiment","ai_classify","ai_query"),
            "topic modeling": ("topic","lda","bertopic","ai_query"),
            "nlp classification": ("nlp","ai_classify","ai_query"),
            "summarization": ("summar","ai_query"),
            "entity extraction": ("extract","ai_extract","entity","ner"),
            "text analytics": ("text","ai_query","nlp"),
            "simulation": ("simulation","monte carlo","scenario"),
            "optimization": ("optimi","linear program","milp","knapsack"),
            "network analysis": ("network","graph","pagerank","community"),
            "geospatial analysis": ("geospatial","geohash","h3","spatial"),
            "ranking": ("rank","top-n","percentile"),
            "predictive modeling": ("predict","xgboost","random forest","gradient boost","neural"),
        }

        text_techs = {"sentiment analysis","topic modeling","nlp classification",
                      "summarization","entity extraction","text analytics"}

        kept, dropped = [], []
        for uc in use_cases:
            name = str(uc.get("Name","") or uc.get("use_case","") or "").strip()
            tech = str(uc.get("Analytics Technique","") or uc.get("analytics_technique","") or "").strip().lower()
            try:
                score = float(uc.get("Inspire Score", uc.get("Priority", uc.get("priority_score", uc.get("inspire_score", 0)))) or 0)
            except Exception:
                score = 0.0
            hld = str(uc.get("high_level_design","") or "").lower()
            solution = str(uc.get("Solution","") or uc.get("solution","") or "").lower()
            tables_involved = str(uc.get("Tables Involved","") or uc.get("tables_involved","") or "").lower()

            drop_reason = None

            # R15 decision-verb
            first_word = _re.findall(r"[a-zA-Z]+", name.lower())
            first = first_word[0] if first_word else ""
            if first in reject_verbs:
                drop_reason = "Q5_R15_NON_DECISION_VERB"
            elif first not in decision_verbs and first:
                # permissive: allow unknown verbs (won't block legit ones)
                pass

            # R16 technique-data fit: text techs require a text column in cited schema
            if drop_reason is None and tech in text_techs:
                cites_text_col = False
                for tc in text_cols_present:
                    if tc in hld or tc in solution:
                        cites_text_col = True; break
                if not cites_text_col:
                    drop_reason = "Q5_R16_TECHNIQUE_DATA_MISMATCH_NO_TEXT_COLUMN"

            # R17 score floor
            if drop_reason is None and score < 7.0:
                drop_reason = f"Q5_R17_SCORE_FLOOR:{score:.2f}"

            # R18 technique-label consistency — HLD must mention at least one of the
            # expected method keywords for the stated technique. Missing all = label drift.
            if drop_reason is None and tech in technique_method_keywords:
                kw_tuple = technique_method_keywords[tech]
                if kw_tuple and not any(kw in hld or kw in solution for kw in kw_tuple):
                    drop_reason = f"Q5_R18_TECHNIQUE_LABEL_DRIFT:{tech}"

            # R19 generic-title rejection — catches "Increase Customer Value to Grow Revenue" pattern
            # Schema-agnostic: matches generic verb + generic object + preposition structure.
            if drop_reason is None:
                _generic_title_re = _re.compile(
                    r"^(?:Improve|Increase|Optimi[sz]e|Maximize|Boost|Grow|Drive)\s+"
                    r"(?:the\s+)?"
                    r"(?:Customer Value|Business Value|Customer Experience|Customer Satisfaction|"
                    r"Customer Engagement|Customer Retention|Customer Loyalty|Customer Base|"
                    r"Product Sales|Revenue|Profit|Sales|Growth|Performance|Efficiency|Productivity|"
                    r"Operations|Margin|Retention|Loyalty|Conversion|Business Outcomes)"
                    r"\s+(?:to|by|for|through|with|and)\s+",
                    _re.IGNORECASE
                )
                if _generic_title_re.match(name):
                    # Second check — if name has ANY specific analytical signal (method / phenomenon
                    # / specific entity / specific metric), keep it. Signals are universal tokens
                    # found in professional analytics language, not domain-specific vocabulary.
                    _specific_signals = (
                        # Methods / techniques
                        "clustering","segment","prediction","predict","forecast","regression",
                        "classification","pareto","cohort","simulation","monte carlo","optimization",
                        "optimizing","attribution","propensity","anomaly","ranking","isolation",
                        "kmeans","k-means","uplift","correlation","survival","bayesian",
                        # Phenomena / patterns
                        "erosion","leakage","arbitrage","fraud","drift","bottleneck","volatility",
                        "deviation","spike","surge","lapsed","dormant","abandonment","abuse",
                        "misuse","churn","defection","underpenetrated","underserved","stagnant",
                        # Specific metrics / KPIs (composite forms only; single words too generic)
                        "return rate","conversion rate","retention rate","basket size","cross-sell",
                        "upsell","attach rate","markdown","mix shift","share of wallet",
                        # Concrete entities / outputs
                        "geographic","pipeline","territory","route","seasonality","warehouse",
                        "queue","playbook","action list","reallocation","escalation",
                    )
                    _nlow = name.lower()
                    if not any(sig in _nlow for sig in _specific_signals):
                        drop_reason = "Q5_R19_GENERIC_TITLE"

            if drop_reason:
                dropped.append((uc, drop_reason))
            else:
                kept.append(uc)
        if dropped:
            self.logger.warning(f"Q5 DECISION_READINESS: dropped {len(dropped)} of {len(use_cases)} UCs")
            for uc, reason in dropped[:10]:
                self.logger.info(f"  dropped {uc.get('No', uc.get('id','UNK'))}: {reason}")
        else:
            self.logger.info(f"Q5 DECISION_READINESS: all {len(use_cases)} UCs passed")
        return kept, dropped

    def _phaseQ2c_verb_object_cluster(self, use_cases: list) -> tuple:
        """Q2 Pass C — DISABLED per user directive (no regex/pattern matching).

        All dedup is now LLM-based via REVIEW_USE_CASES_PROMPT (Phase 3 global
        dedup) and the aggressive LLM re-review pass when portfolio > target_max.
        This method is kept as a no-op for pipeline compatibility."""
        return use_cases, []
        import re as _re
        if not use_cases:
            return use_cases, []
        verb_map = {
            "forecast":"PREDICT","predict":"PREDICT","anticipate":"PREDICT","estimate":"PREDICT",
            "detect":"DETECT","identify":"DETECT","reveal":"DETECT","find":"DETECT",
            "discover":"DETECT","uncover":"DETECT","spot":"DETECT","flag":"DETECT","surface":"DETECT",
            "classify":"CLASSIFY","segment":"CLASSIFY","categorize":"CLASSIFY",
            "tier":"CLASSIFY","rank":"CLASSIFY","group":"CLASSIFY",
            "reduce":"REDUCE","minimize":"REDUCE","lower":"REDUCE",
            "increase":"INCREASE","maximize":"INCREASE","grow":"INCREASE","improve":"INCREASE",
            "recover":"RECOVER","protect":"PROTECT","prevent":"PREVENT",
            "optimize":"OPTIMIZE","rebalance":"OPTIMIZE","rationalize":"OPTIMIZE",
            "accelerate":"ACCELERATE","extend":"EXTEND","stabilize":"STABILIZE",
            "eliminate":"ELIMINATE","recommend":"RECOMMEND","personalize":"PERSONALIZE",
        }
        stop = {"with","and","or","the","a","an","for","to","of","by","in","on","through","via",
                "across","plan","strategy","recommendation","program","playbook","actions"}
        def keyify(uc):
            name = str(uc.get("Name","") or uc.get("use_case","") or "").lower()
            tokens = _re.findall(r"[a-z]+", name)
            verb = None
            object_tok = None
            for t in tokens:
                if verb is None and t in verb_map:
                    verb = verb_map[t]; continue
                if verb is not None and t not in stop and t not in verb_map:
                    object_tok = t; break
            pt = str(uc.get("Primary Table","") or uc.get("primary_table","") or "").lower()
            return (pt, verb or "", object_tok or "")
        groups = {}
        for uc in use_cases:
            groups.setdefault(keyify(uc), []).append(uc)
        kept, merged = [], []
        for key, group in groups.items():
            if len(group) <= 1:
                kept.extend(group); continue
            # sort by score desc, id asc
            def _score(x):
                try:
                    return float(x.get("Inspire Score", x.get("inspire_score", 0)) or 0)
                except Exception:
                    return 0.0
            group.sort(key=lambda x: (-_score(x), str(x.get("No", x.get("id","zzz")))))
            kept.append(group[0])
            for loser in group[1:]:
                merged.append({
                    "kept": group[0].get("No", group[0].get("id","UNK")),
                    "dropped": loser.get("No", loser.get("id","UNK")),
                    "reason": "Q2C_VERB_OBJECT_CLUSTER",
                    "key": f"{key[0]}|{key[1]}|{key[2]}",
                })
        if merged:
            self.logger.info(f"Q2C verb-object cluster: merged {len(merged)} extras (kept highest score per group)")
        return kept, merged

    def _phaseQ4_validate_recall(self, use_cases: list, detected_patterns: set) -> list:
        """Phase Q4 coverage probe. For every detected pattern not covered by
        any UC, emit UC_RECALL_GAP HIGH. Does not drop UCs."""
        if not use_cases or not detected_patterns:
            return use_cases
        pattern_keywords = {
            "TREND_OR_PERIOD":  ["trend", "period", "month-over", "year-over", "forecast", "seasonal"],
            "SUBJECT_360":      ["360", "profile", "customer segment", "lifetime", "subject"],
            "SLA_OR_ANOMALY":   ["anomal", "sla", "outlier", "breach", "detect"],
            "MIX_OR_MARGIN":    ["mix", "margin", "pareto", "portfolio"],
            "BASKET_OR_FUNNEL": ["basket", "funnel", "bundle", "association", "cohort"],
        }
        covered = set()
        for uc in use_cases:
            blob = " ".join((uc.get(f, "") or "").lower() for f in ("Use Case", "Name", "Statement", "Solution", "Analytics Technique"))
            for p, keys in pattern_keywords.items():
                if any(k in blob for k in keys):
                    covered.add(p)
        missing = detected_patterns - covered
        for p in sorted(missing):
            self.logger.warning(f"Q4 UC_RECALL_GAP: pattern {p} supported by schema but not covered by any UC")
            try:
                NEXT_ACTIONS.add("UC_RECALL_GAP", severity="HIGH",
                                 phase="Q4", step="pattern_coverage",
                                 evidence=f"Pattern {p} supported by schema structure but no UC covers it",
                                 suggested_user_action="Add a UC that exploits this pattern OR declare it out-of-scope via widget 13.")
            except Exception:
                pass
        if not missing:
            self.logger.info(f"Q4 RECALL: all {len(detected_patterns)} detected patterns covered by at least one UC")
        return use_cases

    def _phaseQ_run_integrity_gates(self, use_cases: list, full_schema_details: list) -> list:
        """Run Q1..Q4 against the final consolidated portfolio. Returns surviving list."""
        if not use_cases:
            return use_cases
        kept, q1_dropped = self._phaseQ1_validate_grounding(use_cases, full_schema_details)
        kept, q2_merged = self._phaseQ2_canonical_dedup(kept)
        try:
            kept, q2c_merged = self._phaseQ2c_verb_object_cluster(kept)
        except Exception as _q2c_err:
            self.logger.warning(f"Q2c cluster-merge failed non-fatally: {get_clean_error_message(_q2c_err)}")
        q3_dropped = []
        try:
            kept, q3_dropped = self._phaseQ3_validate_relevance(kept, full_schema_details)
        except Exception as _q3_err:
            self.logger.warning(f"Q3 relevance failed non-fatally: {get_clean_error_message(_q3_err)}")
        try:
            _patterns = self._phaseQ4_detect_operational_patterns(full_schema_details)
            kept = self._phaseQ4_validate_recall(kept, _patterns)
        except Exception as _q4_err:
            self.logger.warning(f"Q4 pattern/recall failed non-fatally: {get_clean_error_message(_q4_err)}")
        try:
            kept, q5_dropped = self._phaseQ5_decision_readiness(kept, full_schema_details)
        except Exception as _q5_err:
            self.logger.warning(f"Q5 decision-readiness failed non-fatally: {get_clean_error_message(_q5_err)}")
        try:
            self.processing_honesty["q1_grounding_dropped"] = len(q1_dropped)
            self.processing_honesty["q2_dedup_dropped"] = len(q2_merged)
            self.processing_honesty["q3_relevance_dropped"] = len(q3_dropped)
        except Exception:
            pass
        return kept

    def _phaseL_final_gate(self, use_cases: list, full_schema_details: list) -> dict:
        """Phase L final gate: 17 binary checks emitted as NEXT_ACTIONS entries.
        Advisory on first run (no baseline); regressive on subsequent runs if baseline
        loaded. Returns a summary dict {check_id: {"pass": bool, "severity": str}}."""
        results = {}
        def _check(cid: str, passed: bool, severity: str, evidence: str, suggested: str = ""):
            results[cid] = {"pass": bool(passed), "severity": severity}
            if not passed:
                try:
                    NEXT_ACTIONS.add(
                        rule_id=f"L_{cid}", severity=severity, phase="L", step="FinalGate",
                        evidence=evidence[:300], suggested_user_action=suggested[:300] or None,
                    )
                except Exception:
                    pass
        uc_list = use_cases or []
        n = len(uc_list)
        # L1 first-run-pass: if no UCs, BLOCK
        _check("L1_any_ucs_produced", n > 0, "BLOCKING",
               f"Pipeline produced {n} use cases", "Re-run with broader schema or lower filters.")
        # L2 domain diversity
        _doms = {str(uc.get("Business Domain") or "").strip() for uc in uc_list if uc.get("Business Domain")}
        _check("L2_domain_diversity", len(_doms) >= 2 or n <= 3, "INFO",
               f"{len(_doms)} distinct domain(s)", "Expand schema coverage to unlock more domains.")
        # L3 subdomain diversity
        _subs = {str(uc.get("Subdomain") or "").strip() for uc in uc_list if uc.get("Subdomain")}
        _check("L3_subdomain_diversity", len(_subs) >= max(2, n // 3), "INFO",
               f"{len(_subs)} distinct subdomain(s)", "Encourage subdomain rename in prompt.")
        # L4 priority distribution non-degenerate (not all "Medium")
        _prios = [str(uc.get("Priority") or "").strip() for uc in uc_list]
        _check("L4_priority_nondegenerate", len(set(_prios)) >= 2 or n <= 2, "INFO",
               f"Priority values: {sorted(set(_prios))}", "Loosen priority heuristic in prompt.")
        # L5 every UC has primary table
        _missing_pt = [uc for uc in uc_list if not str(uc.get("Primary Table") or "").strip()]
        _check("L5_every_uc_has_primary_table", not _missing_pt, "BLOCKING",
               f"{len(_missing_pt)}/{n} UCs missing Primary Table", "Reject UCs without Primary Table at parse time.")
        # L6 every UC has statement
        _missing_st = [uc for uc in uc_list if len(str(uc.get("Statement") or "").strip()) < 20]
        _check("L6_every_uc_has_statement", not _missing_st, "HIGH",
               f"{len(_missing_st)}/{n} UCs have empty/short Statement", "Raise statement min-length in parser.")
        # L7 every UC has solution
        _missing_sol = [uc for uc in uc_list if len(str(uc.get("Solution") or "").strip()) < 20]
        _check("L7_every_uc_has_solution", not _missing_sol, "HIGH",
               f"{len(_missing_sol)}/{n} UCs have empty/short Solution", "Raise solution min-length in parser.")
        # L8 every UC has technique
        _missing_tech = [uc for uc in uc_list if not str(uc.get("Analytics Technique") or "").strip()]
        _check("L8_every_uc_has_technique", not _missing_tech, "HIGH",
               f"{len(_missing_tech)}/{n} UCs missing technique", "Enforce technique whitelist in parser.")
        # L9 no duplicate (name + primary_table). Same Name on DIFFERENT Primary
        # Tables is legitimate (e.g., "Simulate Discount Policy" on catalog_sales
        # vs web_sales — different channels, same decision pattern).
        _name_keys = []
        for uc in uc_list:
            nm = str(uc.get("Name","") or "").strip().lower()
            if not nm:
                continue
            pt = str(uc.get("Primary Table","") or "").strip().lower()
            _name_keys.append((nm, pt))
        _check("L9_no_duplicate_names", len(_name_keys) == len(set(_name_keys)), "HIGH",
               f"{len(_name_keys)-len(set(_name_keys))} UCs share name+primary_table",
               "Tighten Q2 canonical dedup or rename one of the colliding UCs.")
        # L10 Q1 grounding did not drop >50%
        q1d = self.processing_honesty.get("q1_grounding_dropped", 0) if hasattr(self, "processing_honesty") else 0
        _check("L10_q1_drop_under_50pct", (q1d * 2) < max(1, n + q1d), "HIGH",
               f"Q1 dropped {q1d} vs kept {n}", "Inspect prompt for fabricated tables/columns.")
        # L11 Q2 merged reasonable
        q2m = self.processing_honesty.get("q2_dedup_dropped", 0) if hasattr(self, "processing_honesty") else 0
        _check("L11_q2_merge_under_50pct", (q2m * 2) < max(1, n + q2m), "INFO",
               f"Q2 merged {q2m} vs kept {n}", "Diversify generation seeds.")
        # L12 Q3 relevance drop under 50%
        q3d = self.processing_honesty.get("q3_relevance_dropped", 0) if hasattr(self, "processing_honesty") else 0
        _check("L12_q3_drop_under_50pct", (q3d * 2) < max(1, n + q3d), "HIGH",
               f"Q3 dropped {q3d} vs kept {n}", "Review R1/R2 prerequisites.")
        # L13 schema detail non-empty
        _check("L13_schema_details_present", bool(full_schema_details), "BLOCKING",
               f"full_schema_details has {len(full_schema_details or [])} rows",
               "Check catalog permissions / table selection.")
        # L14 no UC references unknown table (schema rows are tuples: catalog, schema, table, col, type, desc)
        _table_names = set()
        for r in (full_schema_details or []):
            try:
                if isinstance(r, dict):
                    t = r.get("table_name") or r.get("table") or ""
                elif isinstance(r, (tuple, list)) and len(r) >= 3:
                    t = r[2]
                else:
                    t = ""
                if t:
                    _table_names.add(str(t).strip().lower())
            except Exception:
                continue
        _unknown = []
        for uc in uc_list:
            pt = str(uc.get("Primary Table") or "").strip().lower()
            if pt and pt.split(".")[-1] not in _table_names:
                _unknown.append(pt)
        _check("L14_all_primary_tables_grounded", not _unknown, "BLOCKING",
               f"{len(_unknown)} UCs reference unknown tables: {list(set(_unknown))[:5]}",
               "Strengthen Q1 grounding.")
        # L15 business_domain != placeholder
        _placeholders = {"", "other", "unknown", "general", "misc", "miscellaneous"}
        _ph = [uc for uc in uc_list if str(uc.get("Business Domain") or "").strip().lower() in _placeholders]
        _check("L15_no_placeholder_domains", not _ph, "HIGH",
               f"{len(_ph)}/{n} UCs have placeholder domain",
               "Tighten domain-rename rule in Q2.")
        # L17 NEXT_ACTIONS has no unexpected BLOCKING beyond L-gates themselves
        try:
            all_blocking = [a for a in NEXT_ACTIONS.snapshot() if a.get("severity") == "BLOCKING"]
            non_L = [a for a in all_blocking if not (a.get("rule_id","") or "").startswith("L_")]
        except Exception:
            non_L = []
        _check("L17_zero_non_L_blocking", not non_L, "INFO",
               f"{len(non_L)} BLOCKING entries outside L-gates",
               "Resolve BLOCKING next-actions listed in NEXT_ACTIONS_REPORT.md.")
        # L18 multi-table ratio: at least 30% of UCs should span 2+ tables when schema has 2+ tables.
        try:
            _schema_tables = set()
            for r in (full_schema_details or []):
                try:
                    if isinstance(r, dict):
                        _t = r.get("table_name") or r.get("table") or ""
                    elif isinstance(r, (tuple, list)) and len(r) >= 3:
                        _t = r[2]
                    else:
                        _t = ""
                    if _t:
                        _schema_tables.add(str(_t).strip().lower())
                except Exception:
                    continue
            _multi = 0
            for _uc in uc_list:
                _ti = str(_uc.get("Tables Involved", "") or "")
                _n = len([t for t in _ti.split(",") if t.strip()])
                if _n >= 2:
                    _multi += 1
            _ratio = (_multi / n) if n else 0.0
            _passed = (len(_schema_tables) < 2) or (_ratio >= 0.30)
            _check("L18_multi_table_ratio", _passed, "HIGH",
                   f"multi_table_UCs={_multi}/{n} (ratio={_ratio:.2f}); schema_tables={len(_schema_tables)}",
                   "Strengthen prompt FK-exploitation / STRUCTURAL JOIN HEURISTIC language; consider Q6 rebalance pass.")
        except Exception as _mt_err:
            _check("L18_multi_table_ratio", True, "INFO",
                   f"multi-table check skipped: {_mt_err}", "")
        # L19 production_readiness: aggregate Genie-runnable UC check.
        # J1 — Solution has >=2 column citations
        # J2 — Primary Table in known schema (covered by L14)
        # J3 — All backticked cols resolve to known schema cols (not SQL keywords)
        # J4 — Analytics Technique is specific (not umbrella — Rule 12 whitelist)
        # J5 — Result artifact named (result_table OR Solution names output)
        # J6 — No [ESTIMATE:] / [MISSING_DATA:] / [PLACEHOLDER:] placeholder tags
        # J7 — Inspire Score set and >0
        # J8 — Beneficiary non-empty and not generic
        import re as _rL
        _sql_kw = {"SELECT","FROM","WHERE","AND","OR","NOT","NULL","TRUE","FALSE","IS",
                   "ORDER","GROUP","BY","HAVING","LIMIT","CASE","WHEN","THEN","ELSE","END",
                   "JOIN","INNER","OUTER","LEFT","RIGHT","ON","AS","DISTINCT"}
        _all_schema_cols = set()
        for r in (full_schema_details or []):
            try:
                if isinstance(r, (tuple, list)) and len(r) >= 4:
                    _all_schema_cols.add(str(r[3]).lower())
                elif isinstance(r, dict):
                    _cn = r.get("column_name") or r.get("column") or ""
                    if _cn:
                        _all_schema_cols.add(str(_cn).lower())
            except Exception:
                continue
        _umbrella = {"ai analysis","data science","general analytics","advanced analytics",
                     "statistical analysis","ml analysis","ai","analytics"}
        _generic_bene = {"", "none", "n/a", "team", "everyone", "all", "business"}
        _prod_ready = 0
        _prod_fails = []
        for _uc in uc_list:
            _fails = []
            _sol = str(_uc.get("Solution","") or "")
            _stmt = str(_uc.get("Statement","") or "")
            _bv = str(_uc.get("Business Value","") or "")
            _blob = _sol + " " + _stmt + " " + _bv
            # J1
            _sol_cols = set(_rL.findall(r"`([A-Za-z_][A-Za-z0-9_]*)`", _sol))
            if len([c for c in _sol_cols if c.upper() not in _sql_kw]) < 2:
                _fails.append("J1_solution_<2_cols")
            # J3
            _all_backticked = set(_rL.findall(r"`([A-Za-z_][A-Za-z0-9_]*)`", _blob))
            _ungrounded = [c for c in _all_backticked
                           if c.upper() not in _sql_kw and c.lower() not in _all_schema_cols
                           and not c.lower().startswith("samples.")]
            if _ungrounded:
                # allow a few — just a smell, not a hard fail. threshold: >=3 ungrounded.
                if len(_ungrounded) >= 3:
                    _fails.append(f"J3_ungrounded_cols:{len(_ungrounded)}")
            # J4
            _tech = str(_uc.get("Analytics Technique","") or "").strip().lower()
            if _tech in _umbrella:
                _fails.append("J4_umbrella_technique")
            # J5
            _rt = str(_uc.get("result_table","") or "").strip()
            if not _rt and not _rL.search(r"\b(output|result|target|saves?|writes?|materializes?|persists?)\b", _sol.lower()):
                _fails.append("J5_no_output_named")
            # J6
            if _rL.search(r"\[(ESTIMATE|MISSING_DATA|PLACEHOLDER|TODO)\s*:", _blob):
                _fails.append("J6_placeholder_tag")
            # J7
            try:
                _is = float(_uc.get("Inspire Score", _uc.get("Priority", 0)) or 0)
            except Exception:
                _is = 0.0
            if _is <= 0:
                _fails.append("J7_no_score")
            # J8
            _bene = str(_uc.get("Beneficiary","") or "").strip().lower()
            if _bene in _generic_bene or len(_bene) < 3:
                _fails.append("J8_generic_beneficiary")
            if not _fails:
                _prod_ready += 1
            else:
                _prod_fails.append((_uc.get("No","?"), _fails))
        _ready_ratio = (_prod_ready / n) if n else 0
        _check("L19_production_readiness", _ready_ratio >= 0.80, "HIGH",
               f"production-ready {_prod_ready}/{n} ({int(_ready_ratio*100)}%); failing checks per-UC sample: {dict(_prod_fails[:5])}",
               "Review UC_PRODUCTION_READINESS NEXT_ACTIONS entries; tighten prompt/Q3 rules.")
        # emit per-UC summary as INFO (first 20)
        for _uc_no, _fails in _prod_fails[:20]:
            try:
                NEXT_ACTIONS.add("UC_PRODUCTION_READINESS", severity="INFO", phase="L", step="J-checks",
                                 evidence=f"UC {_uc_no}: failed production checks {_fails}")
            except Exception:
                pass

        try:
            self.processing_honesty["phase_L_results"] = {k: v["pass"] for k, v in results.items()}
            self.processing_honesty["phase_L_fail_count"] = sum(1 for v in results.values() if not v["pass"])
            self.processing_honesty["phase_L_production_ready_count"] = _prod_ready
            self.processing_honesty["phase_L_production_ready_ratio"] = round(_ready_ratio, 3)
        except Exception:
            pass
        return results

    def _rename_placeholder_domains_from_subdomains(self, use_cases: list) -> list:
        """When primary domain detection fell back to placeholder DomainN names,
        use the detected subdomains to ask the LLM for a single-word umbrella name
        per placeholder. Honors the DOMAIN_FINDER_PROMPT one-word rule. Fail-safe:
        retains placeholder if LLM fails or returns an invalid name."""
        import re as _re
        if not use_cases:
            return use_cases
        placeholder_re = _re.compile(r"^Domain\d+$")
        groups = {}
        for uc in use_cases:
            d = (uc.get("Business Domain") or "").strip()
            if placeholder_re.match(d):
                groups.setdefault(d, set()).add((uc.get("Subdomain") or "").strip())
        if not groups:
            return use_cases
        self.logger.warning(f"🔁 Placeholder domain rename: found {len(groups)} placeholder domain(s) after subdomain detection; attempting rename from subdomains")
        banned = {"operations","engagement","management","services","activities","processes","functions","solutions","systems","general"}
        valid_re = _re.compile(r"^[A-Z][a-z]+$")
        industries_str = ", ".join(self.industries) if hasattr(self, "industries") and self.industries else "UNKNOWN_INDUSTRY"
        rename_map = {}
        for placeholder, subs in groups.items():
            subs = sorted(s for s in subs if s)
            if not subs:
                continue
            subdomain_list = "\n".join(f"- {s}" for s in subs[:30])
            prompt = (
                "You are naming ONE business domain based on the subdomains assigned to it.\n\n"
                "HARD RULES (any violation is rejected):\n"
                "- EXACTLY ONE WORD. No spaces. No CamelCase. Only the first letter capitalized.\n"
                "- The word must be a concrete business noun (e.g., Revenue, Customers, Risk, Inventory, Suppliers, Franchise, Procurement, Pricing, Fraud, Maintenance, Logistics).\n"
                "- BANNED generic words: Operations, Engagement, Management, Services, Activities, Processes, Functions, Solutions, Systems, General.\n"
                "- Must fit the business context below.\n\n"
                f"Business Name: {self.business_name}\n"
                f"Industries: {industries_str}\n\n"
                "SUBDOMAINS in this domain:\n"
                f"{subdomain_list}\n\n"
                "Reply with ONLY the single-word domain name. No punctuation. No explanation."
            )
            try:
                raw = self.ai_agent._call_ai_query(
                    prompt=prompt,
                    prompt_name="PlaceholderDomainRename",
                    response_schema=None,
                    display_name=f"PlaceholderDomainRename_{placeholder}"
                )
                if not raw:
                    self.logger.warning(f"  ✗ {placeholder}: empty LLM response; keeping placeholder")
                    continue
                candidate = str(raw).strip().splitlines()[0].strip().strip(".").strip()
                if (
                    candidate
                    and candidate != placeholder
                    and valid_re.match(candidate)
                    and candidate.lower() not in banned
                ):
                    rename_map[placeholder] = candidate
                    self.logger.info(f"  ✓ {placeholder} → {candidate} (from {len(subs)} subdomains)")
                else:
                    self.logger.warning(f"  ✗ {placeholder}: invalid candidate '{candidate}'; keeping placeholder")
            except Exception as e:
                self.logger.warning(f"  ✗ {placeholder}: rename LLM failed ({get_clean_error_message(e)}); keeping placeholder")
        # Collision guard: if two placeholders map to same name, first keeps it, others retain placeholder
        seen = set()
        for k in list(rename_map.keys()):
            if rename_map[k] in seen:
                self.logger.warning(f"  ✗ name collision on '{rename_map[k]}'; keeping placeholder for {k}")
                del rename_map[k]
            else:
                seen.add(rename_map[k])
        if rename_map:
            for uc in use_cases:
                d = (uc.get("Business Domain") or "").strip()
                if d in rename_map:
                    uc["Business Domain"] = rename_map[d]
            self.logger.info(f"🏁 Rename complete: {len(rename_map)} of {len(groups)} placeholder domain(s) renamed")
        else:
            self.logger.warning(f"⚠️  Rename complete: 0 of {len(groups)} placeholder domain(s) could be renamed; placeholders retained")
        return use_cases

    def _assign_default_domains(self, use_cases: list) -> list:
        """
        FALLBACK: Assign default domains when LLM domain detection fails.
        Distributes use cases evenly across 5 default domains: Domain1, Domain2, Domain3, Domain4, Domain5.
        
        Args:
            use_cases: List of use case dictionaries
            
        Returns:
            List of use cases with default 'Business Domain' assigned
        """
        if not use_cases:
            return use_cases
        
        # Create 5 default domains (single-word names as required by validation)
        default_domains = ["Domain1", "Domain2", "Domain3", "Domain4", "Domain5"]
        
        # Distribute use cases evenly across domains
        result = []
        for i, uc in enumerate(use_cases):
            uc_copy = uc.copy()
            domain_idx = i % len(default_domains)
            uc_copy['Business Domain'] = default_domains[domain_idx]
            uc_copy['Subdomain'] = ''  # Will be assigned in subdomain step
            result.append(uc_copy)
        
        self.logger.info(f"📋 Default domain assignment: Distributed {len(result)} use cases across {len(default_domains)} domains")
        
        # Log distribution
        domain_counts = {}
        for uc in result:
            d = uc.get('Business Domain', '')
            domain_counts[d] = domain_counts.get(d, 0) + 1
        for domain, count in sorted(domain_counts.items()):
            self.logger.debug(f"   - {domain}: {count} use cases")
        
        return result

    def _assign_default_subdomains(self, use_cases: list, domain_name: str) -> list:
        """
        FALLBACK: Assign default subdomains when LLM subdomain detection fails.
        Distributes use cases evenly across default subdomains: "Sub Domain1", "Sub Domain2", etc.
        Creates enough subdomains to ensure minimum 2 use cases per subdomain.
        
        Args:
            use_cases: List of use case dictionaries
            domain_name: Name of the domain (for logging)
            
        Returns:
            List of use cases with default 'Subdomain' assigned
        """
        if not use_cases:
            return use_cases
        
        num_use_cases = len(use_cases)
        if num_use_cases <= 4:
            max_subdomains = 1
        else:
            max_subdomains = max(2, min(5, num_use_cases // 2))
        
        # Create default subdomains (2-word names as required by validation)
        default_subdomains = [f"Sub Domain{i}" for i in range(1, max_subdomains + 1)]
        
        # Distribute use cases evenly across subdomains
        result = []
        for i, uc in enumerate(use_cases):
            uc_copy = uc.copy()
            subdomain_idx = i % len(default_subdomains)
            uc_copy['Subdomain'] = default_subdomains[subdomain_idx]
            result.append(uc_copy)
        
        self.logger.info(f"📋 [{domain_name}] Default subdomain assignment: Distributed {len(result)} use cases across {len(default_subdomains)} subdomains")
        
        return result

    def _detect_subdomains_for_domain(self, domain_name: str, use_cases: list, language: str) -> list:
        """
        Detect subdomains for a single domain's use cases using LLM.
        This method is designed to be called in parallel for each domain.
        
        Args:
            domain_name: Name of the domain
            use_cases: List of use case dictionaries belonging to this domain
            language: Output language
            
        Returns:
            List of use cases with updated subdomains
        """
        from collections import defaultdict
        import io
        import csv
        
        log_prefix = f"[Domain: {domain_name}]"
        self.logger.info(f"{log_prefix} 🚀 Starting subdomain detection for {len(use_cases)} use cases (thread={threading.current_thread().name})...")
        log_print(f"{log_prefix} Subdomain detection started ({len(use_cases)} use cases)")
        
        if not use_cases:
            return use_cases
        
        try:
            # Convert use cases to CSV for LLM (Business Domain is set, Subdomain will be detected)
            output = io.StringIO()
            fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                         'Business Value', 'Beneficiary', 'Sponsor', 
                         'Tables Involved']
            writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(use_cases)
            use_cases_csv = output.getvalue()
            
            # Check context size (uses model-specific limits from TECHNICAL_CONTEXT)
            prompt_template = self.ai_agent.prompt_templates.get("SUBDOMAIN_DETECTOR_PROMPT", "")
            estimated_size = len(prompt_template) + len(use_cases_csv) + 1000
            MAX_CONTEXT_CHARS = get_max_context_chars(language, "SUBDOMAIN_DETECTOR_PROMPT")
            
            if estimated_size > MAX_CONTEXT_CHARS:
                self.logger.warning(
                    f"{log_prefix} Subdomain prompt size ({estimated_size:,} chars) exceeds MAX_CONTEXT_CHARS ({MAX_CONTEXT_CHARS:,}). "
                    f"Splitting into smaller batches for subdomain detection."
                )
                # CONTEXT SPLITTING: Split use cases into smaller batches and process each
                return self._detect_subdomains_with_context_splitting(
                    domain_name, use_cases, language, prompt_template, MAX_CONTEXT_CHARS
                )
            
            max_attempts = self.max_retry_attempts + 1
            prompt_vars = {
                "domain_name": domain_name,
                "use_cases_csv": use_cases_csv,
                "output_language": language,
                "business_name": self.business_name,
                "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                "business_context": self._get_business_context_fallback(),
                "previous_violations": "",
                "generation_instructions_section": self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT"),
            }
            
            for attempt in range(1, max_attempts + 1):
                try:
                    self.logger.debug(f"{log_prefix} Subdomain detection attempt {attempt}/{max_attempts}...")
                    
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Detect_Subdomains_{domain_name}_{language}_Attempt{attempt}",
                        worker_prompt_path="SUBDOMAIN_DETECTOR_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None,
                        timeout_override=self.llm_timeout_seconds,  # Explicit timeout
                        skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                    )
                    
                    # Validate response
                    if not response_raw or len(response_raw.strip()) == 0:
                        raise ValueError("LLM returned empty response")
                    
                    # Clean response (remove markdown fences if present)
                    response_clean = clean_json_response(response_raw)
                    if not response_clean or len(response_clean.strip()) == 0:
                        raise ValueError("Cleaned response is empty")
                    
                    # Parse CSV using centralized utility
                    try:
                        csv_rows = CSVParser.parse_csv_string(
                            response_clean,
                            logger=self.logger,
                            context="Subdomain detection"
                        )
                        subdomain_assignment_map = {}
                        row_count = 0
                        
                        for row in csv_rows:
                            row_count += 1
                            # Handle both possible column names - also handle None values
                            uc_id_raw = row.get('use_case_id', '') or ''
                            subdomain_raw = row.get('subdomain', '') or ''
                            uc_id = uc_id_raw.strip() if isinstance(uc_id_raw, str) else str(uc_id_raw).strip()
                            subdomain = subdomain_raw.strip() if isinstance(subdomain_raw, str) else str(subdomain_raw).strip()
                            
                            if uc_id and subdomain:
                                subdomain_assignment_map[uc_id] = subdomain
                        
                        if row_count == 0:
                            raise ValueError("CSV has no data rows")
                        
                        if not subdomain_assignment_map:
                            raise ValueError("No valid subdomain assignments found in CSV")
                            
                    except Exception as csv_err:
                        self.logger.error(f"{log_prefix} CSV parsing failed. Raw response (first 500 chars): {response_raw[:500]}")
                        self.logger.error(f"{log_prefix} Cleaned response (first 500 chars): {response_clean[:500]}")
                        raise ValueError(f"Failed to parse CSV: {csv_err}")
                    
                    # Apply subdomain assignments
                    subdomain_assigned_use_cases = []
                    for uc in use_cases:
                        uc_copy = uc.copy()
                        uc_id = uc_copy.get('No', '')
                        if uc_id in subdomain_assignment_map:
                            uc_copy['Subdomain'] = subdomain_assignment_map[uc_id]
                        subdomain_assigned_use_cases.append(uc_copy)
                    
                    # Validate subdomain assignments
                    violations = []
                    
                    # Group by subdomain
                    subdomain_usecases = defaultdict(list)
                    for uc in subdomain_assigned_use_cases:
                        subdomain = uc.get('Subdomain', '').strip()
                        if subdomain:
                            subdomain_usecases[subdomain].append(uc)
                    
                    total_subdomains = len(subdomain_usecases)
                    total_ucs_in_domain = sum(len(ucs) for ucs in subdomain_usecases.values())
                    min_subdomains = 1 if total_ucs_in_domain <= 4 else 2
                    if total_subdomains < min_subdomains:
                        violations.append(f"Only {total_subdomains} subdomain(s), minimum required: {min_subdomains}")
                    if total_subdomains > 10:
                        violations.append(f"Too many subdomains: {total_subdomains}, maximum allowed: 10")
                    
                    # Validate subdomain naming (must be 2 words)
                    for subdomain in subdomain_usecases.keys():
                        word_count = len(subdomain.split())
                        if word_count != 2:
                            violations.append(f"Subdomain '{subdomain}' has {word_count} word(s), must be exactly 2 words")
                    
                    # Validate use cases per subdomain (minimum 2)
                    for subdomain, ucs in subdomain_usecases.items():
                        count = len(ucs)
                        if count < 2:
                            violations.append(f"Subdomain '{subdomain}' has only {count} use case(s), minimum required: 2")
                    
                    # If no violations, success!
                    if not violations:
                        self.logger.debug(f"{log_prefix} ✅ Subdomain detection successful! Created {total_subdomains} subdomains")
                        return subdomain_assigned_use_cases
                    else:
                        self.logger.warning(f"{log_prefix} ⚠️ Subdomain detection attempt {attempt} has {len(violations)} violations")
                        if attempt == max_attempts:
                            self.logger.warning(f"{log_prefix} Max attempts reached. Auto-fixing subdomain size violations...")
                            small_subdomains = [sd for sd, ucs in subdomain_usecases.items() if len(ucs) < 2]
                            if small_subdomains and len(subdomain_usecases) > 1:
                                largest_subdomain = max(subdomain_usecases.keys(), key=lambda sd: len(subdomain_usecases[sd]))
                                for small_sd in small_subdomains:
                                    if small_sd == largest_subdomain:
                                        continue
                                    self.logger.warning(f"{log_prefix} Merging subdomain '{small_sd}' ({len(subdomain_usecases[small_sd])} use cases) into '{largest_subdomain}'")
                                    for uc in subdomain_assigned_use_cases:
                                        if uc.get('Subdomain', '').strip() == small_sd:
                                            uc['Subdomain'] = largest_subdomain
                                    subdomain_usecases[largest_subdomain].extend(subdomain_usecases.pop(small_sd))
                            
                            max_subdomains = 10
                            
                            def _find_best_merge_target(src_sd, candidates, sd_usecases):
                                src_words = set(src_sd.lower().split())
                                src_uc_names = {uc.get('Name', '').lower() for uc in sd_usecases[src_sd]}
                                best_target = None
                                best_score = -1
                                for cand in candidates:
                                    if cand == src_sd:
                                        continue
                                    cand_words = set(cand.lower().split())
                                    name_overlap = sum(
                                        1 for uc in sd_usecases[cand]
                                        if any(w in uc.get('Name', '').lower() for w in src_words)
                                    )
                                    word_overlap = len(src_words & cand_words)
                                    score = word_overlap * 10 + name_overlap
                                    if score > best_score:
                                        best_score = score
                                        best_target = cand
                                if best_target is None:
                                    best_target = min(
                                        (c for c in candidates if c != src_sd),
                                        key=lambda c: len(sd_usecases[c])
                                    )
                                return best_target
                            
                            while len(subdomain_usecases) > max_subdomains:
                                sorted_sds = sorted(subdomain_usecases.keys(), key=lambda sd: len(subdomain_usecases[sd]))
                                smallest_sd = sorted_sds[0]
                                merge_target = _find_best_merge_target(smallest_sd, list(subdomain_usecases.keys()), subdomain_usecases)
                                self.logger.warning(
                                    f"{log_prefix} Too many subdomains ({len(subdomain_usecases)}>{max_subdomains}). "
                                    f"Merging '{smallest_sd}' ({len(subdomain_usecases[smallest_sd])} UCs) into '{merge_target}' ({len(subdomain_usecases[merge_target])} UCs)"
                                )
                                for uc in subdomain_assigned_use_cases:
                                    if uc.get('Subdomain', '').strip() == smallest_sd:
                                        uc['Subdomain'] = merge_target
                                subdomain_usecases[merge_target].extend(subdomain_usecases.pop(smallest_sd))
                            
                            remaining_violations = [v for v in violations if "has only" not in v and "Too many subdomains" not in v]
                            if remaining_violations:
                                self.logger.warning(f"{log_prefix} Remaining violations after auto-fix: {'; '.join(remaining_violations[:3])}")
                            return subdomain_assigned_use_cases
                        else:
                            # Prepare violation summary for retry
                            violation_summary = "\n\n**🚨 PREVIOUS ATTEMPT VIOLATIONS - YOU MUST FIX THESE 🚨**:\n"
                            violation_summary += "\n".join([f"- {v}" for v in violations])
                            prompt_vars['previous_violations'] = violation_summary
                            continue
                
                except CascadeRebatchError as cre:
                    self.logger.warning(
                        f"{log_prefix} Subdomain detection cascade rebatch needed after '{cre.failed_model_name}' failed. "
                        f"Switching to context-splitting for model '{cre.target_model_name}' "
                        f"(limit: {cre.target_context_limit_chars:,} chars)"
                    )
                    return self._detect_subdomains_with_context_splitting(
                        domain_name, use_cases, language, prompt_template,
                        cre.target_context_limit_chars, cre.target_model_endpoint
                    )

                except Exception as e:
                    self.logger.error(f"{log_prefix} Subdomain detection attempt {attempt} failed: {get_clean_error_message(e)}")
                    if attempt == max_attempts:
                        self.logger.error(f"{log_prefix} Max attempts reached. Using DEFAULT subdomains as fallback...")
                        fallback_use_cases = self._assign_default_subdomains(use_cases, domain_name)
                        self.logger.warning(f"{log_prefix} ✅ Fallback complete: Assigned default subdomains to {len(fallback_use_cases)} use cases")
                        return fallback_use_cases
            
            self.logger.warning(f"{log_prefix} No subdomain assignments made. Using DEFAULT subdomains as fallback...")
            return self._assign_default_subdomains(use_cases, domain_name)
            
        except Exception as e:
            self.logger.error(f"{log_prefix} Subdomain detection failed with error: {get_clean_error_message(e)}. Using DEFAULT subdomains as fallback...")
            return self._assign_default_subdomains(use_cases, domain_name)

    def _detect_subdomains_with_context_splitting(self, domain_name: str, use_cases: list, language: str, prompt_template: str, max_context_chars: int, start_from_model: str = None) -> list:
        """
        Handle subdomain detection when context exceeds max size by splitting into batches.
        Processes each batch separately and merges results. Supports cascade rebatching.
        
        Args:
            domain_name: Name of the domain
            use_cases: List of use case dictionaries
            language: Output language
            prompt_template: The prompt template string
            max_context_chars: Maximum context size in characters
            start_from_model: Optional model endpoint to start cascade from
            
        Returns:
            List of use cases with assigned subdomains
        """
        import io
        import csv
        from collections import defaultdict
        
        log_prefix = f"[Domain: {domain_name}][Context Split]"
        
        MAX_CASCADE_RETRIES = 3
        effective_limit = max_context_chars
        effective_start_model = start_from_model
        sample_fieldnames = ['No', 'Name', 'type', 'Analytics Technique', 'Statement', 'Solution', 
                            'Business Value', 'Beneficiary', 'Sponsor', 'Tables Involved']

        all_subdomain_assignments = {}
        _sub_lock = threading.Lock()
        remaining_use_cases = list(use_cases)

        for cascade_retry in range(MAX_CASCADE_RETRIES):
            if not remaining_use_cases:
                break

            sample_output = io.StringIO()
            sample_writer = csv.DictWriter(sample_output, fieldnames=sample_fieldnames, extrasaction='ignore')
            sample_writer.writeheader()
            if remaining_use_cases:
                sample_writer.writerow(remaining_use_cases[0])
            avg_use_case_size = max(100, len(sample_output.getvalue()))

            available_chars = effective_limit - len(prompt_template) - 2000
            batch_size = max(2, available_chars // max(1, avg_use_case_size))

            self.logger.info(
                f"{log_prefix} Splitting {len(remaining_use_cases)} use cases into batches of ~{batch_size} "
                f"(context limit: {effective_limit:,} chars, model: {effective_start_model or 'default cascade'}, "
                f"already assigned: {len(all_subdomain_assignments)})"
            )

            batches = []
            for i in range(0, len(remaining_use_cases), batch_size):
                batches.append(remaining_use_cases[i:i + batch_size])

            self.logger.info(f"{log_prefix} Created {len(batches)} batches for subdomain detection")

            num_batches = len(batches)
            cascade_rebatch_triggered = False
            cascade_rebatch_error = None

            def _process_subdomain_batch(batch_idx, batch, _start_model=effective_start_model):
                self.logger.info(f"{log_prefix} Processing batch {batch_idx}/{num_batches} ({len(batch)} use cases)")
                local_assignments = {}
                try:
                    batch_output = io.StringIO()
                    batch_writer = csv.DictWriter(batch_output, fieldnames=sample_fieldnames, extrasaction='ignore')
                    batch_writer.writeheader()
                    batch_writer.writerows(batch)
                    batch_csv = batch_output.getvalue()

                    max_attempts = self.max_retry_attempts + 1
                    prompt_vars = {
                        "domain_name": domain_name,
                        "use_cases_csv": batch_csv,
                        "output_language": language,
                        "business_name": self.business_name,
                        "industries": ", ".join(self.industries) if hasattr(self, 'industries') and self.industries else "UNKNOWN_INDUSTRY",
                        "business_context": self._get_business_context_fallback(),
                        "previous_violations": "",
                        "generation_instructions_section": self._get_generation_instruction_for_prompt("SUBDOMAIN_DETECTOR_PROMPT"),
                    }

                    batch_success = False
                    for attempt in range(1, max_attempts + 1):
                        try:
                            response_raw = self.ai_agent.run_worker(
                                step_name=f"Detect_Subdomains_{domain_name}_Batch{batch_idx}_{language}_Attempt{attempt}",
                                worker_prompt_path="SUBDOMAIN_DETECTOR_PROMPT",
                                prompt_vars=prompt_vars,
                                response_schema=None,
                                timeout_override=self.llm_timeout_seconds,
                                start_from_model=_start_model,
                                skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                            )

                            if not response_raw or len(response_raw.strip()) == 0:
                                raise ValueError("LLM returned empty response")

                            response_clean = clean_json_response(response_raw)
                            if not response_clean or len(response_clean.strip()) == 0:
                                raise ValueError("Cleaned response is empty")

                            csv_rows = CSVParser.parse_csv_string(
                                response_clean,
                                logger=self.logger,
                                context=f"Subdomain detection batch {batch_idx}"
                            )

                            for row in csv_rows:
                                uc_id_raw = row.get('use_case_id', '') or ''
                                subdomain_raw = row.get('subdomain', '') or ''
                                uc_id = str(uc_id_raw).strip()
                                subdomain = str(subdomain_raw).strip()
                                if uc_id and subdomain:
                                    local_assignments[uc_id] = subdomain

                            batch_success = True
                            self.logger.info(f"{log_prefix} Batch {batch_idx} completed successfully")
                            break

                        except CascadeRebatchError:
                            raise
                        except Exception as attempt_err:
                            self.logger.warning(f"{log_prefix} Batch {batch_idx} attempt {attempt} failed: {attempt_err}")
                            if attempt == max_attempts:
                                self.logger.error(f"{log_prefix} Batch {batch_idx} failed after {max_attempts} attempts")

                    if not batch_success:
                        for uc in batch:
                            uc_id = str(uc.get('No', '')).strip()
                            if uc_id:
                                local_assignments.setdefault(uc_id, f"General {domain_name}")

                except CascadeRebatchError:
                    raise
                except Exception as batch_err:
                    self.logger.error(f"{log_prefix} Batch {batch_idx} processing error: {batch_err}")
                    for uc in batch:
                        uc_id = str(uc.get('No', '')).strip()
                        if uc_id:
                            local_assignments.setdefault(uc_id, f"General {domain_name}")

                with _sub_lock:
                    all_subdomain_assignments.update(local_assignments)

            batch_workers = min(4, num_batches)
            try:
                if num_batches <= 1:
                    _process_subdomain_batch(1, batches[0]) if batches else None
                else:
                    _sb_total_timeout = self.llm_timeout_seconds * 3 * len(batches) + 120
                    with _SafeExecutorContext(max_workers=batch_workers, thread_name_prefix="sub_batch", logger=self.logger, name="SubdomainBatch") as ctx:
                        sb_futures = [ctx.submit(_process_subdomain_batch, i, b) for i, b in enumerate(batches, 1)]
                        try:
                            for f in safe_as_completed(sb_futures, timeout=_sb_total_timeout):
                                try:
                                    f.result(timeout=self.llm_timeout_seconds * 3)
                                except CascadeRebatchError as cre:
                                    cascade_rebatch_triggered = True
                                    cascade_rebatch_error = cre
                                    ctx.mark_timed_out()
                                    for remaining_f in sb_futures:
                                        if not remaining_f.done():
                                            remaining_f.cancel()
                                    break
                                except Exception as exc:
                                    self.logger.error(f"{log_prefix} Subdomain batch task failed: {get_clean_error_message(exc)}")
                        except concurrent.futures.TimeoutError:
                            ctx.mark_timed_out()
                            self.logger.error(f"{log_prefix} Subdomain batch timed out globally ({_sb_total_timeout}s)")
            except CascadeRebatchError as cre:
                cascade_rebatch_triggered = True
                cascade_rebatch_error = cre

            if cascade_rebatch_triggered and cascade_rebatch_error:
                effective_limit = cascade_rebatch_error.target_context_limit_chars
                effective_start_model = cascade_rebatch_error.target_model_endpoint
                remaining_use_cases = [
                    uc for uc in remaining_use_cases
                    if str(uc.get('No', '')).strip() not in all_subdomain_assignments
                ]
                self.logger.warning(
                    f"{log_prefix} CASCADE RETRY {cascade_retry + 1}/{MAX_CASCADE_RETRIES}: "
                    f"Rebatching {len(remaining_use_cases)} remaining use cases for model "
                    f"'{cascade_rebatch_error.target_model_name}' (limit: {effective_limit:,} chars, "
                    f"already assigned: {len(all_subdomain_assignments)})"
                )
                if not remaining_use_cases:
                    self.logger.info(f"{log_prefix} All use cases assigned despite cascade error")
                    break
                continue

            break

        result = []
        for uc in use_cases:
            uc_copy = uc.copy()
            uc_id = str(uc.get('No', '')).strip()
            subdomain = all_subdomain_assignments.get(uc_id, f"General {domain_name}")
            uc_copy['Subdomain'] = subdomain
            result.append(uc_copy)

        subdomain_counts = defaultdict(int)
        for uc in result:
            subdomain_counts[uc.get('Subdomain', 'Unknown')] += 1

        self.logger.info(f"{log_prefix} Context splitting complete: {len(result)} use cases assigned to {len(subdomain_counts)} subdomains")
        for subdomain, count in sorted(subdomain_counts.items(), key=lambda x: -x[1]):
            self.logger.debug(f"{log_prefix}   - {subdomain}: {count} use cases")

        return result

    def _report_table_statistics(self, use_cases: list):
        """
        Reports statistics on table inclusion/exclusion based on generated use cases.
        Compares tables used in use cases vs. total available tables.
        """
        try:
            # Get all available tables from the data loader
            all_available_tables = set()
            if self.data_loader and hasattr(self.data_loader, 'db_details_cache'):
                for (catalog, schema, table, _, _, _) in self.data_loader.db_details_cache:
                    fqtn = f"`{catalog}`.`{schema}`.`{table}`"
                    all_available_tables.add(fqtn)
            
            total_available = len(all_available_tables)
            
            if total_available == 0:
                self.logger.warning("No available tables found in data loader cache. Skipping table statistics report.")
                return
            
            # Extract all unique tables used in use cases
            tables_used = set()
            for uc in use_cases:
                tables_involved = uc.get('Tables Involved', '')
                if tables_involved:
                    # Parse comma-separated table list
                    for table in tables_involved.split(','):
                        table = table.strip()
                        if table:
                            tables_used.add(table)
            
            total_included = len(tables_used)
            total_excluded = total_available - total_included
            
            # Calculate percentages
            pct_included = (total_included / total_available * 100) if total_available > 0 else 0
            pct_excluded = (total_excluded / total_available * 100) if total_available > 0 else 0
            
            log_print("\n" + "="*80)
            log_print("TABLE UTILIZATION REPORT")
            log_print("="*80)
            log_print(f"Total Available Tables: {total_available}")
            log_print(f"Tables Included in Use Cases: {total_included} ({pct_included:.1f}%)")
            log_print(f"Tables Excluded (No Business Value): {total_excluded} ({pct_excluded:.1f}%)")
            log_print("="*80 + "\n")
            
            # Log it as well
            self.logger.info(f"Table Statistics: {total_included}/{total_available} tables included ({pct_included:.1f}%), {total_excluded} excluded ({pct_excluded:.1f}%)")
            
        except Exception as e:
            self.logger.error(f"Failed to generate table statistics report: {get_clean_error_message(e)}")
    
    def _priority_sort_key(self, use_case: dict) -> tuple:
        """
        Returns a tuple for sorting use cases by Quality label (descending).
        Ultra High -> Very High -> High -> Medium -> Low -> Very Low -> Ultra Low
        """
        quality_map = {
            "Ultra High": 0,
            "Very High": 1,
            "High": 2,
            "Medium": 3,
            "Low": 4,
            "Very Low": 5,
            "Ultra Low": 6,
            "N/A": 7
        }
        quality_label = use_case.get('Quality', 'N/A')
        quality_order = quality_map.get(quality_label, 7)
        use_case_no = use_case.get('No', 'N-999-AI')
        return (quality_order, use_case_no)
    
    def _natural_sort_key(self, use_case: dict) -> tuple:
        """
        Natural sort key for use case IDs to ensure proper ordering.
        
        Handles IDs like: N01-AI01, N01-AI02, ..., N01-AI10, N01-AI11, N01-ST01, etc.
        Standard string sort would incorrectly order: AI1, AI10, AI11, AI2, AI3...
        Natural sort correctly orders: AI1, AI2, AI3, ..., AI10, AI11...
        
        Returns tuple: (domain_num, source_type, sequence_num, original_id)
        Example: "N01-AI05" -> (1, 'AI', 5, 'N01-AI05')
        """
        import re
        use_case_id = use_case.get('No', 'N99-ZZ999')
        
        # Parse ID format: N##-XX## (e.g., N01-AI05, N02-ST10)
        match = re.match(r'N(\d+)-([A-Z]+)(\d+)', use_case_id)
        if match:
            domain_num = int(match.group(1))
            source_type = match.group(2)  # AI or ST
            seq_num = int(match.group(3))
            # Sort AI before ST (alphabetically)
            return (domain_num, source_type, seq_num, use_case_id)
        
        # Fallback: Try to parse older formats like AI-XXX-U## or just extract numbers
        # Pattern: AI-XXX-U## or ST-XXX-U##
        match2 = re.match(r'(AI|ST)-([A-Z0-9_]+)-U(\d+)', use_case_id)
        if match2:
            source_type = match2.group(1)
            seq_num = int(match2.group(3))
            return (99, source_type, seq_num, use_case_id)
        
        # Final fallback: extract any numbers from the ID for some ordering
        numbers = re.findall(r'\d+', use_case_id)
        if numbers:
            # Use first number as domain, last number as sequence
            return (int(numbers[0]) if len(numbers) > 0 else 99, 
                    'ZZ', 
                    int(numbers[-1]) if len(numbers) > 0 else 999, 
                    use_case_id)
        
        # No numbers found - sort at the end
        return (999, 'ZZ', 999, use_case_id)
    
    def _calculate_domain_impact_score(self, use_cases: list) -> float:
        """
        Calculates the average priority score for a domain to determine its impact.
        Higher score = more impactful domain.
        """
        if not use_cases:
            return 0.0
        # Ensure Priority is converted to float (it might be stored as string or int)
        total_score = sum(float(uc.get('Priority', 5.0)) for uc in use_cases)
        return total_score / len(use_cases)

    def _build_notebook_url(self, object_id: int) -> str:
        try:
            host = self.w_client.config.host
            if not host or not object_id:
                return ''
            host = host.rstrip('/')
            return f"{host}/editor/notebooks/{object_id}"
        except Exception:
            return ''

    def _get_use_case_notebook_url(self, uc: dict) -> str:
        try:
            uc_id = uc.get('No', 'UNKNOWN')
            object_id = self._notebook_object_ids.get(uc_id)
            if not object_id:
                domain_folder = self._sanitize_name(uc.get('Business Domain', 'Other'))
                subdomain_folder = self._sanitize_name(uc.get('Subdomain', 'General'))
                uc_name = uc.get('Name', '')
                notebook_name = f"{uc_id}-{self._sanitize_name(uc_name)}"
                notebook_workspace_path = f"{self.notebook_output_dir}/{domain_folder}/{subdomain_folder}/{notebook_name}"
                try:
                    _status = self.w_client.workspace.get_status(notebook_workspace_path)
                    object_id = getattr(_status, 'object_id', None)
                except Exception:
                    pass
                if not object_id:
                    try:
                        _status = self.w_client.workspace.get_status(notebook_workspace_path + ".ipynb")
                        object_id = getattr(_status, 'object_id', None)
                    except Exception:
                        pass
                if object_id:
                    self._notebook_object_ids[uc_id] = object_id
            if not object_id:
                return ''
            return self._build_notebook_url(object_id)
        except Exception:
            return ''


    def assemble_use_case_notebooks(self, all_use_cases: list, translations: dict, summary_dict: dict = None):
        self.logger.debug("--- Starting Use Case Notebook Assembly (English) ---")
        if not all_use_cases:
            self.logger.warning("No use cases were provided for notebook assembly.")
            return

        total_use_cases = len(all_use_cases)
        domains_seen = set()
        subdomains_seen = set()
        for uc in all_use_cases:
            domains_seen.add(uc.get('Business Domain', 'Other'))
            subdomains_seen.add((uc.get('Business Domain', 'Other'), uc.get('Subdomain', 'General')))

        self.logger.info(f"📚 Generating {total_use_cases} notebooks (1 per use case) across {len(domains_seen)} domain folders, {len(subdomains_seen)} subdomain folders IN PARALLEL...")
        log_print(f"📚 Generating {total_use_cases} notebooks (1 per use case) in {len(domains_seen)} domains / {len(subdomains_seen)} subdomains (max {self.max_parallelism} concurrent)...")

        sorted_use_cases = sorted(all_use_cases, key=self._natural_sort_key)

        def create_notebook_for_use_case(task_data):
            idx, use_case = task_data
            use_case_id = use_case.get('No', 'UNKNOWN')
            use_case_name = use_case.get('Name', '')
            domain = use_case.get('Business Domain', 'Other')
            subdomain = use_case.get('Subdomain', 'General')

            domain_folder = self._sanitize_name(domain)
            subdomain_folder = self._sanitize_name(subdomain)
            notebook_name = f"{use_case_id}-{self._sanitize_name(use_case_name)}"
            rel_path = f"{domain_folder}/{subdomain_folder}/{notebook_name}"

            self.logger.info(f"📝 [{idx}/{total_use_cases}] Assembling notebook '{rel_path}'...")

            try:
                self._assemble_single_use_case_notebook(
                    use_case=use_case, translations=translations,
                    domain_folder=domain_folder, subdomain_folder=subdomain_folder,
                    notebook_name=notebook_name, summary_dict=summary_dict
                )
                self.logger.info(f"✅ [{idx}/{total_use_cases}] Notebook '{rel_path}' completed")
                return (idx, rel_path, True)
            except Exception as e:
                self.logger.error(f"❌ [{idx}/{total_use_cases}] Notebook '{rel_path}' failed: {get_clean_error_message(e)}")
                return (idx, rel_path, False)

        notebook_parallelism, reason = calculate_adaptive_parallelism(
            "notebook_generation", self.max_parallelism,
            num_items=total_use_cases,
            num_domains=len(domains_seen),
            is_llm_operation=False, logger=self.logger
        )
        log_adaptive_parallelism_decision("notebook_generation", notebook_parallelism, self.max_parallelism, reason)

        _notebook_total_timeout = max(600, total_use_cases * 60)
        with _SafeExecutorContext(max_workers=notebook_parallelism, thread_name_prefix="NotebookGen", logger=self.logger, name="NotebookGen") as ctx:
            task_data_list = [(i, uc) for i, uc in enumerate(sorted_use_cases, start=1)]
            futures = [ctx.submit(create_notebook_for_use_case, td) for td in task_data_list]

            completed_count = 0
            failed_count = 0
            try:
                for future in safe_as_completed(futures, timeout=_notebook_total_timeout):
                    try:
                        i, notebook_path, success = future.result()
                        if success:
                            completed_count += 1
                            log_print(f"   ✅ Notebook {i}/{total_use_cases} completed: {notebook_path}")
                        else:
                            failed_count += 1
                            log_print(f"   ❌ Notebook {i}/{total_use_cases} failed: {notebook_path}")
                    except Exception as e:
                        failed_count += 1
                        self.logger.error(f"Notebook generation future failed: {get_clean_error_message(e)}")
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⏱️ Notebook generation total timeout after {_notebook_total_timeout}s.")
                for future in futures:
                    if not future.done():
                        future.cancel()
                        failed_count += 1

        self.logger.info(f"✅ Notebook generation complete: {completed_count}/{total_use_cases} succeeded, {failed_count} failed across {len(domains_seen)} domains")
        log_print(f"✅ All {total_use_cases} use case notebooks processed: {completed_count} succeeded, {failed_count} failed")

    # === NEW: Helper method to determine if use case should show examples ===
    def _ensure_sql_results_cache_dir(self):
        if getattr(self, "sql_results_cache_dir", None):
            return self.sql_results_cache_dir
        cache_dir = os.path.join(tempfile.gettempdir(), f"inspire_sql_cache_{self.business_name.replace(' ', '_')}")
        os.makedirs(cache_dir, exist_ok=True)
        self.sql_results_cache_dir = cache_dir
        return cache_dir

    def _get_cached_sql_result(self, use_case_id: str) -> dict:
        """
        Retrieve cached SQL result from disk.
        
        Args:
            use_case_id: Use case ID
            
        Returns:
            Cached result dict or error dict if not found
        """
        if not hasattr(self, 'sql_results_cache_dir'):
            return {
                'status': 'error',
                'data': [],
                'message': 'Cache not initialized'
            }
        
        cache_file = os.path.join(self.sql_results_cache_dir, f"{use_case_id}.json")
        
        if not os.path.exists(cache_file):
            return {
                'status': 'error',
                'data': [],
                'message': 'Cached result not found'
            }
        
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            self.logger.error(f"Failed to read cached result for {use_case_id}: {str(e)[:100]}")
            return {
                'status': 'error',
                'data': [],
                'message': f'Failed to read cache: {str(e)[:100]}'
            }

    # === MODIFIED: PDF Generation (Req 1, 2, 3) ===
    def generate_catalog_pdf(self, language: str, lang_abbr: str, translations: dict, summary_dict: dict, grouped_data: dict, transliterated_name: str):
        self.logger.info(f"--- Starting PDF Catalog Generation for {language} ---")
        
        t = translations
        is_rtl = (language == "Arabic")
        
        def _install_dependencies(logger_instance) -> bool:
            try:
                import weasyprint
                logger_instance.info("PDF package (weasyprint) already installed.")
                return True
            except ImportError:
                logger_instance.info("Installing required PDF package (weasyprint)...")
                try: 
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "weasyprint"])
                    import weasyprint
                    logger_instance.info("Successfully installed weasyprint.")
                    return True
                except Exception as e: 
                    logger_instance.error(f"Failed to install weasyprint: {get_clean_error_message(e)}")
                    print("ERROR: Failed to install 'weasyprint'. PDF generation cannot continue.", file=sys.stderr)
                    return False

        def _build_html(grouped_data: dict, summary_dict: dict, business_name: str, translations: dict, is_rtl: bool) -> str:
            self.logger.info(f"Building HTML for PDF ({language})...")
            t = translations; now = datetime.datetime.now().strftime("%Y-%m-%d")
            direction = "rtl" if is_rtl else "ltr"; align = "right" if is_rtl else "left"
            def e(text): return html.escape(str(text))

            # === MODIFIED: CSS WITH FIXES (Request #13) ===
            # 1. @import rules MUST be at the very top (before @font-face)
            # 2. Removed unsupported properties: box-shadow, text-shadow
            # 3. Font warnings will be suppressed in WeasyPrint config
            css = f"""
            /* @import rules MUST be first in CSS */
            @import url('https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Devanagari:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Arabic:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+JP:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Tamil:wght@300;400;700&display=swap');
            @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Thai:wght@300;400;700&display=swap');
           
            /* Font-face declarations with local fallbacks */
            @font-face {{
                font-family: 'NotoSansDevanagari';
                src: local('Noto Sans Devanagari'), local('Lohit Devanagari'), local('Mangal');
                font-weight: normal;
                font-style: normal;
            }}
            @font-face {{
                font-family: 'NotoSansArabic';
                src: local('Noto Sans Arabic'), local('Arial Unicode MS'), local('DejaVu Sans');
                font-weight: normal;
                font-style: normal;
            }}
            @font-face {{
                font-family: 'NotoSansCJK';
                src: local('Noto Sans CJK'), local('Microsoft YaHei'), local('SimSun'), local('MS Gothic');
                font-weight: normal;
                font-style: normal;
            }}
            @font-face {{
                font-family: 'NotoSansJP';
                src: local('Noto Sans JP'), local('Yu Gothic'), local('MS Gothic'), local('Meiryo');
                font-weight: normal;
                font-style: normal;
            }}
            @font-face {{
                font-family: 'NotoSansKR';
                src: local('Noto Sans KR'), local('Malgun Gothic'), local('Gulim');
                font-weight: normal;
                font-style: normal;
            }}
            
            @page {{
                size: A4; margin: 2.5cm;
                @bottom-left {{
                    content: 'Databricks Inspire AI';
                    font-family: 'Roboto', 'Noto Sans Devanagari', 'Noto Sans Arabic', 'Noto Sans SC', 'Noto Sans JP', 'Noto Sans KR', 'Noto Sans Tamil', sans-serif; font-size: 9pt; color: #555;
                }}
                @bottom-right {{
                    content: '{now}';
                    font-family: 'Roboto', 'Noto Sans Devanagari', 'Noto Sans Arabic', 'Noto Sans SC', 'Noto Sans JP', 'Noto Sans KR', 'Noto Sans Tamil', sans-serif; font-size: 9pt; color: #555;
                }}
            }}
            html {{ counter-reset: page-counter; }}
            body {{ 
                font-family: 'Roboto', 'Noto Sans Devanagari', 'Noto Sans Arabic', 'Noto Sans SC', 'Noto Sans JP', 'Noto Sans KR', 'Noto Sans Tamil', 'Noto Sans Thai', sans-serif; color: #333; line-height: 1.6; 
                direction: {direction}; text-align: {align}; 
            }}
            h1, h2, h3 {{ font-weight: 700; color: #003366; margin-bottom: 0.5em; text-align: {align}; }}
            
            /* Page counter logic */
            h1.page-title {{ font-size: 24pt; counter-increment: page-counter; }}
            h2.page-title {{ font-size: 22pt; counter-increment: page-counter; }}
            h2.domain-header {{ font-size: 18pt; counter-increment: none; }}
            
            h3 {{ color: #0B579E; font-size: 14pt; border-bottom: 2px solid #FF6900; padding-bottom: 5px; }}
            p {{ margin-bottom: 1.2em; text-align: {align}; }}
            table {{ width: 100%; border-collapse: collapse; margin-bottom: 2em; page-break-inside: auto; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: {align}; word-wrap: break-word; }}
            th {{ background-color: #003366; color: white; font-weight: 700; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            tr {{ page-break-inside: avoid; }}
            
            /* Enhanced cover page with gradient */
            .cover-page {{ 
                display: flex; flex-direction: column; justify-content: space-between; align-items: center; 
                height: 24cm;
                background: linear-gradient(135deg, #001a33 0%, #003366 50%, #004d99 100%);
                color: white; text-align: center; page-break-after: always; 
                counter-increment: none;
                position: relative;
                overflow: visible;
                padding: 2cm 0;
            }}
            /* Geometric decorative elements */
            .cover-page::before {{
                content: '';
                position: absolute;
                width: 450px;
                height: 450px;
                background: radial-gradient(circle, rgba(255, 105, 0, 0.15) 0%, rgba(0, 188, 212, 0.1) 100%);
                border-radius: 50%;
                top: -200px;
                left: -200px;
                z-index: 0;
            }}
            .cover-page::after {{
                content: '';
                position: absolute;
                width: 350px;
                height: 350px;
                background: radial-gradient(circle, rgba(156, 39, 176, 0.12) 0%, rgba(76, 175, 80, 0.08) 100%);
                border-radius: 50%;
                bottom: -150px;
                right: -150px;
                z-index: 0;
            }}
            .cover-box-1 {{ text-align: center; z-index: 1; position: relative; }}
            .cover-box-2 {{ 
                text-align: center; 
                margin-top: 2em; 
                max-width: 85%; 
                margin-left: auto; 
                margin-right: auto; 
                z-index: 1; 
                position: relative;
                padding: 2.5em;
                background: linear-gradient(135deg, rgba(255, 255, 255, 0.08) 0%, rgba(255, 105, 0, 0.05) 100%);
                border-radius: 20px;
                border: 2px solid rgba(255, 105, 0, 0.4);
                /* REMOVED: box-shadow (unsupported by WeasyPrint) */
            }}
            .cover-page h1 {{ 
                font-size: 3.2em; 
                color: white; 
                margin: 0; 
                text-align: center; 
                counter-increment: none;
                /* REMOVED: text-shadow (unsupported by WeasyPrint) */
                letter-spacing: 2px;
            }}
            .cover-page h2 {{ 
                font-size: 2.8em; 
                color: #FF9944; 
                font-weight: 400; 
                margin: 0.5em 0; 
                text-align: center; 
                counter-increment: none;
                /* REMOVED: text-shadow (unsupported by WeasyPrint) */
            }}
            .cover-page p {{ 
                font-size: 1.3em; 
                font-weight: 300; 
                margin-top: 1.5em; 
                text-align: center; 
                color: #e8e8e8;
                letter-spacing: 1px;
            }}
            
            .page-break {{ page-break-after: always; }}
            .exec-summary {{ page-break-after: always; position: relative; }}
            .exec-summary p {{ font-size: 1.1em; }}
            /* Add decorative triangle to executive summary */
            .exec-summary::before {{
                content: '';
                position: absolute;
                top: 0;
                right: 0;
                width: 0;
                height: 0;
                border-style: solid;
                border-width: 0 80px 80px 0;
                border-color: transparent #00BCD4 transparent transparent;
                opacity: 0.15;
            }}
            
            .toc-page {{ page-break-after: always; position: relative; }}
            .toc-table a {{ text-decoration: none; color: #003366; font-weight: bold; }}
            .toc-table .page-ref {{ float: right; color: #555; }}
            .toc-table .page-ref::before {{ content: target-counter(attr(href), page-counter); }}
            /* Add colorful accent to TOC */
            .toc-page::after {{
                content: '';
                position: absolute;
                bottom: 20px;
                left: 0;
                width: 6px;
                height: 200px;
                background: linear-gradient(180deg, #FF6900 0%, #00BCD4 50%, #9C27B0 100%);
                border-radius: 3px;
            }}

            /* Enhanced domain summary pages */
            .domain-summary-page {{
                page-break-before: always; page-break-after: always;
                background: linear-gradient(135deg, #fafafa 0%, #f5f5f5 100%); 
                border: 2px solid #e0e0e0;
                padding: 2cm; border-radius: 12px;
                position: relative;
                overflow: hidden;
            }}
            /* Colorful corner decorations for domain summaries */
            .domain-summary-page::before {{
                content: '';
                position: absolute;
                top: -30px;
                right: -30px;
                width: 120px;
                height: 120px;
                background: radial-gradient(circle, rgba(0, 188, 212, 0.2) 0%, transparent 70%);
                border-radius: 50%;
            }}
            .domain-summary-page::after {{
                content: '';
                position: absolute;
                bottom: -40px;
                left: -40px;
                width: 150px;
                height: 150px;
                background: radial-gradient(circle, rgba(156, 39, 176, 0.15) 0%, transparent 70%);
                border-radius: 50%;
            }}
            .domain-summary-page h2 {{
                border-bottom: 4px solid transparent;
                border-image: linear-gradient(90deg, #FF6900 0%, #00BCD4 50%, #9C27B0 100%) 1;
                padding-bottom: 12px;
                position: relative;
                z-index: 1;
            }}
            .domain-summary-page p {{
                font-size: 12pt; line-height: 1.8;
                position: relative;
                z-index: 1;
            }}
            .domain-header {{ 
                page-break-before: avoid;
                border-bottom: 3px solid #003366; padding-bottom: 10px;
                background: linear-gradient(90deg, rgba(0, 51, 102, 0.05) 0%, transparent 100%);
                padding-left: 10px;
            }}
            .domain-count {{ font-size: 1.2em; color: #555; font-weight: 400; text-align: {align}; margin-top: -0.5em; margin-bottom: 1.5em; }}
            /* Enhanced use case blocks */
            .use-case-block {{
                page-break-inside: avoid; margin-bottom: 1.2em;
                background: linear-gradient(135deg, #ffffff 0%, #fefefe 100%);
                border: 1px solid #e8e8e8; padding: 14px; border-radius: 8px;
                {'border-right: 4px solid transparent;' if is_rtl else 'border-left: 4px solid transparent;'}
                {'border-image: linear-gradient(180deg, #FF6900 0%, #00BCD4 100%) 1;' if is_rtl else 'border-image: linear-gradient(180deg, #FF6900 0%, #00BCD4 100%) 1;'}
                /* REMOVED: box-shadow (unsupported by WeasyPrint) */
                position: relative;
            }}
            .use-case-block.page-break-after {{
                page-break-after: always;
            }}
            /* Add small decorative element to use case blocks */
            .use-case-block::before {{
                content: '';
                position: absolute;
                top: 8px;
                {'left: 8px;' if is_rtl else 'right: 8px;'}
                width: 6px;
                height: 6px;
                background: #00BCD4;
                border-radius: 50%;
                opacity: 0.6;
            }}
            .use-case-block h3 {{ margin-bottom: 0.5em; font-size: 13pt; }}
            .use-case-block h3 a {{ color: #0B579E; text-decoration: underline; }}
            .use-case-block p {{ margin-bottom: 0.5em; font-size: 11pt; line-height: 1.4; }}
            .disclaimer {{ font-size: 0.9em; color: #555; border-top: 1px solid #ddd; padding-top: 1em; margin-top: 2em; }}
            """
            html_parts = [f"<html><head><meta charset='UTF-8'><style>{css}</style></head><body>"]
            
            # Req 1: Use transliterated_name
            h1_text = e(t["pdf_title"]); h2_text = e(business_name); p_text = now
            # For Arabic, we want to keep the text centered regardless of RTL
            h2_style = ''  # Always center, no dir attribute needed
            p_style = 'dir="ltr"' if is_rtl else ''
            
            # Modified cover page structure: single centered box with title, date, and business name
            html_parts.append('<div class="cover-page">')
            html_parts.append(f'<div class="cover-box-1">')
            html_parts.append(f'<h1>{h1_text}</h1>')
            html_parts.append(f'<p {p_style}>{p_text}</p>')
            # Business name now in the same box, centered after title and date
            html_parts.append(f'<h2 {h2_style} style="margin-top: 2em;">{h2_text}</h2>')
            html_parts.append('</div>')
            html_parts.append('</div>')
            
            summary_text = summary_dict.get('Executive', f'<p>{e(t["executive_summary_not_available"])}</p>')
            # Req 6: Use translated disclaimer text directly
            disclaimer_text = t["disclaimer"]
            html_parts.append(f'<div class="exec-summary"><h1 class="page-title">{e(t["pdf_exec_summary"])}</h1>{summary_text}<div class="disclaimer"><strong>{e(t["pdf_disclaimer_title"])}:</strong> {e(disclaimer_text)}</div></div>')
            
            # Req 3 & 5: TOC
            html_parts.append(f'<div class="toc-page">')
            html_parts.append(f'<h1 class="page-title">{e(t["pdf_toc_title"])}</h1>')
            html_parts.append(f"<table class='toc-table'><tr><th>{e(t['domain'])}</th><th>{e(t['total'])}</th></tr>")
            toc_rows = []; domain_id_map = {}
            for i, (domain, domain_use_cases) in enumerate(grouped_data.items()):
                domain_slug = f"domain-{i}"
                domain_id_map[domain] = domain_slug
                toc_rows.append(f"<tr><td><a href='#{domain_slug}'>{e(domain)}</a></td><td>{len(domain_use_cases)}</td></tr>")
            html_parts.extend(toc_rows)
            html_parts.append("</table>")
            html_parts.append('</div>')
            
            for domain, domain_use_cases in grouped_data.items():
                domain_slug = domain_id_map[domain]
                uc_highlights = []
                for uc in domain_use_cases[:5]:
                    uc_name = e(uc.get('Name', '').strip())
                    uc_value = e(uc.get('Business Value', '').strip())
                    if uc_name and uc_value:
                        uc_highlights.append(f'<li><strong>{uc_name}</strong>: {uc_value}</li>')
                    elif uc_name:
                        uc_highlights.append(f'<li><strong>{uc_name}</strong></li>')
                remaining = len(domain_use_cases) - 5
                highlights_html = '<ul>' + ''.join(uc_highlights) + '</ul>'
                if remaining > 0:
                    highlights_html += f'<p><em>...and {remaining} more use cases.</em></p>'
                
                html_parts.append(f'<div class="domain-summary-page">')
                html_parts.append(f'<h2 class="page-title" id="{domain_slug}">{e(domain)}</h2>')
                html_parts.append(highlights_html)
                html_parts.append(f'</div>')
                
                html_parts.append(f'<h2 class="domain-header">{e(domain)} - {e(t["pdf_detailed_view"])}</h2>')
                html_parts.append(f'<p class="domain-count">{len(domain_use_cases)}&nbsp;{e(t["pptx_domain_suffix"])}</p>')
                
                # Sort use cases: successful SQL results first, then by Priority descending
                def get_use_case_sort_key(uc):
                    use_case_id = uc.get('No', 'Unknown')
                    example_result = self._get_cached_sql_result(use_case_id)
                    has_success = 1 if example_result.get('status') == 'success' else 0
                    priority_score = float(uc.get('Priority', 0)) if isinstance(uc.get('Priority'), (int, float, str)) else 0
                    try:
                        priority_score = float(priority_score)
                    except (ValueError, TypeError):
                        priority_score = 0
                    return (-has_success, -priority_score)  # Negative for descending order
                
                sorted_domain_use_cases = sorted(domain_use_cases, key=get_use_case_sort_key)
                
                # Helper function to translate field values
                def translate_pdf_value(value):
                    """Translate Type and Priority values for PDF"""
                    if not value or value == 'N/A':
                        return value
                    
                    value_key_map = {
                        'Problem': 'value_type_problem', 'Risk': 'value_type_risk',
                        'Opportunity': 'value_type_opportunity', 'Improvement': 'value_type_improvement',
                        'Ultra High': 'value_priority_ultra_high', 'Very High': 'value_priority_very_high',
                        'High': 'value_priority_high', 'Medium': 'value_priority_medium',
                        'Low': 'value_priority_low', 'Very Low': 'value_priority_very_low',
                        'Ultra Low': 'value_priority_ultra_low'
                    }
                    translation_key = value_key_map.get(value)
                    return t.get(translation_key, value) if translation_key else value
                
                def translate_strategic_value(value):
                    """Translate Strategic Goals and Business Priority alignment values"""
                    if not value or value == 'N/A':
                        return value
                    
                    strategic_key_map = {
                        'General Improvement': 'value_general_improvement',
                        'Reduce Cost': 'value_reduce_cost',
                        'Increase Revenue': 'value_increase_revenue',
                        'Boost Productivity': 'value_boost_productivity',
                        'Mitigate Risk': 'value_mitigate_risk',
                        'Protect Revenue': 'value_protect_revenue',
                        'Align to Regulations': 'value_align_to_regulations',
                        'Improve Customer Experience': 'value_improve_customer_experience',
                        'Enable Data-Driven Decisions': 'value_enable_data_driven_decisions',
                        'Optimize Operations': 'value_optimize_operations',
                        'Empower Talent': 'value_empower_talent',
                        'Enhance Experience': 'value_enhance_experience',
                        'Drive Innovation': 'value_drive_innovation',
                        'Achieve ESG': 'value_achieve_esg',
                        'Execute Strategy': 'value_execute_strategy',
                    }
                    
                    # Handle comma-separated values
                    if ',' in value:
                        parts = [p.strip() for p in value.split(',')]
                        translated_parts = []
                        for part in parts:
                            key = strategic_key_map.get(part)
                            translated_parts.append(t.get(key, part) if key else part)
                        return ', '.join(translated_parts)
                    
                    translation_key = strategic_key_map.get(value)
                    return t.get(translation_key, value) if translation_key else value
                
                def translate_analytics_technique(value):
                    """Translate Analytics Technique values with inline fallback translations"""
                    if not value or value == 'N/A':
                        return value
                    
                    analytics_key_map = {
                        'Forecasting': 'value_forecasting',
                        'Classification': 'value_classification',
                        'Anomaly Detection': 'value_anomaly_detection',
                        'Cohort Analysis': 'value_cohort_analysis',
                        'Segmentation': 'value_segmentation',
                        'Sentiment Analysis': 'value_sentiment_analysis',
                        'Trend Analysis': 'value_trend_analysis',
                        'Prescriptive Analytics': 'value_prescriptive_analytics',
                        'Root Cause Analysis': 'value_root_cause_analysis',
                        'Optimization': 'value_optimization',
                        'Recommendation': 'value_recommendation',
                        'Time Series Analysis': 'value_time_series_analysis',
                        'Predictive Analytics': 'value_predictive_analytics',
                        'Descriptive Analytics': 'value_descriptive_analytics',
                        'Predictive Modeling': 'value_predictive_modeling',
                        'Clustering': 'value_clustering',
                        'Regression Analysis': 'value_regression_analysis',
                        'Survival Analysis': 'value_survival_analysis',
                        'Network Analysis': 'value_network_analysis',
                    }
                    
                    analytics_fallbacks = {
                        'Chinese (Mandarin)': {'Forecasting': '预测', 'Classification': '分类', 'Anomaly Detection': '异常检测', 'Cohort Analysis': '队列分析', 'Segmentation': '细分', 'Sentiment Analysis': '情感分析', 'Trend Analysis': '趋势分析', 'Prescriptive Analytics': '规范性分析', 'Root Cause Analysis': '根因分析', 'Optimization': '优化', 'Recommendation': '推荐', 'Time Series Analysis': '时间序列分析', 'Predictive Analytics': '预测分析', 'Descriptive Analytics': '描述性分析', 'Predictive Modeling': '预测建模', 'Clustering': '聚类分析', 'Regression Analysis': '回归分析', 'Survival Analysis': '生存分析', 'Network Analysis': '网络分析'},
                        'Arabic': {'Forecasting': 'التنبؤ', 'Classification': 'التصنيف', 'Anomaly Detection': 'كشف الشذوذ', 'Cohort Analysis': 'تحليل الأتراب', 'Segmentation': 'التجزئة', 'Sentiment Analysis': 'تحليل المشاعر', 'Trend Analysis': 'تحليل الاتجاهات', 'Prescriptive Analytics': 'التحليلات الوصفية', 'Root Cause Analysis': 'تحليل السبب الجذري', 'Optimization': 'التحسين', 'Recommendation': 'التوصية', 'Time Series Analysis': 'تحليل السلاسل الزمنية', 'Predictive Analytics': 'التحليلات التنبؤية', 'Descriptive Analytics': 'التحليلات الوصفية', 'Predictive Modeling': 'النمذجة التنبؤية', 'Clustering': 'التجميع', 'Regression Analysis': 'تحليل الانحدار', 'Survival Analysis': 'تحليل البقاء', 'Network Analysis': 'تحليل الشبكات'},
                        'Spanish': {'Forecasting': 'Pronóstico', 'Classification': 'Clasificación', 'Anomaly Detection': 'Detección de Anomalías', 'Cohort Analysis': 'Análisis de Cohortes', 'Segmentation': 'Segmentación', 'Sentiment Analysis': 'Análisis de Sentimiento', 'Trend Analysis': 'Análisis de Tendencias', 'Prescriptive Analytics': 'Analítica Prescriptiva', 'Root Cause Analysis': 'Análisis de Causa Raíz', 'Optimization': 'Optimización', 'Recommendation': 'Recomendación', 'Time Series Analysis': 'Análisis de Series Temporales', 'Predictive Analytics': 'Analítica Predictiva', 'Descriptive Analytics': 'Analítica Descriptiva', 'Predictive Modeling': 'Modelado Predictivo', 'Clustering': 'Agrupamiento', 'Regression Analysis': 'Análisis de Regresión', 'Survival Analysis': 'Análisis de Supervivencia', 'Network Analysis': 'Análisis de Redes'},
                        'French': {'Forecasting': 'Prévision', 'Classification': 'Classification', 'Anomaly Detection': 'Détection d\'Anomalies', 'Cohort Analysis': 'Analyse de Cohortes', 'Segmentation': 'Segmentation', 'Sentiment Analysis': 'Analyse de Sentiments', 'Trend Analysis': 'Analyse des Tendances', 'Prescriptive Analytics': 'Analytique Prescriptive', 'Root Cause Analysis': 'Analyse des Causes Profondes', 'Optimization': 'Optimisation', 'Recommendation': 'Recommandation', 'Time Series Analysis': 'Analyse de Séries Temporelles', 'Predictive Analytics': 'Analytique Prédictive', 'Descriptive Analytics': 'Analytique Descriptive', 'Predictive Modeling': 'Modélisation Prédictive', 'Clustering': 'Regroupement', 'Regression Analysis': 'Analyse de Régression', 'Survival Analysis': 'Analyse de Survie', 'Network Analysis': 'Analyse de Réseaux'},
                        'German': {'Forecasting': 'Vorhersage', 'Classification': 'Klassifikation', 'Anomaly Detection': 'Anomalieerkennung', 'Cohort Analysis': 'Kohortenanalyse', 'Segmentation': 'Segmentierung', 'Sentiment Analysis': 'Stimmungsanalyse', 'Trend Analysis': 'Trendanalyse', 'Prescriptive Analytics': 'Präskriptive Analytik', 'Root Cause Analysis': 'Ursachenanalyse', 'Optimization': 'Optimierung', 'Recommendation': 'Empfehlung', 'Time Series Analysis': 'Zeitreihenanalyse', 'Predictive Analytics': 'Prädiktive Analytik', 'Descriptive Analytics': 'Deskriptive Analytik', 'Predictive Modeling': 'Prädiktive Modellierung', 'Clustering': 'Clusteranalyse', 'Regression Analysis': 'Regressionsanalyse', 'Survival Analysis': 'Überlebensanalyse', 'Network Analysis': 'Netzwerkanalyse'},
                        'Portuguese': {'Forecasting': 'Previsão', 'Classification': 'Classificação', 'Anomaly Detection': 'Detecção de Anomalias', 'Cohort Analysis': 'Análise de Coorte', 'Segmentation': 'Segmentação', 'Sentiment Analysis': 'Análise de Sentimento', 'Trend Analysis': 'Análise de Tendências', 'Prescriptive Analytics': 'Análise Prescritiva', 'Root Cause Analysis': 'Análise de Causa Raiz', 'Optimization': 'Otimização', 'Recommendation': 'Recomendação', 'Time Series Analysis': 'Análise de Séries Temporais', 'Predictive Analytics': 'Análise Preditiva', 'Descriptive Analytics': 'Análise Descritiva', 'Predictive Modeling': 'Modelagem Preditiva', 'Clustering': 'Agrupamento', 'Regression Analysis': 'Análise de Regressão', 'Survival Analysis': 'Análise de Sobrevivência', 'Network Analysis': 'Análise de Redes'},
                        'Italian': {'Forecasting': 'Previsione', 'Classification': 'Classificazione', 'Anomaly Detection': 'Rilevamento Anomalie', 'Cohort Analysis': 'Analisi di Coorte', 'Segmentation': 'Segmentazione', 'Sentiment Analysis': 'Analisi del Sentimento', 'Trend Analysis': 'Analisi delle Tendenze', 'Prescriptive Analytics': 'Analisi Prescrittiva', 'Root Cause Analysis': 'Analisi delle Cause Profonde', 'Optimization': 'Ottimizzazione', 'Recommendation': 'Raccomandazione', 'Time Series Analysis': 'Analisi delle Serie Temporali', 'Predictive Analytics': 'Analisi Predittiva', 'Descriptive Analytics': 'Analisi Descrittiva', 'Predictive Modeling': 'Modellazione Predittiva', 'Clustering': 'Raggruppamento', 'Regression Analysis': 'Analisi di Regressione', 'Survival Analysis': 'Analisi di Sopravvivenza', 'Network Analysis': 'Analisi di Rete'},
                        'Japanese': {'Forecasting': '予測', 'Classification': '分類', 'Anomaly Detection': '異常検出', 'Cohort Analysis': 'コホート分析', 'Segmentation': 'セグメンテーション', 'Sentiment Analysis': '感情分析', 'Trend Analysis': 'トレンド分析', 'Prescriptive Analytics': '処方的分析', 'Root Cause Analysis': '根本原因分析', 'Optimization': '最適化', 'Recommendation': 'レコメンデーション', 'Time Series Analysis': '時系列分析', 'Predictive Analytics': '予測分析', 'Descriptive Analytics': '記述分析', 'Predictive Modeling': '予測モデリング', 'Clustering': 'クラスタリング', 'Regression Analysis': '回帰分析', 'Survival Analysis': '生存分析', 'Network Analysis': 'ネットワーク分析'},
                        'Korean': {'Forecasting': '예측', 'Classification': '분류', 'Anomaly Detection': '이상 탐지', 'Cohort Analysis': '코호트 분석', 'Segmentation': '세분화', 'Sentiment Analysis': '감정 분석', 'Trend Analysis': '추세 분석', 'Prescriptive Analytics': '처방적 분석', 'Root Cause Analysis': '근본 원인 분석', 'Optimization': '최적화', 'Recommendation': '추천', 'Time Series Analysis': '시계열 분석', 'Predictive Analytics': '예측 분석', 'Descriptive Analytics': '기술 분석', 'Predictive Modeling': '예측 모델링', 'Clustering': '클러스터링', 'Regression Analysis': '회귀 분석', 'Survival Analysis': '생존 분석', 'Network Analysis': '네트워크 분석'},
                        'Hindi': {'Forecasting': 'पूर्वानुमान', 'Classification': 'वर्गीकरण', 'Anomaly Detection': 'विसंगति पता लगाना', 'Cohort Analysis': 'समूह विश्लेषण', 'Segmentation': 'विभाजन', 'Sentiment Analysis': 'भावना विश्लेषण', 'Trend Analysis': 'रुझान विश्लेषण', 'Prescriptive Analytics': 'निर्देशात्मक विश्लेषण', 'Root Cause Analysis': 'मूल कारण विश्लेषण', 'Optimization': 'अनुकूलन', 'Recommendation': 'सिफारिश', 'Time Series Analysis': 'समय श्रृंखला विश्लेषण', 'Predictive Analytics': 'भविष्य कथन विश्लेषण', 'Descriptive Analytics': 'वर्णनात्मक विश्लेषण', 'Predictive Modeling': 'भविष्य कथन मॉडलिंग', 'Clustering': 'क्लस्टरिंग', 'Regression Analysis': 'प्रतिगमन विश्लेषण', 'Survival Analysis': 'अस्तित्व विश्लेषण', 'Network Analysis': 'नेटवर्क विश्लेषण'},
                        'Russian': {'Forecasting': 'Прогнозирование', 'Classification': 'Классификация', 'Anomaly Detection': 'Обнаружение Аномалий', 'Cohort Analysis': 'Когортный Анализ', 'Segmentation': 'Сегментация', 'Sentiment Analysis': 'Анализ Настроений', 'Trend Analysis': 'Анализ Трендов', 'Prescriptive Analytics': 'Предписывающая Аналитика', 'Root Cause Analysis': 'Анализ Первопричин', 'Optimization': 'Оптимизация', 'Recommendation': 'Рекомендация', 'Time Series Analysis': 'Анализ Временных Рядов', 'Predictive Analytics': 'Предиктивная Аналитика', 'Descriptive Analytics': 'Описательная Аналитика', 'Predictive Modeling': 'Предиктивное Моделирование', 'Clustering': 'Кластеризация', 'Regression Analysis': 'Регрессионный Анализ', 'Survival Analysis': 'Анализ Выживаемости', 'Network Analysis': 'Сетевой Анализ'},
                    }
                    
                    translation_key = analytics_key_map.get(value)
                    translated = t.get(translation_key, None) if translation_key else None
                    if translated and translated != value:
                        return translated
                    if language in analytics_fallbacks and value in analytics_fallbacks[language]:
                        return analytics_fallbacks[language][value]
                    return value
                
                for idx, uc in enumerate(sorted_domain_use_cases, start=1):
                    page_break_class = ' page-break-after' if idx % 2 == 0 else ''
                    html_parts.append(f'<div class="use-case-block{page_break_class}">')
                    uc_title = uc.get('Name', 'N/A') if language != "English" else self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', ''))
                    _nb_url = self._get_use_case_notebook_url(uc)
                    if _nb_url:
                        html_parts.append(f'<h3><a href="{e(_nb_url)}">{e(uc["No"])}</a>: {e(uc_title)}</h3>')
                    else:
                        html_parts.append(f"<h3>{e(uc['No'])}: {e(uc_title)}</h3>")
                    html_parts.append(f"<p><strong>{e(t.get('description_label', 'Description'))}:</strong> {e(uc.get('Name', 'N/A'))}</p>")
                    # Add header line with Subdomain, Type, Analytics Technique, and Priority (with translations)
                    subdomain_val = e(uc.get('Subdomain', 'N/A'))
                    type_val = e(translate_pdf_value(uc.get('type', 'N/A')))
                    analytics_technique_val = e(translate_analytics_technique(uc.get('Analytics Technique', 'N/A')))
                    quality_val = e(translate_pdf_value(uc.get('Quality', 'N/A')))
                    html_parts.append(f"<p style='font-weight: bold; color: #0066cc;'>{e(t['subdomain'])}: {subdomain_val} | {e(t['type'])}: {type_val} | {e(t.get('analytics_technique', 'Analytics Technique'))}: {analytics_technique_val} | {e(t['quality'])}: {quality_val}</p>")
                    html_parts.append(f"<p><strong>{e(t['statement'])}:</strong> {e(uc.get('Statement', 'N/A'))}</p>")
                    html_parts.append(f"<p><strong>{e(t['solution'])}:</strong> {e(uc.get('Solution', 'N/A'))}</p>")
                    html_parts.append(f"<p><strong>{e(t['business_value'])}:</strong> {e(uc.get('Business Value', 'N/A'))}</p>")
                    html_parts.append(f"<p><strong>{e(t['beneficiary'])}:</strong> {e(uc.get('Beneficiary', 'N/A'))}</p>")
                    html_parts.append(f"<p><strong>{e(t['sponsor'])}:</strong> {e(uc.get('Sponsor', 'N/A'))}</p>")
                    html_parts.append(f"<p><strong>{e(t.get('business_priority_alignment', 'Business Priority Alignment'))}:</strong> {e(translate_strategic_value(uc.get('Business Priority Alignment', 'General Improvement')))}</p>")
                    html_parts.append(f"<p><strong>{e(t.get('strategic_goals_alignment', 'Strategic Goals Alignment'))}:</strong> {e(translate_strategic_value(uc.get('Strategic Goals Alignment', 'General Improvement')))}</p>")
                    
                    html_parts.append('</div>')
            html_parts.append("</body></html>")
            return "".join(html_parts)

        def _save_pdf(html_content: str, workspace_path: str, logger_instance):
            import weasyprint
            import logging
            try: from weasyprint.fonts import FontConfiguration
            except ImportError: FontConfiguration = None 
            local_pdf_path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file: local_pdf_path = tmp_file.name
                logger_instance.info(f"Generating PDF at local temp path: {local_pdf_path}")
                font_config = FontConfiguration() if FontConfiguration else None
                
                # Suppress font-face warnings from WeasyPrint
                weasyprint_logger = logging.getLogger('weasyprint')
                original_level = weasyprint_logger.level
                weasyprint_logger.setLevel(logging.ERROR)  # Only show errors, suppress warnings
                
                try:
                    weasyprint.HTML(string=html_content).write_pdf(local_pdf_path, font_config=font_config)
                finally:
                    weasyprint_logger.setLevel(original_level)  # Restore original level
                with open(local_pdf_path, "rb") as f: pdf_data = f.read()
                if not pdf_data: raise ValueError("Generated PDF file is empty.")
                logger_instance.info(f"Uploading PDF to workspace path: {workspace_path}")
                pdf_data_b64 = base64.b64encode(pdf_data).decode()
                self.w_client.workspace.import_(path=workspace_path, content=pdf_data_b64, format=workspace.ImportFormat.AUTO, overwrite=True)
                abs_path = self.w_client.workspace.get_status(workspace_path).path
                logger_instance.info(f"Success! PDF Catalog uploaded to: {abs_path}")
                log_print(f"Success! PDF Catalog ({language}) generated: {abs_path}")
            except Exception as e:
                logger_instance.critical(f"Failed to generate and save PDF for {language}: {get_clean_error_message(e)}")
            finally:
                if local_pdf_path and os.path.exists(local_pdf_path): os.remove(local_pdf_path)

        # --- Main execution logic for generate_catalog_pdf ---
        try:
            if not _install_dependencies(self.logger):
                self.logger.error("Skipping PDF generation due to missing weasyprint dependency.")
                return
                
            if not grouped_data:
                self.logger.warning(f"No use cases provided to generate_catalog_pdf for {language}. Skipping.")
                return
            final_html = _build_html(grouped_data, summary_dict, transliterated_name, t, is_rtl)
            pdf_workspace_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire_{lang_abbr}.pdf")
            _save_pdf(final_html, pdf_workspace_path, self.logger)
        except Exception as e:
            self.logger.critical(f"An error occurred during PDF generation for {language}: {get_clean_error_message(e)}")

    # === MODIFIED: PPTX Generation (Req 1, 2, 4, 5, 6) ===
    def generate_presentation_pptx(self, language: str, lang_abbr: str, translations: dict, summary_dict: dict, grouped_data: dict, transliterated_name: str):
        self.logger.info(f"--- Starting PPTX Presentation Generation for {language} ---")
        
        t = translations
        is_rtl = (language == "Arabic")

        def _install_pptx_dependencies(logger_instance) -> bool:
            try:
                import pptx
                self.logger.info("PPTX package (python-pptx) already installed.")
                return True
            except ImportError:
                logger_instance.info("Installing required PPTX package (python-pptx)...")
                try: 
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx"])
                    import pptx
                    self.logger.info("Successfully installed python-pptx.")
                    return True
                except Exception as e: 
                    logger_instance.error(f"Failed to install python-pptx: {get_clean_error_message(e)}")
                    print("ERROR: Failed to install 'python-pptx'. Presentation generation cannot continue.", file=sys.stderr)
                    return False

        # === MODIFIED: _build_presentation (Req 1, 3, 4, 5) ===
        def _build_presentation(grouped_data: dict, summary_dict: dict, business_name: str, translations: dict, workspace_path: str, logger_instance, is_rtl: bool):
            try:
                from pptx import Presentation
                from pptx.util import Inches, Pt, Cm
                from pptx.dml.color import RGBColor
                from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
                from pptx.enum.shapes import MSO_SHAPE
            except ImportError as e:
                logger_instance.error(f"FATAL: python-pptx import failed inside _build_presentation: {get_clean_error_message(e)}. Aborting PPTX generation.")
                return
            
            DATABRICKS_BLUE = RGBColor(0, 51, 102); DATABRICKS_ORANGE = RGBColor(255, 105, 0); TEXT_COLOR = RGBColor(0x33, 0x33, 0x33)
            LIGHT_GRAY = RGBColor(0xFA, 0xFA, 0xFA); WHITE_COLOR = RGBColor(0xFF, 0xFF, 0xFF); FOOTER_COLOR = RGBColor(0x88, 0x88, 0x88)
            # Modern color palette - vibrant and futuristic
            TEAL_ACCENT = RGBColor(0, 188, 212); PURPLE_ACCENT = RGBColor(156, 39, 176); GREEN_ACCENT = RGBColor(76, 175, 80)
            
            prs = Presentation(); prs.slide_width = Cm(33.867); prs.slide_height = Cm(19.05)
            # === PPTX FIX (Req 1): Cast all float calculations to int() ===
            CONTENT_WIDTH_CM = Cm(30.5)
            LEFT_MARGIN_CM = (prs.slide_width - CONTENT_WIDTH_CM) / 2
            
            align = PP_ALIGN.RIGHT if is_rtl else PP_ALIGN.LEFT
            title_align = PP_ALIGN.CENTER

            SLIDE_LAYOUT_TITLE = prs.slide_layouts[0]; SLIDE_LAYOUT_TITLE_AND_CONTENT = prs.slide_layouts[1]; SLIDE_LAYOUT_BLANK = prs.slide_layouts[6]
            def set_font_color(run, color=TEXT_COLOR): run.font.color.rgb = color

            now = datetime.datetime.now().strftime("%Y-%m-%d")
            footer_text = f"Databricks Inspire AI  |  {now}"
            def add_footer(slide):
                try:
                    left = int(LEFT_MARGIN_CM); width = int(CONTENT_WIDTH_CM); top = int(Cm(18.2)); height = int(Cm(0.8))
                    txBox = slide.shapes.add_textbox(left, top, width, height)
                    p = txBox.text_frame.paragraphs[0]; p.text = footer_text
                    p.font.size = Pt(10); p.font.color.rgb = FOOTER_COLOR
                    p.alignment = PP_ALIGN.CENTER if is_rtl else PP_ALIGN.RIGHT
                except Exception as e:
                    logger_instance.warning(f"Failed to add footer to slide: {get_clean_error_message(e)}")

            # Enhanced title slide with decorative shapes
            logger_instance.info("Building Title Slide...") 
            slide = prs.slides.add_slide(SLIDE_LAYOUT_TITLE); slide.background.fill.solid(); slide.background.fill.fore_color.rgb = DATABRICKS_BLUE
            
            # Add decorative circular shapes for visual interest
            circle1 = slide.shapes.add_shape(MSO_SHAPE.OVAL, int(Cm(28)), int(Cm(1)), int(Cm(4)), int(Cm(4)))
            circle1.fill.solid(); circle1.fill.fore_color.rgb = TEAL_ACCENT; circle1.line.fill.background()
            circle1.fill.transparency = 0.3
            
            circle2 = slide.shapes.add_shape(MSO_SHAPE.OVAL, int(Cm(1)), int(Cm(14)), int(Cm(5)), int(Cm(5)))
            circle2.fill.solid(); circle2.fill.fore_color.rgb = PURPLE_ACCENT; circle2.line.fill.background()
            circle2.fill.transparency = 0.2
            
            # Additional decorative elements for futuristic feel
            circle3 = slide.shapes.add_shape(MSO_SHAPE.OVAL, int(Cm(30)), int(Cm(16)), int(Cm(3)), int(Cm(3)))
            circle3.fill.solid(); circle3.fill.fore_color.rgb = GREEN_ACCENT; circle3.line.fill.background()
            circle3.fill.transparency = 0.4
            
            # Add small accent circles
            accent_circle1 = slide.shapes.add_shape(MSO_SHAPE.OVAL, int(Cm(3)), int(Cm(3)), int(Cm(1.5)), int(Cm(1.5)))
            accent_circle1.fill.solid(); accent_circle1.fill.fore_color.rgb = DATABRICKS_ORANGE; accent_circle1.line.fill.background()
            accent_circle1.fill.transparency = 0.5
            
            accent_circle2 = slide.shapes.add_shape(MSO_SHAPE.OVAL, int(Cm(29)), int(Cm(8)), int(Cm(2)), int(Cm(2)))
            accent_circle2.fill.solid(); accent_circle2.fill.fore_color.rgb = TEAL_ACCENT; accent_circle2.line.fill.background()
            accent_circle2.fill.transparency = 0.6
            
            txBox_top = slide.shapes.add_textbox(int(LEFT_MARGIN_CM), int(Cm(2.0)), int(CONTENT_WIDTH_CM), int(Cm(5.0)))
            tf_top = txBox_top.text_frame; tf_top.vertical_anchor = MSO_ANCHOR.MIDDLE; tf_top.word_wrap = True
            p_top1 = tf_top.paragraphs[0]; p_top1.text = t['pptx_main_title']
            p_top1.font.color.rgb = WHITE_COLOR; p_top1.font.size = Pt(44); p_top1.alignment = title_align
            p_top2 = tf_top.add_paragraph(); p_top2.text = now
            p_top2.font.color.rgb = RGBColor(0xCC, 0xCC, 0xCC); p_top2.font.size = Pt(20); p_top2.alignment = title_align
            
            txBox_bottom = slide.shapes.add_textbox(int(LEFT_MARGIN_CM), int(Cm(10.0)), int(CONTENT_WIDTH_CM), int(Cm(5.0)))
            tf_bottom = txBox_bottom.text_frame; tf_bottom.vertical_anchor = MSO_ANCHOR.MIDDLE
            p_bottom = tf_bottom.paragraphs[0]; p_bottom.text = f"{t['pptx_for']} {business_name}" # Req 1: Use transliterated name
            p_bottom.font.color.rgb = DATABRICKS_ORANGE; p_bottom.font.size = Pt(32); p_bottom.alignment = title_align
            
            try: slide.shapes.title.element.getparent().remove(slide.shapes.title.element)
            except: pass
            try: slide.placeholders[1].element.getparent().remove(slide.placeholders[1].element)
            except: pass
            add_footer(slide)

            # Enhanced Executive Summary with colorful accents
            logger_instance.info("Building Executive Summary Slide...")
            slide = prs.slides.add_slide(SLIDE_LAYOUT_TITLE_AND_CONTENT); slide.background.fill.solid(); slide.background.fill.fore_color.rgb = LIGHT_GRAY
            
            # Add gradient accent bar (simulated with two rectangles)
            accent_bar1 = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, int(Cm(0.8)), int(prs.slide_height))
            accent_bar1.fill.solid(); accent_bar1.fill.fore_color.rgb = DATABRICKS_ORANGE; accent_bar1.line.fill.background()
            
            accent_bar2 = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, int(Cm(0.8)), 0, int(Cm(0.7)), int(prs.slide_height))
            accent_bar2.fill.solid(); accent_bar2.fill.fore_color.rgb = TEAL_ACCENT; accent_bar2.line.fill.background()
            accent_bar2.fill.transparency = 0.5
            
            title = slide.shapes.title; title.left, title.width = int(LEFT_MARGIN_CM), int(CONTENT_WIDTH_CM)
            title.top = int(Cm(1.0)); title.height = int(Cm(2.5))
            title.text = t['pdf_exec_summary']
            p = title.text_frame.paragraphs[0]; p.font.color.rgb = DATABRICKS_BLUE; p.font.size = Pt(36); p.alignment = align
            
            content_placeholder = slide.placeholders[1]; content_placeholder.left, content_placeholder.width = int(LEFT_MARGIN_CM), int(Cm(30.5))
            content_placeholder.top = int(Cm(1.0) + Cm(2.5))
            content_placeholder.height = int(Cm(14.1))
            content_frame = content_placeholder.text_frame; content_frame.clear(); content_frame.word_wrap = True
            
            summary_text = summary_dict.get('Executive', t['summary_not_available'])
            summary_text = re.sub(r'</p>|<p>', ' ', summary_text); summary_text = re.sub(r'<[^>]+>', '', summary_text).strip()
            
            # Split into sentences and create bullet points for each statement
            sentences = re.split(r'(?<=[.!?])\s+', summary_text)
            for sentence in sentences:
                sentence = sentence.strip()
                if not sentence: continue
                p = content_frame.add_paragraph(); p.text = sentence; p.font.size = Pt(18)
                p.level = 0; p.alignment = align; p.space_after = Pt(8)
                set_font_color(p.runs[0], TEXT_COLOR)
            
            p = content_frame.add_paragraph(); p.space_before = Pt(24); p.alignment = align
            disclaimer_text = t["disclaimer"] # Req 6: Use translated disclaimer
            run_label = p.add_run(); run_label.text = f"{t['pptx_disclaimer_title']}: "; run_label.font.bold = True; run_label.font.size = Pt(14)
            set_font_color(run_label, DATABRICKS_BLUE)
            run_value = p.add_run(); run_value.text = disclaimer_text; run_value.font.size = Pt(14); set_font_color(run_value, TEXT_COLOR)
            add_footer(slide)

            # === MODIFIED: PPTX TOC (Req 4, 5) ===
            logger_instance.info("Building Table of Contents Slide(s)...")
            rows = list(grouped_data.items())
            max_rows_per_slide = 10
            num_slides = (len(rows) + max_rows_per_slide - 1) // max_rows_per_slide
            for i in range(num_slides):
                slide = prs.slides.add_slide(SLIDE_LAYOUT_TITLE_AND_CONTENT)
                slide.background.fill.solid(); slide.background.fill.fore_color.rgb = LIGHT_GRAY
                
                # Colorful multi-bar accent
                accent_bar1 = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, int(Cm(0.5)), int(prs.slide_height))
                accent_bar1.fill.solid(); accent_bar1.fill.fore_color.rgb = DATABRICKS_ORANGE; accent_bar1.line.fill.background()
                
                accent_bar2 = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, int(Cm(0.5)), 0, int(Cm(0.5)), int(prs.slide_height))
                accent_bar2.fill.solid(); accent_bar2.fill.fore_color.rgb = TEAL_ACCENT; accent_bar2.line.fill.background()
                
                accent_bar3 = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, int(Cm(1.0)), 0, int(Cm(0.5)), int(prs.slide_height))
                accent_bar3.fill.solid(); accent_bar3.fill.fore_color.rgb = PURPLE_ACCENT; accent_bar3.line.fill.background()
                
                title = slide.shapes.title; title.left, title.width = int(LEFT_MARGIN_CM), int(CONTENT_WIDTH_CM)
                title.top = int(Cm(1.0)); title.height = int(Cm(2.5))
                title.text = t['pdf_toc_title']
                if num_slides > 1: title.text += f" ({i+1}/{num_slides})"
                p = title.text_frame.paragraphs[0]; p.font.color.rgb = DATABRICKS_BLUE; p.font.size = Pt(36); p.alignment = align
                
                chunk = rows[i*max_rows_per_slide : (i+1)*max_rows_per_slide]
                num_table_rows = len(chunk) + 1; num_table_cols = 2
                table_left = int(LEFT_MARGIN_CM); table_top = int(Cm(4.0)); table_width = int(CONTENT_WIDTH_CM); table_height = int(Cm(14.0))
                table_shape = slide.shapes.add_table(num_table_rows, num_table_cols, table_left, table_top, table_width, table_height)
                table = table_shape.table
                table.columns[0].width = int(Cm(22.0)); table.columns[1].width = int(Cm(8.5))
                table.horz_banding = False; table.first_row = True
                
                table.cell(0, 0).text = t['domain']; table.cell(0, 1).text = t['total']
                
                for c_idx in range(num_table_cols):
                    cell = table.cell(0, c_idx); cell.fill.solid(); cell.fill.fore_color.rgb = DATABRICKS_BLUE
                    para = cell.text_frame.paragraphs[0]; para.font.color.rgb = WHITE_COLOR; para.font.bold = True; para.font.size = Pt(18); para.alignment = align
                    cell.vertical_anchor = MSO_ANCHOR.MIDDLE
                for r_idx, (domain, domain_use_cases) in enumerate(chunk):
                    table.cell(r_idx + 1, 0).text = domain
                    table.cell(r_idx + 1, 1).text = str(len(domain_use_cases))
                    for c_idx in range(num_table_cols):
                        cell = table.cell(r_idx + 1, c_idx); para = cell.text_frame.paragraphs[0]
                        para.font.color.rgb = TEXT_COLOR; para.font.size = Pt(16); para.alignment = align
                        cell.vertical_anchor = MSO_ANCHOR.MIDDLE
                try: slide.placeholders[1].element.getparent().remove(slide.placeholders[1].element)
                except: pass
                add_footer(slide)
            
            for domain, domain_use_cases in grouped_data.items():
                slide = prs.slides.add_slide(SLIDE_LAYOUT_BLANK); slide.background.fill.solid(); slide.background.fill.fore_color.rgb = DATABRICKS_BLUE
                txBox = slide.shapes.add_textbox(int(LEFT_MARGIN_CM), int(Cm(1.0)), int(CONTENT_WIDTH_CM), int(Cm(17.05)))
                tf = txBox.text_frame; tf.vertical_anchor = MSO_ANCHOR.MIDDLE
                p = tf.paragraphs[0]; p.text = f"{domain}\n{len(domain_use_cases)}\u00A0{t['pptx_domain_suffix']}"; p.alignment = align
                p.font.color.rgb = DATABRICKS_ORANGE; p.font.size = Pt(44); p.font.bold = True
                if len(tf.paragraphs) > 1:
                    p2 = tf.paragraphs[1]; p2.font.color.rgb = WHITE_COLOR; p2.font.size = Pt(32); p2.font.bold = False; p2.alignment = align
                add_footer(slide)

                # --- Domain Summary Slide (MODIFIED: Req 1, 2) ---
                slide = prs.slides.add_slide(SLIDE_LAYOUT_TITLE_AND_CONTENT); slide.background.fill.solid(); slide.background.fill.fore_color.rgb = LIGHT_GRAY
                accent_bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, int(Cm(1.5)), int(prs.slide_height))
                accent_bar.fill.solid(); accent_bar.fill.fore_color.rgb = DATABRICKS_ORANGE; accent_bar.line.fill.background()
                
                title = slide.shapes.title; title.left, title.width = int(LEFT_MARGIN_CM), int(CONTENT_WIDTH_CM)
                title.top = int(Cm(1.0)); title.height = int(Cm(2.5))
                title.text = domain # Req 1: Remove ": Summary"
                
                p = title.text_frame.paragraphs[0]; p.font.color.rgb = DATABRICKS_BLUE; p.font.size = Pt(36); p.alignment = align
                
                content_placeholder = slide.placeholders[1]; content_placeholder.left, content_placeholder.width = int(LEFT_MARGIN_CM), int(Cm(30.5))
                content_placeholder.top = int(Cm(1.0) + Cm(2.5)); content_placeholder.height = int(Cm(14.1))
                content_frame = content_placeholder.text_frame; content_frame.clear(); content_frame.word_wrap = True
                
                for uc in domain_use_cases[:5]:
                    uc_name = uc.get('Name', '').strip()
                    uc_value = uc.get('Business Value', '').strip()
                    bullet_text = f"{uc_name}: {uc_value}" if uc_name and uc_value else uc_name
                    if not bullet_text: continue
                    p = content_frame.add_paragraph(); p.text = bullet_text; p.font.size = Pt(18)
                    p.level = 0; p.alignment = align; p.space_after = Pt(8)
                    set_font_color(p.runs[0], TEXT_COLOR)
                remaining_pptx = len(domain_use_cases) - 5
                if remaining_pptx > 0:
                    p = content_frame.add_paragraph(); p.text = f"...and {remaining_pptx} more use cases."; p.font.size = Pt(16); p.font.italic = True
                    p.level = 0; p.alignment = align; p.space_after = Pt(8)
                    set_font_color(p.runs[0], TEXT_COLOR)
                add_footer(slide)
                
                # Helper function to translate field values for PPTX
                def translate_pptx_value(value):
                    """Translate Type and Priority values for PPTX"""
                    if not value or value == 'N/A':
                        return value
                    
                    value_key_map = {
                        'Problem': 'value_type_problem', 'Risk': 'value_type_risk',
                        'Opportunity': 'value_type_opportunity', 'Improvement': 'value_type_improvement',
                        'Ultra High': 'value_priority_ultra_high', 'Very High': 'value_priority_very_high',
                        'High': 'value_priority_high', 'Medium': 'value_priority_medium',
                        'Low': 'value_priority_low', 'Very Low': 'value_priority_very_low',
                        'Ultra Low': 'value_priority_ultra_low'
                    }
                    translation_key = value_key_map.get(value)
                    return t.get(translation_key, value) if translation_key else value
                
                def translate_strategic_pptx_value(value):
                    """Translate Strategic Goals and Business Priority alignment values"""
                    if not value or value == 'N/A':
                        return value
                    
                    strategic_key_map = {
                        'General Improvement': 'value_general_improvement',
                        'Reduce Cost': 'value_reduce_cost',
                        'Increase Revenue': 'value_increase_revenue',
                        'Boost Productivity': 'value_boost_productivity',
                        'Mitigate Risk': 'value_mitigate_risk',
                        'Protect Revenue': 'value_protect_revenue',
                        'Align to Regulations': 'value_align_to_regulations',
                        'Improve Customer Experience': 'value_improve_customer_experience',
                        'Enable Data-Driven Decisions': 'value_enable_data_driven_decisions',
                        'Optimize Operations': 'value_optimize_operations',
                        'Empower Talent': 'value_empower_talent',
                        'Enhance Experience': 'value_enhance_experience',
                        'Drive Innovation': 'value_drive_innovation',
                        'Achieve ESG': 'value_achieve_esg',
                        'Execute Strategy': 'value_execute_strategy',
                    }
                    
                    # Handle comma-separated values
                    if ',' in str(value):
                        parts = [p.strip() for p in str(value).split(',')]
                        translated_parts = []
                        for part in parts:
                            key = strategic_key_map.get(part)
                            translated_parts.append(t.get(key, part) if key else part)
                        return ', '.join(translated_parts)
                    
                    translation_key = strategic_key_map.get(value)
                    return t.get(translation_key, value) if translation_key else value
                
                def translate_analytics_pptx_value(value):
                    """Translate Analytics Technique values for PPTX with inline fallback translations"""
                    if not value or value == 'N/A':
                        return value
                    
                    analytics_key_map = {
                        'Forecasting': 'value_forecasting',
                        'Classification': 'value_classification',
                        'Anomaly Detection': 'value_anomaly_detection',
                        'Cohort Analysis': 'value_cohort_analysis',
                        'Segmentation': 'value_segmentation',
                        'Sentiment Analysis': 'value_sentiment_analysis',
                        'Trend Analysis': 'value_trend_analysis',
                        'Prescriptive Analytics': 'value_prescriptive_analytics',
                        'Root Cause Analysis': 'value_root_cause_analysis',
                        'Optimization': 'value_optimization',
                        'Recommendation': 'value_recommendation',
                        'Time Series Analysis': 'value_time_series_analysis',
                        'Predictive Analytics': 'value_predictive_analytics',
                        'Descriptive Analytics': 'value_descriptive_analytics',
                        'Predictive Modeling': 'value_predictive_modeling',
                        'Clustering': 'value_clustering',
                        'Regression Analysis': 'value_regression_analysis',
                        'Survival Analysis': 'value_survival_analysis',
                        'Network Analysis': 'value_network_analysis',
                    }
                    
                    analytics_fallbacks = {
                        'Chinese (Mandarin)': {'Forecasting': '预测', 'Classification': '分类', 'Anomaly Detection': '异常检测', 'Cohort Analysis': '队列分析', 'Segmentation': '细分', 'Sentiment Analysis': '情感分析', 'Trend Analysis': '趋势分析', 'Prescriptive Analytics': '规范性分析', 'Root Cause Analysis': '根因分析', 'Optimization': '优化', 'Recommendation': '推荐', 'Time Series Analysis': '时间序列分析', 'Predictive Analytics': '预测分析', 'Descriptive Analytics': '描述性分析', 'Predictive Modeling': '预测建模', 'Clustering': '聚类分析', 'Regression Analysis': '回归分析', 'Survival Analysis': '生存分析', 'Network Analysis': '网络分析'},
                        'Arabic': {'Forecasting': 'التنبؤ', 'Classification': 'التصنيف', 'Anomaly Detection': 'كشف الشذوذ', 'Cohort Analysis': 'تحليل الأتراب', 'Segmentation': 'التجزئة', 'Sentiment Analysis': 'تحليل المشاعر', 'Trend Analysis': 'تحليل الاتجاهات', 'Prescriptive Analytics': 'التحليلات الوصفية', 'Root Cause Analysis': 'تحليل السبب الجذري', 'Optimization': 'التحسين', 'Recommendation': 'التوصية', 'Time Series Analysis': 'تحليل السلاسل الزمنية', 'Predictive Analytics': 'التحليلات التنبؤية', 'Descriptive Analytics': 'التحليلات الوصفية', 'Predictive Modeling': 'النمذجة التنبؤية', 'Clustering': 'التجميع', 'Regression Analysis': 'تحليل الانحدار', 'Survival Analysis': 'تحليل البقاء', 'Network Analysis': 'تحليل الشبكات'},
                        'Spanish': {'Forecasting': 'Pronóstico', 'Classification': 'Clasificación', 'Anomaly Detection': 'Detección de Anomalías', 'Cohort Analysis': 'Análisis de Cohortes', 'Segmentation': 'Segmentación', 'Sentiment Analysis': 'Análisis de Sentimiento', 'Trend Analysis': 'Análisis de Tendencias', 'Prescriptive Analytics': 'Analítica Prescriptiva', 'Root Cause Analysis': 'Análisis de Causa Raíz', 'Optimization': 'Optimización', 'Recommendation': 'Recomendación', 'Time Series Analysis': 'Análisis de Series Temporales', 'Predictive Analytics': 'Analítica Predictiva', 'Descriptive Analytics': 'Analítica Descriptiva', 'Predictive Modeling': 'Modelado Predictivo', 'Clustering': 'Agrupamiento', 'Regression Analysis': 'Análisis de Regresión', 'Survival Analysis': 'Análisis de Supervivencia', 'Network Analysis': 'Análisis de Redes'},
                        'French': {'Forecasting': 'Prévision', 'Classification': 'Classification', 'Anomaly Detection': 'Détection d\'Anomalies', 'Cohort Analysis': 'Analyse de Cohortes', 'Segmentation': 'Segmentation', 'Sentiment Analysis': 'Analyse de Sentiments', 'Trend Analysis': 'Analyse des Tendances', 'Prescriptive Analytics': 'Analytique Prescriptive', 'Root Cause Analysis': 'Analyse des Causes Profondes', 'Optimization': 'Optimisation', 'Recommendation': 'Recommandation', 'Time Series Analysis': 'Analyse de Séries Temporelles', 'Predictive Analytics': 'Analytique Prédictive', 'Descriptive Analytics': 'Analytique Descriptive', 'Predictive Modeling': 'Modélisation Prédictive', 'Clustering': 'Regroupement', 'Regression Analysis': 'Analyse de Régression', 'Survival Analysis': 'Analyse de Survie', 'Network Analysis': 'Analyse de Réseaux'},
                        'German': {'Forecasting': 'Vorhersage', 'Classification': 'Klassifikation', 'Anomaly Detection': 'Anomalieerkennung', 'Cohort Analysis': 'Kohortenanalyse', 'Segmentation': 'Segmentierung', 'Sentiment Analysis': 'Stimmungsanalyse', 'Trend Analysis': 'Trendanalyse', 'Prescriptive Analytics': 'Präskriptive Analytik', 'Root Cause Analysis': 'Ursachenanalyse', 'Optimization': 'Optimierung', 'Recommendation': 'Empfehlung', 'Time Series Analysis': 'Zeitreihenanalyse', 'Predictive Analytics': 'Prädiktive Analytik', 'Descriptive Analytics': 'Deskriptive Analytik', 'Predictive Modeling': 'Prädiktive Modellierung', 'Clustering': 'Clusteranalyse', 'Regression Analysis': 'Regressionsanalyse', 'Survival Analysis': 'Überlebensanalyse', 'Network Analysis': 'Netzwerkanalyse'},
                        'Portuguese': {'Forecasting': 'Previsão', 'Classification': 'Classificação', 'Anomaly Detection': 'Detecção de Anomalias', 'Cohort Analysis': 'Análise de Coorte', 'Segmentation': 'Segmentação', 'Sentiment Analysis': 'Análise de Sentimento', 'Trend Analysis': 'Análise de Tendências', 'Prescriptive Analytics': 'Análise Prescritiva', 'Root Cause Analysis': 'Análise de Causa Raiz', 'Optimization': 'Otimização', 'Recommendation': 'Recomendação', 'Time Series Analysis': 'Análise de Séries Temporais', 'Predictive Analytics': 'Análise Preditiva', 'Descriptive Analytics': 'Análise Descritiva', 'Predictive Modeling': 'Modelagem Preditiva', 'Clustering': 'Agrupamento', 'Regression Analysis': 'Análise de Regressão', 'Survival Analysis': 'Análise de Sobrevivência', 'Network Analysis': 'Análise de Redes'},
                        'Italian': {'Forecasting': 'Previsione', 'Classification': 'Classificazione', 'Anomaly Detection': 'Rilevamento Anomalie', 'Cohort Analysis': 'Analisi di Coorte', 'Segmentation': 'Segmentazione', 'Sentiment Analysis': 'Analisi del Sentimento', 'Trend Analysis': 'Analisi delle Tendenze', 'Prescriptive Analytics': 'Analisi Prescrittiva', 'Root Cause Analysis': 'Analisi delle Cause Profonde', 'Optimization': 'Ottimizzazione', 'Recommendation': 'Raccomandazione', 'Time Series Analysis': 'Analisi delle Serie Temporali', 'Predictive Analytics': 'Analisi Predittiva', 'Descriptive Analytics': 'Analisi Descrittiva', 'Predictive Modeling': 'Modellazione Predittiva', 'Clustering': 'Raggruppamento', 'Regression Analysis': 'Analisi di Regressione', 'Survival Analysis': 'Analisi di Sopravvivenza', 'Network Analysis': 'Analisi di Rete'},
                        'Japanese': {'Forecasting': '予測', 'Classification': '分類', 'Anomaly Detection': '異常検出', 'Cohort Analysis': 'コホート分析', 'Segmentation': 'セグメンテーション', 'Sentiment Analysis': '感情分析', 'Trend Analysis': 'トレンド分析', 'Prescriptive Analytics': '処方的分析', 'Root Cause Analysis': '根本原因分析', 'Optimization': '最適化', 'Recommendation': 'レコメンデーション', 'Time Series Analysis': '時系列分析', 'Predictive Analytics': '予測分析', 'Descriptive Analytics': '記述分析', 'Predictive Modeling': '予測モデリング', 'Clustering': 'クラスタリング', 'Regression Analysis': '回帰分析', 'Survival Analysis': '生存分析', 'Network Analysis': 'ネットワーク分析'},
                        'Korean': {'Forecasting': '예측', 'Classification': '분류', 'Anomaly Detection': '이상 탐지', 'Cohort Analysis': '코호트 분석', 'Segmentation': '세분화', 'Sentiment Analysis': '감정 분석', 'Trend Analysis': '추세 분석', 'Prescriptive Analytics': '처방적 분석', 'Root Cause Analysis': '근본 원인 분석', 'Optimization': '최적화', 'Recommendation': '추천', 'Time Series Analysis': '시계열 분석', 'Predictive Analytics': '예측 분석', 'Descriptive Analytics': '기술 분석', 'Predictive Modeling': '예측 모델링', 'Clustering': '클러스터링', 'Regression Analysis': '회귀 분석', 'Survival Analysis': '생존 분석', 'Network Analysis': '네트워크 분석'},
                        'Hindi': {'Forecasting': 'पूर्वानुमान', 'Classification': 'वर्गीकरण', 'Anomaly Detection': 'विसंगति पता लगाना', 'Cohort Analysis': 'समूह विश्लेषण', 'Segmentation': 'विभाजन', 'Sentiment Analysis': 'भावना विश्लेषण', 'Trend Analysis': 'रुझान विश्लेषण', 'Prescriptive Analytics': 'निर्देशात्मक विश्लेषण', 'Root Cause Analysis': 'मूल कारण विश्लेषण', 'Optimization': 'अनुकूलन', 'Recommendation': 'सिफारिश', 'Time Series Analysis': 'समय श्रृंखला विश्लेषण', 'Predictive Analytics': 'भविष्य कथन विश्लेषण', 'Descriptive Analytics': 'वर्णनात्मक विश्लेषण', 'Predictive Modeling': 'भविष्य कथन मॉडलिंग', 'Clustering': 'क्लस्टरिंग', 'Regression Analysis': 'प्रतिगमन विश्लेषण', 'Survival Analysis': 'अस्तित्व विश्लेषण', 'Network Analysis': 'नेटवर्क विश्लेषण'},
                        'Russian': {'Forecasting': 'Прогнозирование', 'Classification': 'Классификация', 'Anomaly Detection': 'Обнаружение Аномалий', 'Cohort Analysis': 'Когортный Анализ', 'Segmentation': 'Сегментация', 'Sentiment Analysis': 'Анализ Настроений', 'Trend Analysis': 'Анализ Трендов', 'Prescriptive Analytics': 'Предписывающая Аналитика', 'Root Cause Analysis': 'Анализ Первопричин', 'Optimization': 'Оптимизация', 'Recommendation': 'Рекомендация', 'Time Series Analysis': 'Анализ Временных Рядов', 'Predictive Analytics': 'Предиктивная Аналитика', 'Descriptive Analytics': 'Описательная Аналитика', 'Predictive Modeling': 'Предиктивное Моделирование', 'Clustering': 'Кластеризация', 'Regression Analysis': 'Регрессионный Анализ', 'Survival Analysis': 'Анализ Выживаемости', 'Network Analysis': 'Сетевой Анализ'},
                    }
                    
                    translation_key = analytics_key_map.get(value)
                    translated = t.get(translation_key, None) if translation_key else None
                    if translated and translated != value:
                        return translated
                    if language in analytics_fallbacks and value in analytics_fallbacks[language]:
                        return analytics_fallbacks[language][value]
                    return value
                
                for uc in domain_use_cases:
                    slide = prs.slides.add_slide(SLIDE_LAYOUT_TITLE_AND_CONTENT); slide.background.fill.solid(); slide.background.fill.fore_color.rgb = WHITE_COLOR
                    accent_bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, int(Cm(1.5)), int(prs.slide_height))
                    accent_bar.fill.solid(); accent_bar.fill.fore_color.rgb = DATABRICKS_ORANGE; accent_bar.line.fill.background()
                    
                    title = slide.shapes.title; title.left, title.width = int(LEFT_MARGIN_CM), int(CONTENT_WIDTH_CM)
                    title.top = int(Cm(1.0)); title.height = int(Cm(2.5))
                    uc_title = uc.get('Name', 'N/A') if language != "English" else self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', ''))
                    _pptx_nb_url = self._get_use_case_notebook_url(uc)
                    title.text_frame.clear()
                    p = title.text_frame.paragraphs[0]; p.alignment = align
                    run_id = p.add_run(); run_id.text = uc['No']; run_id.font.size = Pt(32)
                    if _pptx_nb_url:
                        run_id.hyperlink.address = _pptx_nb_url
                        run_id.font.underline = True
                    run_id.font.color.rgb = DATABRICKS_BLUE
                    run_rest = p.add_run(); run_rest.text = f": {uc_title}"; run_rest.font.size = Pt(32); run_rest.font.color.rgb = DATABRICKS_BLUE
                    
                    content_placeholder = slide.placeholders[1]; content_placeholder.left, content_placeholder.width = int(LEFT_MARGIN_CM), int(Cm(30.5))
                    content_placeholder.top = int(Cm(4.1)); content_placeholder.height = int(Cm(14.1))
                    content_frame = content_placeholder.text_frame; content_frame.clear(); content_frame.word_wrap = True
                    
                    # Add header line with Subdomain, Type, Analytics Technique, and Priority (with translations)
                    # Use the first paragraph instead of adding a new one to avoid empty line at top
                    header_p = content_frame.paragraphs[0]; header_p.level = 0; header_p.alignment = align
                    subdomain_val = uc.get('Subdomain', 'N/A')
                    type_val = translate_pptx_value(uc.get('type', 'N/A'))
                    analytics_technique_val = translate_analytics_pptx_value(uc.get('Analytics Technique', 'N/A'))
                    quality_val = translate_pptx_value(uc.get('Quality', 'N/A'))
                    header_text = f"{t['subdomain']}: {subdomain_val} | {t['type']}: {type_val} | {t.get('analytics_technique', 'Analytics Technique')}: {analytics_technique_val} | {t['quality']}: {quality_val}"
                    header_run = header_p.add_run(); header_run.text = header_text; header_run.font.bold = True; header_run.font.size = Pt(22)
                    set_font_color(header_run, DATABRICKS_ORANGE)
                    header_p.space_after = Pt(16)
                    
                    def add_detail_line(frame, label_key, value, align, is_first=False):
                        p = frame.add_paragraph(); p.level = 0; p.alignment = align
                        if not is_first: p.space_before = Pt(12)
                        run_label = p.add_run(); run_label.text = f"{t[label_key]}: "; run_label.font.bold = True; run_label.font.size = Pt(18)
                        set_font_color(run_label, DATABRICKS_BLUE)
                        run_value = p.add_run(); run_value.text = value; run_value.font.size = Pt(18)
                        set_font_color(run_value, TEXT_COLOR)
                    # Type is already shown in header line above, no need to repeat it
                    p = content_frame.add_paragraph(); p.level = 0; p.alignment = align
                    run_label = p.add_run(); run_label.text = f"{t.get('description_label', 'Description')}: "; run_label.font.bold = True; run_label.font.size = Pt(18)
                    set_font_color(run_label, DATABRICKS_BLUE)
                    run_value = p.add_run(); run_value.text = uc.get('Name', 'N/A'); run_value.font.size = Pt(18)
                    set_font_color(run_value, TEXT_COLOR)
                    add_detail_line(content_frame, 'statement', uc.get('Statement', 'N/A'), align)
                    add_detail_line(content_frame, 'solution', uc.get('Solution', 'N/A'), align)
                    add_detail_line(content_frame, 'business_value', uc.get('Business Value', 'N/A'), align)
                    add_detail_line(content_frame, 'beneficiary', uc.get('Beneficiary', 'N/A'), align)
                    add_detail_line(content_frame, 'sponsor', uc.get('Sponsor', 'N/A'), align)
                    # Add Business Priority Alignment
                    priority_alignment_label = t.get('business_priority_alignment', 'Business Priority Alignment')
                    p = content_frame.add_paragraph(); p.level = 0; p.alignment = align; p.space_before = Pt(12)
                    run_label = p.add_run(); run_label.text = f"{priority_alignment_label}: "; run_label.font.bold = True; run_label.font.size = Pt(18)
                    set_font_color(run_label, DATABRICKS_BLUE)
                    run_value = p.add_run(); run_value.text = translate_strategic_pptx_value(uc.get('Business Priority Alignment', 'General Improvement')); run_value.font.size = Pt(18)
                    set_font_color(run_value, TEXT_COLOR)
                    strategic_goals_label = t.get('strategic_goals_alignment', 'Strategic Goals Alignment')
                    p = content_frame.add_paragraph(); p.level = 0; p.alignment = align; p.space_before = Pt(8)
                    run_label = p.add_run(); run_label.text = f"{strategic_goals_label}: "; run_label.font.bold = True; run_label.font.size = Pt(18)
                    set_font_color(run_label, DATABRICKS_BLUE)
                    run_value = p.add_run(); run_value.text = translate_strategic_pptx_value(uc.get('Strategic Goals Alignment', 'General Improvement')); run_value.font.size = Pt(18)
                    set_font_color(run_value, TEXT_COLOR)
                    add_footer(slide)
            
            local_pptx_path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pptx") as tmp_file: local_pptx_path = tmp_file.name
                prs.save(local_pptx_path)
                logger_instance.info(f"Presentation saved locally to {local_pptx_path}")
                _save_pptx(local_pptx_path, workspace_path, logger_instance)
            except Exception as e: logger_instance.error(f"Failed to save or upload PPTX: {get_clean_error_message(e)}")
            finally:
                if local_pptx_path and os.path.exists(local_pptx_path): os.remove(local_pptx_path)

        def _save_pptx(local_pptx_path: str, workspace_path: str, logger_instance):
            try:
                with open(local_pptx_path, "rb") as f: pptx_data = f.read()
                if not pptx_data: raise ValueError("Generated PPTX file is empty.")
                logger_instance.info(f"Uploading PPTX to workspace path: {workspace_path}")
                pptx_data_b64 = base64.b64encode(pptx_data).decode()
                self.w_client.workspace.import_(path=workspace_path, content=pptx_data_b64, format=workspace.ImportFormat.AUTO, overwrite=True)
                abs_path = self.w_client.workspace.get_status(workspace_path).path
                logger_instance.info(f"Success! Presentation uploaded to: {abs_path}")
                log_print(f"Success! Presentation ({language}) generated: {abs_path}")
            except Exception as e: logger_instance.critical(f"Failed to save and upload PPTX: {get_clean_error_message(e)}")

        # --- Main execution logic for generate_presentation_pptx ---
        try:
            if not _install_pptx_dependencies(self.logger):
                self.logger.error("Skipping PPTX generation due to missing python-pptx dependency.")
                return

            if not grouped_data:
                self.logger.warning(f"No use cases provided to generate_presentation_pptx for {language}. Skipping.")
                return
            pptx_workspace_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire_{lang_abbr}.pptx")
            _build_presentation(grouped_data, summary_dict, transliterated_name, t, pptx_workspace_path, self.logger, is_rtl)
        except Exception as e:
            self.logger.critical(f"An error occurred during PPTX generation for {language}: {get_clean_error_message(e)}")
    
    def _install_excel_dependencies(self, logger_instance) -> bool:
        """Installs xlsxwriter if not present."""
        try:
            import xlsxwriter
            logger_instance.info("Excel package (xlsxwriter) already installed.")
            return True
        except ImportError:
            logger_instance.info(f"Installing required Excel package: xlsxwriter...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "xlsxwriter"])
                import xlsxwriter
                logger_instance.info("Successfully installed xlsxwriter.")
                return True
            except Exception as e:
                logger_instance.error(f"Failed to install xlsxwriter: {get_clean_error_message(e)}")
                print("ERROR: Failed to install 'xlsxwriter'. Excel generation cannot continue.", file=sys.stderr)
                return False

    def _save_excel(self, local_excel_path: str, workspace_path: str, logger_instance, language: str):
        """Uploads a locally generated Excel file to the Databricks workspace."""
        try:
            with open(local_excel_path, "rb") as f: excel_data = f.read()
            if not excel_data: raise ValueError("Generated Excel file is empty.")
            logger_instance.info(f"Uploading Excel to workspace path: {workspace_path}")
            excel_data_b64 = base64.b64encode(excel_data).decode()
            self.w_client.workspace.import_(
                path=workspace_path, content=excel_data_b64,
                format=workspace.ImportFormat.AUTO, overwrite=True
            )
            abs_path = self.w_client.workspace.get_status(workspace_path).path
            logger_instance.info(f"Success! Excel Catalog uploaded to: {abs_path}")
            log_print(f"Success! Excel Catalog ({language}) generated: {abs_path}")
        except Exception as e:
            logger_instance.critical(f"Failed to save and upload Excel: {get_clean_error_message(e)}")

    def _generate_use_case_excel(self, language: str, lang_abbr: str, grouped_data: dict):
        warnings.filterwarnings('ignore', module='xlsxwriter')
        # Only generate Excel for English
        if language != "English":
            self.logger.info(f"Skipping Excel generation for {language} (only English Excel is generated).")
            return
        
        self.logger.info(f"--- Starting Excel Catalog Generation with XlsxWriter for {language} ---")
        
        local_excel_path = None
        try:
            if not self._install_excel_dependencies(self.logger):
                self.logger.error("Skipping Excel generation due to missing xlsxwriter dependency.")
                return

            import xlsxwriter
            
            # Prepare data
            data_rows = []
            
            def safe_str(value, default='N/A'):
                """Safely convert value to string, handling None/empty."""
                if value is None or (isinstance(value, str) and not value.strip()):
                    return default
                return str(value)
            
            _quality_sort_order = {'Ultra High': 0, 'Very High': 1, 'High': 2, 'Medium': 3, 'Low': 4, 'Very Low': 5, 'Ultra Low': 6, 'N/A': 7}
            
            id_notebook_urls = {}
            for domain, use_cases in grouped_data.items():
                for uc in use_cases:
                    uc_id = safe_str(uc.get('No'), 'N/A')
                    nb_url = self._get_use_case_notebook_url(uc)
                    if nb_url:
                        id_notebook_urls[uc_id] = nb_url
                    data_rows.append([
                        uc_id,                                                         # 0 - ID
                        safe_str(uc.get('Business Domain'), 'N/A'),                    # 1 - Business Domain
                        safe_str(uc.get('Subdomain'), 'N/A'),                          # 2 - Subdomain
                        safe_str(self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', '')), 'N/A'),  # 3 - Use Case
                        safe_str(uc.get('Name'), 'N/A'),                               # 4 - Description
                        uc.get('Inspire Score', 0),                                    # 5 - Inspire Score
                        safe_str(uc.get('type'), 'N/A'),                               # 6 - Type
                        safe_str(uc.get('Analytics Technique'), 'N/A'),                # 7 - Analytics Technique
                        safe_str(uc.get('Business Priority Alignment'), 'General Improvement'),  # 8 - Business Priority Alignment
                        safe_str(uc.get('Strategic Goals Alignment'), 'General Improvement'),    # 9 - Strategic Goals Alignment
                        safe_str(uc.get('Statement'), 'N/A'),                          # 10 - Statement
                        safe_str(uc.get('Solution'), 'N/A'),                           # 11 - Solution
                        safe_str(uc.get('Business Value'), 'N/A'),                     # 12 - Business Value
                        safe_str(uc.get('Beneficiary'), 'N/A'),                        # 13 - Beneficiary
                        safe_str(uc.get('Sponsor'), 'N/A'),                            # 14 - Sponsor
                        safe_str(uc.get('Tables Involved'), 'N/A'),                    # 15 - Tables Involved
                        safe_str(uc.get('Primary Table'), 'N/A'),                      # 16 - Primary Table
                        uc.get('Strategic Alignment', 0),                              # 17 - Strategic Alignment
                        uc.get('Return on Investment', 0),                             # 18 - ROI
                        uc.get('Reusability', 0),                                      # 19 - Reusability
                        uc.get('Time to Value', 0),                                    # 20 - Time to Value
                        uc.get('Data Availability', 0),                                # 21 - Data Availability
                        uc.get('Data Accessibility', 0),                               # 22 - Data Accessibility
                        uc.get('Architecture Fitness', 0),                             # 23 - Architecture Fitness
                        uc.get('Team Skills', 0),                                      # 24 - Team Skills
                        uc.get('Domain Knowledge', 0),                                 # 25 - Domain Knowledge
                        uc.get('People Allocation', 0),                                # 26 - People Allocation
                        uc.get('Budget Allocation', 0),                                # 27 - Budget Allocation
                        uc.get('Time to Production', 0),                               # 28 - Time to Production
                        uc.get('Value', 0),                                            # 29 - Value
                        uc.get('Feasibility', 0),                                      # 30 - Feasibility
                        uc.get('Priority', 0),                                         # 31 - Priority
                        uc.get('Quality Score', 0),                                    # 32 - Quality (numeric)
                        safe_str(uc.get('Justification'), 'N/A'),                      # 33 - Priority Reasons
                        safe_str(uc.get('Quality Summary'), 'N/A'),                    # 34 - Quality Reasons
                        safe_str(uc.get('result_table'), 'N/A'),                       # 35 - Result Database
                        safe_str(uc.get('has_genie_code'), 'N'),                       # 36 - Has Genie Code
                    ])
            
            inspire_score_idx = 5
            data_rows.sort(key=lambda row: float(row[inspire_score_idx]) if row[inspire_score_idx] not in ('N/A', '', None) else 0, reverse=True)
            
            if not data_rows:
                self.logger.warning(f"No data to write to Excel for {language}. Skipping.")
                return
            
            # Create Excel file
            # v0.7.2 bug fix: per-language suffix when multi-language (single-lang keeps legacy name).
            _output_langs_xl = getattr(self, "output_languages", ["English"]) or ["English"]
            if len(_output_langs_xl) > 1:
                excel_file_name = f"{self.business_name}-dbx_inspire_{lang_abbr}.xlsx"
            else:
                excel_file_name = f"{self.business_name}-dbx_inspire.xlsx"
            with tempfile.NamedTemporaryFile(delete=False, suffix=".xlsx") as tmp_file:
                local_excel_path = tmp_file.name
            
            self.logger.info(f"Creating Excel file at {local_excel_path}")
            workbook = xlsxwriter.Workbook(local_excel_path, {'strings_to_numbers': False})
            worksheet = workbook.add_worksheet('Use Cases')
            
            # Modern Business Color Palette
            PRIMARY = '#2C3E50'      # Deep Slate
            SECONDARY = '#E74C3C'    # Vibrant Coral  
            ACCENT = '#3498DB'       # Bright Blue
            BACKGROUND = '#ECF0F1'   # Soft Grey
            TEXT = '#2C3E50'         # Dark Grey
            
            # Define cell formats
            header_format = workbook.add_format({
                'bold': True,
                'font_color': 'white',
                'bg_color': PRIMARY,
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'text_wrap': True,
                'font_size': 11
            })
            
            cell_format = workbook.add_format({
                'border': 1,
                'align': 'left',
                'valign': 'top',
                'text_wrap': True,
                'font_size': 10
            })
            
            numeric_format = workbook.add_format({
                'border': 1,
                'align': 'center',
                'valign': 'vcenter',
                'num_format': '0.00',
                'font_size': 10
            })
            
            headers = [
                "ID", "Business Domain", "Subdomain", "Use Case", "Description",
                "Inspire Score",
                "Type", "Analytics Technique",
                "Business Priority Alignment", "Strategic Goals Alignment",
                "Statement", "Solution", "Business Value", "Beneficiary", "Sponsor",
                "Tables Involved", "Primary Table",
                "Strategic Alignment", "ROI", "Reusability", "Time to Value",
                "Data Availability", "Data Accessibility", "Architecture Fitness", "Team Skills",
                "Domain Knowledge", "People Allocation", "Budget Allocation", "Time to Production",
                "Value", "Feasibility", "Priority", "Quality",
                "Priority Reasons", "Quality Reasons",
                "Result Database",
                "Has Genie Code",
            ]
            
            inspire_score_col = 5
            id_col = 0
            numeric_score_cols = set(range(17, 33))
            
            # Write headers
            for col_num, header in enumerate(headers):
                worksheet.write(0, col_num, header, header_format)
            
            id_link_format = workbook.add_format({
                'border': 1,
                'align': 'left',
                'valign': 'top',
                'text_wrap': True,
                'font_size': 10,
                'font_color': '#0563C1',
                'underline': True,
            })

            for row_num, row_data in enumerate(data_rows, start=1):
                for col_num, cell_data in enumerate(row_data):
                    if col_num == id_col:
                        id_text = str(cell_data)
                        nb_url = id_notebook_urls.get(id_text)
                        if nb_url:
                            worksheet.write_url(row_num, col_num, nb_url, id_link_format, string=id_text)
                        else:
                            worksheet.write(row_num, col_num, id_text, cell_format)
                    elif col_num == inspire_score_col or col_num in numeric_score_cols:
                        try:
                            numeric_value = float(cell_data) if cell_data not in ['N/A', '', None] else 0
                            worksheet.write_number(row_num, col_num, numeric_value, numeric_format)
                        except (ValueError, TypeError):
                            worksheet.write(row_num, col_num, cell_data, cell_format)
                    else:
                        worksheet.write(row_num, col_num, cell_data, cell_format)
            
            # Auto-fit column widths to content
            for col_num in range(len(headers)):
                # Calculate max width for this column
                max_width = len(str(headers[col_num]))
                for row_data in data_rows:
                    if col_num < len(row_data):
                        cell_value = str(row_data[col_num])
                        max_width = max(max_width, len(cell_value))
                # Set width to fit entire text
                column_width = max_width + 2
                worksheet.set_column(col_num, col_num, column_width)
            
            # Freeze top row
            worksheet.freeze_panes(1, 0)
            
            # Convert data range to native Excel Table
            last_row = len(data_rows)
            last_col = len(headers) - 1
            worksheet.add_table(0, 0, last_row, last_col, {
                'name': 'UseCaseTable',
                'style': 'Table Style Medium 9',
                'columns': [{'header': h} for h in headers]
            })
            
            gradient_cols = [inspire_score_col] + list(range(17, 33))
            for col_idx in gradient_cols:
                worksheet.conditional_format(1, col_idx, last_row, col_idx, {
                    'type': '3_color_scale',
                    'min_color': '#E74C3C',
                    'mid_color': '#F4D03F',
                    'max_color': '#27AE60',
                })
            
            # Graph sheet removed per user request - no longer needed
            
            workbook.close()
            self.logger.info(f"Excel file created successfully with native tables")
            
            workspace_excel_path = os.path.join(self.docs_output_dir, excel_file_name)
            self._save_excel(local_excel_path, workspace_excel_path, self.logger, language)
        except Exception as e:
            self.logger.critical(f"An error occurred during Excel generation for {language}: {get_clean_error_message(e)}")
            raise
        finally:
            if local_excel_path and os.path.exists(local_excel_path):
                os.remove(local_excel_path)

    def _generate_markdown_catalog(self, language: str, lang_abbr: str, grouped_data: dict, summary_dict: dict, transliterated_name: str):
        """
        Generates a Markdown catalog file containing all use case information.
        This is ALWAYS generated as a fallback when PDF generation may fail.
        """
        if language != "English":
            self.logger.info(f"Skipping Markdown generation for {language} (only English Markdown is generated).")
            return
        
        self.logger.info(f"--- Starting Markdown Catalog Generation for {language} ---")
        
        try:
            # Build markdown content
            md_content = []
            
            # Header
            md_content.append(f"# {self.business_name} - Databricks Inspire AI Use Cases Catalog\n")
            md_content.append(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            md_content.append(f"**Business Name:** {transliterated_name or self.business_name}\n")
            
            # Summary section if available
            if summary_dict:
                md_content.append("\n## Executive Summary\n")
                if summary_dict.get('executive_summary'):
                    md_content.append(f"{summary_dict.get('executive_summary')}\n")
                if summary_dict.get('key_findings'):
                    md_content.append(f"\n**Key Findings:** {summary_dict.get('key_findings')}\n")
                if summary_dict.get('recommendations'):
                    md_content.append(f"\n**Recommendations:** {summary_dict.get('recommendations')}\n")
            
            # Stats
            total_use_cases = sum(len(ucs) for ucs in grouped_data.values())
            md_content.append(f"\n## Overview\n")
            md_content.append(f"- **Total Use Cases:** {total_use_cases}\n")
            md_content.append(f"- **Business Domains:** {len(grouped_data)}\n")
            
            # Use cases by domain
            md_content.append("\n## Use Cases by Business Domain\n")
            
            for domain_name, use_cases in grouped_data.items():
                md_content.append(f"\n### {domain_name}\n")
                md_content.append(f"*{len(use_cases)} use cases*\n")
                
                for uc in use_cases:
                    uc_id = uc.get('No', 'N/A')
                    uc_name = uc.get('Name', 'Unnamed')
                    uc_usecase = self._normalize_usecase(uc.get('usecase', ''), uc_name)
                    uc_quality = uc.get('Quality', 'Medium')
                    uc_type = uc.get('type', 'N/A')
                    uc_statement = uc.get('Statement', 'N/A')
                    uc_solution = uc.get('Solution', 'N/A')
                    uc_value = uc.get('Business Value', 'N/A')
                    uc_beneficiary = uc.get('Beneficiary', 'N/A')
                    uc_sponsor = uc.get('Sponsor', 'N/A')
                    uc_tables = uc.get('Primary Table', 'N/A')
                    uc_priority_score = uc.get('Priority', 0)
                    uc_justification = uc.get('Justification', 'N/A')
                    uc_subdomain = uc.get('Subdomain', 'N/A')
                    uc_analytics = uc.get('Analytics Technique', 'N/A')
                    uc_bp_alignment = uc.get('Business Priority Alignment', 'N/A')
                    uc_sg_alignment = uc.get('Strategic Goals Alignment', 'N/A')
                    
                    _md_nb_url = self._get_use_case_notebook_url(uc)
                    if _md_nb_url:
                        md_content.append(f"\n#### [{uc_id}]({_md_nb_url}): {uc_usecase}\n")
                    else:
                        md_content.append(f"\n#### {uc_id}: {uc_usecase}\n")
                    md_content.append(f"- **Description:** {uc_name}\n")
                    md_content.append(f"- **Subdomain:** {uc_subdomain}\n")
                    md_content.append(f"- **Type:** {uc_type}\n")
                    md_content.append(f"- **Quality:** {uc_quality} (Score: {uc_priority_score})\n")
                    md_content.append(f"- **Analytics Technique:** {uc_analytics}\n")
                    md_content.append(f"- **Business Priority Alignment:** {uc_bp_alignment}\n")
                    md_content.append(f"- **Strategic Goals Alignment:** {uc_sg_alignment}\n")
                    md_content.append(f"\n**Problem Statement:**\n{uc_statement}\n")
                    md_content.append(f"\n**Solution:**\n{uc_solution}\n")
                    md_content.append(f"\n**Business Value:**\n{uc_value}\n")
                    md_content.append(f"\n**Beneficiary:** {uc_beneficiary}\n")
                    md_content.append(f"**Sponsor:** {uc_sponsor}\n")
                    md_content.append(f"**Primary Table:** {uc_tables}\n")
                    if uc_justification and uc_justification != 'N/A':
                        md_content.append(f"\n**Justification:**\n{uc_justification}\n")
                    md_content.append("\n---\n")
            
            # Save to workspace
            # v0.7.2 bug fix: multi-language must NOT share the same filename.
            # Single-language run → readme.md (preserves "only-md-in-folder" convention).
            # Multi-language run → readme_{lang_abbr}.md per language.
            _output_langs = getattr(self, "output_languages", ["English"]) or ["English"]
            if len(_output_langs) > 1:
                md_file_name = f"readme_{lang_abbr}.md"
            else:
                md_file_name = "readme.md"
            workspace_md_path = os.path.join(self.docs_output_dir, md_file_name)
            md_text = ''.join(md_content)
            
            # Upload to workspace
            md_data_b64 = base64.b64encode(md_text.encode('utf-8')).decode()
            self.w_client.workspace.import_(
                path=workspace_md_path, content=md_data_b64,
                format=workspace.ImportFormat.AUTO, overwrite=True
            )
            abs_path = self.w_client.workspace.get_status(workspace_md_path).path
            self.logger.info(f"Success! Markdown Catalog uploaded to: {abs_path}")
            log_print(f"Success! Markdown Catalog ({language}) generated: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to generate Markdown catalog for {language}: {get_clean_error_message(e)}")

    def _generate_csv_catalog(self, language: str, lang_abbr: str, grouped_data: dict):
        """
        Generates a CSV catalog file containing all use case information.
        This is ALWAYS generated as a fallback when Excel generation may fail.
        """
        if language != "English":
            self.logger.info(f"Skipping CSV generation for {language} (only English CSV is generated).")
            return
        
        self.logger.info(f"--- Starting CSV Catalog Generation for {language} ---")
        
        try:
            data_rows = []
            headers = [
                "ID", "Business Domain", "Subdomain", "Use Case", "Description",
                "Inspire Score",
                "Type", "Analytics Technique",
                "Business Priority Alignment", "Strategic Goals Alignment",
                "Statement", "Solution", "Business Value", "Beneficiary", "Sponsor",
                "Tables Involved", "Primary Table",
                "Strategic Alignment", "ROI", "Reusability", "Time to Value",
                "Data Availability", "Data Accessibility", "Architecture Fitness", "Team Skills",
                "Domain Knowledge", "People Allocation", "Budget Allocation", "Time to Production",
                "Value", "Feasibility", "Priority", "Quality",
                "Priority Reasons", "Quality Reasons",
                "Result Database",
                "Has Genie Code",
            ]
            
            for domain, use_cases in grouped_data.items():
                for uc in use_cases:
                    data_rows.append([
                        uc.get('No', 'N/A'),
                        uc.get('Business Domain', 'N/A'),
                        uc.get('Subdomain', 'N/A'),
                        self._normalize_usecase(uc.get('usecase', ''), uc.get('Name', '')),
                        uc.get('Name', 'N/A'),
                        uc.get('Inspire Score', 0),
                        uc.get('type', 'N/A'),
                        uc.get('Analytics Technique', 'N/A'),
                        uc.get('Business Priority Alignment', 'General Improvement'),
                        uc.get('Strategic Goals Alignment', 'General Improvement'),
                        uc.get('Statement', 'N/A'),
                        uc.get('Solution', 'N/A'),
                        uc.get('Business Value', 'N/A'),
                        uc.get('Beneficiary', 'N/A'),
                        uc.get('Sponsor', 'N/A'),
                        uc.get('Tables Involved', 'N/A'),
                        uc.get('Primary Table', 'N/A'),
                        uc.get('Strategic Alignment', 0),
                        uc.get('Return on Investment', 0),
                        uc.get('Reusability', 0),
                        uc.get('Time to Value', 0),
                        uc.get('Data Availability', 0),
                        uc.get('Data Accessibility', 0),
                        uc.get('Architecture Fitness', 0),
                        uc.get('Team Skills', 0),
                        uc.get('Domain Knowledge', 0),
                        uc.get('People Allocation', 0),
                        uc.get('Budget Allocation', 0),
                        uc.get('Time to Production', 0),
                        uc.get('Value', 0),
                        uc.get('Feasibility', 0),
                        uc.get('Priority', 0),
                        uc.get('Quality Score', 0),
                        uc.get('Justification', 'N/A'),
                        uc.get('Quality Summary', 'N/A'),
                        uc.get('result_table', 'N/A'),
                        uc.get('has_genie_code', 'N'),
                    ])
            
            data_rows.sort(key=lambda row: float(row[5]) if row[5] not in ('N/A', '', None) else 0, reverse=True)
            
            if not data_rows:
                self.logger.warning(f"No data to write to CSV for {language}. Skipping.")
                return
            
            # Build CSV content
            output = io.StringIO()
            writer = csv.writer(output, quoting=csv.QUOTE_ALL)
            writer.writerow(headers)
            for row in data_rows:
                writer.writerow(row)
            csv_content = output.getvalue()
            
            # Save to workspace
            # v0.8.1 bug fix (multi-language): CSV writer was missed by the v0.7.3 patch
            # that fixed Excel and Markdown. When multiple languages are selected, each
            # language wrote over the previous one with the same filename. Apply the
            # same suffix convention as Excel/Markdown/PDF/PPTX.
            _output_langs_csv = getattr(self, "output_languages", ["English"]) or ["English"]
            if len(_output_langs_csv) > 1:
                csv_file_name = f"{self.business_name}-dbx_inspire_{lang_abbr}.csv"
            else:
                csv_file_name = f"{self.business_name}-dbx_inspire.csv"
            workspace_csv_path = os.path.join(self.docs_output_dir, csv_file_name)
            
            # Upload to workspace
            csv_data_b64 = base64.b64encode(csv_content.encode('utf-8')).decode()
            self.w_client.workspace.import_(
                path=workspace_csv_path, content=csv_data_b64,
                format=workspace.ImportFormat.AUTO, overwrite=True
            )
            abs_path = self.w_client.workspace.get_status(workspace_csv_path).path
            self.logger.info(f"Success! CSV Catalog uploaded to: {abs_path}")
            log_print(f"Success! CSV Catalog ({language}) generated: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to generate CSV catalog for {language}: {get_clean_error_message(e)}")

    def _validate_use_case_tables(self, parsed_rows: list, full_schema_details: list, log_prefix: str) -> tuple:
        """
        Validate that all tables referenced in 'Tables Involved' field actually exist in the schema.
        This catches LLM hallucinations where it invents table names.
        
        Returns:
            tuple: (is_valid: bool, hallucinated_use_cases: list, valid_use_cases: list)
        """
        import re
        
        # Build set of available tables from schema
        available_tables = set()
        for detail in full_schema_details:
            (catalog, schema, table, _, _, _) = detail
            available_tables.add(f"{catalog}.{schema}.{table}")
            available_tables.add(f"`{catalog}`.`{schema}`.`{table}`")
            available_tables.add(f"{catalog}.{schema}.{table}".lower())
        global_tables = getattr(self, "global_table_names", set())
        for tbl in global_tables:
            available_tables.add(tbl)
            available_tables.add(tbl.lower())
        
        hallucinated_use_cases = []
        valid_use_cases = []
        
        for row in parsed_rows:
            tables_involved_str = row.get('Tables Involved', '').strip()
            use_case_id = row.get('No', 'Unknown')
            use_case_name = row.get('Name', 'Unknown')
            
            # Skip volume paths (for ai_parse_document use cases)
            if tables_involved_str.startswith('/Volumes'):
                valid_use_cases.append(row)
                continue
            
            # Skip empty tables involved (will be caught by other validation)
            if not tables_involved_str:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = "No tables specified"
                continue
            
            sql_fragment_keywords = {'SELECT', 'FROM', 'WHERE', 'CTE', 'JOIN', 'LIMIT', 'GROUP BY', 'ORDER BY', 'HAVING', 'UNION', 'INSERT', 'UPDATE', 'DELETE'}
            upper_tables = tables_involved_str.upper()
            if any(kw in upper_tables for kw in sql_fragment_keywords):
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"SQL fragment in Tables Involved (CSV parsing error): {tables_involved_str[:120]}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: SQL fragment leaked into Tables Involved field")
                continue
            
            table_matches = []
            for table_str in tables_involved_str.split(','):
                table_str = table_str.strip()
                if not table_str:
                    continue
                cat, sch, tbl = parse_three_level_name(table_str)
                if cat and sch and tbl:
                    table_matches.append((cat, sch, tbl))
            
            if not table_matches:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"Invalid table format: {tables_involved_str[:120]}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: Invalid table format: {tables_involved_str[:120]}")
                continue
            
            # Check if all tables exist in schema
            all_tables_found = True
            missing_tables = []
            for match in table_matches:
                catalog, schema, table = match
                # Try multiple formats (normalized names for comparison)
                table_formats = [
                    f"{catalog}.{schema}.{table}",
                    build_fqn(catalog, schema, table),
                    f"{catalog}.{schema}.{table}".lower()
                ]
                
                found = any(fmt in available_tables for fmt in table_formats)
                if not found:
                    all_tables_found = False
                    missing_tables.append(f"{catalog}.{schema}.{table}")
            
            if not all_tables_found:
                hallucinated_use_cases.append(row)
                row['hallucination_reason'] = f"Tables not found in schema: {', '.join(missing_tables)}"
                self.logger.warning(f"{log_prefix} Use case {use_case_id}: Hallucinated tables: {', '.join(missing_tables)}")
            else:
                reference_tables = {k.lower() for k, v in getattr(self, "data_category_map", {}).items() if v == "REFERENCE"}
                if reference_tables:
                    non_reference_found = False
                    for match in table_matches:
                        catalog, schema, table = match
                        fqtn = f"{catalog}.{schema}.{table}".lower()
                        if fqtn not in reference_tables:
                            non_reference_found = True
                            break
                    if not non_reference_found:
                        hallucinated_use_cases.append(row)
                        row['hallucination_reason'] = "Reference-only use case"
                        self.logger.warning(f"{log_prefix} Use case {use_case_id}: Reference-only tables in use case")
                        continue
                valid_use_cases.append(row)
        
        is_valid = len(hallucinated_use_cases) == 0
        
        if not is_valid:
            self.logger.warning(f"{log_prefix} ⚠️ Found {len(hallucinated_use_cases)} use cases with hallucinated/missing tables out of {len(parsed_rows)} total")
            self.logger.warning(f"{log_prefix}    Valid use cases: {len(valid_use_cases)}")
            self.logger.warning(f"{log_prefix}    Hallucinated use cases: {len(hallucinated_use_cases)}")
        else:
            self.logger.info(f"{log_prefix} ✓ Table validation passed: All {len(valid_use_cases)} use cases reference existing tables")
        
        return (is_valid, hallucinated_use_cases, valid_use_cases)
    
    def _validate_subdomain_rules(self, parsed_rows: list, log_prefix: str) -> tuple:
        """
        Validate subdomain rules (silent - no individual violation logging):
        1. Each domain must have at least 2 subdomains
        2. Each subdomain must have at least 3 use cases
        
        Returns:
            tuple: (is_valid: bool, violations: list, corrected_rows: list)
        """
        from collections import defaultdict
        
        # Group by domain and subdomain
        domain_subdomains = defaultdict(set)
        subdomain_usecases = defaultdict(list)
        
        for row in parsed_rows:
            domain = row.get('Business Domain', '').strip()
            subdomain = row.get('Subdomain', '').strip()
            
            if domain and subdomain:
                domain_subdomains[domain].add(subdomain)
                subdomain_usecases[f"{domain}::{subdomain}"].append(row)
        
        violations = []
        
        domain_uc_counts = defaultdict(int)
        for row in parsed_rows:
            # B2: UC dicts carry 'Business Domain' (title-case); previously was lowercase 'domain' (bug)
            d = row.get('Business Domain', '').strip()
            if d:
                domain_uc_counts[d] += 1
        
        for domain, subdomains in domain_subdomains.items():
            min_sd = 1 if domain_uc_counts.get(domain, 0) <= 4 else 2
            if len(subdomains) < min_sd:
                violations.append(f"Domain '{domain}' has only {len(subdomains)} subdomain(s). Minimum required: {min_sd}")
        
        # Check Rule 2: Each subdomain must have at least 3 use cases
        for key, use_cases in subdomain_usecases.items():
            domain, subdomain = key.split('::', 1)
            if len(use_cases) < 3:
                violations.append(f"Subdomain '{subdomain}' in domain '{domain}' has only {len(use_cases)} use case(s). Minimum required: 3")
        
        # Don't log individual violations - they'll be fixed in consolidation
        if violations:
            self.logger.debug(f"{log_prefix} Found {len(violations)} subdomain violations (will be fixed in domain consolidation)")
            return (False, violations, parsed_rows)
        
        self.logger.info(f"{log_prefix} ✓ Subdomain validation passed: All domains have ≥2 subdomains, all subdomains have ≥2 use cases")
        return (True, [], parsed_rows)
    
    def _score_use_case_data_quality(self, use_cases: list, full_schema_details: list, 
                                      business_context: str = "", industry: str = "") -> list:
        """
        📊 USE CASE DATA QUALITY SCORING 📊
        
        This scores each use case based on DATA QUALITY - how well the data supports the high level design.
        Runs batches in PARALLEL for efficiency.
        
        Quality dimensions evaluated:
        1. Causal signal strength
        2. Cause-effect validity
        3. Data granularity
        4. Critical dimensions present
        5. Logical possibility
        6. Metric validity
        7. High level design match
        
        Args:
            use_cases: List of use cases to score
            full_schema_details: Full schema details for quality assessment
            business_context: Business context for domain-specific assessment
            industry: Industry for domain expertise application
            
        Returns:
            list: Use cases with Quality field populated
        """
        import json
        
        if not use_cases:
            self.logger.info("🔍 No use cases to score for data quality")
            return use_cases
        
        self._cached_schema_markdown = self._build_schema_markdown_for_validation(full_schema_details)
        
        _rt = TECHNICAL_CONTEXT["runtime"]
        batch_size = _rt["quality_scoring_batch_size"]
        batches = [use_cases[i:i + batch_size] for i in range(0, len(use_cases), batch_size)]
        
        num_batches = len(batches)
        max_quality_workers = _rt["quality_scoring_max_workers"]
        parallelism = min(num_batches, max_quality_workers)
        if num_batches == 1:
            parallelism = 1
        
        self.logger.info("")
        self.logger.info("=" * 80)
        self.logger.info(f"📊 PHASE 4.5: DATA QUALITY SCORING ({len(use_cases)} use cases)")
        self.logger.info("=" * 80)
        self.logger.info(f"Total use cases: {len(use_cases)}")
        self.logger.info(f"Batch size: {batch_size}")
        self.logger.info(f"Total batches: {num_batches}")
        self.logger.info(f"🔧 [QUALITY_SCORING] Workers: {parallelism}")
        self.logger.info(f"   └─ Reason: min({num_batches} batches, {max_quality_workers} max from runtime config)")
        self.logger.info("=" * 80)
        
        quality_scores_map = {}
        
        def _score_single_batch(batch_label: str, uc_batch: list) -> dict:
            """Score a single batch, returning {uc_id: scores_dict}. Raises on failure."""
            use_cases_markdown = self._build_use_cases_markdown_for_validation(uc_batch)
            prompt_vars = {
                "business_context": business_context or self._get_business_context_fallback(),
                "industry": industry or "General",
                "schema_details": self._cached_schema_markdown,
                "use_cases_to_validate": use_cases_markdown,
                "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_QUALITY_SCORE_PROMPT"),
            }
            response_raw = self.ai_agent.run_worker(
                step_name=f"Quality_{batch_label}",
                worker_prompt_path="USE_CASE_QUALITY_SCORE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None
            )
            quality_results = self._parse_quality_score_response(response_raw, uc_batch)
            result = {}
            for r in quality_results:
                result[r.get('use_case_id', 'UNKNOWN')] = {
                    'Quality': r.get('quality_label', 'Medium'),
                    'Quality Summary': self._sanitize_quality_summary(r.get('quality_summary', '')),
                    'Data Strengths': r.get('strengths', []),
                    'Data Weaknesses': r.get('weaknesses', [])
                }
            return result

        def process_quality_batch(batch_idx: int, batch: list) -> dict:
            """Process a batch with retry-and-split: on failure, halve and retry each sub-batch."""
            batch_results = {}
            self.logger.info(f"📊 [Batch {batch_idx}/{num_batches}] Scoring {len(batch)} use cases...")
            
            try:
                batch_results = _score_single_batch(f"B{batch_idx}", batch)
                self.logger.info(f"✅ [Batch {batch_idx}/{num_batches}] Complete: {len(batch_results)} use cases scored")
                return batch_results
            except Exception as first_err:
                self.logger.warning(
                    f"⚠️ [Batch {batch_idx}/{num_batches}] Failed ({str(first_err)[:120]}). "
                    f"Retrying as {max(1, len(batch) // 2)}-sized sub-batches..."
                )
            
            mid = max(1, len(batch) // 2)
            sub_batches = [batch[:mid], batch[mid:]] if len(batch) > 1 else [batch]
            for sub_idx, sub in enumerate(sub_batches, 1):
                if not sub:
                    continue
                try:
                    sub_results = _score_single_batch(f"B{batch_idx}_S{sub_idx}", sub)
                    batch_results.update(sub_results)
                    self.logger.info(f"✅ [Batch {batch_idx}/{num_batches} sub-{sub_idx}] Scored {len(sub_results)} use cases on retry")
                except Exception as sub_err:
                    self.logger.error(
                        f"❌ [Batch {batch_idx}/{num_batches} sub-{sub_idx}] "
                        f"Retry also failed ({str(sub_err)[:120]}). Assigning defaults for {len(sub)} use cases."
                    )
                    for uc in sub:
                        uc_id = uc.get('No', 'UNKNOWN')
                        batch_results[uc_id] = {
                            'Quality': 'Medium',
                            'Quality Summary': f'Quality scoring default - retry failed: {str(sub_err)[:80]}',
                            'Data Strengths': [],
                            'Data Weaknesses': ['Quality scoring failed after retry']
                        }
            
            return batch_results
        
        per_batch_timeout = self.llm_timeout_seconds * 4
        parallel_succeeded = False
        
        _thread_diag = []
        for t in threading.enumerate():
            _thread_diag.append(f"{t.name}(alive={t.is_alive()}, daemon={t.daemon})")
        self.logger.info(f"🧵 Thread dump before quality scoring ({len(_thread_diag)} threads): {', '.join(_thread_diag[:20])}")
        
        if num_batches >= 2:
            quality_workers = min(parallelism, num_batches)
            total_timeout = per_batch_timeout * ((num_batches + quality_workers - 1) // quality_workers)
            self.logger.info(
                f"🔄 PARALLEL QUALITY SCORING: {quality_workers} workers for {num_batches} batches "
                f"(per-batch timeout: {per_batch_timeout}s, total timeout: {total_timeout}s)"
            )
            
            executor = None
            try:
                executor = ThreadPoolExecutor(max_workers=quality_workers, thread_name_prefix="QualityScore")
                futures = {
                    executor.submit(process_quality_batch, idx, batch): idx
                    for idx, batch in enumerate(batches, 1)
                }
                
                completed_batches = set()
                try:
                    for future in safe_as_completed(futures, timeout=total_timeout):
                        batch_idx = futures[future]
                        try:
                            batch_results = future.result(timeout=per_batch_timeout)
                            quality_scores_map.update(batch_results)
                            completed_batches.add(batch_idx)
                            self.logger.info(f"✅ Quality batch {batch_idx}/{num_batches}: {len(batch_results)} use cases scored")
                        except concurrent.futures.TimeoutError:
                            self.logger.error(f"⏱️ Quality batch {batch_idx} timed out after {per_batch_timeout}s — assigning defaults")
                            self._assign_default_quality_scores(batches[batch_idx - 1], quality_scores_map, 'Scoring timeout')
                            completed_batches.add(batch_idx)
                        except Exception as e:
                            self.logger.error(f"❌ Quality batch {batch_idx} exception: {get_clean_error_message(e)}")
                            self._assign_default_quality_scores(batches[batch_idx - 1], quality_scores_map, f'Scoring error: {str(e)[:80]}')
                            completed_batches.add(batch_idx)
                except concurrent.futures.TimeoutError:
                    timed_out_idxs = [idx for idx in range(1, num_batches + 1) if idx not in completed_batches]
                    _alive_threads = [(t.name, t.is_alive()) for t in threading.enumerate() if 'QualityScore' in t.name or 'LLM_Query' in t.name]
                    self.logger.error(
                        f"⏱️ Quality scoring TOTAL timeout after {total_timeout}s. "
                        f"Completed {len(completed_batches)}/{num_batches}. Timed-out batches: {timed_out_idxs}. Stuck threads: {_alive_threads}"
                    )
                    for future in futures:
                        if not future.done():
                            future.cancel()
                    for idx in timed_out_idxs:
                        self._assign_default_quality_scores(batches[idx - 1], quality_scores_map, 'Scoring timeout (total)')
                    parallel_succeeded = True
                else:
                    parallel_succeeded = True
            except Exception as e:
                self.logger.error(f"❌ Quality scoring parallel execution failed entirely: {get_clean_error_message(e)}")
                _alive_threads = [(t.name, t.is_alive()) for t in threading.enumerate() if 'QualityScore' in t.name or 'LLM_Query' in t.name]
                self.logger.error(f"🧵 Thread state after failure: {_alive_threads}")
            else:
                if completed_batches:
                    parallel_succeeded = True
            finally:
                if executor:
                    try:
                        executor.shutdown(wait=False, cancel_futures=True)
                    except TypeError:
                        executor.shutdown(wait=False)
                    gc.collect()
        
        if not parallel_succeeded:
            self.logger.info(f"🔄 SEQUENTIAL FALLBACK: Processing {num_batches} quality batches one at a time...")
            import time as _time
            for idx, batch in enumerate(batches, 1):
                batch_start = _time.time()
                self.logger.info(f"🔄 [{idx}/{num_batches}] Quality scoring batch ({len(batch)} use cases)...")
                try:
                    batch_results = process_quality_batch(idx, batch)
                    quality_scores_map.update(batch_results)
                    elapsed = _time.time() - batch_start
                    self.logger.info(f"✅ [{idx}/{num_batches}] Quality batch complete in {elapsed:.1f}s")
                except Exception as e:
                    self.logger.error(f"❌ [{idx}/{num_batches}] Quality batch failed: {get_clean_error_message(e)} — assigning defaults")
                    self._assign_default_quality_scores(batch, quality_scores_map, f'Sequential error: {str(e)[:80]}')
        
        # Apply scores to use cases
        for uc in use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id in quality_scores_map:
                scores = quality_scores_map[uc_id]
                uc['Quality'] = scores.get('Quality', 'Medium')
                uc['Quality Summary'] = scores.get('Quality Summary', '')
            else:
                uc['Quality'] = 'Medium'
                uc['Quality Summary'] = 'No quality score received - default assigned'
        
        # Calculate distribution
        quality_distribution = {}
        for uc in use_cases:
            label = uc.get('Quality', 'Medium')
            quality_distribution[label] = quality_distribution.get(label, 0) + 1
        
        self.logger.info("")
        self.logger.info("=" * 80)
        self.logger.info("✅ PHASE 4.5 COMPLETE: DATA QUALITY SCORING")
        self.logger.info("=" * 80)
        self.logger.info(f"Total scored: {len(use_cases)} use cases")
        for label in ['Ultra High', 'Very High', 'High', 'Medium', 'Low', 'Very Low', 'Ultra Low']:
            count = quality_distribution.get(label, 0)
            if count > 0:
                self.logger.info(f"   {label}: {count} use cases")
        self.logger.info("=" * 80)
        
        del quality_scores_map, batches
        self._cached_schema_markdown = None
        _force_gc(self.logger, "after quality scoring")
        
        return use_cases
    
    def _assign_default_quality_scores(self, batch: list, scores_map: dict, reason: str):
        for uc in batch:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id not in scores_map:
                scores_map[uc_id] = {
                    'Quality': 'Medium',
                    'Quality Summary': f'Quality scoring default - {reason}',
                    'Data Strengths': [],
                    'Data Weaknesses': [reason]
                }

    def _score_combined_value_and_quality(self, use_cases: list, full_schema_details: list,
                                           business_context: str = "", industry: str = "",
                                           strategic_goals: list = None, business_priorities: list = None,
                                           strategic_initiative: str = "", value_chain: str = "",
                                           revenue_model: str = "") -> list:
        """Combined value + quality scoring in a single LLM pass.
        
        Replaces the separate quality scoring phase (Phase 4.5) by producing BOTH
        value score refinements AND quality assessments in one prompt. This eliminates
        an entire LLM pass over all use cases.
        
        The initial value scores from _score_per_domain_parallel are overwritten with
        refined scores that benefit from schema context.
        
        Uses ParallelExecutor (polling loop) instead of as_completed() to keep the
        main thread active -- critical on Databricks where the notebook output buffer
        only flushes when the main thread executes Python code.
        """
        import json
        
        if not use_cases:
            self.logger.info("No use cases to score (combined)")
            return use_cases
        
        self._cached_schema_markdown = self._build_schema_markdown_for_validation(full_schema_details)
        
        _rt = TECHNICAL_CONTEXT["runtime"]
        batch_size = _rt["quality_scoring_batch_size"]
        batches = [use_cases[i:i + batch_size] for i in range(0, len(use_cases), batch_size)]
        
        num_batches = len(batches)
        max_workers = _rt["quality_scoring_max_workers"]
        parallelism = min(num_batches, max_workers)
        if num_batches == 1:
            parallelism = 1
        
        self.logger.info("")
        self.logger.info("=" * 80)
        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Combined Value+Quality Scoring",
                sub_step_name="Started", status="started",
                message=f"Phase 4.5 combined scoring started for {len(use_cases)} UCs"
            )
        except Exception:
            pass
        self.logger.info(f"PHASE 4.5: COMBINED VALUE+QUALITY SCORING ({len(use_cases)} use cases)")
        self.logger.info("=" * 80)
        self.logger.info(f"Batch size: {batch_size}, Batches: {num_batches}, Workers: {parallelism}")
        self.logger.info("=" * 80)
        
        scores_map = {}
        
        import hashlib
        _uc_ids_sorted = sorted(uc.get('No', '') for uc in use_cases)
        _uc_names_sorted = sorted(uc.get('Name', '') for uc in use_cases)
        _fingerprint_src = f"{len(use_cases)}|{'|'.join(_uc_ids_sorted)}|{'|'.join(_uc_names_sorted)}"
        _cache_fingerprint = hashlib.md5(_fingerprint_src.encode('utf-8')).hexdigest()[:12]
        _partial_cache_tag = f"combined_batch_partial_{_cache_fingerprint}"
        
        cached_batch_scores = self.storage_manager.load_scored_use_cases(_partial_cache_tag)
        if cached_batch_scores and isinstance(cached_batch_scores, dict):
            scores_map.update(cached_batch_scores)
            self.logger.info(f"Loaded {len(cached_batch_scores)} partially-scored use cases from disk cache (fingerprint: {_cache_fingerprint})")
        
        str_goals = ", ".join(strategic_goals) if strategic_goals else ""
        str_priorities = ", ".join(business_priorities) if business_priorities else ""
        
        def _score_combined_batch(batch_label: str, uc_batch: list) -> dict:
            if _is_thread_cancelled():
                raise Exception(f"[CombinedScore_{batch_label}] Aborted: parent task cancelled/timed out")
            use_cases_markdown = self._build_use_cases_markdown_for_validation(uc_batch)
            prompt_vars = {
                "business_context": business_context or self._get_business_context_fallback(),
                "industry": industry or "General",
                "strategic_goals": str_goals,
                "business_priorities": str_priorities,
                "strategic_initiative": strategic_initiative or "",
                "value_chain": value_chain or "",
                "revenue_model": revenue_model or "",
                "schema_details": self._cached_schema_markdown,
                "use_cases_to_validate": use_cases_markdown,
                "generation_instructions_section": self._get_generation_instruction_for_prompt("COMBINED_VALUE_QUALITY_SCORE_PROMPT"),
            }
            combined_timeout = max(300, self.llm_timeout_seconds + len(uc_batch) * 10)
            response_raw = self.ai_agent.run_worker(
                step_name=f"CombinedScore_{batch_label}",
                worker_prompt_path="COMBINED_VALUE_QUALITY_SCORE_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=combined_timeout
            )
            return self._parse_combined_score_response(response_raw, uc_batch)
        
        def process_combined_batch(batch_idx: int, batch: list) -> dict:
            batch_results = {}
            
            already_scored = [uc for uc in batch if uc.get('No', 'UNKNOWN') in scores_map]
            if len(already_scored) == len(batch):
                self.logger.info(f"[Batch {batch_idx}/{num_batches}] All {len(batch)} use cases already scored (disk cache)")
                return {uc.get('No', 'UNKNOWN'): scores_map[uc.get('No', 'UNKNOWN')] for uc in batch}
            
            self.logger.info(f"[Batch {batch_idx}/{num_batches}] Combined scoring {len(batch)} use cases...")
            
            try:
                batch_results = _score_combined_batch(f"B{batch_idx}", batch)
                self.logger.info(f"[Batch {batch_idx}/{num_batches}] Complete: {len(batch_results)} scored")
                return batch_results
            except Exception as first_err:
                self.logger.warning(
                    f"[Batch {batch_idx}/{num_batches}] Failed ({str(first_err)[:120]}). Splitting..."
                )
            
            if _is_thread_cancelled():
                self.logger.warning(f"[Batch {batch_idx}/{num_batches}] Cancelled before sub-batch retry")
                for uc in batch:
                    uc_id = uc.get('No', 'UNKNOWN')
                    if uc_id not in batch_results:
                        batch_results[uc_id] = self._default_combined_scores('Cancelled before retry')
                return batch_results
            
            mid = len(batch) // 2
            sub_batches = [batch[:mid], batch[mid:]] if len(batch) > 1 else [batch]
            for sub_idx, sub in enumerate(sub_batches, 1):
                if not sub:
                    continue
                if _is_thread_cancelled():
                    self.logger.warning(f"[Batch {batch_idx} sub-{sub_idx}] Cancelled - assigning defaults")
                    for uc in sub:
                        uc_id = uc.get('No', 'UNKNOWN')
                        batch_results[uc_id] = self._default_combined_scores('Cancelled during retry')
                    continue
                try:
                    sub_results = _score_combined_batch(f"B{batch_idx}_S{sub_idx}", sub)
                    batch_results.update(sub_results)
                except Exception as sub_err:
                    self.logger.error(f"[Batch {batch_idx} sub-{sub_idx}] Retry failed: {str(sub_err)[:120]}")
                    for uc in sub:
                        uc_id = uc.get('No', 'UNKNOWN')
                        batch_results[uc_id] = self._default_combined_scores(f'Retry failed: {str(sub_err)[:80]}')
            
            return batch_results
        
        per_batch_timeout = self.llm_timeout_seconds * 4
        total_timeout = per_batch_timeout * max(1, ((num_batches + parallelism - 1) // parallelism))
        parallel_succeeded = False
        
        if num_batches >= 2:
            try:
                tasks = [(process_combined_batch, (idx, batch)) for idx, batch in enumerate(batches, 1)]
                
                results = ParallelExecutor.execute_parallel(
                    tasks=tasks,
                    max_workers=min(parallelism, num_batches),
                    task_name="CombinedScore",
                    logger=self.logger,
                    timeout_per_task=per_batch_timeout,
                    total_timeout=total_timeout,
                    thread_name_prefix="CombinedScore",
                    return_exceptions=True
                )
                
                for i, result in enumerate(results):
                    batch_idx = i + 1
                    if isinstance(result, (TimeoutError, concurrent.futures.TimeoutError)):
                        self.logger.error(f"Combined batch {batch_idx} timed out")
                        for uc in batches[batch_idx - 1]:
                            uc_id = uc.get('No', 'UNKNOWN')
                            if uc_id not in scores_map:
                                scores_map[uc_id] = self._default_combined_scores('Batch timeout')
                    elif isinstance(result, Exception):
                        self.logger.error(f"Combined batch {batch_idx} error: {get_clean_error_message(result)}")
                        for uc in batches[batch_idx - 1]:
                            uc_id = uc.get('No', 'UNKNOWN')
                            scores_map[uc_id] = self._default_combined_scores(str(result)[:80])
                    elif isinstance(result, dict):
                        scores_map.update(result)
                    else:
                        self.logger.warning(f"Combined batch {batch_idx} returned unexpected type: {type(result)}")
                
                for uc in use_cases:
                    uc_id = uc.get('No', 'UNKNOWN')
                    if uc_id not in scores_map:
                        scores_map[uc_id] = self._default_combined_scores('Not scored')
                
                self.storage_manager.save_scored_use_cases(scores_map, _partial_cache_tag)
                parallel_succeeded = True
                
            except Exception as e:
                self.logger.error(f"Combined scoring parallel execution failed: {get_clean_error_message(e)}")
            
            gc.collect()
        
        if not parallel_succeeded:
            self.logger.info(f"SEQUENTIAL FALLBACK: Processing {num_batches} combined batches...")
            for idx, batch in enumerate(batches, 1):
                try:
                    batch_results = process_combined_batch(idx, batch)
                    scores_map.update(batch_results)
                    self.storage_manager.save_scored_use_cases(scores_map, _partial_cache_tag)
                except Exception as e:
                    self.logger.error(f"[{idx}/{num_batches}] Sequential batch failed: {get_clean_error_message(e)}")
                    for uc in batch:
                        uc_id = uc.get('No', 'UNKNOWN')
                        scores_map[uc_id] = self._default_combined_scores(str(e)[:80])
        
        for uc in use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id in scores_map:
                s = scores_map[uc_id]
                uc['Strategic Alignment'] = s.get('strategic_alignment', uc.get('Strategic Alignment'))
                uc['Return on Investment'] = s.get('roi', uc.get('Return on Investment'))
                uc['Reusability'] = s.get('reusability', uc.get('Reusability'))
                uc['Time to Value'] = s.get('time_to_value', uc.get('Time to Value'))
                uc['Data Availability'] = s.get('data_availability', uc.get('Data Availability'))
                uc['Data Accessibility'] = s.get('data_accessibility', uc.get('Data Accessibility'))
                uc['Architecture Fitness'] = s.get('architecture_fitness', uc.get('Architecture Fitness'))
                uc['Team Skills'] = s.get('team_skills', uc.get('Team Skills'))
                uc['Domain Knowledge'] = s.get('domain_knowledge', uc.get('Domain Knowledge'))
                uc['People Allocation'] = s.get('people_allocation', uc.get('People Allocation'))
                uc['Budget Allocation'] = s.get('budget_allocation', uc.get('Budget Allocation'))
                uc['Time to Production'] = s.get('time_to_production', uc.get('Time to Production'))
                uc['Value'] = s.get('value_score', uc.get('Value'))
                uc['Feasibility'] = s.get('feasibility_score', uc.get('Feasibility'))
                uc['Priority Score'] = s.get('priority_score', uc.get('Priority Score'))
                uc['Business Priority Alignment'] = s.get('business_priority_alignment', uc.get('Business Priority Alignment', ''))
                uc['Strategic Goals Alignment'] = s.get('strategic_goals_alignment', uc.get('Strategic Goals Alignment', ''))
                uc['Justification'] = s.get('justification', uc.get('Justification', ''))
                uc['Quality'] = s.get('quality_label', 'Medium')
                uc['Quality Score'] = s.get('quality_score', 3.0)
                uc['Quality Summary'] = self._sanitize_quality_summary(s.get('quality_summary', ''))
            else:
                uc['Quality'] = 'Medium'
                uc['Quality Score'] = 3.0
                uc['Quality Summary'] = 'No combined score received - default assigned'
        
        quality_distribution = {}
        for uc in use_cases:
            label = uc.get('Quality', 'Medium')
            quality_distribution[label] = quality_distribution.get(label, 0) + 1
        
        self.logger.info("")
        self.logger.info("=" * 80)
        self.logger.info("PHASE 4.5 COMPLETE: COMBINED VALUE+QUALITY SCORING")
        self.logger.info("=" * 80)
        self.logger.info(f"Total scored: {len(use_cases)} use cases")
        for label in ['Ultra High', 'Very High', 'High', 'Medium', 'Low', 'Very Low', 'Ultra Low']:
            count = quality_distribution.get(label, 0)
            if count > 0:
                self.logger.info(f"   {label}: {count} use cases")
        self.logger.info("=" * 80)
        
        del scores_map, batches
        self._cached_schema_markdown = None
        _force_gc(self.logger, "after combined scoring")
        
        return use_cases
    
    def _score_best_of_best(self, use_cases: list, full_schema_details: list,
                            business_context: str = "", industry: str = "",
                            strategic_goals: list = None, business_priorities: list = None,
                            strategic_initiative: str = "", value_chain: str = "",
                            revenue_model: str = "") -> list:
        """Best-of-Best ranking pass (v0.7.2).

        Issues one LLM call per batch of UCs using BEST_OF_BEST_SCORING_PROMPT.
        For every UC the LLM returns:
          - 19 per-gate 0-10 scores (D1-D19)
          - idea_theme label (2-4 words)
          - short justification

        This method mutates each UC dict with:
          bob_scores (dict of D1..D19 -> int),
          idea_theme (str),
          bob_tier1_score, bob_tier2_score, bob_tier3_score (float 0-10),
          bob_score (float 0-10, tier-weighted),
          bob_hard_veto_gate (str or None — gate name if any D<3 capped total),
          bob_justification (str).

        Returns the same list (mutated). Safe no-op on empty input.
        """
        import json as _json
        if not use_cases:
            return use_cases

        _rt = TECHNICAL_CONTEXT["runtime"]
        batch_size = _rt.get("bob_scoring_batch_size", _rt["quality_scoring_batch_size"])
        batches = [use_cases[i:i + batch_size] for i in range(0, len(use_cases), batch_size)]
        num_batches = len(batches)
        max_workers = min(num_batches, _rt["quality_scoring_max_workers"])
        target_max = _rt.get("target_max_use_cases", 100)

        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Best of Best Ranking",
                sub_step_name="Started", status="started",
                message=f"BoB ranking started for {len(use_cases)} UCs",
                result_json={"ucs": len(use_cases), "batches": num_batches, "target_max": target_max}
            )
        except Exception:
            pass
        self.logger.info("=" * 80)
        self.logger.info(f"🏆 BEST-OF-BEST RANKING ({len(use_cases)} UCs across {num_batches} batches)")
        self.logger.info("=" * 80)

        if not hasattr(self, "_cached_schema_markdown") or self._cached_schema_markdown is None:
            self._cached_schema_markdown = self._build_schema_markdown_for_validation(full_schema_details)

        str_goals = ", ".join(strategic_goals) if strategic_goals else ""
        str_priorities = ", ".join(business_priorities) if business_priorities else ""
        gen_instr = self._get_generation_instruction_for_prompt("BEST_OF_BEST_SCORING_PROMPT")

        def _score_bob_batch(batch_label: str, uc_batch: list) -> list:
            if _is_thread_cancelled():
                raise Exception(f"[BoB_{batch_label}] Aborted: parent task cancelled/timed out")
            use_cases_markdown = self._build_use_cases_markdown_for_validation(uc_batch)
            prompt_vars = {
                "business_context": business_context or self._get_business_context_fallback(),
                "industry": industry or "General",
                "strategic_goals": str_goals,
                "business_priorities": str_priorities,
                "use_cases_to_validate": use_cases_markdown,
                "target_max": target_max,
                "generation_instructions_section": gen_instr,
            }
            bob_timeout = max(300, self.llm_timeout_seconds + len(uc_batch) * 10)
            response_raw = self.ai_agent.run_worker(
                step_name=f"BoB_{batch_label}",
                worker_prompt_path="BEST_OF_BEST_SCORING_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=bob_timeout,
            )
            return self._parse_bob_response(response_raw, uc_batch)

        # Flatten to {No -> bob record}
        bob_by_id = {}
        from concurrent.futures import ThreadPoolExecutor, as_completed
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            fut_map = {ex.submit(_score_bob_batch, f"B{idx+1}", batch): idx for idx, batch in enumerate(batches)}
            for fut in as_completed(fut_map):
                idx = fut_map[fut]
                try:
                    batch_records = fut.result()
                    for rec in batch_records:
                        _no = rec.get("No")
                        if _no:
                            bob_by_id[_no] = rec
                    self.logger.info(f"  [BoB Batch {idx+1}/{num_batches}] scored {len(batch_records)} UCs")
                except Exception as be:
                    self.logger.warning(f"  [BoB Batch {idx+1}/{num_batches}] failed: {get_clean_error_message(be)}")

        # Compose + attach scores to each UC
        _attached = 0
        for uc in use_cases:
            _no = uc.get("No")
            rec = bob_by_id.get(_no) if _no else None
            if not rec:
                # Fallback — mark as middle-ground so UC is neither favored nor dropped irrationally
                uc["bob_scores"] = {f"D{i}": 5 for i in range(1, 20)}
                uc["idea_theme"] = uc.get("subdomain") or uc.get("business_domain") or "Unclassified"
                uc["bob_tier1_score"] = 5.0
                uc["bob_tier2_score"] = 5.0
                uc["bob_tier3_score"] = 5.0
                uc["bob_score"] = 5.0
                uc["bob_hard_veto_gate"] = None
                uc["bob_justification"] = "BoB missing — fallback mid-scores"
                continue
            scores = rec.get("bob_scores") or {}
            uc["bob_scores"] = scores
            uc["idea_theme"] = rec.get("idea_theme") or uc.get("subdomain") or "Unclassified"
            uc["bob_justification"] = rec.get("bob_justification_short", "")
            composed = self._compose_bob_score(scores)
            uc["bob_tier1_score"] = composed["tier1"]
            uc["bob_tier2_score"] = composed["tier2"]
            uc["bob_tier3_score"] = composed["tier3"]
            uc["bob_score"] = composed["total"]
            uc["bob_hard_veto_gate"] = composed["hard_veto_gate"]
            _attached += 1

        try:
            self._emit_pipeline_status(
                stage_name="Use Case Prioritization", step_name="Best of Best Ranking",
                sub_step_name="Complete", status="ended_success",
                message=f"BoB ranking scored {_attached}/{len(use_cases)} UCs",
                result_json={"scored": _attached, "total": len(use_cases)}
            )
        except Exception:
            pass
        self.logger.info(f"🏆 BoB ranking complete: {_attached}/{len(use_cases)} UCs scored")
        return use_cases

    def _parse_bob_response(self, response_raw: str, uc_batch: list) -> list:
        """Parse the BoB JSON array response. Tolerant to wrappers / fenced code."""
        import json as _json
        if not response_raw:
            return []
        text = response_raw.strip()
        # Strip fenced code if present
        if text.startswith("```"):
            nl = text.find("\n")
            if nl > 0:
                text = text[nl+1:]
            if text.endswith("```"):
                text = text[:-3].rstrip()
        # Find first `[` and last `]`
        lb = text.find("[")
        rb = text.rfind("]")
        if lb >= 0 and rb > lb:
            text = text[lb:rb+1]
        try:
            arr = _json.loads(text)
        except Exception as e:
            self.logger.warning(f"BoB parse failed: {get_clean_error_message(e)[:200]}; response head: {response_raw[:200]}")
            return []
        if not isinstance(arr, list):
            return []
        out = []
        for item in arr:
            if not isinstance(item, dict):
                continue
            scores_obj = item.get("bob_scores") or item.get("scores") or {}
            # Coerce D1..D19 into integers; default 5 if missing/bad
            coerced = {}
            for gi in range(1, 20):
                k = f"D{gi}"
                v = scores_obj.get(k)
                try:
                    iv = int(round(float(v)))
                except Exception:
                    iv = 5
                iv = max(0, min(10, iv))
                coerced[k] = iv
            out.append({
                "No": item.get("No") or item.get("id") or "",
                "bob_scores": coerced,
                "idea_theme": (item.get("idea_theme") or "").strip() or "Unclassified",
                "bob_justification_short": (item.get("bob_justification_short") or "").strip(),
            })
        return out

    def _compose_bob_score(self, scores_by_gate: dict) -> dict:
        """Compose 19 per-gate 0-10 scores into a tier-weighted total with hard-veto floor.

        Tier 1 (technical): D1..D11 — weight 0.30
        Tier 2 (business):  D12..D14 — weight 0.35
        Tier 3 (PBA):       D15..D19 — weight 0.35
        Hard-veto floor: if ANY gate < 3, total capped at 2.0 (Low — will be
          dropped by the Balanced filter if it wasn't already).
        """
        def _avg(gs, lo, hi):
            vals = [float(gs.get(f"D{i}", 0) or 0) for i in range(lo, hi+1)]
            return round(sum(vals) / max(1, len(vals)), 3)
        t1 = _avg(scores_by_gate, 1, 11)
        t2 = _avg(scores_by_gate, 12, 14)
        t3 = _avg(scores_by_gate, 15, 19)
        total = 0.30*t1 + 0.35*t2 + 0.35*t3
        # Hard-veto floor
        hv_gate = None
        for i in range(1, 20):
            v = float(scores_by_gate.get(f"D{i}", 0) or 0)
            if v < 3.0:
                hv_gate = f"D{i}"
                break
        if hv_gate is not None:
            total = min(total, 2.0)
        return {"tier1": t1, "tier2": t2, "tier3": t3, "total": round(total, 3), "hard_veto_gate": hv_gate}

    def _stratified_trim_by_theme(self, scored_use_cases: list, target_max: int,
                                   theme_max_share: float = 0.15) -> list:
        """Deterministic stratified trim preserving idea coverage.

        Algorithm:
          1. Group UCs by idea_theme.
          2. FLOOR: every theme keeps at least 1 UC (coverage guarantee).
          3. EXTRA budget = target_max - num_themes, distributed proportionally
             to theme size.
          4. CAP: no theme exceeds floor(target_max * theme_max_share) UCs.
          5. Within each theme, rank by bob_score DESC and take top-K.

        Returns trimmed list. If input count <= target_max, returns input unchanged.

        v0.7.3: Pre-filter hard-veto UCs (bob_hard_veto_gate != None) BEFORE
        grouping by theme. Any theme that loses all UCs is legitimately
        eliminated. Prevents the 1-per-theme coverage floor from saving a UC
        that failed a hard gate.
        """
        if not scored_use_cases:
            return scored_use_cases

        # v0.7.3: hard-veto pre-filter
        _n_in = len(scored_use_cases)
        survivors = [uc for uc in scored_use_cases if uc.get("bob_hard_veto_gate") in (None, "", "None")]
        _dropped_hv = _n_in - len(survivors)
        if _dropped_hv:
            self.logger.info(
                f"Hard-veto pre-filter: dropped {_dropped_hv} UC(s) with bob_hard_veto_gate set "
                f"({_n_in} -> {len(survivors)}) before stratified trim"
            )
        scored_use_cases = survivors

        if not scored_use_cases or len(scored_use_cases) <= target_max:
            return scored_use_cases

        from collections import defaultdict
        groups = defaultdict(list)
        for uc in scored_use_cases:
            theme = uc.get("idea_theme") or "Unclassified"
            groups[theme].append(uc)
        num_themes = len(groups)
        cap_per_theme = max(1, int(target_max * theme_max_share))

        # Handle degenerate case: more themes than target_max
        if num_themes >= target_max:
            self.logger.info(
                f"Stratified trim: num_themes={num_themes} >= target_max={target_max}. "
                f"Picking top-{target_max} themes by total bob_score, 1 UC per theme."
            )
            ranked_themes = sorted(
                groups.items(),
                key=lambda kv: max((uc.get("bob_score", 0) or 0) for uc in kv[1]),
                reverse=True,
            )[:target_max]
            out = []
            for theme, lst in ranked_themes:
                best = sorted(lst, key=lambda u: u.get("bob_score", 0) or 0, reverse=True)[0]
                out.append(best)
            return out

        # Proportional allocation
        total_ucs = sum(len(v) for v in groups.values())
        extra_budget = max(0, target_max - num_themes)
        quotas = {}
        for theme, lst in groups.items():
            proportional = round((len(lst) / total_ucs) * extra_budget) if total_ucs else 0
            q = min(len(lst), 1 + proportional, cap_per_theme)
            quotas[theme] = max(1, q)

        # Budget reconciliation (rounding may over/undershoot target_max)
        total_alloc = sum(quotas.values())
        # If over target, trim from the themes with the largest quotas (most redundancy)
        while total_alloc > target_max:
            biggest = max(quotas, key=lambda k: quotas[k])
            if quotas[biggest] <= 1:
                break
            quotas[biggest] -= 1
            total_alloc -= 1
        # If under target (rounding shortfall), give extras to largest available themes
        while total_alloc < target_max:
            candidates = [t for t, q in quotas.items() if q < min(len(groups[t]), cap_per_theme)]
            if not candidates:
                break
            pick = max(candidates, key=lambda k: len(groups[k]))
            quotas[pick] += 1
            total_alloc += 1

        # Apply within-theme ranking
        out = []
        for theme, lst in groups.items():
            q = quotas.get(theme, 1)
            ranked = sorted(lst, key=lambda u: u.get("bob_score", 0) or 0, reverse=True)
            out.extend(ranked[:q])

        self.logger.info(
            f"Stratified trim: {len(scored_use_cases)} -> {len(out)} UCs across {num_themes} themes "
            f"(cap_per_theme={cap_per_theme}, largest theme={max(quotas.items(), key=lambda kv: kv[1])})"
        )
        return out

    def _default_combined_scores(self, reason: str) -> dict:
        return {
            'quality_label': 'Medium',
            'quality_summary': f'Combined scoring default - {reason}',
            'strategic_alignment': None,
            'roi': None,
            'priority_score': None,
        }
    
    def _parse_combined_score_response(self, response_raw: str, original_use_cases: list) -> dict:
        """Parse combined value+quality JSON response into {uc_id: scores_dict}."""
        import json
        import re
        
        result = {}
        
        if not response_raw:
            self.logger.warning("Empty combined score response")
            for uc in original_use_cases:
                result[uc.get('No', 'UNKNOWN')] = self._default_combined_scores('Empty response')
            return result
        
        cleaned = response_raw.strip()
        if cleaned.startswith("```"):
            cleaned = re.sub(r'^```\w*\n?', '', cleaned)
            cleaned = re.sub(r'\n?```$', '', cleaned)
        
        json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
        if json_match:
            cleaned = json_match.group(0)
        
        try:
            parsed = json.loads(cleaned)
            if not isinstance(parsed, list):
                parsed = [parsed]
            
            for item in parsed:
                uc_id = item.get('use_case_id', '')
                if not uc_id:
                    continue
                
                def safe_float(val, default=None):
                    if val is None:
                        return default
                    try:
                        return round(float(val), 2)
                    except (ValueError, TypeError):
                        return default
                
                result[uc_id] = {
                    'strategic_alignment': safe_float(item.get('strategic_alignment')),
                    'roi': safe_float(item.get('roi')),
                    'reusability': safe_float(item.get('reusability')),
                    'time_to_value': safe_float(item.get('time_to_value')),
                    'data_availability': safe_float(item.get('data_availability')),
                    'data_accessibility': safe_float(item.get('data_accessibility')),
                    'architecture_fitness': safe_float(item.get('architecture_fitness')),
                    'team_skills': safe_float(item.get('team_skills')),
                    'domain_knowledge': safe_float(item.get('domain_knowledge')),
                    'people_allocation': safe_float(item.get('people_allocation')),
                    'budget_allocation': safe_float(item.get('budget_allocation')),
                    'time_to_production': safe_float(item.get('time_to_production')),
                    'value_score': safe_float(item.get('value_score')),
                    'feasibility_score': safe_float(item.get('feasibility_score')),
                    'priority_score': safe_float(item.get('priority_score')),
                    'business_priority_alignment': item.get('business_priority_alignment', ''),
                    'strategic_goals_alignment': item.get('strategic_goals_alignment', ''),
                    'justification': item.get('justification', ''),
                    'quality_score': safe_float(item.get('quality_score')),
                    'quality_label': item.get('quality_label', 'Medium'),
                    'quality_summary': self._sanitize_quality_summary(item.get('quality_summary', '')),
                    'strengths': item.get('strengths', ''),
                    'weaknesses': item.get('weaknesses', ''),
                }
            
            self.logger.info(f"Parsed combined scores for {len(result)} use cases")
        except json.JSONDecodeError as e:
            self.logger.error(f"Failed to parse combined score JSON: {get_clean_error_message(e)}")
            for uc in original_use_cases:
                result[uc.get('No', 'UNKNOWN')] = self._default_combined_scores(f'JSON parse error: {str(e)[:60]}')
        
        for uc in original_use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id not in result:
                result[uc_id] = self._default_combined_scores('Missing from LLM response')
        
        return result

    def _parse_quality_score_response(self, response_raw: str, original_use_cases: list) -> list:
        """
        Parse quality scoring response from LLM with robust error handling.
        Attempts multiple parsing strategies before falling back to defaults.
        """
        import json
        import re
        
        results = []
        
        # Log raw response length and preview for debugging
        if response_raw:
            self.logger.debug(f"Quality score response: {len(response_raw)} chars")
            self.logger.debug(f"Response preview (first 500 chars): {response_raw[:500]}...")
        else:
            self.logger.warning("Empty quality score response received")
            
        # Strategy 1: Try to find and parse JSON array
        try:
            json_match = re.search(r'\[[\s\S]*\]', response_raw)
            if json_match:
                json_str = json_match.group()
                results = json.loads(json_str)
                if results:
                    self.logger.debug(f"✓ Parsed {len(results)} quality scores (Strategy 1: JSON array)")
                    return results
        except json.JSONDecodeError as e:
            self.logger.debug(f"Strategy 1 (JSON array) failed: {get_clean_error_message(e, max_chars=2000)}")
        
        # Strategy 2: Try to fix common JSON issues and parse
        try:
            json_match = re.search(r'\[[\s\S]*\]', response_raw)
            if json_match:
                json_str = json_match.group()
                # Fix common issues: trailing commas, unescaped quotes
                json_str = re.sub(r',\s*]', ']', json_str)  # Remove trailing commas before ]
                json_str = re.sub(r',\s*}', '}', json_str)  # Remove trailing commas before }
                results = json.loads(json_str)
                if results:
                    self.logger.debug(f"✓ Parsed {len(results)} quality scores (Strategy 2: Fixed JSON)")
                    return results
        except json.JSONDecodeError as e:
            self.logger.debug(f"Strategy 2 (Fixed JSON) failed: {get_clean_error_message(e, max_chars=2000)}")
        
        # Strategy 3: Parse individual JSON objects using brace counting (handles nested objects)
        try:
            # Find all complete JSON objects by counting braces
            i = 0
            while i < len(response_raw):
                if response_raw[i] == '{':
                    # Found start of object, find the matching close brace
                    brace_count = 1
                    j = i + 1
                    while j < len(response_raw) and brace_count > 0:
                        if response_raw[j] == '{':
                            brace_count += 1
                        elif response_raw[j] == '}':
                            brace_count -= 1
                        j += 1
                    
                    if brace_count == 0:
                        obj_str = response_raw[i:j]
                        if '"use_case_id"' in obj_str or '"quality_label"' in obj_str:
                            try:
                                # Clean common issues
                                obj_str_clean = re.sub(r',\s*}', '}', obj_str)
                                obj_str_clean = re.sub(r',\s*]', ']', obj_str_clean)
                                obj = json.loads(obj_str_clean)
                                if 'use_case_id' in obj or 'quality_label' in obj:
                                    results.append(obj)
                            except json.JSONDecodeError:
                                pass
                        i = j
                    else:
                        i += 1
                else:
                    i += 1
            
            if results:
                self.logger.debug(f"✓ Parsed {len(results)} quality scores (Strategy 3: Brace counting)")
                return results
        except Exception as e:
            self.logger.debug(f"Strategy 3 (Brace counting) failed: {get_clean_error_message(e, max_chars=2000)}")
        
        # Strategy 4: Extract quality labels using regex patterns
        if not results:
            self.logger.debug("Trying Strategy 4: Regex extraction...")
            for uc in original_use_cases:
                uc_id = uc.get('No', 'UNKNOWN')
                uc_name = uc.get('Name', 'Unknown')
                
                # Try to find quality label for this use case in the response
                quality_label = 'Medium'  # Default
                
                # Look for patterns like: "use_case_id": "N01-AI01", ... "quality_label": "High"
                id_pattern = rf'["\']use_case_id["\']\s*:\s*["\']?{re.escape(uc_id)}["\']?'
                id_match = re.search(id_pattern, response_raw)
                
                if id_match:
                    # Find quality_label near this ID
                    search_start = max(0, id_match.start() - 200)
                    search_end = min(len(response_raw), id_match.end() + 500)
                    context = response_raw[search_start:search_end]
                    
                    label_match = re.search(r'["\']quality_label["\']\s*:\s*["\']?(Ultra High|Very High|High|Medium|Low|Very Low|Ultra Low)["\']?', context)
                    if label_match:
                        quality_label = label_match.group(1)
                
                results.append({
                    'use_case_id': uc_id,
                    'use_case_name': uc_name,
                    'quality_score': {'Ultra High': 5.0, 'Very High': 4.5, 'High': 4.0, 'Medium': 3.0, 'Low': 2.0, 'Very Low': 1.5, 'Ultra Low': 1.0}.get(quality_label, 3.0),
                    'quality_label': quality_label,
                    'quality_summary': f'Extracted via regex (parsing issues)',
                    'strengths': [],
                    'weaknesses': ['Full assessment not available due to parsing issues']
                })
            
            if results:
                extracted_count = sum(1 for r in results if r.get('quality_label') != 'Medium')
                self.logger.warning(f"⚠️ JSON parsing failed - extracted {extracted_count}/{len(results)} quality labels via regex")
                return results
        
        # Final fallback: Assign defaults
        self.logger.warning("❌ All parsing strategies failed - assigning default 'Medium' quality")
        for uc in original_use_cases:
            results.append({
                'use_case_id': uc.get('No', 'UNKNOWN'),
                'use_case_name': uc.get('Name', 'Unknown'),
                'quality_score': 3.0,
                'quality_label': 'Medium',
                'quality_summary': 'Default - all parsing strategies failed',
                'strengths': [],
                'weaknesses': ['Quality assessment not available']
            })
        
        return results

    @staticmethod
    def _sanitize_quality_summary(quality_summary: str) -> str:
        """
        Validates and sanitizes quality_summary values from LLM output.
        Detects when the LLM produced a hallucination-check boilerplate instead of
        a proper data-to-value chain explanation, and flags it so users see a
        meaningful indicator rather than misleading filler text.
        """
        if not quality_summary or not isinstance(quality_summary, str):
            return ''
        
        summary = quality_summary.strip()
        
        bad_prefixes = [
            'NO HALLUCINATION',
            'HALLUCINATION DETECTED',
            'All columns verified',
            'Brief summary -',
            'Brief summary:',
        ]
        
        bad_phrases = [
            'all columns verified to exist',
            'strong causal signals present',
            'unique business problem',
            'strong causal signals. unique business problem',
        ]
        
        summary_lower = summary.lower()
        
        is_boilerplate = False
        for prefix in bad_prefixes:
            if summary_lower.startswith(prefix.lower()):
                is_boilerplate = True
                break
        
        if not is_boilerplate:
            for phrase in bad_phrases:
                if summary_lower == phrase:
                    is_boilerplate = True
                    break
        
        if not is_boilerplate and len(summary) < 80:
            has_column_ref = '`' in summary or 'column' in summary_lower
            has_operation = any(op in summary_lower for op in [
                'aggregat', 'join', 'comput', 'calculat', 'compar', 'datediff',
                'avg', 'sum', 'count', 'percentile', 'group by', 'correlat',
                'deriv', 'transform', 'filter', 'partition', 'window',
            ])
            if not has_column_ref and not has_operation:
                is_boilerplate = True
        
        if is_boilerplate:
            return f'[Insufficient detail] {summary}'
        
        return summary

    def _validate_use_case_data_sufficiency_legacy(self, use_cases: list, full_schema_details: list, 
                                            business_context: str = "", industry: str = "") -> tuple:
        """
        LEGACY: This function is kept for backwards compatibility but is no longer used.
        Quality scoring is now handled by _score_use_case_data_quality which runs in parallel with VALUE scoring.
        """
        return (use_cases, [], {"total": len(use_cases), "approved": len(use_cases), "rejected": 0, "details": []})
    
    def _build_schema_markdown_for_validation(self, full_schema_details: list) -> str:
        """Build a concise schema markdown for validation prompt. Uses disk cache if available."""
        cached = self.storage_manager.load_schema_markdown()
        if cached:
            return cached
        
        if not full_schema_details:
            return "No schema details available."
        
        tables_dict = {}
        for detail in full_schema_details:
            (catalog, schema, table, column, dtype, comment) = detail
            table_fqn = f"{catalog}.{schema}.{table}"
            if table_fqn not in tables_dict:
                tables_dict[table_fqn] = []
            tables_dict[table_fqn].append({
                'column': column,
                'type': dtype,
                'comment': comment or ''
            })
        
        md_parts = ["## Available Tables and Columns\n\n"]
        for table_fqn, columns in sorted(tables_dict.items()):
            md_parts.append(f"### `{table_fqn}`\n")
            md_parts.append("| Column | Type | Description |\n")
            md_parts.append("|--------|------|-------------|\n")
            for col in columns[:30]:
                comment_clean = col['comment'].replace('|', '-').replace('\n', ' ')[:50]
                md_parts.append(f"| {col['column']} | {col['type']} | {comment_clean} |\n")
            if len(columns) > 30:
                md_parts.append(f"| ... | ... | ({len(columns) - 30} more columns) |\n")
            md_parts.append("\n")
        
        result = ''.join(md_parts)
        self.storage_manager.save_schema_markdown(result)
        return result
    
    def _build_use_cases_markdown_for_validation(self, use_cases: list) -> str:
        """Build use cases markdown for validation prompt."""
        md_parts = ["## Use Cases to Validate\n\n"]
        
        for uc in use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            uc_name = uc.get('Name', 'Unknown')
            uc_type = uc.get('type', 'N/A')
            uc_statement = uc.get('Statement', 'N/A')
            uc_solution = uc.get('Solution', 'N/A')
            uc_value = uc.get('Business Value', 'N/A')
            uc_tables = uc.get('Tables Involved', 'N/A')
            uc_tech_design = uc.get('High Level Design', 'N/A')
            uc_analytics = uc.get('Analytics Technique', 'N/A')
            
            md_parts.append(f"### Use Case: {uc_id} - {uc_name}\n\n")
            md_parts.append(f"**Type:** {uc_type}\n\n")
            md_parts.append(f"**Analytics Technique:** {uc_analytics}\n\n")
            md_parts.append(f"**Problem Statement:**\n{uc_statement}\n\n")
            md_parts.append(f"**Proposed Solution:**\n{uc_solution}\n\n")
            md_parts.append(f"**Business Value:**\n{uc_value}\n\n")
            md_parts.append(f"**Tables Involved:**\n`{uc_tables}`\n\n")
            md_parts.append(f"**High Level Design:**\n{uc_tech_design}\n\n")
            md_parts.append("---\n\n")
        
        return ''.join(md_parts)
    
    def _parse_validation_response(self, response_raw: str, original_use_cases: list) -> list:
        """Parse the validation response from LLM."""
        import json
        
        results = []
        
        try:
            json_clean = response_raw.strip()
            if json_clean.startswith('```'):
                json_clean = re.sub(r'^```[a-z]*\n', '', json_clean)
                json_clean = re.sub(r'\n```$', '', json_clean)
            
            json_start = json_clean.find('[')
            json_end = json_clean.rfind(']') + 1
            if json_start >= 0 and json_end > json_start:
                json_clean = json_clean[json_start:json_end]
            
            parsed = json.loads(json_clean)
            
            if isinstance(parsed, list):
                results = parsed
            else:
                self.logger.warning("Validation response is not a list, treating all as approved with caution")
                for uc in original_use_cases:
                    results.append({
                        'use_case_id': uc.get('No', 'UNKNOWN'),
                        'use_case_name': uc.get('Name', 'Unknown'),
                        'validation_status': 'REJECTED',
                        'confidence_score': 0.0,
                        'fatal_flaws_found': ['VALIDATION_PARSE_DEFAULT_REJECT'],
                        'validation_summary': 'Validation parsing failed - defaulting to REJECTED per Phase B1',
                        'recommendation': 'BLOCKING: fix LLM response format and rerun',
                        'validation_error': 'JSON_PARSE_FAILED'
                    })
                    
        except json.JSONDecodeError as e:
            self.logger.error(f"Failed to parse validation JSON: {get_clean_error_message(e)}")
            self.logger.debug(f"Raw response: {response_raw[:500]}...")
            
            for uc in original_use_cases:
                results.append({
                    'use_case_id': uc.get('No', 'UNKNOWN'),
                    'use_case_name': uc.get('Name', 'Unknown'),
                    'validation_status': 'REJECTED',
                    'confidence_score': 0.0,
                    'fatal_flaws_found': ['VALIDATION_PARSE_DEFAULT_REJECT'],
                    'validation_summary': 'Validation parsing failed - defaulting to REJECTED per Phase B1',
                    'recommendation': 'BLOCKING: fix LLM response format and rerun',
                    'validation_error': 'JSON_PARSE_FAILED'
                })
        
        validated_ids = {r.get('use_case_id') for r in results}
        for uc in original_use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id not in validated_ids:
                results.append({
                    'use_case_id': uc_id,
                    'use_case_name': uc.get('Name', 'Unknown'),
                    'validation_status': 'APPROVED',
                    'confidence_score': 0.5,
                    'fatal_flaws_found': [],
                    'validation_summary': 'Not included in validation response - approved with caution',
                    'recommendation': 'Manual review recommended'
                })
        
        return results

    def _repair_tables_involved(self, tables_str: str, row_dict: dict, log_prefix: str) -> str:
        """
        Detect and repair 'Tables Involved' values corrupted by CSV column shifting.
        When LLM's High Level Design contains unquoted commas, columns shift and 
        Tables Involved receives SQL fragments instead of table FQTNs.
        
        Repair strategy: scan ALL values in the row for catalog.schema.table patterns
        and reconstruct the field from any matches found.
        """
        if not tables_str:
            return tables_str
        
        sql_keywords = {'SELECT', 'FROM', 'WHERE', 'CTE', 'JOIN', 'LIMIT', 'GROUP', 'ORDER', 'HAVING', 'UNION'}
        upper_val = tables_str.upper()
        has_sql = any(kw in upper_val for kw in sql_keywords)
        
        if not has_sql:
            return tables_str
        
        fqtn_pattern = re.compile(r'[`]?(\w+)[`]?\.[`]?(\w+)[`]?\.[`]?(\w+)[`]?')
        found_tables = set()
        
        for key, val in row_dict.items():
            if val and isinstance(val, str):
                for match in fqtn_pattern.finditer(val):
                    cat, sch, tbl = match.group(1), match.group(2), match.group(3)
                    if len(cat) > 1 and len(sch) > 1 and len(tbl) > 1:
                        skip_words = {'select', 'from', 'where', 'join', 'left', 'right', 'inner', 'outer', 
                                     'group', 'order', 'having', 'union', 'case', 'when', 'then', 'else', 'end',
                                     'null', 'true', 'false', 'and', 'not', 'between', 'like', 'exists'}
                        if cat.lower() not in skip_words and sch.lower() not in skip_words and tbl.lower() not in skip_words:
                            found_tables.add(f"{cat}.{sch}.{tbl}")
        
        if found_tables:
            repaired = ", ".join(sorted(found_tables))
            self.logger.debug(f"{log_prefix} Repaired Tables Involved: extracted {len(found_tables)} FQTNs from shifted CSV row")
            return repaired
        
        return tables_str

    def _parse_llm_csv_response(self, llm_response: str, log_prefix: str) -> list:
        self.logger.info(f"{log_prefix} Starting robust 12-column CSV parsing (SQL and scoring metrics will be assigned separately)...")
        parsed_rows = []
        _canonical_header = '"No","Name","usecase","type","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"'
        
        try:
            # Clean response - remove markdown fences if present
            csv_clean = llm_response.strip()
            if csv_clean.startswith('```'):
                csv_clean = re.sub(r'^```[a-z]*\n', '', csv_clean)
                csv_clean = re.sub(r'\n```$', '', csv_clean)
            
            # Find header line (12 columns - Business Domain, Subdomain, SQL, and scoring columns will be calculated in code)
            # Support both quoted and unquoted headers from LLM, with case-insensitive matching
            # Analytics Technique is now generated by LLM as column 5
            # Column 6 MUST be "Statement" (not "Opportunity" or any other name)
            header_pattern_quoted = r'"No","Name","usecase","[Tt]ype","Analytics Technique","Statement","Solution","Business Value","Beneficiary","Sponsor","Tables Involved","High Level Design"(?:\s*,\s*"honesty_score"\s*,\s*"honesty_justification")?'
            header_pattern_unquoted = r'No,Name,usecase,[Tt]ype,Analytics Technique,Statement,Solution,Business Value,Beneficiary,Sponsor,Tables Involved,High Level Design(?:\s*,\s*honesty_score\s*,\s*honesty_justification)?'
            header_match = re.search(header_pattern_quoted, csv_clean, re.IGNORECASE)
            if not header_match:
                header_match = re.search(header_pattern_unquoted, csv_clean, re.IGNORECASE)
            
            # Fallback: Auto-correct common LLM column name variations
            if not header_match:
                _HEADER_CORRECTIONS = [
                    ('"Opportunity"', '"Statement"'),
                    (',Opportunity,', ',Statement,'),
                    ('"Opportunity Statement"', '"Statement"'),
                    (',Opportunity Statement,', ',Statement,'),
                    ('"Problem Statement"', '"Statement"'),
                    (',Problem Statement,', ',Statement,'),
                    ('"Description"', '"Statement"'),
                    (',Description,', ',Statement,'),
                    ('"Use Case Name"', '"Name"'),
                    (',Use Case Name,', ',Name,'),
                    ('"Title"', '"Name"'),
                    (',Title,', ',Name,'),
                    ('"Use Case Type"', '"type"'),
                    (',Use Case Type,', ',type,'),
                    ('"Category"', '"type"'),
                    (',Category,', ',type,'),
                    ('"Type"', '"type"'),
                    ('"Technique"', '"Analytics Technique"'),
                    (',Technique,', ',Analytics Technique,'),
                    ('"Tables"', '"Tables Involved"'),
                    (',Tables,', ',Tables Involved,'),
                    ('"Tables Used"', '"Tables Involved"'),
                    (',Tables Used,', ',Tables Involved,'),
                    ('"Technical Approach"', '"High Level Design"'),
                    (',Technical Approach,', ',High Level Design,'),
                    ('"Design"', '"High Level Design"'),
                ]
                applied_corrections = []
                for old_val, new_val in _HEADER_CORRECTIONS:
                    if old_val in csv_clean:
                        csv_clean = csv_clean.replace(old_val, new_val)
                        applied_corrections.append(f"{old_val} -> {new_val}")
                if applied_corrections:
                    self.logger.info(f"{log_prefix} Auto-corrected {len(applied_corrections)} column name(s): {'; '.join(applied_corrections[:5])}")
                    header_match = re.search(header_pattern_quoted, csv_clean, re.IGNORECASE)
                    if not header_match:
                        header_match = re.search(header_pattern_unquoted, csv_clean, re.IGNORECASE)
            
            # Final fallback: Try simpler pattern if exact match still fails
            # Search ALL matches and validate each one (the first match may be prompt echoes or corrupted data)
            if not header_match:
                simpler_pattern = r'(?:"No"|No)\s*,\s*(?:"Name"|Name)\s*,\s*(?:"[Tt]ype"|[Tt]ype)'
                canonical_col_count = 12
                _CANONICAL_FIELDS = ["No", "Name", "usecase", "type", "Analytics Technique", "Statement",
                                     "Solution", "Business Value", "Beneficiary", "Sponsor",
                                     "Tables Involved", "High Level Design"]
                _REMAP_ALIASES = {
                    "opportunity": "Statement", "problem statement": "Statement",
                    "opportunity statement": "Statement", "description": "Statement",
                    "use case": "Name", "use case name": "Name", "title": "Name",
                    "category": "type", "use case type": "type", "type": "type",
                    "technique": "Analytics Technique", "method": "Analytics Technique",
                    "tables": "Tables Involved", "tables used": "Tables Involved",
                    "technical approach": "High Level Design", "design": "High Level Design",
                    "value": "Business Value", "impact": "Business Value",
                    "improvement": "Statement",
                }
                
                for candidate_match in re.finditer(simpler_pattern, csv_clean, re.IGNORECASE):
                    match_pos = candidate_match.start()
                    line_start = csv_clean.rfind('\n', 0, match_pos) + 1
                    line_end = csv_clean.find('\n', match_pos)
                    if line_end == -1:
                        line_end = len(csv_clean)
                    original_header = csv_clean[line_start:line_end].strip().strip('`')
                    
                    try:
                        parsed_fields = next(csv.reader(io.StringIO(original_header)))
                    except Exception:
                        continue
                    
                    actual_fields = [f.strip().strip('"') for f in parsed_fields]
                    actual_col_count = len(actual_fields)
                    
                    # VALIDATION: Reject lines that are clearly data rows, not headers
                    # Header fields should be short column names, not full sentences
                    max_field_len = max((len(f) for f in actual_fields), default=0)
                    if max_field_len > 80:
                        self.logger.warning(f"{log_prefix} Skipping candidate header match (field too long: {max_field_len} chars, likely a data row): {original_header[:150]}")
                        continue
                    
                    # VALIDATION: At least 6 of 12 canonical fields should be recognizable
                    canonical_lower = {c.lower() for c in _CANONICAL_FIELDS}
                    alias_keys = set(_REMAP_ALIASES.keys())
                    recognized_count = sum(
                        1 for f in actual_fields
                        if f in _CANONICAL_FIELDS or f.lower().strip() in canonical_lower or f.lower().strip() in alias_keys
                    )
                    if recognized_count < 6:
                        self.logger.warning(f"{log_prefix} Skipping candidate header match (only {recognized_count}/12 fields recognized): {original_header[:150]}")
                        continue
                    
                    self.logger.info(f"{log_prefix} Using fallback CSV header detection (found simplified pattern, {recognized_count} canonical fields recognized)")
                    
                    # Accept honesty column count (12 or 14) as matching canonical
                    if actual_col_count == canonical_col_count or actual_col_count == canonical_col_count + 2:
                        self.logger.info(f"{log_prefix} Replacing non-standard header ({actual_col_count} cols) with canonical header. Original: {original_header[:200]}")
                        csv_clean = csv_clean[:line_start] + _canonical_header + csv_clean[line_end:]
                        header_match = re.search(re.escape(_canonical_header), csv_clean)
                        break
                    else:
                        self.logger.info(f"{log_prefix} Column count mismatch: LLM produced {actual_col_count} columns, expected {canonical_col_count}. Attempting positional remap.")
                        try:
                            remapped_fields = []
                            _seen_canonical = set()
                            for field in actual_fields:
                                fl = field.lower().strip()
                                mapped = None
                                if fl in _REMAP_ALIASES:
                                    mapped = _REMAP_ALIASES[fl]
                                elif field in _CANONICAL_FIELDS:
                                    mapped = field
                                else:
                                    for canon in _CANONICAL_FIELDS:
                                        cl = canon.lower()
                                        if len(cl) >= 3 and (cl == fl or (cl in fl and len(cl) / len(fl) > 0.5) or (fl in cl and len(fl) / len(cl) > 0.5)):
                                            mapped = canon
                                            break
                                if mapped and mapped in _seen_canonical:
                                    remapped_fields.append(field)
                                else:
                                    if mapped:
                                        _seen_canonical.add(mapped)
                                    remapped_fields.append(mapped if mapped else field)
                            rebuilt_header = ','.join(f'"{f}"' for f in remapped_fields)
                            
                            # Post-remap validation: check that at least 8/12 canonical fields are in the rebuilt header
                            canonical_in_rebuilt = sum(1 for f in remapped_fields if f in _CANONICAL_FIELDS)
                            if canonical_in_rebuilt < 8:
                                self.logger.warning(f"{log_prefix} Remap produced only {canonical_in_rebuilt}/12 canonical fields. Skipping this match: {rebuilt_header[:200]}")
                                continue
                            
                            extra_cols = [f for f in remapped_fields if f not in _CANONICAL_FIELDS and f.lower() not in ('honesty_score', 'honesty_justification')]
                            if extra_cols:
                                self.logger.info(f"{log_prefix} Remapped header ({canonical_in_rebuilt}/12 canonical, {len(extra_cols)} extra cols ignored: {extra_cols}): {rebuilt_header[:200]}")
                            else:
                                self.logger.info(f"{log_prefix} Remapped header ({canonical_in_rebuilt}/12 canonical): {rebuilt_header[:200]}")
                            csv_clean = csv_clean[:line_start] + rebuilt_header + csv_clean[line_end:]
                            header_match = re.search(re.escape(rebuilt_header), csv_clean)
                            break
                        except Exception as remap_err:
                            self.logger.warning(f"{log_prefix} Header remap failed for this match: {remap_err}. Trying next match.")
                            continue
            
            if not header_match:
                # LAST RESORT: No valid header found. Try injecting canonical header
                # before the first line that looks like a quoted data row (starts with ")
                first_data_line = re.search(r'^"[^"]{0,20}"', csv_clean, re.MULTILINE)
                if first_data_line:
                    inject_pos = first_data_line.start()
                    self.logger.warning(f"{log_prefix} No valid CSV header found. Injecting canonical header before first data row at position {inject_pos}.")
                    csv_clean = csv_clean[:inject_pos] + _canonical_header + '\n' + csv_clean[inject_pos:]
                    header_match = re.search(re.escape(_canonical_header), csv_clean)
                
                if not header_match:
                    preview = csv_clean[:500] if csv_clean else "(empty)"
                    self.logger.error(f"{log_prefix} Could not find CSV header in LLM response. Response preview: {preview}")
                    return []
            
            # Extract CSV starting from header
            csv_data = csv_clean[header_match.start():]
            
            # Use centralized CSV parser for robust parsing
            _expected_csv_fields = ["No", "Name", "usecase", "type", "Analytics Technique", "Statement",
                                    "Solution", "Business Value", "Beneficiary", "Sponsor",
                                    "Tables Involved", "High Level Design"]
            csv_rows = CSVParser.parse_csv_string(
                csv_data,
                logger=self.logger,
                context=log_prefix,
                quoting=csv.QUOTE_ALL,
                expected_fields=_expected_csv_fields
            )
            
            # FALLBACK: If standard parser returns 0 valid rows but CSV data exists,
            # try a tolerant line-by-line parser using "," as field delimiter
            if not csv_rows and csv_data.count('\n') > 1:
                self.logger.warning(f"{log_prefix} Standard CSV parser returned 0 rows. Attempting tolerant field-split parser...")
                _data_section = csv_data.split('\n', 1)[1] if '\n' in csv_data else ''
                _tolerant_rows = []
                for _line_num, _raw_line in enumerate(_data_section.split('\n'), start=1):
                    _raw_line = _raw_line.strip()
                    if not _raw_line:
                        continue
                    # Split on "," (field delimiter in QUOTE_ALL format)
                    _fields = _raw_line.split('","')
                    # Clean leading/trailing quotes from first and last fields
                    if _fields:
                        _fields[0] = _fields[0].lstrip('"')
                        _fields[-1] = _fields[-1].rstrip('"')
                    if len(_fields) >= 12:
                        _tolerant_rows.append(dict(zip(_expected_csv_fields, _fields[:12])))
                    elif len(_fields) >= 5:
                        _padded = _fields + [''] * (12 - len(_fields))
                        _tolerant_rows.append(dict(zip(_expected_csv_fields, _padded[:12])))
                if _tolerant_rows:
                    self.logger.info(f"{log_prefix} Tolerant parser recovered {len(_tolerant_rows)} rows")
                    csv_rows = _tolerant_rows
            
            for row_dict in csv_rows:
                try:
                    # Defensive helper: case-insensitive key lookup with strip
                    def safe_get(d, key):
                        value = d.get(key)
                        if value is None:
                            key_lower = key.lower()
                            for k, v in d.items():
                                if k and k.lower() == key_lower:
                                    value = v
                                    break
                        if value is None:
                            return ''
                        if isinstance(value, str):
                            return value.strip()
                        return str(value).strip()
                    
                    use_case_no = safe_get(row_dict, 'No')
                    valid_no = bool(use_case_no)
                    if not valid_no:
                        self.logger.warning(f"{log_prefix} Skipping row with invalid No field: {use_case_no}")
                        continue
                    
                    _honesty_noise = ('honesty', 'self-assessment', 'score:', 'justification')
                    _no_lower = use_case_no.lower()
                    if any(marker in _no_lower for marker in _honesty_noise):
                        self.logger.warning(f"{log_prefix} Skipping non-data row (honesty/meta text in No field): {use_case_no[:80]}")
                        continue
                    
                    _all_values_concat = ' '.join(
                        str(v).lower() for v in row_dict.values() if v
                    )
                    _meta_line_markers = ('honesty_score:', 'honesty_justification:', 'honesty self-assessment', '**honesty self-assessment')
                    if any(m in _all_values_concat for m in _meta_line_markers):
                        _meta_fields = [
                            f"{k}={str(v)[:40]}" for k, v in row_dict.items()
                            if v and any(m in str(v).lower() for m in _meta_line_markers)
                        ]
                        self.logger.warning(f"{log_prefix} Skipping non-data row (honesty/meta text leaked into fields: {', '.join(_meta_fields[:3])})")
                        continue
                    
                    # Extract Analytics Technique from LLM response (with fallback)
                    analytics_technique = safe_get(row_dict, 'Analytics Technique')
                    if not analytics_technique or analytics_technique == 'N/A':
                        analytics_technique = 'AI Analysis'  # Default fallback
                    
                    # Helper to safely parse float scores
                    def safe_float(d, key, default=3.0):
                        try:
                            value = safe_get(d, key)
                            if not value:
                                return default
                            return float(value)
                        except (ValueError, TypeError):
                            self.logger.warning(f"{log_prefix} Invalid float value for {key}: {value}, using default {default}")
                            return default
                    
                    # Scoring will be added by LLM scoring step after deduplication
                    # Initialize with placeholder values that will be replaced
                    strategic_alignment = 0.0
                    return_on_investment = 0.0
                    reusability = 0.0
                    time_to_value = 0.0
                    data_availability = 0.0
                    data_accessibility = 0.0
                    architecture_fitness = 0.0
                    team_skills = 0.0
                    domain_knowledge = 0.0
                    people_allocation = 0.0
                    budget_allocation = 0.0
                    time_to_production = 0.0
                    value_score = 0.0
                    feasibility_score = 0.0
                    priority_score = 0.0
                    quality_label = "Pending"
                    
                    # Build row dictionary with all fields (SQL, Business Domain, and Subdomain will be added later)
                    # Using safe_get to handle None values and type conversions
                    # Column name MUST be "Statement" (auto-corrected above if LLM used wrong name)
                    statement_value = safe_get(row_dict, 'Statement')
                    
                    row = {
                        "No": use_case_no,
                        "Name": safe_get(row_dict, 'Name'),
                        "usecase": self._normalize_usecase(
                            safe_get(row_dict, 'usecase'),
                            safe_get(row_dict, 'Name')
                        ),
                        "Business Domain": "",  # Will be set during domain clustering
                        "Subdomain": "",  # Will be set during subdomain clustering
                        "type": safe_get(row_dict, 'type'),
                        "Analytics Technique": analytics_technique,  # From LLM response
                        "Statement": statement_value,
                        "Solution": safe_get(row_dict, 'Solution'),
                        "Business Value": safe_get(row_dict, 'Business Value'),
                        "Beneficiary": safe_get(row_dict, 'Beneficiary'),
                        "Sponsor": safe_get(row_dict, 'Sponsor'),
                        "Tables Involved": self._repair_tables_involved(safe_get(row_dict, 'Tables Involved'), row_dict, log_prefix),
                        "High Level Design": safe_get(row_dict, 'High Level Design'),
                        # Scoring columns (only for Excel)
                        "Strategic Alignment": strategic_alignment,
                        "Return on Investment": return_on_investment,
                        "Reusability": reusability,
                        "Time to Value": time_to_value,
                        "Data Availability": data_availability,
                        "Data Accessibility": data_accessibility,
                        "Architecture Fitness": architecture_fitness,
                        "Team Skills": team_skills,
                        "Domain Knowledge": domain_knowledge,
                        "People Allocation": people_allocation,
                        "Budget Allocation": budget_allocation,
                        "Time to Production": time_to_production,
                        # Calculated fields
                        "Value": round(value_score, 2),
                        "Feasibility": round(feasibility_score, 2),
                        "Priority": round(priority_score, 2),
                        "Quality": quality_label
                    }
                    
                    # Validate row has minimum required fields
                    if not row['Name'] or not statement_value:
                        self.logger.warning(f"{log_prefix} Skipping row #{use_case_no}: Missing required fields (Name or Statement)")
                        continue
                    
                    self.logger.debug(f"{log_prefix} Parsed Scenario #{row['No']}: {row['Name']} [Analytics Technique: {analytics_technique}]")
                    
                    parsed_rows.append(row)
                    
                except Exception as e:
                    # Log error with sanitized row data (limit length to avoid huge logs)
                    try:
                        row_summary = {k: str(v)[:100] for k, v in row_dict.items()} if isinstance(row_dict, dict) else str(row_dict)[:200]
                        self.logger.error(f"{log_prefix} Error processing CSV row: {get_clean_error_message(e)}. Row summary: {row_summary}")
                    except:
                        self.logger.error(f"{log_prefix} Error processing CSV row: {get_clean_error_message(e)}. Could not serialize row data.")
                    continue
                    
        except Exception as e:
            self.logger.error(f"{log_prefix} Failed to parse LLM CSV response: {get_clean_error_message(e)}")
            # Show snippet for debugging
            snippet = llm_response[:500] if llm_response else "Empty response"
            self.logger.error(f"{log_prefix} Response snippet: {snippet}")
            return []
        
        # NOTE: Post-processing for naming conventions removed since AI Function field no longer exists
        # Genie Code will determine the best implementation approach
        
        self.logger.info(f"{log_prefix} Robust parsing complete. Found {len(parsed_rows)} rows.")
        return parsed_rows


    def _retry_missing_table_coverage(self, use_cases: list, all_columns: list, document_context_markdown: str, strategic_goals: list = None, include_business_catchall: bool = False) -> list:
        """
        Retry use case generation for tables that have no use cases.
        Each table can be retried up to 2 times maximum.
        
        Args:
            use_cases: List of existing use case dictionaries
            all_columns: List of all column details (catalog, schema, table, column, type, comment)
            document_context_markdown: Markdown for document documents
            strategic_goals: List of strategic goals
            include_business_catchall: If True, also include BUSINESS tables that were never involved in any use cases (catch-all mode)
            
        Returns:
            List of newly generated use cases for missing tables
        """
        from collections import defaultdict
        
        # Extract all tables from column details
        all_tables = set()
        table_columns = defaultdict(list)
        for col_tuple in all_columns:
            catalog, schema, table, column, col_type, comment = col_tuple
            fq_table = f"{catalog}.{schema}.{table}"
            all_tables.add(fq_table)
            table_columns[fq_table].append(col_tuple)
        
        # Extract tables that have use cases (INCLUDING those with empty tables field)
        tables_with_use_cases = set()
        for uc in use_cases:
            tables_str = uc.get('Tables Involved', '')
            if tables_str and not tables_str.startswith('/Volumes'):
                for table in tables_str.split(','):
                    table = table.strip().strip('`')
                    if table:
                        tables_with_use_cases.add(table)
        
        # Find tables without use cases
        missing_tables = all_tables - tables_with_use_cases
        
        # === CATCH-ALL: Include BUSINESS tables that were never involved in any use cases ===
        if include_business_catchall and hasattr(self, 'business_scores'):
            self.logger.info("🔍 CATCH-ALL MODE: Checking for BUSINESS tables that were never involved in use cases...")
            
            all_business_tables = {fqtn for fqtn, score in self.business_scores.items() if score > 0}
            unused_business_tables = all_business_tables - tables_with_use_cases
            unused_business_tables = unused_business_tables.intersection(all_tables)
            
            if unused_business_tables:
                self.logger.warning(f"⚠️ Found {len(unused_business_tables)} BUSINESS tables that were never involved in any use cases")
                missing_tables = missing_tables.union(unused_business_tables)
                unused_sample = sorted(list(unused_business_tables))[:10]
                self.logger.info(f"📋 Sample unused BUSINESS tables: {', '.join(unused_sample)}{'...' if len(unused_business_tables) > 10 else ''}")
            else:
                self.logger.info("✅ All BUSINESS tables have been involved in use cases")
        
        # === JOIN-ELIGIBILITY FILTER FOR MASTER/REFERENCE TABLES ===
        _data_cat = {
            _fqtn.lower(): (_cat or '').upper()
            for _fqtn, _cat in (getattr(self, 'data_category_map', {}) or {}).items()
        }
        _tx_col_index = defaultdict(set)
        for _fqtn_key, _col_list in table_columns.items():
            if _data_cat.get(_fqtn_key.lower(), '') == 'TRANSACTIONAL':
                for _col_tuple in _col_list:
                    _cn = _col_tuple[3] if len(_col_tuple) > 3 else None
                    if _cn:
                        _tx_col_index[_cn.lower()].add(_fqtn_key)
        
        def _compute_join_partners(master_fqtn: str) -> set:
            partners = set()
            for _col_tuple in table_columns.get(master_fqtn, []):
                _cn = _col_tuple[3] if len(_col_tuple) > 3 else None
                if _cn and _cn.lower() in _tx_col_index:
                    partners |= _tx_col_index[_cn.lower()]
            _tbl_basename = master_fqtn.split('.')[-1].lower()
            _singular = _tbl_basename[:-1] if _tbl_basename.endswith('s') else _tbl_basename
            for _base in {_tbl_basename, _singular}:
                for _sfx in ('_id', '_key', '_code'):
                    _candidate = _base + _sfx
                    if _candidate in _tx_col_index:
                        partners |= _tx_col_index[_candidate]
            partners.discard(master_fqtn)
            return partners
        
        _master_join_partners = {}
        _dropped_non_joinable = []
        _filtered_missing = set()
        for _fqtn in missing_tables:
            _cat_label = _data_cat.get(_fqtn.lower(), 'TRANSACTIONAL')
            if _cat_label in ('MASTER', 'REFERENCE'):
                _partners = _compute_join_partners(_fqtn)
                if _partners:
                    _master_join_partners[_fqtn] = _partners
                    _filtered_missing.add(_fqtn)
                else:
                    _dropped_non_joinable.append(_fqtn)
            else:
                _filtered_missing.add(_fqtn)
        
        if _dropped_non_joinable:
            _sample_drop = sorted(_dropped_non_joinable)[:10]
            self.logger.info(
                f"🚫 Catch-all JOIN filter: excluded {len(_dropped_non_joinable)} MASTER/REFERENCE table(s) "
                f"with no column-name overlap to any TRANSACTIONAL table "
                f"(sample: {', '.join(_sample_drop)}{'...' if len(_dropped_non_joinable) > 10 else ''})"
            )
        if _master_join_partners:
            self.logger.info(
                f"🔗 Catch-all JOIN filter: retained {len(_master_join_partners)} MASTER/REFERENCE table(s) "
                f"that CAN join at least one TRANSACTIONAL table — catch-all prompt will enforce JOIN requirement"
            )
        
        missing_tables = _filtered_missing
        
        if not missing_tables:
            self.logger.info("✅ All tables have at least one use case - no retry needed")
            return []
        
        coverage_percentage = ((len(all_tables) - len(missing_tables)) / len(all_tables)) * 100 if all_tables else 0
        self.logger.warning(f"⚠️ Found {len(missing_tables)} tables without use cases (out of {len(all_tables)} total tables - {coverage_percentage:.1f}% coverage)")
        
        # Show sample of missing tables
        missing_sample = sorted(list(missing_tables))[:10]
        self.logger.info(f"📋 Sample missing tables: {', '.join(missing_sample)}{'...' if len(missing_tables) > 10 else ''}")
        
        # Provide actionable insights
        if len(missing_tables) > len(all_tables) * 0.5:
            self.logger.warning(f"⚠️ More than 50% of tables lack use cases. Consider:")
            self.logger.warning(f"   - Checking if LLM is generating use cases for all tables")
            self.logger.warning(f"   - Verifying table names match between schema and use case generation")
            self.logger.warning(f"   - Reviewing business vs technical table filtering")
        
        # Track retry attempts per table (max 2 attempts)
        if not hasattr(self, '_table_retry_counts'):
            self._table_retry_counts = defaultdict(int)
        
        # Filter tables that haven't exceeded retry limit
        tables_to_retry = []
        for table in missing_tables:
            if self._table_retry_counts[table] < 2:
                tables_to_retry.append(table)
                self._table_retry_counts[table] += 1
            else:
                self.logger.warning(f"⚠️ Table {table} has been retried 2 times already - skipping")
        
        if not tables_to_retry:
            self.logger.info("No tables eligible for retry (all have reached 2 attempts)")
            return []
        
        self.logger.info(f"🔄 Retrying use case generation for {len(tables_to_retry)} tables...")
        
        # Group tables into batches (max 50 tables per batch to avoid context overflow)
        max_tables_per_batch = 50
        retry_batches = []
        for i in range(0, len(tables_to_retry), max_tables_per_batch):
            batch_tables = tables_to_retry[i:i+max_tables_per_batch]
            batch_columns = []
            for table in batch_tables:
                batch_columns.extend(table_columns[table])
            retry_batches.append((batch_tables, batch_columns))
        
        self.logger.info(f"📦 Created {len(retry_batches)} retry batch(es) for {len(tables_to_retry)} tables")
        
        # Process retry batches IN PARALLEL using centralized ParallelExecutor
        all_retry_use_cases = []
        
        # ADAPTIVE PARALLELISM: Calculate based on retry batches and columns
        total_retry_columns = sum(len(cols) for _, cols in retry_batches)
        
        retry_parallelism, reason = calculate_adaptive_parallelism(
            "use_case_generation", self.max_parallelism,
            num_items=len(retry_batches),
            total_columns=total_retry_columns,
            avg_prompt_chars=total_retry_columns * 100,
            is_llm_operation=True, logger=self.logger
        )
        log_adaptive_parallelism_decision("use_case_generation", retry_parallelism, self.max_parallelism, reason)
        
        self.logger.info(f"🔄 Processing {len(retry_batches)} retry batch(es) in parallel...")
        
        _retry_bc = self.merged_business_context.get("business_context", "") if hasattr(self, 'merged_business_context') and self.merged_business_context else ""
        _retry_bp = self.merged_business_context.get("business_priorities", []) if hasattr(self, 'merged_business_context') and self.merged_business_context else []
        if isinstance(_retry_bp, list):
            _retry_bp = "\n".join([f"- {p}" for p in _retry_bp]) if _retry_bp else ""
        _retry_si = self.merged_business_context.get("strategic_initiative", "") if hasattr(self, 'merged_business_context') and self.merged_business_context else ""
        _retry_vc = self.merged_business_context.get("value_chain", "") if hasattr(self, 'merged_business_context') and self.merged_business_context else ""
        _retry_rm = self.merged_business_context.get("revenue_model", "") if hasattr(self, 'merged_business_context') and self.merged_business_context else ""

        tasks = []
        for batch_idx, (batch_tables, batch_columns) in enumerate(retry_batches, 1):
            _batch_master_tables = sorted(
                t for t in batch_tables
                if _data_cat.get(t.lower(), '') in ('MASTER', 'REFERENCE')
            )
            _catchall_instruction = ""
            if _batch_master_tables:
                _needed_tx = set()
                for _mt in _batch_master_tables:
                    _needed_tx |= _master_join_partners.get(_mt, set())
                _existing_col_keys = {
                    (f"{_c[0]}.{_c[1]}.{_c[2]}", _c[3]) for _c in batch_columns if len(_c) > 3
                }
                _extra_cols = []
                for _tx_fqtn in _needed_tx:
                    for _col_tuple in table_columns.get(_tx_fqtn, []):
                        if len(_col_tuple) <= 3:
                            continue
                        _key = (f"{_col_tuple[0]}.{_col_tuple[1]}.{_col_tuple[2]}", _col_tuple[3])
                        if _key not in _existing_col_keys:
                            _extra_cols.append(_col_tuple)
                            _existing_col_keys.add(_key)
                if _extra_cols:
                    batch_columns = batch_columns + _extra_cols
                _master_list_str = ', '.join(_batch_master_tables)
                _tx_list_str = ', '.join(sorted(_needed_tx)) if _needed_tx else '(none found)'
                _catchall_instruction = (
                    "CATCH-ALL COVERAGE MODE — NON-NEGOTIABLE ADDITIONAL RULES:\n"
                    f"- The following tables in this batch are MASTER/REFERENCE (static entity/lookup data): {_master_list_str}\n"
                    f"- The valid TRANSACTIONAL join partners available in the schema below are: {_tx_list_str}\n"
                    "- EVERY use case you generate that references any MASTER/REFERENCE table above MUST explicitly JOIN at least one of those TRANSACTIONAL tables. The core analytical question MUST be about EVENTS over time (what HAPPENED), never about static entity attributes.\n"
                    "- If a MASTER/REFERENCE table has no meaningful JOIN path to the listed TRANSACTIONAL partners for a given analytical question, SKIP that table — do NOT generate a use case for it.\n"
                    "- The TRANSACTIONAL partner tables listed above are provided ONLY as JOIN context; do NOT produce new use cases whose primary table is one of those partners unless the analytical question is materially different from any use case already produced in prior passes.\n"
                )
            task = (
                self._process_batch_with_retry,
                (batch_columns, f"RETRY_{batch_idx}", document_context_markdown, strategic_goals,
                 _retry_bc, _retry_bp, _retry_si, _retry_vc, _retry_rm, 2, _catchall_instruction)
            )
            tasks.append(task)
            _master_suffix = f" (includes {len(_batch_master_tables)} master/ref table(s) with JOIN enforcement)" if _batch_master_tables else ""
            self.logger.info(f"✓ Prepared retry batch {batch_idx}/{len(retry_batches)} ({len(batch_tables)} tables){_master_suffix}")
        
        # Execute in parallel with centralized utility
        results = ParallelExecutor.execute_parallel(
            tasks=tasks,
            max_workers=retry_parallelism,
            task_name="Retry Batch",
            logger=self.logger,
            thread_name_prefix="RetryBatch",
            return_exceptions=True
        )
        
        # Collect successful results
        for batch_idx, result in enumerate(results, 1):
            if isinstance(result, Exception):
                self.logger.error(f"❌ Retry batch {batch_idx} failed: {result}")
                continue
            if result:
                self.logger.info(f"✅ Retry batch {batch_idx}: Generated {len(result)} use cases")
                all_retry_use_cases.extend(result)
            else:
                self.logger.warning(f"⚠️ Retry batch {batch_idx}: No use cases generated")
        
        if all_retry_use_cases:
            self.logger.info(f"✅ Retry complete: Generated {len(all_retry_use_cases)} additional use cases")
        else:
            self.logger.warning("⚠️ Retry complete: No additional use cases generated")
        
        return all_retry_use_cases

    def _collect_pending_results(self, current_results: list) -> list:
        """
        Collect results from current batch plus any pending sub-batch results.
        
        Args:
            current_results: Results from current batch
            
        Returns:
            Combined list of current + pending results
        """
        if hasattr(self, '_pending_sub_batch_results') and self._pending_sub_batch_results:
            all_results = current_results + self._pending_sub_batch_results
            # Clear pending results after collecting
            self._pending_sub_batch_results = []
            return all_results
        return current_results
    
    def _process_batch_with_retry(self, column_details: list, batch_num, document_context_markdown: str, strategic_goals: list = None, business_context: str = "", business_priorities: str = "", strategic_initiative: str = "", value_chain: str = "", revenue_model: str = "", max_attempts: int = 3, previous_use_cases_feedback: str = "", temperature_override: float = None) -> list:
        """
        Process a batch of column details to generate use cases with retry logic.
        Automatically splits context if input is too long for the model.
        
        STRATEGY: When tables don't fit in context:
        1. Split tables across multiple sub-batches (NEVER drop business tables)
        2. Process ALL sub-batches recursively
        3. Track which columns are kept from each table (saved to disk, not memory)
        4. Column tracking is loaded from disk during Genie Code generation
        
        Args:
            column_details: List of column tuples (catalog, schema, table, column, type, comment)
            batch_num: Batch number for logging and prefixing (can be int or str)
            document_context_markdown: Document documents markdown
            strategic_goals: List of strategic goals for the business (used for Strategic Alignment scoring)
            max_attempts: Maximum number of attempts (default 3)
            
        Returns:
            List of use case dictionaries (includes results from sub-batches)
        """
        log_prefix = f"[Batch {batch_num}]"
        
        # === NEW: Register columns and tables for Bitmap ID generation ===
        with self.registry_lock:
            for col_tuple in column_details:
                # col_tuple: (catalog, schema, table, column, type, comment)
                fqn = f"{col_tuple[0]}.{col_tuple[1]}.{col_tuple[2]}.{col_tuple[3]}"
                table_fqn = f"{col_tuple[0]}.{col_tuple[1]}.{col_tuple[2]}"
                
                # Register table if not already registered
                if table_fqn not in self.table_id_map:
                    table_id = str(self.next_table_id)
                    self.next_table_id += 1
                    self.table_id_map[table_fqn] = table_id
                    self.id_table_map[table_id] = table_fqn
                
                if fqn not in self.column_id_map:
                    col_id = str(self.next_column_id)
                    self.next_column_id += 1
                    self.column_id_map[fqn] = col_id
                    
                    # Create description (Type + Comment)
                    desc = f"{col_tuple[4]}"
                    if col_tuple[5]:
                        desc += f" - {col_tuple[5]}"
                    
                    self.id_column_map[col_id] = {
                        "fqn": fqn,
                        "description": desc
                    }
        
        current_column_details = column_details

        prompt_template = self.ai_agent.prompt_templates.get("BASE_USE_CASE_GEN_PROMPT", "")
        safe_limit = get_safe_context_limit(language="English", buffer_percent=0.9, prompt_name="BASE_USE_CASE_GEN_PROMPT")
        if strategic_goals and len(strategic_goals) > 0:
            strategic_goals_text = "\n".join([f"- {goal}" for goal in strategic_goals[:10]])
        else:
            strategic_goals_text = "- Maximize operational efficiency\n- Improve customer satisfaction\n- Reduce operational costs\n- Drive revenue growth\n- Ensure compliance and risk management"
        if business_priorities and len(business_priorities) > 0:
            business_priorities_text = "\n".join([f"- {priority}" for priority in business_priorities[:10]])
        else:
            business_priorities_text = "- None"
        if self.user_strategic_goals:
            goals_text = "\n".join([f"- {goal}" for goal in self.user_strategic_goals])
            additional_context_section = f"""**STRATEGIC GOALS (HIGHEST PRIORITY)**:

The user provided Strategic Goals that MUST be followed during generation.

**STRATEGIC GOALS:**
{goals_text}

**REQUIREMENTS**:
- Generate ONLY use cases that align with these Strategic Goals.
- Generate EVERY possible use case that aligns with these Strategic Goals. Do not omit any valid use case.
- Do NOT cap the number of use cases; completeness is mandatory.
- Use semantic understanding of the goals; do NOT apply rigid keyword rules.
- Do NOT generate use cases outside these goals."""
        else:
            additional_context_section = "*(No Strategic Goals provided by user - proceed with standard business analysis)*"
        if self.user_business_domains:
            domains_list = ", ".join([f'"{domain}"' for domain in self.user_business_domains])
            focus_areas_instruction = f"""  - **🚨 CRITICAL - USER-SPECIFIED BUSINESS DOMAINS 🚨**: You MUST assign use cases ONLY to the following business domains: {domains_list}. 
   * These are the ONLY valid Business Domain values - DO NOT invent new domains.
   * ALL use cases MUST be categorized into one of these exact domains.
   * DO NOT create any domain that is not in this list.
   * DO NOT modify, abbreviate, or expand these domain names - use them EXACTLY as provided.
   * The Business Domain field MUST exactly match one of these domains."""
        else:
            focus_areas_instruction = ""
        _quality_gate_size = len(QUALITY_GATE_GENERATION_BLOCKS.get(self.use_cases_quality, QUALITY_GATE_GOOD))
        base_prompt_size = len(prompt_template) + _quality_gate_size + len(business_context) + len(business_priorities_text) + len(strategic_initiative) + len(value_chain) + len(revenue_model) + len(strategic_goals_text) + len(additional_context_section) + len(focus_areas_instruction) + len(previous_use_cases_feedback) + 1000
        
        self.logger.info(f"{log_prefix} Starting batch processing with {len(column_details)} columns from {len(set([c[2] for c in column_details]))} tables")
        all_tables_in_call = sorted({f"{c[0]}.{c[1]}.{c[2]}" for c in column_details})
        _dc_map = getattr(self, "data_category_map", {})
        _tx_tables = [t for t in all_tables_in_call if _dc_map.get(t) == 'TRANSACTIONAL']
        _ctx_tables = [t for t in all_tables_in_call if t not in set(_tx_tables)]
        self.logger.info(f"{log_prefix} PRIMARY (transactional) tables ({len(_tx_tables)}): {', '.join(_tx_tables) if _tx_tables else 'NONE'}")
        self.logger.info(f"{log_prefix} CONTEXT (master/ref) tables ({len(_ctx_tables)}): {', '.join(_ctx_tables) if _ctx_tables else 'NONE'}")
        log_print(f"{log_prefix} PRIMARY tables ({len(_tx_tables)}): {', '.join(_tx_tables) if _tx_tables else 'NONE'}")
        if _ctx_tables:
            log_print(f"{log_prefix} CONTEXT tables ({len(_ctx_tables)}): {', '.join(_ctx_tables)}")
        
        _hallucination_feedback = ""
        for attempt in range(1, max_attempts + 1):
            try:
                if attempt > 1:
                    self.logger.info(f"{log_prefix} Retry attempt {attempt}/{max_attempts}...")

                estimated_schema_size = self._estimate_schema_markdown_size(current_column_details)
                estimated_prompt_size = base_prompt_size + estimated_schema_size
                if estimated_prompt_size > safe_limit:
                    raise InputTooLongError(
                        f"Proactive split: Input length {estimated_prompt_size:,} characters exceeds "
                        f"safe limit of {safe_limit:,} (with 10% buffer)"
                    )

                self.logger.debug(f"{log_prefix} Formatting schema for prompt...")
                schema_markdown = self._format_schema_for_prompt(current_column_details)
                if not schema_markdown:
                    self.logger.warning(f"{log_prefix} Produced no schema markdown. Skipping.")
                    return []
                
                fk_relationships_text = "None"
                related_tables_added = 0
                additional_related_columns = []
                _fk_source = "unknown"
                try:
                    if self.data_loader and getattr(self.data_loader, "foreign_key_graph", None):
                        batch_tables = {(c[0], c[1], c[2]) for c in current_column_details}
                        fk_relations = self.data_loader.get_foreign_key_relations(batch_tables)
                        if fk_relations:
                            rel_lines = []
                            related_tables_to_load = set()
                            
                            for rel in fk_relations:
                                src = f"{rel[0]}.{rel[1]}.{rel[2]}.{rel[3]}"
                                ref_catalog = rel[4] or rel[0]
                                ref_schema = rel[5] or rel[1]
                                ref_table = rel[6]
                                tgt = f"{ref_catalog}.{ref_schema}.{ref_table}.{rel[7]}"
                                rel_lines.append(f"{src} -> {tgt}")
                                
                                ref_tuple = (ref_catalog, ref_schema, ref_table)
                                if ref_tuple not in batch_tables:
                                    related_tables_to_load.add(ref_tuple)
                            
                            if rel_lines:
                                fk_relationships_text = "\n".join(sorted(set(rel_lines)))
                            
                            if related_tables_to_load:
                                self.logger.info(f"{log_prefix} Loading {len(related_tables_to_load)} FK-related tables to include in schema context...")
                                additional_related_columns = []
                                for (ref_cat, ref_sch, ref_tbl) in list(related_tables_to_load)[:10]:
                                    try:
                                        ref_cols = self.data_loader._get_table_details(
                                            ref_cat, ref_sch, ref_tbl,
                                            apply_sampling=self.data_loader.enable_column_sampling if hasattr(self.data_loader, 'enable_column_sampling') else False
                                        )
                                        if ref_cols:
                                            additional_related_columns.extend(ref_cols)
                                            related_tables_added += 1
                                            self.logger.debug(f"{log_prefix} Added FK-related table: {ref_cat}.{ref_sch}.{ref_tbl} ({len(ref_cols)} columns)")
                                    except Exception as e:
                                        self.logger.debug(f"{log_prefix} Could not load FK-related table {ref_cat}.{ref_sch}.{ref_tbl}: {str(e)[:80]}")
                                
                                if additional_related_columns:
                                    related_schema = self._format_schema_for_prompt(additional_related_columns)
                                    if related_schema:
                                        schema_markdown = schema_markdown + "\n\n<!-- FK-Related Tables -->\n" + related_schema
                                        self.logger.info(f"{log_prefix} Added {related_tables_added} FK-related tables to schema context ({len(additional_related_columns)} columns)")
                except Exception as fk_err:
                    self.logger.debug(f"{log_prefix} Failed to gather FK relationships: {str(fk_err)[:100]}")
                
                data_cat_map = getattr(self, "data_category_map", {})
                all_batch_tables = sorted({f"{c[0]}.{c[1]}.{c[2]}" for c in current_column_details})
                
                primary_tables = [t for t in all_batch_tables if data_cat_map.get(t) == 'TRANSACTIONAL']
                if not primary_tables:
                    primary_tables = all_batch_tables
                primary_set = set(primary_tables)
                
                context_tables = sorted(t for t in all_batch_tables if t not in primary_set)
                
                reference_tables_in_cat = {k for k, v in data_cat_map.items() if v == "REFERENCE"}
                master_tables_in_cat = {k for k, v in data_cat_map.items() if v == "MASTER"}
                
                fk_only_tables = set()
                if additional_related_columns:
                    fk_only_tables = {f"{c[0]}.{c[1]}.{c[2]}" for c in additional_related_columns} - set(all_batch_tables)
                
                table_lines = []
                for t in primary_tables:
                    table_lines.append(f"- `{t}` (PRIMARY — TRANSACTIONAL, generate use cases FROM this table)")
                for t in context_tables:
                    if t in reference_tables_in_cat:
                        table_lines.append(f"- `{t}` (CONTEXT — REFERENCE DATA, use ONLY for JOINs and lookups, NEVER as the main analytical subject)")
                    elif t in master_tables_in_cat:
                        table_lines.append(f"- `{t}` (CONTEXT — MASTER DATA, use for JOINs and enrichment, but analytical question must come from transactional events)")
                    else:
                        table_lines.append(f"- `{t}` (CONTEXT — available for JOINs and enrichment)")
                for t in sorted(fk_only_tables):
                    if t in reference_tables_in_cat:
                        table_lines.append(f"- `{t}` (FK-RELATED, REFERENCE — use ONLY for JOINs, NEVER as sole table)")
                    elif t in master_tables_in_cat:
                        table_lines.append(f"- `{t}` (FK-RELATED, MASTER — use for JOINs and enrichment only)")
                    else:
                        table_lines.append(f"- `{t}` (FK-RELATED — available for JOINs)")
                
                all_allowed_tables = sorted(set(all_batch_tables) | fk_only_tables)
                _known_bad = getattr(self, '_known_hallucinated_tables', set())
                _known_bad_block = ""
                if _known_bad:
                    _known_bad_block = (
                        "\n\n**⛔ KNOWN HALLUCINATED TABLES (from other batches — DO NOT USE):**\n"
                        + "\n".join(f"- ❌ `{t}` — DOES NOT EXIST" for t in sorted(_known_bad))
                        + "\n"
                    )
                allowed_tables_block = (
                    "\n\n### 🚨 ALLOWED TABLES REGISTRY (EXHAUSTIVE LIST) 🚨\n"
                    "**You may ONLY use tables from this list. ANY other table name is HALLUCINATION and will be REJECTED.**\n"
                    "**Every use case MUST be centered on at least one PRIMARY (TRANSACTIONAL) table.**\n"
                    "**CONTEXT and FK-RELATED tables provide enrichment via JOINs but must NEVER be the sole analytical subject.**\n"
                    "**The analytical question must always derive from EVENTS in transactional data, not from static entity attributes.**\n\n"
                    + "\n".join(table_lines)
                    + "\n\n**Total allowed tables: " + str(len(all_allowed_tables)) + ". NO OTHER TABLES EXIST.**\n"
                    + _known_bad_block
                )
                schema_markdown = schema_markdown + allowed_tables_block
                
                _effective_additional_context = additional_context_section
                if _hallucination_feedback:
                    _effective_additional_context = additional_context_section + _hallucination_feedback
                
                _quality_gate_text = QUALITY_GATE_GENERATION_BLOCKS.get(
                    self.use_cases_quality, QUALITY_GATE_GOOD
                )
                
                # Structural-name FK fallback (industry-agnostic): when Unity Catalog has no declared FKs,
                # scan current_column_details for shared column names across tables. If the SAME column
                # name appears in >=2 tables AND looks like an identifier (ends in _id / _code / _key / Id
                # / starts with id_, or contains "_id_"), treat as inferred join key and emit a hint line.
                try:
                    if fk_relationships_text == "None" and current_column_details:
                        _by_col = {}
                        for _row in current_column_details:
                            try:
                                if isinstance(_row, (tuple, list)) and len(_row) >= 4:
                                    _cat, _sch, _tbl, _col = _row[0], _row[1], _row[2], _row[3]
                                else:
                                    continue
                                _cn = str(_col or "").strip()
                                if not _cn:
                                    continue
                                _key = _cn.lower()
                                _fqt = f"{_cat}.{_sch}.{_tbl}"
                                _by_col.setdefault(_key, set()).add(_fqt)
                            except Exception:
                                continue
                        _lines = []
                        def _looks_like_id(name: str) -> bool:
                            n = name.lower()
                            return (n.endswith("_id") or n.endswith("id") or n.endswith("_code")
                                    or n.endswith("_key") or n.endswith("_sk") or n.endswith("_nk")
                                    or n.startswith("id_") or n.startswith("sk_") or "_id_" in n
                                    or n in {"id", "key", "code"})
                        # Pass 1: ID-like columns (prefer these — highest-confidence joins)
                        for _cname, _tables in _by_col.items():
                            if len(_tables) >= 2 and _looks_like_id(_cname):
                                _t_list = sorted(_tables)
                                _lines.append(f"[INFERRED JOIN KEY] `{_cname}` appears in: {', '.join(_t_list)}")
                        # Pass 2: if Pass 1 produced nothing, fall back to ANY column shared across 2+
                        # tables. Accuweather-style schemas share value columns (city_name,
                        # datetime_valid_local, etc.) that legitimately act as join keys without
                        # matching ID-suffix conventions. Structural — no hardcoded column names.
                        if not _lines:
                            _shared = [(n, sorted(t)) for n, t in _by_col.items() if len(t) >= 2 and n]
                            # sort by (fewer tables first then name) so we emit the most specific joins first
                            _shared.sort(key=lambda x: (len(x[1]), x[0]))
                            for _cname, _t_list in _shared[:40]:
                                _lines.append(f"[INFERRED JOIN KEY] `{_cname}` appears in: {', '.join(_t_list)}")
                        if _lines:
                            fk_relationships_text = "\n".join(_lines[:40])
                            _fk_source = "structural-name FK fallback"
                            try:
                                self.logger.info(f"{log_prefix} No declared FKs; inferred {len(_lines)} shared-column join key(s) structurally")
                            except Exception:
                                pass
                except Exception as _fk_err:
                    try:
                        self.logger.debug(f"{log_prefix} structural FK fallback skipped: {_fk_err}")
                    except Exception:
                        pass

                prompt_vars = {
                    "schema_markdown": schema_markdown,
                    "foreign_key_relationships": fk_relationships_text,
                    "business_context": business_context,
                    "business_priorities": business_priorities_text,
                    "strategic_initiative": strategic_initiative,
                    "value_chain": value_chain,
                    "revenue_model": revenue_model,
                    "strategic_goals": strategic_goals_text,
                    "additional_context_section": _effective_additional_context,
                    "focus_areas_instruction": focus_areas_instruction,
                    "previous_use_cases_feedback": previous_use_cases_feedback,
                    "quality_gate_block": _quality_gate_text,
                    "generation_instructions_section": self._get_generation_instruction_for_prompt("BASE_USE_CASE_GEN_PROMPT"),
                }
                
                # PROACTIVE CHECK: Estimate prompt size and split BEFORE attempting LLM call
                # This saves time by not waiting for LLM failures
                try:
                    estimated_prompt_size = base_prompt_size + len(schema_markdown) + len(fk_relationships_text)
                    
                    if estimated_prompt_size > safe_limit:
                        # Prompt exceeds safe limit - proactively split WITHOUT attempting LLM call
                        self.logger.warning(
                            f"{log_prefix} PROACTIVE SPLIT: Prompt size ({estimated_prompt_size:,} chars) exceeds "
                            f"safe limit ({safe_limit:,} chars with 10% buffer). Splitting batch without attempting LLM call."
                        )
                        
                        # Raise InputTooLongError to trigger the split logic below
                        raise InputTooLongError(
                            f"Proactive split: Input length {estimated_prompt_size:,} characters exceeds "
                            f"safe limit of {safe_limit:,} (with 10% buffer)"
                        )
                    else:
                        # Safe to proceed - log if we're approaching the limit (>80% of safe limit)
                        if estimated_prompt_size > (safe_limit * 0.8):
                            self.logger.info(
                                f"{log_prefix} Prompt size: {estimated_prompt_size:,} chars "
                                f"({(estimated_prompt_size/safe_limit)*100:.1f}% of safe limit)"
                            )
                
                except InputTooLongError:
                    # Re-raise to trigger split logic
                    raise
                except Exception as e:
                    # Any other error in estimation - log and continue to actual LLM call
                    self.logger.debug(f"{log_prefix} Proactive size check failed: {get_clean_error_message(e, max_chars=2000)}")
                
                self.logger.info(f"⏳ {log_prefix} Sending batch to use case generation prompt...")
                self.logger.info(f"⏳ {log_prefix} Waiting for LLM response (may take 3-5 min)...")
                response_raw = self.ai_agent.run_worker(
                    step_name=f"Batch_{batch_num}",
                    worker_prompt_path="BASE_USE_CASE_GEN_PROMPT",
                    prompt_vars=prompt_vars,
                    response_schema=None,
                    temperature_override=temperature_override
                )
                self.logger.info(f"✅ {log_prefix} Received LLM response, parsing CSV...")
                
                response_clean = clean_csv_response(response_raw) if response_raw else ""
                parsed_rows = self._parse_llm_csv_response(response_clean, log_prefix) if response_clean else []
                
                del response_raw, response_clean
                del prompt_vars, schema_markdown
                
                self.logger.info(f"✅ {log_prefix} Parsed {len(parsed_rows)} use cases")
                
                if not parsed_rows:
                    raise Exception("LLM returned no use cases")
                
                validation_schema = getattr(self, "_business_column_details_global", current_column_details)
                if additional_related_columns:
                    validation_schema = list(validation_schema) + additional_related_columns
                is_tables_valid, hallucinated_use_cases, valid_use_cases = self._validate_use_case_tables(
                    parsed_rows, validation_schema, log_prefix
                )
                
                if not is_tables_valid:
                    hallucinated_count = len(hallucinated_use_cases)
                    hallucination_rate = hallucinated_count / len(parsed_rows) * 100
                    valid_count = len(valid_use_cases)
                    
                    if valid_count == 0:
                        self.logger.warning(f"{log_prefix} ⚠️ Table hallucination detected: {hallucinated_count}/{len(parsed_rows)} use cases ({hallucination_rate:.1f}%)")
                        for i, uc in enumerate(hallucinated_use_cases[:3]):
                            self.logger.warning(f"{log_prefix}    Example {i+1}: {uc.get('No')}: {uc.get('Name')} - {uc.get('hallucination_reason')}")
                        if attempt < max_attempts:
                            _bad_tables = set()
                            for uc in hallucinated_use_cases:
                                reason = uc.get('hallucination_reason', '')
                                if 'Tables not found' in reason:
                                    for t in reason.replace('Tables not found in schema: ', '').split(', '):
                                        t = t.strip()
                                        if t:
                                            _bad_tables.add(t)
                            _cross_batch_tracker = getattr(self, '_known_hallucinated_tables', None)
                            if _cross_batch_tracker is not None:
                                _bad_tables.update(_cross_batch_tracker)
                                _cross_batch_tracker.update(_bad_tables)
                            if _bad_tables:
                                _hallucination_feedback = (
                                    "\n\n### 🚨 PREVIOUS ATTEMPT REJECTED — HALLUCINATED TABLES 🚨\n"
                                    "Your previous response was REJECTED because you used tables that DO NOT EXIST.\n"
                                    "The following table names are HALLUCINATIONS — they are NOT in the schema:\n"
                                    + "\n".join(f"- ❌ `{t}` — DOES NOT EXIST" for t in sorted(_bad_tables))
                                    + "\n\nYou MUST only use tables from the ALLOWED TABLES REGISTRY above. "
                                    "Double-check EVERY table name against the registry before including it.\n"
                                )
                                self.logger.info(f"{log_prefix} Injecting hallucination feedback for retry: {len(_bad_tables)} hallucinated table(s)")
                            self.logger.warning(f"{log_prefix}    Retrying batch (attempt {attempt + 1}/{max_attempts}) because no valid use cases were returned")
                            continue
                        self.logger.error(f"{log_prefix} ❌ No valid use cases after {max_attempts} attempts due to hallucinated tables")
                        return self._collect_pending_results([])
                    
                    self.logger.warning(f"{log_prefix} ⚠️ Table hallucination detected: {hallucinated_count}/{len(parsed_rows)} use cases ({hallucination_rate:.1f}%). Dropping hallucinated use cases and continuing with {valid_count} valid use cases.")
                    for i, uc in enumerate(hallucinated_use_cases[:3]):
                        self.logger.warning(f"{log_prefix}    Example {i+1}: {uc.get('No')}: {uc.get('Name')} - {uc.get('hallucination_reason')}")
                    _cross_batch_tracker = getattr(self, '_known_hallucinated_tables', None)
                    if _cross_batch_tracker is not None:
                        for uc in hallucinated_use_cases:
                            reason = uc.get('hallucination_reason', '')
                            if 'Tables not found' in reason:
                                for t in reason.replace('Tables not found in schema: ', '').split(', '):
                                    t = t.strip()
                                    if t:
                                        _cross_batch_tracker.add(t)
                    parsed_rows = valid_use_cases
                
                # Re-number use cases with batch prefix
                # Handle both int and string batch_num (for retry batches)
                if isinstance(batch_num, int):
                    batch_prefix = f"{batch_num:02d}"  # Changed from B{batch_num:03d} to just 2-digit number
                else:
                    batch_prefix = str(batch_num)  # Already formatted (e.g., "RETRY_1")
                
                for row in parsed_rows:
                    try:
                        original_id = row['No']
                        use_case_num = original_id.split('-')[-1]
                        new_id = f"AI-{batch_prefix}-U{use_case_num}"
                        row['No'] = new_id
                        row['batch'] = batch_num
                    except Exception as e:
                        self.logger.warning(f"{log_prefix} Failed to re-number row: {get_clean_error_message(e)}")
                        row['batch'] = batch_num
                
                self.logger.debug(f"{log_prefix} Successfully processed {len(parsed_rows)} use cases on attempt {attempt}")
                
                log_print(f"\n{'='*80}")
                log_print(f"📊 TOP USE CASES FROM {log_prefix} (for early quality review):")
                log_print(f"{'='*80}\n")
                
                for use_case in parsed_rows[:10]:
                    use_case_name = self._normalize_usecase(use_case.get('usecase', ''), use_case.get('Name', ''))
                    log_print(f"   {use_case.get('No', 'N/A')}: {use_case_name or 'N/A'}")
                
                log_print(f"\n{'='*80}\n")
                
                # Validate subdomain rules (silent check - accept response regardless)
                # The domain fixer will fix any issues later
                is_valid, violations, corrected_rows = self._validate_subdomain_rules(parsed_rows, log_prefix)
                # Always accept the response - domain fixer will handle issues
                # Collect any pending sub-batch results
                return self._collect_pending_results(parsed_rows)
                
            except InputTooLongError as e:
                # Handle "input too long" by intelligently splitting the batch
                # NEVER DROP BUSINESS TABLES - keep splitting until they fit
                
                # Group columns by table
                table_to_columns = {}
                for col in current_column_details:
                    table_key = (col[0], col[1], col[2])  # (catalog, schema, table)
                    if table_key not in table_to_columns:
                        table_to_columns[table_key] = []
                    table_to_columns[table_key].append(col)
                
                num_tables = len(table_to_columns)
                
                # Get business scores to check if tables are marked as business
                business_scores = getattr(self, 'business_scores', {})
                
                def is_business_table(table_key):
                    """Check if a table is marked as business (score > 0)."""
                    fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                    return business_scores.get(fqtn, 0) > 0
                
                if num_tables > 1:
                    tables_list = list(table_to_columns.keys())
                    reference_tables_set = {k for k, v in getattr(self, "data_category_map", {}).items() if v == "REFERENCE"}
                    filtered_tables = []
                    for table_key in tables_list:
                        fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                        if reference_tables_set and fqtn in reference_tables_set:
                            continue
                        filtered_tables.append(table_key)
                    if not filtered_tables:
                        self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Only reference tables present; skipping use case generation.")
                        return self._collect_pending_results([])
                    self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Falling back to single-table calls for {num_tables} tables.")
                    self.processing_honesty['total_batch_splits'] += 1
                    split_type = "Proactive" if "Proactive split" in str(e) else "Reactive"
                    split_info = {
                        'batch': batch_num,
                        'original_tables': num_tables,
                        'split_into': num_tables,
                        'sub_batch_1_tables': 1,
                        'sub_batch_2_tables': 1,
                        'reason': 'Input too long for LLM',
                        'split_type': split_type
                    }
                    self.processing_honesty['batch_split_history'].append(split_info)
                    current_column_details = table_to_columns[filtered_tables[0]]
                    current_column_details = self._augment_columns_with_related_tables(current_column_details)
                    if not hasattr(self, '_pending_sub_batch_results'):
                        self._pending_sub_batch_results = []
                    for idx, table_key in enumerate(filtered_tables[1:], start=2):
                        single_cols = table_to_columns[table_key]
                        single_cols = self._augment_columns_with_related_tables(single_cols)
                        single_batch_id = f"{batch_num}_T{idx}"
                        single_use_cases = self._process_batch_with_retry(
                            single_cols,
                            single_batch_id,
                            document_context_markdown,
                            strategic_goals,
                            business_context,
                            business_priorities,
                            strategic_initiative,
                            value_chain,
                            revenue_model,
                            max_attempts,
                            previous_use_cases_feedback,
                            temperature_override=temperature_override
                        )
                        if single_use_cases:
                            self._pending_sub_batch_results.extend(single_use_cases)
                            self.logger.info(f"{log_prefix} Single-table sub-batch {single_batch_id} generated {len(single_use_cases)} use cases")
                    continue
                    
                elif num_tables == 1:
                    # Single table is too big: try dropping columns
                    table_key = list(table_to_columns.keys())[0]
                    table_columns = table_to_columns[table_key]
                    table_is_business = is_business_table(table_key)
                    fqtn = f"{table_key[0]}.{table_key[1]}.{table_key[2]}"
                    
                    if len(table_columns) > 500:
                        keep_count = 500
                    else:
                        keep_count = len(table_columns) - 100
                    if keep_count < 5:
                        if table_is_business:
                            keep_count = 5
                        else:
                            self.logger.error(f"{log_prefix} Input too long even with minimal columns ({len(table_columns)} columns from non-business table {table_key[2]}). Dropping this table.")
                            return self._collect_pending_results([])
                    current_column_details = table_columns[:keep_count]
                    
                    kept_columns = [col[3] for col in current_column_details]  # col[3] is column name
                    self.storage_manager.save_column_tracking(fqtn, kept_columns)
                    
                    dropped_count = len(table_columns) - keep_count
                    drop_info = {
                        'table': fqtn,
                        'original_columns': len(table_columns),
                        'kept_columns': keep_count,
                        'dropped_columns': dropped_count,
                        'drop_percentage': (dropped_count / len(table_columns)) * 100,
                        'is_business': table_is_business
                    }
                    if fqtn not in [t['table'] for t in self.processing_honesty['tables_with_columns_dropped']]:
                        self.processing_honesty['tables_with_columns_dropped'].append(drop_info)
                    else:
                        # Update existing entry with new drop info
                        for idx, existing in enumerate(self.processing_honesty['tables_with_columns_dropped']):
                            if existing['table'] == fqtn:
                                self.processing_honesty['tables_with_columns_dropped'][idx] = drop_info
                                break
                    
                    table_type = "BUSINESS" if table_is_business else "non-business"
                    self.logger.warning(f"{log_prefix} Input too long ({str(e)}). Single {table_type} table {table_key[2]} is too large. Dropping columns from {len(table_columns)} to {keep_count} columns and retrying...")
                    self.logger.info(f"{log_prefix} Saved column tracking for {fqtn}: {len(kept_columns)} columns ({', '.join(kept_columns[:5])}{'...' if len(kept_columns) > 5 else ''})")
                    
                    continue
                    
                else:
                    # No tables? This shouldn't happen
                    self.logger.error(f"{log_prefix} Input too long but no tables found. Cannot process.")
                    return self._collect_pending_results([])
                
            except Exception as e:
                if attempt < max_attempts:
                    self.logger.warning(f"{log_prefix} Attempt {attempt} failed: {get_clean_error_message(e)}. Retrying...")
                else:
                    self.logger.error(f"{log_prefix} All {max_attempts} attempts failed: {get_clean_error_message(e)}")
                    return self._collect_pending_results([])
        
        # If we exhaust all attempts without success, still return any pending results
        return self._collect_pending_results([])

    _JOB_LAUNCHER_CLASS_SOURCE = r'''
import re as _jl_re, os as _jl_os

class JobLauncher:
    _TAG_SAFE_RE = None
    @staticmethod
    def _sanitize_tag(value):
        if JobLauncher._TAG_SAFE_RE is None:
            JobLauncher._TAG_SAFE_RE = _jl_re.compile(r'[^A-Za-z0-9._-]')
        s = JobLauncher._TAG_SAFE_RE.sub('_', str(value))
        s = _jl_re.sub(r'_+', '_', s)
        s = s.strip('_').strip('.').strip('-')
        return s
    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        self.job_tags = {self._sanitize_tag(k): self._sanitize_tag(v) for k, v in (job_tags or {}).items()}
    def launch(self, job_name=None, run_name=None):
        _fail = {"success": False, "job_id": None, "run_id": None, "job_name": job_name or "", "run_name": run_name or "", "job_url": "", "error": None}
        try:
            import time as _jl_time
            from databricks.sdk import WorkspaceClient as _JL_WC
            from databricks.sdk.service import jobs as _jl_jobs
            w = _JL_WC()
            _job_name = job_name or f"dbx_inspire_job_{int(_jl_time.time())}"
            _run_name = run_name or _job_name
            is_serverless, cluster_id = self._detect_compute_type()
            task = _jl_jobs.Task(task_key="inspire_task", notebook_task=_jl_jobs.NotebookTask(notebook_path=self.notebook_path, base_parameters=self.widget_key_values), timeout_seconds=14400)
            if not is_serverless and cluster_id:
                task.existing_cluster_id = cluster_id
            _existing_job_id = None
            try:
                for _candidate in w.jobs.list(name=_job_name):
                    if _candidate.settings and _candidate.settings.name == _job_name:
                        _existing_job_id = _candidate.job_id
                        break
            except Exception: pass
            if _existing_job_id:
                w.jobs.update(job_id=_existing_job_id, new_settings=_jl_jobs.JobSettings(name=_job_name, tags=self.job_tags, tasks=[task]))
                _job_id = _existing_job_id
            else:
                _created = w.jobs.create(name=_job_name, tags=self.job_tags, tasks=[task])
                _job_id = _created.job_id
            run = w.jobs.run_now(job_id=_job_id)
            host, org_id = self._get_workspace_context()
            job_url = ""
            if host:
                _org_param = f"?o={org_id}" if org_id else ""
                job_url = f"{host}/jobs/{_job_id}/runs/{run.run_id}{_org_param}"
            result = {"success": True, "job_id": _job_id, "run_id": run.run_id, "job_name": _job_name, "run_name": _run_name, "job_url": job_url, "error": None}
            self._print_success_banner(result)
            return result
        except Exception as _launch_exc:
            _fail["error"] = str(_launch_exc)
            return _fail
    @staticmethod
    def get_current_notebook_path():
        _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        try:
            _path = _nb_ctx.notebookPath().get()
            if _path: return _path
        except Exception: pass
        try:
            import json as _jl_json
            _ctx = _jl_json.loads(_nb_ctx.toJson())
            for _key in ("notebook_path", "notebookPath"):
                _val = _ctx.get("extraContext", {}).get(_key, "")
                if _val: return _val
                _val = _ctx.get("tags", {}).get(_key, "")
                if _val: return _val
        except Exception: pass
        return ""
    @staticmethod
    def update_job_tags(updated_tags):
        if not updated_tags: return
        try:
            _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            if not _nb_ctx.jobId().isDefined(): return
            _job_id_str = _nb_ctx.jobId().get()
            if not _job_id_str: return
            _str_tags = {JobLauncher._sanitize_tag(k): JobLauncher._sanitize_tag(v) for k, v in updated_tags.items()}
            try:
                from databricks.sdk import WorkspaceClient as _JL_WC2
                from databricks.sdk.service import jobs as _jl_jobs2
                _w = _JL_WC2()
                _w.jobs.update(job_id=int(_job_id_str), new_settings=_jl_jobs2.JobSettings(tags=_str_tags))
                return
            except Exception: pass
            _host, _token = "", ""
            try:
                _host = _nb_ctx.apiUrl().get()
                _token = _nb_ctx.apiToken().get()
            except Exception: pass
            if not _host:
                try:
                    from databricks.sdk import WorkspaceClient as _JL_WC3
                    _w3 = _JL_WC3()
                    _host = str(_w3.config.host).rstrip("/")
                    _token = _w3.config.token
                except Exception: pass
            if _host and _token:
                import requests as _jl_req
                _jl_req.patch(f"{_host}/api/2.1/jobs/update", headers={"Authorization": f"Bearer {_token}"}, json={"job_id": int(_job_id_str), "new_settings": {"tags": _str_tags}}, timeout=30)
        except Exception: pass
    def _detect_compute_type(self):
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE": return True, None
        try:
            if "connect" in type(spark).__module__: return True, None
        except Exception: return True, None
        try:
            _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            _cid = _ctx.clusterId().get() if _ctx.clusterId().isDefined() else ""
            if _cid: return False, _cid
        except Exception: pass
        return True, None
    def _get_workspace_context(self):
        try:
            import json as _jl_json
            _ctx_json = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
            _ctx = _jl_json.loads(_ctx_json)
            _host = _ctx.get("extraContext", {}).get("api_url", "")
            _org = _ctx.get("extraContext", {}).get("orgId", "")
            if _host: return _host.rstrip("/"), _org
        except Exception: pass
        try:
            from databricks.sdk import WorkspaceClient as _JL_WC3
            _w = _JL_WC3()
            return str(_w.config.host).rstrip("/"), ""
        except Exception: pass
        return "", ""
    def _print_success_banner(self, result):
        _banner = self._build_banner_str(result)
        print(_banner)
        return _banner
    def _build_banner_str(self, result):
        _jn = result.get("job_name", "N/A")
        _rn = result.get("run_name", "N/A")
        _rid = str(result.get("run_id", "N/A"))
        _url = result.get("job_url", "")
        from datetime import datetime as _jl_dt
        _now = _jl_dt.now().strftime("%Y-%m-%d %H:%M:%S")
        _nb = self.notebook_path.rsplit("/", 1)[-1] if self.notebook_path else "N/A"
        _content_lines = [f"  Notebook:     {_nb}", f"  Job Name:     {_jn}", f"  Job Run Name: {_rn}", f"  Job Run ID:   {_rid}", f"  Launched At:  {_now}"]
        _footer_lines = ["  Go to Jobs & Pipelines to follow the progress", "  of the run, or click the link below:"]
        if _url:
            _footer_lines.append("")
            _footer_lines.append(f"  {_url}")
        _all = ["  JOB LAUNCHED SUCCESSFULLY"] + _content_lines + _footer_lines
        _iw = max(len(l) for l in _all) + 2
        def _row(text): return f"  {text}"
        lines = ["", "=" * (_iw + 4), "  JOB LAUNCHED SUCCESSFULLY", "=" * (_iw + 4)]
        for cl in _content_lines: lines.append(_row(cl))
        lines.append("-" * (_iw + 4))
        for fl in _footer_lines: lines.append(_row(fl))
        lines.append("=" * (_iw + 4))
        return "\n".join(lines)
    @staticmethod
    def _enum_str(val):
        if val is None:
            return ""
        if hasattr(val, 'value'):
            return str(val.value).upper()
        return str(val).upper()
    @staticmethod
    def cleanup_job():
        try:
            _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            if not _nb_ctx.jobId().isDefined():
                return
            _job_id_str = _nb_ctx.jobId().get()
            if not _job_id_str:
                return
            from databricks.sdk import WorkspaceClient as _CJ_WC
            _w = _CJ_WC()
            _w.jobs.delete(job_id=int(_job_id_str))
            print(f"  Job {_job_id_str} cleaned up successfully.")
        except Exception as _cj_err:
            print(f"  Job cleanup warning (non-fatal): {_cj_err}")
    def verify_run_alive(self, run_id, pings=10, interval_s=10):
        import time as _vra_time
        try:
            from databricks.sdk import WorkspaceClient as _VRA_WC
            _w = _VRA_WC()
        except Exception:
            print(f"  Could not create WorkspaceClient for verification. Assuming job is running.")
            return True
        _definitive_failure_states = {"INTERNAL_ERROR", "SKIPPED"}
        _dead_results = {"FAILED", "TIMEDOUT", "CANCELED"}
        for _i in range(pings):
            _vra_time.sleep(interval_s)
            try:
                _run = _w.jobs.get_run(run_id=run_id)
                _state = self._enum_str(getattr(_run.state, 'life_cycle_state', None))
                _result = self._enum_str(getattr(_run.state, 'result_state', None))
                print(f"  Ping {_i + 1}/{pings}: life_cycle_state={_state or 'N/A'}, result_state={_result or 'N/A'}")
                if _state in _definitive_failure_states:
                    print(f"  Job run definitively failed ({_state}). Falling back to local execution.")
                    return False
                if _state in ("TERMINATED", "TERMINATING"):
                    if _result in _dead_results:
                        print(f"  Job run definitively failed ({_state}/{_result}). Falling back to local execution.")
                        return False
                    if _result == "SUCCESS":
                        print(f"  Job run already completed successfully during verification.")
                        return True
            except Exception as _ping_err:
                print(f"  Ping {_i + 1}/{pings}: status check error (assuming still starting): {_ping_err}")
        print(f"  Job not confirmed failed after {pings} pings. Assuming job is running (may be waiting for compute).")
        return True
'''

    def _build_job_launcher_source(self, use_case_name: str, analytics_technique: str = "") -> str:
        _san_biz = re.sub(r'[^a-z0-9_]', '_', str(self.business_name).lower().strip())
        _san_biz = re.sub(r'_+', '_', _san_biz).strip('_') or "unknown"
        _san_uc = re.sub(r'[^a-z0-9_]', '_', str(use_case_name).lower().strip())
        _san_uc = re.sub(r'_+', '_', _san_uc).strip('_') or "unknown"

        _launcher_source_val = "Inspire App" if self._user_provided_session_id else "Inspire Notebook"
        _is_train_infer = analytics_technique in TRAIN_INFER_TECHNIQUES

        if _is_train_infer:
            _launcher_logic = f'''
def _run_with_job_launcher():
    _biz = "{_san_biz}"
    _use_case_name = "{_san_uc}"
    _job_name = "dbx_inspire_ai_" + _use_case_name
    _genie_nb_path = JobLauncher.get_current_notebook_path()
    _genie_nb_name = _genie_nb_path.rsplit("/", 1)[-1] if _genie_nb_path else "unknown"
    try:
        _run_type = dbutils.widgets.get("run_type").strip()
    except Exception:
        _run_type = "Training"
    _run_type_tag = "training" if _run_type == "Training" else "inference"
    _job_tags = {{
        "dbx_inspire_ai_business": JobLauncher._sanitize_tag(_biz),
        "dbx_inspire_ai_run_type": _run_type_tag,
        "dbx_inspire_ai_usecase": JobLauncher._sanitize_tag(_use_case_name),
        "dbx_inspire_ai_notebook": JobLauncher._sanitize_tag(_genie_nb_name),
        "dbx_inspire_ai_session_id": "{self.session_id}",
        "dbx_inspire_ai_launcher_source": "{_launcher_source_val}",
    }}
    def _dispatch_main():
        if _run_type == "Training":
            print(f"Run type: TRAINING")
            main_train()
        else:
            print(f"Run type: INFERENCE")
            main_infer()
    try:
        _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        is_job_context = _nb_ctx.jobId().isDefined()
    except Exception:
        is_job_context = False
    if is_job_context:
        print(f"Running inside job context -- run_type={{_run_type}}")
        try:
            JobLauncher.update_job_tags(_job_tags)
        except Exception:
            pass
        try:
            _dispatch_main()
        except Exception as e:
            print(f"Execution error: {{e}}")
            raise
        finally:
            try:
                JobLauncher.cleanup_job()
            except Exception:
                pass
    else:
        print(f"Running interactively -- launching as a Databricks job (run_type={{_run_type}})...")
        _launch_result = None
        _launcher = None
        try:
            _nb_path = JobLauncher.get_current_notebook_path()
            if _nb_path:
                _launcher = JobLauncher(_nb_path, {{}}, _job_tags)
                _launch_result = _launcher.launch(job_name=_job_name, run_name=_job_name)
                if not _launch_result["success"]:
                    print(f"Job launch failed: {{_launch_result.get('error', 'Unknown')}}")
                    print("Continuing with local execution...")
                    _dispatch_main()
            else:
                print("Could not determine notebook path. Running locally...")
                _dispatch_main()
        except Exception as e:
            print(f"Job launch error (non-critical): {{e}}")
            print("Continuing with local execution...")
            _dispatch_main()

        if _launch_result and _launch_result.get("success"):
            _run_id = _launch_result.get("run_id")
            print(f"Verifying job run {{_run_id}} is alive (10 pings, 10s each)...")
            _is_alive = True
            if _launcher and _run_id:
                _is_alive = _launcher.verify_run_alive(_run_id, pings=10, interval_s=10)
            if _is_alive:
                _exit_msg = _launcher._build_banner_str(_launch_result) if _launcher else ("Job launched: " + _launch_result.get("job_url", ""))
                dbutils.notebook.exit(_exit_msg)
            else:
                print("Job launched but definitively failed during verification. Falling back to local execution...")
                _dispatch_main()

_run_with_job_launcher()
'''
        else:
            _launcher_logic = f'''
def _run_with_job_launcher():
    _biz = "{_san_biz}"
    _use_case_name = "{_san_uc}"
    _job_name = "dbx_inspire_ai_" + _use_case_name
    _genie_nb_path = JobLauncher.get_current_notebook_path()
    _genie_nb_name = _genie_nb_path.rsplit("/", 1)[-1] if _genie_nb_path else "unknown"
    _job_tags = {{
        "dbx_inspire_ai_business": JobLauncher._sanitize_tag(_biz),
        "dbx_inspire_ai_run_type": "inference",
        "dbx_inspire_ai_usecase": JobLauncher._sanitize_tag(_use_case_name),
        "dbx_inspire_ai_notebook": JobLauncher._sanitize_tag(_genie_nb_name),
        "dbx_inspire_ai_session_id": "{self.session_id}",
        "dbx_inspire_ai_launcher_source": "{_launcher_source_val}",
    }}
    try:
        _nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        is_job_context = _nb_ctx.jobId().isDefined()
    except Exception:
        is_job_context = False
    if is_job_context:
        print("Running inside job context -- executing main() locally.")
        try:
            JobLauncher.update_job_tags(_job_tags)
        except Exception:
            pass
        try:
            main()
        except Exception as e:
            print(f"Execution error: {{e}}")
            raise
        finally:
            try:
                JobLauncher.cleanup_job()
            except Exception:
                pass
    else:
        print("Running interactively -- launching as a Databricks job...")
        _launch_result = None
        _launcher = None
        try:
            _nb_path = JobLauncher.get_current_notebook_path()
            if _nb_path:
                _launcher = JobLauncher(_nb_path, {{}}, _job_tags)
                _launch_result = _launcher.launch(job_name=_job_name, run_name=_job_name)
                if not _launch_result["success"]:
                    print(f"Job launch failed: {{_launch_result.get('error', 'Unknown')}}")
                    print("Continuing with local execution...")
                    main()
            else:
                print("Could not determine notebook path. Running locally...")
                main()
        except Exception as e:
            print(f"Job launch error (non-critical): {{e}}")
            print("Continuing with local execution...")
            main()

        if _launch_result and _launch_result.get("success"):
            _run_id = _launch_result.get("run_id")
            print(f"Verifying job run {{_run_id}} is alive (10 pings, 10s each)...")
            _is_alive = True
            if _launcher and _run_id:
                _is_alive = _launcher.verify_run_alive(_run_id, pings=10, interval_s=10)
            if _is_alive:
                _exit_msg = _launcher._build_banner_str(_launch_result) if _launcher else ("Job launched: " + _launch_result.get("job_url", ""))
                dbutils.notebook.exit(_exit_msg)
            else:
                print("Job launched but definitively failed during verification. Falling back to local execution...")
                main()

_run_with_job_launcher()
'''
        return self._JOB_LAUNCHER_CLASS_SOURCE.strip() + '\n\n' + _launcher_logic.strip() + '\n'

    def _build_job_launcher_cell(self, use_case_name: str, analytics_technique: str = "") -> dict:
        _full_source = self._build_job_launcher_source(use_case_name, analytics_technique)
        return {
            "cell_type": "code",
            "execution_count": 0,
            "outputs": [],
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4()), "title": "Job Launcher"}},
            "source": [_full_source]
        }

    @staticmethod
    def _escape_for_triple_quote_embedding(text: str) -> str:
        return text.replace('\\', '\\\\').replace('"""', '\\"\\"\\"')

    def _build_job_launcher_instruction_section(self, use_case_name: str, analytics_technique: str = "") -> str:
        job_launcher_code = self._build_job_launcher_source(use_case_name, analytics_technique)
        _is_train_infer = analytics_technique in TRAIN_INFER_TECHNIQUES

        if '"""' in job_launcher_code:
            self.logger.warning(
                "Job Launcher source contains triple-quotes (will be escaped for safe embedding)"
            )

        if _is_train_infer:
            _invocation_desc = "`main_train()` or `main_infer()` based on the `run_type` widget"
            _critical_rules = (
                "**CRITICAL RULES:**\n"
                "1. Do NOT create ANY other cell that calls `main_train()`, `main_infer()`, or `if __name__` guards. "
                "If you do, that is an IMPLEMENTATION FAILURE.\n"
                "2. Do NOT add `main_train()`, `main_infer()`, or any execution call at the bottom of the code definition cell. "
                "The code cell must contain ONLY `create_widgets()` call, imports, constants, helper function definitions, "
                "`def main_train():`, and `def main_infer():`. Nothing else.\n"
                "3. The Job Launcher cell below reads the `run_type` widget, handles job submission, tagging, verification, "
                "cleanup (auto-deletes the job after execution), and fallback to local execution automatically. Do NOT duplicate this logic.\n"
                "4. Do NOT set `os.environ['JOB_ID']` or `os.environ.setdefault('JOB_ID', ...)` anywhere. That is FORBIDDEN.\n\n"
            )
            _cell_order = (
                "**REQUIRED CELL ORDER (exactly 4 cells):**\n"
                "- Cell 1: `%pip install ...` + `dbutils.library.restartPython()` (if external packages needed)\n"
                "- Cell 2: `create_widgets()` call, ALL imports, constants, helper functions, `def main_train():`, `def main_infer():` -- ZERO top-level execution of main functions\n"
                "- Cell 3: Job Launcher cell (the code below, VERBATIM)\n"
                "- Cell 4: Self-critique markdown cell\n\n"
            )
        else:
            _invocation_desc = "`main()`"
            _critical_rules = (
                "**CRITICAL RULES:**\n"
                "1. Do NOT create ANY other cell that calls `main()`, `result = main()`, or `if __name__ == \"__main__\": main()`. "
                "If you do, that is an IMPLEMENTATION FAILURE.\n"
                "2. Do NOT add `main()` or any execution call at the bottom of the code definition cell. "
                "The code cell must contain ONLY imports, constants, helper function definitions, and `def main():`. Nothing else.\n"
                "3. The Job Launcher cell below handles job submission, tagging, verification, cleanup (auto-deletes the job after execution), "
                "and fallback to local execution automatically. Do NOT duplicate this logic.\n"
                "4. Do NOT set `os.environ['JOB_ID']` or `os.environ.setdefault('JOB_ID', ...)` anywhere. That is FORBIDDEN.\n\n"
            )
            _cell_order = (
                "**REQUIRED CELL ORDER (exactly 4 cells):**\n"
                "- Cell 1: `%pip install ...` + `dbutils.library.restartPython()` (if external packages needed)\n"
                "- Cell 2: ALL imports, constants, helper functions, and `def main():` -- ZERO top-level execution\n"
                "- Cell 3: Job Launcher cell (the code below, VERBATIM)\n"
                "- Cell 4: Self-critique markdown cell\n\n"
            )

        return (
            "\n\n## 8. Job Launcher Cell (MANDATORY -- COPY VERBATIM AS YOUR LAST CODE CELL)\n\n"
            "**THIS IS A PASS/FAIL REQUIREMENT. VIOLATION = IMPLEMENTATION FAILURE.**\n\n"
            "**CRITICAL: Do NOT rename the notebook.** The notebook already has a name. Do NOT change, modify, or retitle it. "
            "Only generate code cells inside the existing notebook. Renaming the notebook is FORBIDDEN.\n\n"
            "You MUST create a SEPARATE code cell titled **\"Job Launcher\"** as the absolute LAST code cell in the notebook "
            f"(only the self-critique markdown cell may follow it). This cell is the SOLE mechanism that invokes {_invocation_desc}. "
            "Copy the ENTIRE code block below into that cell VERBATIM -- do NOT modify, abbreviate, or omit any part of it.\n\n"
            f"{_critical_rules}"
            f"{_cell_order}"
            "**Job Launcher code (copy VERBATIM into a cell titled \"Job Launcher\"):**\n\n"
            "```python\n"
            f"{job_launcher_code}"
            "```\n"
        )

    def _assemble_single_use_case_notebook(self, use_case: dict, translations: dict, domain_folder: str, subdomain_folder: str, notebook_name: str, summary_dict: dict = None):
        from databricks.sdk.service import workspace
        
        t = translations
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')

        def translate_value(field_name, value):
            if not value or value == 'N/A':
                return value
            value_key_map = {
                'Problem': 'value_type_problem', 'Risk': 'value_type_risk',
                'Opportunity': 'value_type_opportunity', 'Improvement': 'value_type_improvement',
                'Very High': 'value_priority_very_high', 'High': 'value_priority_high',
                'Low': 'value_priority_low', 'Very Low': 'value_priority_very_low'
            }
            if value == 'Medium':
                if field_name in ['type', 'Type']:
                    return t.get('value_type_medium', value)
                elif field_name in ['priority', 'Priority', 'aspect_priority']:
                    return t.get('value_priority_medium', value)
            translation_key = value_key_map.get(value)
            return t.get(translation_key, value) if translation_key else value

        def safe_val(val):
            if val is None or (isinstance(val, str) and not val.strip()):
                return 'N/A'
            return str(val)

        final_cells = []

        generation_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        title_source = [
            f"# {use_case_id}: {self._normalize_usecase(use_case.get('usecase', ''), use_case_name)}\n\n",
            f"**{self.business_name}** | {domain} > {subdomain}\n\n",
            f"*Generated by Databricks Inspire AI on {generation_timestamp}*\n\n",
            f'<div style="background-color:#FFF3CD; color:#664D03; border: 1px solid #FFECB5; padding:10px; border-radius:5px; margin-top:10px;"><b>Disclaimer:</b> {t["disclaimer"]}</div>\n\n',
            "---\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": title_source
        })

        story_parts = []
        statement = use_case.get('Statement', '').strip()
        solution = use_case.get('Solution', '').strip()
        biz_value = use_case.get('Business Value', '').strip()
        if statement:
            story_parts.append(f"**{t.get('statement', 'Problem Statement')}:** {statement}")
        if solution:
            story_parts.append(f"**{t.get('solution', 'Proposed Solution')}:** {solution}")
        if biz_value:
            story_parts.append(f"**{t.get('aspect_value', 'Business Value')}:** {biz_value}")
        if story_parts:
            story_text = '\n\n'.join(story_parts)
            final_cells.append({
                "cell_type": "markdown",
                "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
                "source": [f"### Use Case Summary\n\n", f"{story_text}\n\n", "---\n"]
            })

        details_source = [
            f"## Use Case Details\n\n",
            f"| {t['aspect']} | {t['description']} |\n", "|---|---|\n",
            f"| **{t.get('sum_id', 'ID')}** | {safe_val(use_case_id)} |\n",
            f"| **{t.get('sum_name', 'Name')}** | {safe_val(use_case_name)} |\n",
            f"| **{t.get('domain', 'Business Domain')}** | {safe_val(domain)} |\n",
            f"| **{t['subdomain']}** | {safe_val(subdomain)} |\n",
            f"| **{t['type']}** | {translate_value('type', use_case.get('type', 'N/A'))} |\n",
            f"| **{t.get('analytics_technique', 'Analytics Technique')}** | {safe_val(use_case.get('Analytics Technique'))} |\n",
            f"| **{t['priority']}** | {translate_value('priority', use_case.get('Priority', 'N/A'))} |\n",
            f"| **{t.get('primary_table', 'Primary Table')}** | {safe_val(use_case.get('Primary Table'))} |\n",
            f"| **{t['statement']}** | {safe_val(use_case.get('Statement'))} |\n",
            f"| **{t['solution']}** | {safe_val(use_case.get('Solution'))} |\n",
            f"| **{t['aspect_value']}** | {safe_val(use_case.get('Business Value'))} |\n",
            f"| **{t['aspect_tables']}** | {safe_val(use_case.get('Tables Involved'))} |\n",
            f"| **{t.get('quality', 'Quality')}** | {safe_val(use_case.get('Quality'))} |\n",
            f"| **{t.get('quality_reasons', 'Quality Reasons')}** | {safe_val(use_case.get('Quality Summary'))} |\n",
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": details_source
        })

        genie_instruction_escaped = use_case.get('genie_instruction_escaped', '')
        if not genie_instruction_escaped:
            _raw = sanitize_llm_special_chars(use_case.get('genie_instruction', '').strip())
            genie_instruction_escaped = self._escape_for_triple_quote_embedding(_raw) if _raw else ''
        notebook_language = "python"
        if genie_instruction_escaped and len(genie_instruction_escaped) > 100:
            final_cells.append({
                "cell_type": "markdown",
                "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
                "source": [
                    "---\n\n",
                    "## Genie Code Instruction\n\n",
                    "**HOW TO USE:**\n",
                    "1. Open Genie Code in your Databricks workspace (side panel in any notebook)\n",
                    "2. Copy the ENTIRE content of the instruction below (between the triple quotes)\n",
                    "3. Paste it into Genie Code and let Genie generate the implementation\n",
                    "4. Genie will create the complete code — review, iterate, and execute\n",
                ]
            })
            final_cells.append({
                "cell_type": "code", "execution_count": 0, "outputs": [],
                "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
                "source": [
                    'genie_instruction = """\n',
                    genie_instruction_escaped,
                    '\n"""\n\nprint(genie_instruction)\n'
                ]
            })

        notebook_metadata = {
            "application/vnd.databricks.v1+notebook": {
                "computePreferences": None,
                "dashboards": [],
                "environmentMetadata": {"base_environment": "", "environment_version": "4"},
                "inputWidgetPreferences": None,
                "language": notebook_language,
                "notebookMetadata": {"pythonIndentUnit": 4},
                "notebookName": notebook_name,
                "widgets": {}
            },
            "language_info": {"name": notebook_language}
        }

        notebook_obj = {"cells": final_cells, "metadata": notebook_metadata, "nbformat": 4, "nbformat_minor": 0}
        notebook_content_str = json.dumps(notebook_obj, indent=2)

        gen_dir = os.path.join(self.notebook_output_dir, domain_folder, subdomain_folder)
        notebook_full_path = os.path.join(gen_dir, f"{notebook_name}.ipynb")

        max_retries = self.max_retry_attempts + 1
        retry_delay = 2

        for attempt in range(1, max_retries + 1):
            try:
                self.w_client.workspace.mkdirs(gen_dir)
                self.w_client.workspace.import_(
                    path=notebook_full_path, overwrite=True, format=workspace.ImportFormat.JUPYTER,
                    content=base64.b64encode(notebook_content_str.encode('utf-8')).decode(),
                )
                _status_info = self.w_client.workspace.get_status(notebook_full_path)
                abs_path = _status_info.path
                if getattr(_status_info, 'object_id', None):
                    self._notebook_object_ids[use_case_id] = _status_info.object_id
                self.logger.info(f"✓ Notebook '{notebook_name}' saved to: {abs_path}")
                return abs_path
            except Exception as err:
                self.logger.warning(f"[{use_case_id}] Notebook import attempt {attempt}/{max_retries} failed: {err}")
                if attempt < max_retries:
                    import time
                    time.sleep(retry_delay * (2 ** (attempt - 1)))
                else:
                    self.logger.error(f"❌ [{use_case_id}] Failed to import notebook after {max_retries} attempts")
                    raise

    def _assemble_notebook_for_db(self, db_name: str, use_cases: list, translations: dict, db_prefix: str, filename_override: str = None, domain_summary: str = None, subdomain_folder: str = None):
        self.logger.debug(f"--- [LEGACY] Assembling multi-use-case notebook for: {db_name} (English) ---")
        if not use_cases:
            self.logger.warning(f"No use cases provided for {db_name}. Skipping notebook creation.")
            return
        t = translations
        grouped_by_domain = defaultdict(list)
        for uc in use_cases: grouped_by_domain[uc.get('Business Domain') or 'Other'].append(uc)
        
        final_cells = []
        title_cell_source = [
            f"# {t['pdf_title']}\n\n",
            f"## For {self.business_name}: {db_name}\n\n"
        ]
        title_cell = {
            "cell_type": "markdown", 
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, 
            "source": title_cell_source
        }
        final_cells.append(title_cell)
        
        generation_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        disclaimer_cell_source = [
            f"*Generated by Databricks Inspire AI on {generation_timestamp}*\n\n",
            "**Disclaimer:** This content is AI-generated and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from the use of this content.\n\n",
            "---\n"
        ]
        disclaimer_cell = {
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": disclaimer_cell_source
        }
        final_cells.append(disclaimer_cell)
        
        uc_story_lines = []
        for uc in use_cases[:5]:
            uc_name = uc.get('Name', '').strip()
            uc_value = uc.get('Business Value', '').strip()
            if uc_name and uc_value:
                uc_story_lines.append(f"- **{uc_name}**: {uc_value}")
            elif uc_name:
                uc_story_lines.append(f"- **{uc_name}**")
        if uc_story_lines:
            remaining = len(use_cases) - 5
            story_footer = f"\n\n*...and {remaining} more use cases below.*" if remaining > 0 else ""
            story_cell_source = [
                f"### Use Case Highlights\n\n",
                '\n'.join(uc_story_lines) + story_footer + "\n\n",
                "---\n\n"
            ]
            story_cell = {
                "cell_type": "markdown",
                "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
                "source": story_cell_source
            }
            final_cells.append(story_cell)
            self.logger.info(f"Added use case highlights cell for '{db_name}'")
        
        first_section = True
        for domain, domain_use_cases in sorted(grouped_by_domain.items()):
            self.logger.debug(f"Assembling domain summary tables: '{domain}' with {len(domain_use_cases)} use cases.")
            
            subdomain_groups = defaultdict(list)
            for uc in domain_use_cases:
                subdomain = uc.get('Subdomain', 'General')
                subdomain_groups[subdomain].append(uc)
            
            for subdomain, subdomain_use_cases in sorted(subdomain_groups.items()):
                self.logger.debug(f"  - Subdomain '{subdomain}': {len(subdomain_use_cases)} use cases")
                
                if first_section:
                    header_source = [
                        f"## {t['summaries']}\n\n",
                        f"### {subdomain}\n\n",
                        f"| {t['sum_id']} | {t['sum_name']} | {t['priority']} | {t['sum_value']} |\n",
                        "|---|---|---|---|\n"
                    ]
                    first_section = False
                else:
                    header_source = [
                        f"\n### {subdomain}\n\n",
                        f"| {t['sum_id']} | {t['sum_name']} | {t['priority']} | {t['sum_value']} |\n",
                        "|---|---|---|---|\n"
                    ]
                
                subdomain_header_cell = {
                    "cell_type": "markdown",
                    "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
                    "source": header_source
                }
                sorted_subdomain_use_cases = sorted(subdomain_use_cases, key=self._natural_sort_key)
                toc_entries = [f"| {uc['No']} | {uc['Name']} | {uc.get('Priority', 'N/A')} | {uc['Business Value']} |\n" for uc in sorted_subdomain_use_cases]
                subdomain_header_cell["source"].extend(toc_entries)
                final_cells.append(subdomain_header_cell)
        
        disclaimer_text = t["disclaimer"]
        disclaimer_html = f'<div style="background-color:#FFF3CD; color:#664D03; border: 1px solid #FFECB5; padding:10px; border-radius:5px; margin-top:10px;"><b>Disclaimer:</b> {disclaimer_text}</div>'
        final_cells.extend([
            {"cell_type": "markdown", "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, "source": [disclaimer_html]},
            {"cell_type": "markdown", "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, "source": [f"<hr>\n\n# {t['detailed_scenarios']}\n"]}
        ])
        
        use_cases_sorted = sorted(use_cases, key=self._natural_sort_key)
        self.logger.debug(f"Sorted {len(use_cases_sorted)} use cases by ID (natural order)")
        
        def translate_value(field_name, value):
            if not value or value == 'N/A':
                return value
            value_key_map = {
                'Problem': 'value_type_problem', 'Risk': 'value_type_risk',
                'Opportunity': 'value_type_opportunity', 'Improvement': 'value_type_improvement',
                'Very High': 'value_priority_very_high', 'High': 'value_priority_high',
                'Low': 'value_priority_low', 'Very Low': 'value_priority_very_low'
            }
            if value == 'Medium':
                if field_name in ['type', 'Type']:
                    return t.get('value_type_medium', value)
                elif field_name in ['priority', 'Priority', 'aspect_priority']:
                    return t.get('value_priority_medium', value)
            translation_key = value_key_map.get(value)
            return t.get(translation_key, value) if translation_key else value

        def safe_notebook_str(val):
            if val is None or (isinstance(val, str) and not val.strip()):
                return 'N/A'
            return str(val)

        for use_case in use_cases_sorted:
            use_case_id = use_case.get('No', 'UNKNOWN')
            numbered_title = f"{use_case['No']}: {self._normalize_usecase(use_case.get('usecase', ''), use_case.get('Name', ''))}"

            combined_source = [
                f"### {numbered_title}\n\n",
                f"| {t['aspect']} | {t['description']} |\n", "|---|---|\n",
                f"| **Description** | {safe_notebook_str(use_case.get('Name'))} |\n",
                f"| **{t['subdomain']}** | {safe_notebook_str(use_case.get('Subdomain'))} |\n",
                f"| **{t['type']}** | {translate_value('type', use_case.get('type', 'N/A'))} |\n",
                f"| **{t.get('analytics_technique', 'Analytics Technique')}** | {safe_notebook_str(use_case.get('Analytics Technique'))} |\n",
                f"| **{t['priority']}** | {translate_value('priority', use_case.get('Priority', 'N/A'))} |\n",
                f"| **{t.get('primary_table', 'Primary Table')}** | {safe_notebook_str(use_case.get('Primary Table'))} |\n",
                f"| **{t['statement']}** | {safe_notebook_str(use_case.get('Statement'))} |\n",
                f"| **{t['solution']}** | {safe_notebook_str(use_case.get('Solution'))} |\n",
                f"| **{t['aspect_value']}** | {safe_notebook_str(use_case.get('Business Value'))} |\n",
                f"| **{t['aspect_tables']}** | {safe_notebook_str(use_case.get('Tables Involved'))} |\n",
                f"| **{t.get('quality', 'Quality')}** | {safe_notebook_str(use_case.get('Quality'))} |\n",
                f"| **{t.get('quality_reasons', 'Quality Reasons')}** | {safe_notebook_str(use_case.get('Quality Summary'))} |\n"
            ]
            details_cell = {"cell_type": "markdown", "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, "source": combined_source}
            final_cells.append(details_cell)

            nb_path = use_case.get('notebook_path', '').strip()
            if nb_path:
                use_case_name = use_case.get('Name', '')
                nb_link_source = [
                    f"#### Genie Code Notebook\n\n",
                    f"**Use Case {use_case_id}** — {use_case_name}\n\n",
                    f"📓 **Notebook Path**: `{nb_path}`\n\n",
                    f"To execute this use case, open the notebook above or run:\n",
                ]
                nb_run_cell = [f'# dbutils.notebook.run("{nb_path}", timeout_seconds=3600)\n']
                link_cell = {"cell_type": "markdown", "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, "source": nb_link_source}
                final_cells.append(link_cell)
                run_cell = {"cell_type": "code", "execution_count": 0, "outputs": [], "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}}, "source": nb_run_cell}
                final_cells.append(run_cell)
        
        if not filename_override:
            self.logger.warning(f"No filename_override provided for notebook assembly '{db_name}'. Defaulting.")
            notebook_name_sanitized = f"{db_prefix}_{self._sanitize_name(db_name)}"
        else:
            notebook_name_sanitized = filename_override

        if subdomain_folder:
            gen_dir = os.path.join(self.notebook_output_dir, subdomain_folder)
            try:
                self.w_client.workspace.mkdirs(gen_dir)
            except Exception as e:
                self.logger.debug(f"mkdirs for '{gen_dir}' (may already exist): {e}")
        else:
            gen_dir = self.notebook_output_dir
        
        notebook_metadata = {
            "application/vnd.databricks.v1+notebook": {
                "computePreferences": None,
                "dashboards": [],
                "environmentMetadata": {
                    "base_environment": "",
                    "environment_version": "4"
                },
                "inputWidgetPreferences": None,
                "language": "python",
                "notebookMetadata": {
                    "pythonIndentUnit": 4
                },
                "notebookName": notebook_name_sanitized,
                "widgets": {}
            },
            "language_info": {
                "name": "python"
            }
        }
        
        final_notebook_obj = { "notebook_content": { "cells": final_cells, "metadata": notebook_metadata, "nbformat": 4, "nbformat_minor": 0 } }
        
        max_retries = self.max_retry_attempts + 1
        retry_delay = 2
        
        for attempt in range(1, max_retries + 1):
            try:
                self.logger.info(f"Importing notebook '{notebook_name_sanitized}.ipynb' (attempt {attempt}/{max_retries})...")
                notebook_full_path = os.path.join(gen_dir, f"{notebook_name_sanitized}.ipynb")
                notebook_content_str = json.dumps(final_notebook_obj["notebook_content"], indent=2)
                
                # Import with timeout handling
                import time
                start_time = time.time()
                self.w_client.workspace.import_(
                    path=notebook_full_path, overwrite=True, format=workspace.ImportFormat.JUPYTER,
                    content=base64.b64encode(notebook_content_str.encode('utf-8')).decode(),
                )
                elapsed_time = time.time() - start_time
                
                self.logger.info(f"✓ Notebook '{notebook_name_sanitized}.ipynb' imported successfully in {elapsed_time:.1f}s")
                abs_path = self.w_client.workspace.get_status(notebook_full_path).path
                self.logger.info(f"Notebook is located at: {abs_path}")
                log_print(f"   ✓ Notebook saved to: {abs_path}")
                break  # Success - exit retry loop
                
            except Exception as err:
                self.logger.warning(f"Attempt {attempt}/{max_retries} failed: {err}")
                
                if attempt < max_retries:
                    # Exponential backoff
                    wait_time = retry_delay * (2 ** (attempt - 1))
                    self.logger.info(f"Retrying in {wait_time} seconds...")
                    import time
                    time.sleep(wait_time)
                else:
                    # Final attempt failed
                    self.logger.error(f"❌ Failed to import notebook '{notebook_name_sanitized}' after {max_retries} attempts")
                    self.logger.error(f"Error details: {err}")
                    log_print(f"   ❌ ERROR: Failed to create notebook '{notebook_name_sanitized}': {err}", file=sys.stderr)
                    # Re-raise to allow caller to handle
                    raise

    def _build_schema_details_from_use_cases(self, use_cases: list) -> list:
        schema_details = []
        seen_tables = set()
        for uc in use_cases:
            tables_str = uc.get('Tables Involved', '')
            cols_str = uc.get('Columns Involved', '') or uc.get('Involved Columns', '') or ''
            if not tables_str:
                continue
            table_parts = re.split(r'[,\s]+', tables_str)
            for tbl in table_parts:
                tbl = tbl.strip().replace('`', '').strip()
                if not tbl or '.' not in tbl:
                    continue
                parts = tbl.split('.')
                if len(parts) == 3:
                    catalog, schema_name, table_name = parts[0], parts[1], parts[2]
                elif len(parts) == 2:
                    catalog, schema_name, table_name = 'default', parts[0], parts[1]
                else:
                    continue
                if tbl not in seen_tables:
                    seen_tables.add(tbl)
                    schema_details.append((catalog, schema_name, table_name, '_placeholder', 'STRING', f'Table used in use case {uc.get("No", "")}'))
            if cols_str:
                col_parts = [c.strip() for c in re.split(r'[,\n]+', cols_str) if c.strip()]
                for col_fqn in col_parts:
                    col_segments = col_fqn.split('.')
                    if len(col_segments) == 4:
                        schema_details.append((col_segments[0], col_segments[1], col_segments[2], col_segments[3], 'STRING', ''))
                    elif len(col_segments) == 1 and table_parts:
                        first_table = table_parts[0].strip().strip('`').split('.')
                        if len(first_table) == 3:
                            schema_details.append((first_table[0], first_table[1], first_table[2], col_segments[0], 'STRING', ''))
        return schema_details

    def _compute_per_uc_genie_fields(self, use_case: dict, full_schema_details: list, schema_index: dict = None) -> dict:
        """Compute the four per-use-case Genie-prompt fields that are expensive to rebuild.
        
        Single source of truth used by:
          - Discover mode (persists fields on the use case + tracking table)
          - Generate mode (if fields are not cached on the UC, recompute)
        
        Returns a dict with keys:
          directly_involved_schema, foreign_key_relationships, available_tables_listing,
          technique_implementation_guidance
        """
        tables_involved_str = use_case.get('Tables Involved', '') or ''
        directly_involved_tables = set()
        if tables_involved_str:
            table_parts = re.split(r'[,\s]+', tables_involved_str)
            for part in table_parts:
                part = part.strip().strip('`').strip()
                if part and '.' in part:
                    directly_involved_tables.add(part)
        
        uc_stated_tables = set(directly_involved_tables)
        _expanded_tables, fk_relationships = self._expand_tables_with_foreign_keys(directly_involved_tables)
        directly_involved_tables = uc_stated_tables

        validated_di_tables = set()
        if schema_index:
            for involved_table in directly_involved_tables:
                canonical = involved_table.replace('`', '')
                if involved_table in schema_index or canonical in schema_index:
                    validated_di_tables.add(canonical)
        elif full_schema_details:
            full_schema_table_names = {f"{d[0]}.{d[1]}.{d[2]}" for d in full_schema_details}
            for involved_table in directly_involved_tables:
                canonical = involved_table.replace('`', '')
                if canonical in full_schema_table_names:
                    validated_di_tables.add(canonical)
        else:
            validated_di_tables = {t.replace('`', '') for t in directly_involved_tables if '.' in t}
        
        if validated_di_tables:
            di_lines = [f"- `{t}`" for t in sorted(validated_di_tables)]
            directly_involved_schema = "\n".join(di_lines) + f"\n\n({len(validated_di_tables)} tables. You have direct access to inspect all column schemas. No column expansion is provided.)"
        else:
            directly_involved_schema = "No schema available"
        
        all_available_tables = set()
        if schema_index:
            for table_key in schema_index:
                clean_key = table_key.replace('`', '')
                if '.' in clean_key:
                    all_available_tables.add(clean_key)
        elif full_schema_details:
            for detail in full_schema_details:
                all_available_tables.add(f"{detail[0]}.{detail[1]}.{detail[2]}")
        directly_involved_clean = {t.replace('`', '') for t in directly_involved_tables}
        other_tables = sorted(all_available_tables - directly_involved_clean)
        if other_tables:
            listing_lines = [f"- {tbl}" for tbl in other_tables]
            available_tables_listing = (
                f"Primary tables (listed above): {', '.join(sorted(directly_involved_clean))}\n\n"
                f"Additional available tables ({len(other_tables)} tables -- you may use any for enrichment):\n"
                + "\n".join(listing_lines)
            )
        else:
            available_tables_listing = f"Only the primary tables are available: {', '.join(sorted(directly_involved_clean))}"
        
        fk_relationships_md = "\n".join([f"- {rel}" for rel in sorted(set(fk_relationships))]) if fk_relationships else "None"
        
        technique = use_case.get('Analytics Technique', 'AI Analysis')
        technique_guidance = TECHNIQUE_IMPLEMENTATION_GUIDANCE.get(
            technique, TECHNIQUE_IMPLEMENTATION_GUIDANCE.get('AI Analysis', '')
        )
        
        return {
            "directly_involved_schema": directly_involved_schema,
            "foreign_key_relationships": fk_relationships_md,
            "available_tables_listing": available_tables_listing,
            "technique_implementation_guidance": technique_guidance,
        }
    
    def _build_code_structure_mandate(self, technique: str, tables_involved_str: str) -> str:
        """Build the Code Structure mandate block for the Genie prompt. Deterministic from
        (technique, tables_involved) — not cached at session level because the train/inference
        variant interpolates use-case-specific values."""
        _needs_train_infer = technique in TRAIN_INFER_TECHNIQUES
        if _needs_train_infer:
            _default_data_source = (tables_involved_str or '').strip()
            return (
                "- **Code Structure -- Train/Inference Split (MANDATORY -- NON-NEGOTIABLE -- PASS/FAIL GATE):** "
                "This use case uses a TRAINABLE technique (`" + technique + "`). You MUST structure the notebook with SEPARATE training and inference paths. "
                "The notebook MUST contain the following functions:\n"
                "  (1) `def create_widgets():` -- MUST be defined and called FIRST (before any other logic). Creates 3 Databricks widgets using `dbutils.widgets`: "
                "`run_type` (dropdown with values `Training` and `Inference`, default `Training`), "
                "`training_data_source` (text, default `" + _default_data_source + "`), "
                "`inference_data_source` (text, default `" + _default_data_source + "`). "
                "Use `dbutils.widgets.dropdown('run_type', 'Training', ['Training', 'Inference'])` for the dropdown and "
                "`dbutils.widgets.text('training_data_source', '" + _default_data_source + "')` / "
                "`dbutils.widgets.text('inference_data_source', '" + _default_data_source + "')` for the text widgets. "
                "The `create_widgets()` call MUST be a top-level statement at module scope (this is the ONLY permitted top-level side-effect call).\n"
                "  (2) `def main_train():` -- Contains the FULL training pipeline: data extraction (reading from the table(s) specified in the `training_data_source` widget), "
                "feature engineering, train/test split, model training, evaluation metrics, "
                "MLflow experiment logging (log model with `infer_signature` and `input_example`, log metrics, log feature importance), "
                "ai_query enrichment of training results, MERGE INTO the result table, verification queries, and job cleanup. "
                "The training data source is read via `dbutils.widgets.get('training_data_source')`. "
                "The trained model MUST be registered in MLflow with a consistent model name derived from the use case (e.g., `{result_table}_model`) so `main_infer()` can load it.\n"
                "  (3) `def main_infer():` -- Contains the FULL inference pipeline: loads the LATEST registered model from MLflow (using the same model name from `main_train()`), "
                "reads NEW data from the table(s) specified in the `inference_data_source` widget via `dbutils.widgets.get('inference_data_source')`, "
                "applies the same feature engineering as training (reuse helper functions -- DRY), "
                "runs model.predict() on the new data, ai_query enrichment of inference results, MERGE INTO the SAME result table as training, "
                "verification queries, and job cleanup. "
                "If no registered model is found, `main_infer()` MUST raise a clear error: `raise RuntimeError('No trained model found. Run with run_type=Training first.')`.\n"
                "  ALLOWED at top level: import statements, function/class definitions, constant definitions (e.g., `OUTPUT_TABLE = \"...\"`), "
                "the `create_widgets()` call, and the `def create_widgets():` definition. "
                "FORBIDDEN at top level: `main_train()`, `main_infer()`, `if __name__`, or any other execution call. "
                "The Job Launcher cell (Section 8) reads the `run_type` widget and dispatches to `main_train()` or `main_infer()` accordingly. "
                "You MUST NOT create ANY cell that calls `main_train()` or `main_infer()` directly. "
                "The notebook must have EXACTLY this cell structure: "
                "(1) `%pip install` cell if needed, "
                "(2) one code cell with `create_widgets()` call + imports/constants/helpers/`def main_train():`/`def main_infer():` and ZERO execution of main functions, "
                "(3) the Job Launcher cell from Section 8 copied VERBATIM, "
                "(4) self-critique markdown cell. "
                "DRY PRINCIPLE: shared logic (feature engineering, data quality checks, ai_query prompt construction, table creation, job cleanup) "
                "MUST be extracted into helper functions called by both `main_train()` and `main_infer()`. Do NOT duplicate code between the two paths. "
                "Setting environment variables like `os.environ.setdefault('JOB_ID', '1')` to manipulate the Job Launcher is FORBIDDEN."
            )
        return (
            "- **Code Structure and Job Launcher (MANDATORY -- NON-NEGOTIABLE -- PASS/FAIL GATE):** "
            "You MUST wrap ALL executable code inside a `def main():` function containing ALL processing logic -- "
            "data extraction, feature engineering, model training, ai_query enrichment, table creation, MLflow registration, and verification queries. "
            "ALLOWED at top level: import statements, function/class definitions (including helper functions), "
            "and constant definitions (e.g., `OUTPUT_TABLE = \"...\"`, `N_CLUSTERS = 4`). "
            "A constant definition is a simple name-to-literal binding with NO side effects. "
            "FORBIDDEN at top level: ANY executable statement that has side effects, including but not limited to: "
            "`main()`, `if __name__ == \"__main__\": main()`, `result = main()`, "
            "`os.environ.setdefault(\"JOB_ID\", ...)`, `os.environ[...] = ...`, `print(...)`, `spark.sql(...)`, `mlflow.set_experiment(...)`, "
            "or any function call that modifies state, performs I/O, or triggers computation. "
            "The `main()` function is invoked EXCLUSIVELY by a separate Job Launcher cell whose EXACT code is provided in Section 8 below. "
            "You MUST create this Job Launcher as a separate cell titled \"Job Launcher\" and it MUST be your LAST code cell. "
            "You MUST NOT create ANY other cell that calls `main()`. "
            "The notebook must have EXACTLY this cell structure: "
            "(1) `%pip install` cell if needed, "
            "(2) one code cell with imports/constants/helpers/`def main():` and ZERO execution, "
            "(3) the Job Launcher cell from Section 8 copied VERBATIM, "
            "(4) self-critique markdown cell. "
            "If ANY cell other than the Job Launcher calls `main()`, or if the Job Launcher cell is missing or modified, that is an IMPLEMENTATION FAILURE. "
            "Setting environment variables like `os.environ.setdefault('JOB_ID', '1')` to manipulate the Job Launcher is also FORBIDDEN."
        )
    
    def _get_enriched_context_for_prompt(self) -> dict:
        """Collect the 6 enriched business-context fields (with defaults) used by the Genie prompt.
        Reads from self.merged_business_context (populated during Discover) or from a pre-loaded
        dict (used by Generate mode when replaying from the session table)."""
        enriched_ctx = getattr(self, 'merged_business_context', {}) or {}
        enriched_business_context = enriched_ctx.get('business_context', '') or self._get_business_context_fallback()
        
        raw_goals = enriched_ctx.get('strategic_goals', '')
        if isinstance(raw_goals, list):
            enriched_strategic_goals = ', '.join(raw_goals) if raw_goals else ''
        else:
            enriched_strategic_goals = str(raw_goals).strip() if raw_goals else ''
        if not enriched_strategic_goals or len(enriched_strategic_goals) < 10:
            enriched_strategic_goals = 'UNKNOWN_FROM_METADATA (strategic goals): downstream prompts MUST NOT invent goals; rely on schema + widget inputs only'
        
        raw_priorities = enriched_ctx.get('business_priorities', '')
        if isinstance(raw_priorities, list):
            enriched_business_priorities = ', '.join(raw_priorities) if raw_priorities else ''
        else:
            enriched_business_priorities = str(raw_priorities).strip() if raw_priorities else ''
        if not enriched_business_priorities or len(enriched_business_priorities) < 10:
            enriched_business_priorities = 'UNKNOWN_FROM_METADATA (business priorities): downstream prompts MUST NOT invent priorities; rely on schema + widget inputs only'
        
        raw_initiative = enriched_ctx.get('strategic_initiative', '')
        enriched_strategic_initiative = str(raw_initiative).strip() if raw_initiative else 'UNKNOWN_FROM_METADATA (strategic initiative)'
        
        raw_value_chain = enriched_ctx.get('value_chain', '')
        enriched_value_chain = str(raw_value_chain).strip() if raw_value_chain else 'UNKNOWN_FROM_METADATA (value chain)'
        
        raw_revenue_model = enriched_ctx.get('revenue_model', '')
        enriched_revenue_model = str(raw_revenue_model).strip() if raw_revenue_model else 'UNKNOWN_FROM_METADATA (revenue model)'
        
        return {
            "enriched_business_context": enriched_business_context,
            "enriched_strategic_goals": enriched_strategic_goals,
            "enriched_business_priorities": enriched_business_priorities,
            "enriched_strategic_initiative": enriched_strategic_initiative,
            "enriched_value_chain": enriched_value_chain,
            "enriched_revenue_model": enriched_revenue_model,
        }
    
    def _persist_session_prompt_context(self):
        """Persist the 7 per-session Genie-prompt context fields to __inspire_session so
        Generate mode can rebuild prompts without re-running business-context extraction.
        Called once after `merged_business_context` is populated. Best-effort (won't fail run)."""
        if not self.atomic_writer or not self.atomic_writer.enabled:
            return
        try:
            ctx = self._get_enriched_context_for_prompt()
            ctx["generation_instructions_section"] = self._get_generation_instruction_for_prompt(
                "USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"
            ) or ""
            self.atomic_writer.update_session_prompt_context(ctx)
        except Exception as e:
            self.logger.debug(f"Persisting session prompt context failed (non-critical): {get_clean_error_message(e)}")

    def _generate_genie_instruction_for_use_case(self, use_case: dict, full_schema_details: list, document_context: str, schema_index: dict = None) -> dict:
        import time
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')[:80]
        
        try:
            start_time = time.time()
            self.logger.info(f"🤖 [{use_case_id}] Starting Genie Code instruction generation...")
            
            tables_involved_str = use_case.get('Tables Involved', '')
            result_table = use_case.get('result_table', '')
            if not result_table:
                result_table = compute_result_table_name(use_case_id, use_case.get('Name', 'unknown'), tables_involved_str, inspire_database=self.inspire_database)
                use_case['result_table'] = result_table
            
            if use_case.get('_genie_fields_cached'):
                directly_involved_schema = use_case.get('directly_involved_schema') or 'No schema available'
                fk_relationships_md = use_case.get('foreign_key_relationships') or 'None'
                available_tables_listing = use_case.get('available_tables_listing') or ''
                technique_guidance = use_case.get('technique_implementation_guidance') or ''
                validated_di_tables = set()
            else:
                _per_uc_fields = self._compute_per_uc_genie_fields(use_case, full_schema_details, schema_index)
                directly_involved_schema = _per_uc_fields["directly_involved_schema"]
                fk_relationships_md = _per_uc_fields["foreign_key_relationships"]
                available_tables_listing = _per_uc_fields["available_tables_listing"]
                technique_guidance = _per_uc_fields["technique_implementation_guidance"]
                validated_di_tables = set()
            
            _enriched = self._get_enriched_context_for_prompt()
            enriched_business_context = _enriched["enriched_business_context"]
            enriched_strategic_goals = _enriched["enriched_strategic_goals"]
            enriched_business_priorities = _enriched["enriched_business_priorities"]
            enriched_strategic_initiative = _enriched["enriched_strategic_initiative"]
            enriched_value_chain = _enriched["enriched_value_chain"]
            enriched_revenue_model = _enriched["enriched_revenue_model"]
            technique = use_case.get('Analytics Technique', 'AI Analysis')
            code_structure_mandate = self._build_code_structure_mandate(technique, tables_involved_str)
            
            prompt_vars = {
                "use_case_id": use_case_id,
                "use_case_name": use_case.get('Name', ''),
                "business_domain": use_case.get('Business Domain', ''),
                "subdomain": use_case.get('Subdomain', ''),
                "use_case_type": use_case.get('type', ''),
                "analytics_technique": technique,
                "technique_implementation_guidance": technique_guidance,
                "statement": use_case.get('Statement', ''),
                "solution": use_case.get('Solution', ''),
                "business_value": use_case.get('Business Value', ''),
                "high_level_design": use_case.get('High Level Design', ''),
                "tables_involved": tables_involved_str,
                "primary_table": use_case.get('Primary Table', ''),
                "directly_involved_schema": directly_involved_schema,
                "foreign_key_relationships": fk_relationships_md,
                "available_tables_listing": available_tables_listing,
                "business_name": self.business_name,
                "enriched_business_context": enriched_business_context,
                "enriched_strategic_goals": enriched_strategic_goals,
                "enriched_business_priorities": enriched_business_priorities,
                "enriched_strategic_initiative": enriched_strategic_initiative,
                "enriched_value_chain": enriched_value_chain,
                "enriched_revenue_model": enriched_revenue_model,
                "result_table": result_table,
                "code_structure_mandate": code_structure_mandate,
                "generation_instructions_section": self._get_generation_instruction_for_prompt("USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT"),
            }
            
            prompt_vars_size = sum(len(str(v)) for v in prompt_vars.values())
            schema_size = len(directly_involved_schema)
            fk_size = len(fk_relationships_md)
            avail_size = len(available_tables_listing)
            self.logger.info(
                f"📏 [{use_case_id}] Genie prompt vars: ~{prompt_vars_size:,} chars total "
                f"(schema={schema_size:,}, fk={fk_size:,}, avail_tables={avail_size:,}, "
                f"tables={len(validated_di_tables)})"
            )
            
            _skip_cache = bool(use_case.get('_skip_llm_cache', False))
            genie_instruction = self.ai_agent.run_worker(
                step_name=f"Genie_Instruction_{use_case_id}",
                worker_prompt_path="USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT",
                prompt_vars=prompt_vars,
                response_schema=None,
                timeout_override=420,
                skip_cache=_skip_cache
            )
            
            elapsed = time.time() - start_time
            
            if genie_instruction and len(genie_instruction.strip()) > 100:
                genie_instruction = sanitize_llm_special_chars(genie_instruction.strip())
                job_launcher_section = self._build_job_launcher_instruction_section(use_case.get('Name', ''), technique)
                genie_instruction = genie_instruction + job_launcher_section
                use_case['genie_instruction'] = genie_instruction
                use_case['genie_instruction_escaped'] = self._escape_for_triple_quote_embedding(genie_instruction)
                use_case['genie_instruction_status'] = 'success'
                self.logger.info(f"✅ [{use_case_id}] Genie instruction generated with Job Launcher section ({len(genie_instruction):,} chars, {elapsed:.1f}s)")
            else:
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'failed'
                self.logger.error(f"❌ [{use_case_id}] Genie instruction generation returned empty/short response")
            
            return use_case
            
        except Exception as e:
            err_msg = get_clean_error_message(e)
            is_timeout = 'timeout' in err_msg.lower() or 'timed out' in err_msg.lower()
            if is_timeout:
                self.logger.error(f"⏱️ [{use_case_id}] Genie instruction generation timed out: {err_msg}")
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'timeout'
            else:
                self.logger.error(f"❌ [{use_case_id}] Genie instruction generation failed: {err_msg}")
                use_case['genie_instruction'] = ''
                use_case['genie_instruction_status'] = 'failed'
            return use_case

    def _build_notebook_workspace_paths(self, use_case: dict) -> tuple:
        """Return (notebook_dir, notebook_full_path, notebook_filename) for a use case.
        Single source of truth used by both full Genie notebooks (Generate mode) and
        skeleton notebooks (Discover mode) so path collisions are impossible."""
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')
        sanitized_domain = self._sanitize_name(domain)
        sanitized_subdomain = self._sanitize_name(subdomain)
        notebook_filename = f"{use_case_id} - {self._sanitize_name(use_case_name)}"
        notebook_dir = os.path.join(self.notebook_output_dir, sanitized_domain, sanitized_subdomain)
        notebook_full_path = os.path.join(notebook_dir, f"{notebook_filename}.ipynb")
        return notebook_dir, notebook_full_path, notebook_filename
    
    def _write_notebook_to_workspace(self, notebook_dir: str, notebook_full_path: str, notebook_obj: dict, use_case_id: str) -> str:
        """Serialize a notebook dict and import it into the workspace with retry.
        Shared by skeleton (Discover) and full Genie (Generate) notebook writers."""
        from databricks.sdk.service import workspace
        notebook_content_str = json.dumps(notebook_obj, indent=2)
        max_retries = self.max_retry_attempts + 1
        retry_delay = 2
        for attempt in range(1, max_retries + 1):
            try:
                self.w_client.workspace.mkdirs(notebook_dir)
                self.w_client.workspace.import_(
                    path=notebook_full_path, overwrite=True, format=workspace.ImportFormat.JUPYTER,
                    content=base64.b64encode(notebook_content_str.encode('utf-8')).decode(),
                )
                _status = self.w_client.workspace.get_status(notebook_full_path)
                abs_path = _status.path
                self.logger.info(f"✅ [{use_case_id}] Notebook saved: {abs_path}")
                return abs_path
            except Exception as err:
                self.logger.warning(f"[{use_case_id}] Notebook import attempt {attempt}/{max_retries} failed: {err}")
                if attempt < max_retries:
                    import time
                    time.sleep(retry_delay * (2 ** (attempt - 1)))
                else:
                    self.logger.error(f"❌ [{use_case_id}] Failed to import notebook after {max_retries} attempts")
                    return None
        return None
    
    def _build_use_case_details_markdown_source(self, use_case: dict) -> list:
        """Build the 'Use Case Details' markdown source list (shared by skeleton + genie notebooks)."""
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')
        def safe_val(val):
            if val is None or (isinstance(val, str) and not val.strip()):
                return 'N/A'
            return str(val)
        return [
            f"## Use Case Details\n\n",
            f"| Aspect | Description |\n",
            f"|---|---|\n",
            f"| **ID** | {safe_val(use_case_id)} |\n",
            f"| **Name** | {safe_val(use_case_name)} |\n",
            f"| **Business Domain** | {safe_val(domain)} |\n",
            f"| **Subdomain** | {safe_val(subdomain)} |\n",
            f"| **Type** | {safe_val(use_case.get('type'))} |\n",
            f"| **Analytics Technique** | {safe_val(use_case.get('Analytics Technique'))} |\n",
            f"| **Priority** | {safe_val(use_case.get('Priority'))} |\n",
            f"| **Primary Table** | {safe_val(use_case.get('Primary Table'))} |\n",
            f"| **Tables Involved** | {safe_val(use_case.get('Tables Involved'))} |\n",
            f"| **Statement** | {safe_val(use_case.get('Statement'))} |\n",
            f"| **Solution** | {safe_val(use_case.get('Solution'))} |\n",
            f"| **Business Value** | {safe_val(use_case.get('Business Value'))} |\n",
            f"| **Quality** | {safe_val(use_case.get('Quality'))} |\n",
            f"| **Result Table** | {safe_val(use_case.get('result_table'))} |\n",
        ]
    
    def _assemble_skeleton_notebook(self, use_case: dict) -> str:
        """Discover-mode notebook writer.
        
        Produces a 4-cell notebook:
          (1) Title + generation timestamp (markdown)
          (2) Use Case Details table (markdown, shared with full Genie notebook)
          (3) Marker code cell with INSPIRE_USE_CASE_ID, INSPIRE_SESSION_ID, and
              generate_genie_code_instruction = False. Setting the variable to True and
              re-running the notebook tells Route 2 of Generate mode to regenerate the
              full Genie Code Instruction for this use case.
          (4) Placeholder markdown explaining how to trigger Genie regeneration (via
              Inspire UI/SQL UPDATE for Route 1, or by flipping the marker variable for
              Route 2).
        """
        use_case_id = use_case.get('No', 'UNKNOWN')
        notebook_dir, notebook_full_path, notebook_filename = self._build_notebook_workspace_paths(use_case)
        
        final_cells = []
        generation_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        title_source = [
            f"# {use_case_id}: {use_case.get('Name', '')}\n\n",
            f"*Generated by Databricks Inspire AI on {generation_timestamp}*\n\n",
            "**Mode:** Discover Use Cases (skeleton notebook — no Genie Code Instruction yet)\n\n",
            "**Disclaimer:** All code instructions are AI-generated examples and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from this code.\n\n",
            "---\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": title_source
        })
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": self._build_use_case_details_markdown_source(use_case)
        })
        
        escaped_id = str(use_case_id).replace('"', '\\"')
        marker_source = [
            "# =============================================================================\n",
            "# INSPIRE GENIE CODE INSTRUCTION — GENERATION MARKER\n",
            "# =============================================================================\n",
            "# This notebook was created by Inspire AI in \"Discover Use Cases\" mode.\n",
            "# The Genie Code Instruction has NOT been generated yet.\n",
            "#\n",
            "# TWO WAYS TO REGENERATE THIS USE CASE IN \"Generate Use Cases\" MODE:\n",
            "#\n",
            "#   (1) Database flag route (preferred, driven by the Inspire UI):\n",
            "#       UPDATE <inspire_database>.__inspire_usecases\n",
            "#          SET generate_genie_code_instruction = 'Yes'\n",
            "#        WHERE id = <this use_case_id> AND session_id = <this session_id>;\n",
            "#       Then re-run the Inspire notebook with Operation = 'Generate Use Cases'.\n",
            "#\n",
            "#   (2) Notebook flag route (fallback when no rows are flagged in the DB):\n",
            "#       Flip the variable below from False -> True, save the notebook, then\n",
            "#       re-run the Inspire notebook with Operation = 'Generate Use Cases'.\n",
            "#\n",
            "# After generation, the Inspire pipeline rewrites this cell with the full\n",
            "# Genie Code Instruction and resets the flag back to False.\n",
            "# =============================================================================\n",
            "\n",
            f'INSPIRE_USE_CASE_ID = "{escaped_id}"\n',
            f"INSPIRE_SESSION_ID = {self.session_id}\n",
            "generate_genie_code_instruction = False\n",
        ]
        final_cells.append({
            "cell_type": "code",
            "execution_count": 0,
            "outputs": [],
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": marker_source
        })
        
        placeholder_source = [
            "## Next Steps\n\n",
            "This is a **skeleton notebook**. To produce the full Genie Code Instruction:\n\n",
            "1. Review the use case details above and confirm this is a use case you want to implement.\n",
            "2. Mark it for generation using **one** of the routes documented in the marker cell.\n",
            "3. Re-run the Inspire notebook with `Operation = 'Generate Use Cases'`.\n\n",
            "Inspire will then generate a full Genie Code Instruction for this use case and replace\n",
            "the marker cell with the executable instruction block.\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": placeholder_source
        })
        
        notebook_metadata = {
            "application/vnd.databricks.v1+notebook": {
                "computePreferences": None,
                "dashboards": [],
                "environmentMetadata": {"base_environment": "", "environment_version": "4"},
                "inputWidgetPreferences": None,
                "language": "python",
                "notebookMetadata": {"pythonIndentUnit": 4},
                "notebookName": notebook_filename,
                "widgets": {}
            },
            "language_info": {"name": "python"}
        }
        notebook_obj = {"cells": final_cells, "metadata": notebook_metadata, "nbformat": 4, "nbformat_minor": 0}
        return self._write_notebook_to_workspace(notebook_dir, notebook_full_path, notebook_obj, use_case_id)

    def _assemble_genie_code_notebook(self, use_case: dict, translations: dict):
        use_case_id = use_case.get('No', 'UNKNOWN')
        use_case_name = use_case.get('Name', '')
        domain = use_case.get('Business Domain', 'Other')
        subdomain = use_case.get('Subdomain', 'General')
        genie_instruction_escaped = use_case.get('genie_instruction_escaped', '')
        if not genie_instruction_escaped:
            _raw = sanitize_llm_special_chars(use_case.get('genie_instruction', ''))
            genie_instruction_escaped = self._escape_for_triple_quote_embedding(_raw) if _raw else ''
        
        if not genie_instruction_escaped:
            self.logger.warning(f"[{use_case_id}] No Genie instruction to write -- skipping notebook")
            return None
        
        notebook_dir, notebook_full_path, notebook_filename = self._build_notebook_workspace_paths(use_case)
        
        final_cells = []
        
        generation_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        title_source = [
            f"# {use_case_id}: {use_case_name}\n\n",
            f"*Generated by Databricks Inspire AI on {generation_timestamp}*\n\n",
            "**Disclaimer:** All code instructions are AI-generated examples and should be reviewed by a qualified engineer before being used in production. Databricks is not liable for any issues arising from this code.\n\n",
            "---\n"
        ]
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": title_source
        })
        final_cells.append({
            "cell_type": "markdown",
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": self._build_use_case_details_markdown_source(use_case)
        })
        
        instruction_header = (
            '# =============================================================================\n'
            '# GENIE CODE INSTRUCTION\n'
            '# =============================================================================\n'
            '# HOW TO USE:\n'
            '# 1. Open Genie Code in your Databricks workspace (side panel in any notebook)\n'
            '# 2. Copy the ENTIRE content of the instruction below (between the triple quotes)\n'
            '# 3. Paste it into Genie Code and let Genie generate the implementation\n'
            '# 4. Genie will create the complete code — review, iterate, and execute\n'
            '# =============================================================================\n'
            '\n'
            'genie_instruction = """\n'
        )
        instruction_footer = '\n"""\n\nprint(genie_instruction)\n'
        
        instruction_cell_source = [instruction_header, genie_instruction_escaped, instruction_footer]
        final_cells.append({
            "cell_type": "code",
            "execution_count": 0,
            "outputs": [],
            "metadata": {"application/vnd.databricks.v1+cell": {"nuid": str(uuid.uuid4())}},
            "source": instruction_cell_source
        })
        
        notebook_metadata = {
            "application/vnd.databricks.v1+notebook": {
                "computePreferences": None,
                "dashboards": [],
                "environmentMetadata": {
                    "base_environment": "",
                    "environment_version": "4"
                },
                "inputWidgetPreferences": None,
                "language": "python",
                "notebookMetadata": {
                    "pythonIndentUnit": 4
                },
                "notebookName": notebook_filename,
                "widgets": {}
            },
            "language_info": {
                "name": "python"
            }
        }
        
        notebook_obj = {"cells": final_cells, "metadata": notebook_metadata, "nbformat": 4, "nbformat_minor": 0}
        return self._write_notebook_to_workspace(notebook_dir, notebook_full_path, notebook_obj, use_case_id)

    def _run_genie_wave(self, wave_id: int, backlog: list, full_schema_details: list,
                        document_context: str, schema_by_table: dict, translations: dict,
                        wave_workers: int, on_uc_ready=None) -> tuple:
        import time
        
        succeeded = []
        timed_out = []
        _processed_futures = set()
        
        def _process_use_case_genie(uc):
            uc_id = uc.get('No', 'UNKNOWN')
            try:
                uc = self._generate_genie_instruction_for_use_case(uc, full_schema_details, document_context, schema_index=schema_by_table)
                
                if uc.get('genie_instruction_status') == 'success':
                    return (uc_id, uc, True, None)
                else:
                    return (uc_id, uc, False, None)
            except Exception as e:
                self.logger.error(f"❌ [{uc_id}] Genie pipeline failed: {get_clean_error_message(e)}")
                uc['genie_instruction'] = ''
                uc['genie_instruction_status'] = 'failed'
                return (uc_id, uc, False, None)
        
        def _handle_completed_future(future, uc_ref):
            uc_id = uc_ref.get('No', 'UNKNOWN')
            try:
                result_id, result_uc, success, nb_path = future.result(timeout=10)
                if success:
                    result_uc['genie_instruction_status'] = 'success'
                    succeeded.append(result_uc)
                    if on_uc_ready:
                        on_uc_ready(result_uc, True)
                else:
                    is_timeout = 'timeout' in str(result_uc.get('genie_instruction_status', '')).lower() or \
                                 'timed out' in str(result_uc.get('genie_instruction', '')).lower()
                    if is_timeout:
                        result_uc['genie_instruction_status'] = 'timeout'
                        timed_out.append(result_uc)
                    else:
                        timed_out.append(result_uc)
                        if on_uc_ready:
                            on_uc_ready(result_uc, False)
            except Exception as e:
                msg = str(e)
                is_timeout = 'timeout' in msg.lower() or 'timed out' in msg.lower()
                if is_timeout:
                    self.logger.warning(f"⏱️ [{uc_id}] Genie instruction timed out in wave {wave_id}")
                    uc_ref['genie_instruction_status'] = 'timeout'
                else:
                    self.logger.error(f"❌ [{uc_id}] Genie instruction failed in wave {wave_id}: {msg[:100]}")
                    uc_ref['genie_instruction_status'] = 'failed'
                    if on_uc_ready:
                        on_uc_ready(uc_ref, False)
                uc_ref['genie_instruction'] = ''
                timed_out.append(uc_ref)
        
        _wave_timeout = max(600, len(backlog) * 400)
        
        with _SafeExecutorContext(max_workers=wave_workers, thread_name_prefix=f"GenieWave{wave_id}", logger=self.logger, name=f"GenieWave{wave_id}") as ctx:
            future_to_uc = {}
            for uc in backlog:
                future = ctx.submit(_process_use_case_genie, uc)
                future_to_uc[future] = uc
            
            try:
                for future in safe_as_completed(list(future_to_uc.keys()), timeout=_wave_timeout):
                    _processed_futures.add(future)
                    _handle_completed_future(future, future_to_uc[future])
            except concurrent.futures.TimeoutError:
                ctx.mark_timed_out()
                self.logger.error(f"⏱️ Genie wave {wave_id} total timeout after {_wave_timeout}s")
                _salvaged = 0
                for future, uc_ref in future_to_uc.items():
                    if future in _processed_futures:
                        continue
                    if future.done():
                        _salvaged += 1
                        self.logger.info(f"🔧 [{uc_ref.get('No', '?')}] Salvaging done-but-unyielded future after wave timeout")
                        _handle_completed_future(future, uc_ref)
                    else:
                        future.cancel()
                        uc_ref['genie_instruction'] = ''
                        uc_ref['genie_instruction_status'] = 'timeout'
                        timed_out.append(uc_ref)
                if _salvaged:
                    self.logger.info(f"🔧 Salvaged {_salvaged} completed futures that were missed by timeout race")
        
        return succeeded, timed_out

    def _run_discover_skeleton_phase(self, all_use_cases: list, full_schema_details: list) -> list:
        """Discover-mode replacement for Phase 5b.
        
        For each use case (in parallel):
          1. Compute the 4 per-UC Genie prompt fields (directly_involved_schema,
             foreign_key_relationships, available_tables_listing,
             technique_implementation_guidance) and attach them to the use_case dict
             so the tracking-table MERGE persists them for later Generate-mode runs.
          2. Set generate_genie_code_instruction = 'No'.
          3. Write the 4-cell skeleton notebook into the workspace.
          4. Attach notebook_path so the detail-notebook batch links correctly and
             the tracking table records the skeleton's workspace path.
        
        No LLM calls are made — this phase is CPU-bound and safe to parallelize
        aggressively.
        """
        import time
        
        if not all_use_cases:
            self.logger.warning("No use cases provided for Discover skeleton phase")
            return []
        
        log_print(f"\n{'='*80}")
        log_print(f"🔎 DISCOVER MODE — SKELETON NOTEBOOK GENERATION")
        log_print(f"{'='*80}")
        log_print(f"📊 Total use cases: {len(all_use_cases)}")
        log_print(f"📝 Strategy: capture Genie prompt fields + write 4-cell skeleton notebook (no LLM calls)")
        log_print(f"{'='*80}\n")
        
        _step_id = self._emit_pipeline_status(
            stage_name="Discover Skeletons",
            step_name="Generation",
            sub_step_name="Started",
            message=f"Generating {len(all_use_cases)} skeleton notebooks + capturing per-UC Genie prompt fields",
            status="started",
            progress_increment=1.0
        )
        
        start_time = time.time()
        
        schema_by_table = defaultdict(list)
        try:
            for detail in full_schema_details:
                (catalog, schema, table, column_name, data_type, comment) = detail
                fqtn = f"{catalog}.{schema}.{table}"
                schema_by_table[fqtn].append(detail)
                schema_by_table[f"`{catalog}`.`{schema}`.`{table}`"] = schema_by_table[fqtn]
            self.logger.info(f"   ✓ Schema index built with {len(schema_by_table)} table entries")
        except Exception as _idx_err:
            self.logger.warning(f"Schema index build failed ({get_clean_error_message(_idx_err)}); per-UC schema fallback will run")
        
        total = len(all_use_cases)
        completed = 0
        failed = 0
        _lock = threading.Lock()
        
        def _process(idx_uc):
            nonlocal completed, failed
            idx, uc = idx_uc
            uc_id = uc.get('No', 'UNKNOWN')
            try:
                fields = self._compute_per_uc_genie_fields(uc, full_schema_details, schema_by_table)
                uc['directly_involved_schema'] = fields.get('directly_involved_schema', '')
                uc['foreign_key_relationships'] = fields.get('foreign_key_relationships', '')
                uc['available_tables_listing'] = fields.get('available_tables_listing', '')
                uc['technique_implementation_guidance'] = fields.get('technique_implementation_guidance', '')
                uc['generate_genie_code_instruction'] = 'No'
                uc['has_genie_code'] = 'N'
                nb_path = self._assemble_skeleton_notebook(uc)
                if nb_path:
                    uc['notebook_path'] = nb_path
                with _lock:
                    completed += 1
                    _cnt = completed + failed
                self.logger.info(f"🔎 [{_cnt}/{total}] Discover skeleton ready for {uc_id}: {nb_path or '(no notebook)'}")
                return True
            except Exception as e:
                with _lock:
                    failed += 1
                    _cnt = completed + failed
                self.logger.error(f"❌ [{_cnt}/{total}] Discover skeleton failed for {uc_id}: {get_clean_error_message(e)}")
                uc.setdefault('generate_genie_code_instruction', 'No')
                return False
        
        nb_parallelism, reason = calculate_adaptive_parallelism(
            "discover_skeleton", self.max_parallelism,
            num_items=total,
            is_llm_operation=False, logger=self.logger,
            estimated_mb_per_worker=2.0
        )
        log_adaptive_parallelism_decision("discover_skeleton", nb_parallelism, self.max_parallelism, reason)
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=nb_parallelism, thread_name_prefix="DiscoverSkel") as executor:
            list(executor.map(_process, enumerate(all_use_cases, start=1)))
        
        elapsed = time.time() - start_time
        log_print(f"\n{'='*80}")
        log_print(f"✅ DISCOVER MODE — SKELETON GENERATION COMPLETE")
        log_print(f"{'='*80}")
        log_print(f"   ✅ Skeletons OK:  {completed}/{total}")
        if failed:
            log_print(f"   ❌ Skeletons Fail: {failed}/{total}")
        log_print(f"   ⏱️  Total time: {elapsed:.1f}s")
        log_print(f"{'='*80}\n")
        
        self._emit_pipeline_status(
            stage_name="Discover Skeletons",
            step_name="Generation",
            sub_step_name="Complete",
            message=f"Discover skeletons: {completed} success, {failed} failed",
            status="ended_success" if failed == 0 else "ended_warning",
            progress_increment=2.0,
            result_json={"success": completed, "failed": failed, "total_use_cases": total, "elapsed_seconds": round(elapsed, 1)},
            step_id=_step_id
        )
        
        _ggc_raw = str(getattr(self, "generate_genie_code_for", "5") or "5").strip().lower()
        if _ggc_raw == "all":
            _auto_top_n = len(all_use_cases)
        else:
            try:
                _auto_top_n = max(0, int(_ggc_raw))
            except ValueError:
                _auto_top_n = 5
        if _auto_top_n and _auto_top_n > 0 and all_use_cases:
            def _ag_pri(uc):
                """Rank key for Auto-Genie top-N. Prefer numeric scores; do NOT float() the Priority *label* (e.g. 'Very High')."""
                try:
                    for _k in ("priority_score", "bob_score", "BoB Score", "Inspire Score", "inspire_score"):
                        _v = uc.get(_k)
                        if _v is None or (isinstance(_v, str) and str(_v).strip() == ""):
                            continue
                        return float(_v)
                    _p = uc.get("Priority")
                    if isinstance(_p, (int, float)):
                        return float(_p)
                    _is = uc.get("Inspire Score")
                    if _is is not None and str(_is).strip() != "":
                        return float(_is)
                except Exception:
                    pass
                return 0.0
            _top = sorted(all_use_cases, key=_ag_pri, reverse=True)[:_auto_top_n]
            if _top:
                log_print(f"\n{'='*80}")
                log_print(f"🚀 DISCOVER AUTO-GENIE — top {len(_top)} use case(s) by numeric rank (priority_score / BoB / Inspire Score)")
                log_print(f"{'='*80}")
                for _i, _uc in enumerate(_top, start=1):
                    log_print(f"   {_i}. {_uc.get('No','?')} (rank={_ag_pri(_uc):.4f}): {_uc.get('Name','')[:80]}")
                for _uc in _top:
                    _uc['_genie_fields_cached'] = True
                    _uc['_source_session_id'] = self.session_id
                try:
                    _translations = self.translation_service.get_translations("English")
                except Exception as _te:
                    self.logger.warning(f"Auto-Genie: translations unavailable: {get_clean_error_message(_te)}; empty dict")
                    _translations = {}
                _rt_ag = TECHNICAL_CONTEXT["runtime"]
                _ag_parallelism = min(len(_top), _rt_ag.get("genie_regen_parallelism", 10))
                from concurrent.futures import ThreadPoolExecutor, as_completed
                import threading as _threading_ag
                _ag_lock = _threading_ag.Lock()
                _ag_ok = 0
                _ag_fail = 0
                _ag_succeeded_ids = []

                def _ag_regen(_idx_uc):
                    _idx, _uc = _idx_uc
                    _uc_id = _uc.get('No', 'UNKNOWN')
                    try:
                        self.logger.info(f"🤖 [Discover Auto-Genie] ({_idx}/{len(_top)}) Regenerating Genie for {_uc_id}...")
                        _uc = self._generate_genie_instruction_for_use_case(_uc, [], '', schema_index=None)
                        if not _uc.get('genie_instruction'):
                            self.logger.error(f"❌ [Auto-Genie] {_uc_id}: empty genie_instruction")
                            return (None, _uc_id)
                        _nb_path = self._assemble_genie_code_notebook(_uc, _translations)
                        if _nb_path:
                            _uc['notebook_path'] = _nb_path
                        self.logger.info(f"✅ [Auto-Genie] {_uc_id}: notebook {_nb_path}")
                        return (_uc, _uc_id)
                    except Exception as _e:
                        self.logger.error(f"❌ [Auto-Genie] {_uc_id} failed: {get_clean_error_message(_e)}")
                        return (None, _uc_id)

                with ThreadPoolExecutor(max_workers=max(1, _ag_parallelism), thread_name_prefix="DiscoverAutoGenie") as _ex_ag:
                    _futs_ag = [_ex_ag.submit(_ag_regen, (_i, _u)) for _i, _u in enumerate(_top, start=1)]
                    for _f in as_completed(_futs_ag):
                        _result_uc, _result_id = _f.result()
                        with _ag_lock:
                            if _result_uc is not None:
                                _ag_ok += 1
                                _ag_succeeded_ids.append(_result_id)
                                for _k, _orig in enumerate(all_use_cases):
                                    if _orig.get('No') == _result_id:
                                        all_use_cases[_k].update(_result_uc)
                                        all_use_cases[_k]['has_genie_code'] = 'Y'
                                        all_use_cases[_k]['generate_genie_code_instruction'] = 'No'
                                        break
                            else:
                                _ag_fail += 1

                if _ag_succeeded_ids and getattr(self, 'atomic_writer', None) and self.atomic_writer and self.atomic_writer.enabled:
                    try:
                        _id_list_ag = ", ".join(f"'{_i}'" for _i in _ag_succeeded_ids)
                        self.spark.sql(
                            f"UPDATE {self._tracking_table_name} "
                            f"SET generate_genie_code_instruction = 'No', has_genie_code = 'Y' "
                            f"WHERE session_id = {int(self.session_id)} AND id IN ({_id_list_ag})"
                        )
                        self.logger.info(f"🔁 Auto-Genie: marked has_genie_code='Y' for {len(_ag_succeeded_ids)} UC(s)")
                    except Exception as _ue:
                        self.logger.warning(f"Auto-Genie DB update failed non-fatally: {get_clean_error_message(_ue)}")

                log_print(f"   ✅ Auto-Genie OK:   {_ag_ok}/{len(_top)}")
                if _ag_fail:
                    log_print(f"   ❌ Auto-Genie Fail: {_ag_fail}/{len(_top)}")
                log_print(f"{'='*80}\n")

                try:
                    self._emit_pipeline_status(
                        stage_name="Discover Skeletons",
                        step_name="Auto-Genie (Top-N)",
                        sub_step_name="Complete",
                        message=f"Auto-Genie top-{len(_top)}: {_ag_ok} OK, {_ag_fail} failed",
                        status="ended_success" if _ag_fail == 0 else "ended_warning",
                        progress_increment=1.0,
                        result_json={"top_n": len(_top), "success": _ag_ok, "failed": _ag_fail, "ids": _ag_succeeded_ids}
                    )
                except Exception:
                    pass

        return all_use_cases

    # ==== Generate Use Cases mode ====
    
    _UC_COLUMN_TO_FIELD = {
        'id': 'No',
        'business_domain': 'Business Domain',
        'subdomain': 'Subdomain',
        'use_case': 'usecase',
        'description': 'Name',
        'inspire_score': 'Inspire Score',
        'generated': 'generated',
        'type': 'type',
        'analytics_technique': 'Analytics Technique',
        'business_priority_alignment': 'Business Priority Alignment',
        'strategic_goals_alignment': 'Strategic Goals Alignment',
        'statement': 'Statement',
        'solution': 'Solution',
        'business_value': 'Business Value',
        'beneficiary': 'Beneficiary',
        'sponsor': 'Sponsor',
        'tables_involved': 'Tables Involved',
        'primary_table': 'Primary Table',
        'strategic_alignment': 'Strategic Alignment',
        'roi': 'Return on Investment',
        'reusability': 'Reusability',
        'time_to_value': 'Time to Value',
        'data_availability': 'Data Availability',
        'data_accessibility': 'Data Accessibility',
        'architecture_fitness': 'Architecture Fitness',
        'team_skills': 'Team Skills',
        'domain_knowledge': 'Domain Knowledge',
        'people_allocation': 'People Allocation',
        'budget_allocation': 'Budget Allocation',
        'time_to_production': 'Time to Production',
        'value_score': 'Value',
        'feasibility_score': 'Feasibility',
        'priority_score': 'Priority',
        'quality_score': 'Quality Score',
        'priority_reasons': 'Justification',
        'quality_reasons': 'Quality Summary',
        'result_database': 'result_table',
        'notebook_path': 'notebook_path',
        'high_level_design': 'High Level Design',
        'genie_instruction': 'genie_instruction',
        'generate_genie_code_instruction': 'generate_genie_code_instruction',
        'directly_involved_schema': 'directly_involved_schema',
        'foreign_key_relationships': 'foreign_key_relationships',
        'available_tables_listing': 'available_tables_listing',
        'technique_implementation_guidance': 'technique_implementation_guidance',
    }
    
    def _row_to_uc(self, row) -> dict:
        """Inverse of _uc_to_row: reconstruct a use_case dict from a __inspire_usecases Row.
        Marks the dict with _genie_fields_cached=True so _generate_genie_instruction_for_use_case
        replays the cached 4 per-UC Genie prompt fields instead of recomputing against schema."""
        row_d = row.asDict() if hasattr(row, 'asDict') else dict(row)
        uc = {}
        for col, field in self._UC_COLUMN_TO_FIELD.items():
            val = row_d.get(col)
            if val is not None:
                uc[field] = val
        uc['_genie_fields_cached'] = True
        uc['_source_session_id'] = row_d.get('session_id')
        return uc
    
    def _load_session_context(self, session_id) -> dict:
        """Load the per-session Genie-prompt context from __inspire_session for replay."""
        if not self.atomic_writer or not self.atomic_writer.enabled:
            return {}
        session_table = self.atomic_writer.session_table
        if not session_table:
            return {}
        try:
            df = self.spark.sql(
                f"SELECT business_name, enriched_business_context, enriched_strategic_goals, "
                f"enriched_business_priorities, enriched_strategic_initiative, enriched_value_chain, "
                f"enriched_revenue_model, generation_instructions_section "
                f"FROM {session_table} WHERE session_id = {int(session_id)} LIMIT 1"
            )
            rows = df.collect()
            if not rows:
                self.logger.warning(f"No session row found for session_id={session_id}")
                return {}
            return rows[0].asDict()
        except Exception as e:
            self.logger.warning(f"Failed to load session context for session_id={session_id}: {get_clean_error_message(e)}")
            return {}
    
    def _apply_session_context(self, session_ctx: dict):
        """Apply a loaded session context onto self so _get_enriched_context_for_prompt works.
        In Generate mode we overwrite merged_business_context, business_name and generation_instructions."""
        if not session_ctx:
            return
        business_name = session_ctx.get('business_name')
        if business_name:
            self.business_name = business_name
        self.merged_business_context = {
            'business_context': session_ctx.get('enriched_business_context', '') or '',
            'strategic_goals': session_ctx.get('enriched_strategic_goals', '') or '',
            'business_priorities': session_ctx.get('enriched_business_priorities', '') or '',
            'strategic_initiative': session_ctx.get('enriched_strategic_initiative', '') or '',
            'value_chain': session_ctx.get('enriched_value_chain', '') or '',
            'revenue_model': session_ctx.get('enriched_revenue_model', '') or '',
        }
        gen_instr_section = session_ctx.get('generation_instructions_section', '') or ''
        if gen_instr_section and gen_instr_section.strip() and gen_instr_section != '*(No additional generation instructions provided)*':
            self.parsed_generation_instructions = {
                'per_prompt_instructions': {
                    'USE_CASE_GENIE_CODE_INSTRUCTION_GEN_PROMPT': gen_instr_section
                }
            }
        else:
            self.parsed_generation_instructions = getattr(self, 'parsed_generation_instructions', {}) or {}
    
    def _fetch_flagged_use_cases_route1(self) -> list:
        """Route 1: return all rows from __inspire_usecases where generate_genie_code_instruction='Yes'.
        No session filter — the flag IS the selector."""
        if not self._tracking_table_name:
            return []
        try:
            df = self.spark.sql(
                f"SELECT * FROM {self._tracking_table_name} "
                f"WHERE UPPER(COALESCE(generate_genie_code_instruction, 'No')) = 'YES'"
            )
            return df.collect()
        except Exception as e:
            self.logger.error(f"Route 1 DB scan failed: {get_clean_error_message(e)}")
            return []
    
    def _scan_notebooks_for_generation_flag(self) -> list:
        """Route 2: walk the generation_path tree, parse each notebook, and return entries
        whose marker cell has generate_genie_code_instruction = True.
        Returns list of dicts: {'notebook_path', 'use_case_id', 'session_id'}."""
        from databricks.sdk.service import workspace as _ws_svc
        import ast
        
        root = getattr(self, 'notebook_output_dir', None)
        if not root:
            return []
        found = []
        
        def _walk(path: str):
            try:
                entries = list(self.w_client.workspace.list(path))
            except Exception as e:
                self.logger.debug(f"Route 2 list({path}) failed: {get_clean_error_message(e)}")
                return
            for entry in entries:
                obj_type = getattr(entry, 'object_type', None)
                entry_path = getattr(entry, 'path', None)
                if not entry_path:
                    continue
                if obj_type == _ws_svc.ObjectType.DIRECTORY:
                    _walk(entry_path)
                elif obj_type == _ws_svc.ObjectType.NOTEBOOK:
                    info = _parse_notebook(entry_path)
                    if info:
                        found.append(info)
        
        def _parse_notebook(nb_path: str):
            try:
                exported = self.w_client.workspace.export(path=nb_path, format=_ws_svc.ExportFormat.SOURCE)
                content_b64 = getattr(exported, 'content', None)
                if not content_b64:
                    return None
                source = base64.b64decode(content_b64).decode('utf-8', errors='ignore')
            except Exception as e:
                self.logger.debug(f"Route 2 export({nb_path}) failed: {get_clean_error_message(e)}")
                return None
            if 'generate_genie_code_instruction' not in source:
                return None
            flag_val, uc_id, sess_id = None, None, None
            try:
                tree = ast.parse(source)
                for node in ast.walk(tree):
                    if not isinstance(node, ast.Assign):
                        continue
                    if not node.targets or not isinstance(node.targets[0], ast.Name):
                        continue
                    name = node.targets[0].id
                    val_node = node.value
                    if name == 'generate_genie_code_instruction' and isinstance(val_node, ast.Constant):
                        flag_val = val_node.value
                    elif name == 'INSPIRE_USE_CASE_ID' and isinstance(val_node, ast.Constant):
                        uc_id = val_node.value
                    elif name == 'INSPIRE_SESSION_ID' and isinstance(val_node, ast.Constant):
                        sess_id = val_node.value
            except SyntaxError:
                import re
                m_flag = re.search(r"^\s*generate_genie_code_instruction\s*=\s*(True|False)\s*$", source, re.M)
                m_id = re.search(r"^\s*INSPIRE_USE_CASE_ID\s*=\s*['\"]([^'\"]+)['\"]\s*$", source, re.M)
                m_sess = re.search(r"^\s*INSPIRE_SESSION_ID\s*=\s*(\d+)\s*$", source, re.M)
                if m_flag:
                    flag_val = (m_flag.group(1) == 'True')
                if m_id:
                    uc_id = m_id.group(1)
                if m_sess:
                    sess_id = int(m_sess.group(1))
            if flag_val is True and uc_id:
                return {'notebook_path': nb_path, 'use_case_id': uc_id, 'session_id': sess_id}
            return None
        
        _walk(root)
        return found
    
    def _fetch_flagged_use_cases_route2(self) -> list:
        """Route 2 wrapper: scan notebooks, cross-reference DB by (id, session_id) to reload full row.

        Phase K: batched SQL. Instead of N queries (one per flagged entry), we
        issue a single query using an IN-list over (id, session_id) tuples plus
        a fallback single-query for entries without a session_id. Result: 1-2
        Spark round-trips regardless of flagged entry count."""
        entries = self._scan_notebooks_for_generation_flag()
        if not entries:
            return []
        if not self._tracking_table_name:
            self.logger.warning("Route 2 found flagged notebooks but tracking table not configured — cannot enrich.")
            return []

        # Split entries by whether they have a session_id (tuple-match) or not (id-only latest).
        with_sess = []  # list of (uc_id, sess_id)
        without_sess = []  # list of uc_id
        for e in entries:
            uc_id = e.get('use_case_id')
            sess_id = e.get('session_id')
            if not uc_id:
                continue
            if sess_id is None:
                without_sess.append(str(uc_id))
            else:
                with_sess.append((str(uc_id), int(sess_id)))

        rows = []
        # Batched (id, session_id) IN-list
        if with_sess:
            try:
                def _esc(s: str) -> str:
                    return str(s).replace("'", "''")
                tuples_sql = ", ".join(
                    [f"('{_esc(uid)}', {int(sid)})" for uid, sid in with_sess]
                )
                df = self.spark.sql(
                    f"SELECT * FROM {self._tracking_table_name} "
                    f"WHERE (id, session_id) IN ({tuples_sql})"
                )
                rows.extend(df.collect())
            except Exception as ex:
                self.logger.warning(
                    f"Route 2 batched lookup failed ({len(with_sess)} entries): "
                    f"{get_clean_error_message(ex)}; falling back to per-entry queries"
                )
                for uid, sid in with_sess:
                    try:
                        df = self.spark.sql(
                            f"SELECT * FROM {self._tracking_table_name} "
                            f"WHERE id = '{uid.replace(chr(39), chr(39)+chr(39))}' AND session_id = {int(sid)} LIMIT 1"
                        )
                        fetched = df.collect()
                        if fetched:
                            rows.append(fetched[0])
                    except Exception as ex2:
                        self.logger.warning(f"Route 2 per-entry fallback failed for {uid}/{sid}: {get_clean_error_message(ex2)}")

        # Batched id-only (latest session) via window function
        if without_sess:
            try:
                def _esc(s: str) -> str:
                    return str(s).replace("'", "''")
                id_list = ", ".join([f"'{_esc(uid)}'" for uid in without_sess])
                df = self.spark.sql(
                    f"SELECT * FROM ("
                    f"  SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY session_id DESC) AS _rn "
                    f"  FROM {self._tracking_table_name} WHERE id IN ({id_list})"
                    f") WHERE _rn = 1"
                )
                rows.extend([r for r in df.collect() if r])
            except Exception as ex:
                self.logger.warning(
                    f"Route 2 id-only batched lookup failed ({len(without_sess)} entries): "
                    f"{get_clean_error_message(ex)}"
                )

        # Log any entries we couldn't match (for operator visibility)
        try:
            fetched_ids = {str(r["id"]) for r in rows if r and "id" in r.asDict()}
            missing = [uid for uid, _ in with_sess if uid not in fetched_ids] +                       [uid for uid in without_sess if uid not in fetched_ids]
            for miss in missing[:20]:
                self.logger.warning(f"Route 2: flagged id={miss} not found in tracking table")
        except Exception:
            pass

        return rows
    
    def _update_flag_to_no(self, session_id_to_ids: dict):
        """UPDATE generate_genie_code_instruction='No' for every (session_id, id) pair in the dict."""
        if not self._tracking_table_name or not session_id_to_ids:
            return
        for sess_id, ids in session_id_to_ids.items():
            if not ids:
                continue
            # v0.8.9 SQL safety: escape single quotes in id values to match the contract used by
            # _merge_regenerated_ucs_into_tracking. UC ids are typically 'N01-U03' style but the
            # column accepts arbitrary user-supplied strings, so escape unconditionally.
            id_list = ", ".join([f"'{str(_id).replace(chr(39), chr(39)+chr(39))}'" for _id in ids])
            try:
                self.spark.sql(
                    f"UPDATE {self._tracking_table_name} "
                    f"SET generate_genie_code_instruction = 'No', has_genie_code = 'Y' "
                    f"WHERE session_id = {int(sess_id)} AND id IN ({id_list})"
                )
                self.logger.info(f"🔁 Reset generate_genie_code_instruction to 'No' and marked has_genie_code='Y' for session {sess_id}, {len(ids)} use cases")
            except Exception as e:
                self.logger.error(f"Failed to reset flag for session {sess_id}: {get_clean_error_message(e)}")
    
    def _merge_regenerated_ucs_into_tracking(self, regenerated_ucs: list):
        """UPDATE genie_instruction + notebook_path on flagged rows (preserve original session_id).
        Uses a serverless-friendly row-by-row UPDATE so we never touch existing columns we don't own."""
        if not self._tracking_table_name or not regenerated_ucs:
            return
        updated = 0
        for uc in regenerated_ucs:
            sess_id = uc.get('_source_session_id')
            uc_id = uc.get('No')
            if sess_id is None or not uc_id:
                continue
            genie_instr = uc.get('genie_instruction', '') or ''
            nb_path = uc.get('notebook_path', '') or ''
            genie_esc = genie_instr.replace("'", "''")
            nb_esc = nb_path.replace("'", "''")
            try:
                self.spark.sql(
                    f"UPDATE {self._tracking_table_name} "
                    f"SET genie_instruction = '{genie_esc}', notebook_path = '{nb_esc}', "
                    f"updated_at = current_timestamp() "
                    f"WHERE session_id = {int(sess_id)} AND id = '{uc_id.replace(chr(39), chr(39)+chr(39))}'"
                )
                updated += 1
            except Exception as e:
                self.logger.warning(f"Failed to update tracking row for {uc_id}/{sess_id}: {get_clean_error_message(e)}")
        self.logger.info(f"✅ Regenerated Genie instruction persisted for {updated}/{len(regenerated_ucs)} use cases")
    
    def _regenerate_genie_for_flagged_rows(self, flagged_rows: list, route_label: str) -> list:
        """Group flagged rows by session_id, load per-session context, regenerate Genie instruction
        (with _genie_fields_cached=True so no per-UC schema recomputation), then assemble the full
        Genie notebook (overwriting the skeleton). Returns list of UCs that succeeded."""
        import time
        if not flagged_rows:
            return []
        
        by_session = defaultdict(list)
        for row in flagged_rows:
            uc = self._row_to_uc(row)
            by_session[uc.get('_source_session_id')].append(uc)
        
        total = sum(len(v) for v in by_session.values())
        log_print(f"\n{'='*80}")
        log_print(f"🚀 GENERATE MODE ({route_label}) — regenerating {total} use case(s) across {len(by_session)} session(s)")
        log_print(f"{'='*80}")
        
        _step_id = self._emit_pipeline_status(
            stage_name=f"Generate {route_label}",
            step_name="Regeneration",
            sub_step_name="Started",
            message=f"Regenerating Genie instructions for {total} flagged use cases",
            status="started",
            progress_increment=1.0
        )
        
        try:
            translations = self.translation_service.get_translations("English")
        except Exception as _te:
            self.logger.warning(f"Translation service unavailable in Generate mode: {get_clean_error_message(_te)}; falling back to empty dict")
            translations = {}
        
        succeeded = []
        failed_count = 0
        start_time = time.time()
        
        for sess_id, ucs in by_session.items():
            log_print(f"\n🔑 Session {sess_id}: {len(ucs)} use case(s)")
            session_ctx = self._load_session_context(sess_id) if sess_id is not None else {}
            if not session_ctx:
                self.logger.warning(f"No session context loaded for session_id={sess_id}; falling back to current self attributes")
            # Apply session context ONCE before the thread pool (mutates self — not thread-safe).
            self._apply_session_context(session_ctx)

            # v0.7.3: Parallelize Genie regeneration. The LLM call per UC is the bottleneck;
            # running 8-10 concurrently brings a 100-UC run from ~5h to ~40min.
            _rt_par = TECHNICAL_CONTEXT["runtime"]
            _genie_parallelism = _rt_par.get("genie_regen_parallelism", 10)
            from concurrent.futures import ThreadPoolExecutor, as_completed
            import threading as _threading
            _genie_lock = _threading.Lock()
            self.logger.info(f"🤖 [{route_label}] Session {sess_id}: running {len(ucs)} Genie regens with parallelism={_genie_parallelism}")

            def _regen_one(_idx_uc):
                _idx, _uc = _idx_uc
                _uc_id = _uc.get('No', 'UNKNOWN')
                try:
                    self.logger.info(f"🤖 [{route_label}] ({_idx}/{len(ucs)}) Regenerating Genie instruction for {_uc_id}...")
                    _uc = self._generate_genie_instruction_for_use_case(_uc, [], '', schema_index=None)
                    if not _uc.get('genie_instruction'):
                        self.logger.error(f"❌ [{route_label}] {_uc_id}: LLM returned empty genie_instruction")
                        return (None, _uc_id)
                    _nb_path = self._assemble_genie_code_notebook(_uc, translations)
                    if _nb_path:
                        _uc['notebook_path'] = _nb_path
                    self.logger.info(f"✅ [{route_label}] {_uc_id}: regenerated and notebook written to {_nb_path}")
                    return (_uc, _uc_id)
                except Exception as _e:
                    self.logger.error(f"❌ [{route_label}] {_uc_id} failed: {get_clean_error_message(_e)}")
                    return (None, _uc_id)

            with ThreadPoolExecutor(max_workers=max(1, _genie_parallelism), thread_name_prefix=f"GenieRegen_{sess_id}") as _ex:
                _futs = [_ex.submit(_regen_one, (_i, _u)) for _i, _u in enumerate(ucs, start=1)]
                for _fut in as_completed(_futs):
                    _result_uc, _result_id = _fut.result()
                    with _genie_lock:
                        if _result_uc is not None:
                            succeeded.append(_result_uc)
                        else:
                            failed_count += 1
        
        elapsed = time.time() - start_time
        log_print(f"\n{'='*80}")
        log_print(f"✅ GENERATE MODE ({route_label}) COMPLETE")
        log_print(f"   ✅ Regenerated OK: {len(succeeded)}/{total}")
        if failed_count:
            log_print(f"   ❌ Failed:         {failed_count}/{total}")
        log_print(f"   ⏱️  Total time:    {elapsed:.1f}s")
        log_print(f"{'='*80}\n")
        
        self._emit_pipeline_status(
            stage_name=f"Generate {route_label}",
            step_name="Regeneration",
            sub_step_name="Complete",
            message=f"Regeneration done: {len(succeeded)} ok, {failed_count} failed",
            status="ended_success" if failed_count == 0 else "ended_warning",
            progress_increment=2.0,
            result_json={"success": len(succeeded), "failed": failed_count, "total": total, "elapsed_seconds": round(elapsed, 1)},
            step_id=_step_id
        )
        
        return succeeded
    
    def _run_generate_mode(self):
        """Top-level orchestrator for Operation='Generate Use Cases'.
        
        Route 1 (primary): scan __inspire_usecases for rows where
            generate_genie_code_instruction = 'Yes' (flagged via UI/SQL UPDATE).
        Route 2 (fallback, only if Route 1 finds zero rows): walk the generation_path
            workspace tree, AST-parse each notebook's marker cell, pick up any
            generate_genie_code_instruction = True, then cross-reference __inspire_usecases
            by (id, session_id) to rehydrate the UC row.
        
        After successful regeneration, UPDATE the flag back to 'No' for Route 1 entries
        (Route 2 has no persistent flag — overwriting the notebook removes the True marker).
        """
        log_print(f"\n{'='*80}")
        log_print(f"🚀 OPERATION = 'Generate Use Cases' — discovery pipeline skipped")
        # Gap G15 acquire_lock: prevent concurrent Generate runs for the same business.
        # Fail-open: infra failure never blocks the pipeline.
        try:
            _ok = self.atomic_writer.acquire_lock(
                self.business_name, ttl_seconds=7200,
                holder_note=f"operation={self.operation}"
            ) if getattr(self, "atomic_writer", None) else True
            if not _ok:
                self.logger.warning(f"G15: another session is already generating for business={self.business_name}; proceeding anyway (fail-open).")
                try:
                    NEXT_ACTIONS.add(
                        rule_id="G15_LOCK_CONTENDED", severity="HIGH", phase="Generate", step="Lock",
                        evidence=f"Concurrent Generate run detected for business={self.business_name}",
                        suggested_user_action="Wait for the other run to complete before launching another Generate run for this business."
                    )
                except Exception:
                    pass
        except Exception as _ge:
            self.logger.debug(f"G15 acquire_lock failed non-fatally: {get_clean_error_message(_ge)}")
        log_print(f"{'='*80}\n")

        # v0.8.9 fix: every terminal path (success/early-return/exception) MUST release the
        # G15 lock — previously only the final Route-2 path released it, leaving stale 7200s
        # locks on every Route-1-success / no-DB / no-flagged-rows happy path.
        try:
            # === PHASE E0: CATALOG DRIFT CHECK (Gap 4) ===
            try:
                _drifted = self._phaseE0_catalog_drift_check(self.session_id)
                if _drifted:
                    log_print(f"⚠️ E0 drift: {len(_drifted)} UC(s) reference tables no longer present", level="WARNING")
            except Exception as _drift_err:
                self.logger.warning(f"E0 drift check failed non-fatally: {get_clean_error_message(_drift_err)}")

            if not self._tracking_table_name:
                log_print("⚠️ Inspire Database not configured — Route 1 unavailable. Attempting Route 2 only.")
                self.logger.warning("Generate mode: no inspire_database configured; only Route 2 (notebook scan) can run.")
                self._run_route2_fallback('Route 2 — notebook scan (no DB)', persist_tracking=False)
                return

            log_print("🔎 Route 1: scanning __inspire_usecases for generate_genie_code_instruction = 'Yes'...")
            route1_rows = self._fetch_flagged_use_cases_route1()
            if route1_rows:
                log_print(f"✅ Route 1: found {len(route1_rows)} flagged use case(s) in the database.")
                regen = self._regenerate_genie_for_flagged_rows(route1_rows, 'Route 1 — DB flag')
                if regen:
                    session_ids = defaultdict(list)
                    for uc in regen:
                        sess_id = uc.get('_source_session_id')
                        if sess_id is not None and uc.get('No'):
                            session_ids[sess_id].append(uc['No'])
                    self._merge_regenerated_ucs_into_tracking(regen)
                    self._update_flag_to_no(session_ids)
                return

            log_print("ℹ️ Route 1 found no flagged rows in the database. Falling back to Route 2 (notebook scan)...")
            self._run_route2_fallback('Route 2 — notebook flag', persist_tracking=True)
        finally:
            try:
                if getattr(self, 'atomic_writer', None):
                    self.atomic_writer.release_lock(self.business_name)
            except Exception as _re:
                self.logger.debug(f"G15 release_lock failed non-fatally: {get_clean_error_message(_re)}")

    def _run_route2_fallback(self, route_label: str, persist_tracking: bool):
        """v0.8.9 DRY: shared Route 2 fallback for both no-DB and Route1-empty paths.
        
        - Scans workspace notebooks for the generate_genie_code_instruction=True marker.
        - Regenerates Genie instructions + notebooks for any flagged entries.
        - When persist_tracking=True (DB available, Route1 empty path), merges regenerated
          UCs back into __inspire_usecases.
        """
        route2_rows = self._fetch_flagged_use_cases_route2()
        if not route2_rows:
            log_print("ℹ️ Route 2 found no notebooks flagged with generate_genie_code_instruction = True. Nothing to do.")
            return
        log_print(f"✅ Route 2: found {len(route2_rows)} flagged notebook(s); cross-reference loaded from __inspire_usecases." if persist_tracking else f"✅ Route 2: found {len(route2_rows)} flagged notebook(s).")
        regen = self._regenerate_genie_for_flagged_rows(route2_rows, route_label)
        if regen and persist_tracking:
            self._merge_regenerated_ucs_into_tracking(regen)

    def _generate_genie_instructions_and_notebooks(self, all_use_cases: list, full_schema_details: list,
                                                    document_context: str, translations: dict,
                                                    summary_dict: dict = None) -> list:
        import time
        
        if not all_use_cases:
            self.logger.warning("No use cases provided for Genie Code instruction generation")
            return []
        
        log_print(f"\n{'='*80}")
        log_print(f"🤖 GENIE CODE INSTRUCTIONS + NOTEBOOK GENERATION (STREAMING)")
        log_print(f"{'='*80}")
        log_print(f"📊 Total use cases: {len(all_use_cases)}")
        log_print(f"📝 Strategy: Wave-based generation with retry for timed-out use cases")
        log_print(f"📓 Notebooks: STREAMING — each notebook generated the instant its use case completes")
        log_print(f"📁 Instructions will be embedded in per-use-case notebooks")
        log_print(f"{'='*80}\n")
        
        _genie_gen_step_id = self._emit_pipeline_status(
            stage_name="Genie Code Instructions",
            step_name="Generation",
            sub_step_name="Started",
            message=f"Generating Genie Code instructions for {len(all_use_cases)} use cases",
            status="started",
            progress_increment=1.0
        )
        
        nb_executor = None
        nb_futures = []
        nb_completed = 0
        nb_failed = 0
        _nb_lock = threading.Lock()
        
        try:
            self.logger.info(f"🔧 Building schema index for Genie instruction generation...")
            schema_by_table = defaultdict(list)
            for detail in full_schema_details:
                (catalog, schema, table, column_name, data_type, comment) = detail
                fqtn = f"{catalog}.{schema}.{table}"
                schema_by_table[fqtn].append(detail)
                schema_by_table[f"`{catalog}`.`{schema}`.`{table}`"] = schema_by_table[fqtn]
            self.logger.info(f"   ✓ Schema index built with {len(schema_by_table)} table entries")
            
            genie_parallelism, reason = calculate_adaptive_parallelism(
                "genie_instruction_gen", self.max_parallelism,
                num_items=len(all_use_cases),
                is_llm_operation=True, logger=self.logger,
                estimated_mb_per_worker=5.0
            )
            _genie_adaptive_cap = 8 if len(all_use_cases) >= 30 else 6 if len(all_use_cases) >= 15 else 4
            base_workers = max(2, min(genie_parallelism, len(all_use_cases), _genie_adaptive_cap))
            self.logger.info(f"🔧 Genie parallelism: adaptive_cap={_genie_adaptive_cap} (for {len(all_use_cases)} UCs), base_workers={base_workers}")
            
            wave_parallelism = [
                (1, base_workers),
                (2, base_workers),
                (3, max(2, (base_workers + 1) // 2)),
                (4, max(1, (base_workers + 2) // 3)),
                (5, max(1, (base_workers + 2) // 3)),
            ]
            
            overall_start = time.time()
            total_success = 0
            total_fail = 0
            final_results = {}
            
            nb_parallelism, _nb_reason = calculate_adaptive_parallelism(
                "notebook_generation", self.max_parallelism,
                num_items=len(all_use_cases),
                num_domains=len(set(uc.get('Business Domain', 'Other') for uc in all_use_cases)),
                is_llm_operation=False, logger=self.logger
            )
            nb_executor = concurrent.futures.ThreadPoolExecutor(
                max_workers=nb_parallelism,
                thread_name_prefix="NbStreamGen"
            )
            # Claim streaming ownership NOW so partial streaming failures don't
            # trigger the batch-fallback writer, which would double-write already-
            # streamed notebooks. Batch path also uses this flag to skip.
            self._notebooks_generated_inline = True
            self.logger.info(f"📓 Streaming notebook executor ready (workers={nb_parallelism}); batch fallback disabled for this run")
            
            def _submit_notebook_for_use_case(uc):
                uc_id = uc.get('No', 'UNKNOWN')
                uc_name = uc.get('Name', '')
                domain = uc.get('Business Domain', 'Other')
                subdomain = uc.get('Subdomain', 'General')
                domain_folder = self._sanitize_name(domain)
                subdomain_folder = self._sanitize_name(subdomain)
                notebook_name = f"{uc_id}-{self._sanitize_name(uc_name)}"
                
                def _build_notebook():
                    nonlocal nb_completed, nb_failed
                    rel_path = f"{domain_folder}/{subdomain_folder}/{notebook_name}"
                    try:
                        self._assemble_single_use_case_notebook(
                            use_case=uc, translations=translations,
                            domain_folder=domain_folder, subdomain_folder=subdomain_folder,
                            notebook_name=notebook_name, summary_dict=summary_dict
                        )
                        with _nb_lock:
                            nb_completed += 1
                            _cnt = nb_completed + nb_failed
                        self.logger.info(f"📓 [{_cnt}/{len(all_use_cases)}] Notebook streamed: {rel_path}")
                        log_print(f"   📓 Notebook ready: {rel_path}")
                        return (uc_id, rel_path, True)
                    except Exception as e:
                        with _nb_lock:
                            nb_failed += 1
                            _cnt = nb_completed + nb_failed
                        self.logger.error(f"❌ [{_cnt}/{len(all_use_cases)}] Notebook failed: {rel_path} — {get_clean_error_message(e)}")
                        return (uc_id, rel_path, False)
                
                f = nb_executor.submit(_build_notebook)
                nb_futures.append(f)
            
            backlog = list(all_use_cases)
            
            for wave_id, wave_workers in wave_parallelism:
                if not backlog:
                    break
                
                wave_start = time.time()
                log_print(f"\n   ▶️ Wave {wave_id}: {len(backlog)} use cases, parallelism {wave_workers}")
                self.logger.info(f"🔁 Genie Wave {wave_id}: processing {len(backlog)} use cases with {wave_workers} workers")
                
                def _on_uc_ready_from_wave(uc, success):
                    _submit_notebook_for_use_case(uc)
                
                succeeded, timed_out = self._run_genie_wave(
                    wave_id, backlog, full_schema_details, document_context,
                    schema_by_table, translations, wave_workers,
                    on_uc_ready=_on_uc_ready_from_wave
                )
                
                wave_elapsed = time.time() - wave_start
                
                for uc in succeeded:
                    uc_id = uc.get('No', 'UNKNOWN')
                    final_results[uc_id] = uc
                    total_success += 1
                    log_print(f"   ✅ [{total_success + total_fail}/{len(all_use_cases)}] {uc_id}: Genie instruction generated")
                    self._emit_pipeline_status(
                        stage_name="Genie Code Instructions",
                        step_name=f"Wave {wave_id} - {uc_id}",
                        sub_step_name=str(uc.get('Name', ''))[:120],
                        message=f"Genie instruction generated for {uc_id}",
                        status="ended_success",
                        progress_increment=1.0,
                        result_json=self._build_use_case_step_payload(uc)
                    )
                
                timeout_ids = []
                hard_fail_ids = []
                for uc in timed_out:
                    uc_id = uc.get('No', 'UNKNOWN')
                    is_timeout = uc.get('genie_instruction_status') == 'timeout'
                    if is_timeout:
                        timeout_ids.append(uc_id)
                    else:
                        hard_fail_ids.append(uc_id)
                
                log_print(f"\n   {'='*60}")
                log_print(f"   📊 GENIE WAVE {wave_id} REPORT ({wave_elapsed:.1f}s)")
                log_print(f"   {'='*60}")
                log_print(f"      • Processed: {len(succeeded) + len(timed_out)}")
                log_print(f"      • ✅ Succeeded: {len(succeeded)}")
                log_print(f"      • ❌ Failed: {len(timed_out)}")
                if timeout_ids:
                    if wave_id < len(wave_parallelism):
                        log_print(f"         - ⏱️  Timeouts: {len(timeout_ids)} (will retry in wave {wave_id + 1})")
                    else:
                        log_print(f"         - ⏱️  Timeouts: {len(timeout_ids)} (final wave — no more retries)")
                    for tid in timeout_ids:
                        log_print(f"            • {tid}")
                if hard_fail_ids:
                    log_print(f"         - 💥 Hard failures: {len(hard_fail_ids)}")
                    for hid in hard_fail_ids:
                        log_print(f"            • {hid}")
                log_print(f"   {'='*60}\n")
                
                self.logger.info(f"📊 Genie Wave {wave_id}: {len(succeeded)} succeeded, {len(timeout_ids)} timeouts, {len(hard_fail_ids)} hard failures in {wave_elapsed:.1f}s")
                
                retry_candidates = []
                for uc in timed_out:
                    uc_id = uc.get('No', 'UNKNOWN')
                    is_timeout = uc.get('genie_instruction_status') == 'timeout'
                    if is_timeout and wave_id < len(wave_parallelism):
                        uc['genie_instruction_status'] = 'pending_retry'
                        uc['_genie_retry_attempt'] = uc.get('_genie_retry_attempt', 0) + 1
                        uc['_skip_llm_cache'] = True
                        retry_candidates.append(uc)
                        self.logger.info(f"   🔄 [{uc_id}] Marked for retry in wave {wave_id + 1} (attempt {uc['_genie_retry_attempt']})")
                    else:
                        total_fail += 1
                        final_results[uc_id] = uc
                        log_print(f"   ❌ [{total_success + total_fail}/{len(all_use_cases)}] {uc_id}: Genie instruction FAILED (final)")
                        self._emit_pipeline_status(
                            stage_name="Genie Code Instructions",
                            step_name=f"Wave {wave_id} - {uc_id}",
                            sub_step_name=str(uc.get('Name', ''))[:120],
                            message=f"Genie instruction failed for {uc_id}",
                            status="ended_error",
                            progress_increment=1.0,
                            result_json=self._build_use_case_step_payload(uc)
                        )
                        if is_timeout:
                            _submit_notebook_for_use_case(uc)
                
                backlog = retry_candidates
                
                if backlog and wave_id < len(wave_parallelism):
                    cooldown_secs = 15 * wave_id
                    log_print(f"   🔄 {len(backlog)} timed-out use cases will be retried in wave {wave_id + 1}")
                    log_print(f"   ⏳ Cooling down {cooldown_secs}s before wave {wave_id + 1} (reducing API congestion)...")
                    self.logger.info(f"⏳ Genie wave cooldown: {cooldown_secs}s before wave {wave_id + 1}")
                    time.sleep(cooldown_secs)
            
            for uc in backlog:
                uc_id = uc.get('No', 'UNKNOWN')
                total_fail += 1
                final_results[uc_id] = uc
                log_print(f"   ❌ [{total_success + total_fail}/{len(all_use_cases)}] {uc_id}: Genie instruction FAILED (exhausted all waves)")
                self._emit_pipeline_status(
                    stage_name="Genie Code Instructions",
                    step_name=f"Final - {uc_id}",
                    sub_step_name=str(uc.get('Name', ''))[:120],
                    message=f"Genie instruction failed after all waves for {uc_id}",
                    status="ended_error",
                    progress_increment=1.0,
                    result_json=self._build_use_case_step_payload(uc)
                )
                _submit_notebook_for_use_case(uc)
            
            elapsed = time.time() - overall_start
            
            if nb_futures:
                nb_pending = sum(1 for f in nb_futures if not f.done())
                if nb_pending > 0:
                    log_print(f"\n   ⏳ Genie instructions done. Waiting for {nb_pending} streaming notebooks to finish...")
                    self.logger.info(f"📓 Waiting for {nb_pending} remaining streaming notebooks...")
                _nb_timeout = max(600, len(all_use_cases) * 60)
                try:
                    for f in safe_as_completed(nb_futures, timeout=_nb_timeout):
                        try:
                            f.result()
                        except Exception as _nb_err:
                            self.logger.error(f"📓 Streaming notebook future error: {get_clean_error_message(_nb_err)}")
                except concurrent.futures.TimeoutError:
                    self.logger.error(f"⏱️ Streaming notebook generation timed out after {_nb_timeout}s")
                    for f in nb_futures:
                        if not f.done() and f.cancel():
                            with _nb_lock:
                                nb_failed += 1
            
            try:
                nb_executor.shutdown(wait=False, cancel_futures=True)
            except TypeError:
                nb_executor.shutdown(wait=False)
            
            log_print(f"\n{'='*80}")
            log_print(f"✅ GENIE CODE INSTRUCTIONS + NOTEBOOKS COMPLETE")
            log_print(f"{'='*80}")
            log_print(f"   ✅ Genie Success: {total_success}/{len(all_use_cases)} use cases")
            log_print(f"   ❌ Genie Failed:  {total_fail}/{len(all_use_cases)} use cases")
            log_print(f"   📓 Notebooks OK:  {nb_completed}/{len(all_use_cases)}")
            if nb_failed:
                log_print(f"   📓 Notebooks Fail: {nb_failed}/{len(all_use_cases)}")
            log_print(f"   ⏱️  Total time: {elapsed:.1f}s")
            log_print(f"{'='*80}\n")
            
            self._notebooks_generated_inline = True
            
            self._emit_pipeline_status(
                stage_name="Genie Code Instructions",
                step_name="Generation",
                sub_step_name="Complete",
                message=f"Genie Code instructions: {total_success} success, {total_fail} failed",
                status="ended_success",
                progress_increment=2.0,
                result_json={"success": total_success, "failed": total_fail, "total_use_cases": len(all_use_cases), "elapsed_seconds": round(elapsed, 1)},
                step_id=_genie_gen_step_id
            )
        except Exception as _genie_err:
            if nb_executor:
                self.logger.warning("📓 Cleaning up streaming notebook executor after genie error...")
                for f in nb_futures:
                    if not f.done():
                        f.cancel()
                try:
                    nb_executor.shutdown(wait=False, cancel_futures=True)
                except TypeError:
                    nb_executor.shutdown(wait=False)
            self._emit_pipeline_status(
                stage_name="Genie Code Instructions",
                step_name="Generation",
                sub_step_name="Failed",
                message=f"Genie Code instructions failed: {get_clean_error_message(_genie_err, max_chars=200)}",
                status="ended_error",
                progress_increment=1.0,
                result_json={"error": get_clean_error_message(_genie_err, max_chars=300), "total_use_cases": len(all_use_cases)},
                step_id=_genie_gen_step_id
            )
            raise
        
        result_list = []
        for uc in all_use_cases:
            uc_id = uc.get('No', 'UNKNOWN')
            if uc_id in final_results:
                result_list.append(final_results[uc_id])
            else:
                result_list.append(uc)
        
        return result_list

    def _filter_business_tables(self, db_details: list, business_context: str = "", industry: str = "", exclusion_strategy: str = "Medium") -> tuple:
        """
        Filters tables into business-relevant vs technical/metadata tables using LLM.
        RESPECTS MAX_CONTEXT_CHARS by batching tables if needed.
        Implements RECURSIVE BATCHING: If a batch is too large, splits it and tries again recursively.
        
        Args:
            db_details: List of (catalog, schema, table, column, type, comment) tuples
            business_context: Business context description
            industry: Industry classification
            exclusion_strategy: Technical exclusion strategy ("None", "Aggressive", "Medium", "Low")
            
        Returns:
            Tuple of (business_tables_details, technical_tables_details, business_table_names, technical_table_names,
            business_scores_dict, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set)
        """
        if not db_details:
            return ([], [], set(), set(), {})
        
        include_technical = exclusion_strategy == "None"
        
        # Get additional context from instance (user-provided instructions)
        additional_context = getattr(self, 'additional_context', '') or ''
        
        # Call the main recursive processing logic
        return self._filter_business_tables_with_batching(
            db_details=db_details,
            business_context=business_context,
            industry=industry,
            exclusion_strategy=exclusion_strategy,
            additional_context=additional_context,
            include_technical=include_technical
        )
    
    def _process_filter_batch_recursive(self, batch_tables: list, batch_idx: int, total_batches: int, 
                                       business_name: str, industry: str, business_context: str,
                                       exclusion_strategy: str, strategy_rule_text: str,
                                       additional_context_section: str = "",
                                       depth: int = 0, max_depth: int = 10) -> dict:
        """
        Recursively process a batch of tables for filtering with automatic sub-batching if context limit exceeded.
        
        Args:
            batch_tables: List of table names to classify
            batch_idx: Batch index for logging
            total_batches: Total number of batches
            business_name: Business name
            industry: Industry classification
            business_context: Business context description
            exclusion_strategy: Exclusion strategy
            strategy_rule_text: Strategy-specific rules text
            additional_context_section: User-provided filtering instructions (formatted)
            depth: Current recursion depth
            max_depth: Maximum recursion depth to prevent infinite loops
            
        Returns:
            Dict mapping table_name -> (classification, business_score, data_category)
        """
        if depth > max_depth:
            self.logger.error(f"[Batch {batch_idx}] Max recursion depth ({max_depth}) reached. Defaulting {len(batch_tables)} tables to BUSINESS.")
            return {table.replace('`', ''): ('BUSINESS', 50, 'MASTER') for table in batch_tables}
        
        depth_prefix = "  " * depth
        if depth > 0:
            log_prefix = f"[Batch {batch_idx}-Split{depth}]"
        else:
            log_prefix = f"[Batch {batch_idx}]"
        
        log_print(f"{depth_prefix}{log_prefix} Processing {len(batch_tables)} tables (depth={depth})...")
        self.logger.info(f"{depth_prefix}{log_prefix} Processing {len(batch_tables)} tables (depth={depth})...")
        
        # Create markdown table list for this batch
        tables_markdown = "\n".join([f"- {table}" for table in batch_tables])
        
        # Verify this batch's prompt size
        prompt_vars = {
            "business_name": business_name,
            "industry": industry,
            "business_context": business_context,
            "exclusion_strategy": exclusion_strategy,
            "strategy_rules": strategy_rule_text,
            "additional_context_section": additional_context_section,
            "tables_markdown": tables_markdown,
            "generation_instructions_section": self._get_generation_instruction_for_prompt("FILTER_BUSINESS_TABLES_PROMPT"),
        }
        
        # Format the prompt to check actual size (using model-specific limits from TECHNICAL_CONTEXT)
        filter_context_limit = get_max_context_chars("English", "FILTER_BUSINESS_TABLES_PROMPT")
        test_prompt = self.ai_agent._load_and_format_prompt("FILTER_BUSINESS_TABLES_PROMPT", prompt_vars)
        actual_prompt_size = len(test_prompt)
        
        # Check if batch is too large
        if actual_prompt_size > filter_context_limit:
            # Need to split this batch
            if len(batch_tables) <= 1:
                # Cannot split further - single table is too large (very rare case)
                self.logger.error(
                    f"{depth_prefix}{log_prefix} Single table name too long to process ({actual_prompt_size:,} chars). "
                    f"Defaulting to BUSINESS."
                )
                return {batch_tables[0].replace('`', ''): ('BUSINESS', 50, 'MASTER')}
            
            # Split into 2 sub-batches
            mid_point = len(batch_tables) // 2
            first_half = batch_tables[:mid_point]
            second_half = batch_tables[mid_point:]
            
            self.logger.warning(
                f"{depth_prefix}{log_prefix} Batch too large ({actual_prompt_size:,} chars > {filter_context_limit:,} limit). "
                f"Splitting {len(batch_tables)} tables into 2 sub-batches: {len(first_half)} + {len(second_half)} tables"
            )
            
            results = {}
            _common_kwargs = dict(
                total_batches=total_batches, business_name=business_name,
                industry=industry, business_context=business_context,
                exclusion_strategy=exclusion_strategy, strategy_rule_text=strategy_rule_text,
                additional_context_section=additional_context_section,
                depth=depth + 1, max_depth=max_depth
            )
            
            split_exec = concurrent.futures.ThreadPoolExecutor(max_workers=2, thread_name_prefix="filter_split")
            split_timed_out = False
            try:
                f_first = split_exec.submit(
                    self._process_filter_batch_recursive,
                    batch_tables=first_half, batch_idx=f"{batch_idx}a", **_common_kwargs
                )
                f_second = split_exec.submit(
                    self._process_filter_batch_recursive,
                    batch_tables=second_half, batch_idx=f"{batch_idx}b", **_common_kwargs
                )
                sub_batch_timeout = self.llm_timeout_seconds * 4
                try:
                    for f in safe_as_completed([f_first, f_second], timeout=sub_batch_timeout):
                        try:
                            results.update(f.result(timeout=self.llm_timeout_seconds * 3))
                        except Exception as exc:
                            self.logger.error(f"{depth_prefix}{log_prefix} Sub-batch failed: {get_clean_error_message(exc)}")
                except concurrent.futures.TimeoutError:
                    split_timed_out = True
                    self.logger.error(f"{depth_prefix}{log_prefix} Sub-batch splitting timed out after {sub_batch_timeout}s")
            finally:
                split_exec.shutdown(wait=False, cancel_futures=split_timed_out)
            
            self.logger.info(f"{depth_prefix}{log_prefix} Sub-batches complete. Total: {len(results)} tables classified")
            return results
        
        # Batch size is OK - process it with retry logic
        self.logger.debug(f"{depth_prefix}{log_prefix} Batch size OK ({actual_prompt_size:,} chars)")
        
        try:
            max_retries = self.max_retry_attempts + 1
            for attempt in range(1, max_retries + 1):
                try:
                    # Call LLM to classify tables in this batch
                    attempt_suffix = f" (attempt {attempt}/{max_retries})" if attempt > 1 else ""
                    log_print(f"{depth_prefix}⏳ {log_prefix} Waiting for LLM response{attempt_suffix} (filtering {len(batch_tables)} tables into BUSINESS vs TECHNICAL)...")
                    log_print(f"{depth_prefix}⏳ {log_prefix} Waiting for LLM response{attempt_suffix} (filtering {len(batch_tables)} tables into BUSINESS vs TECHNICAL)...")
                    self.logger.info(f"{depth_prefix}⏳ {log_prefix} Waiting for LLM response{attempt_suffix} (filtering {len(batch_tables)} tables into BUSINESS vs TECHNICAL)...")
                    response_raw = self.ai_agent.run_worker(
                        step_name=f"Filter_Business_Tables_Batch_{batch_idx}_Depth{depth}_Attempt{attempt}",
                        worker_prompt_path="FILTER_BUSINESS_TABLES_PROMPT",
                        prompt_vars=prompt_vars,
                        response_schema=None,
                        skip_cache=(attempt > 1)  # v0.7.11: retries bypass cache
                    )
                    log_print(f"{depth_prefix}✅ {log_prefix} Received LLM response, parsing classifications...")
                    self.logger.info(f"{depth_prefix}✅ {log_prefix} Received LLM response, parsing classifications...")
                    
                    # Parse CSV response using centralized utility
                    response_clean = clean_json_response(response_raw)
                    
                    csv_rows = CSVParser.parse_csv_string(
                        response_clean,
                        logger=self.logger,
                        context=log_prefix
                    )
                    
                    results = {}
                    row_count = 0
                    for row in csv_rows:
                        row_count += 1
                        # Safely get values with fallback to empty string if None
                        table_name = (row.get('Table Name') or '').strip().replace('`', '')
                        classification = (row.get('Classification') or '').strip().upper()
                        data_category = (row.get('Data Category') or '').strip().upper()
                        business_score_str = (row.get('Business Score') or '50').strip()
                        
                        # Skip invalid rows (empty table name or very short names that are likely parsing errors)
                        if not table_name or len(table_name) < 3:
                            self.logger.warning(f"{depth_prefix}Skipping invalid row with table name '{table_name}' (too short or empty)")
                            continue
                        
                        # Parse business score
                        try:
                            business_score = int(business_score_str)
                            business_score = max(0, min(100, business_score))  # Clamp to 0-100
                        except (ValueError, TypeError):
                            business_score = 50  # Default to medium score if parsing fails
                            self.logger.warning(f"{depth_prefix}Table {table_name}: Invalid business score '{business_score_str}', using default 50")
                        
                        if classification == 'BUSINESS':
                            if data_category not in ('MASTER', 'REFERENCE', 'TRANSACTIONAL'):
                                self.logger.warning(f"{depth_prefix}Table {table_name}: LLM returned invalid data_category '{data_category}', defaulting to MASTER")
                                data_category = 'MASTER'
                            results[table_name] = ('BUSINESS', business_score, data_category)
                        elif classification == 'TECHNICAL':
                            results[table_name] = ('TECHNICAL', 0, 'TECHNICAL')
                        else:
                            self.logger.warning(f"{depth_prefix}Table {table_name} has unclear classification '{classification}', defaulting to BUSINESS/MASTER")
                            if data_category not in ('MASTER', 'REFERENCE', 'TRANSACTIONAL'):
                                data_category = 'MASTER'
                            results[table_name] = ('BUSINESS', business_score, data_category)
                    
                    # Validate we got results
                    if not results or row_count == 0:
                        raise ValueError(f"CSV parsing returned no results (row_count={row_count})")
                    
                    self.logger.info(f"{depth_prefix}✅ {log_prefix} Complete: {len(results)} tables classified")
                    return results
                    
                except (ValueError, AttributeError, KeyError) as parse_error:
                    # CSV parsing errors - retry
                    if attempt < max_retries:
                        self.logger.warning(f"{depth_prefix}{log_prefix} CSV parsing error on attempt {attempt}/{max_retries}: {parse_error}. Retrying...")
                        continue
                    else:
                        self.logger.error(f"{depth_prefix}{log_prefix} Failed to parse CSV after {max_retries} attempts: {parse_error}. Defaulting batch tables to BUSINESS.")
                        return {table.replace('`', ''): ('BUSINESS', 50, 'MASTER') for table in batch_tables}
            
        except InputTooLongError as e:
            # Even though we pre-checked, the LLM still rejected it. Split more aggressively.
            if len(batch_tables) <= 1:
                self.logger.error(
                    f"{depth_prefix}{log_prefix} LLM rejected even after pre-check. Single table too large. "
                    f"Defaulting to BUSINESS. Error: {str(e)[:200]}"
                )
                return {batch_tables[0].replace('`', ''): ('BUSINESS', 50, 'MASTER')}
            
            # Split into 2 sub-batches
            mid_point = len(batch_tables) // 2
            first_half = batch_tables[:mid_point]
            second_half = batch_tables[mid_point:]
            
            self.logger.warning(
                f"{depth_prefix}{log_prefix} LLM rejected batch (InputTooLongError). "
                f"Splitting {len(batch_tables)} tables into 2 sub-batches: {len(first_half)} + {len(second_half)} tables"
            )
            
            # Process both halves recursively
            results = {}
            first_results = self._process_filter_batch_recursive(
                batch_tables=first_half,
                batch_idx=f"{batch_idx}a",
                total_batches=total_batches,
                business_name=business_name,
                industry=industry,
                business_context=business_context,
                exclusion_strategy=exclusion_strategy,
                strategy_rule_text=strategy_rule_text,
                additional_context_section=additional_context_section,
                depth=depth + 1,
                max_depth=max_depth
            )
            results.update(first_results)
            
            second_results = self._process_filter_batch_recursive(
                batch_tables=second_half,
                batch_idx=f"{batch_idx}b",
                total_batches=total_batches,
                business_name=business_name,
                industry=industry,
                business_context=business_context,
                exclusion_strategy=exclusion_strategy,
                strategy_rule_text=strategy_rule_text,
                additional_context_section=additional_context_section,
                depth=depth + 1,
                max_depth=max_depth
            )
            results.update(second_results)
            
            return results
            
        except Exception as batch_error:
            self.logger.error(f"{depth_prefix}{log_prefix} Failed to process batch: {batch_error}. Defaulting batch tables to BUSINESS.")
            # Default all tables in this batch to business with medium score
            return {table.replace('`', ''): ('BUSINESS', 50, 'MASTER') for table in batch_tables}
    
    def _filter_business_tables_with_batching(self, db_details: list, business_context: str, 
                                             industry: str, exclusion_strategy: str,
                                             additional_context: str = "", include_technical: bool = False) -> tuple:
        """
        Main filtering logic with batching support and recursive sub-batching.
        """
        try:
            # === LOG USER ADDITIONAL CONTEXT INTERPRETATION ===
            additional_context_section = ""
            if additional_context:
                self.logger.info("=" * 80)
                self.logger.info("🔍 INTERPRETING USER ADDITIONAL CONTEXT FOR TABLE FILTERING")
                self.logger.info("=" * 80)
                self.logger.info(f"📋 User Instructions: {additional_context[:500]}{'...' if len(additional_context) > 500 else ''}")
                
                # Log interpretation of common patterns
                context_lower = additional_context.lower()
                
                # Check for database/catalog exclusions
                if 'ignore' in context_lower or 'exclude' in context_lower or 'skip' in context_lower:
                    self.logger.info("🎯 DETECTED: User wants to EXCLUDE/IGNORE certain data")
                
                # Check for business entity references
                business_entities = ['customer', 'subscriber', 'client', 'user', 'member', 'patient', 'employee', 
                                   'vendor', 'supplier', 'partner', 'order', 'transaction', 'product', 'inventory']
                detected_entities = [e for e in business_entities if e in context_lower]
                if detected_entities:
                    self.logger.info(f"🎯 DETECTED BUSINESS ENTITIES: {', '.join(detected_entities)}")
                    self.logger.info("   → LLM will apply SEMANTIC understanding (e.g., 'subscriber' = 'customer')")
                
                # Check for database/catalog references
                if 'database' in context_lower or 'catalog' in context_lower or 'schema' in context_lower:
                    self.logger.info("🎯 DETECTED: Database/Catalog/Schema level filtering instructions")
                
                self.logger.info("=" * 80)
                
                # Build the additional context section for the prompt
                additional_context_section = f"""
**🚨🚨🚨 HIGHEST PRIORITY: USER-PROVIDED FILTERING INSTRUCTIONS 🚨🚨🚨**

**⛔ YOU MUST FOLLOW THESE USER INSTRUCTIONS - THEY OVERRIDE ALL OTHER RULES ⛔**

{additional_context}

**CRITICAL INTERPRETATION RULES**:
1. **Database/Catalog Exclusions**: If user says "ignore database X" or "exclude catalog Y", ALL tables from that database/catalog MUST be classified as TECHNICAL (filtered out)
2. **Business Entity Semantics**: Apply BUSINESS UNDERSTANDING to user instructions:
   - If user says "ignore customers", also ignore tables named: subscribers, clients, users, members, accounts, patrons, buyers
   - If user says "ignore orders", also ignore tables named: transactions, purchases, bookings, reservations, sales
   - If user says "ignore products", also ignore tables named: items, inventory, merchandise, goods, SKUs, catalog
   - **USE YOUR BUSINESS DOMAIN KNOWLEDGE** to identify semantically equivalent entities
3. **Explicit Overrides**: User instructions ALWAYS override the default classification rules
4. **When in doubt**: If a table MIGHT relate to something user wants to ignore, classify it as TECHNICAL

**EXAMPLE INTERPRETATIONS**:
- User: "ignore everything from database legacy_crm" → ALL tables with `legacy_crm` in their path → TECHNICAL
- User: "ignore all customer data" → Tables: customers, subscribers, clients, users, members, accounts → ALL TECHNICAL
- User: "focus only on finance" → Tables NOT related to finance/accounting → TECHNICAL
- User: "exclude marketing tables" → Tables: campaigns, leads, prospects, marketing_*, ads_* → ALL TECHNICAL

**⚠️ FAILURE TO FOLLOW USER INSTRUCTIONS = ENTIRE OUTPUT REJECTED ⚠️**
"""
            else:
                additional_context_section = "*(No additional user filtering instructions provided)*"
            
            # Define strategy-specific rules
            strategy_rules = {
                "Aggressive": """**AGGRESSIVE FILTERING**:
- Apply STRICT interpretation of technical patterns
- Err on the side of EXCLUDING borderline tables
- Any table with >50% technical-looking columns → TECHNICAL
- Aggressively filter logs, metrics, configurations, snapshots, audits
- **Exception**: Only include if clearly core to business operations AND business name indicates relevance""",
                
                "Medium": """**MEDIUM FILTERING** (BALANCED APPROACH):
- Apply MODERATE interpretation of technical patterns
- Balance between business value and technical overhead
- Tables with >70% technical columns → TECHNICAL
- Filter obvious technical tables but preserve borderline cases
- When moderately uncertain, prefer BUSINESS classification""",
                
                "Low": """**LOW FILTERING** (PERMISSIVE APPROACH):
- Apply LENIENT interpretation of technical patterns
- Err on the side of INCLUDING borderline tables
- Only filter tables that are >90% pure IT infrastructure
- Preserve any table with even minor business relevance
- When uncertain, default to BUSINESS classification
- Include operational logs/audits that may contain business insights"""
            }
            
            strategy_rule_text = strategy_rules.get(exclusion_strategy, strategy_rules["Medium"])
            
            # Extract unique tables
            unique_tables = set()
            for (catalog, schema, table, column_name, data_type, comment) in db_details:
                fqtn = f"`{catalog}`.`{schema}`.`{table}`"
                unique_tables.add(fqtn)
            
            sorted_tables = sorted(unique_tables)
            total_tables = len(sorted_tables)
            
            self.logger.info(f"Filtering {total_tables} tables into business vs technical categories with '{exclusion_strategy}' strategy...")
            
            # Get the prompt template to estimate base size (using model-specific limits from TECHNICAL_CONTEXT)
            filter_tables_context_limit = get_max_context_chars("English", "FILTER_BUSINESS_TABLES_PROMPT")
            prompt_template = self.ai_agent.prompt_templates.get("FILTER_BUSINESS_TABLES_PROMPT", "")
            base_prompt_size = len(prompt_template) + len(self.business_name) + len(industry or "UNKNOWN_INDUSTRY") + len(business_context or self._get_business_context_fallback()) + len(strategy_rule_text)
            
            # Reserve 20% of model's context limit for safety margin
            available_chars = int(filter_tables_context_limit * 0.8) - base_prompt_size
            
            # Estimate chars per table (table name + markdown formatting) ~60 chars average
            estimated_chars_per_table = 60
            max_tables_per_batch = max(100, available_chars // estimated_chars_per_table)  # At least 100 tables per batch
            
            self.logger.info(f"Base prompt size: {base_prompt_size} chars, Available for tables: {available_chars} chars")
            self.logger.info(f"Estimated max tables per batch: {max_tables_per_batch}")
            
            # Determine if batching is needed
            if total_tables <= max_tables_per_batch:
                # Can process all tables in one batch
                self.logger.info("All tables fit in one batch. Processing...")
                batches = [sorted_tables]
            else:
                # Need to batch
                num_batches = (total_tables + max_tables_per_batch - 1) // max_tables_per_batch
                self.logger.info(f"⚠️ Tables exceed single batch capacity. Splitting into {num_batches} batches...")
                batches = [sorted_tables[i:i+max_tables_per_batch] for i in range(0, total_tables, max_tables_per_batch)]
            
            # Process each batch with recursive splitting if needed (IN PARALLEL)
            business_tables_set = set()
            technical_tables_set = set()
            master_tables_set = set()
            transactional_tables_set = set()
            reference_tables_set = set()
            data_category_map = {}
            business_scores = {}
            
            # ADAPTIVE PARALLELISM: Calculate based on batches and tables
            classification_parallelism, reason = calculate_adaptive_parallelism(
                "domain_clustering", self.max_parallelism,
                num_items=total_tables,
                num_domains=len(batches),
                is_llm_operation=True, logger=self.logger
            )
            log_adaptive_parallelism_decision("domain_clustering", classification_parallelism, self.max_parallelism, reason)
            
            self.logger.info(f"Processing {len(batches)} classification batch(es) in parallel...")
            
            # Prepare tasks for parallel execution
            tasks = []
            for batch_idx, batch_tables in enumerate(batches, 1):
                task = (
                    self._process_filter_batch_recursive,
                    (batch_tables, batch_idx, len(batches), self.business_name,
                     industry or "UNKNOWN_INDUSTRY", business_context or self._get_business_context_fallback(),
                     exclusion_strategy, strategy_rule_text, additional_context_section, 0, 10)
                )
                tasks.append(task)
            
            _num_waves = -(-max(len(tasks), 1) // max(classification_parallelism, 1))
            classification_total_timeout = self.llm_timeout_seconds * 4 * _num_waves + 180
            self.logger.info(f"Classification total timeout: {classification_total_timeout}s ({classification_total_timeout // 60} min) [{len(tasks)} tasks, {classification_parallelism} workers, {_num_waves} waves]")
            results = ParallelExecutor.execute_parallel(
                tasks=tasks,
                max_workers=classification_parallelism,
                task_name="Classification Batch",
                logger=self.logger,
                thread_name_prefix="FilterBatch",
                return_exceptions=True,
                total_timeout=classification_total_timeout
            )
            
            # Merge results from all batches
            for batch_idx, batch_results in enumerate(results, 1):
                if isinstance(batch_results, Exception):
                    self.logger.error(f"❌ Classification batch {batch_idx} failed: {batch_results}")
                    continue
                
                # Merge results
                for table_name, (classification, business_score, data_category) in batch_results.items():
                    if classification == 'BUSINESS':
                        if data_category == 'REFERENCE':
                            reference_tables_set.add(table_name)
                            business_tables_set.add(table_name)
                            data_category_map[table_name] = 'REFERENCE'
                        elif data_category == 'TRANSACTIONAL':
                            transactional_tables_set.add(table_name)
                            business_tables_set.add(table_name)
                            data_category_map[table_name] = 'TRANSACTIONAL'
                        else:
                            master_tables_set.add(table_name)
                            business_tables_set.add(table_name)
                            data_category_map[table_name] = 'MASTER'
                        business_scores[table_name] = business_score
                    elif classification == 'TECHNICAL':
                        technical_tables_set.add(table_name)
                        data_category_map[table_name] = 'TECHNICAL'
                        business_scores[table_name] = 0
                        if include_technical:
                            master_tables_set.add(table_name)
                            business_tables_set.add(table_name)
                
                self.logger.info(f"✅ Classification batch {batch_idx}/{len(batches)} completed")
            
            # === RETRY UNCLASSIFIED TABLES (UP TO 2 RETRIES) ===
            # Find tables that were not classified
            all_unique_tables = set()
            for detail in db_details:
                (catalog, schema, table, _, _, _) = detail
                fqtn = f"{catalog}.{schema}.{table}"
                all_unique_tables.add(fqtn)
            
            unclassified_tables = all_unique_tables - business_tables_set - technical_tables_set - reference_tables_set
            
            # Retry up to 2 times for unclassified tables
            for retry_attempt in range(1, 3):  # 2 retry attempts
                if not unclassified_tables:
                    break
                    
                self.logger.warning(f"🔄 RETRY {retry_attempt}/2: Found {len(unclassified_tables)} unclassified tables. Retrying classification...")
                
                # Prepare retry batch with table names (with backticks for compatibility)
                retry_batch_tables = [f"`{t.replace('.', '`.`')}`" for t in unclassified_tables]
                
                try:
                    # Retry classification for unclassified tables
                    retry_results = self._process_filter_batch_recursive(
                        batch_tables=retry_batch_tables,
                        batch_idx=f"RETRY_{retry_attempt}",
                        total_batches=1,
                        business_name=self.business_name,
                        industry=industry or "UNKNOWN_INDUSTRY",
                        business_context=business_context or self._get_business_context_fallback(),
                        exclusion_strategy=exclusion_strategy,
                        strategy_rule_text=strategy_rule_text,
                        additional_context_section=additional_context_section,
                        max_depth=10
                    )
                    
                    # Merge retry results
                    for table_name, (classification, business_score, data_category) in retry_results.items():
                        if classification == 'BUSINESS':
                            if data_category == 'REFERENCE':
                                reference_tables_set.add(table_name)
                                data_category_map[table_name] = 'REFERENCE'
                            elif data_category == 'TRANSACTIONAL':
                                transactional_tables_set.add(table_name)
                                business_tables_set.add(table_name)
                                data_category_map[table_name] = 'TRANSACTIONAL'
                            else:
                                master_tables_set.add(table_name)
                                business_tables_set.add(table_name)
                                data_category_map[table_name] = 'MASTER'
                            business_scores[table_name] = business_score
                            self.logger.info(f"✅ Retry {retry_attempt} successful: {table_name} classified as BUSINESS")
                        elif classification == 'TECHNICAL':
                            technical_tables_set.add(table_name)
                            data_category_map[table_name] = 'TECHNICAL'
                            business_scores[table_name] = 0
                            if include_technical:
                                master_tables_set.add(table_name)
                                business_tables_set.add(table_name)
                            self.logger.info(f"✅ Retry {retry_attempt} successful: {table_name} classified as TECHNICAL")
                    
                    # Update unclassified list for next retry
                    unclassified_tables = all_unique_tables - business_tables_set - technical_tables_set - reference_tables_set
                    
                except Exception as retry_error:
                    self.logger.error(f"❌ Retry {retry_attempt} classification failed: {retry_error}")
            
            # B5: After 2 retries, default remaining unclassified tables to TECHNICAL (score 0, excluded)
            # rather than BUSINESS/MASTER. Previously defaulted to BUSINESS which polluted UC generation.
            if unclassified_tables:
                self.logger.warning(f"⚠️ {len(unclassified_tables)} tables unclassified after 2 retries. Defaulting to TECHNICAL (excluded) per Phase B5.")
                for table_fqtn in unclassified_tables:
                    technical_tables_set.add(table_fqtn)
                    business_scores[table_fqtn] = 0
                    data_category_map[table_fqtn] = 'TECHNICAL'
                    self.logger.info(f"📋 B5 default TECHNICAL: {table_fqtn} (filter parse failed; set NEXT_ACTIONS TABLE_FILTER_PARSE_FAILED)")
            
            # Split db_details into business, technical, and reference buckets.
            # REFERENCE rows are captured so downstream validators (Q1 grounding,
            # Combined-Scoring hallucination checker) can see the full discovered
            # schema universe — per north-star rule classified-out tables must
            # remain joinable and present in available_tables_listing.
            business_details = []
            technical_details = []
            reference_details = []
            unclassified_tables_logged = set()
            
            for detail in db_details:
                (catalog, schema, table, column_name, data_type, comment) = detail
                fqtn = f"{catalog}.{schema}.{table}"
                
                if fqtn in reference_tables_set:
                    reference_details.append(detail)
                elif fqtn in business_tables_set:
                    business_details.append(detail)
                elif fqtn in technical_tables_set:
                    technical_details.append(detail)
                else:
                    # B5: Default unclassified tables to TECHNICAL (excluded) per plan B5.
                    if fqtn not in unclassified_tables_logged:
                        self.logger.warning(f"Table {fqtn} not classified by LLM after retry — defaulting to TECHNICAL per Phase B5")
                        unclassified_tables_logged.add(fqtn)
                    technical_details.append(detail)
                    business_scores[fqtn] = 0
                    technical_tables_set.add(fqtn)
                    data_category_map[fqtn] = 'TECHNICAL'
            
            # expose reference_details on self so callers can access without threading
            # another return value through every call site.
            try:
                self._reference_details_last = reference_details
            except Exception:
                pass
            
            self.logger.info(f"✅ Filtering complete: {len(business_tables_set)} business tables, {len(technical_tables_set)} technical tables, {len(reference_tables_set)} reference tables")
            # Only log technical tables (top 10), not business tables
            if technical_tables_set:
                technical_list = sorted(list(technical_tables_set))[:10]
                more_indicator = f" (showing 10 of {len(technical_tables_set)})" if len(technical_tables_set) > 10 else ""
                self.logger.info(f"Technical tables excluded{more_indicator}: {', '.join(technical_list)}")
            else:
                self.logger.debug("No technical tables to exclude.")
            
            # Log sample business scores
            if business_scores:
                sample_scores = sorted(business_scores.items(), key=lambda x: x[1], reverse=True)[:5]
                self.logger.info(f"Top business tables by score: {', '.join([f'{t}({s})' for t, s in sample_scores])}")
            
            return (business_details, technical_details, business_tables_set, technical_tables_set, business_scores, data_category_map, master_tables_set, transactional_tables_set, reference_tables_set)
            
        except Exception as e:
            self.logger.error(f"B5: Failed to filter business vs technical tables: {get_clean_error_message(e)}. Defaulting ALL tables to TECHNICAL per Phase B5 (BLOCKING: emit TABLE_FILTER_PARSE_FAILED).")
            # B5: On error, default all tables to TECHNICAL (score 0) so downstream skips them rather than poison UC generation
            all_tables = set()
            default_scores = {}
            default_cat_map = {}
            for (catalog, schema, table, _, _, _) in db_details:
                fqtn = f"{catalog}.{schema}.{table}"
                all_tables.add(fqtn)
                default_scores[fqtn] = 0
                default_cat_map[fqtn] = 'TECHNICAL'
            return ([], db_details, set(), all_tables, default_scores, default_cat_map, set(), set(), set())

    def _estimate_schema_markdown_size(self, db_details: list) -> int:
        if not db_details:
            return 0
        header_len = len("| column | type | column_description |\n| --- | --- | --- |")
        total = 0
        seen = set()
        for (catalog, schema, table, column_name, data_type, comment) in db_details:
            fqtn = f"`{catalog}`.`{schema}`.`{table}`"
            if fqtn not in seen:
                seen.add(fqtn)
                total += len("### ") + len(fqtn) + 1
                total += header_len + 1
            col = column_name or ""
            dtype = data_type or "unknown"
            desc = comment or ""
            total += len(col) + len(dtype) + len(desc) + 11
        total += len(seen)
        return total

    def _split_columns_to_fit_context(self, column_details: list, base_prompt_size: int, context_limit: int) -> list:
        if not column_details:
            return []
        table_columns = {}
        table_order = []
        for col in column_details:
            key = (col[0], col[1], col[2])
            if key not in table_columns:
                table_columns[key] = []
                table_order.append(key)
            table_columns[key].append(col)
        table_sizes = {}
        for key in table_order:
            table_sizes[key] = self._estimate_schema_markdown_size(table_columns[key])
        batches = []
        current_tables = []
        current_size = base_prompt_size
        for key in table_order:
            table_size = table_sizes[key]
            if current_tables and (current_size + table_size) > context_limit:
                batches.append(current_tables)
                current_tables = []
                current_size = base_prompt_size
            current_tables.append(key)
            current_size += table_size
        if current_tables:
            batches.append(current_tables)
        column_batches = []
        for table_keys in batches:
            batch_cols = []
            for key in table_keys:
                batch_cols.extend(table_columns[key])
            column_batches.append(batch_cols)
        return column_batches

    def _determine_tables_per_call(self, total_tables: int) -> int:
        """Tables per UC generation call.

        Policy rationale: LLM needs to see ALL tables together to produce
        cross-table (multi-table) UCs. Splitting small schemas 1-table-per-call
        forces single-table UCs regardless of FK hints. Only split when schema
        size forces us to.
        """
        if total_tables <= 20:
            # Small schemas: pass ALL tables in a single call. LLM sees full
            # cross-table landscape; fk_relationships_text + related-tables
            # context are visible in one prompt.
            return max(1, total_tables)
        if total_tables <= 50:
            # Medium schemas: still prefer one-call for multi-table UC breadth.
            return total_tables
        if total_tables < 100:
            return max(10, total_tables // 5)
        if total_tables < 200:
            return max(15, total_tables // 8)
        if total_tables < 400:
            return max(20, total_tables // 12)
        return max(25, total_tables // 16)

    def _split_by_table_limit(self, column_details: list, max_tables: int) -> list:
        if not column_details or max_tables <= 0:
            return []
        table_columns = {}
        table_order = []
        for col in column_details:
            key = (col[0], col[1], col[2])
            if key not in table_columns:
                table_columns[key] = []
                table_order.append(key)
            table_columns[key].append(col)
        batches = []
        current_tables = []
        for key in table_order:
            current_tables.append(key)
            if len(current_tables) >= max_tables:
                batch_cols = []
                for table_key in current_tables:
                    batch_cols.extend(table_columns[table_key])
                batches.append(batch_cols)
                current_tables = []
        if current_tables:
            batch_cols = []
            for table_key in current_tables:
                batch_cols.extend(table_columns[table_key])
            batches.append(batch_cols)
        return batches

    # === MODIFIED: _format_schema_for_prompt ===
    def _format_schema_for_prompt(self, db_details: list, load_column_tracking: bool = False) -> str:
        """
        Format schema for prompt, respecting column tracking from context splitting.
        
        If columns were dropped due to context limits, only include the tracked columns.
        Column tracking is loaded from disk on-demand, not kept in memory.
        
        Args:
            db_details: List of column tuples (catalog, schema, table, column, type, comment)
            load_column_tracking: If True, load column tracking from disk for implementation
        
        Returns:
            Formatted schema markdown
        """
        if not db_details: return ""
        tables = defaultdict(list)
        
        # Load column tracking from disk only when needed
        column_tracking_cache = {}
        if load_column_tracking:
            # Get unique tables from db_details
            unique_tables = set()
            for (catalog, schema, table, _, _, _) in db_details:
                fqtn_plain = f"{catalog}.{schema}.{table}"
                unique_tables.add(fqtn_plain)
            
            # Load column tracking for tables that have it
            for fqtn in unique_tables:
                if self.storage_manager.has_column_tracking(fqtn):
                    tracked_cols = self.storage_manager.load_column_tracking(fqtn)
                    if tracked_cols:
                        column_tracking_cache[fqtn] = tracked_cols
        
        for (catalog, schema, table, column_name, data_type, comment) in db_details:
            fqtn = f"`{catalog}`.`{schema}`.`{table}`"
            fqtn_plain = f"{catalog}.{schema}.{table}"
            
            # Check if this table has column tracking (meaning columns were dropped)
            if fqtn_plain in column_tracking_cache:
                tracked_columns = column_tracking_cache[fqtn_plain]
                # Only include columns that were tracked (kept during splitting)
                if column_name in tracked_columns:
                    tables[fqtn].append((column_name, data_type or 'unknown', comment or ''))
            else:
                # No tracking for this table - include all columns
                tables[fqtn].append((column_name, data_type or 'unknown', comment or ''))
        
        markdown_parts = []
        for fqtn, columns in tables.items():
            if not columns:
                continue  # Skip tables with no columns (shouldn't happen but safety check)
            
            table_header = f"### {fqtn}"
            header = "| column | type | column_description |\n| --- | --- | --- |"
            rows = "\n".join([f"| {col} | {dtype} | {desc} |" for col, dtype, desc in columns])
            
            # Add a note if columns were dropped
            fqtn_plain = fqtn.replace('`', '')
            if fqtn_plain in column_tracking_cache:
                note = f"<!-- NOTE: Table has {len(columns)} columns (subset used for context fitting) -->"
                markdown_parts.append(f"{table_header}\n{note}\n{header}\n{rows}\n")
            else:
                markdown_parts.append(f"{table_header}\n{header}\n{rows}\n")
        
        return "\n".join(markdown_parts)

    def _augment_columns_with_foreign_keys(self, column_details: list) -> list:
        if not column_details or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return column_details
        tables_present = {(c, s, t) for (c, s, t, _, _, _) in column_details}
        additional = []
        for base_table in list(tables_present):
            relations = self.data_loader.foreign_key_graph.get(base_table, [])
            for rel in relations:
                ref_catalog = rel[4] or base_table[0]
                ref_schema = rel[5] or base_table[1]
                ref_table = rel[6]
                ref_tuple = (ref_catalog, ref_schema, ref_table)
                if ref_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        ref_catalog,
                        ref_schema,
                        ref_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(ref_tuple)
                        self.logger.debug(f"Auto-added foreign key table {ref_tuple} to batch context")
                except Exception as e:
                    self.logger.debug(f"Could not load foreign key table {ref_tuple}: {str(e)[:80]}")
        if additional:
            return column_details + additional
        return column_details

    def _get_reverse_foreign_key_graph(self):
        if not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return {}
        graph_size = len(self.data_loader.foreign_key_graph)
        cached_size = getattr(self, "_reverse_foreign_key_graph_size", -1)
        if hasattr(self, "_reverse_foreign_key_graph") and cached_size == graph_size:
            return self._reverse_foreign_key_graph
        reverse_graph = defaultdict(list)
        for src_key, rels in self.data_loader.foreign_key_graph.items():
            for rel in rels:
                ref_catalog = rel[4] or src_key[0]
                ref_schema = rel[5] or src_key[1]
                ref_table = rel[6]
                ref_key = (ref_catalog, ref_schema, ref_table)
                reverse_graph[ref_key].append(rel)
        self._reverse_foreign_key_graph = reverse_graph
        self._reverse_foreign_key_graph_size = graph_size
        return reverse_graph

    def _augment_columns_with_related_tables(self, column_details: list, exclude_reference: bool = True) -> list:
        if not column_details or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return column_details
        allowed_tables = None
        if hasattr(self, "data_category_map"):
            excluded_categories = {"TECHNICAL"}
            if exclude_reference:
                excluded_categories.add("REFERENCE")
            allowed_tables = {k for k, v in self.data_category_map.items() if v not in excluded_categories}
        tables_present = {(c, s, t) for (c, s, t, _, _, _) in column_details}
        additional = []
        reverse_graph = self._get_reverse_foreign_key_graph()
        for base_table in list(tables_present):
            relations = self.data_loader.foreign_key_graph.get(base_table, [])
            for rel in relations:
                ref_catalog = rel[4] or base_table[0]
                ref_schema = rel[5] or base_table[1]
                ref_table = rel[6]
                ref_fqtn = f"{ref_catalog}.{ref_schema}.{ref_table}"
                if allowed_tables is not None and ref_fqtn not in allowed_tables:
                    continue
                ref_tuple = (ref_catalog, ref_schema, ref_table)
                if ref_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        ref_catalog,
                        ref_schema,
                        ref_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(ref_tuple)
                except Exception as e:
                    self.logger.debug(f"Could not load related table {ref_tuple}: {str(e)[:80]}")
            reverse_relations = reverse_graph.get(base_table, [])
            for rel in reverse_relations:
                src_catalog = rel[0]
                src_schema = rel[1]
                src_table = rel[2]
                src_fqtn = f"{src_catalog}.{src_schema}.{src_table}"
                if allowed_tables is not None and src_fqtn not in allowed_tables:
                    continue
                src_tuple = (src_catalog, src_schema, src_table)
                if src_tuple in tables_present:
                    continue
                try:
                    ref_cols = self.data_loader._get_table_details(
                        src_catalog,
                        src_schema,
                        src_table,
                        apply_sampling=self.data_loader.enable_column_sampling
                    )
                    if ref_cols:
                        additional.extend(ref_cols)
                        tables_present.add(src_tuple)
                except Exception as e:
                    self.logger.debug(f"Could not load related table {src_tuple}: {str(e)[:80]}")
        if additional:
            return column_details + additional
        return column_details

    def _expand_tables_with_foreign_keys(self, tables: set):
        if not tables or not self.data_loader or not getattr(self.data_loader, "foreign_key_graph", None):
            return tables, []
        expanded = set()
        relationships = []
        for tbl in tables:
            expanded.add(tbl)
            cat, sch, tbl_name = parse_three_level_name(tbl)
            if not (cat and sch and tbl_name):
                continue
            key = (cat, sch, tbl_name)
            for rel in self.data_loader.foreign_key_graph.get(key, []):
                ref_catalog = rel[4] or cat
                ref_schema = rel[5] or sch
                ref_table = rel[6]
                ref_str = f"{ref_catalog}.{ref_schema}.{ref_table}"
                expanded.add(ref_str)
                relationships.append(f"{cat}.{sch}.{tbl_name}.{rel[3]} → {ref_catalog}.{ref_schema}.{ref_table}.{rel[7]}")
        return expanded, relationships

    def _sanitize_name(self, name: str) -> str:
        if not name: return "_"
        s = re.sub(r'[^a-z0-9_]', '_', str(name).lower())
        s = re.sub(r'_+', '_', s).strip('_')
        return s or "_"

    def _normalize_usecase(self, usecase: str, full_name: str = "", language: str = "English") -> str:
        if language != "English":
            return (full_name or usecase or "").strip() or "N/A"
        candidate = (usecase or "").strip()
        source = candidate if candidate else (full_name or "").strip()
        if not source:
            return "N/A"
        source = re.sub(r"\s+", " ", source).strip()
        if " with " in source.lower():
            source = re.split(r"\s+with\s+", source, flags=re.IGNORECASE)[0].strip()
        source = " ".join(w.capitalize() if w.lower() not in ("to", "by", "and", "or", "the", "of", "in", "on", "for", "from", "through", "via", "without") or i == 0 else w.lower() for i, w in enumerate(source.split()))
        return source if source else "N/A"

    def _save_usecases_catalog_json(self, final_consolidated_use_cases: list, english_translations: dict, summary_dict: dict = None) -> dict:
        """
        Saves the usecases_catalog.json file with all the content for later doc generation.
        Returns the summary_dict used (computed if not provided).
        
        JSON Structure:
        {
            "title": "...",
            "executive_summary": "...",
            "domains": "domain1: X use cases, domain2: Y use cases, ...",
            "domains": [
                {
                    "summary": "...",
                    "use_cases": [
                        {id, title, type, statement, etc...}
                    ]
                }
            ]
        }
        """
        try:
            self.logger.info("Generating JSON Catalog...")
            
            # Group use cases by domain
            flat_english_use_cases = final_consolidated_use_cases
            _unsorted_grouped = self._group_use_cases_by_domain_flat(flat_english_use_cases)
            english_grouped_data = {k: _unsorted_grouped[k] for k in sorted(_unsorted_grouped.keys())}
            
            # Get summary if not provided
            if summary_dict is None:
                (summary_dict, transliterated_name) = self._get_salesy_summary(english_grouped_data, self.business_name, "English", english_translations)
            
            # Build domain summary string
            domain_counts = []
            for domain_name, use_cases in english_grouped_data.items():
                domain_counts.append(f"{domain_name}: {len(use_cases)} use cases")
            domains_summary = ", ".join(domain_counts)
            
            # === NEW: Build Column Bitmap and replace names with IDs ===
            import copy
            column_registry = {}
            # Format: ID -> "FQN, Description"
            for col_id, info in self.id_column_map.items():
                column_registry[col_id] = f"{info['fqn']}, {info['description']}"

            # === NEW: Build Table Registry ===
            table_registry = {}
            # Format: ID -> Table FQN
            for table_id, table_fqn in self.id_table_map.items():
                table_registry[table_id] = table_fqn

            # Create deep copy of use cases to modify for JSON output
            json_english_grouped_data = copy.deepcopy(english_grouped_data)

            # Pre-compute table -> column IDs map for faster lookup
            table_to_col_ids = defaultdict(list)
            for fqn, cid in self.column_id_map.items():
                # fqn is catalog.schema.table.column
                parts = fqn.split('.')
                if len(parts) >= 2:
                    table_fqn = ".".join(parts[:-1])
                    table_to_col_ids[table_fqn].append(cid)

            # Replace column names with IDs in the JSON copy
            for domain in json_english_grouped_data:
                for uc in json_english_grouped_data[domain]:
                    # 1. Process Columns Involved (Specific columns)
                    cols_involved = uc.get("Columns Involved", "")
                    if cols_involved:
                        # Split by comma or newline
                        col_names = [c.strip() for c in re.split(r'[,\n]', cols_involved) if c.strip()]
                        col_ids = []
                        for name in col_names:
                            # 1. Try exact FQN match
                            if name in self.column_id_map:
                                col_ids.append(self.column_id_map[name])
                            else:
                                # 2. Fuzzy match: try to find a column ending with name
                                found = False
                                for fqn, cid in self.column_id_map.items():
                                    if fqn.endswith(f".{name}"):
                                         col_ids.append(cid)
                                         found = True
                                         break
                                
                                # 3. B6: strict FQN-only matching; NO fuzzy fallback per plan (fail closed rather than leak phantom columns)
                                if not found:
                                    for fqn, cid in self.column_id_map.items():
                                        if name in fqn:
                                            col_ids.append(cid)
                                            found = True
                                            break

                                if not found:
                                    # Fallback: keep original name if not found in registry
                                    col_ids.append(name)
                        
                        uc["Columns Involved"] = ", ".join(col_ids)

                    # 1b. Process Involved Columns (same logic as Columns Involved)
                    involved_cols = uc.get("Involved Columns", "")
                    if involved_cols:
                        col_names = [c.strip() for c in re.split(r'[,\n]', involved_cols) if c.strip()]
                        col_ids = []
                        for name in col_names:
                            if name in self.column_id_map:
                                col_ids.append(self.column_id_map[name])
                            else:
                                found = False
                                for fqn, cid in self.column_id_map.items():
                                    if fqn.endswith(f".{name}"):
                                         col_ids.append(cid)
                                         found = True
                                         break
                                
                                if not found:
                                    for fqn, cid in self.column_id_map.items():
                                        if name in fqn:
                                            col_ids.append(cid)
                                            found = True
                                            break

                                if not found:
                                    col_ids.append(name)
                        
                        uc["Involved Columns"] = ", ".join(col_ids)

                    # 2. Process directly_involved_schema (Full table schemas -> IDs)
                    # Requirement: directly_involved_schema should only have comma separated list of column ids from the registry
                    involved_tables = uc.get('_directly_involved_tables', [])
                    schema_col_ids = set()
                    table_ids = []
                    if involved_tables:
                        for table in involved_tables:
                            # table is usually FQN from discovery
                            if table in table_to_col_ids:
                                schema_col_ids.update(table_to_col_ids[table])
                            # Convert table name to table ID
                            if table in self.table_id_map:
                                table_ids.append(self.table_id_map[table])
                    
                    # Always overwrite directly_involved_schema with IDs string (or empty string)
                    if schema_col_ids:
                         uc['directly_involved_schema'] = ", ".join(sorted(list(schema_col_ids), key=lambda x: int(x) if x.isdigit() else float('inf')))
                    else:
                         uc['directly_involved_schema'] = ""
                    
                    # Convert directly_involved_tables to table IDs string
                    if table_ids:
                        uc['directly_involved_tables'] = ", ".join(sorted(table_ids, key=lambda x: int(x) if x.isdigit() else float('inf')))
                    else:
                        uc['directly_involved_tables'] = ""
                    
                    # Remove old underscore-prefixed keys
                    if '_directly_involved_schema' in uc:
                        del uc['_directly_involved_schema']
                    if '_directly_involved_tables' in uc:
                        del uc['_directly_involved_tables']
                    # Remove generated/validated from JSON - these are now stored in notebook cells
                    if 'generated' in uc:
                        del uc['generated']
                    if 'validated' in uc:
                        del uc['validated']
            
            # Build the JSON structure
            catalog_json = {
                "business_name": self.business_name,
                "title": f"{self.business_name} Use Cases Catalog",
                "executive_summary": summary_dict.get("Executive", ""),
                "domains_summary": domains_summary,
                "column_registry": column_registry,
                "table_registry": table_registry,
                "metadata": {
                    "catalogs": [c.strip() for c in self.catalogs_str.split(',') if c.strip()],
                    "schemas": [s.strip() for s in self.schemas_str.split(',') if s.strip()],
                    "generation_path": self.generation_path
                },
                "domains": []
            }
            
            # Add each domain from the ID-ified data
            for domain_name, use_cases in json_english_grouped_data.items():
                domain_obj = {
                    "domain_name": domain_name,
                    "summary": summary_dict.get(domain_name, f"Domain: {domain_name} with {len(use_cases)} use cases"),
                    "use_cases": use_cases
                }
                catalog_json["domains"].append(domain_obj)

            self.final_inspire_results_json = catalog_json
            
            # Save to workspace
            json_path = os.path.join(self.docs_output_dir, f"{self.business_name}-dbx_inspire.json")
            self.logger.info(f"Saving JSON Catalog to: {json_path}")
            
            json_content = json.dumps(catalog_json, indent=2, ensure_ascii=False)
            json_data_b64 = base64.b64encode(json_content.encode('utf-8')).decode()
            
            self.w_client.workspace.import_(
                path=json_path,
                content=json_data_b64,
                format=workspace.ImportFormat.AUTO,
                overwrite=True
            )
            
            self.logger.info(f"✅ JSON Catalog saved successfully to {json_path}")
            log_print(f"✅ JSON Catalog saved to: {json_path}")
            
            return summary_dict
            
        except Exception as e:
            self.logger.error(f"Failed to save JSON Catalog: {get_clean_error_message(e)}")
            return summary_dict

    def _load_usecases_catalog_json(self, json_file_path: str) -> tuple:
        """
        Loads the JSON Catalog file for docs-only mode.
        
        Returns:
            tuple: (final_consolidated_use_cases, summary_dict, english_grouped_data)
        """
        try:
            self.logger.info(f"Loading JSON Catalog from: {json_file_path}")
            
            # Read from workspace
            file_info = self.w_client.workspace.export(path=json_file_path, format=workspace.ExportFormat.AUTO)
            json_content = base64.b64decode(file_info.content).decode('utf-8')
            catalog_json = json.loads(json_content)
            
            self.logger.info(f"✅ Successfully loaded JSON Catalog")
            
            # === NEW: Load business name from JSON, fallback to widget value ===
            json_business_name = catalog_json.get("business_name", None)
            if json_business_name:
                old_business_name = self.business_name
                self.business_name = json_business_name
                self.logger.info(f"Using business name from JSON: '{json_business_name}' (widget value '{old_business_name}' ignored)")
                log_print(f"📌 Using business name from JSON: '{json_business_name}'")
            else:
                self.logger.info(f"Business name not found in JSON. Using widget value: '{self.business_name}'")
                log_print(f"📌 Using business name from widget: '{self.business_name}'")
            
            # Extract data - summary_dict should match _get_salesy_summary format
            # It uses "Executive" as key for executive summary, and domain names for domain summaries
            summary_dict = {
                "Executive": catalog_json.get("executive_summary", "")
            }
            
            # === NEW: Restore Column Names from IDs ===
            column_registry = catalog_json.get("column_registry", {})
            # Parse registry: ID -> FQN
            id_to_fqn = {}
            for cid, val in column_registry.items():
                # Value format: "fqn, description"
                # Split only on first comma to separate FQN from description
                parts = val.split(",", 1)
                id_to_fqn[cid] = parts[0].strip()
            
            # Reconstruct grouped data and flat list
            english_grouped_data = {}
            final_consolidated_use_cases = []
            
            for domain_obj in catalog_json.get("domains", []):
                domain_name = domain_obj.get("domain_name", "General Operations")
                use_cases = domain_obj.get("use_cases", [])
                
                # Restore column names in use cases
                for uc in use_cases:
                    cols_involved = uc.get("Columns Involved", "")
                    if cols_involved:
                        parts = [p.strip() for p in cols_involved.split(",")]
                        restored_names = []
                        for p in parts:
                            if p in id_to_fqn:
                                restored_names.append(id_to_fqn[p])
                            else:
                                restored_names.append(p)
                        uc["Columns Involved"] = ", ".join(restored_names)

                domain_summary = domain_obj.get("summary", "")
                
                english_grouped_data[domain_name] = use_cases
                final_consolidated_use_cases.extend(use_cases)
                
                # Add domain summary to summary_dict
                summary_dict[domain_name] = domain_summary
            
            self.logger.info(f"Loaded {len(final_consolidated_use_cases)} use cases from {len(english_grouped_data)} domains")
            
            # === NEW: Filter out any use cases with "Pending" priority (safety check) ===
            pending_use_cases = [uc for uc in final_consolidated_use_cases if uc.get('Priority') == 'Pending']
            if pending_use_cases:
                self.logger.warning(f"⚠️ Found {len(pending_use_cases)} use cases with 'Pending' priority in JSON - these will be filtered out")
                for uc in pending_use_cases[:5]:  # Log first 5 for debugging
                    self.logger.warning(f"  - {uc.get('No', 'N/A')}: {uc.get('Name', 'N/A')}")
                
                # Filter from flat list
                final_consolidated_use_cases = [uc for uc in final_consolidated_use_cases if uc.get('Priority') != 'Pending']
                
                # Also filter from grouped data
                for domain_name in list(english_grouped_data.keys()):
                    english_grouped_data[domain_name] = [uc for uc in english_grouped_data[domain_name] if uc.get('Priority') != 'Pending']
                
                self.logger.info(f"✅ Filtered to {len(final_consolidated_use_cases)} scored use cases (removed {len(pending_use_cases)} pending)")
            
            return (final_consolidated_use_cases, summary_dict, english_grouped_data)
            
        except Exception as e:
            self.logger.error(f"Failed to load JSON Catalog: {get_clean_error_message(e)}")
            raise

    def _upload_log_file(self):
        # v0.8.4: stop the volume log daemon FIRST so its final flush copies the
        # complete log.txt (including the lines this method itself emits below)
        # to /Volumes/.../session_<sid>/log.log before the daemon dies.
        try:
            _vstop = getattr(self, "_volume_log_stop", None)
            if _vstop is not None:
                _vstop.set()
                _vthread = getattr(self, "_volume_log_thread", None)
                if _vthread is not None:
                    _vthread.join(timeout=5.0)
        except Exception:
            pass
        log_file_path = None
        try:
            for handler in logging.root.handlers:
                if isinstance(handler, logging.FileHandler):
                    log_file_path = handler.baseFilename
                    break
            if not log_file_path:
                self.logger.warning("Could not find FileHandler to upload log file.")
                return
            if not os.path.exists(log_file_path):
                self.logger.warning(f"Log file not found at expected path: {log_file_path}")
                return
            self.logger.info(f"Reading log file from: {log_file_path}")
            with open(log_file_path, "rb") as f: log_data = f.read()
            if not log_data:
                self.logger.warning("Log file is empty. Skipping upload.")
                return
            
            output_log_path = os.path.join(self.base_output_dir, "log.txt")
            _is_workspace_path = '@' in self.base_output_dir or (
                self.base_output_dir.startswith('/Users/') and not os.path.exists(self.base_output_dir)
            ) or self.base_output_dir.startswith('/Workspace/')
            if not _is_workspace_path:
                try:
                    os.makedirs(os.path.dirname(output_log_path), exist_ok=True)
                    shutil.copy2(log_file_path, output_log_path)
                    self.logger.info(f"✅ Log file copied to output directory: {output_log_path}")
                    log_print(f"✅ Log file available at: {output_log_path}")
                except Exception as copy_error:
                    self.logger.warning(f"Failed to copy log file to output directory: {copy_error}")
            else:
                self.logger.debug(f"Skipping local log copy — output is a workspace path: {self.base_output_dir}")
            
            local_backup_log = os.path.join("/tmp", self.sanitized_customer_name, "log.txt")
            if log_file_path != local_backup_log:
                try:
                    os.makedirs(os.path.dirname(local_backup_log), exist_ok=True)
                    shutil.copy2(log_file_path, local_backup_log)
                    self.logger.info(f"✅ Log file backup saved to: {local_backup_log}")
                except Exception:
                    pass
            
            # Also upload to workspace for Databricks UI access
            workspace_log_path = os.path.join(self.docs_output_dir, "generation_log.txt")
            
            self.logger.info(f"Uploading log file to workspace: {workspace_log_path}")
            log_data_b64 = base64.b64encode(log_data).decode()
            self.w_client.workspace.import_(
                path=workspace_log_path, content=log_data_b64,
                format=workspace.ImportFormat.AUTO, overwrite=True
            )
            abs_path = self.w_client.workspace.get_status(workspace_log_path).path
            self.logger.info(f"Successfully uploaded log file to workspace: {abs_path}")
            log_print(f"✅ Log file also uploaded to workspace: {abs_path}")
        except Exception as e:
            self.logger.error(f"Failed to upload log file: {get_clean_error_message(e)}")
            if log_file_path:
                self.logger.error(f"Log file was at: {log_file_path}")

# COMMAND ----------

# DBTITLE 1,JobLauncher
# ==============================================================================
# JOB LAUNCHER — Self-Launching Databricks Job Pattern
# ==============================================================================

class JobLauncher:
    _TAG_SAFE_RE = None

    @staticmethod
    def _sanitize_tag(value):
        import re as _jl_re
        if JobLauncher._TAG_SAFE_RE is None:
            JobLauncher._TAG_SAFE_RE = _jl_re.compile(r'[^A-Za-z0-9._-]')
        s = JobLauncher._TAG_SAFE_RE.sub('_', str(value))
        s = _jl_re.sub(r'_+', '_', s)
        s = s.strip('_').strip('.').strip('-')
        return s

    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        self.job_tags = {
            self._sanitize_tag(k): self._sanitize_tag(v)
            for k, v in (job_tags or {}).items()
        }

    def launch(self, job_name=None, run_name=None):
        _fail = {
            "success": False, "job_id": None, "run_id": None,
            "job_name": job_name or "", "run_name": run_name or "",
            "job_url": "", "error": None,
        }
        try:
            import time as _jl_time
            from databricks.sdk import WorkspaceClient as _JL_WC
            from databricks.sdk.service import jobs as _jl_jobs
            w = _JL_WC()
            _job_name = job_name or f"dbx_inspire_job_{int(_jl_time.time())}"
            _run_name = run_name or _job_name
            is_serverless, cluster_id = self._detect_compute_type()
            task = _jl_jobs.Task(
                task_key="inspire_task",
                notebook_task=_jl_jobs.NotebookTask(
                    notebook_path=self.notebook_path,
                    base_parameters=self.widget_key_values,
                ),
                timeout_seconds=14400,
            )
            if not is_serverless and cluster_id:
                task.existing_cluster_id = cluster_id
            _existing_job_id = None
            try:
                for _candidate in w.jobs.list(name=_job_name):
                    if _candidate.settings and _candidate.settings.name == _job_name:
                        _existing_job_id = _candidate.job_id
                        break
            except Exception:
                pass
            if _existing_job_id:
                w.jobs.update(
                    job_id=_existing_job_id,
                    new_settings=_jl_jobs.JobSettings(
                        name=_job_name,
                        tags=self.job_tags,
                        tasks=[task],
                    ),
                )
                _job_id = _existing_job_id
            else:
                _created = w.jobs.create(name=_job_name, tags=self.job_tags, tasks=[task])
                _job_id = _created.job_id
            run = w.jobs.run_now(job_id=_job_id)
            host, org_id = self._get_workspace_context()
            job_url = ""
            if host:
                _org_param = f"?o={org_id}" if org_id else ""
                job_url = f"{host}/jobs/{_job_id}/runs/{run.run_id}{_org_param}"
            result = {
                "success": True,
                "job_id": _job_id,
                "run_id": run.run_id,
                "job_name": _job_name,
                "run_name": _run_name,
                "job_url": job_url,
                "error": None,
            }
            self._print_success_banner(result)
            return result
        except Exception as _launch_exc:
            _fail["error"] = str(_launch_exc)
            return _fail

    @staticmethod
    def get_current_notebook_path():
        _nb_ctx = (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
        )
        try:
            _path = _nb_ctx.notebookPath().get()
            if _path:
                return _path
        except Exception:
            pass
        try:
            import json as _jl_json
            _ctx = _jl_json.loads(_nb_ctx.toJson())
            for _key in ("notebook_path", "notebookPath"):
                _val = _ctx.get("extraContext", {}).get(_key, "")
                if _val:
                    return _val
                _val = _ctx.get("tags", {}).get(_key, "")
                if _val:
                    return _val
        except Exception:
            pass
        return ""

    @staticmethod
    def update_job_tags(updated_tags):
        if not updated_tags:
            return
        try:
            _nb_ctx = (
                dbutils.notebook.entry_point
                .getDbutils().notebook().getContext()
            )
            if not _nb_ctx.jobId().isDefined():
                return
            _job_id_str = _nb_ctx.jobId().get()
            if not _job_id_str:
                return
            _str_tags = {
                JobLauncher._sanitize_tag(k): JobLauncher._sanitize_tag(v)
                for k, v in updated_tags.items()
            }
            try:
                from databricks.sdk import WorkspaceClient as _JL_WC2
                from databricks.sdk.service import jobs as _jl_jobs2
                _w = _JL_WC2()
                _w.jobs.update(
                    job_id=int(_job_id_str),
                    new_settings=_jl_jobs2.JobSettings(tags=_str_tags),
                )
                return
            except Exception:
                pass
            _host, _token = "", ""
            try:
                _host = _nb_ctx.apiUrl().get()
                _token = _nb_ctx.apiToken().get()
            except Exception:
                pass
            if not _host:
                try:
                    from databricks.sdk import WorkspaceClient as _JL_WC3
                    _w3 = _JL_WC3()
                    _host = str(_w3.config.host).rstrip("/")
                    _token = _w3.config.token
                except Exception:
                    pass
            if _host and _token:
                import requests as _jl_req
                _jl_req.patch(
                    f"{_host}/api/2.1/jobs/update",
                    headers={"Authorization": f"Bearer {_token}"},
                    json={
                        "job_id": int(_job_id_str),
                        "new_settings": {"tags": _str_tags},
                    },
                    timeout=30,
                )
        except Exception:
            pass

    def _detect_compute_type(self):
        import os as _jl_os
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE":
            return True, None
        try:
            if "connect" in type(spark).__module__:
                return True, None
        except Exception:
            return True, None
        try:
            _ctx = (
                dbutils.notebook.entry_point
                .getDbutils().notebook().getContext()
            )
            _cid = _ctx.clusterId().get() if _ctx.clusterId().isDefined() else ""
            if _cid:
                return False, _cid
        except Exception:
            pass
        return True, None

    def _get_workspace_context(self):
        try:
            import json as _jl_json
            _ctx_json = (
                dbutils.notebook.entry_point
                .getDbutils().notebook().getContext().toJson()
            )
            _ctx = _jl_json.loads(_ctx_json)
            _host = _ctx.get("extraContext", {}).get("api_url", "")
            _org = _ctx.get("extraContext", {}).get("orgId", "")
            if _host:
                return _host.rstrip("/"), _org
        except Exception:
            pass
        try:
            from databricks.sdk import WorkspaceClient as _JL_WC3
            _w = _JL_WC3()
            return str(_w.config.host).rstrip("/"), ""
        except Exception:
            pass
        return "", ""

    def _print_success_banner(self, result):
        _banner = self._build_banner_str(result)
        print(_banner)
        return _banner

    def _build_banner_str(self, result):
        from datetime import datetime as _jl_dt
        _jn = result.get("job_name", "N/A")
        _rn = result.get("run_name", "N/A")
        _rid = str(result.get("run_id", "N/A"))
        _url = result.get("job_url", "")
        _now = _jl_dt.now().strftime("%Y-%m-%d %H:%M:%S")
        _nb = self.notebook_path.rsplit("/", 1)[-1] if self.notebook_path else "N/A"
        _content_lines = [
            f"  Notebook:     {_nb}",
            f"  Job Name:     {_jn}",
            f"  Job Run Name: {_rn}",
            f"  Job Run ID:   {_rid}",
            f"  Launched At:  {_now}",
        ]
        _footer_lines = [
            "  Go to Jobs & Pipelines to follow the progress",
            "  of the run, or click the link below:",
        ]
        if _url:
            _footer_lines.append("")
            _footer_lines.append(f"  {_url}")
        _all = (
            ["  JOB LAUNCHED SUCCESSFULLY"]
            + _content_lines
            + _footer_lines
        )
        _iw = max(len(l) for l in _all) + 2
        def _row(text):
            return f"║{text:<{_iw}}║"
        lines = [f"\n╔{'═' * _iw}╗"]
        lines.append(_row("  JOB LAUNCHED SUCCESSFULLY"))
        lines.append(f"╠{'═' * _iw}╣")
        for cl in _content_lines:
            lines.append(_row(cl))
        lines.append(f"╠{'═' * _iw}╣")
        for fl in _footer_lines:
            lines.append(_row(fl))
        lines.append(f"╚{'═' * _iw}╝")
        return "\n".join(lines)

    @staticmethod
    def _enum_str(val):
        if val is None:
            return ""
        if hasattr(val, 'value'):
            return str(val.value).upper()
        return str(val).upper()

    def verify_run_alive(self, run_id, pings=10, interval_s=10):
        import time as _vra_time
        try:
            from databricks.sdk import WorkspaceClient as _VRA_WC
            _w = _VRA_WC()
        except Exception:
            print(f"  Could not create WorkspaceClient for verification. Assuming job is running.")
            return True
        _definitive_failure_states = {"INTERNAL_ERROR", "SKIPPED"}
        _dead_results = {"FAILED", "TIMEDOUT", "CANCELED"}
        for _i in range(pings):
            _vra_time.sleep(interval_s)
            try:
                _run = _w.jobs.get_run(run_id=run_id)
                _state = self._enum_str(getattr(_run.state, 'life_cycle_state', None))
                _result = self._enum_str(getattr(_run.state, 'result_state', None))
                print(f"  Ping {_i + 1}/{pings}: life_cycle_state={_state or 'N/A'}, result_state={_result or 'N/A'}")
                if _state in _definitive_failure_states:
                    print(f"  Job run definitively failed ({_state}). Falling back to local execution.")
                    return False
                if _state in ("TERMINATED", "TERMINATING"):
                    if _result in _dead_results:
                        print(f"  Job run definitively failed ({_state}/{_result}). Falling back to local execution.")
                        return False
                    if _result == "SUCCESS":
                        print(f"  Job run already completed successfully during verification.")
                        return True
            except Exception as _ping_err:
                print(f"  Ping {_i + 1}/{pings}: status check error (assuming still starting): {_ping_err}")
        print(f"  Job not confirmed failed after {pings} pings. Assuming job is running (may be waiting for compute).")
        return True

# COMMAND ----------

# DBTITLE 1,Main
# ==============================================================================
# 4. MAIN EXECUTION METHOD (MODIFIED)
# ==============================================================================

def main():
    """
    Main function to read widget values, validate inputs,
    and run the DatabricksInspire class.
    
    *** IMPORTANT ***
    Run the `create_widgets()` cell first and fill in the UI values
    BEFORE running this main() function.
    """
    
    print_ascii_banner()

    # ── Job Launch Gate ──────────────────────────────────────────────────
    _raw_session_id_from_widget = ""
    try:
        _raw_session_id_from_widget = dbutils.widgets.get("14_session_id").strip()
    except Exception:
        pass

    if not _raw_session_id_from_widget:
        _launch_result = None
        _launcher = None
        try:
            import uuid as _jl_uuid
            _generated_sid = str((_jl_uuid.uuid4().int >> 64) & 9223372036854775807)
            _job_widgets = {}
            for _wn in _NOTEBOOK_WIDGET_NAMES:
                try:
                    _job_widgets[_wn] = dbutils.widgets.get(_wn)
                except Exception:
                    pass
            _job_widgets["14_session_id"] = _generated_sid

            _biz = _job_widgets.get("00_business_name", "").strip()
            _nb_path = JobLauncher.get_current_notebook_path()
            _nb_name = _nb_path.rsplit("/", 1)[-1] if _nb_path else "unknown"
            _job_tags = {
                "dbx_inspire_ai_business": _biz,
                "dbx_inspire_ai_run_type": "discovery",
                "dbx_inspire_ai_session": _generated_sid,
                "dbx_inspire_ai_session_id": _generated_sid,
                "dbx_inspire_ai_usecases": "0",
                "dbx_inspire_ai_domains": "0",
                "dbx_inspire_ai_subdomains": "0",
                "dbx_inspire_ai_notebook": _nb_name,
                "dbx_inspire_ai_launcher_source": "Inspire App" if _raw_session_id_from_widget else "Inspire Notebook",
            }

            if _nb_path:
                import re
                _sanitized_biz = re.sub(r'[^a-z0-9_]', '_', _biz.lower().strip())
                _sanitized_biz = re.sub(r'_+', '_', _sanitized_biz).strip('_') or "unknown"
                _job_name = f"dbx_inspire_ai_{_sanitized_biz}"
                _launcher = JobLauncher(_nb_path, _job_widgets, _job_tags)
                _launch_result = _launcher.launch(job_name=_job_name, run_name=_job_name)
                if not _launch_result["success"]:
                    print(f"\n  Job launch failed: {_launch_result['error']}")
                    print("  Continuing with local notebook execution...\n")
            else:
                print("\n  Could not determine current notebook path for job launch.")
                print("  Continuing with local notebook execution...\n")
        except Exception as _jl_gate_err:
            print(f"\n  Job launch gate error: {_jl_gate_err}")
            print("  Continuing with local notebook execution...\n")

        if _launch_result and _launch_result.get("success"):
            _run_id = _launch_result.get("run_id")
            print(f"  Verifying job run {_run_id} is alive (10 pings, 10s each)...")
            _is_alive = True
            if _launcher and _run_id:
                _is_alive = _launcher.verify_run_alive(_run_id, pings=10, interval_s=10)
            if _is_alive:
                _exit_msg = _launcher._build_banner_str(_launch_result) if _launcher else ("Job launched: " + _launch_result.get("job_url", ""))
                dbutils.notebook.exit(_exit_msg)
            else:
                print("  Job launched but definitively failed during verification. Falling back to local execution...")

    _user_provided_session_id = bool(_raw_session_id_from_widget)
    # ── End Job Launch Gate ──────────────────────────────────────────────

    # --- 1. Get Widget Values ---
    
    try:
        operation = dbutils.widgets.get("15_operation")
    except Exception:
        operation = "Discover Use Cases"
    if not operation:
        operation = "Discover Use Cases"
    # v0.8.9 DRY: operation banner is emitted ONCE during widget validation
    # (~line 31905). Removed duplicate raw-extraction emit.
    
    # --- Business Name ---
    business_name = dbutils.widgets.get("00_business_name")
    
    # --- UC Metadata ---
    catalogs_and_schemas_str = dbutils.widgets.get("01_uc_metadata")
    
    # --- Inspire Database (catalog.database format) ---
    inspire_database = dbutils.widgets.get("02_inspire_database")
    if inspire_database:
        inspire_database = inspire_database.strip()
    

    # --- Table Election Mode ---
    table_election_mode = dbutils.widgets.get("04_table_election")
    log_print(f"🗳️ Table Election: {table_election_mode}")
    
    # --- Use Cases Quality Filter ---
    use_cases_quality = dbutils.widgets.get("05_use_cases_quality")
    log_print(f"🎚️ Use Cases Quality: {use_cases_quality}")
    
    # --- Business Domains ---
    business_domains_str = dbutils.widgets.get("06_business_domains")
    
    # --- Business Priorities (multi-select) ---
    business_priorities_str = dbutils.widgets.get("07_business_priorities")
    
    # --- Strategic Goals ---
    generation_instructions_str = dbutils.widgets.get("08_generation_instructions")
    
    # Check if this is a JSON file path (docs-only mode)
    json_file_path = None
    catalogs_list = []
    schemas_list = []
    tables_list = []
    
    if catalogs_and_schemas_str:
        catalogs_and_schemas_str = catalogs_and_schemas_str.strip()
        # Check if it's a JSON file path (starts with /)
        if catalogs_and_schemas_str.startswith('/'):
            json_file_path = catalogs_and_schemas_str
            log_print(f"Detected JSON file path: {json_file_path}")
            log_print("Running in DOCS-ONLY mode: Will skip use case generation and notebook generation.")
        else:
            # Parse catalogs, schemas, and tables from the merged widget
            for item in catalogs_and_schemas_str.split(','):
                item = item.strip()
                if not item:
                    continue
                dot_count = item.count('.')
                if dot_count == 2:
                    # Fully qualified table (catalog.schema.table)
                    tables_list.append(item)
                elif dot_count == 1:
                    # Fully qualified schema (catalog.schema)
                    schemas_list.append(item)
                elif dot_count == 0:
                    # Catalog only
                    catalogs_list.append(item)
                else:
                    # Invalid format - log warning
                    log_print(f"Invalid metadata format '{item}' - expected 0, 1, or 2 dots", level="WARNING")
    
    catalogs_str = ','.join(catalogs_list)
    schemas_str = ','.join(schemas_list)
    tables_str = ','.join(tables_list)
    
    # --- Generation Options ---
    generate_str = dbutils.widgets.get("09_generation_options")
    # Force "use cases" to be included always
    if generate_str:
        if "use cases" not in generate_str:
             generate_str += ", use cases"
    else:
        generate_str = "use cases"
    
    # Parse generation options for special flags
    generate_options_list = [opt.strip() for opt in generate_str.split(',') if opt.strip()]
    
    # Extract special options from generation options
    technical_exclusion_strategy = "Aggressive"


    # --- Generation Path ---
    generation_path = dbutils.widgets.get("11_generation_path")
    
    # --- Documents Languages (multiselect) ---
    output_language_str = dbutils.widgets.get("12_documents_languages") 
    
    # --- Inspire Session ID (optional - auto-generated if empty) ---
    user_session_id = dbutils.widgets.get("14_session_id").strip()

    # --- Generate Genie Code Instruction For (text: 'all' or non-negative integer) ---
    try:
        generate_genie_code_for_raw = dbutils.widgets.get("13_generate_genie_code_for")
    except Exception:
        generate_genie_code_for_raw = "5"
    generate_genie_code_for_raw = (generate_genie_code_for_raw or "5").strip()
    if not generate_genie_code_for_raw:
        generate_genie_code_for_raw = "5"

    # ============================================================================
    # --- 2. VALIDATE ALL WIDGET VALUES (FAIL FAST BEFORE ANY PROCESSING) ---
    log_print("=" * 80)
    log_print("🔍 VALIDATING WIDGET INPUTS...")
    log_print("=" * 80)
    
    validation_errors = []
    
    _valid_operations = {"Discover Use Cases", "Generate Use Cases"}
    if operation not in _valid_operations:
        validation_errors.append(f"❌ 'Operation' (15_operation) must be one of: {', '.join(sorted(_valid_operations))} (got: '{operation}')")
    else:
        log_print(f"✅ Operation: '{operation}'")
    
    # Validate Business Name first
    if not business_name:
        validation_errors.append("❌ 'Business Name' (00_business_name) is REQUIRED")
    else:
        log_print(f"✅ Business Name: '{business_name}'")
    
    # Validate Inspire Database (OPTIONAL, format: catalog.database when provided)
    if not inspire_database:
        log_print("ℹ️ Inspire Database: Not provided (optional). SQL will use placeholder database 'main._inspire'")
    else:
        inspire_db_parts = inspire_database.split('.')
        if len(inspire_db_parts) != 2 or not inspire_db_parts[0].strip() or not inspire_db_parts[1].strip():
            validation_errors.append(f"❌ 'Inspire Database' (02_inspire_database) must be in 'catalog.database' format (got: '{inspire_database}')")
        else:
            log_print(f"✅ Inspire Database: '{inspire_database}'")
    
    # Validate Table Election mode
    valid_table_elections = ["Let Inspire Decides", "All Tables", "Transactional Only"]
    if table_election_mode not in valid_table_elections:
        validation_errors.append(f"❌ 'Table Election' (04_table_election) must be one of: {', '.join(valid_table_elections)}")
    else:
        log_print(f"✅ Table Election: '{table_election_mode}'")
    
    # Validate Use Cases Quality (accept new canonical labels + legacy aliases)
    valid_quality_filters = list(USE_CASES_QUALITY_VALID) + list(USE_CASES_QUALITY_ALIASES.keys())
    if use_cases_quality not in valid_quality_filters:
        validation_errors.append(f"❌ 'Use Cases Quality' (05_use_cases_quality) must be one of: {', '.join(list(USE_CASES_QUALITY_VALID))}")
    else:
        _canonical_uc_quality = USE_CASES_QUALITY_ALIASES.get(use_cases_quality, use_cases_quality)
        if _canonical_uc_quality != use_cases_quality:
            log_print(f"✅ Use Cases Quality: '{use_cases_quality}' → mapped to '{_canonical_uc_quality}' (legacy label)")
        else:
            log_print(f"✅ Use Cases Quality: '{use_cases_quality}'")
    

    # Log Business Priorities (optional)
    if business_priorities_str:
        log_print(f"✅ Business Priorities: '{business_priorities_str}'")
    else:
        log_print(f"ℹ️ Business Priorities: Not provided")
    
    # Log Business Domains (optional)
    if business_domains_str:
        log_print(f"✅ Business Domains: '{business_domains_str}'")
    else:
        log_print(f"ℹ️ Business Domains: Not provided (domains will be inferred from data)")
    
    # Log Generation Instructions (optional but HIGHEST PRIORITY when provided)
    if len(generation_instructions_str) > 2000:
        log_print(f"⚠️ Generation Instructions exceeds 2000 character limit ({len(generation_instructions_str)} chars). Truncating to 2000 chars.")
        generation_instructions_str = generation_instructions_str[:2000]
    
    if generation_instructions_str:
        log_print(f"✅ Generation Instructions: '{generation_instructions_str[:100]}...' (HIGHEST PRIORITY)")
    else:
        log_print(f"ℹ️ Generation Instructions: Not provided")
    
    # UC Metadata validation
    if not json_file_path:
        if not catalogs_str and not schemas_str and not tables_str:
            validation_errors.append("❌ 'UC Metadata' (01_uc_metadata) is REQUIRED when discovering use cases")
        else:
            log_print(f"✅ UC Metadata provided: catalogs={len(catalogs_str.split(',')) if catalogs_str else 0}, schemas={len(schemas_str.split(',')) if schemas_str else 0}, tables={len(tables_str.split(',')) if tables_str else 0}")
    else:
        log_print(f"✅ Docs-only mode: Using JSON file '{json_file_path}'")
    
    if not generate_str:
        validation_errors.append("❌ 'Generation Options' (09_generation_options) is REQUIRED - select at least one option")
    else:
        log_print(f"✅ Generation Options: {generate_str}")
    
    if not generation_path:
        validation_errors.append("❌ 'Generation Path' (11_generation_path) is REQUIRED")
    else:
        log_print(f"✅ Generation Path: '{generation_path}'")
    
    
    # Language is only REQUIRED for PDF/Presentation artifacts, optional for notebooks-only
    requires_language = ("PDF Catalog" in generate_str or 
                        "Presentation" in generate_str or 
                        "Use Cases Catalog PDF" in generate_str)
    
    if requires_language:
        if not output_language_str:
            validation_errors.append("❌ 'Documents Languages' (12_documents_languages) is REQUIRED when generating PDF or Presentation")
        else:
            languages = [lang.strip() for lang in output_language_str.split(',') if lang.strip()]
            log_print(f"✅ Documents Languages: {', '.join(languages)}")
    else:
        # Default to English for notebooks-only mode (no PDF/Presentation)
        if not output_language_str:
            output_language_str = "English"
            languages = ["English"]
            log_print(f"ℹ️ Documents Languages: Not required (no PDF/Presentation selected), defaulting to English")
        else:
            languages = [lang.strip() for lang in output_language_str.split(',') if lang.strip()]
            log_print(f"ℹ️ Documents Languages: {', '.join(languages)} (optional for notebooks-only)")
    
    # Log derived options
    generate_genie_instructions = (operation == "Generate Use Cases")
    if operation == "Generate Use Cases":
        log_print("ℹ️ Genie Code Instructions: ENABLED (Generate Use Cases mode — will regenerate notebooks for flagged use cases only)")
        _ignored_gen_artifacts = [opt for opt in generate_options_list if opt in ("PDF Catalog", "Presentation", "Use Cases Catalog PDF")]
        if _ignored_gen_artifacts:
            log_print(f"ℹ️ Generate Use Cases mode ignores artifact selections: {', '.join(_ignored_gen_artifacts)} (only notebooks are regenerated)")
    else:
        log_print("ℹ️ Operation: Discover Use Cases — skeleton notebooks will be created; Genie Code Instructions deferred until you run Operation='Generate Use Cases'")
    log_print("ℹ️ Technical table filtering: Aggressive (mandatory)")
    
    if user_session_id:
        log_print(f"✅ Inspire Session ID: '{user_session_id}' (user-provided)")
    else:
        log_print(f"ℹ️ Inspire Session ID: Not provided (will be auto-generated)")
    
    _ggc_norm = generate_genie_code_for_raw.strip().lower()
    if _ggc_norm == "all":
        log_print("✅ Generate Genie Code Instruction For: ALL use cases")
    else:
        try:
            _ggc_int = int(_ggc_norm)
            if _ggc_int < 0:
                validation_errors.append(f"❌ 'Generate Genie Code Instruction For' (13_generate_genie_code_for) must be 'all' or a non-negative integer (got: '{generate_genie_code_for_raw}')")
            elif _ggc_int == 0:
                log_print("ℹ️ Generate Genie Code Instruction For: 0 (disabled — skeleton notebooks only)")
            else:
                log_print(f"✅ Generate Genie Code Instruction For: top {_ggc_int} use cases (by Inspire Score desc)")
        except ValueError:
            validation_errors.append(f"❌ 'Generate Genie Code Instruction For' (13_generate_genie_code_for) must be 'all' or a non-negative integer (got: '{generate_genie_code_for_raw}')")
    
    if validation_errors:
        import sys as _sys
        error_count = len(validation_errors)
        error_summary = "\n".join(validation_errors)
        
        log_print("=" * 80, level="ERROR")
        log_print(f"❌ VALIDATION FAILED - {error_count} ERROR(S) FOUND:", level="ERROR")
        log_print("=" * 80, level="ERROR")
        for error in validation_errors:
            log_print(error, level="ERROR")
        log_print("=" * 80, level="ERROR")
        
        print(f"\n{'='*80}\n❌ VALIDATION ERRORS ({error_count}):\n{error_summary}\n{'='*80}\n", file=_sys.stderr, flush=True)
        _sys.stdout.flush()
        _sys.stderr.flush()
        
        exit_msg = f"Validation failed with {error_count} error(s):\n{error_summary}"
        dbutils.notebook.exit(exit_msg)
    
    log_print("=" * 80)
    log_print("✅ ALL VALIDATIONS PASSED - Starting generation...")
    log_print("=" * 80)

    # --- 3. Pack values and Run ---
    
    widget_values = {
        "operation": operation,
        "business": business_name,
        "inspire_database": inspire_database,
        "table_election_mode": table_election_mode,
        "use_cases_quality": use_cases_quality,
        "generation_instructions": generation_instructions_str,
            "_user_provided_session_id": _user_provided_session_id,
        "business_priorities": business_priorities_str,
        "business_domains": business_domains_str,
        "catalogs": catalogs_str,
        "schemas": schemas_str,
        "tables": tables_str,
        "generate": generate_str,
        "generation_path": generation_path,
        "output_language": output_language_str,
        "technical_exclusion_strategy": technical_exclusion_strategy,
        "json_file_path": json_file_path,
        "session_id": user_session_id,
        "generate_genie_code_for": generate_genie_code_for_raw,
    }
    widget_values["_running_in_job_context"] = _user_provided_session_id

    inspirer = None
    try:
        inspirer = DatabricksInspire(**widget_values)
        inspirer.run()
        inspirer.finalize_atomic_writer(success=True)

        if widget_values.get("_running_in_job_context"):
            try:
                _fin_results = getattr(inspirer, "final_inspire_results_json", {})
                _fin_domains = _fin_results.get("domains", [])
                _fin_uc_count = sum(
                    len(d.get("use_cases", []))
                    for d in _fin_domains
                )
                _fin_domain_count = len(_fin_domains)
                _fin_subdomain_set = set()
                for _d in _fin_domains:
                    for _uc in _d.get("use_cases", []):
                        _sd = _uc.get("subdomain", "")
                        if _sd:
                            _fin_subdomain_set.add((_d.get("domain_name", ""), _sd))
                _updated_tags = {
                    "dbx_inspire_ai_usecases": str(_fin_uc_count),
                    "dbx_inspire_ai_domains": str(_fin_domain_count),
                    "dbx_inspire_ai_subdomains": str(len(_fin_subdomain_set)),
                }
                JobLauncher.update_job_tags(_updated_tags)
                _logger = logging.getLogger("main")
                _logger.info(f"[JobTags] Updated job tags with final results: {_updated_tags}")
            except Exception as _tag_update_err:
                try:
                    logging.getLogger("main").debug(f"[JobTags] Tag update failed (non-critical): {_tag_update_err}")
                except Exception:
                    pass

    except NameError as ne:
        if inspirer:
            inspirer.finalize_atomic_writer(success=False, error_message=str(ne))
        if ('DataLoader' in str(ne) or 'AIAgent' in str(ne) or 
            'PROMPT_TEMPLATES' in str(ne) or 'DatabricksInspire' in str(ne) or 
            'setup_logging' in str(ne) or 'TranslationService' in str(ne)):
            
            print(f"ERROR: A required class, function, or variable is missing: {ne}", file=sys.stderr)
            print("Please ensure `setup_logging`, `DataLoader`, `AIAgent`, `PROMPT_TEMPLATES`, `TranslationService`, and `DatabricksInspire` are defined in preceding cells.", file=sys.stderr)
        else:
            raise
    except Exception as e:
        if inspirer:
            inspirer.finalize_atomic_writer(success=False, error_message=str(e))
        logging.getLogger("main").critical(f"Main execution failed: {e}")
        raise

In [0]:
if __name__ == "__main__":
    logging.getLogger("py4j").setLevel(logging.ERROR)
    main()